In [ ]:
from pathlib import Path

DESKTOP = Path.home() / "Desktop"
DATA_DIR_CANDIDATES = [
    DESKTOP / "NGFAID" / "NGFAID" / "6624956" / "2days" / "2days",
    DESKTOP / "NGFAID" / "2days",
    DESKTOP / "2days",
]


def resolve_data_dir() -> Path:
    for candidate in DATA_DIR_CANDIDATES:
        if candidate.exists():
            return candidate

    for marker_name in ["flight_header.csv", "flight_data.pkl"]:
        for marker in DESKTOP.rglob(marker_name):
            if marker.is_file():
                return marker.parent

    raise FileNotFoundError(
        "Unable to locate the local NGFAID 2days dataset. "
        f"Checked: {DATA_DIR_CANDIDATES} and recursive matches under {DESKTOP}"
    )


DATA_DIR = resolve_data_dir()


def resolve_data_file(filename: str) -> Path:
    for candidate_dir in [DATA_DIR, *DATA_DIR_CANDIDATES]:
        candidate = candidate_dir / filename
        if candidate.exists():
            return candidate

    for candidate in DESKTOP.rglob(filename):
        if candidate.is_file():
            return candidate

    raise FileNotFoundError(
        f"Unable to locate {filename!r} under {DESKTOP}"
    )


print(f"Using local dataset directory: {DATA_DIR}")



In [ ]:
import pickle
import pandas as pd

# Load data files
print("Loading data files...")

# Load training data
with open(str(resolve_data_file("train_pairs_sample_split_window_250_step_100.pkl")), 'rb') as f:
    train_pairs = pickle.load(f)

# Load label mappings
with open(str(resolve_data_file("label_mappings_sample_split_window_250_step_100.pkl")), 'rb') as f:
    label_mappings = pickle.load(f)

# Load original flight_header to get sensor information
flight_header = pd.read_csv(str(resolve_data_file("flight_header.csv")))

print("Data loading completed!\n")

# Extract sensor information
if len(train_pairs) > 0:
    sample_key = list(train_pairs.keys())[0]
    sample = train_pairs[sample_key]
    num_sensors = sample['x_before'].shape[1]
    window_length = sample['x_before'].shape[0]

    print(f"Sensor and Data Information:")
    print(f"  Number of Sensors: {num_sensors}")
    print(f"  Window Length: {window_length}")
    print(f"  Total Samples: {len(train_pairs)}\n")

# Create reverse label mappings
reverse_mappings = {}
for mapping_name, mapping in label_mappings.items():
    reverse_mappings[mapping_name] = {v: k for k, v in mapping.items()}

# Statistical analysis of maintenance operation categories
print("="*80)
print("Maintenance Operation Category Statistics")
print("="*80)

# Create statistics data list
stats_data = []

# Analyze each classification field
classification_fields = [
    ('class', 'class_encoded', 'class_to_id'),
    ('target_class', 'target_class_encoded', 'target_class_to_id'),
    ('label', 'label_encoded', 'label_to_id'),
    ('hclass', 'hclass_encoded', 'hclass_to_id')
]

for original_field, encoded_field, mapping_name in classification_fields:
    if mapping_name in label_mappings:
        print(f"\n[{original_field.upper()} Classification Statistics]")

        # Count encoded value distribution
        encoded_counts = {}
        for sample in train_pairs.values():
            if encoded_field in sample:
                enc_val = sample[encoded_field]
                encoded_counts[enc_val] = encoded_counts.get(enc_val, 0) + 1

        # Create table data
        table_data = []
        for enc_val, count in sorted(encoded_counts.items()):
            if enc_val in reverse_mappings[mapping_name]:
                original_val = reverse_mappings[mapping_name][enc_val]
            else:
                original_val = "Unknown"

            percentage = (count / len(train_pairs)) * 100
            table_data.append({
                'Original Category': original_val,
                'Encoded Value': enc_val,
                'Sample Count': count,
                'Percentage(%)': f"{percentage:.2f}%"
            })

        # Print table
        df_table = pd.DataFrame(table_data)
        print(df_table.to_string(index=False))

        stats_data.append({
            'Classification Field': original_field,
            'Number of Categories': len(encoded_counts),
            'Total Samples': len(train_pairs)
        })

# Print overall statistics summary
print("\n" + "="*80)
print("Overall Statistics Summary")
print("="*80)
df_summary = pd.DataFrame(stats_data)
print(df_summary.to_string(index=False))

# Print detailed sensor and data information table
print("\n" + "="*80)
print("Sensor and Data Detailed Information")
print("="*80)

sensor_info = {
    'Attribute': ['Number of Sensors', 'Window Length', 'Step Size', 'Total Training Samples', 'Data Dimensions'],
    'Value': [
        num_sensors,
        window_length,
        100,  # step_size
        len(train_pairs),
        f"({window_length}, {num_sensors})"
    ]
}
df_sensor = pd.DataFrame(sensor_info)
print(df_sensor.to_string(index=False))

print("\nData loading and statistics completed!")


## Part 1: Data Preprocessing and Dataset Construction


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, RobustScaler
from scipy.optimize import linear_sum_assignment
import warnings
warnings.filterwarnings('ignore')
import pickle

# ------------------------------
# Reproducibility
# ------------------------------
def set_random_seeds(seed=42):
    """Set all random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_random_seeds(42)

# ------------------------------
# Load data
# ------------------------------
flight_header = pd.read_csv(str(resolve_data_file("flight_header.csv")))
flight_data_dict = pd.read_pickle(str(resolve_data_file("flight_data.pkl")))

print(flight_header.head())
print("flight_header shape:", flight_header.shape)
# guard in case index 2 doesn't exist
example_key = 2 if 2 in flight_data_dict else next(iter(flight_data_dict))
print("flight_data_dict shape:", flight_data_dict[example_key].shape)
print("flight_data_dict length:", len(flight_data_dict))

# ------------------------------
# Preprocess
# ------------------------------
def preprocess_flight_data(flight_data_dict, flight_header):
    """
    Data preprocessing: keep only sequences that are in header and have finite values
    """
    processed_data_dict = {}
    print("Starting data preprocessing...")

    valid_master_indices = set(flight_header['Master Index'].values.tolist())

    for master_idx, seq_data in flight_data_dict.items():
        if master_idx not in valid_master_indices:
            continue

        # ensure float32
        seq_data = np.asarray(seq_data, dtype=np.float32)

        # skip NaN/inf or empty
        if seq_data.size == 0 or seq_data.shape[0] == 0:
            continue
        if not np.isfinite(seq_data).all():
            continue

        processed_data_dict[master_idx] = seq_data

    print(f"Data preprocessing completed! Number of processed flights: {len(processed_data_dict)}")
    return processed_data_dict

def compute_cost_matrix(before_group, after_group, lambda_weight=0.1):
    """
    Compute cost matrix for optimal matching between before and after maintenance flights
    """
    n_before, n_after = len(before_group), len(after_group)
    cost_matrix = np.full((n_before, n_after), np.inf, dtype=np.float32)

    # availability checks
    has_flight_length = ('flight_length' in before_group.columns) and ('flight_length' in after_group.columns)
    has_num_before = ('number_flights_before' in before_group.columns) and ('number_flights_before' in after_group.columns)

    for i, (_, br) in enumerate(before_group.iterrows()):
        for j, (_, ar) in enumerate(after_group.iterrows()):
            # Relaxed constraint: mirrored date difference within tolerance 3 (was 1)
            if abs(abs(br['date_diff']) - abs(ar['date_diff'])) > 3:
                continue

            # Fleet history count constraint (skip negativity like -1 meaning unknown)
            if has_num_before:
                nb_b = br['number_flights_before']
                nb_a = ar['number_flights_before']
                if nb_b >= 0 and nb_a >= 0 and abs(nb_b - nb_a) > 10:  # Relaxed from 5 to 10
                    continue

            # Base cost: date difference gap
            date_cost = abs(abs(br['date_diff']) - abs(ar['date_diff']))
            flight_length_cost = 0.0

            if has_flight_length:
                fl_b, fl_a = br['flight_length'], ar['flight_length']
                denom = max(fl_b, fl_a, 1)  # avoid zero division
                # Relaxed relative difference tolerance from 0.1 to 0.3
                if abs(fl_b - fl_a) / denom > 0.3:
                    continue
                flight_length_cost = abs(fl_b - fl_a)

            cost_matrix[i, j] = date_cost + lambda_weight * flight_length_cost

    return cost_matrix

def create_pair_mapping(flight_header):
    """
    Create pair mapping using maintenance grouping column (auto-detected) and Hungarian algorithm
    """
    print("Creating maintenance ID-based pair mapping...")

    # Auto-detect grouping column with fallback
    candidates = ['maintenance_id', 'maint_id', 'maintenance', 'maint', 'class', 'label']
    group_col = next((c for c in candidates if c in flight_header.columns), None)

    if group_col is None:
        print("Warning: No grouping column found; using single global group.")
        flight_header['_global_group_'] = 'ALL'
        group_col = '_global_group_'
    else:
        print(f"Using {group_col} for grouping")

    pair_mapping = {}
    pair_id = 0

    grouped = flight_header.groupby(group_col)

    total_pairs = 0
    total_possible_pairs = 0
    all_costs = []
    group_stats = []

    for group_key, group in grouped:
        # Separate before and after maintenance flights
        before_flights = group[(group['before_after'] == 1) & (group['date_diff'] < 0)]
        after_flights  = group[(group['before_after'] == 0) & (group['date_diff'] > 0)]

        n_before = len(before_flights)
        n_after  = len(after_flights)

        if n_before == 0 or n_after == 0:
            continue

        possible_pairs = min(n_before, n_after)
        total_possible_pairs += possible_pairs

        # Compute cost matrix
        cost_matrix = compute_cost_matrix(before_flights, after_flights)

        # Remove all-inf rows/cols
        valid_rows = ~np.all(np.isinf(cost_matrix), axis=1)
        valid_cols = ~np.all(np.isinf(cost_matrix), axis=0)

        if not np.any(valid_rows) or not np.any(valid_cols):
            print(f"Group {group_key}: No valid pairs found (all costs infinite)")
            group_stats.append({
                'group': group_key, 'before': n_before, 'after': n_after,
                'matched': 0, 'efficiency': 0.0
            })
            continue

        filtered_cost_matrix = cost_matrix[valid_rows][:, valid_cols]
        valid_before_indices = np.where(valid_rows)[0]
        valid_after_indices  = np.where(valid_cols)[0]

        # Still feasible?
        if filtered_cost_matrix.size == 0 or np.all(np.isinf(filtered_cost_matrix)):
            print(f"Group {group_key}: Filtered cost matrix is empty or all infinite")
            group_stats.append({
                'group': group_key, 'before': n_before, 'after': n_after,
                'matched': 0, 'efficiency': 0.0
            })
            continue

        # Check if the filtered cost matrix is square and feasible for Hungarian algorithm
        if filtered_cost_matrix.shape[0] == 0 or filtered_cost_matrix.shape[1] == 0:
            print(f"Group {group_key}: Cost matrix has zero dimensions")
            group_stats.append({
                'group': group_key, 'before': n_before, 'after': n_after,
                'matched': 0, 'efficiency': 0.0
            })
            continue

        # Handle rectangular cost matrix by padding with high cost values
        rows, cols = filtered_cost_matrix.shape
        if rows != cols:
            max_dim = max(rows, cols)
            padded_matrix = np.full((max_dim, max_dim), np.inf, dtype=np.float32)
            padded_matrix[:rows, :cols] = filtered_cost_matrix

            # Set padding cells to a high but finite cost to make matrix feasible
            max_finite_cost = np.max(filtered_cost_matrix[np.isfinite(filtered_cost_matrix)])
            if np.isfinite(max_finite_cost):
                padding_cost = max_finite_cost * 10  # High penalty for dummy assignments
            else:
                padding_cost = 1000.0  # Default high cost

            # Fill padding areas with high finite cost
            if rows < max_dim:
                padded_matrix[rows:, :cols] = padding_cost
            if cols < max_dim:
                padded_matrix[:rows, cols:] = padding_cost

            filtered_cost_matrix = padded_matrix

        try:
            row_indices, col_indices = linear_sum_assignment(filtered_cost_matrix)
        except ValueError as e:
            print(f"Group {group_key}: Hungarian algorithm failed - {e}")
            group_stats.append({
                'group': group_key, 'before': n_before, 'after': n_after,
                'matched': 0, 'efficiency': 0.0
            })
            continue

        # Group-level and global thresholds (more lenient)
        group_costs = []
        valid_assignments = []

        for r, c in zip(row_indices, col_indices):
            # Only consider assignments within original matrix bounds
            if r < len(valid_before_indices) and c < len(valid_after_indices):
                cost = float(filtered_cost_matrix[r, c])
                if np.isfinite(cost):
                    group_costs.append(cost)
                    valid_assignments.append((r, c, cost))

        if group_costs:
            local_thr = np.percentile(group_costs, 95)  # More lenient from 80 to 95
            all_costs.extend(group_costs)
        else:
            local_thr = np.inf

        global_thr = np.percentile(all_costs, 95) if len(all_costs) > 10 else np.inf  # More lenient from 90 to 95
        threshold = min(local_thr, global_thr, 10.0)  # Add absolute threshold cap

        # Meta columns (if present in both)
        meta_cols = [c for c in ['class', 'target_class', 'label', 'hclass']
                     if c in before_flights.columns and c in after_flights.columns]

        matched_pairs = 0
        for row_idx, col_idx, cost in valid_assignments:
            if cost > threshold:
                continue

            orig_before_idx = valid_before_indices[row_idx]
            orig_after_idx  = valid_after_indices[col_idx]

            before_master_idx = int(before_flights.iloc[orig_before_idx]['Master Index'])
            after_master_idx  = int(after_flights.iloc[orig_after_idx]['Master Index'])

            pair_info = {
                'before_master_idx': before_master_idx,
                'after_master_idx':  after_master_idx,
                'maintenance_group': group_key,
                'cost': cost
            }

            before_row = before_flights.iloc[orig_before_idx]
            for col in meta_cols:
                if col in before_row:
                    pair_info[col] = before_row[col]

            pair_mapping[pair_id] = pair_info
            pair_id += 1
            matched_pairs += 1

        group_stats.append({
            'group': group_key,
            'before': n_before,
            'after': n_after,
            'matched': matched_pairs,
            'efficiency': matched_pairs / possible_pairs if possible_pairs > 0 else 0.0,
            'avg_cost': float(np.mean(group_costs)) if group_costs else np.inf
        })

        total_pairs += matched_pairs

    # Quality report
    print(f"\nPair mapping completed:")
    print(f"  Total groups: {len(group_stats)}")
    print(f"  Total possible pairs: {total_possible_pairs}")
    print(f"  Successfully matched pairs: {total_pairs}")
    if total_possible_pairs > 0:
        overall_efficiency = total_pairs / total_possible_pairs
        print(f"  Overall efficiency: {overall_efficiency*100:.1f}%")
    if all_costs:
        print(f"  Cost statistics:")
        print(f"    Mean: {np.mean(all_costs):.3f}")
        print(f"    P50: {np.percentile(all_costs, 50):.3f}")
        print(f"    P80: {np.percentile(all_costs, 80):.3f}")
        print(f"    P90: {np.percentile(all_costs, 90):.3f}")

    # Sample quality check
    print(f"\nQuality check sample pairs:")
    sample_pairs = list(pair_mapping.keys())[:3]
    for pid in sample_pairs:
        pair_info = pair_mapping[pid]
        before_idx = pair_info['before_master_idx']
        after_idx  = pair_info['after_master_idx']

        before_row = flight_header[flight_header['Master Index'] == before_idx].iloc[0]
        after_row  = flight_header[flight_header['Master Index'] == after_idx].iloc[0]

        print(f"  Pair {pid}: cost={pair_info['cost']:.3f}")
        print(f"    Before maintenance: date_diff={before_row['date_diff']}, before_after={before_row['before_after']}")
        print(f"    After  maintenance: date_diff={after_row['date_diff']},  before_after={after_row['before_after']}")
        if 'flight_length' in flight_header.columns:
            print(f"    Flight length: {before_row['flight_length']:.1f} -> {after_row['flight_length']:.1f}")

    return pair_mapping

def create_sliding_windows_all_pairs(flight_data_dict, pair_mapping, window_size=500, step_size=250):
    """
    Generate sliding window samples for ALL pairs at once
    """
    all_samples = {}
    sample_id = 0

    print(f"Generating sliding window samples for all {len(pair_mapping)} pairs...")

    for pair_id, pair_info in pair_mapping.items():
        before_idx = pair_info['before_master_idx']
        after_idx  = pair_info['after_master_idx']

        # Check data exists
        if before_idx not in flight_data_dict or after_idx not in flight_data_dict:
            continue

        before_data = flight_data_dict[before_idx]
        after_data  = flight_data_dict[after_idx]

        # Strict filtering
        if (not np.isfinite(before_data).all()) or (not np.isfinite(after_data).all()):
            continue

        # Generate windows (before)
        before_windows = []
        if len(before_data) >= window_size:
            for start in range(0, len(before_data) - window_size + 1, step_size):
                before_windows.append(before_data[start:start + window_size])
        else:
            padded = np.zeros((window_size, before_data.shape[1]), dtype=np.float32)
            padded[:len(before_data)] = before_data
            before_windows.append(padded)

        # Generate windows (after)
        after_windows = []
        if len(after_data) >= window_size:
            for start in range(0, len(after_data) - window_size + 1, step_size):
                after_windows.append(after_data[start:start + window_size])
        else:
            padded = np.zeros((window_size, after_data.shape[1]), dtype=np.float32)
            padded[:len(after_data)] = after_data
            after_windows.append(padded)

        # Pair windows (round-robin)
        max_windows = max(len(before_windows), len(after_windows))
        for i in range(max_windows):
            b_win = before_windows[i % len(before_windows)]
            a_win = after_windows[i % len(after_windows)]

            if (not np.isfinite(b_win).all()) or (not np.isfinite(a_win).all()):
                continue

            sample_data = {
                "x_before": b_win.astype(np.float32),
                "x_after":  a_win.astype(np.float32),
                "pair_id":  pair_id,
                "matching_cost": float(pair_info['cost'])
            }

            for field in ['class', 'target_class', 'label', 'hclass']:
                if field in pair_info:
                    sample_data[field] = pair_info[field]

            all_samples[sample_id] = sample_data
            sample_id += 1

    print(f"Generated {len(all_samples)} sliding window samples from all pairs")
    return all_samples

def stratified_split_samples(all_samples, train_ratio=0.55, val_ratio=0.15, test_ratio=0.30, random_state=42):
    """
    Perform stratified split on samples based on class labels
    """
    print("Performing stratified split on samples...")

    # Extract sample IDs and labels for stratification
    sample_ids = list(all_samples.keys())

    # Determine stratification field (priority: class > label > default)
    stratify_field = None
    if any('class' in sample for sample in all_samples.values()):
        stratify_field = 'class'
    elif any('label' in sample for sample in all_samples.values()):
        stratify_field = 'label'

    if stratify_field is None:
        print("Warning: No class or label field found, using random split")
        # Random split without stratification
        np.random.seed(random_state)
        shuffled_ids = sample_ids.copy()
        np.random.shuffle(shuffled_ids)

        n_train = int(train_ratio * len(shuffled_ids))
        n_val = int(val_ratio * len(shuffled_ids))

        train_ids = shuffled_ids[:n_train]
        val_ids = shuffled_ids[n_train:n_train + n_val]
        test_ids = shuffled_ids[n_train + n_val:]
    else:
        print(f"Using {stratify_field} for stratified split")

        # Extract labels for stratification
        labels = []
        for sample_id in sample_ids:
            sample = all_samples[sample_id]
            if stratify_field in sample:
                labels.append(sample[stratify_field])
            else:
                labels.append('unknown')  # fallback for missing labels

        # First split: train vs (val + test)
        train_ids, temp_ids, _, temp_labels = train_test_split(
            sample_ids, labels,
            test_size=(val_ratio + test_ratio),
            stratify=labels,
            random_state=random_state
        )

        # Second split: val vs test
        val_test_ratio = val_ratio / (val_ratio + test_ratio)
        val_ids, test_ids, _, _ = train_test_split(
            temp_ids, temp_labels,
            test_size=(1 - val_test_ratio),
            stratify=temp_labels,
            random_state=random_state
        )

    print(f"Stratified split completed:")
    print(f"  Train samples: {len(train_ids)} ({len(train_ids)/len(sample_ids)*100:.1f}%)")
    print(f"  Val samples: {len(val_ids)} ({len(val_ids)/len(sample_ids)*100:.1f}%)")
    print(f"  Test samples: {len(test_ids)} ({len(test_ids)/len(sample_ids)*100:.1f}%)")

    # Create split dictionaries
    train_samples = {sid: all_samples[sid] for sid in train_ids}
    val_samples = {sid: all_samples[sid] for sid in val_ids}
    test_samples = {sid: all_samples[sid] for sid in test_ids}

    return train_samples, val_samples, test_samples

def compute_scaling_params(train_samples):
    """
    Compute normalization parameters using training set only
    """
    print("Computing normalization parameters...")
    all_data = []

    for sample in train_samples.values():
        xb = sample['x_before'].reshape(-1, sample['x_before'].shape[-1])
        xa = sample['x_after'].reshape(-1, sample['x_after'].shape[-1])
        all_data.append(xb)
        all_data.append(xa)

    if not all_data:
        raise ValueError("No training samples to compute scaling parameters.")

    combined_data = np.vstack(all_data)

    scaler = RobustScaler()
    scaler.fit(combined_data)

    print(f"Normalization parameters computed! Features: {combined_data.shape[1]}, samples: {combined_data.shape[0]}")
    return scaler

def apply_scaling(samples, scaler):
    """
    Apply normalization to samples
    """
    scaled_samples = {}

    for sample_id, sample in samples.items():
        scaled_sample = sample.copy()

        # before
        x_before = sample['x_before']
        xb = x_before.reshape(-1, x_before.shape[-1])
        xb_scaled = scaler.transform(xb)
        scaled_sample['x_before'] = xb_scaled.reshape(x_before.shape).astype(np.float32)

        # after
        x_after = sample['x_after']
        xa = x_after.reshape(-1, x_after.shape[-1])
        xa_scaled = scaler.transform(xa)
        scaled_sample['x_after'] = xa_scaled.reshape(x_after.shape).astype(np.float32)

        scaled_samples[sample_id] = scaled_sample

    return scaled_samples

def build_label_mappings(datasets):
    """
    Build label mappings from the union of labels across provided datasets.
    datasets: iterable of dicts {sample_id: sample_dict}
    """
    unique_classes = set()
    unique_target_classes = set()
    unique_labels = set()
    unique_hclasses = set()

    for ds in datasets:
        for sample in ds.values():
            if 'class' in sample:
                unique_classes.add(int(sample['class']))
            if 'target_class' in sample:
                unique_target_classes.add(int(sample['target_class']))
            if 'label' in sample:
                unique_labels.add(sample['label'])  # string labels are fine
            if 'hclass' in sample:
                unique_hclasses.add(int(sample['hclass']))

    label_mappings = {}

    if unique_classes:
        label_mappings['class_to_id'] = {cls: idx for idx, cls in enumerate(sorted(unique_classes))}
    if unique_target_classes:
        label_mappings['target_class_to_id'] = {cls: idx for idx, cls in enumerate(sorted(unique_target_classes))}
    if unique_labels:
        label_mappings['label_to_id'] = {lbl: idx for idx, lbl in enumerate(sorted(unique_labels))}
    if unique_hclasses:
        label_mappings['hclass_to_id'] = {hcls: idx for idx, hcls in enumerate(sorted(unique_hclasses))}

    return label_mappings

def encode_with_mappings(samples, label_mappings, unknown_id=-1):
    """
    Encode labels using provided mappings. Unseen labels map to unknown_id.
    """
    encoded_samples = {}
    for sample_id, sample in samples.items():
        enc = sample.copy()
        if 'class' in sample and 'class_to_id' in label_mappings:
            enc['class_encoded'] = label_mappings['class_to_id'].get(int(sample['class']), unknown_id)
        if 'target_class' in sample and 'target_class_to_id' in label_mappings:
            enc['target_class_encoded'] = label_mappings['target_class_to_id'].get(int(sample['target_class']), unknown_id)
        if 'label' in sample and 'label_to_id' in label_mappings:
            enc['label_encoded'] = label_mappings['label_to_id'].get(sample['label'], unknown_id)
        if 'hclass' in sample and 'hclass_to_id' in label_mappings:
            enc['hclass_encoded'] = label_mappings['hclass_to_id'].get(int(sample['hclass']), unknown_id)
        encoded_samples[sample_id] = enc
    return encoded_samples

def final_validation(samples):
    """
    Final validation to ensure no NaN or inf values
    """
    valid_samples = {}
    removed_count = 0

    for sample_id, sample in samples.items():
        x_before = sample['x_before']
        x_after  = sample['x_after']

        if (not np.isfinite(x_before).all()) or (not np.isfinite(x_after).all()):
            removed_count += 1
            continue

        valid_samples[sample_id] = sample

    print(f"Final validation: {len(valid_samples)} valid samples, {removed_count} removed")
    return valid_samples

def save_data(data, filename, save_dir):
    """
    Save processed data to specified directory
    """
    os.makedirs(save_dir, exist_ok=True)
    filepath = os.path.join(save_dir, filename)

    with open(filepath, 'wb') as f:
        pickle.dump(data, f)

    print(f"Data saved to: {filepath}")

def analyze_pair_information(pairs_train, pairs_val, pairs_test, label_mappings):
    """
    Analyze and print detailed information about pairs including class distribution
    """
    print("\n" + "="*80)
    print("DETAILED PAIR ANALYSIS")
    print("="*80)

    datasets = {
        'Training': pairs_train,
        'Validation': pairs_val,
        'Test': pairs_test
    }

    for dataset_name, dataset in datasets.items():
        print(f"\n{dataset_name.upper()} SET ANALYSIS:")
        print("-" * 50)
        print(f"Total samples: {len(dataset)}")

        if len(dataset) == 0:
            print("No samples in this dataset")
            continue

        # Analyze class distributions
        class_fields = ['class', 'target_class', 'label', 'hclass']

        for field in class_fields:
            if any(field in sample for sample in dataset.values()):
                print(f"\n{field.upper()} DISTRIBUTION:")
                class_counts = {}
                for sample in dataset.values():
                    if field in sample:
                        class_val = sample[field]
                        class_counts[class_val] = class_counts.get(class_val, 0) + 1

                # Sort by class value for consistent output
                sorted_classes = sorted(class_counts.items())
                for class_val, count in sorted_classes:
                    percentage = (count / len(dataset)) * 100
                    print(f"  {field} {class_val}: {count} samples ({percentage:.1f}%)")

        # Analyze encoded class distributions
        encoded_fields = ['class_encoded', 'target_class_encoded', 'label_encoded', 'hclass_encoded']

        for field in encoded_fields:
            if any(field in sample for sample in dataset.values()):
                print(f"\n{field.upper()} DISTRIBUTION:")
                class_counts = {}
                for sample in dataset.values():
                    if field in sample:
                        class_val = sample[field]
                        class_counts[class_val] = class_counts.get(class_val, 0) + 1

                # Sort by encoded value
                sorted_classes = sorted(class_counts.items())
                for encoded_val, count in sorted_classes:
                    percentage = (count / len(dataset)) * 100
                    print(f"  Encoded value {encoded_val}: {count} samples ({percentage:.1f}%)")

        # Analyze matching costs
        costs = [sample['matching_cost'] for sample in dataset.values() if 'matching_cost' in sample]
        if costs:
            print(f"\nMATCHING COST STATISTICS:")
            print(f"  Mean cost: {np.mean(costs):.4f}")
            print(f"  Median cost: {np.median(costs):.4f}")
            print(f"  Min cost: {np.min(costs):.4f}")
            print(f"  Max cost: {np.max(costs):.4f}")
            print(f"  Std deviation: {np.std(costs):.4f}")

        # Analyze pair IDs
        pair_ids = [sample['pair_id'] for sample in dataset.values() if 'pair_id' in sample]
        if pair_ids:
            unique_pairs = len(set(pair_ids))
            print(f"\nPAIR ID STATISTICS:")
            print(f"  Unique pairs: {unique_pairs}")
            print(f"  Total samples: {len(pair_ids)}")
            print(f"  Samples per pair (avg): {len(pair_ids) / unique_pairs:.1f}")

    # Overall summary
    total_samples = len(pairs_train) + len(pairs_val) + len(pairs_test)
    print(f"\n" + "="*80)
    print("OVERALL SUMMARY:")
    print("="*80)
    print(f"Total samples across all sets: {total_samples}")
    print(f"Training samples: {len(pairs_train)} ({len(pairs_train)/total_samples*100:.1f}%)")
    print(f"Validation samples: {len(pairs_val)} ({len(pairs_val)/total_samples*100:.1f}%)")
    print(f"Test samples: {len(pairs_test)} ({len(pairs_test)/total_samples*100:.1f}%)")

    # Label mapping summary
    print(f"\nLABEL MAPPING SUMMARY:")
    for mapping_name, mapping in label_mappings.items():
        print(f"  {mapping_name}: {len(mapping)} unique values")
        if len(mapping) <= 20:  # Only show details for small mappings
            print(f"    Values: {sorted(mapping.keys())}")

# ------------------------------
# Pipeline
# ------------------------------
window_size = 250
step_size   = 100

print("="*60)
print("Starting sample-level stratified split data processing pipeline")
print("="*60)

# Step 1: Data preprocessing
processed_data_dict = preprocess_flight_data(flight_data_dict, flight_header)

# Step 2: Pair mapping
pair_mapping = create_pair_mapping(flight_header)

# Step 3: Generate ALL sliding window samples first
print("\nGenerating all sliding window samples...")
all_samples = create_sliding_windows_all_pairs(processed_data_dict, pair_mapping, window_size, step_size)

# Step 4: Stratified split on sample level (55% train, 15% val, 30% test)
train_samples, val_samples, test_samples = stratified_split_samples(
    all_samples, train_ratio=0.55, val_ratio=0.15, test_ratio=0.30, random_state=42
)

print(f"\nSample-level stratified split completed!")
print(f"Training samples: {len(train_samples)}")
print(f"Validation samples: {len(val_samples)}")
print(f"Test samples: {len(test_samples)}")

# Step 5: Normalization on train only
scaler = compute_scaling_params(train_samples)

# Step 6: Apply normalization to all
print("\nApplying normalization...")
train_samples_scaled = apply_scaling(train_samples, scaler)
val_samples_scaled   = apply_scaling(val_samples, scaler)
test_samples_scaled  = apply_scaling(test_samples, scaler)

# Step 7: Create label mappings from union (train+val+test)
print("Creating label mappings (union of train/val/test)...")
label_mappings = build_label_mappings([train_samples_scaled, val_samples_scaled, test_samples_scaled])

# Encode all sets safely (unknown → -1)
train_samples_encoded = encode_with_mappings(train_samples_scaled, label_mappings, unknown_id=-1)
val_samples_encoded   = encode_with_mappings(val_samples_scaled,   label_mappings, unknown_id=-1)
test_samples_encoded  = encode_with_mappings(test_samples_scaled,  label_mappings, unknown_id=-1)

# Step 8: Final validation
print("\nFinal validation...")
pairs_train = final_validation(train_samples_encoded)
pairs_val   = final_validation(val_samples_encoded)
pairs_test  = final_validation(test_samples_encoded)

# Step 9: Detailed pair analysis
analyze_pair_information(pairs_train, pairs_val, pairs_test, label_mappings)

# Step 10: Save
save_dir = str(DATA_DIR)

print("\nSaving processed data...")
save_data(pairs_train, f"train_pairs_sample_split_window_{window_size}_step_{step_size}.pkl", save_dir)
save_data(pairs_val,   f"val_pairs_sample_split_window_{window_size}_step_{step_size}.pkl",   save_dir)
save_data(pairs_test,  f"test_pairs_sample_split_window_{window_size}_step_{step_size}.pkl",  save_dir)
save_data(pair_mapping, f"pair_mapping_sample_split_window_{window_size}_step_{step_size}.pkl", save_dir)
save_data(label_mappings, f"label_mappings_sample_split_window_{window_size}_step_{step_size}.pkl", save_dir)
save_data(scaler, f"scaler_sample_split_window_{window_size}_step_{step_size}.pkl", save_dir)

print("\n" + "="*60)
print("Sample-level stratified split data processing pipeline completed")
print("="*60)

print(f"\nFinal dataset statistics:")
print(f"  Training set:   {len(pairs_train)} pair samples")
print(f"  Validation set: {len(pairs_val)} pair samples")
print(f"  Test set:       {len(pairs_test)} pair samples")

print(f"\nLabel mapping information:")
if 'class_to_id' in label_mappings:
    print(f"  Main classes:   {len(label_mappings['class_to_id'])}")
if 'target_class_to_id' in label_mappings:
    print(f"  Target classes: {len(label_mappings['target_class_to_id'])}")
if 'label_to_id' in label_mappings:
    print(f"  Labels:         {len(label_mappings['label_to_id'])}")
if 'hclass_to_id' in label_mappings:
    print(f"  H classes:      {len(label_mappings['hclass_to_id'])}")

print(f"\nData preprocessing information:")
print(f"  Window size: {window_size}")
print(f"  Step size:   {step_size}")
print(f"  Normalization method: RobustScaler")
print(f"  Pairing method: Maintenance ID-based Hungarian algorithm")
print(f"  Dataset split: Sample-level stratified split")
print(f"  Random seed: 42")

print(f"\nSaved files:")
print(f"  Directory: {save_dir}")
print(f"  train_pairs_sample_split_window_{window_size}_step_{step_size}.pkl")
print(f"  val_pairs_sample_split_window_{window_size}_step_{step_size}.pkl")
print(f"  test_pairs_sample_split_window_{window_size}_step_{step_size}.pkl")
print(f"  pair_mapping_sample_split_window_{window_size}_step_{step_size}.pkl")
print(f"  label_mappings_sample_split_window_{window_size}_step_{step_size}.pkl")
print(f"  scaler_sample_split_window_{window_size}_step_{step_size}.pkl")

# Display sample info
if len(pairs_train) > 0:
    sample_key = list(pairs_train.keys())[0]
    sample = pairs_train[sample_key]
    print(f"\nSample information:")
    print(f"  Before maintenance data shape: {sample['x_before'].shape}")
    print(f"  After maintenance data shape:  {sample['x_after'].shape}")
    if 'class_encoded' in sample:
        print(f"  Encoded label: class={sample['class_encoded']}")
    if 'target_class_encoded' in sample:
        print(f"  Target class encoded: {sample['target_class_encoded']}")
    if 'label_encoded' in sample:
        print(f"  Label encoded: {sample['label_encoded']}")
    if 'hclass_encoded' in sample:
        print(f"  H class encoded: {sample['hclass_encoded']}")
    print(f"  Matching cost: {sample['matching_cost']:.4f}")

print(f"\nAll sample-level stratified split data processing completed and saved, ready for model training!")


In [ ]:
all_ids = set(all_samples.keys())
used_ids = set(pairs_train.keys()) | set(pairs_val.keys()) | set(pairs_test.keys())

print("All samples used:", all_ids == used_ids)
print("Number of unused samples:", len(all_ids - used_ids))
print("Number of duplicated samples:", len(used_ids) - (len(pairs_train) + len(pairs_val) + len(pairs_test)))


In [ ]:
from collections import Counter

def _count_by_field(dataset: dict, field: str) -> dict:
    """Count samples per value of a given field in a dataset dict."""
    ctr = Counter()
    for sample in dataset.values():
        if field in sample:
            ctr[sample[field]] += 1
    # Sort numeric values in ascending order and strings alphabetically
    try:
        return dict(sorted(ctr.items(), key=lambda kv: (isinstance(kv[0], str), kv[0])))
    except TypeError:
        # Avoid sorting errors when mixed data types are present
        return dict(sorted(ctr.items(), key=lambda kv: str(kv[0])))

def print_split_counts(name: str, dataset: dict, fields=('class',)):
    """Print how many categories and how many samples per category for given fields."""
    total = len(dataset)
    print(f"\n===== {name.upper()} =====")
    print(f"Total samples: {total}")

    for field in fields:
        counts = _count_by_field(dataset, field)
        if not counts:
            print(f"[{field}] not found in this split.")
            continue
        print(f"\n[{field}] categories: {len(counts)}")
        for k, v in counts.items():
            print(f"  {field} {k}: {v}")

# Example: summarize counts by class (most commonly used)
print_split_counts("train", pairs_train, fields=("class",))
print_split_counts("val",   pairs_val,   fields=("class",))
print_split_counts("test",  pairs_test,  fields=("class",))

print_split_counts("train", pairs_train, fields=("class","target_class","label"))
print_split_counts("val",   pairs_val,   fields=("class","target_class","label"))
print_split_counts("test",  pairs_test,  fields=("class","target_class","label"))


## Part 2: Reload Saved Processed Data


In [ ]:
# -*- coding: utf-8 -*-
"""
Load sample-level split dataset saved by preprocessing pipeline,
print split counts for multiple fields, and compute/print class weights.
"""

import os
import pickle
from typing import Dict, Any, Tuple, List
from collections import Counter

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


# ----------------------------
# Custom Dataset Class
# ----------------------------
class SampleDataset(Dataset):
    """PyTorch Dataset class for wrapping sample dictionaries"""

    def __init__(self, samples_dict: dict):
        self.samples = list(samples_dict.values())
        self.keys = list(samples_dict.keys())

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # Get class_encoded with proper handling of missing values
        class_encoded = sample.get('class_encoded', -100)  # Use -100 as ignore_index
        if class_encoded is None or class_encoded < 0:
            class_encoded = -100

        return {
            'x_before': torch.from_numpy(sample['x_before']).float(),
            'x_after': torch.from_numpy(sample['x_after']).float(),
            'y': torch.tensor(int(class_encoded), dtype=torch.long),  # Fixed: return y in dict
            'class_encoded': sample.get('class_encoded', -100),  # Use -100 instead of 0
            'label_encoded': sample.get('label_encoded', -100),  # Use -100 instead of 0
            'target_class_encoded': sample.get('target_class_encoded', -100),  # Use -100 instead of 0
            'hclass_encoded': sample.get('hclass_encoded', -100),  # Use -100 instead of 0
            'fault_class': sample.get('class', ''),  # Renamed to avoid keyword conflict
            'label': sample.get('label', ''),
            'target_class': sample.get('target_class', ''),
            'meta': {
                'pair_id': self.keys[idx],
                'matching_cost': sample.get('matching_cost', 0.0)
            }
        }


# ----------------------------
# I/O Helper Functions
# ----------------------------
def _load_pickle(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)


def load_sample_split_artifacts(
    save_dir: str,
    window_size: int,
    step_size: int,
) -> Dict[str, Any]:
    """
    Load artifacts saved by *sample-level stratified split* pipeline:
      - train_pairs_sample_split_window_{W}_step_{S}.pkl

      - val_pairs_sample_split_window_{W}_step_{S}.pkl
      - test_pairs_sample_split_window_{W}_step_{S}.pkl
      - pair_mapping_sample_split_window_{W}_step_{S}.pkl
      - label_mappings_sample_split_window_{W}_step_{S}.pkl
      - scaler_sample_split_window_{W}_step_{S}.pkl
    """
    def fname(stem: str) -> str:
        return os.path.join(
            save_dir,
            f"{stem}_window_{window_size}_step_{step_size}.pkl"
        )

    artifacts = {
        "train": _load_pickle(fname("train_pairs_sample_split")),
        "val": _load_pickle(fname("val_pairs_sample_split")),
        "test": _load_pickle(fname("test_pairs_sample_split")),
        "pair_mapping": _load_pickle(fname("pair_mapping_sample_split")),
        "label_mappings": _load_pickle(fname("label_mappings_sample_split")),
        "scaler": _load_pickle(fname("scaler_sample_split")),
    }

    print("[OK] Loaded artifacts (sample-level split) from:", save_dir)
    print(f"  Train samples: {len(artifacts['train'])} | Val samples: {len(artifacts['val'])} | Test samples: {len(artifacts['test'])}")

    # Shape probing
    for name in ("train", "val", "test"):
        ds = artifacts[name]
        if len(ds) > 0:
            probe = next(iter(ds.values()))
            xb, xa = probe["x_before"], probe["x_after"]
            print(f"  Example window shape ({name}) -> x_before: {xb.shape}, x_after: {xa.shape}")
            break

    return artifacts


# ----------------------------
# Statistics Printer
# ----------------------------
def _count_by_field(dataset: dict, field: str) -> dict:
    """Count number of samples for each value of given field in dataset dict."""
    ctr = Counter()
    for sample in dataset.values():
        if field in sample:
            ctr[sample[field]] += 1
    # Unified string sorting to avoid mixed type errors
    return dict(sorted(ctr.items(), key=lambda kv: str(kv[0])))


def print_split_counts(name: str, dataset: dict, fields=('class',)):
    """Print how many classes and samples per class for given fields."""
    total = len(dataset)
    print(f"\n===== {name.upper()} =====")
    print(f"Total samples: {total}")

    for field in fields:
        counts = _count_by_field(dataset, field)
        if not counts:
            print(f"[{field}] Not found in this split.")
            continue
        print(f"\n[{field}] Number of classes: {len(counts)}")
        for k, v in counts.items():
            print(f"  {field} {k}: {v}")


# ----------------------------
# Class Weight Helper Functions
# ----------------------------
def _extract_labels(dataset: dict, label_key: str) -> np.ndarray:
    """Extract label array for given label_key from dataset dict."""
    ys: List[int] = []
    for s in dataset.values():
        if label_key in s:
            y = s[label_key]
            if y is not None and y >= 0:  # Only include valid labels (>= 0)
                ys.append(int(y))
    return np.asarray(ys, dtype=np.int64)


def compute_class_weights(
    labels: np.ndarray,
    strategy: str = "balanced"
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute weight for each class.
    Returns (classes_sorted, weights_aligned).
      - "balanced": N / (C * n_c)  —— equivalent to sklearn.compute_class_weight('balanced', ...)
      - "inv_freq": 1 / n_c
    """
    if labels.size == 0:
        raise ValueError("No labels found to compute class weights.")
    ctr = Counter(labels.tolist())
    classes = np.array(sorted(ctr.keys()), dtype=np.int64)
    counts  = np.array([ctr[c] for c in classes], dtype=np.float64)

    if strategy == "balanced":
        N = float(labels.size)
        C = float(len(classes))
        weights = N / (C * counts)
    elif strategy == "inv_freq":
        weights = 1.0 / counts
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    return classes, weights


def print_class_weights_for_field(
    dataset_train: dict,
    label_key: str,
    title: str = None,
    strategy: str = "balanced"
) -> torch.Tensor:
    """
    Compute and print class weights for given field.
    Returns weight tensor.
    """
    if title is None:
        title = f"Class weights ({label_key}, {strategy})"
    y = _extract_labels(dataset_train, label_key)
    if y.size == 0:
        print(f"\n[{title}] Key '{label_key}' has no available labels.")
        return torch.ones(1, dtype=torch.float32)

    classes, weights = compute_class_weights(y, strategy=strategy)
    print(f"\n[{title}] Length={len(classes)}")
    for c, w in zip(classes, weights):
        print(f"  id={int(c):2d}  weight={float(w):.6f}")

    # Also show tensor aligned with max id (useful for CE loss)
    max_id = int(classes.max())
    class_weights_tensor = torch.ones(max_id + 1, dtype=torch.float32)
    class_weights_tensor[classes] = torch.from_numpy(weights).float()
    print(f"  -> Weight tensor shape: {tuple(class_weights_tensor.shape)}")

    return class_weights_tensor


def find_best_label_key(dataset: dict, preferred_keys: tuple) -> str:
    """Find the label key with highest coverage in the dataset."""
    coverage_scores = {}
    for key in preferred_keys:
        coverage = sum(1 for s in dataset.values() if key in s and s.get(key) is not None and s.get(key) >= 0)
        coverage_scores[key] = coverage

    if not coverage_scores or max(coverage_scores.values()) == 0:
        return None

    # Return key with highest coverage
    return max(coverage_scores.items(), key=lambda x: x[1])[0]


# ----------------------------
# Main Function
# ----------------------------
if __name__ == "__main__":
    # ==== Configuration ====
    SAVE_DIR = str(DATA_DIR)
    WINDOW_SIZE = 250
    STEP_SIZE = 100
    BATCH_SIZE = 128
    NUM_WORKERS = min(os.cpu_count() // 2, 2) if os.cpu_count() else 2
    PIN_MEMORY = torch.cuda.is_available()
    # ===============================================

    # Set random seeds for reproducibility
    torch.manual_seed(42)
    np.random.seed(42)

    artifacts = load_sample_split_artifacts(SAVE_DIR, WINDOW_SIZE, STEP_SIZE)

    pairs_train = artifacts["train"]
    pairs_val   = artifacts["val"]
    pairs_test  = artifacts["test"]

    # ---- Print counts (original fields) ----
    print_split_counts("train", pairs_train, fields=("class",))
    print_split_counts("val",   pairs_val,   fields=("class",))
    print_split_counts("test",  pairs_test,  fields=("class",))

    print_split_counts("train", pairs_train, fields=("class","target_class","label"))
    print_split_counts("val",   pairs_val,   fields=("class","target_class","label"))
    print_split_counts("test",  pairs_test,  fields=("class","target_class","label"))

    # ---- Print counts (encoded fields, if present) ----
    print_split_counts("train-ENC", pairs_train, fields=("class_encoded","target_class_encoded","label_encoded","hclass_encoded"))
    print_split_counts("val-ENC",   pairs_val,   fields=("class_encoded","target_class_encoded","label_encoded","hclass_encoded"))
    print_split_counts("test-ENC",  pairs_test,  fields=("class_encoded","target_class_encoded","label_encoded","hclass_encoded"))

    # ---- Compute and print class weights (choose your key) ----
    # Use improved key selection based on coverage
    preferred_keys = ("class_encoded", "label_encoded", "target_class_encoded", "hclass_encoded", "class")
    chosen_key = find_best_label_key(pairs_train, preferred_keys)

    class_weights = None
    if chosen_key is None:
        raise ValueError("[ERROR] No suitable label key found to compute class weights. Cannot proceed with training.")
    else:
        print(f"\n[INFO] Selected label key: '{chosen_key}' (highest coverage)")
        class_weights = print_class_weights_for_field(pairs_train, chosen_key, title=f"Balanced class weights (train) - {chosen_key}", strategy="balanced")
        # Also print inv_freq version for reference:
        print_class_weights_for_field(pairs_train, chosen_key, title=f"Inverse frequency class weights (train) - {chosen_key}", strategy="inv_freq")

    # ---- Create DataLoaders with improved performance settings ----
    train_dataset = SampleDataset(pairs_train)
    val_dataset = SampleDataset(pairs_val)
    test_dataset = SampleDataset(pairs_test)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=True if NUM_WORKERS > 0 else False
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=True if NUM_WORKERS > 0 else False
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=True if NUM_WORKERS > 0 else False
    )

    print(f"\n===== DATALOADER INFO =====")
    print(f"Train DataLoader: {len(train_loader)} batches, batch size: {BATCH_SIZE}")
    print(f"Val DataLoader: {len(val_loader)} batches, batch size: {BATCH_SIZE}")
    print(f"Test DataLoader: {len(test_loader)} batches, batch size: {BATCH_SIZE}")
    print(f"Performance settings: num_workers={NUM_WORKERS}, pin_memory={PIN_MEMORY}")

    # Save metadata for consistency
    metadata = {
        'K': int(class_weights.shape[0]),
        'sensor_dim': pairs_train[list(pairs_train.keys())[0]]['x_before'].shape[-1],
        'class_weights': class_weights.numpy().tolist(),
        'chosen_label_key': chosen_key,
        'id_to_name': {i: f"Class_{i}" for i in range(class_weights.shape[0])}
    }

    import json
    with open("dataset_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)
    print(f"\n[INFO] Saved dataset metadata to dataset_metadata.json")


## Part 3A: Baseline Model Architecture and Training


In [ ]:


# ============================================================
# Monotonic-Decreasing HI + aligned plotting (before -> after)
# Adapted to external train/val/test loaders that yield:
#   batch = {
#       "x_before": (B, L, C), "x_after": (B, L, C),
#       "y": LongTensor[B] or None,
#       "meta": list[dict] (each may contain 'pair_id', 'matching_cost', ...)
#   }
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import os
import random
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix

# ---------------------------------------------------------------------
# Utilities for the NEW loaders
# ---------------------------------------------------------------------
def adapt_batch(batch, device):
    """
    Adapt a batch from your loaders to the tensors expected by the model.
    Ensures fixed-length mask/lengths and extracts optional meta.
    """
    xb = batch["x_before"].to(device)  # (B, L, C)
    xa = batch["x_after"].to(device)   # (B, L, C)

    B, L, _ = xb.shape
    mask = torch.ones(B, L, dtype=xb.dtype, device=device)
    lengths = torch.full((B,), L, dtype=torch.long, device=device)

    labels = batch.get("y", None)
    if labels is not None and isinstance(labels, torch.Tensor):
        # Filter out ignore_index labels
        valid_mask = labels != -100
        if valid_mask.any():
            labels = labels.to(device)
        else:
            labels = None
    elif labels is not None:
        labels = None

    meta = batch.get("meta", None)
    sample_ids = []
    matching_costs = torch.zeros(B, dtype=torch.float32, device=device)
    if isinstance(meta, (list, tuple)) and len(meta) == B:
        for i, m in enumerate(meta):
            if isinstance(m, dict):
                sample_ids.append(str(m.get("pair_id", f"sample_{i}")))
                if "matching_cost" in m and m["matching_cost"] is not None:
                    matching_costs[i] = float(m["matching_cost"])
            else:
                sample_ids.append(f"sample_{i}")
    else:
        sample_ids = [f"sample_{i}" for i in range(B)]

    return xb, xa, labels, mask, lengths, sample_ids, matching_costs


def scan_num_classes_from_loader(train_loader):
    """
    Determine K (number of classes) from the training loader.
    """
    max_class = -1
    for batch in train_loader:
        y = batch.get("y", None)
        if y is None:
            continue
        if isinstance(y, torch.Tensor):
            y_np = y.detach().cpu().numpy()
            valid_labels = y_np[y_np >= 0]  # filter out negative labels
            if len(valid_labels) > 0:
                max_class = max(max_class, int(valid_labels.max()))

    if max_class < 0:
        raise ValueError("No valid labels found in the training loader to determine K.")

    K = max_class + 1
    return K


def build_default_id_to_name(K):
    return {i: f"Class_{i}" for i in range(K)}


# ---------------------------------------------------------------------
# Core model components
# ---------------------------------------------------------------------
class BoltzmannKAN(nn.Module):
    def __init__(self, in_ch, out_ch=4):
        super().__init__()
        self.E  = nn.Parameter(torch.zeros(out_ch, in_ch))
        self.kT = nn.Parameter(torch.ones(out_ch, in_ch))
    def forward(self, x):
        B,T,C = x.shape
        kT = torch.clamp(F.softplus(self.kT), 0.01, 10.0).unsqueeze(0).unsqueeze(2)
        E  = torch.clamp(self.E, -10.0, 10.0).unsqueeze(0).unsqueeze(2)
        x_ = torch.clamp(x.unsqueeze(1), -10.0, 10.0)
        exp = torch.clamp((E - x_) / kT, -50, 50)
        w   = torch.sigmoid(exp)
        h   = (w * x_).sum(dim=3).permute(0, 2, 1)
        return torch.clamp(F.softplus(h), 0.0, 100.0)

class BaseOp(nn.Module):
    def __init__(self):
        super().__init__()
        self.param_modulator = None
    def set_param_modulator(self, modulator): self.param_modulator = modulator
    def get_modulated_params(self, context):
        if self.param_modulator is None: return {}
        return self.param_modulator(context)

class MonotonicLinearOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_bias = self.bias
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        bias  = torch.clamp(base_bias + mod.get('bias_offset', 0.0), -5.0, 5.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * (xm + bias)), 0.0, 100.0)

class MonotonicFlatOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 1e-3, 1.0
    def _cum(self, x):
        B, T = x.shape
        diff = F.relu(x[:, 1:] - x[:, :-1])
        zero_init = torch.zeros(B, 1, device=x.device, dtype=x.dtype)
        return torch.cat([zero_init, torch.cumsum(diff, dim=1)], dim=1)
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_bias = self.bias
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        bias  = torch.clamp(base_bias + mod.get('bias_offset', 0.0), -2.0, 2.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        cum_result = self._cum(xm)
        return torch.clamp(F.softplus(scale * (cum_result + bias)), 0.0, 100.0)

class ConcaveLogOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.eps = 1e-3
        self.smin, self.smax = 0.01, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        eps  = torch.clamp(self.eps + mod.get('eps_offset', 0.0), 1e-6, 1e-1)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * torch.log(torch.abs(xm) + eps)), 0.0, 100.0)

class SaturationSigmoidOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.raw_slope = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
        self.lmin, self.lmax = 0.1, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_slope = torch.clamp(F.softplus(self.raw_slope), self.lmin, self.lmax)
        base_bias  = self.bias
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        slope = torch.clamp(base_slope + mod.get('slope_offset', 0.0), self.lmin, self.lmax)
        bias  = torch.clamp(base_bias  + mod.get('bias_offset', 0.0), -3.0, 3.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * torch.sigmoid(slope * (xm - bias))), 0.0, 100.0)

class HingeReLUOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.threshold = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_threshold = self.threshold
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        threshold = torch.clamp(base_threshold + mod.get('threshold_offset', 0.0), -3.0, 3.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * F.relu(xm - threshold)), 0.0, 100.0)

class PolynomialOp(BaseOp):
    def __init__(self, deg=2):  # Reduced default degree for stability
        super().__init__()
        self.raw_coeff = nn.Parameter(torch.zeros(deg))
        self.deg = deg
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_coeff = torch.clamp(F.softplus(self.raw_coeff), 0.01, 5.0)
        xm = torch.clamp(torch.tanh(h.mean(-1)), -3.0, 3.0)  # Added tanh for stability
        coeff_offset = mod.get('coeff_offset', torch.zeros_like(base_coeff))
        if coeff_offset.dim() > 1:
            coeff_offset = coeff_offset.mean(0)
        coeff = torch.clamp(base_coeff + coeff_offset, 0.01, 5.0)
        y = torch.zeros_like(xm)
        for i in range(self.deg):
            y = y + coeff[i] * torch.clamp(xm ** (i + 1), -100.0, 100.0)
        return torch.clamp(F.softplus(y), 0.0, 100.0)

class DampedSinOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.raw_freq = nn.Parameter(torch.tensor(0.0))
        self.raw_lambda = nn.Parameter(torch.tensor(0.0))
        self.phase = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
        self.fmin, self.fmax = 0.1, 5.0
        self.lmin, self.lmax = 0.01, 3.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_freq  = torch.clamp(F.softplus(self.raw_freq), self.fmin, self.fmax)
        base_lam   = torch.clamp(F.softplus(self.raw_lambda), self.lmin, self.lmax)
        base_phase = torch.clamp(self.phase, -10.0, 10.0)
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        freq  = torch.clamp(base_freq  + mod.get('freq_offset', 0.0), self.fmin, self.fmax)
        lam   = torch.clamp(base_lam   + mod.get('lambda_offset', 0.0), self.lmin, self.lmax)
        phase = torch.clamp(base_phase + mod.get('phase_offset', 0.0), -10.0, 10.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        damp = 1 / (1 + lam * torch.abs(xm))
        return torch.clamp(F.softplus(scale * damp * (torch.sin(freq * xm + phase) + 1)), 0.0, 100.0)

class PiecewiseLinearOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_k1 = nn.Parameter(torch.tensor(0.0))
        self.raw_k2 = nn.Parameter(torch.tensor(0.0))
        self.threshold = nn.Parameter(torch.tensor(0.0))
        self.kmin, self.kmax = 0.01, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_k1 = torch.clamp(F.softplus(self.raw_k1), self.kmin, self.kmax)
        base_k2 = torch.clamp(F.softplus(self.raw_k2), self.kmin, self.kmax)
        base_thr = torch.clamp(self.threshold, -5.0, 5.0)
        k1 = torch.clamp(base_k1 + mod.get('k1_offset', 0.0), self.kmin, self.kmax)
        k2 = torch.clamp(base_k2 + mod.get('k2_offset', 0.0), self.kmin, self.kmax)
        thr= torch.clamp(base_thr + mod.get('threshold_offset', 0.0), -5.0, 5.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        left = k1 * xm
        right = k1 * thr + k2 * (xm - thr)
        out = torch.where(xm <= thr, left, right)
        return torch.clamp(F.softplus(out), 0.0, 100.0)

class ParameterModulator(nn.Module):
    def __init__(self, context_dim, param_configs):
        super().__init__()
        self.param_configs = param_configs
        self.param_predictors = nn.ModuleDict()
        for name, info in param_configs.items():
            dim = info.get('dim', 1)
            self.param_predictors[name] = nn.Sequential(
                nn.Linear(context_dim, 64), nn.ReLU(), nn.Dropout(0.1),
                nn.Linear(64, 32), nn.ReLU(),
                nn.Linear(32, dim), nn.Tanh()
            )
    def forward(self, context):
        global_context = context.mean(dim=1)  # (B, context_dim)
        modulated = {}
        for name, predictor in self.param_predictors.items():
            info = self.param_configs[name]
            raw_off = predictor(global_context)
            scale = info.get('scale', 0.1)
            modulated[name] = raw_off * scale
        return modulated

class LiquidWeightGenerator(nn.Module):
    def __init__(self, h_dim, x_dim, n_ops, hidden_dim=64, tau_min=1.0, tau_max=3.0):
        super().__init__()
        self.n_ops = n_ops
        self.tau_min, self.tau_max = tau_min, tau_max
        self.h_feature_net = nn.Sequential(nn.Linear(h_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(0.1))
        self.x_feature_net = nn.Sequential(nn.Linear(x_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(0.1))
        self.temporal_encoder = nn.Sequential(nn.Linear(3, hidden_dim // 4), nn.ReLU())
        combined_dim = hidden_dim + hidden_dim // 4
        self.feature_fusion = nn.Sequential(
            nn.Linear(combined_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.op_branches = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 4), nn.ReLU(), nn.Linear(hidden_dim // 4, 1))
            for _ in range(n_ops)
        ])
        self.temp_predictor = nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 4), nn.ReLU(), nn.Linear(hidden_dim // 4, 1))
        self.global_bias_scale = 0.01
        for branch in self.op_branches:
            nn.init.xavier_uniform_(branch[0].weight)
            nn.init.xavier_uniform_(branch[2].weight)

    def forward(self, h_multi, x):
        B, T, h_dim = h_multi.shape
        _, _, x_dim = x.shape
        h_features = self.h_feature_net(h_multi)
        x_features = self.x_feature_net(x)
        t_norm = torch.arange(T, device=x.device, dtype=x.dtype).unsqueeze(0).expand(B, -1) / max(T-1, 1)
        dt = torch.zeros_like(t_norm)
        dt[:, 1:] = t_norm[:, 1:] - t_norm[:, :-1]
        phase = torch.sin(2 * 3.14159 * t_norm)
        temporal_input = torch.stack([t_norm, dt, phase], dim=-1)
        temporal_features = self.temporal_encoder(temporal_input)
        combined = torch.cat([h_features, x_features, temporal_features], dim=-1)
        fused = self.feature_fusion(combined)
        raw_logits = []
        for branch in self.op_branches:
            raw_logits.append(branch(fused).squeeze(-1))
        raw_weights = torch.stack(raw_logits, dim=-1)                     # (B, T, K)

        # NaN protection
        raw_weights = torch.nan_to_num(raw_weights, nan=0.0, posinf=1e6, neginf=-1e6)
        raw_weights = raw_weights - raw_weights.mean(dim=-1, keepdim=True)

        if self.global_bias_scale > 0:
            global_bias = torch.randn(self.n_ops, device=raw_weights.device) * self.global_bias_scale
            raw_weights = raw_weights + global_bias.unsqueeze(0).unsqueeze(0)
        raw_temp = self.temp_predictor(fused).squeeze(-1)                 # (B, T)
        raw_temp = torch.nan_to_num(raw_temp, nan=0.0, posinf=1e6, neginf=-1e6)
        temperature = torch.clamp(F.softplus(raw_temp) + self.tau_min, self.tau_min, self.tau_max)
        weights = F.softmax(raw_weights / temperature.unsqueeze(-1), dim=-1)
        return weights, temperature

class CustomKAN(nn.Module):
    def __init__(self, ops, h_dim, x_dim):
        super().__init__()
        self.ops = nn.ModuleList(ops)
        self.n_ops = len(ops)
        self.weight_generator = LiquidWeightGenerator(h_dim=h_dim, x_dim=x_dim, n_ops=self.n_ops, hidden_dim=128)
        self.op_feature_extractors = nn.ModuleList([
            nn.Sequential(nn.Linear(h_dim, h_dim), nn.ReLU(), nn.Linear(h_dim, h_dim))
            for _ in range(self.n_ops)
        ])
        self.param_modulators = nn.ModuleList()
        context_dim = h_dim + x_dim
        for op in self.ops:
            param_configs = self._get_param_configs_for_op(op)
            if param_configs:
                modulator = ParameterModulator(context_dim, param_configs)
                self.param_modulators.append(modulator)
                op.set_param_modulator(modulator)
            else:
                self.param_modulators.append(None)
        self.raw_gain = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))

    def _get_param_configs_for_op(self, op):
        if isinstance(op, MonotonicLinearOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.2}, 'bias_offset': {'dim': 1, 'scale': 0.3}}
        if isinstance(op, MonotonicFlatOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.1}, 'bias_offset': {'dim': 1, 'scale': 0.2}}
        if isinstance(op, ConcaveLogOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.2}, 'eps_offset': {'dim': 1, 'scale': 0.01}}
        if isinstance(op, SaturationSigmoidOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.2}, 'slope_offset': {'dim': 1, 'scale': 0.3}, 'bias_offset': {'dim': 1, 'scale': 0.3}}
        if isinstance(op, HingeReLUOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.2}, 'threshold_offset': {'dim': 1, 'scale': 0.3}}
        if isinstance(op, PolynomialOp):
            return {'coeff_offset': {'dim': op.deg, 'scale': 0.2}}
        if isinstance(op, DampedSinOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.2}, 'freq_offset': {'dim': 1, 'scale': 0.3},
                    'lambda_offset': {'dim': 1, 'scale': 0.2}, 'phase_offset': {'dim': 1, 'scale': 0.5}}
        if isinstance(op, PiecewiseLinearOp):
            return {'k1_offset': {'dim': 1, 'scale': 0.2}, 'k2_offset': {'dim': 1, 'scale': 0.2},
                    'threshold_offset': {'dim': 1, 'scale': 0.3}}
        return {}

    def forward(self, h_multi, x):
        op_features = [ext(h_multi) for ext in self.op_feature_extractors]
        context = torch.cat([h_multi, x], dim=-1)
        outs = [op(op_features[i], context) for i, op in enumerate(self.ops)]
        Tm = min(o.size(1) for o in outs)
        outs = [o[:, :Tm] for o in outs]
        h_multi_aligned = h_multi[:, :Tm, :]
        x_aligned = x[:, :Tm, :]
        op_stack = torch.stack(outs, dim=-1)  # (B, Tm, K)
        weights, temperature = self.weight_generator(h_multi_aligned, x_aligned)
        damage = torch.sum(op_stack * weights, dim=-1)  # (B, Tm)
        damage = torch.clamp(damage, 0.0, 100.0)
        gain = torch.clamp(F.softplus(self.raw_gain), 0.1, 2.0)
        bias_val = torch.clamp(self.bias, -2.0, 2.0)
        damage = torch.clamp(gain * damage + bias_val, 0.0, 100.0)
        return damage, weights, temperature

class TrendEncoder(nn.Module):
    """
    Vectorized monotonic decreasing HI projection.
    """
    def __init__(self, in_ch, trend_ch=4):
        super().__init__()
        self.boltz = BoltzmannKAN(in_ch, out_ch=trend_ch)
        ops = [MonotonicLinearOp(), MonotonicFlatOp(), ConcaveLogOp(),
               SaturationSigmoidOp(), HingeReLUOp(), PolynomialOp(),
               DampedSinOp(), PiecewiseLinearOp()]
        self.customkan = CustomKAN(ops, h_dim=trend_ch, x_dim=in_ch)

        self.proj_gain = nn.Parameter(torch.tensor(1.0))
        self.proj_bias = nn.Parameter(torch.tensor(0.0))

        self.health_aware_transform = nn.Sequential(
            nn.Linear(in_ch, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )
        self.linear_enforcer = nn.Sequential(
            nn.Linear(in_ch, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )

    def forward(self, x):  # x:(B,T,C)
        B, T, C = x.shape
        h_multi = self.boltz(x)                               # (B,T,trend_ch)
        damage, weights, temperature = self.customkan(h_multi, x)

        x_reshaped = x.view(-1, C)
        health_direct = self.health_aware_transform(x_reshaped).view(B, T)  # (B,T)
        linear_weight = self.linear_enforcer(x_reshaped).view(B, T)

        t = torch.arange(T, device=x.device, dtype=x.dtype).unsqueeze(0).expand(B, -1)
        t_normalized = t / (T - 1 + 1e-6)

        g = torch.clamp(F.softplus(self.proj_gain), 0.1, 5.0)
        b = torch.clamp(self.proj_bias, -5.0, 5.0)
        damage_normalized = torch.sigmoid(g*(damage + b))

        combined_health = health_direct * (1 - 0.3 * damage_normalized)
        linear_decay = 1.0 - t_normalized * 0.5
        hi = linear_weight * linear_decay + (1 - linear_weight) * combined_health

        # ---- Vectorized strictly non-increasing enforcement with epsilon step ----
        eps = 1e-3
        idx = torch.arange(T, device=hi.device, dtype=hi.dtype).unsqueeze(0)  # (1, T)
        hi_adj = hi - eps * idx
        hi_mon, _ = torch.cummin(hi_adj, dim=1)
        hi = hi_mon + eps * idx
        # -------------------------------------------------------------------------

        return hi, h_multi, weights, temperature

class ReconHead(nn.Module):
    def __init__(self, C, hidden=128):
        super().__init__()
        self.gru = nn.GRU(input_size=C+1, hidden_size=hidden, batch_first=True)
        self.out = nn.Linear(hidden, C)
    def forward(self, x_in, h_in):
        B,T,C = x_in.shape
        h_in_clamped = torch.clamp(h_in, 0.0, 10.0)
        feat = torch.cat([x_in, h_in_clamped.unsqueeze(-1)], dim=-1)
        H,_ = self.gru(feat)
        y = self.out(H)
        return y

class MaintClassifier(nn.Module):
    def __init__(self, sensor_dim, hidden=64, n_classes=14):
        super().__init__()
        self.hi_mlp = nn.Sequential(
            nn.Linear(6, hidden), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden//2)
        )
        self.sensor_mlp = nn.Sequential(
            nn.Linear(sensor_dim * 2, hidden), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden//2)
        )
        self.final_classifier = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, n_classes)
        )

    def forward(self, h_b, h_a, x_b, x_a, mask):
        m = mask
        T = m.size(1)
        valid = m.sum(1, keepdim=True).clamp_min(1.0)

        db = h_a - h_b
        mean_d = (db*m).sum(1,keepdim=True)/valid
        var_d  = (((db-mean_d)*m)**2).sum(1,keepdim=True)/valid
        std_d  = (var_d + 1e-8).sqrt()
        d0     = (db[:, :1])
        dmax   = (db.masked_fill(m==0, -1e9)).max(dim=1, keepdim=True).values
        pos_ratio = ((db>0).float()*m).sum(1,keepdim=True)/valid

        t = torch.arange(T, device=h_b.device, dtype=h_b.dtype)
        t = (t - t.mean())/(t.std()+1e-6)
        def slope(x):
            num = (x*t).sum(1) - x.sum(1)*t.sum()/T
            den = (t**2).sum() - (t.sum()**2)/T + 1e-8
            return (num/den).unsqueeze(1)
        slope_diff = slope(h_a) - slope(h_b)

        hi_feat = torch.cat([mean_d, d0, dmax, std_d, slope_diff, pos_ratio], dim=1)
        hi_feat = torch.clamp(hi_feat, -10.0, 10.0)
        hi_features = self.hi_mlp(hi_feat)

        x_b_mean = (x_b * m.unsqueeze(-1)).sum(1) / valid
        x_a_mean = (x_a * m.unsqueeze(-1)).sum(1) / valid
        sensor_change = torch.cat([x_b_mean, x_a_mean], dim=1)
        sensor_features = self.sensor_mlp(sensor_change)

        combined = torch.cat([hi_features, sensor_features], dim=1)
        logits = self.final_classifier(combined)
        return logits

def sanitize_tensors(*tensors):
    result = []
    for t in tensors:
        if t is not None:
            t_clean = torch.nan_to_num(t, nan=0.0, posinf=1e6, neginf=-1e6)
            result.append(t_clean)
        else:
            result.append(t)
    return result[0] if len(result) == 1 else result

class DiffAwareReconstructor(nn.Module):
    """
    - Two encoders share parameters: encode x_before / x_after → h_b / h_a (monotonic decreasing HI)
    - Two reconstruction heads: one-step recursive reconstruction
    - Classification head: ΔHI + sensor change features
    """
    def __init__(self, in_ch, trend_ch=4, hidden=128, n_classes=14):
        super().__init__()
        self.encoder = TrendEncoder(in_ch, trend_ch)
        self.recon_b = ReconHead(in_ch, hidden)
        self.recon_a = ReconHead(in_ch, hidden)
        self.clf     = MaintClassifier(sensor_dim=in_ch, hidden=64, n_classes=n_classes)
        self.consistency_weight = nn.Parameter(torch.tensor(1.0))

    def forward(self, x_b, x_a, mask):
        h_b, h_multi_b, weights_b, temp_b = self.encoder(x_b)
        h_a, h_multi_a, weights_a, temp_a = self.encoder(x_a)

        h_b, h_a = sanitize_tensors(h_b, h_a)
        weights_b, weights_a = sanitize_tensors(weights_b, weights_a)

        xb_in, hb_in = x_b[:, :-2], h_b[:, :-2]
        xa_in, ha_in = x_a[:, :-2], h_a[:, :-2]
        yb_hat = self.recon_b(xb_in, hb_in)   # (B, L-2, C) ≈ x_b[:,1:L-1]
        ya_hat = self.recon_a(xa_in, ha_in)   # (B, L-2, C) ≈ x_a[:,1:L-1]

        yb_hat, ya_hat = sanitize_tensors(yb_hat, ya_hat)
        logits = self.clf(h_b, h_a, x_b, x_a, mask)
        logits = sanitize_tensors(logits)
        return yb_hat, ya_hat, h_b, h_a, logits, weights_b, weights_a, temp_b, temp_a

# ---------------------------------------------------------------------
# Losses / Regularizers
# ---------------------------------------------------------------------
def masked_mse(a, b, mask):
    diff = (a - b)**2
    mse  = (diff.mean(-1) * mask).sum() / (mask.sum() + 1e-6)
    return mse

def masked_mse_batchwise(a, b, mask):
    diff = (a - b)**2  # (B, T, C)
    mse_t = diff.mean(-1) * mask  # (B, T)
    denom = mask.sum(1).clamp_min(1.0)  # (B,)
    return mse_t.sum(1) / denom

def slope_loss(h, mask, kind="smooth"):
    if kind=="smooth":
        d2 = h[:,2:] - 2*h[:,1:-1] + h[:,:-2]
        m  = mask[:,2:]
        return (d2.abs() * m).sum() / (m.sum()+1e-6)
    elif kind=="mono_dec":
        dh = h[:,1:] - h[:,:-1]
        m  = mask[:,1:]
        return (F.relu(dh) * m).sum() / (m.sum()+1e-6)
    else:
        return torch.tensor(0., device=h.device)

def enhanced_liquid_weight_regularization(weights_b, weights_a, mask,
                                          lambda_tv=0.05, lambda_ent=0.2,
                                          lambda_bal=0.1, lambda_div=0.1):
    total_loss = torch.tensor(0.0, device=weights_b.device)

    def tv_loss(weights, mask):
        if weights.size(1) <= 1:
            return torch.tensor(0.0, device=weights.device)
        diff = weights[:, 1:, :] - weights[:, :-1, :]
        mask_diff = mask[:, 1:]
        tv = (diff.abs().sum(-1) * mask_diff).sum() / (mask_diff.sum() + 1e-6)
        return tv

    tv_loss_b = tv_loss(weights_b, mask)
    tv_loss_a = tv_loss(weights_a, mask)
    total_loss += lambda_tv * (tv_loss_b + tv_loss_a)

    def entropy_loss(weights, mask):
        eps = 1e-8
        entropy = -(weights * torch.log(weights + eps)).sum(-1)
        target_entropy = np.log(weights.size(-1)) * 0.8
        entropy_penalty = F.relu(target_entropy - entropy)
        return (entropy_penalty * mask).sum() / (mask.sum() + 1e-6)

    ent_loss_b = entropy_loss(weights_b, mask)
    ent_loss_a = entropy_loss(weights_a, mask)
    total_loss += lambda_ent * (ent_loss_b + ent_loss_a)

    def balance_loss(weights, mask):
        valid_weights = weights * mask.unsqueeze(-1)  # (B, T, K)
        mean_usage = valid_weights.sum([0, 1]) / (mask.sum() + 1e-6)  # (K,)
        target_usage = 1.0 / weights.size(-1)
        balance = ((mean_usage - target_usage) ** 2).sum()
        return balance

    bal_loss_b = balance_loss(weights_b, mask)
    bal_loss_a = balance_loss(weights_a, mask)
    total_loss += lambda_bal * (bal_loss_b + bal_loss_a)

    def diversity_loss(weights, mask):
        B, T, K = weights.shape
        if T <= 1:
            return torch.tensor(0.0, device=weights.device)
        valid_weights = weights * mask.unsqueeze(-1)
        weights_flat = valid_weights.view(-1, K)
        weights_centered = weights_flat - weights_flat.mean(0, keepdim=True)
        cov = torch.mm(weights_centered.T, weights_centered) / (weights_flat.size(0) - 1 + 1e-6)
        mask_offdiag = torch.ones_like(cov) - torch.eye(K, device=cov.device)
        corr_penalty = (cov.abs() * mask_offdiag).sum()
        return corr_penalty

    div_loss_b = diversity_loss(weights_b, mask)
    div_loss_a = diversity_loss(weights_a, mask)
    total_loss += lambda_div * (div_loss_b + div_loss_a)
    return total_loss

def smoothness_enhancement_loss(h, mask, order=2):
    if order == 2:
        d2 = h[:,2:] - 2*h[:,1:-1] + h[:,:-2]
        m = mask[:,2:]
        return (d2.abs() * m).sum() / (m.sum() + 1e-6)
    elif order == 3:
        d3 = h[:,3:] - 3*h[:,2:-1] + 3*h[:,1:-2] - h[:,:-3]
        m = mask[:,3:]
        return (d3.abs() * m).sum() / (m.sum() + 1e-6)
    else:
        return torch.tensor(0., device=h.device)

def compute_sample_weights(matching_costs, alpha=0.5):
    return 1.0 / (1.0 + alpha * matching_costs)

def maintenance_effect_loss(h_b, h_a, mask, lambda_diff=1.0, lambda_smooth_after=0.5, lambda_level_constraint=2.0):
    """
    Encourage maintenance effect: after-maintenance HI should be different from before,
    after-maintenance HI should be smoother, and min(h_a) > max(h_b) for proper separation.
    """
    total_loss = torch.tensor(0.0, device=h_b.device)

    # 1. Encourage difference between before and after HI (improved stability)
    hi_diff = torch.abs(h_a - h_b)  # (B, T)
    hi_diff_clamped = torch.clamp(hi_diff, min=1e-3)  # Prevent extreme gradients
    diff_loss = F.softplus(-10.0 * hi_diff_clamped)  # Smooth penalty for small differences
    diff_loss = (diff_loss * mask).sum() / (mask.sum() + 1e-6)
    total_loss += lambda_diff * diff_loss

    # 2. Encourage after-maintenance HI to be smoother (less variation)
    if h_a.size(1) > 1:
        h_a_diff = torch.abs(h_a[:, 1:] - h_a[:, :-1])  # (B, T-1)
        mask_diff = mask[:, 1:]
        smooth_after_loss = (h_a_diff * mask_diff).sum() / (mask_diff.sum() + 1e-6)
        total_loss += lambda_smooth_after * smooth_after_loss

    # 3. Encourage min(h_a) > max(h_b) for proper level separation
    # Use masked operations to handle variable lengths
    h_b_masked = h_b.masked_fill(mask == 0, -1e9)  # Fill invalid positions with very low values
    h_a_masked = h_a.masked_fill(mask == 0, 1e9)   # Fill invalid positions with very high values

    h_b_max = h_b_masked.max(dim=1)[0]  # (B,) - max of each sequence
    h_a_min = h_a_masked.min(dim=1)[0]  # (B,) - min of each sequence

    # Encourage h_a_min > h_b_max with a margin
    margin = 0.1
    level_constraint = F.relu(h_b_max - h_a_min + margin)  # (B,)
    level_constraint_loss = level_constraint.mean()
    total_loss += lambda_level_constraint * level_constraint_loss

    return total_loss

# ---------------------------------------------------------------------
# Evaluation / Prediction
# ---------------------------------------------------------------------
@torch.no_grad()
def eval_epoch(model, loader, device, K, compute_acc=True):
    model.eval()
    total = {"mse_b":0.0, "mse_a":0.0, "acc":0.0, "delta_mean":0.0}
    n_batch=0
    n_acc = 0; n_tot=0
    for batch in loader:
        xb, xa, labels, mask, lengths, sample_ids, matching_costs = adapt_batch(batch, device)
        m_tgt  = (torch.arange(xb.size(1)-2, device=device)[None,:] < (lengths-2)[:,None]).float()
        yb_hat, ya_hat, h_b, h_a, logits, _, _, _, _ = model(xb, xa, mask)
        yb_hat, ya_hat, h_b, h_a, logits = sanitize_tensors(yb_hat, ya_hat, h_b, h_a, logits)

        # Ensure logits dimension matches expected K
        if logits.shape[-1] != K:
            print(f"Warning: Logits dim {logits.shape[-1]} != expected K={K}, adjusting...")
            if logits.shape[-1] < K:
                # Pad with zeros
                padding = torch.zeros(logits.shape[0], K - logits.shape[-1], device=logits.device)
                logits = torch.cat([logits, padding], dim=1)
            else:
                # Truncate
                logits = logits[:, :K]

        yb_tgt = xb[:,1:-1,:]
        ya_tgt = xa[:,1:-1,:]
        mse_b = masked_mse(yb_hat, yb_tgt, m_tgt)
        mse_a = masked_mse(ya_hat, ya_tgt, m_tgt)
        if compute_acc and labels is not None:
            # Filter out ignore_index labels
            valid_labels = labels != -100
            if valid_labels.any():
                pred = logits[valid_labels].argmax(dim=1)
                labels_valid = labels[valid_labels]
                n_acc += (pred == labels_valid).sum().item()
                n_tot += labels_valid.numel()
        valid = mask.sum(1, keepdim=True).clamp_min(1.0)
        delta_mean = ((h_a - h_b)*mask).sum(1,keepdim=True)/valid
        delta_mean = torch.nan_to_num(delta_mean, nan=0.0)
        total["delta_mean"] += delta_mean.mean().item()
        total["mse_b"] += mse_b.item()
        total["mse_a"] += mse_a.item()
        n_batch += 1
    for k in ["mse_b","mse_a","delta_mean"]:
        total[k] /= max(n_batch,1)
    total["acc"] = (n_acc / max(n_tot,1)) if n_tot>0 else 0.0
    return total

@torch.no_grad()
def collect_test_predictions(model, loader, device, max_curve_keep=24, K=14, id_to_name=None):
    model.eval()
    y_true, y_pred = [], []
    all_delta_mean, all_sample_ids = [], []
    keep_curves = []
    for batch in loader:
        xb, xa, labels, mask, lengths, sample_ids, matching_costs = adapt_batch(batch, device)
        yb_hat, ya_hat, h_b, h_a, logits, weights_b, weights_a, temp_b, temp_a = model(xb, xa, mask)
        yb_hat, ya_hat, h_b, h_a, logits = sanitize_tensors(yb_hat, ya_hat, h_b, h_a, logits)

        # Ensure logits dimension matches expected K
        if logits.shape[-1] != K:
            if logits.shape[-1] < K:
                padding = torch.zeros(logits.shape[0], K - logits.shape[-1], device=logits.device)
                logits = torch.cat([logits, padding], dim=1)
            else:
                logits = logits[:, :K]

        prob = F.softmax(logits, dim=1)     # (B,K)
        pred = prob.argmax(1)               # (B,)

        valid = mask.sum(1, keepdim=True).clamp_min(1.0)
        delta_mean = ((h_a - h_b) * mask).sum(1, keepdim=True) / valid  # (B,1)
        delta_mean = torch.nan_to_num(delta_mean, nan=0.0)

        if labels is not None:
            # Filter out ignore_index labels
            valid_labels = labels != -100
            if valid_labels.any():
                y_true.append(labels[valid_labels].detach().cpu().numpy())
                y_pred.append(pred[valid_labels].detach().cpu().numpy())
        all_delta_mean.append(delta_mean.squeeze(1).cpu().numpy())
        all_sample_ids.extend(sample_ids)

        for i in range(xb.size(0)):
            if len(keep_curves) >= max_curve_keep: break
            L_i = int(lengths[i].item())
            L_recon = min(L_i - 2, yb_hat.size(1))
            keep_curves.append({
                "sample_id": sample_ids[i],
                "true": int(labels[i].cpu().item()) if labels is not None and labels[i] != -100 else -1,
                "pred": int(pred[i].cpu().item()),
                "prob": prob[i].cpu().numpy(),
                "h_before": h_b[i, :L_i].cpu().numpy(),
                "h_after":  h_a[i, :L_i].cpu().numpy(),
                "weights_before": weights_b[i, :L_i].cpu().numpy() if weights_b is not None else None,
                "weights_after": weights_a[i, :L_i].cpu().numpy() if weights_a is not None else None,
                "temp_before": temp_b[i, :L_i].cpu().numpy() if temp_b is not None else None,
                "temp_after": temp_a[i, :L_i].cpu().numpy() if temp_a is not None else None,
                "op_outputs_before": [],
                "op_outputs_after":  [],
                "yb_hat":   yb_hat[i, :L_recon].cpu().numpy(),
                "ya_hat":   ya_hat[i, :L_recon].cpu().numpy(),
                "x_before": xb[i, :L_i].cpu().numpy(),
                "x_after":  xa[i, :L_i].cpu().numpy(),
            })
    y_true = np.concatenate(y_true, axis=0) if len(y_true)>0 else np.array([])
    y_pred = np.concatenate(y_pred, axis=0) if len(y_pred)>0 else np.array([])
    all_delta_mean = np.concatenate(all_delta_mean, axis=0) if len(all_delta_mean)>0 else np.array([])
    return y_true, y_pred, all_delta_mean, all_sample_ids, keep_curves

# ---------------------------------------------------------------------
# Plotting
# ---------------------------------------------------------------------
def remove_constant_segments(hi_values, threshold=1e-6):
    if len(hi_values) <= 1:
        return hi_values, np.arange(len(hi_values))
    diffs = np.abs(np.diff(hi_values))
    valid_mask = np.ones(len(hi_values), dtype=bool)
    valid_mask[1:] = diffs > threshold
    if np.sum(valid_mask) < 2:
        valid_mask[0] = True
        valid_mask[-1] = True
    valid_indices = np.where(valid_mask)[0]
    return hi_values[valid_indices], valid_indices

def plot_hi_examples_aligned(curves, id_to_name, n_show=6, seed=0):
    if len(curves)==0:
        print("(No visualization samples)")
        return
    random.Random(seed).shuffle(curves)
    n_show = min(n_show, len(curves))
    cols = 3
    rows = int(np.ceil(n_show/cols))
    plt.figure(figsize=(cols*4.6, rows*3.2))
    for k in range(n_show):
        ex = curves[k]
        hb = ex["h_before"]; ha = ex["h_after"]
        L  = min(len(hb), len(ha))
        hb, ha = hb[:L], ha[:L]
        hb_clean, hb_indices = remove_constant_segments(hb)
        ha_clean, ha_indices = remove_constant_segments(ha)
        t_b = hb_indices
        t_a = ha_indices + L

        sample_id = ex["sample_id"]
        pred = ex["pred"]; true = ex["true"]; prob = ex["prob"]
        true_name = id_to_name.get(true, f"Class_{true}")
        pred_name = id_to_name.get(pred, f"Class_{pred}")

        plt.subplot(rows, cols, k+1)
        if len(hb_clean) > 1:
            plt.plot(t_b, hb_clean, label="Learned HI (Pre)", linewidth=1.8, marker='o', markersize=3)
        if len(ha_clean) > 1:
            plt.plot(t_a, ha_clean, label="Learned HI (Post)",  linewidth=1.8, linestyle='--', marker='s', markersize=3)
        plt.axvline(L-1, color='k', linestyle=':', linewidth=1.0, alpha=0.7)
        d_mean = float(np.mean(ha - hb))
        d_max  = float(np.max(ha - hb))
        plt.title(f"ID={sample_id}\nTrue={true_name} | Pred={pred_name} "
                  f"| p={prob[pred]:.2f}\nΔHI_mean={d_mean:.3f}, ΔHI_max={d_max:.3f}", fontsize=9)
        plt.xlabel("Cycle (Pre | Post)")
        plt.ylabel("Health Index (higher=better)")
        plt.grid(ls='--', alpha=.35)
        if k==0: plt.legend()
    plt.tight_layout(); plt.show()

def plot_enhanced_liquid_weights(curves, n_show=3, seed=0):
    if len(curves) == 0:
        print("(No visualization samples)")
        return
    curves_with_weights = [c for c in curves if c.get("weights_before") is not None]
    if len(curves_with_weights) == 0:
        print("(No weight information available)")
        return
    random.Random(seed).shuffle(curves_with_weights)
    n_show = min(n_show, len(curves_with_weights))
    op_names = ["MonotonicLinear", "MonotonicFlat", "ConcaveLog", "SaturationSigmoid",
                "HingeReLU", "Polynomial", "DampedSin", "PiecewiseLinear"]
    for i in range(n_show):
        ex = curves_with_weights[i]
        weights_b = ex["weights_before"]
        weights_a = ex["weights_after"]
        temp_b = ex.get("temp_before")
        temp_a = ex.get("temp_after")
        sample_id = ex["sample_id"]
        true_label = ex["true"]
        pred_label = ex["pred"]
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle(f"Dynamic Parameter Modulation - ID={sample_id}, True=Class_{true_label}, Pred=Class_{pred_label}", fontsize=14)
        ax = axes[0, 0]
        T_b, K = weights_b.shape
        for k in range(K):
            ax.plot(range(T_b), weights_b[:, k], label=op_names[k] if k < len(op_names) else f"Op_{k}",
                   marker='o', markersize=2, linewidth=1.5)
        ax.set_title("Pre-maintenance Weights"); ax.set_xlabel("Time Step"); ax.set_ylabel("Weight")
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left'); ax.grid(True, alpha=0.3)
        ax = axes[0, 1]
        T_a, _ = weights_a.shape
        for k in range(K):
            ax.plot(range(T_a), weights_a[:, k], label=op_names[k] if k < len(op_names) else f"Op_{k}",
                   marker='s', markersize=2, linewidth=1.5, linestyle='--')
        ax.set_title("Post-maintenance Weights"); ax.set_xlabel("Time Step"); ax.set_ylabel("Weight")
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left'); ax.grid(True, alpha=0.3)
        ax = axes[0, 2]
        entropy_b = -np.sum(weights_b * np.log(weights_b + 1e-8), axis=1)
        entropy_a = -np.sum(weights_a * np.log(weights_a + 1e-8), axis=1)
        ax.plot(range(T_b), entropy_b, label="Pre", marker='o', linewidth=2)
        ax.plot(range(T_a), entropy_a, label="Post", marker='s', linewidth=2, linestyle='--')
        ax.axhline(np.log(K), color='r', linestyle=':', label=f'Max Entropy ({np.log(K):.2f})')
        ax.set_title("Weight Entropy Over Time"); ax.set_xlabel("Time Step"); ax.set_ylabel("Entropy")
        ax.legend(); ax.grid(True, alpha=0.3)
        ax = axes[1, 0]
        if temp_b is not None and temp_a is not None:
            ax.plot(range(T_b), temp_b, label="Pre", marker='o', linewidth=2)
            ax.plot(range(T_a), temp_a, label="Post", marker='s', linewidth=2, linestyle='--')
            ax.set_title("Temperature Evolution"); ax.set_xlabel("Time Step"); ax.set_ylabel("Temperature"); ax.legend()
        else:
            ax.text(0.5, 0.5, "Temperature data\nnot available", ha='center', va='center', transform=ax.transAxes)
            ax.set_title("Temperature Evolution")
        ax.grid(True, alpha=0.3)
        ax = axes[1, 1]
        ax.text(0.5, 0.5, "Operator output variance\n(not collected)", ha='center', va='center', transform=ax.transAxes)
        ax.set_title("Operator Output Variance"); ax.grid(True, alpha=0.3)
        ax = axes[1, 2]
        combined_weights = np.concatenate([weights_b, weights_a], axis=0)
        # Apply smoothing for correlation stability
        from scipy.ndimage import uniform_filter1d
        smoothed_weights = uniform_filter1d(combined_weights, size=3, axis=0)
        corr_matrix = np.corrcoef(smoothed_weights.T)
        im = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
        ax.set_title("Operator Weight Correlation (Smoothed)")
        ax.set_xticks(range(K)); ax.set_yticks(range(K))
        short_names = [op_names[k][:8] if k < len(op_names) else f"Op_{k}" for k in range(K)]
        ax.set_xticklabels(short_names, rotation=45); ax.set_yticklabels(short_names)
        cbar = plt.colorbar(im, ax=ax); cbar.set_label('Correlation')
        plt.tight_layout(); plt.show()

def plot_liquid_weights(curves, n_show=3, seed=0):
    plot_enhanced_liquid_weights(curves, n_show, seed)

def plot_sensor_examples_aligned(curves, sensor_idx_list=None, n_cols=4):
    if len(curves)==0:
        print("(No visualization samples)")
        return
    ex = curves[0]
    xb = ex["x_before"]; xa = ex["x_after"]; ya = ex["ya_hat"]
    L, C = xb.shape
    Lhat = ya.shape[0]
    if sensor_idx_list is None:
        step = max(1, C//8); sensor_idx_list = list(range(0, C, step))[:8]
    n = len(sensor_idx_list)
    n_rows = int(np.ceil(n/n_cols))
    plt.figure(figsize=(n_cols*4.3, n_rows*3.1))
    for i, s in enumerate(sensor_idx_list):
        plt.subplot(n_rows, n_cols, i+1)
        t_b = np.arange(L); t_a = np.arange(L, 2*L)
        plt.plot(t_b, xb[:,s], label="Original (pre)", linewidth=1.2)
        plt.plot(t_a, xa[:,s], label="Original (post)",  linewidth=1.0, linestyle="--", alpha=0.7)
        t_pred = np.arange(L+1, L+1+Lhat)
        plt.plot(t_pred, ya[:,s], label="Post prediction", linewidth=1.6)
        plt.axvline(L-1, color='k', linestyle=':', linewidth='1.0')
        plt.title(f"Sensor_{s:02d}")
        if i==0: plt.legend()
        plt.grid(ls="--", alpha=.35)
    plt.tight_layout(); plt.show()

def topk_by_delta(df_delta, id_to_name, k=3):
    if len(df_delta)==0:
        return
    unique_classes = sorted(df_delta["true"].unique())
    for cls in unique_classes[:5]:
        sub = df_delta[df_delta["true"]==cls].sort_values("delta_hi_mean", ascending=False).head(k)
        class_name = id_to_name.get(cls, f"Class_{cls}")
        print(f"\n[True={class_name}] Top {k} samples with highest ΔHI_mean:")
        print(sub[["uid","delta_hi_mean","pred"]].reset_index(drop=True))

# ---------------------------------------------------------------------
# Training (separate) and Testing (separate)
# ---------------------------------------------------------------------
def build_model_and_setup(train_loader, device, class_weights=None):
    """
    Build model and setup training components.

    Args:
        train_loader: Training data loader
        device: Device to use
        class_weights: Pre-computed class weights tensor (optional)
    """
    # Determine K from training loader
    K = scan_num_classes_from_loader(train_loader)
    id_to_name = build_default_id_to_name(K)

    first_batch = next(iter(train_loader))
    C = first_batch["x_before"].shape[-1]

    print(f"\n[Setup] Sensor dim C={C}, Num classes K={K}")

    # Use provided class weights or create uniform weights
    if class_weights is not None:
        if isinstance(class_weights, np.ndarray):
            class_weights = torch.from_numpy(class_weights).float().to(device)
        elif isinstance(class_weights, torch.Tensor):
            class_weights = class_weights.float().to(device)

        # Ensure class_weights has the right size
        if len(class_weights) < K:
            # Pad with ones if needed
            padded_weights = torch.ones(K, dtype=torch.float32, device=device)
            padded_weights[:len(class_weights)] = class_weights
            class_weights = padded_weights
        elif len(class_weights) > K:
            # Truncate if needed
            class_weights = class_weights[:K]

        print(f"[Class Weights] Using provided weights: {class_weights.detach().cpu().numpy()}")
    else:
        class_weights = torch.ones(K, dtype=torch.float32, device=device)
        print(f"[Class Weights] Using uniform weights: {class_weights.detach().cpu().numpy()}")

    model = DiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=K).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    ce  = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100, reduction='none')

    return model, opt, ce, K, id_to_name, C


def train_only(train_loader, val_loader, epochs=300, lr=1e-3, device=None, patience=20,
               use_matching_cost=True, save_path="best_model.pth", class_weights=None,
               es_alpha=1.0):
    """
    Train and save the best model. Returns path to checkpoint and metadata.

    Args:
        class_weights: Pre-computed class weights (numpy array or torch tensor)
        es_alpha: Weight for accuracy in early stopping metric
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Build model and training setup
    model, opt, ce, K, id_to_name, C = build_model_and_setup(train_loader, device, class_weights)
    # Override LR if provided
    for g in opt.param_groups:
        g['lr'] = lr

    best_v = float("inf")
    best_state = None
    best_epoch = 0
    no_improve = 0
    nan_batches = 0

    print(f"\nStarting training for {epochs} epochs with early stopping (patience={patience})...")
    print(f"Device: {device}")
    print(f"Early stopping alpha: {es_alpha}")

    for ep in range(1, epochs+1):
        print(f"\n[Epoch {ep}/{epochs}] Training...")
        model.train()
        logs = {"mse_b":0.0, "mse_a":0.0, "smooth":0.0, "mono":0.0,
                "cls":0.0, "smooth_enh":0.0, "liquid_weight":0.0, "maintenance_effect":0.0, "all":0.0}
        n_bt = 0
        batch_nan_count = 0

        progress = ep / epochs
        lambda_tv  = 0.03 * (1 - progress * 0.3)
        lambda_ent = 0.3  * (1 - progress * 0.7)
        lambda_bal = 0.15 * (1 - progress * 0.5)
        lambda_div = 0.2  * (1 - progress * 0.6)

        # Gradient norm tracking
        grad_norms = []

        for batch_idx, batch in enumerate(train_loader):
            xb, xa, labels, mask, lengths, sample_ids, matching_costs = adapt_batch(batch, device)
            m_tgt  = (torch.arange(xb.size(1)-2, device=device)[None,:] < (lengths-2)[:,None]).float()
            yb_hat, ya_hat, h_b, h_a, logits, weights_b, weights_a, temp_b, temp_a = model(xb, xa, mask)
            yb_hat, ya_hat, h_b, h_a, logits = sanitize_tensors(yb_hat, ya_hat, h_b, h_a, logits)
            weights_b, weights_a = sanitize_tensors(weights_b, weights_a)

            # Reconstruction targets
            yb_tgt = xb[:,1:-1,:]
            ya_tgt = xa[:,1:-1,:]

            # Per-sample reconstruction losses (for optional cost-weighting)
            rec_b_per = masked_mse_batchwise(yb_hat, yb_tgt, m_tgt)
            rec_a_per = masked_mse_batchwise(ya_hat, ya_tgt, m_tgt)
            rec_per = rec_b_per + rec_a_per

            # Classification loss (per-sample)
            if labels is None:
                cls_per = torch.zeros(xb.size(0), device=device)
            else:
                cls_per = ce(logits, labels)

            # Optional matching-cost weighting
            if use_matching_cost and matching_costs is not None:
                w_i = compute_sample_weights(matching_costs)
            else:
                w_i = torch.ones(xb.size(0), device=device)

            loss_rec = (rec_per * w_i).mean()
            loss_cls = (cls_per * w_i).mean()

            # HI regularizations
            loss_smooth = slope_loss(h_a, mask, "smooth") + slope_loss(h_b, mask, "smooth")
            loss_mono   = slope_loss(h_a, mask, "mono_dec") + slope_loss(h_b, mask, "mono_dec")
            loss_smooth_enhanced = (smoothness_enhancement_loss(h_b, mask, order=2) +
                                   smoothness_enhancement_loss(h_a, mask, order=2) +
                                   smoothness_enhancement_loss(h_b, mask, order=3) * 0.3 +
                                   smoothness_enhancement_loss(h_a, mask, order=3) * 0.3)
            loss_liquid_weight = enhanced_liquid_weight_regularization(
                weights_b, weights_a, mask,
                lambda_tv=lambda_tv, lambda_ent=lambda_ent,
                lambda_bal=lambda_bal, lambda_div=lambda_div
            )

            # Maintenance effect loss to encourage proper HI behavior
            loss_maintenance_effect = maintenance_effect_loss(
                h_b, h_a, mask,
                lambda_diff=1.0,           # Encourage difference between before/after
                lambda_smooth_after=0.5,   # Encourage after-maintenance HI to be smoother
                lambda_level_constraint=2.0 # Encourage min(h_a) > max(h_b)
            )

            # Total loss
            w_rec=1.0; w_smooth=0.03; w_mono=0.05; w_cls=1.0; w_smooth_enh=0.3; w_liquid=0.5; w_maintenance=1.5
            loss = (w_rec*loss_rec + w_smooth*loss_smooth + w_mono*loss_mono +
                    w_cls*loss_cls + w_smooth_enh*loss_smooth_enhanced + w_liquid*loss_liquid_weight +
                    w_maintenance*loss_maintenance_effect)

            loss_rec, loss_cls, loss_smooth, loss_mono, loss_smooth_enhanced, loss_liquid_weight, loss_maintenance_effect, loss = sanitize_tensors(
                loss_rec, loss_cls, loss_smooth, loss_mono, loss_smooth_enhanced, loss_liquid_weight, loss_maintenance_effect, loss
            )

            if torch.isnan(loss) or torch.isinf(loss):
                batch_nan_count += 1
                if batch_nan_count == 1:
                    print(f"    [Warning] Epoch {ep}: Found NaN/Inf loss, skipping abnormal batch")
                continue

            opt.zero_grad()
            loss.backward()

            # Adaptive gradient clipping
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            grad_norms.append(grad_norm.item())

            opt.step()

            logs["mse_b"] += rec_b_per.mean().item()
            logs["mse_a"] += rec_a_per.mean().item()
            logs["smooth"]+= loss_smooth.item()
            logs["mono"]  += loss_mono.item()
            logs["cls"]   += loss_cls.item()
            logs["smooth_enh"]+= loss_smooth_enhanced.item()
            logs["liquid_weight"]+= loss_liquid_weight.item()
            logs["maintenance_effect"]+= loss_maintenance_effect.item()
            logs["all"]   += loss.item()
            n_bt += 1

        if batch_nan_count > 0:
            nan_batches += batch_nan_count

        for k in logs: logs[k] /= max(n_bt,1)

        print("    Validating...")
        vl = eval_epoch(model, val_loader, device, K, compute_acc=True)
        # Improved early-stopping metric with configurable weight
        vl_total = (vl['mse_b'] + vl['mse_a']) + es_alpha * (1.0 - vl['acc'])

        # Log gradient statistics
        avg_grad_norm = np.mean(grad_norms) if grad_norms else 0.0
        max_grad_norm = np.max(grad_norms) if grad_norms else 0.0

        print(f"[Epoch {ep:03d}] "
              f"Train: L={logs['all']:.4f} rec_b={logs['mse_b']:.4f} rec_a={logs['mse_a']:.4f} "
              f"smooth={logs['smooth']:.4f} mono={logs['mono']:.4f} liquid_w={logs['liquid_weight']:.4f} "
              f"maint_eff={logs['maintenance_effect']:.4f} cls={logs['cls']:.4f} | "
              f"Val: mse_b={vl['mse_b']:.4f} mse_a={vl['mse_a']:.4f} acc={vl['acc']:.3f} ΔHI_impr={vl['delta_mean']:.3f} "
              f"(ES metric={vl_total:.4f}) | Grad: avg={avg_grad_norm:.3f} max={max_grad_norm:.3f}")

        if vl_total < best_v:
            best_v = vl_total
            best_epoch = ep
            best_state = {k: v.clone() if hasattr(v, 'clone') else v for k, v in model.state_dict().items()}
            no_improve = 0
            print(f"    ✓ Saved new best model (ES metric: {best_v:.4f})")
        else:
            no_improve += 1
            print(f"    No improvement for {no_improve}/{patience} epochs")
            if no_improve >= patience:
                print(f"\n[Early Stopping] Patience reached at epoch {ep}.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"\n[Best Checkpoint] ES metric: {best_v:.4f} (Epoch {best_epoch})")
        torch.save(model.state_dict(), save_path)
        print(f"[Model Saved] {save_path}")

        # Save metadata alongside model
        meta_path = save_path.replace('.pth', '_meta.json')
        meta_data = {
            "K": K,
            "id_to_name": id_to_name,
            "sensor_dim": C,
            "best_es": best_v,
            "best_epoch": best_epoch,
            "class_weights": class_weights.cpu().numpy().tolist() if class_weights is not None else None
        }
        import json
        with open(meta_path, 'w') as f:
            json.dump(meta_data, f, indent=2)
        print(f"[Metadata Saved] {meta_path}")

    if nan_batches > 0:
        print(f"[Warning] Total skipped {nan_batches} batches containing NaN/Inf during training.")

    meta = {"K": K, "id_to_name": id_to_name, "sensor_dim": C, "best_es": best_v, "best_epoch": best_epoch}
    return save_path, meta


def test_only(test_loader, ckpt_path, K=None, id_to_name=None, device=None,
              visualize=True, max_curve_keep=24):
    """
    Load a saved checkpoint and evaluate on the official test set.
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Peek one batch to get sensor dimension C (and fallback K if needed)
    first_batch = next(iter(test_loader))
    C = first_batch["x_before"].shape[-1]

    if K is None:
        # If K not provided, try to infer from labels present in test loader
        # (May undercount if some classes are absent. Prefer passing K from training.)
        print("[Info] K not provided; attempting to infer from test loader labels (may be lower than full K).")
        K = scan_num_classes_from_loader(test_loader)

    if id_to_name is None:
        id_to_name = build_default_id_to_name(K)

    # Build model and load weights
    model = DiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=K).to(device)
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state, strict=False)
    model.eval()

    print("\n" + "="*60)
    print("Final Test Set Evaluation")
    print("="*60)

    te = eval_epoch(model, test_loader, device, K, compute_acc=True)
    print(f"[Test] mse_b={te['mse_b']:.4f} mse_a={te['mse_a']:.4f} acc={te['acc']:.3f} ΔHI_impr={te['delta_mean']:.3f}")

    # Predictions & visualization data
    y_true, y_pred, all_delta_mean, all_uids, keep_curves = collect_test_predictions(
        model, test_loader, device, max_curve_keep=max_curve_keep, K=K, id_to_name=id_to_name
    )

    # Reports (only if labels present)
    if y_true.size > 0:
        print("\n[Classification Report]")
        labels_full = list(range(K))
        target_names = [id_to_name.get(i, f"Class_{i}") for i in labels_full]
        # --- FIX: force labels + names to avoid mismatch when some classes are absent in predictions ---
        print(classification_report(
            y_true, y_pred,
            labels=labels_full,
            target_names=target_names,
            zero_division=0
        ))

        print("\n[Confusion Matrix]")
        print(confusion_matrix(y_true, y_pred, labels=labels_full))

        df_delta = pd.DataFrame({
            "uid": all_uids[:len(y_true)],
            "true": y_true,
            "pred": y_pred,
            "delta_hi_mean": all_delta_mean[:len(y_true)]
        })
        print("\n[Learned ΔHI Statistics] By true class")
        print(df_delta.groupby("true")["delta_hi_mean"].describe())
        topk_by_delta(df_delta, id_to_name, k=3)

        if visualize:
            print("\n[Visualization] Learned health index curves with liquid weight adaptation...")
            plot_hi_examples_aligned(keep_curves, id_to_name, n_show=6, seed=0)

            print("\n[Visualization] Liquid operator weights evolution...")
            plot_liquid_weights(keep_curves, n_show=3, seed=0)

            print("\n[Visualization] Sensor reconstruction predictions...")
            plot_sensor_examples_aligned(keep_curves, sensor_idx_list=None, n_cols=4)
    else:
        print("\n[Info] No labels in test loader; skipping classification report and confusion matrix.")

    return {"test_metrics": te, "K": K, "id_to_name": id_to_name, "curves": keep_curves}


# ---------------------------------------------------------------------
# Example usage (replace with your real loaders and class weights)
# ---------------------------------------------------------------------
# Assuming you have computed class_weights from your data loader script:
# class_weights = your_computed_class_weights_tensor  # shape: (K,)

ckpt_path, meta = train_only(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=300,
    lr=1e-3,
    device=None,     # auto
    patience=20,
    use_matching_cost=True,
    save_path="best_model.pth",
    class_weights=torch.as_tensor(class_weights, dtype=torch.float32)   # Pass your computed class weights here
)


In [ ]:
import shutil

# Source model path
src_path = "/content/best_model.pth"

# Destination path
dst_path = str(resolve_data_file("best_model_NGFAID_250_100.pth"))

# Copy the file
shutil.copy(src_path, dst_path)

print(f"✅ Model saved to: {dst_path}")


In [ ]:
# ================== Post-Training Loader & Eval (no retraining) ==================
# Purpose: once best_model.pth is available, skip retraining, restore the model, rebuild metadata, and run evaluation directly.
# Dependencies: existing functions/classes such as scan_num_classes_from_loader, build_default_id_to_name,
#       DiffAwareReconstructor, test_only, adapt_batch (used indirectly inside the loaders), etc.

import os, json, torch

# -------- Paths and device --------
ckpt_path = "best_model.pth"             # Your trained checkpoint weights
meta_path = "best_model_meta.json"       # Metadata filename (kept distinct from the original _meta.json to avoid conflicts)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

assert os.path.exists(ckpt_path), f"Checkpoint file not found: {ckpt_path}"

# -------- Infer sensor dimension C and number of classes K from the existing loaders --------
# Prefer train_loader when inferring K; fall back to test_loader if needed.
def _infer_C_and_K(train_loader=None, test_loader=None):
    # Inspect one batch to obtain C
    probe_loader = train_loader if (train_loader is not None) else test_loader
    assert probe_loader is not None, "An available train_loader or test_loader is required to infer the dimensions."
    first_batch = next(iter(probe_loader))
    C = first_batch["x_before"].shape[-1]

    K = None
    if train_loader is not None:
        try:
            K = scan_num_classes_from_loader(train_loader)
        except Exception:
            K = None
    if K is None and test_loader is not None:
        try:
            K = scan_num_classes_from_loader(test_loader)
        except Exception:
            K = None
    assert K is not None and K > 0, "Failed to infer K (number of classes) from the data. Please check the loader labels y."
    return C, K

# train_loader / test_loader should already be constructed outside this snippet
# If train_loader is unavailable, replace it with None below and infer using only test_loader.
C, K = _infer_C_and_K(train_loader=train_loader, test_loader=test_loader)
id_to_name = build_default_id_to_name(K)

print(f"[Info] Inferred sensor dim C={C}, num classes K={K}")

# -------- Backfill / rebuild metadata (optional) --------
# If you want to store the training-time class_weights in the metadata as well, provide them here (tensor / numpy / list are all acceptable).
# Demonstration: write class_weights_obj if it is available; otherwise keep None.
class_weights_obj = None
# For example, if you still have the class_weights array, you can use:
# import numpy as np
# class_weights_obj = np.asarray(class_weights, dtype=float)

def _serialize_class_weights(obj):
    if obj is None:
        return None
    try:
        import numpy as np
        if isinstance(obj, torch.Tensor):
            return obj.detach().cpu().numpy().tolist()
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, (list, tuple)):
            return list(obj)
        else:
            return None
    except Exception:
        return None

meta = {
    "K": int(K),
    "id_to_name": {int(k): str(v) for k, v in id_to_name.items()},
    "sensor_dim": int(C),
    "best_es": None,             # Leave blank or fill manually if the early-stopping metric from training is unknown
    "best_epoch": None,          # Same as above
    "class_weights": _serialize_class_weights(class_weights_obj)
}

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print(f"[Meta Saved] {meta_path}")

# -------- Load checkpoint weights and evaluate directly (no retraining required) --------
# Option A: directly reuse test_only from your code base (recommended)
results = test_only(
    test_loader=test_loader,
    ckpt_path=ckpt_path,
    K=K,
    id_to_name=id_to_name,
    device=device,
    visualize=True,          # Set to True to generate plots; use False in headless environments
    max_curve_keep=24
)

# If you prefer not to use test_only, you can also restore the model manually (Option B):
# model = DiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=K).to(device)
# state = torch.load(ckpt_path, map_location=device)
# model.load_state_dict(state, strict=False)
# model.eval()
# Then write your own evaluation loop or reuse helpers such as eval_epoch / collect_test_predictions.


## Part 3B: Post-Training Evaluation and Metadata Recovery


In [ ]:


# ============================================================
# Monotonic-Decreasing HI + aligned plotting (before -> after)
# Adapted to external train/val/test loaders that yield:
#   batch = {
#       "x_before": (B, L, C), "x_after": (B, L, C),
#       "y": LongTensor[B] or None,
#       "meta": list[dict] (each may contain 'pair_id', 'matching_cost', ...)
#   }
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import os
import random
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix

# ---------------------------------------------------------------------
# Utilities for the NEW loaders
# ---------------------------------------------------------------------
def adapt_batch(batch, device):
    """
    Adapt a batch from your loaders to the tensors expected by the model.
    Ensures fixed-length mask/lengths and extracts optional meta.
    """
    xb = batch["x_before"].to(device)  # (B, L, C)
    xa = batch["x_after"].to(device)   # (B, L, C)

    B, L, _ = xb.shape
    mask = torch.ones(B, L, dtype=xb.dtype, device=device)
    lengths = torch.full((B,), L, dtype=torch.long, device=device)

    labels = batch.get("y", None)
    if labels is not None and isinstance(labels, torch.Tensor):
        # Filter out ignore_index labels
        valid_mask = labels != -100
        if valid_mask.any():
            labels = labels.to(device)
        else:
            labels = None
    elif labels is not None:
        labels = None

    meta = batch.get("meta", None)
    sample_ids = []
    matching_costs = torch.zeros(B, dtype=torch.float32, device=device)
    if isinstance(meta, (list, tuple)) and len(meta) == B:
        for i, m in enumerate(meta):
            if isinstance(m, dict):
                sample_ids.append(str(m.get("pair_id", f"sample_{i}")))
                if "matching_cost" in m and m["matching_cost"] is not None:
                    matching_costs[i] = float(m["matching_cost"])
            else:
                sample_ids.append(f"sample_{i}")
    else:
        sample_ids = [f"sample_{i}" for i in range(B)]

    return xb, xa, labels, mask, lengths, sample_ids, matching_costs


def scan_num_classes_from_loader(train_loader):
    """
    Determine K (number of classes) from the training loader.
    """
    max_class = -1
    for batch in train_loader:
        y = batch.get("y", None)
        if y is None:
            continue
        if isinstance(y, torch.Tensor):
            y_np = y.detach().cpu().numpy()
            valid_labels = y_np[y_np >= 0]  # filter out negative labels
            if len(valid_labels) > 0:
                max_class = max(max_class, int(valid_labels.max()))

    if max_class < 0:
        raise ValueError("No valid labels found in the training loader to determine K.")

    K = max_class + 1
    return K


def build_default_id_to_name(K):
    return {i: f"Class_{i}" for i in range(K)}


# ---------------------------------------------------------------------
# Core model components
# ---------------------------------------------------------------------
class BoltzmannKAN(nn.Module):
    def __init__(self, in_ch, out_ch=4):
        super().__init__()
        self.E  = nn.Parameter(torch.zeros(out_ch, in_ch))
        self.kT = nn.Parameter(torch.ones(out_ch, in_ch))
    def forward(self, x):
        B,T,C = x.shape
        kT = torch.clamp(F.softplus(self.kT), 0.01, 10.0).unsqueeze(0).unsqueeze(2)
        E  = torch.clamp(self.E, -10.0, 10.0).unsqueeze(0).unsqueeze(2)
        x_ = torch.clamp(x.unsqueeze(1), -10.0, 10.0)
        exp = torch.clamp((E - x_) / kT, -50, 50)
        w   = torch.sigmoid(exp)
        h   = (w * x_).sum(dim=3).permute(0, 2, 1)
        return torch.clamp(F.softplus(h), 0.0, 100.0)

class BaseOp(nn.Module):
    def __init__(self):
        super().__init__()
        self.param_modulator = None
    def set_param_modulator(self, modulator): self.param_modulator = modulator
    def get_modulated_params(self, context):
        if self.param_modulator is None: return {}
        return self.param_modulator(context)

class MonotonicLinearOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_bias = self.bias
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        bias  = torch.clamp(base_bias + mod.get('bias_offset', 0.0), -5.0, 5.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * (xm + bias)), 0.0, 100.0)

class MonotonicFlatOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 1e-3, 1.0
    def _cum(self, x):
        B, T = x.shape
        diff = F.relu(x[:, 1:] - x[:, :-1])
        zero_init = torch.zeros(B, 1, device=x.device, dtype=x.dtype)
        return torch.cat([zero_init, torch.cumsum(diff, dim=1)], dim=1)
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_bias = self.bias
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        bias  = torch.clamp(base_bias + mod.get('bias_offset', 0.0), -2.0, 2.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        cum_result = self._cum(xm)
        return torch.clamp(F.softplus(scale * (cum_result + bias)), 0.0, 100.0)

class ConcaveLogOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.eps = 1e-3
        self.smin, self.smax = 0.01, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        eps  = torch.clamp(self.eps + mod.get('eps_offset', 0.0), 1e-6, 1e-1)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * torch.log(torch.abs(xm) + eps)), 0.0, 100.0)

class SaturationSigmoidOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.raw_slope = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
        self.lmin, self.lmax = 0.1, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_slope = torch.clamp(F.softplus(self.raw_slope), self.lmin, self.lmax)
        base_bias  = self.bias
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        slope = torch.clamp(base_slope + mod.get('slope_offset', 0.0), self.lmin, self.lmax)
        bias  = torch.clamp(base_bias  + mod.get('bias_offset', 0.0), -3.0, 3.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * torch.sigmoid(slope * (xm - bias))), 0.0, 100.0)

class HingeReLUOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.threshold = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_threshold = self.threshold
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        threshold = torch.clamp(base_threshold + mod.get('threshold_offset', 0.0), -3.0, 3.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * F.relu(xm - threshold)), 0.0, 100.0)

class PolynomialOp(BaseOp):
    def __init__(self, deg=2):  # Reduced default degree for stability
        super().__init__()
        self.raw_coeff = nn.Parameter(torch.zeros(deg))
        self.deg = deg
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_coeff = torch.clamp(F.softplus(self.raw_coeff), 0.01, 5.0)
        xm = torch.clamp(torch.tanh(h.mean(-1)), -3.0, 3.0)  # Added tanh for stability
        coeff_offset = mod.get('coeff_offset', torch.zeros_like(base_coeff))
        if coeff_offset.dim() > 1:
            coeff_offset = coeff_offset.mean(0)
        coeff = torch.clamp(base_coeff + coeff_offset, 0.01, 5.0)
        y = torch.zeros_like(xm)
        for i in range(self.deg):
            y = y + coeff[i] * torch.clamp(xm ** (i + 1), -100.0, 100.0)
        return torch.clamp(F.softplus(y), 0.0, 100.0)

class DampedSinOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.raw_freq = nn.Parameter(torch.tensor(0.0))
        self.raw_lambda = nn.Parameter(torch.tensor(0.0))
        self.phase = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
        self.fmin, self.fmax = 0.1, 5.0
        self.lmin, self.lmax = 0.01, 3.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_freq  = torch.clamp(F.softplus(self.raw_freq), self.fmin, self.fmax)
        base_lam   = torch.clamp(F.softplus(self.raw_lambda), self.lmin, self.lmax)
        base_phase = torch.clamp(self.phase, -10.0, 10.0)
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        freq  = torch.clamp(base_freq  + mod.get('freq_offset', 0.0), self.fmin, self.fmax)
        lam   = torch.clamp(base_lam   + mod.get('lambda_offset', 0.0), self.lmin, self.lmax)
        phase = torch.clamp(base_phase + mod.get('phase_offset', 0.0), -10.0, 10.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        damp = 1 / (1 + lam * torch.abs(xm))
        return torch.clamp(F.softplus(scale * damp * (torch.sin(freq * xm + phase) + 1)), 0.0, 100.0)

class PiecewiseLinearOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_k1 = nn.Parameter(torch.tensor(0.0))
        self.raw_k2 = nn.Parameter(torch.tensor(0.0))
        self.threshold = nn.Parameter(torch.tensor(0.0))
        self.kmin, self.kmax = 0.01, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_k1 = torch.clamp(F.softplus(self.raw_k1), self.kmin, self.kmax)
        base_k2 = torch.clamp(F.softplus(self.raw_k2), self.kmin, self.kmax)
        base_thr = torch.clamp(self.threshold, -5.0, 5.0)
        k1 = torch.clamp(base_k1 + mod.get('k1_offset', 0.0), self.kmin, self.kmax)
        k2 = torch.clamp(base_k2 + mod.get('k2_offset', 0.0), self.kmin, self.kmax)
        thr= torch.clamp(base_thr + mod.get('threshold_offset', 0.0), -5.0, 5.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        left = k1 * xm
        right = k1 * thr + k2 * (xm - thr)
        out = torch.where(xm <= thr, left, right)
        return torch.clamp(F.softplus(out), 0.0, 100.0)

class ParameterModulator(nn.Module):
    def __init__(self, context_dim, param_configs):
        super().__init__()
        self.param_configs = param_configs
        self.param_predictors = nn.ModuleDict()
        for name, info in param_configs.items():
            dim = info.get('dim', 1)
            self.param_predictors[name] = nn.Sequential(
                nn.Linear(context_dim, 64), nn.ReLU(), nn.Dropout(0.1),
                nn.Linear(64, 32), nn.ReLU(),
                nn.Linear(32, dim), nn.Tanh()
            )
    def forward(self, context):
        global_context = context.mean(dim=1)  # (B, context_dim)
        modulated = {}
        for name, predictor in self.param_predictors.items():
            info = self.param_configs[name]
            raw_off = predictor(global_context)
            scale = info.get('scale', 0.1)
            modulated[name] = raw_off * scale
        return modulated

class LiquidWeightGenerator(nn.Module):
    def __init__(self, h_dim, x_dim, n_ops, hidden_dim=64, tau_min=1.0, tau_max=3.0):
        super().__init__()
        self.n_ops = n_ops
        self.tau_min, self.tau_max = tau_min, tau_max
        self.h_feature_net = nn.Sequential(nn.Linear(h_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(0.1))
        self.x_feature_net = nn.Sequential(nn.Linear(x_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(0.1))
        self.temporal_encoder = nn.Sequential(nn.Linear(3, hidden_dim // 4), nn.ReLU())
        combined_dim = hidden_dim + hidden_dim // 4
        self.feature_fusion = nn.Sequential(
            nn.Linear(combined_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.op_branches = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 4), nn.ReLU(), nn.Linear(hidden_dim // 4, 1))
            for _ in range(n_ops)
        ])
        self.temp_predictor = nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 4), nn.ReLU(), nn.Linear(hidden_dim // 4, 1))
        self.global_bias_scale = 0.01
        for branch in self.op_branches:
            nn.init.xavier_uniform_(branch[0].weight)
            nn.init.xavier_uniform_(branch[2].weight)

    def forward(self, h_multi, x):
        B, T, h_dim = h_multi.shape
        _, _, x_dim = x.shape
        h_features = self.h_feature_net(h_multi)
        x_features = self.x_feature_net(x)
        t_norm = torch.arange(T, device=x.device, dtype=x.dtype).unsqueeze(0).expand(B, -1) / max(T-1, 1)
        dt = torch.zeros_like(t_norm)
        dt[:, 1:] = t_norm[:, 1:] - t_norm[:, :-1]
        phase = torch.sin(2 * 3.14159 * t_norm)
        temporal_input = torch.stack([t_norm, dt, phase], dim=-1)
        temporal_features = self.temporal_encoder(temporal_input)
        combined = torch.cat([h_features, x_features, temporal_features], dim=-1)
        fused = self.feature_fusion(combined)
        raw_logits = []
        for branch in self.op_branches:
            raw_logits.append(branch(fused).squeeze(-1))
        raw_weights = torch.stack(raw_logits, dim=-1)                     # (B, T, K)

        # NaN protection
        raw_weights = torch.nan_to_num(raw_weights, nan=0.0, posinf=1e6, neginf=-1e6)
        raw_weights = raw_weights - raw_weights.mean(dim=-1, keepdim=True)

        if self.global_bias_scale > 0:
            global_bias = torch.randn(self.n_ops, device=raw_weights.device) * self.global_bias_scale
            raw_weights = raw_weights + global_bias.unsqueeze(0).unsqueeze(0)
        raw_temp = self.temp_predictor(fused).squeeze(-1)                 # (B, T)
        raw_temp = torch.nan_to_num(raw_temp, nan=0.0, posinf=1e6, neginf=-1e6)
        temperature = torch.clamp(F.softplus(raw_temp) + self.tau_min, self.tau_min, self.tau_max)
        weights = F.softmax(raw_weights / temperature.unsqueeze(-1), dim=-1)
        return weights, temperature

class CustomKAN(nn.Module):
    def __init__(self, ops, h_dim, x_dim):
        super().__init__()
        self.ops = nn.ModuleList(ops)
        self.n_ops = len(ops)
        self.weight_generator = LiquidWeightGenerator(h_dim=h_dim, x_dim=x_dim, n_ops=self.n_ops, hidden_dim=128)
        self.op_feature_extractors = nn.ModuleList([
            nn.Sequential(nn.Linear(h_dim, h_dim), nn.ReLU(), nn.Linear(h_dim, h_dim))
            for _ in range(self.n_ops)
        ])
        self.param_modulators = nn.ModuleList()
        context_dim = h_dim + x_dim
        for op in self.ops:
            param_configs = self._get_param_configs_for_op(op)
            if param_configs:
                modulator = ParameterModulator(context_dim, param_configs)
                self.param_modulators.append(modulator)
                op.set_param_modulator(modulator)
            else:
                self.param_modulators.append(None)
        self.raw_gain = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))

    def _get_param_configs_for_op(self, op):
        if isinstance(op, MonotonicLinearOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.2}, 'bias_offset': {'dim': 1, 'scale': 0.3}}
        if isinstance(op, MonotonicFlatOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.1}, 'bias_offset': {'dim': 1, 'scale': 0.2}}
        if isinstance(op, ConcaveLogOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.2}, 'eps_offset': {'dim': 1, 'scale': 0.01}}
        if isinstance(op, SaturationSigmoidOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.2}, 'slope_offset': {'dim': 1, 'scale': 0.3}, 'bias_offset': {'dim': 1, 'scale': 0.3}}
        if isinstance(op, HingeReLUOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.2}, 'threshold_offset': {'dim': 1, 'scale': 0.3}}
        if isinstance(op, PolynomialOp):
            return {'coeff_offset': {'dim': op.deg, 'scale': 0.2}}
        if isinstance(op, DampedSinOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.2}, 'freq_offset': {'dim': 1, 'scale': 0.3},
                    'lambda_offset': {'dim': 1, 'scale': 0.2}, 'phase_offset': {'dim': 1, 'scale': 0.5}}
        if isinstance(op, PiecewiseLinearOp):
            return {'k1_offset': {'dim': 1, 'scale': 0.2}, 'k2_offset': {'dim': 1, 'scale': 0.2},
                    'threshold_offset': {'dim': 1, 'scale': 0.3}}
        return {}

    def forward(self, h_multi, x):
        op_features = [ext(h_multi) for ext in self.op_feature_extractors]
        context = torch.cat([h_multi, x], dim=-1)
        outs = [op(op_features[i], context) for i, op in enumerate(self.ops)]
        Tm = min(o.size(1) for o in outs)
        outs = [o[:, :Tm] for o in outs]
        h_multi_aligned = h_multi[:, :Tm, :]
        x_aligned = x[:, :Tm, :]
        op_stack = torch.stack(outs, dim=-1)  # (B, Tm, K)
        weights, temperature = self.weight_generator(h_multi_aligned, x_aligned)
        damage = torch.sum(op_stack * weights, dim=-1)  # (B, Tm)
        damage = torch.clamp(damage, 0.0, 100.0)
        gain = torch.clamp(F.softplus(self.raw_gain), 0.1, 2.0)
        bias_val = torch.clamp(self.bias, -2.0, 2.0)
        damage = torch.clamp(gain * damage + bias_val, 0.0, 100.0)
        return damage, weights, temperature

class TrendEncoder(nn.Module):
    """
    Vectorized monotonic decreasing HI projection.
    """
    def __init__(self, in_ch, trend_ch=4):
        super().__init__()
        self.boltz = BoltzmannKAN(in_ch, out_ch=trend_ch)
        ops = [MonotonicLinearOp(), MonotonicFlatOp(), ConcaveLogOp(),
               SaturationSigmoidOp(), HingeReLUOp(), PolynomialOp(),
               DampedSinOp(), PiecewiseLinearOp()]
        self.customkan = CustomKAN(ops, h_dim=trend_ch, x_dim=in_ch)

        self.proj_gain = nn.Parameter(torch.tensor(1.0))
        self.proj_bias = nn.Parameter(torch.tensor(0.0))

        self.health_aware_transform = nn.Sequential(
            nn.Linear(in_ch, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )
        self.linear_enforcer = nn.Sequential(
            nn.Linear(in_ch, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )

    def forward(self, x):  # x:(B,T,C)
        B, T, C = x.shape
        h_multi = self.boltz(x)                               # (B,T,trend_ch)
        damage, weights, temperature = self.customkan(h_multi, x)

        x_reshaped = x.view(-1, C)
        health_direct = self.health_aware_transform(x_reshaped).view(B, T)  # (B,T)
        linear_weight = self.linear_enforcer(x_reshaped).view(B, T)

        t = torch.arange(T, device=x.device, dtype=x.dtype).unsqueeze(0).expand(B, -1)
        t_normalized = t / (T - 1 + 1e-6)

        g = torch.clamp(F.softplus(self.proj_gain), 0.1, 5.0)
        b = torch.clamp(self.proj_bias, -5.0, 5.0)
        damage_normalized = torch.sigmoid(g*(damage + b))

        combined_health = health_direct * (1 - 0.3 * damage_normalized)
        linear_decay = 1.0 - t_normalized * 0.5
        hi = linear_weight * linear_decay + (1 - linear_weight) * combined_health

        # ---- Vectorized strictly non-increasing enforcement with epsilon step ----
        eps = 1e-3
        idx = torch.arange(T, device=hi.device, dtype=hi.dtype).unsqueeze(0)  # (1, T)
        hi_mon, _ = torch.cummin(hi + eps * idx, dim=1)
        hi = hi_mon - eps * idx
        # -------------------------------------------------------------------------

        return hi, h_multi, weights, temperature

class ReconHead(nn.Module):
    def __init__(self, C, hidden=128):
        super().__init__()
        self.gru = nn.GRU(input_size=C+1, hidden_size=hidden, batch_first=True)
        self.out = nn.Linear(hidden, C)
    def forward(self, x_in, h_in):
        B,T,C = x_in.shape
        h_in_clamped = torch.clamp(h_in, 0.0, 10.0)
        feat = torch.cat([x_in, h_in_clamped.unsqueeze(-1)], dim=-1)
        H,_ = self.gru(feat)
        y = self.out(H)
        return y

class MaintClassifier(nn.Module):
    def __init__(self, sensor_dim, hidden=64, n_classes=14):
        super().__init__()
        self.hi_mlp = nn.Sequential(
            nn.Linear(6, hidden), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden//2)
        )
        self.sensor_mlp = nn.Sequential(
            nn.Linear(sensor_dim * 2, hidden), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden//2)
        )
        self.final_classifier = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, n_classes)
        )

    def forward(self, h_b, h_a, x_b, x_a, mask):
        m = mask
        T = m.size(1)
        valid = m.sum(1, keepdim=True).clamp_min(1.0)

        db = h_a - h_b
        mean_d = (db*m).sum(1,keepdim=True)/valid
        var_d  = (((db-mean_d)*m)**2).sum(1,keepdim=True)/valid
        std_d  = (var_d + 1e-8).sqrt()
        d0     = (db[:, :1])
        dmax   = (db.masked_fill(m==0, -1e9)).max(dim=1, keepdim=True).values
        pos_ratio = ((db>0).float()*m).sum(1,keepdim=True)/valid

        t = torch.arange(T, device=h_b.device, dtype=h_b.dtype)
        t = (t - t.mean())/(t.std()+1e-6)
        def slope(x):
            num = (x*t).sum(1) - x.sum(1)*t.sum()/T
            den = (t**2).sum() - (t.sum()**2)/T + 1e-8
            return (num/den).unsqueeze(1)
        slope_diff = slope(h_a) - slope(h_b)

        hi_feat = torch.cat([mean_d, d0, dmax, std_d, slope_diff, pos_ratio], dim=1)
        hi_feat = torch.clamp(hi_feat, -10.0, 10.0)
        hi_features = self.hi_mlp(hi_feat)

        x_b_mean = (x_b * m.unsqueeze(-1)).sum(1) / valid
        x_a_mean = (x_a * m.unsqueeze(-1)).sum(1) / valid
        sensor_change = torch.cat([x_b_mean, x_a_mean], dim=1)
        sensor_features = self.sensor_mlp(sensor_change)

        combined = torch.cat([hi_features, sensor_features], dim=1)
        logits = self.final_classifier(combined)
        return logits

def sanitize_tensors(*tensors):
    result = []
    for t in tensors:
        if t is not None:
            t_clean = torch.nan_to_num(t, nan=0.0, posinf=1e6, neginf=-1e6)
            result.append(t_clean)
        else:
            result.append(t)
    return result[0] if len(result) == 1 else result

class DiffAwareReconstructor(nn.Module):
    """
    - Two encoders share parameters: encode x_before / x_after → h_b / h_a (monotonic decreasing HI)
    - Two reconstruction heads: one-step recursive reconstruction
    - Classification head: ΔHI + sensor change features
    """
    def __init__(self, in_ch, trend_ch=4, hidden=128, n_classes=14):
        super().__init__()
        self.encoder = TrendEncoder(in_ch, trend_ch)
        self.recon_b = ReconHead(in_ch, hidden)
        self.recon_a = ReconHead(in_ch, hidden)
        self.clf     = MaintClassifier(sensor_dim=in_ch, hidden=64, n_classes=n_classes)
        self.consistency_weight = nn.Parameter(torch.tensor(1.0))

    def forward(self, x_b, x_a, mask):
        h_b, h_multi_b, weights_b, temp_b = self.encoder(x_b)
        h_a, h_multi_a, weights_a, temp_a = self.encoder(x_a)

        h_b, h_a = sanitize_tensors(h_b, h_a)
        weights_b, weights_a = sanitize_tensors(weights_b, weights_a)

        xb_in, hb_in = x_b[:, :-2], h_b[:, :-2]
        xa_in, ha_in = x_a[:, :-2], h_a[:, :-2]
        yb_hat = self.recon_b(xb_in, hb_in)   # (B, L-2, C) ≈ x_b[:,1:L-1]
        ya_hat = self.recon_a(xa_in, ha_in)   # (B, L-2, C) ≈ x_a[:,1:L-1]

        yb_hat, ya_hat = sanitize_tensors(yb_hat, ya_hat)
        logits = self.clf(h_b, h_a, x_b, x_a, mask)
        logits = sanitize_tensors(logits)
        return yb_hat, ya_hat, h_b, h_a, logits, weights_b, weights_a, temp_b, temp_a

# ---------------------------------------------------------------------
# Losses / Regularizers
# ---------------------------------------------------------------------
def masked_mse(a, b, mask):
    diff = (a - b)**2
    mse  = (diff.mean(-1) * mask).sum() / (mask.sum() + 1e-6)
    return mse

def masked_mse_batchwise(a, b, mask):
    diff = (a - b)**2  # (B, T, C)
    mse_t = diff.mean(-1) * mask  # (B, T)
    denom = mask.sum(1).clamp_min(1.0)  # (B,)
    return mse_t.sum(1) / denom

def slope_loss(h, mask, kind="smooth"):
    if kind=="smooth":
        d2 = h[:,2:] - 2*h[:,1:-1] + h[:,:-2]
        m  = mask[:,2:]
        return (d2.abs() * m).sum() / (m.sum()+1e-6)
    elif kind=="mono_dec":
        dh = h[:,1:] - h[:,:-1]
        m  = mask[:,1:]
        return (F.relu(dh) * m).sum() / (m.sum()+1e-6)
    else:
        return torch.tensor(0., device=h.device)

def enhanced_liquid_weight_regularization(weights_b, weights_a, mask,
                                          lambda_tv=0.05, lambda_ent=0.2,
                                          lambda_bal=0.1, lambda_div=0.1):
    total_loss = torch.tensor(0.0, device=weights_b.device)

    def tv_loss(weights, mask):
        if weights.size(1) <= 1:
            return torch.tensor(0.0, device=weights.device)
        diff = weights[:, 1:, :] - weights[:, :-1, :]
        mask_diff = mask[:, 1:]
        tv = (diff.abs().sum(-1) * mask_diff).sum() / (mask_diff.sum() + 1e-6)
        return tv

    tv_loss_b = tv_loss(weights_b, mask)
    tv_loss_a = tv_loss(weights_a, mask)
    total_loss += lambda_tv * (tv_loss_b + tv_loss_a)

    def entropy_loss(weights, mask):
        eps = 1e-8
        entropy = -(weights * torch.log(weights + eps)).sum(-1)
        target_entropy = np.log(weights.size(-1)) * 0.8
        entropy_penalty = F.relu(target_entropy - entropy)
        return (entropy_penalty * mask).sum() / (mask.sum() + 1e-6)

    ent_loss_b = entropy_loss(weights_b, mask)
    ent_loss_a = entropy_loss(weights_a, mask)
    total_loss += lambda_ent * (ent_loss_b + ent_loss_a)

    def balance_loss(weights, mask):
        valid_weights = weights * mask.unsqueeze(-1)  # (B, T, K)
        mean_usage = valid_weights.sum([0, 1]) / (mask.sum() + 1e-6)  # (K,)
        target_usage = 1.0 / weights.size(-1)
        balance = ((mean_usage - target_usage) ** 2).sum()
        return balance

    bal_loss_b = balance_loss(weights_b, mask)
    bal_loss_a = balance_loss(weights_a, mask)
    total_loss += lambda_bal * (bal_loss_b + bal_loss_a)

    def diversity_loss(weights, mask):
        B, T, K = weights.shape
        if T <= 1:
            return torch.tensor(0.0, device=weights.device)
        valid_weights = weights * mask.unsqueeze(-1)
        weights_flat = valid_weights.view(-1, K)
        weights_centered = weights_flat - weights_flat.mean(0, keepdim=True)
        cov = torch.mm(weights_centered.T, weights_centered) / (weights_flat.size(0) - 1 + 1e-6)
        mask_offdiag = torch.ones_like(cov) - torch.eye(K, device=cov.device)
        corr_penalty = (cov.abs() * mask_offdiag).sum()
        return corr_penalty

    div_loss_b = diversity_loss(weights_b, mask)
    div_loss_a = diversity_loss(weights_a, mask)
    total_loss += lambda_div * (div_loss_b + div_loss_a)
    return total_loss

def smoothness_enhancement_loss(h, mask, order=2):
    if order == 2:
        d2 = h[:,2:] - 2*h[:,1:-1] + h[:,:-2]
        m = mask[:,2:]
        return (d2.abs() * m).sum() / (m.sum() + 1e-6)
    elif order == 3:
        d3 = h[:,3:] - 3*h[:,2:-1] + 3*h[:,1:-2] - h[:,:-3]
        m = mask[:,3:]
        return (d3.abs() * m).sum() / (m.sum() + 1e-6)
    else:
        return torch.tensor(0., device=h.device)

def compute_sample_weights(matching_costs, alpha=0.5):
    return 1.0 / (1.0 + alpha * matching_costs)

def maintenance_effect_loss(h_b, h_a, mask, lambda_diff=1.0, lambda_smooth_after=0.5, lambda_level_constraint=3.0):
    """
    Encourage maintenance effect: after-maintenance HI should be different from before,
    after-maintenance HI should be smoother, and min(h_a) > max(h_b) for proper separation.
    """
    total_loss = torch.tensor(0.0, device=h_b.device)

    # 1. Encourage difference between before and after HI (improved stability)
    hi_diff = torch.abs(h_a - h_b)  # (B, T)
    hi_diff_clamped = torch.clamp(hi_diff, min=1e-3)  # Prevent extreme gradients
    diff_loss = F.softplus(-10.0 * hi_diff_clamped)  # Smooth penalty for small differences
    diff_loss = (diff_loss * mask).sum() / (mask.sum() + 1e-6)
    total_loss += lambda_diff * diff_loss

    # 2. Encourage after-maintenance HI to be smoother (less variation)
    if h_a.size(1) > 1:
        h_a_diff = torch.abs(h_a[:, 1:] - h_a[:, :-1])  # (B, T-1)
        mask_diff = mask[:, 1:]
        smooth_after_loss = (h_a_diff * mask_diff).sum() / (mask_diff.sum() + 1e-6)
        total_loss += lambda_smooth_after * smooth_after_loss

    # 3. Encourage min(h_a) > max(h_b) for proper level separation
    # Use masked operations to handle variable lengths
    h_b_masked = h_b.masked_fill(mask == 0, -1e9)  # Fill invalid positions with very low values
    h_a_masked = h_a.masked_fill(mask == 0, 1e9)   # Fill invalid positions with very high values

    h_b_max = h_b_masked.max(dim=1)[0]  # (B,) - max of each sequence
    h_a_min = h_a_masked.min(dim=1)[0]  # (B,) - min of each sequence

    # Encourage h_a_min > h_b_max with a margin
    margin = 0.1
    level_constraint = F.relu(h_b_max - h_a_min + margin)  # (B,)
    level_constraint_loss = level_constraint.mean()
    total_loss += lambda_level_constraint * level_constraint_loss
    margin_pt = 0.05
    pointwise_gap = (h_a - h_b) + (-margin_pt)
    pointwise_loss = F.relu(-pointwise_gap)      # Penalize cases where h_a < h_b + margin_pt
    pointwise_loss = (pointwise_loss * mask).sum() / (mask.sum() + 1e-6)

    total_loss += 1.0 * pointwise_loss   # Tunable coefficient: try values between 0.5 and 2.0
    return total_loss

# ---------------------------------------------------------------------
# Evaluation / Prediction
# ---------------------------------------------------------------------
@torch.no_grad()
def eval_epoch(model, loader, device, K, compute_acc=True):
    model.eval()
    total = {"mse_b":0.0, "mse_a":0.0, "acc":0.0, "delta_mean":0.0}
    n_batch=0
    n_acc = 0; n_tot=0
    for batch in loader:
        xb, xa, labels, mask, lengths, sample_ids, matching_costs = adapt_batch(batch, device)
        m_tgt  = (torch.arange(xb.size(1)-2, device=device)[None,:] < (lengths-2)[:,None]).float()
        yb_hat, ya_hat, h_b, h_a, logits, _, _, _, _ = model(xb, xa, mask)
        yb_hat, ya_hat, h_b, h_a, logits = sanitize_tensors(yb_hat, ya_hat, h_b, h_a, logits)

        # Ensure logits dimension matches expected K
        if logits.shape[-1] != K:
            print(f"Warning: Logits dim {logits.shape[-1]} != expected K={K}, adjusting...")
            if logits.shape[-1] < K:
                # Pad with zeros
                padding = torch.zeros(logits.shape[0], K - logits.shape[-1], device=logits.device)
                logits = torch.cat([logits, padding], dim=1)
            else:
                # Truncate
                logits = logits[:, :K]

        yb_tgt = xb[:,1:-1,:]
        ya_tgt = xa[:,1:-1,:]
        mse_b = masked_mse(yb_hat, yb_tgt, m_tgt)
        mse_a = masked_mse(ya_hat, ya_tgt, m_tgt)
        if compute_acc and labels is not None:
            # Filter out ignore_index labels
            valid_labels = labels != -100
            if valid_labels.any():
                pred = logits[valid_labels].argmax(dim=1)
                labels_valid = labels[valid_labels]
                n_acc += (pred == labels_valid).sum().item()
                n_tot += labels_valid.numel()
        valid = mask.sum(1, keepdim=True).clamp_min(1.0)
        delta_mean = ((h_a - h_b)*mask).sum(1,keepdim=True)/valid
        delta_mean = torch.nan_to_num(delta_mean, nan=0.0)
        total["delta_mean"] += delta_mean.mean().item()
        total["mse_b"] += mse_b.item()
        total["mse_a"] += mse_a.item()
        n_batch += 1
    for k in ["mse_b","mse_a","delta_mean"]:
        total[k] /= max(n_batch,1)
    total["acc"] = (n_acc / max(n_tot,1)) if n_tot>0 else 0.0
    return total

@torch.no_grad()
def collect_test_predictions(model, loader, device, max_curve_keep=24, K=14, id_to_name=None):
    model.eval()
    y_true, y_pred = [], []
    all_delta_mean, all_sample_ids = [], []
    keep_curves = []
    for batch in loader:
        xb, xa, labels, mask, lengths, sample_ids, matching_costs = adapt_batch(batch, device)
        yb_hat, ya_hat, h_b, h_a, logits, weights_b, weights_a, temp_b, temp_a = model(xb, xa, mask)
        yb_hat, ya_hat, h_b, h_a, logits = sanitize_tensors(yb_hat, ya_hat, h_b, h_a, logits)

        # Ensure logits dimension matches expected K
        if logits.shape[-1] != K:
            if logits.shape[-1] < K:
                padding = torch.zeros(logits.shape[0], K - logits.shape[-1], device=logits.device)
                logits = torch.cat([logits, padding], dim=1)
            else:
                logits = logits[:, :K]

        prob = F.softmax(logits, dim=1)     # (B,K)
        pred = prob.argmax(1)               # (B,)

        valid = mask.sum(1, keepdim=True).clamp_min(1.0)
        delta_mean = ((h_a - h_b) * mask).sum(1, keepdim=True) / valid  # (B,1)
        delta_mean = torch.nan_to_num(delta_mean, nan=0.0)

        if labels is not None:
            # Filter out ignore_index labels
            valid_labels = labels != -100
            if valid_labels.any():
                y_true.append(labels[valid_labels].detach().cpu().numpy())
                y_pred.append(pred[valid_labels].detach().cpu().numpy())
        all_delta_mean.append(delta_mean.squeeze(1).cpu().numpy())
        all_sample_ids.extend(sample_ids)

        for i in range(xb.size(0)):
            if len(keep_curves) >= max_curve_keep: break
            L_i = int(lengths[i].item())
            L_recon = min(L_i - 2, yb_hat.size(1))
            keep_curves.append({
                "sample_id": sample_ids[i],
                "true": int(labels[i].cpu().item()) if labels is not None and labels[i] != -100 else -1,
                "pred": int(pred[i].cpu().item()),
                "prob": prob[i].cpu().numpy(),
                "h_before": h_b[i, :L_i].cpu().numpy(),
                "h_after":  h_a[i, :L_i].cpu().numpy(),
                "weights_before": weights_b[i, :L_i].cpu().numpy() if weights_b is not None else None,
                "weights_after": weights_a[i, :L_i].cpu().numpy() if weights_a is not None else None,
                "temp_before": temp_b[i, :L_i].cpu().numpy() if temp_b is not None else None,
                "temp_after": temp_a[i, :L_i].cpu().numpy() if temp_a is not None else None,
                "op_outputs_before": [],
                "op_outputs_after":  [],
                "yb_hat":   yb_hat[i, :L_recon].cpu().numpy(),
                "ya_hat":   ya_hat[i, :L_recon].cpu().numpy(),
                "x_before": xb[i, :L_i].cpu().numpy(),
                "x_after":  xa[i, :L_i].cpu().numpy(),
            })
    y_true = np.concatenate(y_true, axis=0) if len(y_true)>0 else np.array([])
    y_pred = np.concatenate(y_pred, axis=0) if len(y_pred)>0 else np.array([])
    all_delta_mean = np.concatenate(all_delta_mean, axis=0) if len(all_delta_mean)>0 else np.array([])
    return y_true, y_pred, all_delta_mean, all_sample_ids, keep_curves

# ---------------------------------------------------------------------
# Plotting
# ---------------------------------------------------------------------
def remove_constant_segments(hi_values, threshold=1e-6):
    if len(hi_values) <= 1:
        return hi_values, np.arange(len(hi_values))
    diffs = np.abs(np.diff(hi_values))
    valid_mask = np.ones(len(hi_values), dtype=bool)
    valid_mask[1:] = diffs > threshold
    if np.sum(valid_mask) < 2:
        valid_mask[0] = True
        valid_mask[-1] = True
    valid_indices = np.where(valid_mask)[0]
    return hi_values[valid_indices], valid_indices

def plot_hi_examples_aligned(curves, id_to_name, n_show=6, seed=0):
    if len(curves)==0:
        print("(No visualization samples)")
        return
    random.Random(seed).shuffle(curves)
    n_show = min(n_show, len(curves))
    cols = 3
    rows = int(np.ceil(n_show/cols))
    plt.figure(figsize=(cols*4.6, rows*3.2))
    for k in range(n_show):
        ex = curves[k]
        hb = ex["h_before"]; ha = ex["h_after"]
        L  = min(len(hb), len(ha))
        hb, ha = hb[:L], ha[:L]
        hb_clean, hb_indices = remove_constant_segments(hb)
        ha_clean, ha_indices = remove_constant_segments(ha)
        t_b = hb_indices
        t_a = ha_indices + L

        sample_id = ex["sample_id"]
        pred = ex["pred"]; true = ex["true"]; prob = ex["prob"]
        true_name = id_to_name.get(true, f"Class_{true}")
        pred_name = id_to_name.get(pred, f"Class_{pred}")

        plt.subplot(rows, cols, k+1)
        if len(hb_clean) > 1:
            plt.plot(t_b, hb_clean, label="Learned HI (Pre)", linewidth=1.8, marker='o', markersize=3)
        if len(ha_clean) > 1:
            plt.plot(t_a, ha_clean, label="Learned HI (Post)",  linewidth=1.8, linestyle='--', marker='s', markersize=3)
        plt.axvline(L-1, color='k', linestyle=':', linewidth=1.0, alpha=0.7)
        d_mean = float(np.mean(ha - hb))
        d_max  = float(np.max(ha - hb))
        plt.title(f"ID={sample_id}\nTrue={true_name} | Pred={pred_name} "
                  f"| p={prob[pred]:.2f}\nΔHI_mean={d_mean:.3f}, ΔHI_max={d_max:.3f}", fontsize=9)
        plt.xlabel("Cycle (Pre | Post)")
        plt.ylabel("Health Index (higher=better)")
        plt.grid(ls='--', alpha=.35)
        if k==0: plt.legend()
    plt.tight_layout(); plt.show()

def plot_enhanced_liquid_weights(curves, n_show=3, seed=0):
    if len(curves) == 0:
        print("(No visualization samples)")
        return
    curves_with_weights = [c for c in curves if c.get("weights_before") is not None]
    if len(curves_with_weights) == 0:
        print("(No weight information available)")
        return
    random.Random(seed).shuffle(curves_with_weights)
    n_show = min(n_show, len(curves_with_weights))
    op_names = ["MonotonicLinear", "MonotonicFlat", "ConcaveLog", "SaturationSigmoid",
                "HingeReLU", "Polynomial", "DampedSin", "PiecewiseLinear"]
    for i in range(n_show):
        ex = curves_with_weights[i]
        weights_b = ex["weights_before"]
        weights_a = ex["weights_after"]
        temp_b = ex.get("temp_before")
        temp_a = ex.get("temp_after")
        sample_id = ex["sample_id"]
        true_label = ex["true"]
        pred_label = ex["pred"]
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle(f"Dynamic Parameter Modulation - ID={sample_id}, True=Class_{true_label}, Pred=Class_{pred_label}", fontsize=14)
        ax = axes[0, 0]
        T_b, K = weights_b.shape
        for k in range(K):
            ax.plot(range(T_b), weights_b[:, k], label=op_names[k] if k < len(op_names) else f"Op_{k}",
                   marker='o', markersize=2, linewidth=1.5)
        ax.set_title("Pre-maintenance Weights"); ax.set_xlabel("Time Step"); ax.set_ylabel("Weight")
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left'); ax.grid(True, alpha=0.3)
        ax = axes[0, 1]
        T_a, _ = weights_a.shape
        for k in range(K):
            ax.plot(range(T_a), weights_a[:, k], label=op_names[k] if k < len(op_names) else f"Op_{k}",
                   marker='s', markersize=2, linewidth=1.5, linestyle='--')
        ax.set_title("Post-maintenance Weights"); ax.set_xlabel("Time Step"); ax.set_ylabel("Weight")
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left'); ax.grid(True, alpha=0.3)
        ax = axes[0, 2]
        entropy_b = -np.sum(weights_b * np.log(weights_b + 1e-8), axis=1)
        entropy_a = -np.sum(weights_a * np.log(weights_a + 1e-8), axis=1)
        ax.plot(range(T_b), entropy_b, label="Pre", marker='o', linewidth=2)
        ax.plot(range(T_a), entropy_a, label="Post", marker='s', linewidth=2, linestyle='--')
        ax.axhline(np.log(K), color='r', linestyle=':', label=f'Max Entropy ({np.log(K):.2f})')
        ax.set_title("Weight Entropy Over Time"); ax.set_xlabel("Time Step"); ax.set_ylabel("Entropy")
        ax.legend(); ax.grid(True, alpha=0.3)
        ax = axes[1, 0]
        if temp_b is not None and temp_a is not None:
            ax.plot(range(T_b), temp_b, label="Pre", marker='o', linewidth=2)
            ax.plot(range(T_a), temp_a, label="Post", marker='s', linewidth=2, linestyle='--')
            ax.set_title("Temperature Evolution"); ax.set_xlabel("Time Step"); ax.set_ylabel("Temperature"); ax.legend()
        else:
            ax.text(0.5, 0.5, "Temperature data\nnot available", ha='center', va='center', transform=ax.transAxes)
            ax.set_title("Temperature Evolution")
        ax.grid(True, alpha=0.3)
        ax = axes[1, 1]
        ax.text(0.5, 0.5, "Operator output variance\n(not collected)", ha='center', va='center', transform=ax.transAxes)
        ax.set_title("Operator Output Variance"); ax.grid(True, alpha=0.3)
        ax = axes[1, 2]
        combined_weights = np.concatenate([weights_b, weights_a], axis=0)
        # Apply smoothing for correlation stability
        from scipy.ndimage import uniform_filter1d
        smoothed_weights = uniform_filter1d(combined_weights, size=3, axis=0)
        corr_matrix = np.corrcoef(smoothed_weights.T)
        im = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
        ax.set_title("Operator Weight Correlation (Smoothed)")
        ax.set_xticks(range(K)); ax.set_yticks(range(K))
        short_names = [op_names[k][:8] if k < len(op_names) else f"Op_{k}" for k in range(K)]
        ax.set_xticklabels(short_names, rotation=45); ax.set_yticklabels(short_names)
        cbar = plt.colorbar(im, ax=ax); cbar.set_label('Correlation')
        plt.tight_layout(); plt.show()

def plot_liquid_weights(curves, n_show=3, seed=0):
    plot_enhanced_liquid_weights(curves, n_show, seed)

def plot_sensor_examples_aligned(curves, sensor_idx_list=None, n_cols=4):
    if len(curves)==0:
        print("(No visualization samples)")
        return
    ex = curves[0]
    xb = ex["x_before"]; xa = ex["x_after"]; ya = ex["ya_hat"]
    L, C = xb.shape
    Lhat = ya.shape[0]
    if sensor_idx_list is None:
        step = max(1, C//8); sensor_idx_list = list(range(0, C, step))[:8]
    n = len(sensor_idx_list)
    n_rows = int(np.ceil(n/n_cols))
    plt.figure(figsize=(n_cols*4.3, n_rows*3.1))
    for i, s in enumerate(sensor_idx_list):
        plt.subplot(n_rows, n_cols, i+1)
        t_b = np.arange(L); t_a = np.arange(L, 2*L)
        plt.plot(t_b, xb[:,s], label="Original (pre)", linewidth=1.2)
        plt.plot(t_a, xa[:,s], label="Original (post)",  linewidth=1.0, linestyle="--", alpha=0.7)
        t_pred = np.arange(L+1, L+1+Lhat)
        plt.plot(t_pred, ya[:,s], label="Post prediction", linewidth=1.6)
        plt.axvline(L-1, color='k', linestyle=':', linewidth='1.0')
        plt.title(f"Sensor_{s:02d}")
        if i==0: plt.legend()
        plt.grid(ls="--", alpha=.35)
    plt.tight_layout(); plt.show()

def topk_by_delta(df_delta, id_to_name, k=3):
    if len(df_delta)==0:
        return
    unique_classes = sorted(df_delta["true"].unique())
    for cls in unique_classes[:5]:
        sub = df_delta[df_delta["true"]==cls].sort_values("delta_hi_mean", ascending=False).head(k)
        class_name = id_to_name.get(cls, f"Class_{cls}")
        print(f"\n[True={class_name}] Top {k} samples with highest ΔHI_mean:")
        print(sub[["uid","delta_hi_mean","pred"]].reset_index(drop=True))

# ---------------------------------------------------------------------
# Training (separate) and Testing (separate)
# ---------------------------------------------------------------------
def build_model_and_setup(train_loader, device, class_weights=None):
    """
    Build model and setup training components.

    Args:
        train_loader: Training data loader
        device: Device to use
        class_weights: Pre-computed class weights tensor (optional)
    """
    # Determine K from training loader
    K = scan_num_classes_from_loader(train_loader)
    id_to_name = build_default_id_to_name(K)

    first_batch = next(iter(train_loader))
    C = first_batch["x_before"].shape[-1]

    print(f"\n[Setup] Sensor dim C={C}, Num classes K={K}")

    # Use provided class weights or create uniform weights
    if class_weights is not None:
        if isinstance(class_weights, np.ndarray):
            class_weights = torch.from_numpy(class_weights).float().to(device)
        elif isinstance(class_weights, torch.Tensor):
            class_weights = class_weights.float().to(device)

        # Ensure class_weights has the right size
        if len(class_weights) < K:
            # Pad with ones if needed
            padded_weights = torch.ones(K, dtype=torch.float32, device=device)
            padded_weights[:len(class_weights)] = class_weights
            class_weights = padded_weights
        elif len(class_weights) > K:
            # Truncate if needed
            class_weights = class_weights[:K]

        print(f"[Class Weights] Using provided weights: {class_weights.detach().cpu().numpy()}")
    else:
        class_weights = torch.ones(K, dtype=torch.float32, device=device)
        print(f"[Class Weights] Using uniform weights: {class_weights.detach().cpu().numpy()}")

    model = DiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=K).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    ce  = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100, reduction='none')

    return model, opt, ce, K, id_to_name, C


def train_only(train_loader, val_loader, epochs=300, lr=1e-3, device=None, patience=20,
               use_matching_cost=True, save_path="best_model.pth", class_weights=None,
               es_alpha=1.0):
    """
    Train and save the best model. Returns path to checkpoint and metadata.

    Args:
        class_weights: Pre-computed class weights (numpy array or torch tensor)
        es_alpha: Weight for accuracy in early stopping metric
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Build model and training setup
    model, opt, ce, K, id_to_name, C = build_model_and_setup(train_loader, device, class_weights)
    # Override LR if provided
    for g in opt.param_groups:
        g['lr'] = lr

    best_v = float("inf")
    best_state = None
    best_epoch = 0
    no_improve = 0
    nan_batches = 0

    print(f"\nStarting training for {epochs} epochs with early stopping (patience={patience})...")
    print(f"Device: {device}")
    print(f"Early stopping alpha: {es_alpha}")

    for ep in range(1, epochs+1):
        print(f"\n[Epoch {ep}/{epochs}] Training...")
        model.train()
        logs = {"mse_b":0.0, "mse_a":0.0, "smooth":0.0, "mono":0.0,
                "cls":0.0, "smooth_enh":0.0, "liquid_weight":0.0, "maintenance_effect":0.0, "all":0.0}
        n_bt = 0
        batch_nan_count = 0

        progress = ep / epochs
        lambda_tv  = 0.03 * (1 - progress * 0.3)
        lambda_ent = 0.3  * (1 - progress * 0.7)
        lambda_bal = 0.15 * (1 - progress * 0.5)
        lambda_div = 0.2  * (1 - progress * 0.6)

        # Gradient norm tracking
        grad_norms = []

        for batch_idx, batch in enumerate(train_loader):
            xb, xa, labels, mask, lengths, sample_ids, matching_costs = adapt_batch(batch, device)
            m_tgt  = (torch.arange(xb.size(1)-2, device=device)[None,:] < (lengths-2)[:,None]).float()
            yb_hat, ya_hat, h_b, h_a, logits, weights_b, weights_a, temp_b, temp_a = model(xb, xa, mask)
            yb_hat, ya_hat, h_b, h_a, logits = sanitize_tensors(yb_hat, ya_hat, h_b, h_a, logits)
            weights_b, weights_a = sanitize_tensors(weights_b, weights_a)

            # Reconstruction targets
            yb_tgt = xb[:,1:-1,:]
            ya_tgt = xa[:,1:-1,:]

            # Per-sample reconstruction losses (for optional cost-weighting)
            rec_b_per = masked_mse_batchwise(yb_hat, yb_tgt, m_tgt)
            rec_a_per = masked_mse_batchwise(ya_hat, ya_tgt, m_tgt)
            rec_per = rec_b_per + rec_a_per

            # Classification loss (per-sample)
            if labels is None:
                cls_per = torch.zeros(xb.size(0), device=device)
            else:
                cls_per = ce(logits, labels)

            # Optional matching-cost weighting
            if use_matching_cost and matching_costs is not None:
                w_i = compute_sample_weights(matching_costs)
            else:
                w_i = torch.ones(xb.size(0), device=device)

            loss_rec = (rec_per * w_i).mean()
            loss_cls = (cls_per * w_i).mean()

            # HI regularizations
            loss_smooth = slope_loss(h_a, mask, "smooth") + slope_loss(h_b, mask, "smooth")
            loss_mono   = slope_loss(h_a, mask, "mono_dec") + slope_loss(h_b, mask, "mono_dec")
            loss_smooth_enhanced = (smoothness_enhancement_loss(h_b, mask, order=2) +
                                   smoothness_enhancement_loss(h_a, mask, order=2) +
                                   smoothness_enhancement_loss(h_b, mask, order=3) * 0.3 +
                                   smoothness_enhancement_loss(h_a, mask, order=3) * 0.3)
            loss_liquid_weight = enhanced_liquid_weight_regularization(
                weights_b, weights_a, mask,
                lambda_tv=lambda_tv, lambda_ent=lambda_ent,
                lambda_bal=lambda_bal, lambda_div=lambda_div
            )

            # Maintenance effect loss to encourage proper HI behavior
            loss_maintenance_effect = maintenance_effect_loss(
                h_b, h_a, mask,
                lambda_diff=1.0,           # Encourage difference between before/after
                lambda_smooth_after=0.5,   # Encourage after-maintenance HI to be smoother
                lambda_level_constraint=2.0 # Encourage min(h_a) > max(h_b)
            )

            # Total loss
            w_rec=1.0; w_smooth=0.03; w_mono=0.05; w_cls=1.0; w_smooth_enh=0.3; w_liquid=0.5; w_maintenance=2.0
            loss = (w_rec*loss_rec + w_smooth*loss_smooth + w_mono*loss_mono +
                    w_cls*loss_cls + w_smooth_enh*loss_smooth_enhanced + w_liquid*loss_liquid_weight +
                    w_maintenance*loss_maintenance_effect)

            loss_rec, loss_cls, loss_smooth, loss_mono, loss_smooth_enhanced, loss_liquid_weight, loss_maintenance_effect, loss = sanitize_tensors(
                loss_rec, loss_cls, loss_smooth, loss_mono, loss_smooth_enhanced, loss_liquid_weight, loss_maintenance_effect, loss
            )

            if torch.isnan(loss) or torch.isinf(loss):
                batch_nan_count += 1
                if batch_nan_count == 1:
                    print(f"    [Warning] Epoch {ep}: Found NaN/Inf loss, skipping abnormal batch")
                continue

            opt.zero_grad()
            loss.backward()

            # Adaptive gradient clipping
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            grad_norms.append(grad_norm.item())

            opt.step()

            logs["mse_b"] += rec_b_per.mean().item()
            logs["mse_a"] += rec_a_per.mean().item()
            logs["smooth"]+= loss_smooth.item()
            logs["mono"]  += loss_mono.item()
            logs["cls"]   += loss_cls.item()
            logs["smooth_enh"]+= loss_smooth_enhanced.item()
            logs["liquid_weight"]+= loss_liquid_weight.item()
            logs["maintenance_effect"]+= loss_maintenance_effect.item()
            logs["all"]   += loss.item()
            n_bt += 1

        if batch_nan_count > 0:
            nan_batches += batch_nan_count

        for k in logs: logs[k] /= max(n_bt,1)

        print("    Validating...")
        vl = eval_epoch(model, val_loader, device, K, compute_acc=True)
        # Improved early-stopping metric with configurable weight
        vl_total = (vl['mse_b'] + vl['mse_a']) + es_alpha * (1.0 - vl['acc'])

        # Log gradient statistics
        avg_grad_norm = np.mean(grad_norms) if grad_norms else 0.0
        max_grad_norm = np.max(grad_norms) if grad_norms else 0.0

        print(f"[Epoch {ep:03d}] "
              f"Train: L={logs['all']:.4f} rec_b={logs['mse_b']:.4f} rec_a={logs['mse_a']:.4f} "
              f"smooth={logs['smooth']:.4f} mono={logs['mono']:.4f} liquid_w={logs['liquid_weight']:.4f} "
              f"maint_eff={logs['maintenance_effect']:.4f} cls={logs['cls']:.4f} | "
              f"Val: mse_b={vl['mse_b']:.4f} mse_a={vl['mse_a']:.4f} acc={vl['acc']:.3f} ΔHI_impr={vl['delta_mean']:.3f} "
              f"(ES metric={vl_total:.4f}) | Grad: avg={avg_grad_norm:.3f} max={max_grad_norm:.3f}")

        if vl_total < best_v:
            best_v = vl_total
            best_epoch = ep
            best_state = {k: v.clone() if hasattr(v, 'clone') else v for k, v in model.state_dict().items()}
            no_improve = 0
            print(f"    ✓ Saved new best model (ES metric: {best_v:.4f})")
        else:
            no_improve += 1
            print(f"    No improvement for {no_improve}/{patience} epochs")
            if no_improve >= patience:
                print(f"\n[Early Stopping] Patience reached at epoch {ep}.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"\n[Best Checkpoint] ES metric: {best_v:.4f} (Epoch {best_epoch})")
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        torch.save(model.state_dict(), save_path)
        print(f"[Model Saved] {save_path}")

        # Save metadata alongside model
        meta_path = save_path.replace('.pth', '_meta.json')
        meta_data = {
            "K": K,
            "id_to_name": id_to_name,
            "sensor_dim": C,
            "best_es": best_v,
            "best_epoch": best_epoch,
            "class_weights": class_weights.cpu().numpy().tolist() if class_weights is not None else None
        }
        import json
        with open(meta_path, 'w') as f:
            json.dump(meta_data, f, indent=2)
        print(f"[Metadata Saved] {meta_path}")

    if nan_batches > 0:
        print(f"[Warning] Total skipped {nan_batches} batches containing NaN/Inf during training.")

    meta = {"K": K, "id_to_name": id_to_name, "sensor_dim": C, "best_es": best_v, "best_epoch": best_epoch}
    return save_path, meta


def test_only(test_loader, ckpt_path, K=None, id_to_name=None, device=None,
              visualize=True, max_curve_keep=24):
    """
    Load a saved checkpoint and evaluate on the official test set.
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Peek one batch to get sensor dimension C (and fallback K if needed)
    first_batch = next(iter(test_loader))
    C = first_batch["x_before"].shape[-1]

    if K is None:
        # If K not provided, try to infer from labels present in test loader
        # (May undercount if some classes are absent. Prefer passing K from training.)
        print("[Info] K not provided; attempting to infer from test loader labels (may be lower than full K).")
        K = scan_num_classes_from_loader(test_loader)

    if id_to_name is None:
        id_to_name = build_default_id_to_name(K)

    # Build model and load weights
    model = DiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=K).to(device)
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state, strict=False)
    model.eval()

    print("\n" + "="*60)
    print("Final Test Set Evaluation")
    print("="*60)

    te = eval_epoch(model, test_loader, device, K, compute_acc=True)
    print(f"[Test] mse_b={te['mse_b']:.4f} mse_a={te['mse_a']:.4f} acc={te['acc']:.3f} ΔHI_impr={te['delta_mean']:.3f}")

    # Predictions & visualization data
    y_true, y_pred, all_delta_mean, all_uids, keep_curves = collect_test_predictions(
        model, test_loader, device, max_curve_keep=max_curve_keep, K=K, id_to_name=id_to_name
    )

    # Reports (only if labels present)
    if y_true.size > 0:
        print("\n[Classification Report]")
        labels_full = list(range(K))
        target_names = [id_to_name.get(i, f"Class_{i}") for i in labels_full]
        # --- FIX: force labels + names to avoid mismatch when some classes are absent in predictions ---
        print(classification_report(
            y_true, y_pred,
            labels=labels_full,
            target_names=target_names,
            zero_division=0
        ))

        print("\n[Confusion Matrix]")
        print(confusion_matrix(y_true, y_pred, labels=labels_full))

        df_delta = pd.DataFrame({
            "uid": all_uids[:len(y_true)],
            "true": y_true,
            "pred": y_pred,
            "delta_hi_mean": all_delta_mean[:len(y_true)]
        })
        print("\n[Learned ΔHI Statistics] By true class")
        print(df_delta.groupby("true")["delta_hi_mean"].describe())
        topk_by_delta(df_delta, id_to_name, k=3)

        if visualize:
            print("\n[Visualization] Learned health index curves with liquid weight adaptation...")
            plot_hi_examples_aligned(keep_curves, id_to_name, n_show=6, seed=0)

            print("\n[Visualization] Liquid operator weights evolution...")
            plot_liquid_weights(keep_curves, n_show=3, seed=0)

            print("\n[Visualization] Sensor reconstruction predictions...")
            plot_sensor_examples_aligned(keep_curves, sensor_idx_list=None, n_cols=4)
    else:
        print("\n[Info] No labels in test loader; skipping classification report and confusion matrix.")

    return {"test_metrics": te, "K": K, "id_to_name": id_to_name, "curves": keep_curves}


# ---------------------------------------------------------------------
# Example usage (replace with your real loaders and class weights)
# ---------------------------------------------------------------------
# Assuming you have computed class_weights from your data loader script:
# class_weights = your_computed_class_weights_tensor  # shape: (K,)

ckpt_path, meta = train_only(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=300,
    lr=1e-3,
    device=None,     # auto
    patience=20,
    use_matching_cost=True,
    save_path=str(resolve_data_file("best_model_NGFAID_250_100_2.pth")),
    class_weights=torch.as_tensor(class_weights, dtype=torch.float32)   # Pass your computed class weights here
)


In [ ]:

# -------- Infer sensor dimension C and number of classes K from the existing loaders --------
# Prefer train_loader when inferring K; fall back to test_loader if needed.
def _infer_C_and_K(train_loader=None, test_loader=None):
    # Inspect one batch to obtain C
    probe_loader = train_loader if (train_loader is not None) else test_loader
    assert probe_loader is not None, "An available train_loader or test_loader is required to infer the dimensions."
    first_batch = next(iter(probe_loader))
    C = first_batch["x_before"].shape[-1]

    K = None
    if train_loader is not None:
        try:
            K = scan_num_classes_from_loader(train_loader)
        except Exception:
            K = None
    if K is None and test_loader is not None:
        try:
            K = scan_num_classes_from_loader(test_loader)
        except Exception:
            K = None
    assert K is not None and K > 0, "Failed to infer K (number of classes) from the data. Please check the loader labels y."
    return C, K

# train_loader / test_loader should already be constructed outside this snippet
# If train_loader is unavailable, replace it with None below and infer using only test_loader.
C, K = _infer_C_and_K(train_loader=train_loader, test_loader=test_loader)
id_to_name = build_default_id_to_name(K)

print(f"[Info] Inferred sensor dim C={C}, num classes K={K}")

# -------- Backfill / rebuild metadata (optional) --------
# If you want to store the training-time class_weights in the metadata as well, provide them here (tensor / numpy / list are all acceptable).
# Demonstration: write class_weights_obj if it is available; otherwise keep None.
class_weights_obj = None
# For example, if you still have the class_weights array, you can use:
# import numpy as np
# class_weights_obj = np.asarray(class_weights, dtype=float)

def _serialize_class_weights(obj):
    if obj is None:
        return None
    try:
        import numpy as np
        if isinstance(obj, torch.Tensor):
            return obj.detach().cpu().numpy().tolist()
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, (list, tuple)):
            return list(obj)
        else:
            return None
    except Exception:
        return None

meta = {
    "K": int(K),
    "id_to_name": {int(k): str(v) for k, v in id_to_name.items()},
    "sensor_dim": int(C),
    "best_es": None,             # Leave blank or fill manually if the early-stopping metric from training is unknown
    "best_epoch": None,          # Same as above
    "class_weights": _serialize_class_weights(class_weights_obj)
}

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print(f"[Meta Saved] {meta_path}")

# -------- Load checkpoint weights and evaluate directly (no retraining required) --------
# Option A: directly reuse test_only from your code base (recommended)
results = test_only(
    test_loader=test_loader,
    ckpt_path=str(resolve_data_file("best_model_NGFAID_250_100_2.pth")),
    K=K,
    id_to_name=id_to_name,
    device=device,
    visualize=True,          # Set to True to generate plots; use False in headless environments
    max_curve_keep=24
)


## Part 3C: Monotonic HI Training Variant


In [ ]:


# ============================================================
# Monotonic-Decreasing HI + aligned plotting (before -> after)
# Adapted to external train/val/test loaders that yield:
#   batch = {
#       "x_before": (B, L, C), "x_after": (B, L, C),
#       "y": LongTensor[B] or None,
#       "meta": list[dict] (each may contain 'pair_id', 'matching_cost', ...)
#   }
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import os
import random
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix

# ---------------------------------------------------------------------
# Utilities for the NEW loaders
# ---------------------------------------------------------------------
def adapt_batch(batch, device):
    """
    Adapt a batch from your loaders to the tensors expected by the model.
    Ensures fixed-length mask/lengths and extracts optional meta.
    """
    xb = batch["x_before"].to(device)  # (B, L, C)
    xa = batch["x_after"].to(device)   # (B, L, C)

    B, L, _ = xb.shape
    mask = torch.ones(B, L, dtype=xb.dtype, device=device)
    lengths = torch.full((B,), L, dtype=torch.long, device=device)

    labels = batch.get("y", None)
    if labels is not None and isinstance(labels, torch.Tensor):
        # Filter out ignore_index labels
        valid_mask = labels != -100
        if valid_mask.any():
            labels = labels.to(device)
        else:
            labels = None
    elif labels is not None:
        labels = None

    meta = batch.get("meta", None)
    sample_ids = []
    matching_costs = torch.zeros(B, dtype=torch.float32, device=device)
    if isinstance(meta, (list, tuple)) and len(meta) == B:
        for i, m in enumerate(meta):
            if isinstance(m, dict):
                sample_ids.append(str(m.get("pair_id", f"sample_{i}")))
                if "matching_cost" in m and m["matching_cost"] is not None:
                    matching_costs[i] = float(m["matching_cost"])
            else:
                sample_ids.append(f"sample_{i}")
    else:
        sample_ids = [f"sample_{i}" for i in range(B)]

    return xb, xa, labels, mask, lengths, sample_ids, matching_costs


def scan_num_classes_from_loader(train_loader):
    """
    Determine K (number of classes) from the training loader.
    """
    max_class = -1
    for batch in train_loader:
        y = batch.get("y", None)
        if y is None:
            continue
        if isinstance(y, torch.Tensor):
            y_np = y.detach().cpu().numpy()
            valid_labels = y_np[y_np >= 0]  # filter out negative labels
            if len(valid_labels) > 0:
                max_class = max(max_class, int(valid_labels.max()))

    if max_class < 0:
        raise ValueError("No valid labels found in the training loader to determine K.")

    K = max_class + 1
    return K


def build_default_id_to_name(K):
    return {i: f"Class_{i}" for i in range(K)}


# ---------------------------------------------------------------------
# Core model components
# ---------------------------------------------------------------------
class BoltzmannKAN(nn.Module):
    def __init__(self, in_ch, out_ch=4):
        super().__init__()
        self.E  = nn.Parameter(torch.zeros(out_ch, in_ch))
        self.kT = nn.Parameter(torch.ones(out_ch, in_ch))
    def forward(self, x):
        B,T,C = x.shape
        kT = torch.clamp(F.softplus(self.kT), 0.01, 10.0).unsqueeze(0).unsqueeze(2)
        E  = torch.clamp(self.E, -10.0, 10.0).unsqueeze(0).unsqueeze(2)
        x_ = torch.clamp(x.unsqueeze(1), -10.0, 10.0)
        exp = torch.clamp((E - x_) / kT, -50, 50)
        w   = torch.sigmoid(exp)
        h   = (w * x_).sum(dim=3).permute(0, 2, 1)
        return torch.clamp(F.softplus(h), 0.0, 100.0)

class BaseOp(nn.Module):
    def __init__(self):
        super().__init__()
        self.param_modulator = None
    def set_param_modulator(self, modulator): self.param_modulator = modulator
    def get_modulated_params(self, context):
        if self.param_modulator is None: return {}
        return self.param_modulator(context)

class MonotonicLinearOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_bias = self.bias
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        bias  = torch.clamp(base_bias + mod.get('bias_offset', 0.0), -5.0, 5.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * (xm + bias)), 0.0, 100.0)

class MonotonicFlatOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 1e-3, 1.0
    def _cum(self, x):
        B, T = x.shape
        diff = F.relu(x[:, 1:] - x[:, :-1])
        zero_init = torch.zeros(B, 1, device=x.device, dtype=x.dtype)
        return torch.cat([zero_init, torch.cumsum(diff, dim=1)], dim=1)
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_bias = self.bias
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        bias  = torch.clamp(base_bias + mod.get('bias_offset', 0.0), -2.0, 2.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        cum_result = self._cum(xm)
        return torch.clamp(F.softplus(scale * (cum_result + bias)), 0.0, 100.0)

class ConcaveLogOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.eps = 1e-3
        self.smin, self.smax = 0.01, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        eps  = torch.clamp(self.eps + mod.get('eps_offset', 0.0), 1e-6, 1e-1)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * torch.log(torch.abs(xm) + eps)), 0.0, 100.0)

class SaturationSigmoidOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.raw_slope = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
        self.lmin, self.lmax = 0.1, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_slope = torch.clamp(F.softplus(self.raw_slope), self.lmin, self.lmax)
        base_bias  = self.bias
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        slope = torch.clamp(base_slope + mod.get('slope_offset', 0.0), self.lmin, self.lmax)
        bias  = torch.clamp(base_bias  + mod.get('bias_offset', 0.0), -3.0, 3.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * torch.sigmoid(slope * (xm - bias))), 0.0, 100.0)

class HingeReLUOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.threshold = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_threshold = self.threshold
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        threshold = torch.clamp(base_threshold + mod.get('threshold_offset', 0.0), -3.0, 3.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * F.relu(xm - threshold)), 0.0, 100.0)

class PolynomialOp(BaseOp):
    def __init__(self, deg=2):  # Reduced default degree for stability
        super().__init__()
        self.raw_coeff = nn.Parameter(torch.zeros(deg))
        self.deg = deg
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_coeff = torch.clamp(F.softplus(self.raw_coeff), 0.01, 5.0)
        xm = torch.clamp(torch.tanh(h.mean(-1)), -3.0, 3.0)  # Added tanh for stability
        coeff_offset = mod.get('coeff_offset', torch.zeros_like(base_coeff))
        if coeff_offset.dim() > 1:
            coeff_offset = coeff_offset.mean(0)
        coeff = torch.clamp(base_coeff + coeff_offset, 0.01, 5.0)
        y = torch.zeros_like(xm)
        for i in range(self.deg):
            y = y + coeff[i] * torch.clamp(xm ** (i + 1), -100.0, 100.0)
        return torch.clamp(F.softplus(y), 0.0, 100.0)

class DampedSinOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.raw_freq = nn.Parameter(torch.tensor(0.0))
        self.raw_lambda = nn.Parameter(torch.tensor(0.0))
        self.phase = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
        self.fmin, self.fmax = 0.1, 5.0
        self.lmin, self.lmax = 0.01, 3.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_freq  = torch.clamp(F.softplus(self.raw_freq), self.fmin, self.fmax)
        base_lam   = torch.clamp(F.softplus(self.raw_lambda), self.lmin, self.lmax)
        base_phase = torch.clamp(self.phase, -10.0, 10.0)
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        freq  = torch.clamp(base_freq  + mod.get('freq_offset', 0.0), self.fmin, self.fmax)
        lam   = torch.clamp(base_lam   + mod.get('lambda_offset', 0.0), self.lmin, self.lmax)
        phase = torch.clamp(base_phase + mod.get('phase_offset', 0.0), -10.0, 10.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        damp = 1 / (1 + lam * torch.abs(xm))
        return torch.clamp(F.softplus(scale * damp * (torch.sin(freq * xm + phase) + 1)), 0.0, 100.0)

class PiecewiseLinearOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_k1 = nn.Parameter(torch.tensor(0.0))
        self.raw_k2 = nn.Parameter(torch.tensor(0.0))
        self.threshold = nn.Parameter(torch.tensor(0.0))
        self.kmin, self.kmax = 0.01, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_k1 = torch.clamp(F.softplus(self.raw_k1), self.kmin, self.kmax)
        base_k2 = torch.clamp(F.softplus(self.raw_k2), self.kmin, self.kmax)
        base_thr = torch.clamp(self.threshold, -5.0, 5.0)
        k1 = torch.clamp(base_k1 + mod.get('k1_offset', 0.0), self.kmin, self.kmax)
        k2 = torch.clamp(base_k2 + mod.get('k2_offset', 0.0), self.kmin, self.kmax)
        thr= torch.clamp(base_thr + mod.get('threshold_offset', 0.0), -5.0, 5.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        left = k1 * xm
        right = k1 * thr + k2 * (xm - thr)
        out = torch.where(xm <= thr, left, right)
        return torch.clamp(F.softplus(out), 0.0, 100.0)

class ParameterModulator(nn.Module):
    def __init__(self, context_dim, param_configs):
        super().__init__()
        self.param_configs = param_configs
        self.param_predictors = nn.ModuleDict()
        for name, info in param_configs.items():
            dim = info.get('dim', 1)
            self.param_predictors[name] = nn.Sequential(
                nn.Linear(context_dim, 64), nn.ReLU(), nn.Dropout(0.1),
                nn.Linear(64, 32), nn.ReLU(),
                nn.Linear(32, dim), nn.Tanh()
            )
    def forward(self, context):
        global_context = context.mean(dim=1)  # (B, context_dim)
        modulated = {}
        for name, predictor in self.param_predictors.items():
            info = self.param_configs[name]
            raw_off = predictor(global_context)
            scale = info.get('scale', 0.1)
            modulated[name] = raw_off * scale
        return modulated

class LiquidWeightGenerator(nn.Module):
    def __init__(self, h_dim, x_dim, n_ops, hidden_dim=64, tau_min=1.0, tau_max=3.0):
        super().__init__()
        self.n_ops = n_ops
        self.tau_min, self.tau_max = tau_min, tau_max
        self.h_feature_net = nn.Sequential(nn.Linear(h_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(0.1))
        self.x_feature_net = nn.Sequential(nn.Linear(x_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(0.1))
        self.temporal_encoder = nn.Sequential(nn.Linear(3, hidden_dim // 4), nn.ReLU())
        combined_dim = hidden_dim + hidden_dim // 4
        self.feature_fusion = nn.Sequential(
            nn.Linear(combined_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.op_branches = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 4), nn.ReLU(), nn.Linear(hidden_dim // 4, 1))
            for _ in range(n_ops)
        ])
        self.temp_predictor = nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 4), nn.ReLU(), nn.Linear(hidden_dim // 4, 1))
        self.global_bias_scale = 0.01
        for branch in self.op_branches:
            nn.init.xavier_uniform_(branch[0].weight)
            nn.init.xavier_uniform_(branch[2].weight)

    def forward(self, h_multi, x):
        B, T, h_dim = h_multi.shape
        _, _, x_dim = x.shape
        h_features = self.h_feature_net(h_multi)
        x_features = self.x_feature_net(x)
        t_norm = torch.arange(T, device=x.device, dtype=x.dtype).unsqueeze(0).expand(B, -1) / max(T-1, 1)
        dt = torch.zeros_like(t_norm)
        dt[:, 1:] = t_norm[:, 1:] - t_norm[:, :-1]
        phase = torch.sin(2 * 3.14159 * t_norm)
        temporal_input = torch.stack([t_norm, dt, phase], dim=-1)
        temporal_features = self.temporal_encoder(temporal_input)
        combined = torch.cat([h_features, x_features, temporal_features], dim=-1)
        fused = self.feature_fusion(combined)
        raw_logits = []
        for branch in self.op_branches:
            raw_logits.append(branch(fused).squeeze(-1))
        raw_weights = torch.stack(raw_logits, dim=-1)                     # (B, T, K)

        # NaN protection
        raw_weights = torch.nan_to_num(raw_weights, nan=0.0, posinf=1e6, neginf=-1e6)
        raw_weights = raw_weights - raw_weights.mean(dim=-1, keepdim=True)

        if self.global_bias_scale > 0:
            global_bias = torch.randn(self.n_ops, device=raw_weights.device) * self.global_bias_scale
            raw_weights = raw_weights + global_bias.unsqueeze(0).unsqueeze(0)
        raw_temp = self.temp_predictor(fused).squeeze(-1)                 # (B, T)
        raw_temp = torch.nan_to_num(raw_temp, nan=0.0, posinf=1e6, neginf=-1e6)
        temperature = torch.clamp(F.softplus(raw_temp) + self.tau_min, self.tau_min, self.tau_max)
        weights = F.softmax(raw_weights / temperature.unsqueeze(-1), dim=-1)
        return weights, temperature

class CustomKAN(nn.Module):
    def __init__(self, ops, h_dim, x_dim):
        super().__init__()
        self.ops = nn.ModuleList(ops)
        self.n_ops = len(ops)
        self.weight_generator = LiquidWeightGenerator(h_dim=h_dim, x_dim=x_dim, n_ops=self.n_ops, hidden_dim=128)
        self.op_feature_extractors = nn.ModuleList([
            nn.Sequential(nn.Linear(h_dim, h_dim), nn.ReLU(), nn.Linear(h_dim, h_dim))
            for _ in range(self.n_ops)
        ])
        self.param_modulators = nn.ModuleList()
        context_dim = h_dim + x_dim
        for op in self.ops:
            param_configs = self._get_param_configs_for_op(op)
            if param_configs:
                modulator = ParameterModulator(context_dim, param_configs)
                self.param_modulators.append(modulator)
                op.set_param_modulator(modulator)
            else:
                self.param_modulators.append(None)
        self.raw_gain = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))

    def _get_param_configs_for_op(self, op):
        if isinstance(op, MonotonicLinearOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.2}, 'bias_offset': {'dim': 1, 'scale': 0.3}}
        if isinstance(op, MonotonicFlatOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.1}, 'bias_offset': {'dim': 1, 'scale': 0.2}}
        if isinstance(op, ConcaveLogOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.2}, 'eps_offset': {'dim': 1, 'scale': 0.01}}
        if isinstance(op, SaturationSigmoidOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.2}, 'slope_offset': {'dim': 1, 'scale': 0.3}, 'bias_offset': {'dim': 1, 'scale': 0.3}}
        if isinstance(op, HingeReLUOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.2}, 'threshold_offset': {'dim': 1, 'scale': 0.3}}
        if isinstance(op, PolynomialOp):
            return {'coeff_offset': {'dim': op.deg, 'scale': 0.2}}
        if isinstance(op, DampedSinOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.2}, 'freq_offset': {'dim': 1, 'scale': 0.3},
                    'lambda_offset': {'dim': 1, 'scale': 0.2}, 'phase_offset': {'dim': 1, 'scale': 0.5}}
        if isinstance(op, PiecewiseLinearOp):
            return {'k1_offset': {'dim': 1, 'scale': 0.2}, 'k2_offset': {'dim': 1, 'scale': 0.2},
                    'threshold_offset': {'dim': 1, 'scale': 0.3}}
        return {}

    def forward(self, h_multi, x):
        op_features = [ext(h_multi) for ext in self.op_feature_extractors]
        context = torch.cat([h_multi, x], dim=-1)
        outs = [op(op_features[i], context) for i, op in enumerate(self.ops)]
        Tm = min(o.size(1) for o in outs)
        outs = [o[:, :Tm] for o in outs]
        h_multi_aligned = h_multi[:, :Tm, :]
        x_aligned = x[:, :Tm, :]
        op_stack = torch.stack(outs, dim=-1)  # (B, Tm, K)
        weights, temperature = self.weight_generator(h_multi_aligned, x_aligned)
        damage = torch.sum(op_stack * weights, dim=-1)  # (B, Tm)
        damage = torch.clamp(damage, 0.0, 100.0)
        gain = torch.clamp(F.softplus(self.raw_gain), 0.1, 2.0)
        bias_val = torch.clamp(self.bias, -2.0, 2.0)
        damage = torch.clamp(gain * damage + bias_val, 0.0, 100.0)
        return damage, weights, temperature

class TrendEncoder(nn.Module):
    """
    Vectorized monotonic decreasing HI projection with learnable nonlinear decay.
    """
    def __init__(self, in_ch, trend_ch=4):
        super().__init__()
        self.boltz = BoltzmannKAN(in_ch, out_ch=trend_ch)
        ops = [MonotonicLinearOp(), MonotonicFlatOp(), ConcaveLogOp(),
               SaturationSigmoidOp(), HingeReLUOp(), PolynomialOp(),
               DampedSinOp(), PiecewiseLinearOp()]
        self.customkan = CustomKAN(ops, h_dim=trend_ch, x_dim=in_ch)

        self.proj_gain = nn.Parameter(torch.tensor(1.0))
        self.proj_bias = nn.Parameter(torch.tensor(0.0))

        self.health_aware_transform = nn.Sequential(
            nn.Linear(in_ch, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )

        # Learnable nonlinear decay rate predictor
        self.rate_head = nn.Sequential(
            nn.Linear(in_ch + 1, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )
        self.rate_scale = nn.Parameter(torch.tensor(0.1))  # Global decay intensity

        # Reduced linear enforcer influence
        self.linear_enforcer = nn.Sequential(
            nn.Linear(in_ch, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )

    def forward(self, x):  # x:(B,T,C)
        B, T, C = x.shape
        h_multi = self.boltz(x)                               # (B,T,trend_ch)
        damage, weights, temperature = self.customkan(h_multi, x)

        x_reshaped = x.view(-1, C)
        health_direct = self.health_aware_transform(x_reshaped).view(B, T)  # (B,T)

        t = torch.arange(T, device=x.device, dtype=x.dtype).unsqueeze(0).expand(B, -1)
        t_normalized = t / (T - 1 + 1e-6)

        # Learnable nonlinear decay instead of fixed linear decay
        time_feat = t_normalized.unsqueeze(-1)                 # (B,T,1)
        rate_in = torch.cat([x, time_feat], dim=-1)           # (B,T,C+1)
        raw_rate = self.rate_head(rate_in).squeeze(-1)         # (B,T)
        rate = F.softplus(raw_rate) * torch.clamp(self.rate_scale, 1e-3, 2.0)
        cum_rate = torch.cumsum(rate, dim=1)                   # (B,T)
        nonlinear_decay = torch.exp(-cum_rate)                 # Monotonic decreasing and flexible

        g = torch.clamp(F.softplus(self.proj_gain), 0.1, 5.0)
        b = torch.clamp(self.proj_bias, -5.0, 5.0)
        damage_normalized = torch.sigmoid(g*(damage + b))

        combined_health = health_direct * (1 - 0.3 * damage_normalized)

        # Reduced linear channel influence with time-dependent weighting
        lw = self.linear_enforcer(x_reshaped).view(B, T)
        linear_weight = 0.2 * lw * (1.0 - t_normalized)   # Early stages more linear, later stages more network-driven

        hi = linear_weight * nonlinear_decay + (1 - linear_weight) * combined_health

        # ---- Vectorized strictly non-increasing enforcement with epsilon step ----
        eps = 1e-3
        idx = torch.arange(T, device=hi.device, dtype=hi.dtype).unsqueeze(0)  # (1, T)
        hi_mon, _ = torch.cummin(hi + eps * idx, dim=1)
        hi = hi_mon - eps * idx
        # -------------------------------------------------------------------------

        return hi, h_multi, weights, temperature

class ReconHead(nn.Module):
    def __init__(self, C, hidden=128):
        super().__init__()
        self.gru = nn.GRU(input_size=C+1, hidden_size=hidden, batch_first=True)
        self.out = nn.Linear(hidden, C)
    def forward(self, x_in, h_in):
        B,T,C = x_in.shape
        h_in_clamped = torch.clamp(h_in, 0.0, 10.0)
        feat = torch.cat([x_in, h_in_clamped.unsqueeze(-1)], dim=-1)
        H,_ = self.gru(feat)
        y = self.out(H)
        return y

class MaintClassifier(nn.Module):
    def __init__(self, sensor_dim, hidden=64, n_classes=14):
        super().__init__()
        self.hi_mlp = nn.Sequential(
            nn.Linear(6, hidden), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden//2)
        )
        self.sensor_mlp = nn.Sequential(
            nn.Linear(sensor_dim * 2, hidden), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden//2)
        )
        self.final_classifier = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, n_classes)
        )

    def forward(self, h_b, h_a, x_b, x_a, mask):
        m = mask
        T = m.size(1)
        valid = m.sum(1, keepdim=True).clamp_min(1.0)

        db = h_a - h_b
        mean_d = (db*m).sum(1,keepdim=True)/valid
        var_d  = (((db-mean_d)*m)**2).sum(1,keepdim=True)/valid
        std_d  = (var_d + 1e-8).sqrt()
        d0     = (db[:, :1])
        dmax   = (db.masked_fill(m==0, -1e9)).max(dim=1, keepdim=True).values
        pos_ratio = ((db>0).float()*m).sum(1,keepdim=True)/valid

        t = torch.arange(T, device=h_b.device, dtype=h_b.dtype)
        t = (t - t.mean())/(t.std()+1e-6)
        def slope(x):
            num = (x*t).sum(1) - x.sum(1)*t.sum()/T
            den = (t**2).sum() - (t.sum()**2)/T + 1e-8
            return (num/den).unsqueeze(1)
        slope_diff = slope(h_a) - slope(h_b)

        hi_feat = torch.cat([mean_d, d0, dmax, std_d, slope_diff, pos_ratio], dim=1)
        hi_feat = torch.clamp(hi_feat, -10.0, 10.0)
        hi_features = self.hi_mlp(hi_feat)

        x_b_mean = (x_b * m.unsqueeze(-1)).sum(1) / valid
        x_a_mean = (x_a * m.unsqueeze(-1)).sum(1) / valid
        sensor_change = torch.cat([x_b_mean, x_a_mean], dim=1)
        sensor_features = self.sensor_mlp(sensor_change)

        combined = torch.cat([hi_features, sensor_features], dim=1)
        logits = self.final_classifier(combined)
        return logits

def sanitize_tensors(*tensors):
    result = []
    for t in tensors:
        if t is not None:
            t_clean = torch.nan_to_num(t, nan=0.0, posinf=1e6, neginf=-1e6)
            result.append(t_clean)
        else:
            result.append(t)
    return result[0] if len(result) == 1 else result

class DiffAwareReconstructor(nn.Module):
    """
    - Two encoders share parameters: encode x_before / x_after → h_b / h_a (monotonic decreasing HI)
    - Two reconstruction heads: one-step recursive reconstruction
    - Classification head: ΔHI + sensor change features
    """
    def __init__(self, in_ch, trend_ch=4, hidden=128, n_classes=14):
        super().__init__()
        self.encoder = TrendEncoder(in_ch, trend_ch)
        self.recon_b = ReconHead(in_ch, hidden)
        self.recon_a = ReconHead(in_ch, hidden)
        self.clf     = MaintClassifier(sensor_dim=in_ch, hidden=64, n_classes=n_classes)
        self.consistency_weight = nn.Parameter(torch.tensor(1.0))

    def forward(self, x_b, x_a, mask):
        h_b, h_multi_b, weights_b, temp_b = self.encoder(x_b)
        h_a, h_multi_a, weights_a, temp_a = self.encoder(x_a)

        h_b, h_a = sanitize_tensors(h_b, h_a)
        weights_b, weights_a = sanitize_tensors(weights_b, weights_a)

        xb_in, hb_in = x_b[:, :-2], h_b[:, :-2]
        xa_in, ha_in = x_a[:, :-2], h_a[:, :-2]
        yb_hat = self.recon_b(xb_in, hb_in)   # (B, L-2, C) ≈ x_b[:,1:L-1]
        ya_hat = self.recon_a(xa_in, ha_in)   # (B, L-2, C) ≈ x_a[:,1:L-1]

        yb_hat, ya_hat = sanitize_tensors(yb_hat, ya_hat)
        logits = self.clf(h_b, h_a, x_b, x_a, mask)
        logits = sanitize_tensors(logits)
        return yb_hat, ya_hat, h_b, h_a, logits, weights_b, weights_a, temp_b, temp_a

# ---------------------------------------------------------------------
# Losses / Regularizers
# ---------------------------------------------------------------------
def masked_mse(a, b, mask):
    diff = (a - b)**2
    mse  = (diff.mean(-1) * mask).sum() / (mask.sum() + 1e-6)
    return mse

def masked_mse_batchwise(a, b, mask):
    diff = (a - b)**2  # (B, T, C)
    mse_t = diff.mean(-1) * mask  # (B, T)
    denom = mask.sum(1).clamp_min(1.0)  # (B,)
    return mse_t.sum(1) / denom

def slope_loss(h, mask, kind="smooth"):
    if kind=="smooth":
        d2 = h[:,2:] - 2*h[:,1:-1] + h[:,:-2]
        m  = mask[:,2:]
        return (d2.abs() * m).sum() / (m.sum()+1e-6)
    elif kind=="mono_dec":
        dh = h[:,1:] - h[:,:-1]
        m  = mask[:,1:]
        return (F.relu(dh) * m).sum() / (m.sum()+1e-6)
    elif kind=="tv":
        # Total variation (first-order differences) - allows more curvature
        dh = h[:,1:] - h[:,:-1]
        m  = mask[:,1:]
        return (dh.abs() * m).sum() / (m.sum()+1e-6)
    else:
        return torch.tensor(0., device=h.device)

def enhanced_liquid_weight_regularization(weights_b, weights_a, mask,
                                          lambda_tv=0.05, lambda_ent=0.2,
                                          lambda_bal=0.1, lambda_div=0.1):
    total_loss = torch.tensor(0.0, device=weights_b.device)

    def tv_loss(weights, mask):
        if weights.size(1) <= 1:
            return torch.tensor(0.0, device=weights.device)
        diff = weights[:, 1:, :] - weights[:, :-1, :]
        mask_diff = mask[:, 1:]
        tv = (diff.abs().sum(-1) * mask_diff).sum() / (mask_diff.sum() + 1e-6)
        return tv

    tv_loss_b = tv_loss(weights_b, mask)
    tv_loss_a = tv_loss(weights_a, mask)
    total_loss += lambda_tv * (tv_loss_b + tv_loss_a)

    def entropy_loss(weights, mask):
        eps = 1e-8
        entropy = -(weights * torch.log(weights + eps)).sum(-1)
        target_entropy = np.log(weights.size(-1)) * 0.8
        entropy_penalty = F.relu(target_entropy - entropy)
        return (entropy_penalty * mask).sum() / (mask.sum() + 1e-6)

    ent_loss_b = entropy_loss(weights_b, mask)
    ent_loss_a = entropy_loss(weights_a, mask)
    total_loss += lambda_ent * (ent_loss_b + ent_loss_a)

    def balance_loss(weights, mask):
        valid_weights = weights * mask.unsqueeze(-1)  # (B, T, K)
        mean_usage = valid_weights.sum([0, 1]) / (mask.sum() + 1e-6)  # (K,)
        target_usage = 1.0 / weights.size(-1)
        balance = ((mean_usage - target_usage) ** 2).sum()
        return balance

    bal_loss_b = balance_loss(weights_b, mask)
    bal_loss_a = balance_loss(weights_a, mask)
    total_loss += lambda_bal * (bal_loss_b + bal_loss_a)

    def diversity_loss(weights, mask):
        B, T, K = weights.shape
        if T <= 1:
            return torch.tensor(0.0, device=weights.device)
        valid_weights = weights * mask.unsqueeze(-1)
        weights_flat = valid_weights.view(-1, K)
        weights_centered = weights_flat - weights_flat.mean(0, keepdim=True)
        cov = torch.mm(weights_centered.T, weights_centered) / (weights_flat.size(0) - 1 + 1e-6)
        mask_offdiag = torch.ones_like(cov) - torch.eye(K, device=cov.device)
        corr_penalty = (cov.abs() * mask_offdiag).sum()
        return corr_penalty

    div_loss_b = diversity_loss(weights_b, mask)
    div_loss_a = diversity_loss(weights_a, mask)
    total_loss += lambda_div * (div_loss_b + div_loss_a)
    return total_loss

def smoothness_enhancement_loss(h, mask, order=2):
    if order == 2:
        d2 = h[:,2:] - 2*h[:,1:-1] + h[:,:-2]
        m = mask[:,2:]
        return (d2.abs() * m).sum() / (m.sum() + 1e-6)
    elif order == 3:
        d3 = h[:,3:] - 3*h[:,2:-1] + 3*h[:,1:-2] - h[:,:-3]
        m = mask[:,3:]
        return (d3.abs() * m).sum() / (m.sum() + 1e-6)
    else:
        return torch.tensor(0., device=h.device)

def compute_sample_weights(matching_costs, alpha=0.5):
    return 1.0 / (1.0 + alpha * matching_costs)

def maintenance_effect_loss(h_b, h_a, mask, lambda_diff=1.0, lambda_smooth_after=0.5, lambda_level_constraint=3.0):
    """
    Encourage maintenance effect: after-maintenance HI should be different from before,
    after-maintenance HI should be smoother, and min(h_a) > max(h_b) for proper separation.
    """
    total_loss = torch.tensor(0.0, device=h_b.device)

    # 1. Encourage difference between before and after HI (improved stability)
    hi_diff = torch.abs(h_a - h_b)  # (B, T)
    hi_diff_clamped = torch.clamp(hi_diff, min=1e-3)  # Prevent extreme gradients
    diff_loss = F.softplus(-10.0 * hi_diff_clamped)  # Smooth penalty for small differences
    diff_loss = (diff_loss * mask).sum() / (mask.sum() + 1e-6)
    total_loss += lambda_diff * diff_loss

    # 2. Encourage after-maintenance HI to be smoother (less variation)
    if h_a.size(1) > 1:
        h_a_diff = torch.abs(h_a[:, 1:] - h_a[:, :-1])  # (B, T-1)
        mask_diff = mask[:, 1:]
        smooth_after_loss = (h_a_diff * mask_diff).sum() / (mask_diff.sum() + 1e-6)
        total_loss += lambda_smooth_after * smooth_after_loss

    # 3. Encourage min(h_a) > max(h_b) for proper level separation
    # Use masked operations to handle variable lengths
    h_b_masked = h_b.masked_fill(mask == 0, -1e9)  # Fill invalid positions with very low values
    h_a_masked = h_a.masked_fill(mask == 0, 1e9)   # Fill invalid positions with very high values

    h_b_max = h_b_masked.max(dim=1)[0]  # (B,) - max of each sequence
    h_a_min = h_a_masked.min(dim=1)[0]  # (B,) - min of each sequence

    # Encourage h_a_min > h_b_max with a margin
    margin = 0.1
    level_constraint = F.relu(h_b_max - h_a_min + margin)  # (B,)
    level_constraint_loss = level_constraint.mean()
    total_loss += lambda_level_constraint * level_constraint_loss
    margin_pt = 0.05
    pointwise_gap = (h_a - h_b) + (-margin_pt)
    pointwise_loss = F.relu(-pointwise_gap)      # Penalize cases where h_a < h_b + margin_pt
    pointwise_loss = (pointwise_loss * mask).sum() / (mask.sum() + 1e-6)

    total_loss += 1.0 * pointwise_loss   # Tunable coefficient: try values between 0.5 and 2.0
    return total_loss

# ---------------------------------------------------------------------
# Evaluation / Prediction
# ---------------------------------------------------------------------
@torch.no_grad()
def eval_epoch(model, loader, device, K, compute_acc=True):
    model.eval()
    total = {"mse_b":0.0, "mse_a":0.0, "acc":0.0, "delta_mean":0.0}
    n_batch=0
    n_acc = 0; n_tot=0
    for batch in loader:
        xb, xa, labels, mask, lengths, sample_ids, matching_costs = adapt_batch(batch, device)
        m_tgt  = (torch.arange(xb.size(1)-2, device=device)[None,:] < (lengths-2)[:,None]).float()
        yb_hat, ya_hat, h_b, h_a, logits, _, _, _, _ = model(xb, xa, mask)
        yb_hat, ya_hat, h_b, h_a, logits = sanitize_tensors(yb_hat, ya_hat, h_b, h_a, logits)

        # Ensure logits dimension matches expected K
        if logits.shape[-1] != K:
            print(f"Warning: Logits dim {logits.shape[-1]} != expected K={K}, adjusting...")
            if logits.shape[-1] < K:
                # Pad with zeros
                padding = torch.zeros(logits.shape[0], K - logits.shape[-1], device=logits.device)
                logits = torch.cat([logits, padding], dim=1)
            else:
                # Truncate
                logits = logits[:, :K]

        yb_tgt = xb[:,1:-1,:]
        ya_tgt = xa[:,1:-1,:]
        mse_b = masked_mse(yb_hat, yb_tgt, m_tgt)
        mse_a = masked_mse(ya_hat, ya_tgt, m_tgt)
        if compute_acc and labels is not None:
            # Filter out ignore_index labels
            valid_labels = labels != -100
            if valid_labels.any():
                pred = logits[valid_labels].argmax(dim=1)
                labels_valid = labels[valid_labels]
                n_acc += (pred == labels_valid).sum().item()
                n_tot += labels_valid.numel()
        valid = mask.sum(1, keepdim=True).clamp_min(1.0)
        delta_mean = ((h_a - h_b)*mask).sum(1,keepdim=True)/valid
        delta_mean = torch.nan_to_num(delta_mean, nan=0.0)
        total["delta_mean"] += delta_mean.mean().item()
        total["mse_b"] += mse_b.item()
        total["mse_a"] += mse_a.item()
        n_batch += 1
    for k in ["mse_b","mse_a","delta_mean"]:
        total[k] /= max(n_batch,1)
    total["acc"] = (n_acc / max(n_tot,1)) if n_tot>0 else 0.0
    return total

@torch.no_grad()
def collect_test_predictions(model, loader, device, max_curve_keep=24, K=14, id_to_name=None):
    model.eval()
    y_true, y_pred = [], []
    all_delta_mean, all_sample_ids = [], []
    keep_curves = []
    for batch in loader:
        xb, xa, labels, mask, lengths, sample_ids, matching_costs = adapt_batch(batch, device)
        yb_hat, ya_hat, h_b, h_a, logits, weights_b, weights_a, temp_b, temp_a = model(xb, xa, mask)
        yb_hat, ya_hat, h_b, h_a, logits = sanitize_tensors(yb_hat, ya_hat, h_b, h_a, logits)

        # Ensure logits dimension matches expected K
        if logits.shape[-1] != K:
            if logits.shape[-1] < K:
                padding = torch.zeros(logits.shape[0], K - logits.shape[-1], device=logits.device)
                logits = torch.cat([logits, padding], dim=1)
            else:
                logits = logits[:, :K]

        prob = F.softmax(logits, dim=1)     # (B,K)
        pred = prob.argmax(1)               # (B,)

        valid = mask.sum(1, keepdim=True).clamp_min(1.0)
        delta_mean = ((h_a - h_b) * mask).sum(1, keepdim=True) / valid  # (B,1)
        delta_mean = torch.nan_to_num(delta_mean, nan=0.0)

        if labels is not None:
            # Filter out ignore_index labels
            valid_labels = labels != -100
            if valid_labels.any():
                y_true.append(labels[valid_labels].detach().cpu().numpy())
                y_pred.append(pred[valid_labels].detach().cpu().numpy())
        all_delta_mean.append(delta_mean.squeeze(1).cpu().numpy())
        all_sample_ids.extend(sample_ids)

        for i in range(xb.size(0)):
            if len(keep_curves) >= max_curve_keep: break
            L_i = int(lengths[i].item())
            L_recon = min(L_i - 2, yb_hat.size(1))
            keep_curves.append({
                "sample_id": sample_ids[i],
                "true": int(labels[i].cpu().item()) if labels is not None and labels[i] != -100 else -1,
                "pred": int(pred[i].cpu().item()),
                "prob": prob[i].cpu().numpy(),
                "h_before": h_b[i, :L_i].cpu().numpy(),
                "h_after":  h_a[i, :L_i].cpu().numpy(),
                "weights_before": weights_b[i, :L_i].cpu().numpy() if weights_b is not None else None,
                "weights_after": weights_a[i, :L_i].cpu().numpy() if weights_a is not None else None,
                "temp_before": temp_b[i, :L_i].cpu().numpy() if temp_b is not None else None,
                "temp_after": temp_a[i, :L_i].cpu().numpy() if temp_a is not None else None,
                "op_outputs_before": [],
                "op_outputs_after":  [],
                "yb_hat":   yb_hat[i, :L_recon].cpu().numpy(),
                "ya_hat":   ya_hat[i, :L_recon].cpu().numpy(),
                "x_before": xb[i, :L_i].cpu().numpy(),
                "x_after":  xa[i, :L_i].cpu().numpy(),
            })
    y_true = np.concatenate(y_true, axis=0) if len(y_true)>0 else np.array([])
    y_pred = np.concatenate(y_pred, axis=0) if len(y_pred)>0 else np.array([])
    all_delta_mean = np.concatenate(all_delta_mean, axis=0) if len(all_delta_mean)>0 else np.array([])
    return y_true, y_pred, all_delta_mean, all_sample_ids, keep_curves

# ---------------------------------------------------------------------
# Plotting
# ---------------------------------------------------------------------
def remove_constant_segments(hi_values, threshold=1e-6):
    if len(hi_values) <= 1:
        return hi_values, np.arange(len(hi_values))
    diffs = np.abs(np.diff(hi_values))
    valid_mask = np.ones(len(hi_values), dtype=bool)
    valid_mask[1:] = diffs > threshold
    if np.sum(valid_mask) < 2:
        valid_mask[0] = True
        valid_mask[-1] = True
    valid_indices = np.where(valid_mask)[0]
    return hi_values[valid_indices], valid_indices

def plot_hi_examples_aligned(curves, id_to_name, n_show=6, seed=0):
    if len(curves)==0:
        print("(No visualization samples)")
        return
    random.Random(seed).shuffle(curves)
    n_show = min(n_show, len(curves))
    cols = 3
    rows = int(np.ceil(n_show/cols))
    plt.figure(figsize=(cols*4.6, rows*3.2))
    for k in range(n_show):
        ex = curves[k]
        hb = ex["h_before"]; ha = ex["h_after"]
        L  = min(len(hb), len(ha))
        hb, ha = hb[:L], ha[:L]
        hb_clean, hb_indices = remove_constant_segments(hb)
        ha_clean, ha_indices = remove_constant_segments(ha)
        t_b = hb_indices
        t_a = ha_indices + L

        sample_id = ex["sample_id"]
        pred = ex["pred"]; true = ex["true"]; prob = ex["prob"]
        true_name = id_to_name.get(true, f"Class_{true}")
        pred_name = id_to_name.get(pred, f"Class_{pred}")

        plt.subplot(rows, cols, k+1)
        if len(hb_clean) > 1:
            plt.plot(t_b, hb_clean, label="Learned HI (Pre)", linewidth=1.8, marker='o', markersize=3)
        if len(ha_clean) > 1:
            plt.plot(t_a, ha_clean, label="Learned HI (Post)",  linewidth=1.8, linestyle='--', marker='s', markersize=3)
        plt.axvline(L-1, color='k', linestyle=':', linewidth=1.0, alpha=0.7)
        d_mean = float(np.mean(ha - hb))
        d_max  = float(np.max(ha - hb))
        plt.title(f"ID={sample_id}\nTrue={true_name} | Pred={pred_name} "
                  f"| p={prob[pred]:.2f}\nΔHI_mean={d_mean:.3f}, ΔHI_max={d_max:.3f}", fontsize=9)
        plt.xlabel("Cycle (Pre | Post)")
        plt.ylabel("Health Index (higher=better)")
        plt.grid(ls='--', alpha=.35)
        if k==0: plt.legend()
    plt.tight_layout(); plt.show()

def plot_enhanced_liquid_weights(curves, n_show=3, seed=0):
    if len(curves) == 0:
        print("(No visualization samples)")
        return
    curves_with_weights = [c for c in curves if c.get("weights_before") is not None]
    if len(curves_with_weights) == 0:
        print("(No weight information available)")
        return
    random.Random(seed).shuffle(curves_with_weights)
    n_show = min(n_show, len(curves_with_weights))
    op_names = ["MonotonicLinear", "MonotonicFlat", "ConcaveLog", "SaturationSigmoid",
                "HingeReLU", "Polynomial", "DampedSin", "PiecewiseLinear"]
    for i in range(n_show):
        ex = curves_with_weights[i]
        weights_b = ex["weights_before"]
        weights_a = ex["weights_after"]
        temp_b = ex.get("temp_before")
        temp_a = ex.get("temp_after")
        sample_id = ex["sample_id"]
        true_label = ex["true"]
        pred_label = ex["pred"]
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle(f"Dynamic Parameter Modulation - ID={sample_id}, True=Class_{true_label}, Pred=Class_{pred_label}", fontsize=14)
        ax = axes[0, 0]
        T_b, K = weights_b.shape
        for k in range(K):
            ax.plot(range(T_b), weights_b[:, k], label=op_names[k] if k < len(op_names) else f"Op_{k}",
                   marker='o', markersize=2, linewidth=1.5)
        ax.set_title("Pre-maintenance Weights"); ax.set_xlabel("Time Step"); ax.set_ylabel("Weight")
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left'); ax.grid(True, alpha=0.3)
        ax = axes[0, 1]
        T_a, _ = weights_a.shape
        for k in range(K):
            ax.plot(range(T_a), weights_a[:, k], label=op_names[k] if k < len(op_names) else f"Op_{k}",
                   marker='s', markersize=2, linewidth=1.5, linestyle='--')
        ax.set_title("Post-maintenance Weights"); ax.set_xlabel("Time Step"); ax.set_ylabel("Weight")
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left'); ax.grid(True, alpha=0.3)
        ax = axes[0, 2]
        entropy_b = -np.sum(weights_b * np.log(weights_b + 1e-8), axis=1)
        entropy_a = -np.sum(weights_a * np.log(weights_a + 1e-8), axis=1)
        ax.plot(range(T_b), entropy_b, label="Pre", marker='o', linewidth=2)
        ax.plot(range(T_a), entropy_a, label="Post", marker='s', linewidth=2, linestyle='--')
        ax.axhline(np.log(K), color='r', linestyle=':', label=f'Max Entropy ({np.log(K):.2f})')
        ax.set_title("Weight Entropy Over Time"); ax.set_xlabel("Time Step"); ax.set_ylabel("Entropy")
        ax.legend(); ax.grid(True, alpha=0.3)
        ax = axes[1, 0]
        if temp_b is not None and temp_a is not None:
            ax.plot(range(T_b), temp_b, label="Pre", marker='o', linewidth=2)
            ax.plot(range(T_a), temp_a, label="Post", marker='s', linewidth=2, linestyle='--')
            ax.set_title("Temperature Evolution"); ax.set_xlabel("Time Step"); ax.set_ylabel("Temperature"); ax.legend()
        else:
            ax.text(0.5, 0.5, "Temperature data\nnot available", ha='center', va='center', transform=ax.transAxes)
            ax.set_title("Temperature Evolution")
        ax.grid(True, alpha=0.3)
        ax = axes[1, 1]
        ax.text(0.5, 0.5, "Operator output variance\n(not collected)", ha='center', va='center', transform=ax.transAxes)
        ax.set_title("Operator Output Variance"); ax.grid(True, alpha=0.3)
        ax = axes[1, 2]
        combined_weights = np.concatenate([weights_b, weights_a], axis=0)
        # Apply smoothing for correlation stability
        from scipy.ndimage import uniform_filter1d
        smoothed_weights = uniform_filter1d(combined_weights, size=3, axis=0)
        corr_matrix = np.corrcoef(smoothed_weights.T)
        im = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
        ax.set_title("Operator Weight Correlation (Smoothed)")
        ax.set_xticks(range(K)); ax.set_yticks(range(K))
        short_names = [op_names[k][:8] if k < len(op_names) else f"Op_{k}" for k in range(K)]
        ax.set_xticklabels(short_names, rotation=45); ax.set_yticklabels(short_names)
        cbar = plt.colorbar(im, ax=ax); cbar.set_label('Correlation')
        plt.tight_layout(); plt.show()

def plot_liquid_weights(curves, n_show=3, seed=0):
    plot_enhanced_liquid_weights(curves, n_show, seed)

def plot_sensor_examples_aligned(curves, sensor_idx_list=None, n_cols=4):
    if len(curves)==0:
        print("(No visualization samples)")
        return
    ex = curves[0]
    xb = ex["x_before"]; xa = ex["x_after"]; ya = ex["ya_hat"]
    L, C = xb.shape
    Lhat = ya.shape[0]
    if sensor_idx_list is None:
        step = max(1, C//8); sensor_idx_list = list(range(0, C, step))[:8]
    n = len(sensor_idx_list)
    n_rows = int(np.ceil(n/n_cols))
    plt.figure(figsize=(n_cols*4.3, n_rows*3.1))
    for i, s in enumerate(sensor_idx_list):
        plt.subplot(n_rows, n_cols, i+1)
        t_b = np.arange(L); t_a = np.arange(L, 2*L)
        plt.plot(t_b, xb[:,s], label="Original (pre)", linewidth=1.2)
        plt.plot(t_a, xa[:,s], label="Original (post)",  linewidth=1.0, linestyle="--", alpha=0.7)
        t_pred = np.arange(L+1, L+1+Lhat)
        plt.plot(t_pred, ya[:,s], label="Post prediction", linewidth=1.6)
        plt.axvline(L-1, color='k', linestyle=':', linewidth='1.0')
        plt.title(f"Sensor_{s:02d}")
        if i==0: plt.legend()
        plt.grid(ls="--", alpha=.35)
    plt.tight_layout(); plt.show()

def topk_by_delta(df_delta, id_to_name, k=3):
    if len(df_delta)==0:
        return
    unique_classes = sorted(df_delta["true"].unique())
    for cls in unique_classes[:5]:
        sub = df_delta[df_delta["true"]==cls].sort_values("delta_hi_mean", ascending=False).head(k)
        class_name = id_to_name.get(cls, f"Class_{cls}")
        print(f"\n[True={class_name}] Top {k} samples with highest ΔHI_mean:")
        print(sub[["uid","delta_hi_mean","pred"]].reset_index(drop=True))

# ---------------------------------------------------------------------
# Training (separate) and Testing (separate)
# ---------------------------------------------------------------------
def build_model_and_setup(train_loader, device, class_weights=None):
    """
    Build model and setup training components.

    Args:
        train_loader: Training data loader
        device: Device to use
        class_weights: Pre-computed class weights tensor (optional)
    """
    # Determine K from training loader
    K = scan_num_classes_from_loader(train_loader)
    id_to_name = build_default_id_to_name(K)

    first_batch = next(iter(train_loader))
    C = first_batch["x_before"].shape[-1]

    print(f"\n[Setup] Sensor dim C={C}, Num classes K={K}")

    # Use provided class weights or create uniform weights
    if class_weights is not None:
        if isinstance(class_weights, np.ndarray):
            class_weights = torch.from_numpy(class_weights).float().to(device)
        elif isinstance(class_weights, torch.Tensor):
            class_weights = class_weights.float().to(device)

        # Ensure class_weights has the right size
        if len(class_weights) < K:
            # Pad with ones if needed
            padded_weights = torch.ones(K, dtype=torch.float32, device=device)
            padded_weights[:len(class_weights)] = class_weights
            class_weights = padded_weights
        elif len(class_weights) > K:
            # Truncate if needed
            class_weights = class_weights[:K]

        print(f"[Class Weights] Using provided weights: {class_weights.detach().cpu().numpy()}")
    else:
        class_weights = torch.ones(K, dtype=torch.float32, device=device)
        print(f"[Class Weights] Using uniform weights: {class_weights.detach().cpu().numpy()}")

    model = DiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=K).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    ce  = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100, reduction='none')

    return model, opt, ce, K, id_to_name, C


def train_only(train_loader, val_loader, epochs=300, lr=1e-3, device=None, patience=20,
               use_matching_cost=True, save_path="best_model.pth", class_weights=None,
               es_alpha=1.0):
    """
    Train and save the best model. Returns path to checkpoint and metadata.

    Args:
        class_weights: Pre-computed class weights (numpy array or torch tensor)
        es_alpha: Weight for accuracy in early stopping metric
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Build model and training setup
    model, opt, ce, K, id_to_name, C = build_model_and_setup(train_loader, device, class_weights)
    # Override LR if provided
    for g in opt.param_groups:
        g['lr'] = lr

    best_v = float("inf")
    best_state = None
    best_epoch = 0
    no_improve = 0
    nan_batches = 0

    print(f"\nStarting training for {epochs} epochs with early stopping (patience={patience})...")
    print(f"Device: {device}")
    print(f"Early stopping alpha: {es_alpha}")

    for ep in range(1, epochs+1):
        print(f"\n[Epoch {ep}/{epochs}] Training...")
        model.train()
        logs = {"mse_b":0.0, "mse_a":0.0, "smooth":0.0, "mono":0.0,
                "cls":0.0, "smooth_enh":0.0, "liquid_weight":0.0, "maintenance_effect":0.0, "all":0.0}
        n_bt = 0
        batch_nan_count = 0

        progress = ep / epochs
        lambda_tv  = 0.03 * (1 - progress * 0.3)
        lambda_ent = 0.3  * (1 - progress * 0.7)
        lambda_bal = 0.15 * (1 - progress * 0.5)
        lambda_div = 0.2  * (1 - progress * 0.6)

        # Gradient norm tracking
        grad_norms = []

        for batch_idx, batch in enumerate(train_loader):
            xb, xa, labels, mask, lengths, sample_ids, matching_costs = adapt_batch(batch, device)
            m_tgt  = (torch.arange(xb.size(1)-2, device=device)[None,:] < (lengths-2)[:,None]).float()
            yb_hat, ya_hat, h_b, h_a, logits, weights_b, weights_a, temp_b, temp_a = model(xb, xa, mask)
            yb_hat, ya_hat, h_b, h_a, logits = sanitize_tensors(yb_hat, ya_hat, h_b, h_a, logits)
            weights_b, weights_a = sanitize_tensors(weights_b, weights_a)

            # Reconstruction targets
            yb_tgt = xb[:,1:-1,:]
            ya_tgt = xa[:,1:-1,:]

            # Per-sample reconstruction losses (for optional cost-weighting)
            rec_b_per = masked_mse_batchwise(yb_hat, yb_tgt, m_tgt)
            rec_a_per = masked_mse_batchwise(ya_hat, ya_tgt, m_tgt)
            rec_per = rec_b_per + rec_a_per

            # Classification loss (per-sample)
            if labels is None:
                cls_per = torch.zeros(xb.size(0), device=device)
            else:
                cls_per = ce(logits, labels)

            # Optional matching-cost weighting
            if use_matching_cost and matching_costs is not None:
                w_i = compute_sample_weights(matching_costs)
            else:
                w_i = torch.ones(xb.size(0), device=device)

            loss_rec = (rec_per * w_i).mean()
            loss_cls = (cls_per * w_i).mean()

            # HI regularizations - reduced smoothness penalties for more curvature
            loss_smooth = slope_loss(h_a, mask, "tv") + slope_loss(h_b, mask, "tv")  # Use TV instead of second-order
            loss_mono   = slope_loss(h_a, mask, "mono_dec") + slope_loss(h_b, mask, "mono_dec")
            loss_smooth_enhanced = (smoothness_enhancement_loss(h_b, mask, order=2) +
                                   smoothness_enhancement_loss(h_a, mask, order=2) +
                                   smoothness_enhancement_loss(h_b, mask, order=3) * 0.3 +
                                   smoothness_enhancement_loss(h_a, mask, order=3) * 0.3)
            loss_liquid_weight = enhanced_liquid_weight_regularization(
                weights_b, weights_a, mask,
                lambda_tv=lambda_tv, lambda_ent=lambda_ent,
                lambda_bal=lambda_bal, lambda_div=lambda_div
            )

            # Maintenance effect loss to encourage proper HI behavior
            loss_maintenance_effect = maintenance_effect_loss(
                h_b, h_a, mask,
                lambda_diff=1.0,           # Encourage difference between before/after
                lambda_smooth_after=0.5,   # Encourage after-maintenance HI to be smoother
                lambda_level_constraint=2.0 # Encourage min(h_a) > max(h_b)
            )

            # Total loss - reduced smoothness weights for more curvature
            w_rec=1.0; w_smooth=0.005; w_mono=0.05; w_cls=1.0; w_smooth_enh=0.05; w_liquid=0.5; w_maintenance=2.0
            loss = (w_rec*loss_rec + w_smooth*loss_smooth + w_mono*loss_mono +
                    w_cls*loss_cls + w_smooth_enh*loss_smooth_enhanced + w_liquid*loss_liquid_weight +
                    w_maintenance*loss_maintenance_effect)

            loss_rec, loss_cls, loss_smooth, loss_mono, loss_smooth_enhanced, loss_liquid_weight, loss_maintenance_effect, loss = sanitize_tensors(
                loss_rec, loss_cls, loss_smooth, loss_mono, loss_smooth_enhanced, loss_liquid_weight, loss_maintenance_effect, loss
            )

            if torch.isnan(loss) or torch.isinf(loss):
                batch_nan_count += 1
                if batch_nan_count == 1:
                    print(f"    [Warning] Epoch {ep}: Found NaN/Inf loss, skipping abnormal batch")
                continue

            opt.zero_grad()
            loss.backward()

            # Adaptive gradient clipping
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            grad_norms.append(grad_norm.item())

            opt.step()

            logs["mse_b"] += rec_b_per.mean().item()
            logs["mse_a"] += rec_a_per.mean().item()
            logs["smooth"]+= loss_smooth.item()
            logs["mono"]  += loss_mono.item()
            logs["cls"]   += loss_cls.item()
            logs["smooth_enh"]+= loss_smooth_enhanced.item()
            logs["liquid_weight"]+= loss_liquid_weight.item()
            logs["maintenance_effect"]+= loss_maintenance_effect.item()
            logs["all"]   += loss.item()
            n_bt += 1

        if batch_nan_count > 0:
            nan_batches += batch_nan_count

        for k in logs: logs[k] /= max(n_bt,1)

        print("    Validating...")
        vl = eval_epoch(model, val_loader, device, K, compute_acc=True)
        # Improved early-stopping metric with configurable weight
        vl_total = (vl['mse_b'] + vl['mse_a']) + es_alpha * (1.0 - vl['acc'])

        # Log gradient statistics
        avg_grad_norm = np.mean(grad_norms) if grad_norms else 0.0
        max_grad_norm = np.max(grad_norms) if grad_norms else 0.0

        print(f"[Epoch {ep:03d}] "
              f"Train: L={logs['all']:.4f} rec_b={logs['mse_b']:.4f} rec_a={logs['mse_a']:.4f} "
              f"smooth={logs['smooth']:.4f} mono={logs['mono']:.4f} liquid_w={logs['liquid_weight']:.4f} "
              f"maint_eff={logs['maintenance_effect']:.4f} cls={logs['cls']:.4f} | "
              f"Val: mse_b={vl['mse_b']:.4f} mse_a={vl['mse_a']:.4f} acc={vl['acc']:.3f} ΔHI_impr={vl['delta_mean']:.3f} "
              f"(ES metric={vl_total:.4f}) | Grad: avg={avg_grad_norm:.3f} max={max_grad_norm:.3f}")

        if vl_total < best_v:
            best_v = vl_total
            best_epoch = ep
            best_state = {k: v.clone() if hasattr(v, 'clone') else v for k, v in model.state_dict().items()}
            no_improve = 0
            print(f"    ✓ Saved new best model (ES metric: {best_v:.4f})")
        else:
            no_improve += 1
            print(f"    No improvement for {no_improve}/{patience} epochs")
            if no_improve >= patience:
                print(f"\n[Early Stopping] Patience reached at epoch {ep}.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"\n[Best Checkpoint] ES metric: {best_v:.4f} (Epoch {best_epoch})")
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        torch.save(model.state_dict(), save_path)
        print(f"[Model Saved] {save_path}")

        # Save metadata alongside model
        meta_path = save_path.replace('.pth', '_meta.json')
        meta_data = {
            "K": K,
            "id_to_name": id_to_name,
            "sensor_dim": C,
            "best_es": best_v,
            "best_epoch": best_epoch,
            "class_weights": class_weights.cpu().numpy().tolist() if class_weights is not None else None
        }
        import json
        with open(meta_path, 'w') as f:
            json.dump(meta_data, f, indent=2)
        print(f"[Metadata Saved] {meta_path}")

    if nan_batches > 0:
        print(f"[Warning] Total skipped {nan_batches} batches containing NaN/Inf during training.")

    meta = {"K": K, "id_to_name": id_to_name, "sensor_dim": C, "best_es": best_v, "best_epoch": best_epoch}
    return save_path, meta




# ---------------------------------------------------------------------
# Example usage (replace with your real loaders and class weights)
# ---------------------------------------------------------------------
# Assuming you have computed class_weights from your data loader script:
# class_weights = your_computed_class_weights_tensor  # shape: (K,)

ckpt_path, meta = train_only(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=1000,
    lr=1e-3,
    device=None,     # auto
    patience=20,
    use_matching_cost=True,
    save_path=str(resolve_data_file("best_model_NGFAID_250_100_3.pth")),
    class_weights=torch.as_tensor(class_weights, dtype=torch.float32)   # Pass your computed class weights here
)


In [ ]:

# -------- Infer sensor dimension C and number of classes K from the existing loaders --------
# Prefer train_loader when inferring K; fall back to test_loader if needed.
def _infer_C_and_K(train_loader=None, test_loader=None):
    # Inspect one batch to obtain C
    probe_loader = train_loader if (train_loader is not None) else test_loader
    assert probe_loader is not None, "An available train_loader or test_loader is required to infer the dimensions."
    first_batch = next(iter(probe_loader))
    C = first_batch["x_before"].shape[-1]

    K = None
    if train_loader is not None:
        try:
            K = scan_num_classes_from_loader(train_loader)
        except Exception:
            K = None
    if K is None and test_loader is not None:
        try:
            K = scan_num_classes_from_loader(test_loader)
        except Exception:
            K = None
    assert K is not None and K > 0, "Failed to infer K (number of classes) from the data. Please check the loader labels y."
    return C, K

# train_loader / test_loader should already be constructed outside this snippet
# If train_loader is unavailable, replace it with None below and infer using only test_loader.
C, K = _infer_C_and_K(train_loader=train_loader, test_loader=test_loader)
id_to_name = build_default_id_to_name(K)

print(f"[Info] Inferred sensor dim C={C}, num classes K={K}")

# -------- Backfill / rebuild metadata (optional) --------
# If you want to store the training-time class_weights in the metadata as well, provide them here (tensor / numpy / list are all acceptable).
# Demonstration: write class_weights_obj if it is available; otherwise keep None.
class_weights_obj = None
# For example, if you still have the class_weights array, you can use:
# import numpy as np
# class_weights_obj = np.asarray(class_weights, dtype=float)

def _serialize_class_weights(obj):
    if obj is None:
        return None
    try:
        import numpy as np
        if isinstance(obj, torch.Tensor):
            return obj.detach().cpu().numpy().tolist()
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, (list, tuple)):
            return list(obj)
        else:
            return None
    except Exception:
        return None

def test_only(test_loader, ckpt_path, K=None, id_to_name=None, device=None,
              visualize=True, max_curve_keep=24):
    """
    Load a saved checkpoint and evaluate on the official test set.
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Peek one batch to get sensor dimension C (and fallback K if needed)
    first_batch = next(iter(test_loader))
    C = first_batch["x_before"].shape[-1]

    if K is None:
        # If K not provided, try to infer from labels present in test loader
        # (May undercount if some classes are absent. Prefer passing K from training.)
        print("[Info] K not provided; attempting to infer from test loader labels (may be lower than full K).")
        K = scan_num_classes_from_loader(test_loader)

    if id_to_name is None:
        id_to_name = build_default_id_to_name(K)

    # Build model and load weights
    model = DiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=K).to(device)
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state, strict=False)
    model.eval()

    print("\n" + "="*60)
    print("Final Test Set Evaluation")
    print("="*60)

    te = eval_epoch(model, test_loader, device, K, compute_acc=True)
    print(f"[Test] mse_b={te['mse_b']:.4f} mse_a={te['mse_a']:.4f} acc={te['acc']:.3f} ΔHI_impr={te['delta_mean']:.3f}")

    # Predictions & visualization data
    y_true, y_pred, all_delta_mean, all_uids, keep_curves = collect_test_predictions(
        model, test_loader, device, max_curve_keep=max_curve_keep, K=K, id_to_name=id_to_name
    )

    # Reports (only if labels present)
    if y_true.size > 0:
        print("\n[Classification Report]")
        labels_full = list(range(K))
        target_names = [id_to_name.get(i, f"Class_{i}") for i in labels_full]
        # --- FIX: force labels + names to avoid mismatch when some classes are absent in predictions ---
        print(classification_report(
            y_true, y_pred,
            labels=labels_full,
            target_names=target_names,
            zero_division=0
        ))

        print("\n[Confusion Matrix]")
        print(confusion_matrix(y_true, y_pred, labels=labels_full))

        df_delta = pd.DataFrame({
            "uid": all_uids[:len(y_true)],
            "true": y_true,
            "pred": y_pred,
            "delta_hi_mean": all_delta_mean[:len(y_true)]
        })
        print("\n[Learned ΔHI Statistics] By true class")
        print(df_delta.groupby("true")["delta_hi_mean"].describe())
        topk_by_delta(df_delta, id_to_name, k=3)

        if visualize:
            print("\n[Visualization] Learned health index curves with liquid weight adaptation...")
            plot_hi_examples_aligned(keep_curves, id_to_name, n_show=16, seed=0)

            print("\n[Visualization] Liquid operator weights evolution...")
            plot_liquid_weights(keep_curves, n_show=16, seed=0)

            print("\n[Visualization] Sensor reconstruction predictions...")
            plot_sensor_examples_aligned(keep_curves, sensor_idx_list=None, n_cols=4)
    else:
        print("\n[Info] No labels in test loader; skipping classification report and confusion matrix.")

    return {"test_metrics": te, "K": K, "id_to_name": id_to_name, "curves": keep_curves}
meta = {
    "K": int(K),
    "id_to_name": {int(k): str(v) for k, v in id_to_name.items()},
    "sensor_dim": int(C),
    "best_es": None,             # Leave blank or fill manually if the early-stopping metric from training is unknown
    "best_epoch": None,          # Same as above
    "class_weights": _serialize_class_weights(class_weights_obj)
}

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print(f"[Meta Saved] {meta_path}")

# -------- Load checkpoint weights and evaluate directly (no retraining required) --------
# Option A: directly reuse test_only from your code base (recommended)
results = test_only(
    test_loader=test_loader,
    ckpt_path=str(resolve_data_file("best_model_NGFAID_250_100_3.pth")),
    K=K,
    id_to_name=id_to_name,
    device=device,
    visualize=True,          # Set to True to generate plots; use False in headless environments
    max_curve_keep=24
)


## Part 3D: Monotonic HI Training With Stability Analysis


In [ ]:


# ============================================================
# Monotonic-Decreasing HI + aligned plotting (before -> after)
# Adapted to external train/val/test loaders that yield:
#   batch = {
#       "x_before": (B, L, C), "x_after": (B, L, C),
#       "y": LongTensor[B] or None,
#       "meta": list[dict] (each may contain 'pair_id', 'matching_cost', ...)
#   }
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import os
import random
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix

# ---------------------------------------------------------------------
# Utilities for the NEW loaders
# ---------------------------------------------------------------------
def adapt_batch(batch, device):
    """
    Adapt a batch from your loaders to the tensors expected by the model.
    Ensures fixed-length mask/lengths and extracts optional meta.
    """
    xb = batch["x_before"].to(device)  # (B, L, C)
    xa = batch["x_after"].to(device)   # (B, L, C)

    B, L, _ = xb.shape
    mask = torch.ones(B, L, dtype=xb.dtype, device=device)
    lengths = torch.full((B,), L, dtype=torch.long, device=device)

    labels = batch.get("y", None)
    if labels is not None and isinstance(labels, torch.Tensor):
        # Filter out ignore_index labels
        valid_mask = labels != -100
        if valid_mask.any():
            labels = labels.to(device)
        else:
            labels = None
    elif labels is not None:
        labels = None

    meta = batch.get("meta", None)
    sample_ids = []
    matching_costs = torch.zeros(B, dtype=torch.float32, device=device)
    if isinstance(meta, (list, tuple)) and len(meta) == B:
        for i, m in enumerate(meta):
            if isinstance(m, dict):
                sample_ids.append(str(m.get("pair_id", f"sample_{i}")))
                if "matching_cost" in m and m["matching_cost"] is not None:
                    matching_costs[i] = float(m["matching_cost"])
            else:
                sample_ids.append(f"sample_{i}")
    else:
        sample_ids = [f"sample_{i}" for i in range(B)]

    return xb, xa, labels, mask, lengths, sample_ids, matching_costs


def scan_num_classes_from_loader(train_loader):
    """
    Determine K (number of classes) from the training loader.
    """
    max_class = -1
    for batch in train_loader:
        y = batch.get("y", None)
        if y is None:
            continue
        if isinstance(y, torch.Tensor):
            y_np = y.detach().cpu().numpy()
            valid_labels = y_np[y_np >= 0]  # filter out negative labels
            if len(valid_labels) > 0:
                max_class = max(max_class, int(valid_labels.max()))

    if max_class < 0:
        raise ValueError("No valid labels found in the training loader to determine K.")

    K = max_class + 1
    return K


def build_default_id_to_name(K):
    return {i: f"Class_{i}" for i in range(K)}


# ---------------------------------------------------------------------
# Core model components
# ---------------------------------------------------------------------
class BoltzmannKAN(nn.Module):
    def __init__(self, in_ch, out_ch=4):
        super().__init__()
        self.E  = nn.Parameter(torch.zeros(out_ch, in_ch))
        self.kT = nn.Parameter(torch.ones(out_ch, in_ch))
    def forward(self, x):
        B,T,C = x.shape
        kT = torch.clamp(F.softplus(self.kT), 0.01, 10.0).unsqueeze(0).unsqueeze(2)
        E  = torch.clamp(self.E, -10.0, 10.0).unsqueeze(0).unsqueeze(2)
        x_ = torch.clamp(x.unsqueeze(1), -10.0, 10.0)
        exp = torch.clamp((E - x_) / kT, -50, 50)
        w   = torch.sigmoid(exp)
        h   = (w * x_).sum(dim=3).permute(0, 2, 1)
        return torch.clamp(F.softplus(h), 0.0, 100.0)

class BaseOp(nn.Module):
    def __init__(self):
        super().__init__()
        self.param_modulator = None
    def set_param_modulator(self, modulator): self.param_modulator = modulator
    def get_modulated_params(self, context):
        if self.param_modulator is None: return {}
        return self.param_modulator(context)

class MonotonicLinearOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_bias = self.bias
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        bias  = torch.clamp(base_bias + mod.get('bias_offset', 0.0), -5.0, 5.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * (xm + bias)), 0.0, 100.0)

class MonotonicFlatOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 1e-3, 1.0
    def _cum(self, x):
        B, T = x.shape
        diff = F.relu(x[:, 1:] - x[:, :-1])
        zero_init = torch.zeros(B, 1, device=x.device, dtype=x.dtype)
        return torch.cat([zero_init, torch.cumsum(diff, dim=1)], dim=1)
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_bias = self.bias
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        bias  = torch.clamp(base_bias + mod.get('bias_offset', 0.0), -2.0, 2.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        cum_result = self._cum(xm)
        return torch.clamp(F.softplus(scale * (cum_result + bias)), 0.0, 100.0)

class ConcaveLogOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.eps = 1e-3
        self.smin, self.smax = 0.01, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        eps  = torch.clamp(self.eps + mod.get('eps_offset', 0.0), 1e-6, 1e-1)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * torch.log(torch.abs(xm) + eps)), 0.0, 100.0)

class SaturationSigmoidOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.raw_slope = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
        self.lmin, self.lmax = 0.1, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_slope = torch.clamp(F.softplus(self.raw_slope), self.lmin, self.lmax)
        base_bias  = self.bias
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        slope = torch.clamp(base_slope + mod.get('slope_offset', 0.0), self.lmin, self.lmax)
        bias  = torch.clamp(base_bias  + mod.get('bias_offset', 0.0), -3.0, 3.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * torch.sigmoid(slope * (xm - bias))), 0.0, 100.0)

class HingeReLUOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.threshold = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_threshold = self.threshold
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        threshold = torch.clamp(base_threshold + mod.get('threshold_offset', 0.0), -3.0, 3.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * F.relu(xm - threshold)), 0.0, 100.0)

class PolynomialOp(BaseOp):
    def __init__(self, deg=2):  # Reduced default degree for stability
        super().__init__()
        self.raw_coeff = nn.Parameter(torch.zeros(deg))
        self.deg = deg
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_coeff = torch.clamp(F.softplus(self.raw_coeff), 0.01, 5.0)
        xm = torch.clamp(torch.tanh(h.mean(-1)), -3.0, 3.0)  # Added tanh for stability
        coeff_offset = mod.get('coeff_offset', torch.zeros_like(base_coeff))
        if coeff_offset.dim() > 1:
            coeff_offset = coeff_offset.mean(0)
        coeff = torch.clamp(base_coeff + coeff_offset, 0.01, 5.0)
        y = torch.zeros_like(xm)
        for i in range(self.deg):
            y = y + coeff[i] * torch.clamp(xm ** (i + 1), -100.0, 100.0)
        return torch.clamp(F.softplus(y), 0.0, 100.0)

class DampedSinOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.raw_freq = nn.Parameter(torch.tensor(0.0))
        self.raw_lambda = nn.Parameter(torch.tensor(0.0))
        self.phase = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
        self.fmin, self.fmax = 0.1, 5.0
        self.lmin, self.lmax = 0.01, 3.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_freq  = torch.clamp(F.softplus(self.raw_freq), self.fmin, self.fmax)
        base_lam   = torch.clamp(F.softplus(self.raw_lambda), self.lmin, self.lmax)
        base_phase = torch.clamp(self.phase, -10.0, 10.0)
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        freq  = torch.clamp(base_freq  + mod.get('freq_offset', 0.0), self.fmin, self.fmax)
        lam   = torch.clamp(base_lam   + mod.get('lambda_offset', 0.0), self.lmin, self.lmax)
        phase = torch.clamp(base_phase + mod.get('phase_offset', 0.0), -10.0, 10.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        damp = 1 / (1 + lam * torch.abs(xm))
        return torch.clamp(F.softplus(scale * damp * (torch.sin(freq * xm + phase) + 1)), 0.0, 100.0)

class PiecewiseLinearOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_k1 = nn.Parameter(torch.tensor(0.0))
        self.raw_k2 = nn.Parameter(torch.tensor(0.0))
        self.threshold = nn.Parameter(torch.tensor(0.0))
        self.kmin, self.kmax = 0.01, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_k1 = torch.clamp(F.softplus(self.raw_k1), self.kmin, self.kmax)
        base_k2 = torch.clamp(F.softplus(self.raw_k2), self.kmin, self.kmax)
        base_thr = torch.clamp(self.threshold, -5.0, 5.0)
        k1 = torch.clamp(base_k1 + mod.get('k1_offset', 0.0), self.kmin, self.kmax)
        k2 = torch.clamp(base_k2 + mod.get('k2_offset', 0.0), self.kmin, self.kmax)
        thr= torch.clamp(base_thr + mod.get('threshold_offset', 0.0), -5.0, 5.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        left = k1 * xm
        right = k1 * thr + k2 * (xm - thr)
        out = torch.where(xm <= thr, left, right)
        return torch.clamp(F.softplus(out), 0.0, 100.0)

class ParameterModulator(nn.Module):
    def __init__(self, context_dim, param_configs):
        super().__init__()
        self.param_configs = param_configs
        self.param_predictors = nn.ModuleDict()
        for name, info in param_configs.items():
            dim = info.get('dim', 1)
            self.param_predictors[name] = nn.Sequential(
                nn.Linear(context_dim, 64), nn.ReLU(), nn.Dropout(0.1),
                nn.Linear(64, 32), nn.ReLU(),
                nn.Linear(32, dim), nn.Tanh()
            )
    def forward(self, context):
        global_context = context.mean(dim=1)  # (B, context_dim)
        modulated = {}
        for name, predictor in self.param_predictors.items():
            info = self.param_configs[name]
            raw_off = predictor(global_context)
            scale = info.get('scale', 0.1)
            modulated[name] = raw_off * scale
        return modulated

class LiquidWeightGenerator(nn.Module):
    def __init__(self, h_dim, x_dim, n_ops, hidden_dim=64, tau_min=1.0, tau_max=3.0):
        super().__init__()
        self.n_ops = n_ops
        self.tau_min, self.tau_max = tau_min, tau_max
        self.h_feature_net = nn.Sequential(nn.Linear(h_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(0.1))
        self.x_feature_net = nn.Sequential(nn.Linear(x_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(0.1))
        self.temporal_encoder = nn.Sequential(nn.Linear(3, hidden_dim // 4), nn.ReLU())
        combined_dim = hidden_dim + hidden_dim // 4
        self.feature_fusion = nn.Sequential(
            nn.Linear(combined_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.op_branches = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 4), nn.ReLU(), nn.Linear(hidden_dim // 4, 1))
            for _ in range(n_ops)
        ])
        self.temp_predictor = nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 4), nn.ReLU(), nn.Linear(hidden_dim // 4, 1))
        self.global_bias_scale = 0.01
        for branch in self.op_branches:
            nn.init.xavier_uniform_(branch[0].weight)
            nn.init.xavier_uniform_(branch[2].weight)

    def forward(self, h_multi, x, is_after=False):
        B, T, h_dim = h_multi.shape
        _, _, x_dim = x.shape
        h_features = self.h_feature_net(h_multi)
        x_features = self.x_feature_net(x)
        t_norm = torch.arange(T, device=x.device, dtype=x.dtype).unsqueeze(0).expand(B, -1) / max(T-1, 1)
        dt = torch.zeros_like(t_norm)
        dt[:, 1:] = t_norm[:, 1:] - t_norm[:, :-1]
        phase = torch.sin(2 * 3.14159 * t_norm)
        temporal_input = torch.stack([t_norm, dt, phase], dim=-1)
        temporal_features = self.temporal_encoder(temporal_input)
        combined = torch.cat([h_features, x_features, temporal_features], dim=-1)
        fused = self.feature_fusion(combined)
        raw_logits = []
        for branch in self.op_branches:
            raw_logits.append(branch(fused).squeeze(-1))
        raw_weights = torch.stack(raw_logits, dim=-1)                     # (B, T, K)

        # NaN protection
        raw_weights = torch.nan_to_num(raw_weights, nan=0.0, posinf=1e6, neginf=-1e6)
        raw_weights = raw_weights - raw_weights.mean(dim=-1, keepdim=True)

        if self.global_bias_scale > 0:
            global_bias = torch.randn(self.n_ops, device=raw_weights.device) * self.global_bias_scale
            raw_weights = raw_weights + global_bias.unsqueeze(0).unsqueeze(0)
        raw_temp = self.temp_predictor(fused).squeeze(-1)                 # (B, T)
        raw_temp = torch.nan_to_num(raw_temp, nan=0.0, posinf=1e6, neginf=-1e6)

        # Adaptive temperature for after-maintenance: slightly higher in later stages for smoother mixing
        base_temp = torch.clamp(F.softplus(raw_temp) + self.tau_min, self.tau_min, self.tau_max)
        if is_after:
            # Increase temperature in later stages for smoother operator mixing
            tau = int(0.3 * T)  # 30% cutoff
            temp_boost = torch.zeros_like(base_temp)
            if T > tau:
                late_stage_mask = torch.arange(T, device=base_temp.device) >= tau
                temp_boost[:, late_stage_mask] = 0.5 * (torch.arange(T - tau, device=base_temp.device).float() / (T - tau))
            temperature = base_temp + temp_boost
        else:
            temperature = base_temp

        weights = F.softmax(raw_weights / temperature.unsqueeze(-1), dim=-1)
        return weights, temperature

class CustomKAN(nn.Module):
    def __init__(self, ops, h_dim, x_dim):
        super().__init__()
        self.ops = nn.ModuleList(ops)
        self.n_ops = len(ops)
        self.weight_generator = LiquidWeightGenerator(h_dim=h_dim, x_dim=x_dim, n_ops=self.n_ops, hidden_dim=128)
        self.op_feature_extractors = nn.ModuleList([
            nn.Sequential(nn.Linear(h_dim, h_dim), nn.ReLU(), nn.Linear(h_dim, h_dim))
            for _ in range(self.n_ops)
        ])
        self.param_modulators = nn.ModuleList()
        context_dim = h_dim + x_dim
        for op in self.ops:
            param_configs = self._get_param_configs_for_op(op)
            if param_configs:
                modulator = ParameterModulator(context_dim, param_configs)
                self.param_modulators.append(modulator)
                op.set_param_modulator(modulator)
            else:
                self.param_modulators.append(None)
        self.raw_gain = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))

    def _get_param_configs_for_op(self, op):
        if isinstance(op, MonotonicLinearOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.2}, 'bias_offset': {'dim': 1, 'scale': 0.3}}
        if isinstance(op, MonotonicFlatOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.1}, 'bias_offset': {'dim': 1, 'scale': 0.2}}
        if isinstance(op, ConcaveLogOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.2}, 'eps_offset': {'dim': 1, 'scale': 0.01}}
        if isinstance(op, SaturationSigmoidOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.2}, 'slope_offset': {'dim': 1, 'scale': 0.3}, 'bias_offset': {'dim': 1, 'scale': 0.3}}
        if isinstance(op, HingeReLUOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.2}, 'threshold_offset': {'dim': 1, 'scale': 0.3}}
        if isinstance(op, PolynomialOp):
            return {'coeff_offset': {'dim': op.deg, 'scale': 0.2}}
        if isinstance(op, DampedSinOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.2}, 'freq_offset': {'dim': 1, 'scale': 0.3},
                    'lambda_offset': {'dim': 1, 'scale': 0.2}, 'phase_offset': {'dim': 1, 'scale': 0.5}}
        if isinstance(op, PiecewiseLinearOp):
            return {'k1_offset': {'dim': 1, 'scale': 0.2}, 'k2_offset': {'dim': 1, 'scale': 0.2},
                    'threshold_offset': {'dim': 1, 'scale': 0.3}}
        return {}

    def forward(self, h_multi, x, is_after=False):
        op_features = [ext(h_multi) for ext in self.op_feature_extractors]
        context = torch.cat([h_multi, x], dim=-1)
        outs = [op(op_features[i], context) for i, op in enumerate(self.ops)]
        Tm = min(o.size(1) for o in outs)
        outs = [o[:, :Tm] for o in outs]
        h_multi_aligned = h_multi[:, :Tm, :]
        x_aligned = x[:, :Tm, :]
        op_stack = torch.stack(outs, dim=-1)  # (B, Tm, K)
        weights, temperature = self.weight_generator(h_multi_aligned, x_aligned, is_after=is_after)
        damage = torch.sum(op_stack * weights, dim=-1)  # (B, Tm)
        damage = torch.clamp(damage, 0.0, 100.0)
        gain = torch.clamp(F.softplus(self.raw_gain), 0.1, 2.0)
        bias_val = torch.clamp(self.bias, -2.0, 2.0)
        damage = torch.clamp(gain * damage + bias_val, 0.0, 100.0)
        return damage, weights, temperature

class TrendEncoder(nn.Module):
    """
    Vectorized monotonic decreasing HI projection with learnable nonlinear decay.
    Enhanced with plateau-specific regularization for after-maintenance sequences.
    """
    def __init__(self, in_ch, trend_ch=4):
        super().__init__()
        self.boltz = BoltzmannKAN(in_ch, out_ch=trend_ch)
        ops = [MonotonicLinearOp(), MonotonicFlatOp(), ConcaveLogOp(),
               SaturationSigmoidOp(), HingeReLUOp(), PolynomialOp(),
               DampedSinOp(), PiecewiseLinearOp()]
        self.customkan = CustomKAN(ops, h_dim=trend_ch, x_dim=in_ch)

        self.proj_gain = nn.Parameter(torch.tensor(1.0))
        self.proj_bias = nn.Parameter(torch.tensor(0.0))

        self.health_aware_transform = nn.Sequential(
            nn.Linear(in_ch, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )

        # Learnable nonlinear decay rate predictor
        self.rate_head = nn.Sequential(
            nn.Linear(in_ch + 1, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )
        self.rate_scale = nn.Parameter(torch.tensor(0.1))  # Global decay intensity

        # Reduced linear enforcer influence
        self.linear_enforcer = nn.Sequential(
            nn.Linear(in_ch, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )

        # Steady-state level predictor for plateau targeting
        self.steady_state_head = nn.Sequential(
            nn.Linear(in_ch, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Sigmoid()
        )

    def forward(self, x, is_after=False):  # x:(B,T,C)
        B, T, C = x.shape
        h_multi = self.boltz(x)                               # (B,T,trend_ch)
        damage, weights, temperature = self.customkan(h_multi, x, is_after=is_after)

        x_reshaped = x.view(-1, C)
        health_direct = self.health_aware_transform(x_reshaped).view(B, T)  # (B,T)

        t = torch.arange(T, device=x.device, dtype=x.dtype).unsqueeze(0).expand(B, -1)
        t_normalized = t / (T - 1 + 1e-6)

        # Learnable nonlinear decay with plateau-aware rate control
        time_feat = t_normalized.unsqueeze(-1)                 # (B,T,1)
        rate_in = torch.cat([x, time_feat], dim=-1)           # (B,T,C+1)
        raw_rate = self.rate_head(rate_in).squeeze(-1)         # (B,T)
        rate = F.softplus(raw_rate) * torch.clamp(self.rate_scale, 1e-3, 2.0)

        # For after-maintenance, encourage rate to approach zero in later stages
        if is_after:
            tau = int(0.3 * T)  # 30% cutoff for plateau region
            if T > tau:
                late_stage_penalty = torch.zeros_like(rate)
                late_indices = torch.arange(T, device=rate.device) >= tau
                late_stage_penalty[:, late_indices] = rate[:, late_indices] * 2.0  # Penalty for high rates in plateau region
                rate = rate - late_stage_penalty
                rate = torch.clamp(rate, 1e-6, None)  # Ensure positive rates

        cum_rate = torch.cumsum(rate, dim=1)                   # (B,T)
        nonlinear_decay = torch.exp(-cum_rate)                 # Monotonic decreasing and flexible

        g = torch.clamp(F.softplus(self.proj_gain), 0.1, 5.0)
        b = torch.clamp(self.proj_bias, -5.0, 5.0)
        damage_normalized = torch.sigmoid(g*(damage + b))

        combined_health = health_direct * (1 - 0.3 * damage_normalized)

        # Reduced linear channel influence with time-dependent weighting
        lw = self.linear_enforcer(x_reshaped).view(B, T)
        linear_weight = 0.2 * lw * (1.0 - t_normalized)   # Early stages more linear, later stages more network-driven

        hi = linear_weight * nonlinear_decay + (1 - linear_weight) * combined_health

        # ---- Vectorized strictly non-increasing enforcement with adaptive epsilon ----
        if is_after:
            # Smaller epsilon step for after-maintenance to allow for plateaus
            eps = 5e-4
        else:
            eps = 1e-3
        idx = torch.arange(T, device=hi.device, dtype=hi.dtype).unsqueeze(0)  # (1, T)
        hi_mon, _ = torch.cummin(hi + eps * idx, dim=1)
        hi = hi_mon - eps * idx
        # -------------------------------------------------------------------------

        # Predict steady-state level for plateau targeting
        x_global = x.mean(dim=1)  # (B, C) - global sensor statistics
        steady_level = self.steady_state_head(x_global)  # (B, 1)

        return hi, h_multi, weights, temperature, steady_level

class ReconHead(nn.Module):
    def __init__(self, C, hidden=128):
        super().__init__()
        self.gru = nn.GRU(input_size=C+1, hidden_size=hidden, batch_first=True)
        self.out = nn.Linear(hidden, C)
    def forward(self, x_in, h_in):
        B,T,C = x_in.shape
        h_in_clamped = torch.clamp(h_in, 0.0, 10.0)
        feat = torch.cat([x_in, h_in_clamped.unsqueeze(-1)], dim=-1)
        H,_ = self.gru(feat)
        y = self.out(H)
        return y

class MaintClassifier(nn.Module):
    def __init__(self, sensor_dim, hidden=64, n_classes=14):
        super().__init__()
        self.hi_mlp = nn.Sequential(
            nn.Linear(6, hidden), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden//2)
        )
        self.sensor_mlp = nn.Sequential(
            nn.Linear(sensor_dim * 2, hidden), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden//2)
        )
        self.final_classifier = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, n_classes)
        )

    def forward(self, h_b, h_a, x_b, x_a, mask):
        m = mask
        T = m.size(1)
        valid = m.sum(1, keepdim=True).clamp_min(1.0)

        db = h_a - h_b
        mean_d = (db*m).sum(1,keepdim=True)/valid
        var_d  = (((db-mean_d)*m)**2).sum(1,keepdim=True)/valid
        std_d  = (var_d + 1e-8).sqrt()
        d0     = (db[:, :1])
        dmax   = (db.masked_fill(m==0, -1e9)).max(dim=1, keepdim=True).values
        pos_ratio = ((db>0).float()*m).sum(1,keepdim=True)/valid

        t = torch.arange(T, device=h_b.device, dtype=h_b.dtype)
        t = (t - t.mean())/(t.std()+1e-6)
        def slope(x):
            num = (x*t).sum(1) - x.sum(1)*t.sum()/T
            den = (t**2).sum() - (t.sum()**2)/T + 1e-8
            return (num/den).unsqueeze(1)
        slope_diff = slope(h_a) - slope(h_b)

        hi_feat = torch.cat([mean_d, d0, dmax, std_d, slope_diff, pos_ratio], dim=1)
        hi_feat = torch.clamp(hi_feat, -10.0, 10.0)
        hi_features = self.hi_mlp(hi_feat)

        x_b_mean = (x_b * m.unsqueeze(-1)).sum(1) / valid
        x_a_mean = (x_a * m.unsqueeze(-1)).sum(1) / valid
        sensor_change = torch.cat([x_b_mean, x_a_mean], dim=1)
        sensor_features = self.sensor_mlp(sensor_change)

        combined = torch.cat([hi_features, sensor_features], dim=1)
        logits = self.final_classifier(combined)
        return logits

def sanitize_tensors(*tensors):
    result = []
    for t in tensors:
        if t is not None:
            t_clean = torch.nan_to_num(t, nan=0.0, posinf=1e6, neginf=-1e6)
            result.append(t_clean)
        else:
            result.append(t)
    return result[0] if len(result) == 1 else result

class DiffAwareReconstructor(nn.Module):
    """
    - Two encoders share parameters: encode x_before / x_after → h_b / h_a (monotonic decreasing HI)
    - Two reconstruction heads: one-step recursive reconstruction
    - Classification head: ΔHI + sensor change features
    """
    def __init__(self, in_ch, trend_ch=4, hidden=128, n_classes=14):
        super().__init__()
        self.encoder = TrendEncoder(in_ch, trend_ch)
        self.recon_b = ReconHead(in_ch, hidden)
        self.recon_a = ReconHead(in_ch, hidden)
        self.clf     = MaintClassifier(sensor_dim=in_ch, hidden=64, n_classes=n_classes)
        self.consistency_weight = nn.Parameter(torch.tensor(1.0))

    def forward(self, x_b, x_a, mask):
        h_b, h_multi_b, weights_b, temp_b, steady_b = self.encoder(x_b, is_after=False)
        h_a, h_multi_a, weights_a, temp_a, steady_a = self.encoder(x_a, is_after=True)

        h_b, h_a = sanitize_tensors(h_b, h_a)
        weights_b, weights_a = sanitize_tensors(weights_b, weights_a)
        steady_b, steady_a = sanitize_tensors(steady_b, steady_a)

        xb_in, hb_in = x_b[:, :-2], h_b[:, :-2]
        xa_in, ha_in = x_a[:, :-2], h_a[:, :-2]
        yb_hat = self.recon_b(xb_in, hb_in)   # (B, L-2, C) ≈ x_b[:,1:L-1]
        ya_hat = self.recon_a(xa_in, ha_in)   # (B, L-2, C) ≈ x_a[:,1:L-1]

        yb_hat, ya_hat = sanitize_tensors(yb_hat, ya_hat)
        logits = self.clf(h_b, h_a, x_b, x_a, mask)
        logits = sanitize_tensors(logits)
        return yb_hat, ya_hat, h_b, h_a, logits, weights_b, weights_a, temp_b, temp_a, steady_b, steady_a

# ---------------------------------------------------------------------
# Losses / Regularizers
# ---------------------------------------------------------------------
def masked_mse(a, b, mask):
    diff = (a - b)**2
    mse  = (diff.mean(-1) * mask).sum() / (mask.sum() + 1e-6)
    return mse

def masked_mse_batchwise(a, b, mask):
    diff = (a - b)**2  # (B, T, C)
    mse_t = diff.mean(-1) * mask  # (B, T)
    denom = mask.sum(1).clamp_min(1.0)  # (B,)
    return mse_t.sum(1) / denom

def slope_loss(h, mask, kind="smooth"):
    if kind=="smooth":
        d2 = h[:,2:] - 2*h[:,1:-1] + h[:,:-2]
        m  = mask[:,2:]
        return (d2.abs() * m).sum() / (m.sum()+1e-6)
    elif kind=="mono_dec":
        dh = h[:,1:] - h[:,:-1]
        m  = mask[:,1:]
        return (F.relu(dh) * m).sum() / (m.sum()+1e-6)
    elif kind=="tv":
        # Total variation (first-order differences) - allows more curvature
        dh = h[:,1:] - h[:,:-1]
        m  = mask[:,1:]
        return (dh.abs() * m).sum() / (m.sum()+1e-6)
    else:
        return torch.tensor(0., device=h.device)

def plateau_tv_loss(h_a, mask, tau_ratio=0.3):
    """
    Time-weighted total variation loss for the plateau region (later 70% of sequence).
    Encourages h_a to become flat in the later stages.
    """
    B, T = h_a.shape
    tau = int(tau_ratio * T)

    if T <= tau + 1:
        return torch.tensor(0.0, device=h_a.device)

    # Focus on the plateau region (later 70%)
    h_plateau = h_a[:, tau:]  # (B, T-tau)
    mask_plateau = mask[:, tau:]  # (B, T-tau)

    if h_plateau.size(1) <= 1:
        return torch.tensor(0.0, device=h_a.device)

    # First-order differences in plateau region
    dh = h_plateau[:, 1:] - h_plateau[:, :-1]  # (B, T-tau-1)
    mask_diff = mask_plateau[:, 1:]  # (B, T-tau-1)

    # Time-increasing weights (later time steps get higher penalty)
    T_plateau = h_plateau.size(1) - 1
    if T_plateau > 0:
        time_weights = torch.linspace(1.0, 3.0, T_plateau, device=h_a.device).unsqueeze(0)  # (1, T-tau-1)
        weighted_tv = (dh.abs() * time_weights * mask_diff).sum() / (mask_diff.sum() + 1e-6)
    else:
        weighted_tv = torch.tensor(0.0, device=h_a.device)

    return weighted_tv

def plateau_curvature_loss(h_a, mask, tau_ratio=0.3):
    """
    Time-weighted second-order difference loss for the plateau region.
    Reduces curvature/oscillations in the later stages.
    """
    B, T = h_a.shape
    tau = int(tau_ratio * T)

    if T <= tau + 2:
        return torch.tensor(0.0, device=h_a.device)

    # Focus on the plateau region
    h_plateau = h_a[:, tau:]  # (B, T-tau)
    mask_plateau = mask[:, tau:]  # (B, T-tau)

    if h_plateau.size(1) <= 2:
        return torch.tensor(0.0, device=h_a.device)

    # Second-order differences in plateau region
    d2h = h_plateau[:, 2:] - 2 * h_plateau[:, 1:-1] + h_plateau[:, :-2]  # (B, T-tau-2)
    mask_d2 = mask_plateau[:, 2:]  # (B, T-tau-2)

    # Time-increasing weights
    T_plateau = h_plateau.size(1) - 2
    if T_plateau > 0:
        time_weights = torch.linspace(1.0, 2.0, T_plateau, device=h_a.device).unsqueeze(0)  # (1, T-tau-2)
        weighted_curv = (d2h.abs() * time_weights * mask_d2).sum() / (mask_d2.sum() + 1e-6)
    else:
        weighted_curv = torch.tensor(0.0, device=h_a.device)

    return weighted_curv

def steady_state_loss(h_a, steady_level, mask, tau_ratio=0.3):
    """
    Encourage h_a to approach the predicted steady-state level in the plateau region.
    """
    B, T = h_a.shape
    tau = int(tau_ratio * T)

    if T <= tau:
        return torch.tensor(0.0, device=h_a.device)

    # Focus on the plateau region
    h_plateau = h_a[:, tau:]  # (B, T-tau)
    mask_plateau = mask[:, tau:]  # (B, T-tau)

    # Expand steady_level to match plateau dimensions
    steady_expanded = steady_level.expand(-1, h_plateau.size(1))  # (B, T-tau)

    # L2 loss to steady state
    steady_loss = ((h_plateau - steady_expanded) ** 2 * mask_plateau).sum() / (mask_plateau.sum() + 1e-6)

    return steady_loss

def flat_operator_prior_loss(weights_a, mask, tau_ratio=0.3, flat_op_indices=[1, 3]):
    """
    Encourage flat/saturation operators to have higher weights in the plateau region.
    flat_op_indices: indices of MonotonicFlatOp and SaturationSigmoidOp
    """
    B, T, K = weights_a.shape
    tau = int(tau_ratio * T)

    if T <= tau:
        return torch.tensor(0.0, device=weights_a.device)

    # Focus on plateau region
    weights_plateau = weights_a[:, tau:, :]  # (B, T-tau, K)
    mask_plateau = mask[:, tau:]  # (B, T-tau)

    # Target distribution: higher probability for flat operators
    target_dist = torch.ones(K, device=weights_a.device) / K
    for idx in flat_op_indices:
        if idx < K:
            target_dist[idx] *= 2.0  # Boost flat operators
    target_dist = target_dist / target_dist.sum()  # Renormalize

    # KL divergence from current weights to target distribution
    eps = 1e-8
    weights_plateau_norm = weights_plateau + eps
    target_expanded = target_dist.unsqueeze(0).unsqueeze(0).expand_as(weights_plateau_norm)

    kl_div = (weights_plateau_norm * torch.log(weights_plateau_norm / target_expanded)).sum(-1)  # (B, T-tau)
    kl_loss = (kl_div * mask_plateau).sum() / (mask_plateau.sum() + 1e-6)

    return kl_loss

def enhanced_liquid_weight_regularization(weights_b, weights_a, mask,
                                          lambda_tv=0.05, lambda_ent=0.2,
                                          lambda_bal=0.1, lambda_div=0.1):
    total_loss = torch.tensor(0.0, device=weights_b.device)

    def tv_loss(weights, mask):
        if weights.size(1) <= 1:
            return torch.tensor(0.0, device=weights.device)
        diff = weights[:, 1:, :] - weights[:, :-1, :]
        mask_diff = mask[:, 1:]
        tv = (diff.abs().sum(-1) * mask_diff).sum() / (mask_diff.sum() + 1e-6)
        return tv

    tv_loss_b = tv_loss(weights_b, mask)
    tv_loss_a = tv_loss(weights_a, mask)
    total_loss += lambda_tv * (tv_loss_b + tv_loss_a)

    def entropy_loss(weights, mask):
        eps = 1e-8
        entropy = -(weights * torch.log(weights + eps)).sum(-1)
        target_entropy = np.log(weights.size(-1)) * 0.8
        entropy_penalty = F.relu(target_entropy - entropy)
        return (entropy_penalty * mask).sum() / (mask.sum() + 1e-6)

    ent_loss_b = entropy_loss(weights_b, mask)
    ent_loss_a = entropy_loss(weights_a, mask)
    total_loss += lambda_ent * (ent_loss_b + ent_loss_a)

    def balance_loss(weights, mask):
        valid_weights = weights * mask.unsqueeze(-1)  # (B, T, K)
        mean_usage = valid_weights.sum([0, 1]) / (mask.sum() + 1e-6)  # (K,)
        target_usage = 1.0 / weights.size(-1)
        balance = ((mean_usage - target_usage) ** 2).sum()
        return balance

    bal_loss_b = balance_loss(weights_b, mask)
    bal_loss_a = balance_loss(weights_a, mask)
    total_loss += lambda_bal * (bal_loss_b + bal_loss_a)

    def diversity_loss(weights, mask):
        B, T, K = weights.shape
        if T <= 1:
            return torch.tensor(0.0, device=weights.device)
        valid_weights = weights * mask.unsqueeze(-1)
        weights_flat = valid_weights.view(-1, K)
        weights_centered = weights_flat - weights_flat.mean(0, keepdim=True)
        cov = torch.mm(weights_centered.T, weights_centered) / (weights_flat.size(0) - 1 + 1e-6)
        mask_offdiag = torch.ones_like(cov) - torch.eye(K, device=cov.device)
        corr_penalty = (cov.abs() * mask_offdiag).sum()
        return corr_penalty

    div_loss_b = diversity_loss(weights_b, mask)
    div_loss_a = diversity_loss(weights_a, mask)
    total_loss += lambda_div * (div_loss_b + div_loss_a)
    return total_loss

def smoothness_enhancement_loss_separated(h_b, h_a, mask, order=2, weight_b=0.3, weight_a=1.0):
    """
    Separated smoothness loss: lower weight for h_b, higher weight for h_a.
    """
    def compute_smoothness(h, mask, order):
        if order == 2:
            if h.size(1) <= 2:
                return torch.tensor(0.0, device=h.device)
            d2 = h[:,2:] - 2*h[:,1:-1] + h[:,:-2]
            m = mask[:,2:]
            return (d2.abs() * m).sum() / (m.sum() + 1e-6)
        elif order == 3:
            if h.size(1) <= 3:
                return torch.tensor(0.0, device=h.device)
            d3 = h[:,3:] - 3*h[:,2:-1] + 3*h[:,1:-2] - h[:,:-3]
            m = mask[:,3:]
            return (d3.abs() * m).sum() / (m.sum() + 1e-6)
        else:
            return torch.tensor(0., device=h.device)

    smooth_b = compute_smoothness(h_b, mask, order)
    smooth_a = compute_smoothness(h_a, mask, order)

    return weight_b * smooth_b + weight_a * smooth_a

def compute_sample_weights(matching_costs, alpha=0.5):
    return 1.0 / (1.0 + alpha * matching_costs)

def maintenance_effect_loss(h_b, h_a, mask, lambda_diff=1.0, lambda_smooth_after=2.5, lambda_level_constraint=3.0):
    """
    Enhanced maintenance effect loss with stronger after-maintenance smoothing.
    """
    total_loss = torch.tensor(0.0, device=h_b.device)

    # 1. Encourage difference between before and after HI (improved stability)
    hi_diff = torch.abs(h_a - h_b)  # (B, T)
    hi_diff_clamped = torch.clamp(hi_diff, min=1e-3)  # Prevent extreme gradients
    diff_loss = F.softplus(-10.0 * hi_diff_clamped)  # Smooth penalty for small differences
    diff_loss = (diff_loss * mask).sum() / (mask.sum() + 1e-6)
    total_loss += lambda_diff * diff_loss

    # 2. Encourage after-maintenance HI to be smoother (INCREASED WEIGHT)
    if h_a.size(1) > 1:
        h_a_diff = torch.abs(h_a[:, 1:] - h_a[:, :-1])  # (B, T-1)
        mask_diff = mask[:, 1:]
        smooth_after_loss = (h_a_diff * mask_diff).sum() / (mask_diff.sum() + 1e-6)
        total_loss += lambda_smooth_after * smooth_after_loss

    # 3. Encourage min(h_a) > max(h_b) for proper level separation
    h_b_masked = h_b.masked_fill(mask == 0, -1e9)  # Fill invalid positions with very low values
    h_a_masked = h_a.masked_fill(mask == 0, 1e9)   # Fill invalid positions with very high values

    h_b_max = h_b_masked.max(dim=1)[0]  # (B,) - max of each sequence
    h_a_min = h_a_masked.min(dim=1)[0]  # (B,) - min of each sequence

    # Encourage h_a_min > h_b_max with a margin
    margin = 0.1
    level_constraint = F.relu(h_b_max - h_a_min + margin)  # (B,)
    level_constraint_loss = level_constraint.mean()
    total_loss += lambda_level_constraint * level_constraint_loss

    # Pointwise gap constraint
    margin_pt = 0.05
    pointwise_gap = (h_a - h_b) + (-margin_pt)
    pointwise_loss = F.relu(-pointwise_gap)      # Penalty when h_a < h_b + margin_pt
    pointwise_loss = (pointwise_loss * mask).sum() / (mask.sum() + 1e-6)
    total_loss += 1.0 * pointwise_loss

    return total_loss

def dynamic_plateau_loss(h_b, h_a, steady_a, mask, k=10.0, m=0.05):
    """
    Dynamic plateau loss that activates when maintenance effect is significant.
    """
    # Calculate maintenance effect magnitude
    delta_mean = ((h_a - h_b) * mask).sum(1, keepdim=True) / (mask.sum(1, keepdim=True) + 1e-6)  # (B, 1)

    # Activation function: stronger plateau enforcement when delta is large
    gamma = torch.sigmoid(k * (delta_mean - m))  # (B, 1)

    # Plateau losses
    tv_loss = plateau_tv_loss(h_a, mask)
    curv_loss = plateau_curvature_loss(h_a, mask)
    steady_loss = steady_state_loss(h_a, steady_a, mask)

    # Weight by activation
    weighted_plateau_loss = gamma.mean() * (tv_loss + 0.5 * curv_loss + steady_loss)

    return weighted_plateau_loss

# ---------------------------------------------------------------------
# Evaluation / Prediction
# ---------------------------------------------------------------------
@torch.no_grad()
def eval_epoch(model, loader, device, K, compute_acc=True):
    model.eval()
    total = {"mse_b":0.0, "mse_a":0.0, "acc":0.0, "delta_mean":0.0}
    n_batch=0
    n_acc = 0; n_tot=0
    for batch in loader:
        xb, xa, labels, mask, lengths, sample_ids, matching_costs = adapt_batch(batch, device)
        m_tgt  = (torch.arange(xb.size(1)-2, device=device)[None,:] < (lengths-2)[:,None]).float()
        yb_hat, ya_hat, h_b, h_a, logits, _, _, _, _, _, _ = model(xb, xa, mask)
        yb_hat, ya_hat, h_b, h_a, logits = sanitize_tensors(yb_hat, ya_hat, h_b, h_a, logits)

        # Ensure logits dimension matches expected K
        if logits.shape[-1] != K:
            print(f"Warning: Logits dim {logits.shape[-1]} != expected K={K}, adjusting...")
            if logits.shape[-1] < K:
                # Pad with zeros
                padding = torch.zeros(logits.shape[0], K - logits.shape[-1], device=logits.device)
                logits = torch.cat([logits, padding], dim=1)
            else:
                # Truncate
                logits = logits[:, :K]

        yb_tgt = xb[:,1:-1,:]
        ya_tgt = xa[:,1:-1,:]
        mse_b = masked_mse(yb_hat, yb_tgt, m_tgt)
        mse_a = masked_mse(ya_hat, ya_tgt, m_tgt)
        if compute_acc and labels is not None:
            # Filter out ignore_index labels
            valid_labels = labels != -100
            if valid_labels.any():
                pred = logits[valid_labels].argmax(dim=1)
                labels_valid = labels[valid_labels]
                n_acc += (pred == labels_valid).sum().item()
                n_tot += labels_valid.numel()
        valid = mask.sum(1, keepdim=True).clamp_min(1.0)
        delta_mean = ((h_a - h_b)*mask).sum(1,keepdim=True)/valid
        delta_mean = torch.nan_to_num(delta_mean, nan=0.0)
        total["delta_mean"] += delta_mean.mean().item()
        total["mse_b"] += mse_b.item()
        total["mse_a"] += mse_a.item()
        n_batch += 1
    for k in ["mse_b","mse_a","delta_mean"]:
        total[k] /= max(n_batch,1)
    total["acc"] = (n_acc / max(n_tot,1)) if n_tot>0 else 0.0
    return total

@torch.no_grad()
def collect_test_predictions(model, loader, device, max_curve_keep=24, K=14, id_to_name=None):
    model.eval()
    y_true, y_pred = [], []
    all_delta_mean, all_sample_ids = [], []
    keep_curves = []
    for batch in loader:
        xb, xa, labels, mask, lengths, sample_ids, matching_costs = adapt_batch(batch, device)
        yb_hat, ya_hat, h_b, h_a, logits, weights_b, weights_a, temp_b, temp_a, steady_b, steady_a = model(xb, xa, mask)
        yb_hat, ya_hat, h_b, h_a, logits = sanitize_tensors(yb_hat, ya_hat, h_b, h_a, logits)

        # Ensure logits dimension matches expected K
        if logits.shape[-1] != K:
            if logits.shape[-1] < K:
                padding = torch.zeros(logits.shape[0], K - logits.shape[-1], device=logits.device)
                logits = torch.cat([logits, padding], dim=1)
            else:
                logits = logits[:, :K]

        prob = F.softmax(logits, dim=1)     # (B,K)
        pred = prob.argmax(1)               # (B,)

        valid = mask.sum(1, keepdim=True).clamp_min(1.0)
        delta_mean = ((h_a - h_b) * mask).sum(1, keepdim=True) / valid  # (B,1)
        delta_mean = torch.nan_to_num(delta_mean, nan=0.0)

        if labels is not None:
            # Filter out ignore_index labels
            valid_labels = labels != -100
            if valid_labels.any():
                y_true.append(labels[valid_labels].detach().cpu().numpy())
                y_pred.append(pred[valid_labels].detach().cpu().numpy())
        all_delta_mean.append(delta_mean.squeeze(1).cpu().numpy())
        all_sample_ids.extend(sample_ids)

        for i in range(xb.size(0)):
            if len(keep_curves) >= max_curve_keep: break
            L_i = int(lengths[i].item())
            L_recon = min(L_i - 2, yb_hat.size(1))
            keep_curves.append({
                "sample_id": sample_ids[i],
                "true": int(labels[i].cpu().item()) if labels is not None and labels[i] != -100 else -1,
                "pred": int(pred[i].cpu().item()),
                "prob": prob[i].cpu().numpy(),
                "h_before": h_b[i, :L_i].cpu().numpy(),
                "h_after":  h_a[i, :L_i].cpu().numpy(),
                "weights_before": weights_b[i, :L_i].cpu().numpy() if weights_b is not None else None,
                "weights_after": weights_a[i, :L_i].cpu().numpy() if weights_a is not None else None,
                "temp_before": temp_b[i, :L_i].cpu().numpy() if temp_b is not None else None,
                "temp_after": temp_a[i, :L_i].cpu().numpy() if temp_a is not None else None,
                "steady_before": steady_b[i].cpu().item() if steady_b is not None else None,
                "steady_after": steady_a[i].cpu().item() if steady_a is not None else None,
                "op_outputs_before": [],
                "op_outputs_after":  [],
                "yb_hat":   yb_hat[i, :L_recon].cpu().numpy(),
                "ya_hat":   ya_hat[i, :L_recon].cpu().numpy(),
                "x_before": xb[i, :L_i].cpu().numpy(),
                "x_after":  xa[i, :L_i].cpu().numpy(),
            })
    y_true = np.concatenate(y_true, axis=0) if len(y_true)>0 else np.array([])
    y_pred = np.concatenate(y_pred, axis=0) if len(y_pred)>0 else np.array([])
    all_delta_mean = np.concatenate(all_delta_mean, axis=0) if len(all_delta_mean)>0 else np.array([])
    return y_true, y_pred, all_delta_mean, all_sample_ids, keep_curves

# ---------------------------------------------------------------------
# Plotting
# ---------------------------------------------------------------------
def remove_constant_segments(hi_values, threshold=1e-6):
    if len(hi_values) <= 1:
        return hi_values, np.arange(len(hi_values))
    diffs = np.abs(np.diff(hi_values))
    valid_mask = np.ones(len(hi_values), dtype=bool)
    valid_mask[1:] = diffs > threshold
    if np.sum(valid_mask) < 2:
        valid_mask[0] = True
        valid_mask[-1] = True
    valid_indices = np.where(valid_mask)[0]
    return hi_values[valid_indices], valid_indices

def plot_hi_examples_aligned(curves, id_to_name, n_show=6, seed=0):
    if len(curves)==0:
        print("(No visualization samples)")
        return
    random.Random(seed).shuffle(curves)
    n_show = min(n_show, len(curves))
    cols = 3
    rows = int(np.ceil(n_show/cols))
    plt.figure(figsize=(cols*4.6, rows*3.2))
    for k in range(n_show):
        ex = curves[k]
        hb = ex["h_before"]; ha = ex["h_after"]
        L  = min(len(hb), len(ha))
        hb, ha = hb[:L], ha[:L]
        hb_clean, hb_indices = remove_constant_segments(hb)
        ha_clean, ha_indices = remove_constant_segments(ha)
        t_b = hb_indices
        t_a = ha_indices + L

        sample_id = ex["sample_id"]
        pred = ex["pred"]; true = ex["true"]; prob = ex["prob"]
        true_name = id_to_name.get(true, f"Class_{true}")
        pred_name = id_to_name.get(pred, f"Class_{pred}")

        plt.subplot(rows, cols, k+1)
        if len(hb_clean) > 1:
            plt.plot(t_b, hb_clean, label="Learned HI (Pre)", linewidth=1.8, marker='o', markersize=3)
        if len(ha_clean) > 1:
            plt.plot(t_a, ha_clean, label="Learned HI (Post)",  linewidth=1.8, linestyle='--', marker='s', markersize=3)
        plt.axvline(L-1, color='k', linestyle=':', linewidth=1.0, alpha=0.7)
        d_mean = float(np.mean(ha - hb))
        d_max  = float(np.max(ha - hb))

        # Add steady state info if available
        steady_info = ""
        if ex.get("steady_after") is not None:
            steady_info = f" | SS={ex['steady_after']:.3f}"

        plt.title(f"ID={sample_id}\nTrue={true_name} | Pred={pred_name} "
                  f"| p={prob[pred]:.2f}\nΔHI_mean={d_mean:.3f}, ΔHI_max={d_max:.3f}{steady_info}", fontsize=9)
        plt.xlabel("Cycle (Pre | Post)")
        plt.ylabel("Health Index (higher=better)")
        plt.grid(ls='--', alpha=.35)
        if k==0: plt.legend()
    plt.tight_layout(); plt.show()

def plot_enhanced_liquid_weights(curves, n_show=3, seed=0):
    if len(curves) == 0:
        print("(No visualization samples)")
        return
    curves_with_weights = [c for c in curves if c.get("weights_before") is not None]
    if len(curves_with_weights) == 0:
        print("(No weight information available)")
        return
    random.Random(seed).shuffle(curves_with_weights)
    n_show = min(n_show, len(curves_with_weights))
    op_names = ["MonotonicLinear", "MonotonicFlat", "ConcaveLog", "SaturationSigmoid",
                "HingeReLU", "Polynomial", "DampedSin", "PiecewiseLinear"]
    for i in range(n_show):
        ex = curves_with_weights[i]
        weights_b = ex["weights_before"]
        weights_a = ex["weights_after"]
        temp_b = ex.get("temp_before")
        temp_a = ex.get("temp_after")
        sample_id = ex["sample_id"]
        true_label = ex["true"]
        pred_label = ex["pred"]
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle(f"Dynamic Parameter Modulation - ID={sample_id}, True=Class_{true_label}, Pred=Class_{pred_label}", fontsize=14)
        ax = axes[0, 0]
        T_b, K = weights_b.shape
        for k in range(K):
            ax.plot(range(T_b), weights_b[:, k], label=op_names[k] if k < len(op_names) else f"Op_{k}",
                   marker='o', markersize=2, linewidth=1.5)
        ax.set_title("Pre-maintenance Weights"); ax.set_xlabel("Time Step"); ax.set_ylabel("Weight")
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left'); ax.grid(True, alpha=0.3)
        ax = axes[0, 1]
        T_a, _ = weights_a.shape
        for k in range(K):
            ax.plot(range(T_a), weights_a[:, k], label=op_names[k] if k < len(op_names) else f"Op_{k}",
                   marker='s', markersize=2, linewidth=1.5, linestyle='--')
        ax.set_title("Post-maintenance Weights"); ax.set_xlabel("Time Step"); ax.set_ylabel("Weight")
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left'); ax.grid(True, alpha=0.3)
        ax = axes[0, 2]
        entropy_b = -np.sum(weights_b * np.log(weights_b + 1e-8), axis=1)
        entropy_a = -np.sum(weights_a * np.log(weights_a + 1e-8), axis=1)
        ax.plot(range(T_b), entropy_b, label="Pre", marker='o', linewidth=2)
        ax.plot(range(T_a), entropy_a, label="Post", marker='s', linewidth=2, linestyle='--')
        ax.axhline(np.log(K), color='r', linestyle=':', label=f'Max Entropy ({np.log(K):.2f})')
        ax.set_title("Weight Entropy Over Time"); ax.set_xlabel("Time Step"); ax.set_ylabel("Entropy")
        ax.legend(); ax.grid(True, alpha=0.3)
        ax = axes[1, 0]
        if temp_b is not None and temp_a is not None:
            ax.plot(range(T_b), temp_b, label="Pre", marker='o', linewidth=2)
            ax.plot(range(T_a), temp_a, label="Post", marker='s', linewidth=2, linestyle='--')
            ax.set_title("Temperature Evolution"); ax.set_xlabel("Time Step"); ax.set_ylabel("Temperature"); ax.legend()
        else:
            ax.text(0.5, 0.5, "Temperature data\nnot available", ha='center', va='center', transform=ax.transAxes)
            ax.set_title("Temperature Evolution")
        ax.grid(True, alpha=0.3)
        ax = axes[1, 1]
        ax.text(0.5, 0.5, "Operator output variance\n(not collected)", ha='center', va='center', transform=ax.transAxes)
        ax.set_title("Operator Output Variance"); ax.grid(True, alpha=0.3)
        ax = axes[1, 2]
        combined_weights = np.concatenate([weights_b, weights_a], axis=0)
        # Apply smoothing for correlation stability
        from scipy.ndimage import uniform_filter1d
        smoothed_weights = uniform_filter1d(combined_weights, size=3, axis=0)
        corr_matrix = np.corrcoef(smoothed_weights.T)
        im = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
        ax.set_title("Operator Weight Correlation (Smoothed)")
        ax.set_xticks(range(K)); ax.set_yticks(range(K))
        short_names = [op_names[k][:8] if k < len(op_names) else f"Op_{k}" for k in range(K)]
        ax.set_xticklabels(short_names, rotation=45); ax.set_yticklabels(short_names)
        cbar = plt.colorbar(im, ax=ax); cbar.set_label('Correlation')
        plt.tight_layout(); plt.show()

def plot_liquid_weights(curves, n_show=3, seed=0):
    plot_enhanced_liquid_weights(curves, n_show, seed)

def plot_sensor_examples_aligned(curves, sensor_idx_list=None, n_cols=4):
    if len(curves)==0:
        print("(No visualization samples)")
        return
    ex = curves[0]
    xb = ex["x_before"]; xa = ex["x_after"]; ya = ex["ya_hat"]
    L, C = xb.shape
    Lhat = ya.shape[0]
    if sensor_idx_list is None:
        step = max(1, C//8); sensor_idx_list = list(range(0, C, step))[:8]
    n = len(sensor_idx_list)
    n_rows = int(np.ceil(n/n_cols))
    plt.figure(figsize=(n_cols*4.3, n_rows*3.1))
    for i, s in enumerate(sensor_idx_list):
        plt.subplot(n_rows, n_cols, i+1)
        t_b = np.arange(L); t_a = np.arange(L, 2*L)
        plt.plot(t_b, xb[:,s], label="Original (pre)", linewidth=1.2)
        plt.plot(t_a, xa[:,s], label="Original (post)",  linewidth=1.0, linestyle="--", alpha=0.7)
        t_pred = np.arange(L+1, L+1+Lhat)
        plt.plot(t_pred, ya[:,s], label="Post prediction", linewidth=1.6)
        plt.axvline(L-1, color='k', linestyle=':', linewidth='1.0')
        plt.title(f"Sensor_{s:02d}")
        if i==0: plt.legend()
        plt.grid(ls="--", alpha=.35)
    plt.tight_layout(); plt.show()

def topk_by_delta(df_delta, id_to_name, k=3):
    if len(df_delta)==0:
        return
    unique_classes = sorted(df_delta["true"].unique())
    for cls in unique_classes[:5]:
        sub = df_delta[df_delta["true"]==cls].sort_values("delta_hi_mean", ascending=False).head(k)
        class_name = id_to_name.get(cls, f"Class_{cls}")
        print(f"\n[True={class_name}] Top {k} samples with highest ΔHI_mean:")
        print(sub[["uid","delta_hi_mean","pred"]].reset_index(drop=True))

# ---------------------------------------------------------------------
# Training (separate) and Testing (separate)
# ---------------------------------------------------------------------
def build_model_and_setup(train_loader, device, class_weights=None):
    """
    Build model and setup training components.

    Args:
        train_loader: Training data loader
        device: Device to use
        class_weights: Pre-computed class weights tensor (optional)
    """
    # Determine K from training loader
    K = scan_num_classes_from_loader(train_loader)
    id_to_name = build_default_id_to_name(K)

    first_batch = next(iter(train_loader))
    C = first_batch["x_before"].shape[-1]

    print(f"\n[Setup] Sensor dim C={C}, Num classes K={K}")

    # Use provided class weights or create uniform weights
    if class_weights is not None:
        if isinstance(class_weights, np.ndarray):
            class_weights = torch.from_numpy(class_weights).float().to(device)
        elif isinstance(class_weights, torch.Tensor):
            class_weights = class_weights.float().to(device)

        # Ensure class_weights has the right size
        if len(class_weights) < K:
            # Pad with ones if needed
            padded_weights = torch.ones(K, dtype=torch.float32, device=device)
            padded_weights[:len(class_weights)] = class_weights
            class_weights = padded_weights
        elif len(class_weights) > K:
            # Truncate if needed
            class_weights = class_weights[:K]

        print(f"[Class Weights] Using provided weights: {class_weights.detach().cpu().numpy()}")
    else:
        class_weights = torch.ones(K, dtype=torch.float32, device=device)
        print(f"[Class Weights] Using uniform weights: {class_weights.detach().cpu().numpy()}")

    model = DiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=K).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    ce  = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100, reduction='none')

    return model, opt, ce, K, id_to_name, C


def train_only(train_loader, val_loader, epochs=300, lr=1e-3, device=None, patience=20,
               use_matching_cost=True, save_path="best_model.pth", class_weights=None,
               es_alpha=1.0):
    """
    Train and save the best model. Returns path to checkpoint and metadata.

    Args:
        class_weights: Pre-computed class weights (numpy array or torch tensor)
        es_alpha: Weight for accuracy in early stopping metric
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Build model and training setup
    model, opt, ce, K, id_to_name, C = build_model_and_setup(train_loader, device, class_weights)
    # Override LR if provided
    for g in opt.param_groups:
        g['lr'] = lr

    best_v = float("inf")
    best_state = None
    best_epoch = 0
    no_improve = 0
    nan_batches = 0

    print(f"\nStarting training for {epochs} epochs with early stopping (patience={patience})...")
    print(f"Device: {device}")
    print(f"Early stopping alpha: {es_alpha}")

    for ep in range(1, epochs+1):
        print(f"\n[Epoch {ep}/{epochs}] Training...")
        model.train()
        logs = {"mse_b":0.0, "mse_a":0.0, "smooth":0.0, "mono":0.0,
                "cls":0.0, "smooth_enh":0.0, "liquid_weight":0.0, "maintenance_effect":0.0,
                "plateau":0.0, "all":0.0}
        n_bt = 0
        batch_nan_count = 0

        progress = ep / epochs
        lambda_tv  = 0.03 * (1 - progress * 0.3)
        lambda_ent = 0.3  * (1 - progress * 0.7)
        lambda_bal = 0.15 * (1 - progress * 0.5)
        lambda_div = 0.2  * (1 - progress * 0.6)

        # Gradient norm tracking
        grad_norms = []

        for batch_idx, batch in enumerate(train_loader):
            xb, xa, labels, mask, lengths, sample_ids, matching_costs = adapt_batch(batch, device)
            m_tgt  = (torch.arange(xb.size(1)-2, device=device)[None,:] < (lengths-2)[:,None]).float()
            yb_hat, ya_hat, h_b, h_a, logits, weights_b, weights_a, temp_b, temp_a, steady_b, steady_a = model(xb, xa, mask)
            yb_hat, ya_hat, h_b, h_a, logits = sanitize_tensors(yb_hat, ya_hat, h_b, h_a, logits)
            weights_b, weights_a = sanitize_tensors(weights_b, weights_a)
            steady_b, steady_a = sanitize_tensors(steady_b, steady_a)

            # Reconstruction targets
            yb_tgt = xb[:,1:-1,:]
            ya_tgt = xa[:,1:-1,:]

            # Per-sample reconstruction losses (for optional cost-weighting)
            rec_b_per = masked_mse_batchwise(yb_hat, yb_tgt, m_tgt)
            rec_a_per = masked_mse_batchwise(ya_hat, ya_tgt, m_tgt)
            rec_per = rec_b_per + rec_a_per

            # Classification loss (per-sample)
            if labels is None:
                cls_per = torch.zeros(xb.size(0), device=device)
            else:
                cls_per = ce(logits, labels)

            # Optional matching-cost weighting
            if use_matching_cost and matching_costs is not None:
                w_i = compute_sample_weights(matching_costs)
            else:
                w_i = torch.ones(xb.size(0), device=device)

            loss_rec = (rec_per * w_i).mean()
            loss_cls = (cls_per * w_i).mean()

            # HI regularizations - separated smoothness for before/after
            loss_smooth = slope_loss(h_a, mask, "tv") + slope_loss(h_b, mask, "tv")  # Use TV instead of second-order
            loss_mono   = slope_loss(h_a, mask, "mono_dec") + slope_loss(h_b, mask, "mono_dec")

            # Separated smoothness enhancement: lower weight for h_b, higher for h_a
            loss_smooth_enhanced = (smoothness_enhancement_loss_separated(h_b, h_a, mask, order=2, weight_b=0.2, weight_a=1.0) +
                                   smoothness_enhancement_loss_separated(h_b, h_a, mask, order=3, weight_b=0.1, weight_a=0.5))

            loss_liquid_weight = enhanced_liquid_weight_regularization(
                weights_b, weights_a, mask,
                lambda_tv=lambda_tv, lambda_ent=lambda_ent,
                lambda_bal=lambda_bal, lambda_div=lambda_div
            )

            # Enhanced maintenance effect loss with stronger after-maintenance smoothing
            loss_maintenance_effect = maintenance_effect_loss(
                h_b, h_a, mask,
                lambda_diff=1.0,           # Encourage difference between before/after
                lambda_smooth_after=2.5,   # INCREASED: Encourage after-maintenance HI to be smoother
                lambda_level_constraint=2.0 # Encourage min(h_a) > max(h_b)
            )

            # NEW: Plateau-specific losses for after-maintenance sequences
            loss_plateau_tv = plateau_tv_loss(h_a, mask)
            loss_plateau_curv = plateau_curvature_loss(h_a, mask)
            loss_steady_state = steady_state_loss(h_a, steady_a, mask)
            loss_flat_prior = flat_operator_prior_loss(weights_a, mask)
            loss_dynamic_plateau = dynamic_plateau_loss(h_b, h_a, steady_a, mask)

            # Combined plateau loss
            loss_plateau = (loss_plateau_tv + 0.5 * loss_plateau_curv +
                           loss_steady_state + 0.3 * loss_flat_prior +
                           loss_dynamic_plateau)

            # Total loss - adjusted weights for plateau emphasis
            w_rec=1.0; w_smooth=0.003; w_mono=0.05; w_cls=1.0; w_smooth_enh=0.03; w_liquid=0.5; w_maintenance=2.0; w_plateau=1.5
            loss = (w_rec*loss_rec + w_smooth*loss_smooth + w_mono*loss_mono +
                    w_cls*loss_cls + w_smooth_enh*loss_smooth_enhanced + w_liquid*loss_liquid_weight +
                    w_maintenance*loss_maintenance_effect + w_plateau*loss_plateau)

            loss_rec, loss_cls, loss_smooth, loss_mono, loss_smooth_enhanced, loss_liquid_weight, loss_maintenance_effect, loss_plateau, loss = sanitize_tensors(
                loss_rec, loss_cls, loss_smooth, loss_mono, loss_smooth_enhanced, loss_liquid_weight, loss_maintenance_effect, loss_plateau, loss
            )

            if torch.isnan(loss) or torch.isinf(loss):
                batch_nan_count += 1
                if batch_nan_count == 1:
                    print(f"    [Warning] Epoch {ep}: Found NaN/Inf loss, skipping abnormal batch")
                continue

            opt.zero_grad()
            loss.backward()

            # Adaptive gradient clipping
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            grad_norms.append(grad_norm.item())

            opt.step()

            logs["mse_b"] += rec_b_per.mean().item()
            logs["mse_a"] += rec_a_per.mean().item()
            logs["smooth"]+= loss_smooth.item()
            logs["mono"]  += loss_mono.item()
            logs["cls"]   += loss_cls.item()
            logs["smooth_enh"]+= loss_smooth_enhanced.item()
            logs["liquid_weight"]+= loss_liquid_weight.item()
            logs["maintenance_effect"]+= loss_maintenance_effect.item()
            logs["plateau"]+= loss_plateau.item()
            logs["all"]   += loss.item()
            n_bt += 1

        if batch_nan_count > 0:
            nan_batches += batch_nan_count

        for k in logs: logs[k] /= max(n_bt,1)

        print("    Validating...")
        vl = eval_epoch(model, val_loader, device, K, compute_acc=True)
        # Improved early-stopping metric with configurable weight
        vl_total = (vl['mse_b'] + vl['mse_a']) + es_alpha * (1.0 - vl['acc'])

        # Log gradient statistics
        avg_grad_norm = np.mean(grad_norms) if grad_norms else 0.0
        max_grad_norm = np.max(grad_norms) if grad_norms else 0.0

        print(f"[Epoch {ep:03d}] "
              f"Train: L={logs['all']:.4f} rec_b={logs['mse_b']:.4f} rec_a={logs['mse_a']:.4f} "
              f"smooth={logs['smooth']:.4f} mono={logs['mono']:.4f} liquid_w={logs['liquid_weight']:.4f} "
              f"maint_eff={logs['maintenance_effect']:.4f} plateau={logs['plateau']:.4f} cls={logs['cls']:.4f} | "
              f"Val: mse_b={vl['mse_b']:.4f} mse_a={vl['mse_a']:.4f} acc={vl['acc']:.3f} ΔHI_impr={vl['delta_mean']:.3f} "
              f"(ES metric={vl_total:.4f}) | Grad: avg={avg_grad_norm:.3f} max={max_grad_norm:.3f}")

        if vl_total < best_v:
            best_v = vl_total
            best_epoch = ep
            best_state = {k: v.clone() if hasattr(v, 'clone') else v for k, v in model.state_dict().items()}
            no_improve = 0
            print(f"    ✓ Saved new best model (ES metric: {best_v:.4f})")
        else:
            no_improve += 1
            print(f"    No improvement for {no_improve}/{patience} epochs")
            if no_improve >= patience:
                print(f"\n[Early Stopping] Patience reached at epoch {ep}.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"\n[Best Checkpoint] ES metric: {best_v:.4f} (Epoch {best_epoch})")
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        torch.save(model.state_dict(), save_path)
        print(f"[Model Saved] {save_path}")

        # Save metadata alongside model
        meta_path = save_path.replace('.pth', '_meta.json')
        meta_data = {
            "K": K,
            "id_to_name": id_to_name,
            "sensor_dim": C,
            "best_es": best_v,
            "best_epoch": best_epoch,
            "class_weights": class_weights.cpu().numpy().tolist() if class_weights is not None else None
        }
        import json
        with open(meta_path, 'w') as f:
            json.dump(meta_data, f, indent=2)
        print(f"[Metadata Saved] {meta_path}")

    if nan_batches > 0:
        print(f"[Warning] Total skipped {nan_batches} batches containing NaN/Inf during training.")

    meta = {"K": K, "id_to_name": id_to_name, "sensor_dim": C, "best_es": best_v, "best_epoch": best_epoch}
    return save_path, meta




# ---------------------------------------------------------------------
# Example usage (replace with your real loaders and class weights)
# ---------------------------------------------------------------------
# Assuming you have computed class_weights from your data loader script:
# class_weights = your_computed_class_weights_tensor  # shape: (K,)

ckpt_path, meta = train_only(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=1000,
    lr=1e-3,
    device=None,     # auto
    patience=20,
    use_matching_cost=True,
    save_path=str(resolve_data_file("best_model_NGFAID_250_100_4.pth")),
    class_weights=torch.as_tensor(class_weights, dtype=torch.float32)   # Pass your computed class weights here
)


In [ ]:

# -------- Infer sensor dimension C and number of classes K from the existing loaders --------
# Prefer train_loader when inferring K; fall back to test_loader if needed.
def _infer_C_and_K(train_loader=None, test_loader=None):
    # Inspect one batch to obtain C
    probe_loader = train_loader if (train_loader is not None) else test_loader
    assert probe_loader is not None, "An available train_loader or test_loader is required to infer the dimensions."
    first_batch = next(iter(probe_loader))
    C = first_batch["x_before"].shape[-1]

    K = None
    if train_loader is not None:
        try:
            K = scan_num_classes_from_loader(train_loader)
        except Exception:
            K = None
    if K is None and test_loader is not None:
        try:
            K = scan_num_classes_from_loader(test_loader)
        except Exception:
            K = None
    assert K is not None and K > 0, "Failed to infer K (number of classes) from the data. Please check the loader labels y."
    return C, K

# train_loader / test_loader should already be constructed outside this snippet
# If train_loader is unavailable, replace it with None below and infer using only test_loader.
C, K = _infer_C_and_K(train_loader=train_loader, test_loader=test_loader)
id_to_name = build_default_id_to_name(K)

print(f"[Info] Inferred sensor dim C={C}, num classes K={K}")

# -------- Backfill / rebuild metadata (optional) --------
# If you want to store the training-time class_weights in the metadata as well, provide them here (tensor / numpy / list are all acceptable).
# Demonstration: write class_weights_obj if it is available; otherwise keep None.
class_weights_obj = None
# For example, if you still have the class_weights array, you can use:
# import numpy as np
# class_weights_obj = np.asarray(class_weights, dtype=float)

def _serialize_class_weights(obj):
    if obj is None:
        return None
    try:
        import numpy as np
        if isinstance(obj, torch.Tensor):
            return obj.detach().cpu().numpy().tolist()
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, (list, tuple)):
            return list(obj)
        else:
            return None
    except Exception:
        return None

def test_only(test_loader, ckpt_path, K=None, id_to_name=None, device=None,
              visualize=True, max_curve_keep=24):
    """
    Load a saved checkpoint and evaluate on the official test set.
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Peek one batch to get sensor dimension C (and fallback K if needed)
    first_batch = next(iter(test_loader))
    C = first_batch["x_before"].shape[-1]

    if K is None:
        # If K not provided, try to infer from labels present in test loader
        # (May undercount if some classes are absent. Prefer passing K from training.)
        print("[Info] K not provided; attempting to infer from test loader labels (may be lower than full K).")
        K = scan_num_classes_from_loader(test_loader)

    if id_to_name is None:
        id_to_name = build_default_id_to_name(K)

    # Build model and load weights
    model = DiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=K).to(device)
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state, strict=False)
    model.eval()

    print("\n" + "="*60)
    print("Final Test Set Evaluation")
    print("="*60)

    te = eval_epoch(model, test_loader, device, K, compute_acc=True)
    print(f"[Test] mse_b={te['mse_b']:.4f} mse_a={te['mse_a']:.4f} acc={te['acc']:.3f} ΔHI_impr={te['delta_mean']:.3f}")

    # Predictions & visualization data
    y_true, y_pred, all_delta_mean, all_uids, keep_curves = collect_test_predictions(
        model, test_loader, device, max_curve_keep=max_curve_keep, K=K, id_to_name=id_to_name
    )

    # Reports (only if labels present)
    if y_true.size > 0:
        print("\n[Classification Report]")
        labels_full = list(range(K))
        target_names = [id_to_name.get(i, f"Class_{i}") for i in labels_full]
        # --- FIX: force labels + names to avoid mismatch when some classes are absent in predictions ---
        print(classification_report(
            y_true, y_pred,
            labels=labels_full,
            target_names=target_names,
            zero_division=0
        ))

        print("\n[Confusion Matrix]")
        print(confusion_matrix(y_true, y_pred, labels=labels_full))

        df_delta = pd.DataFrame({
            "uid": all_uids[:len(y_true)],
            "true": y_true,
            "pred": y_pred,
            "delta_hi_mean": all_delta_mean[:len(y_true)]
        })
        print("\n[Learned ΔHI Statistics] By true class")
        print(df_delta.groupby("true")["delta_hi_mean"].describe())
        topk_by_delta(df_delta, id_to_name, k=3)

        if visualize:
            print("\n[Visualization] Learned health index curves with liquid weight adaptation...")
            plot_hi_examples_aligned(keep_curves, id_to_name, n_show=16, seed=0)

            print("\n[Visualization] Liquid operator weights evolution...")
            plot_liquid_weights(keep_curves, n_show=16, seed=0)

            print("\n[Visualization] Sensor reconstruction predictions...")
            plot_sensor_examples_aligned(keep_curves, sensor_idx_list=None, n_cols=4)
    else:
        print("\n[Info] No labels in test loader; skipping classification report and confusion matrix.")

    return {"test_metrics": te, "K": K, "id_to_name": id_to_name, "curves": keep_curves}
meta = {
    "K": int(K),
    "id_to_name": {int(k): str(v) for k, v in id_to_name.items()},
    "sensor_dim": int(C),
    "best_es": None,             # Leave blank or fill manually if the early-stopping metric from training is unknown
    "best_epoch": None,          # Same as above
    "class_weights": _serialize_class_weights(class_weights_obj)
}

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print(f"[Meta Saved] {meta_path}")

results = test_only(
    test_loader=test_loader,
    ckpt_path=str(resolve_data_file("best_model_NGFAID_250_100_4.pth")),
    K=K,
    id_to_name=id_to_name,
    device=device,
    visualize=True,          # Set to True to generate plots; use False in headless environments
    max_curve_keep=24
)


## Part 3E: Extended Training Configuration


In [ ]:


# ============================================================
# Monotonic-Decreasing HI + aligned plotting (before -> after)
# Adapted to external train/val/test loaders that yield:
#   batch = {
#       "x_before": (B, L, C), "x_after": (B, L, C),
#       "y": LongTensor[B] or None,
#       "meta": list[dict] (each may contain 'pair_id', 'matching_cost', ...)
#   }
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import os
import random
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix

# ---------------------------------------------------------------------
# Utilities for the NEW loaders
# ---------------------------------------------------------------------
def adapt_batch(batch, device):
    """
    Adapt a batch from your loaders to the tensors expected by the model.
    Ensures fixed-length mask/lengths and extracts optional meta.
    """
    xb = batch["x_before"].to(device)  # (B, L, C)
    xa = batch["x_after"].to(device)   # (B, L, C)

    B, L, _ = xb.shape
    mask = torch.ones(B, L, dtype=xb.dtype, device=device)
    lengths = torch.full((B,), L, dtype=torch.long, device=device)

    labels = batch.get("y", None)
    if labels is not None and isinstance(labels, torch.Tensor):
        # Filter out ignore_index labels
        valid_mask = labels != -100
        if valid_mask.any():
            labels = labels.to(device)
        else:
            labels = None
    elif labels is not None:
        labels = None

    meta = batch.get("meta", None)
    sample_ids = []
    matching_costs = torch.zeros(B, dtype=torch.float32, device=device)
    if isinstance(meta, (list, tuple)) and len(meta) == B:
        for i, m in enumerate(meta):
            if isinstance(m, dict):
                sample_ids.append(str(m.get("pair_id", f"sample_{i}")))
                if "matching_cost" in m and m["matching_cost"] is not None:
                    matching_costs[i] = float(m["matching_cost"])
            else:
                sample_ids.append(f"sample_{i}")
    else:
        sample_ids = [f"sample_{i}" for i in range(B)]

    return xb, xa, labels, mask, lengths, sample_ids, matching_costs


def scan_num_classes_from_loader(train_loader):
    """
    Determine K (number of classes) from the training loader.
    """
    max_class = -1
    for batch in train_loader:
        y = batch.get("y", None)
        if y is None:
            continue
        if isinstance(y, torch.Tensor):
            y_np = y.detach().cpu().numpy()
            valid_labels = y_np[y_np >= 0]  # filter out negative labels
            if len(valid_labels) > 0:
                max_class = max(max_class, int(valid_labels.max()))

    if max_class < 0:
        raise ValueError("No valid labels found in the training loader to determine K.")

    K = max_class + 1
    return K


def build_default_id_to_name(K):
    return {i: f"Class_{i}" for i in range(K)}


# ---------------------------------------------------------------------
# Core model components
# ---------------------------------------------------------------------
class BoltzmannKAN(nn.Module):
    def __init__(self, in_ch, out_ch=4):
        super().__init__()
        self.E  = nn.Parameter(torch.zeros(out_ch, in_ch))
        self.kT = nn.Parameter(torch.ones(out_ch, in_ch))
    def forward(self, x):
        B,T,C = x.shape
        kT = torch.clamp(F.softplus(self.kT), 0.01, 10.0).unsqueeze(0).unsqueeze(2)
        E  = torch.clamp(self.E, -10.0, 10.0).unsqueeze(0).unsqueeze(2)
        x_ = torch.clamp(x.unsqueeze(1), -10.0, 10.0)
        exp = torch.clamp((E - x_) / kT, -50, 50)
        w   = torch.sigmoid(exp)
        h   = (w * x_).sum(dim=3).permute(0, 2, 1)
        return torch.clamp(F.softplus(h), 0.0, 100.0)

class BaseOp(nn.Module):
    def __init__(self):
        super().__init__()
        self.param_modulator = None
    def set_param_modulator(self, modulator): self.param_modulator = modulator
    def get_modulated_params(self, context):
        if self.param_modulator is None: return {}
        return self.param_modulator(context)

class MonotonicLinearOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_bias = self.bias
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        bias  = torch.clamp(base_bias + mod.get('bias_offset', 0.0), -5.0, 5.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * (xm + bias)), 0.0, 100.0)

class MonotonicFlatOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 1e-3, 1.0
    def _cum(self, x):
        B, T = x.shape
        diff = F.relu(x[:, 1:] - x[:, :-1])
        zero_init = torch.zeros(B, 1, device=x.device, dtype=x.dtype)
        return torch.cat([zero_init, torch.cumsum(diff, dim=1)], dim=1)
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_bias = self.bias
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        bias  = torch.clamp(base_bias + mod.get('bias_offset', 0.0), -2.0, 2.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        cum_result = self._cum(xm)
        return torch.clamp(F.softplus(scale * (cum_result + bias)), 0.0, 100.0)

class ConcaveLogOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.eps = 1e-3
        self.smin, self.smax = 0.01, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        eps  = torch.clamp(self.eps + mod.get('eps_offset', 0.0), 1e-6, 1e-1)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * torch.log(torch.abs(xm) + eps)), 0.0, 100.0)

class SaturationSigmoidOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.raw_slope = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
        self.lmin, self.lmax = 0.1, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_slope = torch.clamp(F.softplus(self.raw_slope), self.lmin, self.lmax)
        base_bias  = self.bias
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        slope = torch.clamp(base_slope + mod.get('slope_offset', 0.0), self.lmin, self.lmax)
        bias  = torch.clamp(base_bias  + mod.get('bias_offset', 0.0), -3.0, 3.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * torch.sigmoid(slope * (xm - bias))), 0.0, 100.0)

class HingeReLUOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.threshold = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_threshold = self.threshold
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        threshold = torch.clamp(base_threshold + mod.get('threshold_offset', 0.0), -3.0, 3.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * F.relu(xm - threshold)), 0.0, 100.0)

class PolynomialOp(BaseOp):
    def __init__(self, deg=2):  # Reduced default degree for stability
        super().__init__()
        self.raw_coeff = nn.Parameter(torch.zeros(deg))
        self.deg = deg
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_coeff = torch.clamp(F.softplus(self.raw_coeff), 0.01, 5.0)
        xm = torch.clamp(torch.tanh(h.mean(-1)), -3.0, 3.0)  # Added tanh for stability
        coeff_offset = mod.get('coeff_offset', torch.zeros_like(base_coeff))
        if coeff_offset.dim() > 1:
            coeff_offset = coeff_offset.mean(0)
        coeff = torch.clamp(base_coeff + coeff_offset, 0.01, 5.0)
        y = torch.zeros_like(xm)
        for i in range(self.deg):
            y = y + coeff[i] * torch.clamp(xm ** (i + 1), -100.0, 100.0)
        return torch.clamp(F.softplus(y), 0.0, 100.0)

class DampedSinOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.raw_freq = nn.Parameter(torch.tensor(0.0))
        self.raw_lambda = nn.Parameter(torch.tensor(0.0))
        self.phase = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
        self.fmin, self.fmax = 0.1, 5.0
        self.lmin, self.lmax = 0.01, 3.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_freq  = torch.clamp(F.softplus(self.raw_freq), self.fmin, self.fmax)
        base_lam   = torch.clamp(F.softplus(self.raw_lambda), self.lmin, self.lmax)
        base_phase = torch.clamp(self.phase, -10.0, 10.0)
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        freq  = torch.clamp(base_freq  + mod.get('freq_offset', 0.0), self.fmin, self.fmax)
        lam   = torch.clamp(base_lam   + mod.get('lambda_offset', 0.0), self.lmin, self.lmax)
        phase = torch.clamp(base_phase + mod.get('phase_offset', 0.0), -10.0, 10.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        damp = 1 / (1 + lam * torch.abs(xm))
        return torch.clamp(F.softplus(scale * damp * (torch.sin(freq * xm + phase) + 1)), 0.0, 100.0)

class PiecewiseLinearOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_k1 = nn.Parameter(torch.tensor(0.0))
        self.raw_k2 = nn.Parameter(torch.tensor(0.0))
        self.threshold = nn.Parameter(torch.tensor(0.0))
        self.kmin, self.kmax = 0.01, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_k1 = torch.clamp(F.softplus(self.raw_k1), self.kmin, self.kmax)
        base_k2 = torch.clamp(F.softplus(self.raw_k2), self.kmin, self.kmax)
        base_thr = torch.clamp(self.threshold, -5.0, 5.0)
        k1 = torch.clamp(base_k1 + mod.get('k1_offset', 0.0), self.kmin, self.kmax)
        k2 = torch.clamp(base_k2 + mod.get('k2_offset', 0.0), self.kmin, self.kmax)
        thr= torch.clamp(base_thr + mod.get('threshold_offset', 0.0), -5.0, 5.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        left = k1 * xm
        right = k1 * thr + k2 * (xm - thr)
        out = torch.where(xm <= thr, left, right)
        return torch.clamp(F.softplus(out), 0.0, 100.0)

class ParameterModulator(nn.Module):
    def __init__(self, context_dim, param_configs):
        super().__init__()
        self.param_configs = param_configs
        self.param_predictors = nn.ModuleDict()
        for name, info in param_configs.items():
            dim = info.get('dim', 1)
            self.param_predictors[name] = nn.Sequential(
                nn.Linear(context_dim, 64), nn.ReLU(), nn.Dropout(0.1),
                nn.Linear(64, 32), nn.ReLU(),
                nn.Linear(32, dim), nn.Tanh()
            )
    def forward(self, context):
        global_context = context.mean(dim=1)  # (B, context_dim)
        modulated = {}
        for name, predictor in self.param_predictors.items():
            info = self.param_configs[name]
            raw_off = predictor(global_context)
            scale = info.get('scale', 0.1)
            modulated[name] = raw_off * scale
        return modulated

class LiquidWeightGenerator(nn.Module):
    def __init__(self, h_dim, x_dim, n_ops, hidden_dim=64, tau_min=0.2, tau_max=1.2):
        super().__init__()
        self.n_ops = n_ops
        self.tau_min, self.tau_max = tau_min, tau_max
        self.h_feature_net = nn.Sequential(nn.Linear(h_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(0.1))
        self.x_feature_net = nn.Sequential(nn.Linear(x_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(0.1))
        self.temporal_encoder = nn.Sequential(nn.Linear(3, hidden_dim // 4), nn.ReLU())
        combined_dim = hidden_dim + hidden_dim // 4
        self.feature_fusion = nn.Sequential(
            nn.Linear(combined_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.op_branches = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 4), nn.ReLU(), nn.Linear(hidden_dim // 4, 1))
            for _ in range(n_ops)
        ])
        self.temp_predictor = nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 4), nn.ReLU(), nn.Linear(hidden_dim // 4, 1))
        self.global_bias_scale = 0.0  # Disabled random bias injection
        for branch in self.op_branches:
            nn.init.xavier_uniform_(branch[0].weight)
            nn.init.xavier_uniform_(branch[2].weight)

    def forward(self, h_multi, x, is_after=False):
        B, T, h_dim = h_multi.shape
        _, _, x_dim = x.shape
        h_features = self.h_feature_net(h_multi)
        x_features = self.x_feature_net(x)
        t_norm = torch.arange(T, device=x.device, dtype=x.dtype).unsqueeze(0).expand(B, -1) / max(T-1, 1)
        dt = torch.zeros_like(t_norm)
        dt[:, 1:] = t_norm[:, 1:] - t_norm[:, :-1]
        phase = torch.sin(2 * 3.14159 * t_norm)
        temporal_input = torch.stack([t_norm, dt, phase], dim=-1)
        temporal_features = self.temporal_encoder(temporal_input)
        combined = torch.cat([h_features, x_features, temporal_features], dim=-1)
        fused = self.feature_fusion(combined)
        raw_logits = []
        for branch in self.op_branches:
            raw_logits.append(branch(fused).squeeze(-1))
        raw_weights = torch.stack(raw_logits, dim=-1)                     # (B, T, K)

        # NaN protection
        raw_weights = torch.nan_to_num(raw_weights, nan=0.0, posinf=1e6, neginf=-1e6)
        raw_weights = raw_weights - raw_weights.mean(dim=-1, keepdim=True)

        raw_temp = self.temp_predictor(fused).squeeze(-1)                 # (B, T)
        raw_temp = torch.nan_to_num(raw_temp, nan=0.0, posinf=1e6, neginf=-1e6)

        # Removed adaptive temperature boost for after-maintenance
        temperature = torch.clamp(F.softplus(raw_temp) + self.tau_min, self.tau_min, self.tau_max)

        weights = F.softmax(raw_weights / temperature.unsqueeze(-1), dim=-1)
        return weights, temperature

class CustomKAN(nn.Module):
    def __init__(self, ops, h_dim, x_dim):
        super().__init__()
        self.ops = nn.ModuleList(ops)
        self.n_ops = len(ops)
        self.weight_generator = LiquidWeightGenerator(h_dim=h_dim, x_dim=x_dim, n_ops=self.n_ops, hidden_dim=128)
        self.op_feature_extractors = nn.ModuleList([
            nn.Sequential(nn.Linear(h_dim, h_dim), nn.ReLU(), nn.Linear(h_dim, h_dim))
            for _ in range(self.n_ops)
        ])
        self.param_modulators = nn.ModuleList()
        context_dim = h_dim + x_dim
        for op in self.ops:
            param_configs = self._get_param_configs_for_op(op)
            if param_configs:
                modulator = ParameterModulator(context_dim, param_configs)
                self.param_modulators.append(modulator)
                op.set_param_modulator(modulator)
            else:
                self.param_modulators.append(None)
        self.raw_gain = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))

    def _get_param_configs_for_op(self, op):
        if isinstance(op, MonotonicLinearOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.2}, 'bias_offset': {'dim': 1, 'scale': 0.3}}
        if isinstance(op, MonotonicFlatOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.1}, 'bias_offset': {'dim': 1, 'scale': 0.2}}
        if isinstance(op, ConcaveLogOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.2}, 'eps_offset': {'dim': 1, 'scale': 0.01}}
        if isinstance(op, SaturationSigmoidOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.2}, 'slope_offset': {'dim': 1, 'scale': 0.3}, 'bias_offset': {'dim': 1, 'scale': 0.3}}
        if isinstance(op, HingeReLUOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.2}, 'threshold_offset': {'dim': 1, 'scale': 0.3}}
        if isinstance(op, PolynomialOp):
            return {'coeff_offset': {'dim': op.deg, 'scale': 0.2}}
        if isinstance(op, DampedSinOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.2}, 'freq_offset': {'dim': 1, 'scale': 0.3},
                    'lambda_offset': {'dim': 1, 'scale': 0.2}, 'phase_offset': {'dim': 1, 'scale': 0.5}}
        if isinstance(op, PiecewiseLinearOp):
            return {'k1_offset': {'dim': 1, 'scale': 0.2}, 'k2_offset': {'dim': 1, 'scale': 0.2},
                    'threshold_offset': {'dim': 1, 'scale': 0.3}}
        return {}

    def forward(self, h_multi, x, is_after=False):
        op_features = [ext(h_multi) for ext in self.op_feature_extractors]
        context = torch.cat([h_multi, x], dim=-1)
        outs = [op(op_features[i], context) for i, op in enumerate(self.ops)]
        Tm = min(o.size(1) for o in outs)
        outs = [o[:, :Tm] for o in outs]
        h_multi_aligned = h_multi[:, :Tm, :]
        x_aligned = x[:, :Tm, :]
        op_stack = torch.stack(outs, dim=-1)  # (B, Tm, K)
        weights, temperature = self.weight_generator(h_multi_aligned, x_aligned, is_after=is_after)
        damage = torch.sum(op_stack * weights, dim=-1)  # (B, Tm)
        damage = torch.clamp(damage, 0.0, 100.0)
        gain = torch.clamp(F.softplus(self.raw_gain), 0.1, 2.0)
        bias_val = torch.clamp(self.bias, -2.0, 2.0)
        damage = torch.clamp(gain * damage + bias_val, 0.0, 100.0)
        return damage, weights, temperature

class TrendEncoder(nn.Module):
    """
    Vectorized monotonic decreasing HI projection with learnable nonlinear decay.
    Enhanced with plateau-specific regularization for after-maintenance sequences.
    """
    def __init__(self, in_ch, trend_ch=4):
        super().__init__()
        self.boltz = BoltzmannKAN(in_ch, out_ch=trend_ch)
        ops = [MonotonicLinearOp(), MonotonicFlatOp(), ConcaveLogOp(),
               SaturationSigmoidOp(), HingeReLUOp(), PolynomialOp(),
               DampedSinOp(), PiecewiseLinearOp()]
        self.customkan = CustomKAN(ops, h_dim=trend_ch, x_dim=in_ch)

        self.proj_gain = nn.Parameter(torch.tensor(1.0))
        self.proj_bias = nn.Parameter(torch.tensor(0.0))

        self.health_aware_transform = nn.Sequential(
            nn.Linear(in_ch, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )

        # Learnable nonlinear decay rate predictor
        self.rate_head = nn.Sequential(
            nn.Linear(in_ch + 1, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )
        self.rate_scale = nn.Parameter(torch.tensor(0.1))  # Global decay intensity

        # Reduced linear enforcer influence
        self.linear_enforcer = nn.Sequential(
            nn.Linear(in_ch, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )

        # Steady-state level predictor for plateau targeting
        self.steady_state_head = nn.Sequential(
            nn.Linear(in_ch, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Sigmoid()
        )

    def forward(self, x, is_after=False):  # x:(B,T,C)
        B, T, C = x.shape
        h_multi = self.boltz(x)                               # (B,T,trend_ch)
        damage, weights, temperature = self.customkan(h_multi, x, is_after=is_after)

        x_reshaped = x.view(-1, C)
        health_direct = self.health_aware_transform(x_reshaped).view(B, T)  # (B,T)

        t = torch.arange(T, device=x.device, dtype=x.dtype).unsqueeze(0).expand(B, -1)
        t_normalized = t / (T - 1 + 1e-6)

        # Learnable nonlinear decay with plateau-aware rate control
        time_feat = t_normalized.unsqueeze(-1)                 # (B,T,1)
        rate_in = torch.cat([x, time_feat], dim=-1)           # (B,T,C+1)
        raw_rate = self.rate_head(rate_in).squeeze(-1)         # (B,T)
        rate = F.softplus(raw_rate) * torch.clamp(self.rate_scale, 1e-3, 2.0)

        # For after-maintenance, encourage rate to approach zero in later stages
        if is_after:
            tau = int(0.3 * T)  # 30% cutoff for plateau region
            if T > tau:
                late_stage_penalty = torch.zeros_like(rate)
                late_indices = torch.arange(T, device=rate.device) >= tau
                late_stage_penalty[:, late_indices] = rate[:, late_indices] * 2.0  # Penalty for high rates in plateau region
                rate = rate - late_stage_penalty
                rate = torch.clamp(rate, 1e-6, None)  # Ensure positive rates

        cum_rate = torch.cumsum(rate, dim=1)                   # (B,T)
        nonlinear_decay = torch.exp(-cum_rate)                 # Monotonic decreasing and flexible

        g = torch.clamp(F.softplus(self.proj_gain), 0.1, 5.0)
        b = torch.clamp(self.proj_bias, -5.0, 5.0)
        damage_normalized = torch.sigmoid(g*(damage + b))

        combined_health = health_direct * (1 - 0.3 * damage_normalized)

        # Reduced linear channel influence with time-dependent weighting
        lw = self.linear_enforcer(x_reshaped).view(B, T)
        linear_weight = 0.2 * lw * (1.0 - t_normalized)   # Early stages more linear, later stages more network-driven

        hi = linear_weight * nonlinear_decay + (1 - linear_weight) * combined_health

        # ---- Vectorized strictly non-increasing enforcement with adaptive epsilon ----
        if is_after:
            # Smaller epsilon step for after-maintenance to allow for plateaus
            eps = 5e-4
        else:
            eps = 1e-3
        idx = torch.arange(T, device=hi.device, dtype=hi.dtype).unsqueeze(0)  # (1, T)
        hi_mon, _ = torch.cummin(hi + eps * idx, dim=1)
        hi = hi_mon - eps * idx
        # -------------------------------------------------------------------------

        # Predict steady-state level for plateau targeting
        x_global = x.mean(dim=1)  # (B, C) - global sensor statistics
        steady_level = self.steady_state_head(x_global)  # (B, 1)

        return hi, h_multi, weights, temperature, steady_level

class ReconHead(nn.Module):
    def __init__(self, C, hidden=128):
        super().__init__()
        self.gru = nn.GRU(input_size=C+1, hidden_size=hidden, batch_first=True)
        self.out = nn.Linear(hidden, C)
    def forward(self, x_in, h_in):
        B,T,C = x_in.shape
        h_in_clamped = torch.clamp(h_in, 0.0, 10.0)
        feat = torch.cat([x_in, h_in_clamped.unsqueeze(-1)], dim=-1)
        H,_ = self.gru(feat)
        y = self.out(H)
        return y

class MaintClassifier(nn.Module):
    def __init__(self, sensor_dim, hidden=64, n_classes=14):
        super().__init__()
        self.hi_mlp = nn.Sequential(
            nn.Linear(6, hidden), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden//2)
        )
        self.sensor_mlp = nn.Sequential(
            nn.Linear(sensor_dim * 2, hidden), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden//2)
        )
        self.final_classifier = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, n_classes)
        )

    def forward(self, h_b, h_a, x_b, x_a, mask):
        m = mask
        T = m.size(1)
        valid = m.sum(1, keepdim=True).clamp_min(1.0)

        db = h_a - h_b
        mean_d = (db*m).sum(1,keepdim=True)/valid
        var_d  = (((db-mean_d)*m)**2).sum(1,keepdim=True)/valid
        std_d  = (var_d + 1e-8).sqrt()
        d0     = (db[:, :1])
        dmax   = (db.masked_fill(m==0, -1e9)).max(dim=1, keepdim=True).values
        pos_ratio = ((db>0).float()*m).sum(1,keepdim=True)/valid

        t = torch.arange(T, device=h_b.device, dtype=h_b.dtype)
        t = (t - t.mean())/(t.std()+1e-6)
        def slope(x):
            num = (x*t).sum(1) - x.sum(1)*t.sum()/T
            den = (t**2).sum() - (t.sum()**2)/T + 1e-8
            return (num/den).unsqueeze(1)
        slope_diff = slope(h_a) - slope(h_b)

        hi_feat = torch.cat([mean_d, d0, dmax, std_d, slope_diff, pos_ratio], dim=1)
        hi_feat = torch.clamp(hi_feat, -10.0, 10.0)
        hi_features = self.hi_mlp(hi_feat)

        x_b_mean = (x_b * m.unsqueeze(-1)).sum(1) / valid
        x_a_mean = (x_a * m.unsqueeze(-1)).sum(1) / valid
        sensor_change = torch.cat([x_b_mean, x_a_mean], dim=1)
        sensor_features = self.sensor_mlp(sensor_change)

        combined = torch.cat([hi_features, sensor_features], dim=1)
        logits = self.final_classifier(combined)
        return logits

def sanitize_tensors(*tensors):
    result = []
    for t in tensors:
        if t is not None:
            t_clean = torch.nan_to_num(t, nan=0.0, posinf=1e6, neginf=-1e6)
            result.append(t_clean)
        else:
            result.append(t)
    return result[0] if len(result) == 1 else result

class DiffAwareReconstructor(nn.Module):
    """
    - Two encoders share parameters: encode x_before / x_after → h_b / h_a (monotonic decreasing HI)
    - Two reconstruction heads: one-step recursive reconstruction
    - Classification head: ΔHI + sensor change features
    """
    def __init__(self, in_ch, trend_ch=4, hidden=128, n_classes=14):
        super().__init__()
        self.encoder = TrendEncoder(in_ch, trend_ch)
        self.recon_b = ReconHead(in_ch, hidden)
        self.recon_a = ReconHead(in_ch, hidden)
        self.clf     = MaintClassifier(sensor_dim=in_ch, hidden=64, n_classes=n_classes)
        self.consistency_weight = nn.Parameter(torch.tensor(1.0))

    def forward(self, x_b, x_a, mask):
        h_b, h_multi_b, weights_b, temp_b, steady_b = self.encoder(x_b, is_after=False)
        h_a, h_multi_a, weights_a, temp_a, steady_a = self.encoder(x_a, is_after=True)

        h_b, h_a = sanitize_tensors(h_b, h_a)
        weights_b, weights_a = sanitize_tensors(weights_b, weights_a)
        steady_b, steady_a = sanitize_tensors(steady_b, steady_a)

        xb_in, hb_in = x_b[:, :-2], h_b[:, :-2]
        xa_in, ha_in = x_a[:, :-2], h_a[:, :-2]
        yb_hat = self.recon_b(xb_in, hb_in)   # (B, L-2, C) ≈ x_b[:,1:L-1]
        ya_hat = self.recon_a(xa_in, ha_in)   # (B, L-2, C) ≈ x_a[:,1:L-1]

        yb_hat, ya_hat = sanitize_tensors(yb_hat, ya_hat)
        logits = self.clf(h_b, h_a, x_b, x_a, mask)
        logits = sanitize_tensors(logits)
        return yb_hat, ya_hat, h_b, h_a, logits, weights_b, weights_a, temp_b, temp_a, steady_b, steady_a

# ---------------------------------------------------------------------
# Losses / Regularizers
# ---------------------------------------------------------------------
def masked_mse(a, b, mask):
    diff = (a - b)**2
    mse  = (diff.mean(-1) * mask).sum() / (mask.sum() + 1e-6)
    return mse

def masked_mse_batchwise(a, b, mask):
    diff = (a - b)**2  # (B, T, C)
    mse_t = diff.mean(-1) * mask  # (B, T)
    denom = mask.sum(1).clamp_min(1.0)  # (B,)
    return mse_t.sum(1) / denom

def slope_loss(h, mask, kind="smooth"):
    if kind=="smooth":
        d2 = h[:,2:] - 2*h[:,1:-1] + h[:,:-2]
        m  = mask[:,2:]
        return (d2.abs() * m).sum() / (m.sum()+1e-6)
    elif kind=="mono_dec":
        dh = h[:,1:] - h[:,:-1]
        m  = mask[:,1:]
        return (F.relu(dh) * m).sum() / (m.sum()+1e-6)
    elif kind=="tv":
        # Total variation (first-order differences) - allows more curvature
        dh = h[:,1:] - h[:,:-1]
        m  = mask[:,1:]
        return (dh.abs() * m).sum() / (m.sum()+1e-6)
    else:
        return torch.tensor(0., device=h.device)

def plateau_tv_loss(h_a, mask, tau_ratio=0.3):
    """
    Time-weighted total variation loss for the plateau region (later 70% of sequence).
    Encourages h_a to become flat in the later stages.
    """
    B, T = h_a.shape
    tau = int(tau_ratio * T)

    if T <= tau + 1:
        return torch.tensor(0.0, device=h_a.device)

    # Focus on the plateau region (later 70%)
    h_plateau = h_a[:, tau:]  # (B, T-tau)
    mask_plateau = mask[:, tau:]  # (B, T-tau)

    if h_plateau.size(1) <= 1:
        return torch.tensor(0.0, device=h_a.device)

    # First-order differences in plateau region
    dh = h_plateau[:, 1:] - h_plateau[:, :-1]  # (B, T-tau-1)
    mask_diff = mask_plateau[:, 1:]  # (B, T-tau-1)

    # Time-increasing weights (later time steps get higher penalty)
    T_plateau = h_plateau.size(1) - 1
    if T_plateau > 0:
        time_weights = torch.linspace(1.0, 3.0, T_plateau, device=h_a.device).unsqueeze(0)  # (1, T-tau-1)
        weighted_tv = (dh.abs() * time_weights * mask_diff).sum() / (mask_diff.sum() + 1e-6)
    else:
        weighted_tv = torch.tensor(0.0, device=h_a.device)

    return weighted_tv

def plateau_curvature_loss(h_a, mask, tau_ratio=0.3):
    """
    Time-weighted second-order difference loss for the plateau region.
    Reduces curvature/oscillations in the later stages.
    """
    B, T = h_a.shape
    tau = int(tau_ratio * T)

    if T <= tau + 2:
        return torch.tensor(0.0, device=h_a.device)

    # Focus on the plateau region
    h_plateau = h_a[:, tau:]  # (B, T-tau)
    mask_plateau = mask[:, tau:]  # (B, T-tau)

    if h_plateau.size(1) <= 2:
        return torch.tensor(0.0, device=h_a.device)

    # Second-order differences in plateau region
    d2h = h_plateau[:, 2:] - 2 * h_plateau[:, 1:-1] + h_plateau[:, :-2]  # (B, T-tau-2)
    mask_d2 = mask_plateau[:, 2:]  # (B, T-tau-2)

    # Time-increasing weights
    T_plateau = h_plateau.size(1) - 2
    if T_plateau > 0:
        time_weights = torch.linspace(1.0, 2.0, T_plateau, device=h_a.device).unsqueeze(0)  # (1, T-tau-2)
        weighted_curv = (d2h.abs() * time_weights * mask_d2).sum() / (mask_d2.sum() + 1e-6)
    else:
        weighted_curv = torch.tensor(0.0, device=h_a.device)

    return weighted_curv

def steady_state_loss(h_a, steady_level, mask, tau_ratio=0.3):
    """
    Encourage h_a to approach the predicted steady-state level in the plateau region.
    """
    B, T = h_a.shape
    tau = int(tau_ratio * T)

    if T <= tau:
        return torch.tensor(0.0, device=h_a.device)

    # Focus on the plateau region
    h_plateau = h_a[:, tau:]  # (B, T-tau)
    mask_plateau = mask[:, tau:]  # (B, T-tau)

    # Expand steady_level to match plateau dimensions
    steady_expanded = steady_level.expand(-1, h_plateau.size(1))  # (B, T-tau)

    # L2 loss to steady state
    steady_loss = ((h_plateau - steady_expanded) ** 2 * mask_plateau).sum() / (mask_plateau.sum() + 1e-6)

    return steady_loss

def flat_operator_prior_loss(weights_a, mask, tau_ratio=0.3, flat_op_indices=[1, 3]):
    """
    Encourage flat/saturation operators to have higher weights in the plateau region.
    flat_op_indices: indices of MonotonicFlatOp and SaturationSigmoidOp
    """
    B, T, K = weights_a.shape
    tau = int(tau_ratio * T)

    if T <= tau:
        return torch.tensor(0.0, device=weights_a.device)

    # Focus on plateau region
    weights_plateau = weights_a[:, tau:, :]  # (B, T-tau, K)
    mask_plateau = mask[:, tau:]  # (B, T-tau)

    # Target distribution: higher probability for flat operators
    target_dist = torch.ones(K, device=weights_a.device) / K
    for idx in flat_op_indices:
        if idx < K:
            target_dist[idx] *= 2.0  # Boost flat operators
    target_dist = target_dist / target_dist.sum()  # Renormalize

    # KL divergence from current weights to target distribution
    eps = 1e-8
    weights_plateau_norm = weights_plateau + eps
    target_expanded = target_dist.unsqueeze(0).unsqueeze(0).expand_as(weights_plateau_norm)

    kl_div = (weights_plateau_norm * torch.log(weights_plateau_norm / target_expanded)).sum(-1)  # (B, T-tau)
    kl_loss = (kl_div * mask_plateau).sum() / (mask_plateau.sum() + 1e-6)

    return kl_loss

def enhanced_liquid_weight_regularization(weights_b, weights_a, mask,
                                          lambda_tv=0.05, lambda_ent=0.05,
                                          lambda_bal=0.0, lambda_div=0.05):
    total_loss = torch.tensor(0.0, device=weights_b.device)

    def tv_loss(weights, mask):
        if weights.size(1) <= 1:
            return torch.tensor(0.0, device=weights.device)
        diff = weights[:, 1:, :] - weights[:, :-1, :]
        mask_diff = mask[:, 1:]
        tv = (diff.abs().sum(-1) * mask_diff).sum() / (mask_diff.sum() + 1e-6)
        return tv

    tv_loss_b = tv_loss(weights_b, mask)
    tv_loss_a = tv_loss(weights_a, mask)
    total_loss += lambda_tv * (tv_loss_b + tv_loss_a)

    def entropy_loss(weights, mask):
        eps = 1e-8
        entropy = -(weights * torch.log(weights + eps)).sum(-1)
        # Encourage lower entropy (sharper distributions)
        target_entropy = np.log(weights.size(-1)) * 0.3
        entropy_penalty = F.relu(entropy - target_entropy)  # Penalty for high entropy
        return (entropy_penalty * mask).sum() / (mask.sum() + 1e-6)

    ent_loss_b = entropy_loss(weights_b, mask)
    ent_loss_a = entropy_loss(weights_a, mask)
    total_loss += lambda_ent * (ent_loss_b + ent_loss_a)

    def balance_loss(weights, mask):
        if lambda_bal == 0.0:
            return torch.tensor(0.0, device=weights.device)
        valid_weights = weights * mask.unsqueeze(-1)  # (B, T, K)
        mean_usage = valid_weights.sum([0, 1]) / (mask.sum() + 1e-6)  # (K,)
        target_usage = 1.0 / weights.size(-1)
        balance = ((mean_usage - target_usage) ** 2).sum()
        return balance

    bal_loss_b = balance_loss(weights_b, mask)
    bal_loss_a = balance_loss(weights_a, mask)
    total_loss += lambda_bal * (bal_loss_b + bal_loss_a)

    def diversity_loss(weights, mask):
        B, T, K = weights.shape
        if T <= 1:
            return torch.tensor(0.0, device=weights.device)
        valid_weights = weights * mask.unsqueeze(-1)
        weights_flat = valid_weights.view(-1, K)
        weights_centered = weights_flat - weights_flat.mean(0, keepdim=True)
        cov = torch.mm(weights_centered.T, weights_centered) / (weights_flat.size(0) - 1 + 1e-6)
        mask_offdiag = torch.ones_like(cov) - torch.eye(K, device=cov.device)
        corr_penalty = (cov.abs() * mask_offdiag).sum()
        return corr_penalty

    div_loss_b = diversity_loss(weights_b, mask)
    div_loss_a = diversity_loss(weights_a, mask)
    total_loss += lambda_div * (div_loss_b + div_loss_a)
    return total_loss

def smoothness_enhancement_loss_separated(h_b, h_a, mask, order=2, weight_b=0.3, weight_a=1.0):
    """
    Separated smoothness loss: lower weight for h_b, higher weight for h_a.
    """
    def compute_smoothness(h, mask, order):
        if order == 2:
            if h.size(1) <= 2:
                return torch.tensor(0.0, device=h.device)
            d2 = h[:,2:] - 2*h[:,1:-1] + h[:,:-2]
            m = mask[:,2:]
            return (d2.abs() * m).sum() / (m.sum() + 1e-6)
        elif order == 3:
            if h.size(1) <= 3:
                return torch.tensor(0.0, device=h.device)
            d3 = h[:,3:] - 3*h[:,2:-1] + 3*h[:,1:-2] - h[:,:-3]
            m = mask[:,3:]
            return (d3.abs() * m).sum() / (m.sum() + 1e-6)
        else:
            return torch.tensor(0., device=h.device)

    smooth_b = compute_smoothness(h_b, mask, order)
    smooth_a = compute_smoothness(h_a, mask, order)

    return weight_b * smooth_b + weight_a * smooth_a

def compute_sample_weights(matching_costs, alpha=0.5):
    return 1.0 / (1.0 + alpha * matching_costs)

def maintenance_effect_loss(h_b, h_a, mask, lambda_diff=1.0, lambda_smooth_after=2.5, lambda_level_constraint=3.0):
    """
    Enhanced maintenance effect loss with stronger after-maintenance smoothing.
    """
    total_loss = torch.tensor(0.0, device=h_b.device)

    # 1. Encourage difference between before and after HI (improved stability)
    hi_diff = torch.abs(h_a - h_b)  # (B, T)
    hi_diff_clamped = torch.clamp(hi_diff, min=1e-3)  # Prevent extreme gradients
    diff_loss = F.softplus(-10.0 * hi_diff_clamped)  # Smooth penalty for small differences
    diff_loss = (diff_loss * mask).sum() / (mask.sum() + 1e-6)
    total_loss += lambda_diff * diff_loss

    # 2. Encourage after-maintenance HI to be smoother (INCREASED WEIGHT)
    if h_a.size(1) > 1:
        h_a_diff = torch.abs(h_a[:, 1:] - h_a[:, :-1])  # (B, T-1)
        mask_diff = mask[:, 1:]
        smooth_after_loss = (h_a_diff * mask_diff).sum() / (mask_diff.sum() + 1e-6)
        total_loss += lambda_smooth_after * smooth_after_loss

    # 3. Encourage min(h_a) > max(h_b) for proper level separation
    h_b_masked = h_b.masked_fill(mask == 0, -1e9)  # Fill invalid positions with very low values
    h_a_masked = h_a.masked_fill(mask == 0, 1e9)   # Fill invalid positions with very high values

    h_b_max = h_b_masked.max(dim=1)[0]  # (B,) - max of each sequence
    h_a_min = h_a_masked.min(dim=1)[0]  # (B,) - min of each sequence

    # Encourage h_a_min > h_b_max with a margin
    margin = 0.1
    level_constraint = F.relu(h_b_max - h_a_min + margin)  # (B,)
    level_constraint_loss = level_constraint.mean()
    total_loss += lambda_level_constraint * level_constraint_loss

    # Pointwise gap constraint
    margin_pt = 0.05
    pointwise_gap = (h_a - h_b) + (-margin_pt)
    pointwise_loss = F.relu(-pointwise_gap)      # Penalty when h_a < h_b + margin_pt
    pointwise_loss = (pointwise_loss * mask).sum() / (mask.sum() + 1e-6)
    total_loss += 1.0 * pointwise_loss

    return total_loss

def dynamic_plateau_loss(h_b, h_a, steady_a, mask, k=10.0, m=0.05):
    """
    Dynamic plateau loss that activates when maintenance effect is significant.
    """
    # Calculate maintenance effect magnitude
    delta_mean = ((h_a - h_b) * mask).sum(1, keepdim=True) / (mask.sum(1, keepdim=True) + 1e-6)  # (B, 1)

    # Activation function: stronger plateau enforcement when delta is large
    gamma = torch.sigmoid(k * (delta_mean - m))  # (B, 1)

    # Plateau losses
    tv_loss = plateau_tv_loss(h_a, mask)
    curv_loss = plateau_curvature_loss(h_a, mask)
    steady_loss = steady_state_loss(h_a, steady_a, mask)

    # Weight by activation
    weighted_plateau_loss = gamma.mean() * (tv_loss + 0.5 * curv_loss + steady_loss)

    return weighted_plateau_loss

# ---------------------------------------------------------------------
# Evaluation / Prediction
# ---------------------------------------------------------------------
@torch.no_grad()
def eval_epoch(model, loader, device, K, compute_acc=True):
    model.eval()
    total = {"mse_b":0.0, "mse_a":0.0, "acc":0.0, "delta_mean":0.0}
    n_batch=0
    n_acc = 0; n_tot=0
    for batch in loader:
        xb, xa, labels, mask, lengths, sample_ids, matching_costs = adapt_batch(batch, device)
        m_tgt  = (torch.arange(xb.size(1)-2, device=device)[None,:] < (lengths-2)[:,None]).float()
        yb_hat, ya_hat, h_b, h_a, logits, _, _, _, _, _, _ = model(xb, xa, mask)
        yb_hat, ya_hat, h_b, h_a, logits = sanitize_tensors(yb_hat, ya_hat, h_b, h_a, logits)

        # Ensure logits dimension matches expected K
        if logits.shape[-1] != K:
            print(f"Warning: Logits dim {logits.shape[-1]} != expected K={K}, adjusting...")
            if logits.shape[-1] < K:
                # Pad with zeros
                padding = torch.zeros(logits.shape[0], K - logits.shape[-1], device=logits.device)
                logits = torch.cat([logits, padding], dim=1)
            else:
                # Truncate
                logits = logits[:, :K]

        yb_tgt = xb[:,1:-1,:]
        ya_tgt = xa[:,1:-1,:]
        mse_b = masked_mse(yb_hat, yb_tgt, m_tgt)
        mse_a = masked_mse(ya_hat, ya_tgt, m_tgt)
        if compute_acc and labels is not None:
            # Filter out ignore_index labels
            valid_labels = labels != -100
            if valid_labels.any():
                pred = logits[valid_labels].argmax(dim=1)
                labels_valid = labels[valid_labels]
                n_acc += (pred == labels_valid).sum().item()
                n_tot += labels_valid.numel()
        valid = mask.sum(1, keepdim=True).clamp_min(1.0)
        delta_mean = ((h_a - h_b)*mask).sum(1,keepdim=True)/valid
        delta_mean = torch.nan_to_num(delta_mean, nan=0.0)
        total["delta_mean"] += delta_mean.mean().item()
        total["mse_b"] += mse_b.item()
        total["mse_a"] += mse_a.item()
        n_batch += 1
    for k in ["mse_b","mse_a","delta_mean"]:
        total[k] /= max(n_batch,1)
    total["acc"] = (n_acc / max(n_tot,1)) if n_tot>0 else 0.0
    return total

@torch.no_grad()
def collect_test_predictions(model, loader, device, max_curve_keep=24, K=14, id_to_name=None):
    model.eval()
    y_true, y_pred = [], []
    all_delta_mean, all_sample_ids = [], []
    keep_curves = []
    for batch in loader:
        xb, xa, labels, mask, lengths, sample_ids, matching_costs = adapt_batch(batch, device)
        yb_hat, ya_hat, h_b, h_a, logits, weights_b, weights_a, temp_b, temp_a, steady_b, steady_a = model(xb, xa, mask)
        yb_hat, ya_hat, h_b, h_a, logits = sanitize_tensors(yb_hat, ya_hat, h_b, h_a, logits)

        # Ensure logits dimension matches expected K
        if logits.shape[-1] != K:
            if logits.shape[-1] < K:
                padding = torch.zeros(logits.shape[0], K - logits.shape[-1], device=logits.device)
                logits = torch.cat([logits, padding], dim=1)
            else:
                logits = logits[:, :K]

        prob = F.softmax(logits, dim=1)     # (B,K)
        pred = prob.argmax(1)               # (B,)

        valid = mask.sum(1, keepdim=True).clamp_min(1.0)
        delta_mean = ((h_a - h_b) * mask).sum(1, keepdim=True) / valid  # (B,1)
        delta_mean = torch.nan_to_num(delta_mean, nan=0.0)

        if labels is not None:
            # Filter out ignore_index labels
            valid_labels = labels != -100
            if valid_labels.any():
                y_true.append(labels[valid_labels].detach().cpu().numpy())
                y_pred.append(pred[valid_labels].detach().cpu().numpy())
        all_delta_mean.append(delta_mean.squeeze(1).cpu().numpy())
        all_sample_ids.extend(sample_ids)

        for i in range(xb.size(0)):
            if len(keep_curves) >= max_curve_keep: break
            L_i = int(lengths[i].item())
            L_recon = min(L_i - 2, yb_hat.size(1))
            keep_curves.append({
                "sample_id": sample_ids[i],
                "true": int(labels[i].cpu().item()) if labels is not None and labels[i] != -100 else -1,
                "pred": int(pred[i].cpu().item()),
                "prob": prob[i].cpu().numpy(),
                "h_before": h_b[i, :L_i].cpu().numpy(),
                "h_after":  h_a[i, :L_i].cpu().numpy(),
                "weights_before": weights_b[i, :L_i].cpu().numpy() if weights_b is not None else None,
                "weights_after": weights_a[i, :L_i].cpu().numpy() if weights_a is not None else None,
                "temp_before": temp_b[i, :L_i].cpu().numpy() if temp_b is not None else None,
                "temp_after": temp_a[i, :L_i].cpu().numpy() if temp_a is not None else None,
                "steady_before": steady_b[i].cpu().item() if steady_b is not None else None,
                "steady_after": steady_a[i].cpu().item() if steady_a is not None else None,
                "op_outputs_before": [],
                "op_outputs_after":  [],
                "yb_hat":   yb_hat[i, :L_recon].cpu().numpy(),
                "ya_hat":   ya_hat[i, :L_recon].cpu().numpy(),
                "x_before": xb[i, :L_i].cpu().numpy(),
                "x_after":  xa[i, :L_i].cpu().numpy(),
            })
    y_true = np.concatenate(y_true, axis=0) if len(y_true)>0 else np.array([])
    y_pred = np.concatenate(y_pred, axis=0) if len(y_pred)>0 else np.array([])
    all_delta_mean = np.concatenate(all_delta_mean, axis=0) if len(all_delta_mean)>0 else np.array([])
    return y_true, y_pred, all_delta_mean, all_sample_ids, keep_curves

# ---------------------------------------------------------------------
# Plotting
# ---------------------------------------------------------------------
def remove_constant_segments(hi_values, threshold=1e-6):
    if len(hi_values) <= 1:
        return hi_values, np.arange(len(hi_values))
    diffs = np.abs(np.diff(hi_values))
    valid_mask = np.ones(len(hi_values), dtype=bool)
    valid_mask[1:] = diffs > threshold
    if np.sum(valid_mask) < 2:
        valid_mask[0] = True
        valid_mask[-1] = True
    valid_indices = np.where(valid_mask)[0]
    return hi_values[valid_indices], valid_indices

def plot_hi_examples_aligned(curves, id_to_name, n_show=6, seed=0):
    if len(curves)==0:
        print("(No visualization samples)")
        return
    random.Random(seed).shuffle(curves)
    n_show = min(n_show, len(curves))
    cols = 3
    rows = int(np.ceil(n_show/cols))
    plt.figure(figsize=(cols*4.6, rows*3.2))
    for k in range(n_show):
        ex = curves[k]
        hb = ex["h_before"]; ha = ex["h_after"]
        L  = min(len(hb), len(ha))
        hb, ha = hb[:L], ha[:L]
        hb_clean, hb_indices = remove_constant_segments(hb)
        ha_clean, ha_indices = remove_constant_segments(ha)
        t_b = hb_indices
        t_a = ha_indices + L

        sample_id = ex["sample_id"]
        pred = ex["pred"]; true = ex["true"]; prob = ex["prob"]
        true_name = id_to_name.get(true, f"Class_{true}")
        pred_name = id_to_name.get(pred, f"Class_{pred}")

        plt.subplot(rows, cols, k+1)
        if len(hb_clean) > 1:
            plt.plot(t_b, hb_clean, label="Learned HI (Pre)", linewidth=1.8, marker='o', markersize=3)
        if len(ha_clean) > 1:
            plt.plot(t_a, ha_clean, label="Learned HI (Post)",  linewidth=1.8, linestyle='--', marker='s', markersize=3)
        plt.axvline(L-1, color='k', linestyle=':', linewidth=1.0, alpha=0.7)
        d_mean = float(np.mean(ha - hb))
        d_max  = float(np.max(ha - hb))

        # Add steady state info if available
        steady_info = ""
        if ex.get("steady_after") is not None:
            steady_info = f" | SS={ex['steady_after']:.3f}"

        plt.title(f"ID={sample_id}\nTrue={true_name} | Pred={pred_name} "
                  f"| p={prob[pred]:.2f}\nΔHI_mean={d_mean:.3f}, ΔHI_max={d_max:.3f}{steady_info}", fontsize=9)
        plt.xlabel("Cycle (Pre | Post)")
        plt.ylabel("Health Index (higher=better)")
        plt.grid(ls='--', alpha=.35)
        if k==0: plt.legend()
    plt.tight_layout(); plt.show()

def plot_enhanced_liquid_weights(curves, n_show=3, seed=0):
    if len(curves) == 0:
        print("(No visualization samples)")
        return
    curves_with_weights = [c for c in curves if c.get("weights_before") is not None]
    if len(curves_with_weights) == 0:
        print("(No weight information available)")
        return
    random.Random(seed).shuffle(curves_with_weights)
    n_show = min(n_show, len(curves_with_weights))
    op_names = ["MonotonicLinear", "MonotonicFlat", "ConcaveLog", "SaturationSigmoid",
                "HingeReLU", "Polynomial", "DampedSin", "PiecewiseLinear"]
    for i in range(n_show):
        ex = curves_with_weights[i]
        weights_b = ex["weights_before"]
        weights_a = ex["weights_after"]
        temp_b = ex.get("temp_before")
        temp_a = ex.get("temp_after")
        sample_id = ex["sample_id"]
        true_label = ex["true"]
        pred_label = ex["pred"]
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle(f"Dynamic Parameter Modulation - ID={sample_id}, True=Class_{true_label}, Pred=Class_{pred_label}", fontsize=14)
        ax = axes[0, 0]
        T_b, K = weights_b.shape
        for k in range(K):
            ax.plot(range(T_b), weights_b[:, k], label=op_names[k] if k < len(op_names) else f"Op_{k}",
                   marker='o', markersize=2, linewidth=1.5)
        ax.set_title("Pre-maintenance Weights"); ax.set_xlabel("Time Step"); ax.set_ylabel("Weight")
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left'); ax.grid(True, alpha=0.3)
        ax = axes[0, 1]
        T_a, _ = weights_a.shape
        for k in range(K):
            ax.plot(range(T_a), weights_a[:, k], label=op_names[k] if k < len(op_names) else f"Op_{k}",
                   marker='s', markersize=2, linewidth=1.5, linestyle='--')
        ax.set_title("Post-maintenance Weights"); ax.set_xlabel("Time Step"); ax.set_ylabel("Weight")
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left'); ax.grid(True, alpha=0.3)
        ax = axes[0, 2]
        entropy_b = -np.sum(weights_b * np.log(weights_b + 1e-8), axis=1)
        entropy_a = -np.sum(weights_a * np.log(weights_a + 1e-8), axis=1)
        ax.plot(range(T_b), entropy_b, label="Pre", marker='o', linewidth=2)
        ax.plot(range(T_a), entropy_a, label="Post", marker='s', linewidth=2, linestyle='--')
        ax.axhline(np.log(K), color='r', linestyle=':', label=f'Max Entropy ({np.log(K):.2f})')
        ax.set_title("Weight Entropy Over Time"); ax.set_xlabel("Time Step"); ax.set_ylabel("Entropy")
        ax.legend(); ax.grid(True, alpha=0.3)
        ax = axes[1, 0]
        if temp_b is not None and temp_a is not None:
            ax.plot(range(T_b), temp_b, label="Pre", marker='o', linewidth=2)
            ax.plot(range(T_a), temp_a, label="Post", marker='s', linewidth=2, linestyle='--')
            ax.set_title("Temperature Evolution"); ax.set_xlabel("Time Step"); ax.set_ylabel("Temperature"); ax.legend()
        else:
            ax.text(0.5, 0.5, "Temperature data\nnot available", ha='center', va='center', transform=ax.transAxes)
            ax.set_title("Temperature Evolution")
        ax.grid(True, alpha=0.3)
        ax = axes[1, 1]
        ax.text(0.5, 0.5, "Operator output variance\n(not collected)", ha='center', va='center', transform=ax.transAxes)
        ax.set_title("Operator Output Variance"); ax.grid(True, alpha=0.3)
        ax = axes[1, 2]
        combined_weights = np.concatenate([weights_b, weights_a], axis=0)
        # Apply smoothing for correlation stability
        from scipy.ndimage import uniform_filter1d
        smoothed_weights = uniform_filter1d(combined_weights, size=3, axis=0)
        corr_matrix = np.corrcoef(smoothed_weights.T)
        im = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
        ax.set_title("Operator Weight Correlation (Smoothed)")
        ax.set_xticks(range(K)); ax.set_yticks(range(K))
        short_names = [op_names[k][:8] if k < len(op_names) else f"Op_{k}" for k in range(K)]
        ax.set_xticklabels(short_names, rotation=45); ax.set_yticklabels(short_names)
        cbar = plt.colorbar(im, ax=ax); cbar.set_label('Correlation')
        plt.tight_layout(); plt.show()

def plot_liquid_weights(curves, n_show=3, seed=0):
    plot_enhanced_liquid_weights(curves, n_show, seed)

def plot_sensor_examples_aligned(curves, sensor_idx_list=None, n_cols=4):
    if len(curves)==0:
        print("(No visualization samples)")
        return
    ex = curves[0]
    xb = ex["x_before"]; xa = ex["x_after"]; ya = ex["ya_hat"]
    L, C = xb.shape
    Lhat = ya.shape[0]
    if sensor_idx_list is None:
        step = max(1, C//8); sensor_idx_list = list(range(0, C, step))[:8]
    n = len(sensor_idx_list)
    n_rows = int(np.ceil(n/n_cols))
    plt.figure(figsize=(n_cols*4.3, n_rows*3.1))
    for i, s in enumerate(sensor_idx_list):
        plt.subplot(n_rows, n_cols, i+1)
        t_b = np.arange(L); t_a = np.arange(L, 2*L)
        plt.plot(t_b, xb[:,s], label="Original (pre)", linewidth=1.2)
        plt.plot(t_a, xa[:,s], label="Original (post)",  linewidth=1.0, linestyle="--", alpha=0.7)
        t_pred = np.arange(L+1, L+1+Lhat)
        plt.plot(t_pred, ya[:,s], label="Post prediction", linewidth=1.6)
        plt.axvline(L-1, color='k', linestyle=':', linewidth='1.0')
        plt.title(f"Sensor_{s:02d}")
        if i==0: plt.legend()
        plt.grid(ls="--", alpha=.35)
    plt.tight_layout(); plt.show()

def topk_by_delta(df_delta, id_to_name, k=3):
    if len(df_delta)==0:
        return
    unique_classes = sorted(df_delta["true"].unique())
    for cls in unique_classes[:5]:
        sub = df_delta[df_delta["true"]==cls].sort_values("delta_hi_mean", ascending=False).head(k)
        class_name = id_to_name.get(cls, f"Class_{cls}")
        print(f"\n[True={class_name}] Top {k} samples with highest ΔHI_mean:")
        print(sub[["uid","delta_hi_mean","pred"]].reset_index(drop=True))

# ---------------------------------------------------------------------
# Training (separate) and Testing (separate)
# ---------------------------------------------------------------------
def build_model_and_setup(train_loader, device, class_weights=None):
    """
    Build model and setup training components.

    Args:
        train_loader: Training data loader
        device: Device to use
        class_weights: Pre-computed class weights tensor (optional)
    """
    # Determine K from training loader
    K = scan_num_classes_from_loader(train_loader)
    id_to_name = build_default_id_to_name(K)

    first_batch = next(iter(train_loader))
    C = first_batch["x_before"].shape[-1]

    print(f"\n[Setup] Sensor dim C={C}, Num classes K={K}")

    # Use provided class weights or create uniform weights
    if class_weights is not None:
        if isinstance(class_weights, np.ndarray):
            class_weights = torch.from_numpy(class_weights).float().to(device)
        elif isinstance(class_weights, torch.Tensor):
            class_weights = class_weights.float().to(device)

        # Ensure class_weights has the right size
        if len(class_weights) < K:
            # Pad with ones if needed
            padded_weights = torch.ones(K, dtype=torch.float32, device=device)
            padded_weights[:len(class_weights)] = class_weights
            class_weights = padded_weights
        elif len(class_weights) > K:
            # Truncate if needed
            class_weights = class_weights[:K]

        print(f"[Class Weights] Using provided weights: {class_weights.detach().cpu().numpy()}")
    else:
        class_weights = torch.ones(K, dtype=torch.float32, device=device)
        print(f"[Class Weights] Using uniform weights: {class_weights.detach().cpu().numpy()}")

    model = DiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=K).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    ce  = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100, reduction='none')

    return model, opt, ce, K, id_to_name, C


def train_only(train_loader, val_loader, epochs=300, lr=1e-3, device=None, patience=20,
               use_matching_cost=True, save_path="best_model.pth", class_weights=None,
               es_alpha=1.0):
    """
    Train and save the best model. Returns path to checkpoint and metadata.

    Args:
        class_weights: Pre-computed class weights (numpy array or torch tensor)
        es_alpha: Weight for accuracy in early stopping metric
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Build model and training setup
    model, opt, ce, K, id_to_name, C = build_model_and_setup(train_loader, device, class_weights)
    # Override LR if provided
    for g in opt.param_groups:
        g['lr'] = lr

    best_v = float("inf")
    best_state = None
    best_epoch = 0
    no_improve = 0
    nan_batches = 0

    print(f"\nStarting training for {epochs} epochs with early stopping (patience={patience})...")
    print(f"Device: {device}")
    print(f"Early stopping alpha: {es_alpha}")

    for ep in range(1, epochs+1):
        print(f"\n[Epoch {ep}/{epochs}] Training...")
        model.train()
        logs = {"mse_b":0.0, "mse_a":0.0, "smooth":0.0, "mono":0.0,
                "cls":0.0, "smooth_enh":0.0, "liquid_weight":0.0, "maintenance_effect":0.0,
                "plateau":0.0, "all":0.0}
        n_bt = 0
        batch_nan_count = 0

        progress = ep / epochs
        lambda_tv  = 0.03 * (1 - progress * 0.3)
        lambda_ent = 0.10 * (1 - progress)      # Linear annealing to 0
        lambda_bal = 0.0   # Disabled balance loss
        lambda_div = 0.05 * (1 - 0.5*progress)  # Light annealing

        # Gradient norm tracking
        grad_norms = []

        for batch_idx, batch in enumerate(train_loader):
            xb, xa, labels, mask, lengths, sample_ids, matching_costs = adapt_batch(batch, device)
            m_tgt  = (torch.arange(xb.size(1)-2, device=device)[None,:] < (lengths-2)[:,None]).float()
            yb_hat, ya_hat, h_b, h_a, logits, weights_b, weights_a, temp_b, temp_a, steady_b, steady_a = model(xb, xa, mask)
            yb_hat, ya_hat, h_b, h_a, logits = sanitize_tensors(yb_hat, ya_hat, h_b, h_a, logits)
            weights_b, weights_a = sanitize_tensors(weights_b, weights_a)
            steady_b, steady_a = sanitize_tensors(steady_b, steady_a)

            # Reconstruction targets
            yb_tgt = xb[:,1:-1,:]
            ya_tgt = xa[:,1:-1,:]

            # Per-sample reconstruction losses (for optional cost-weighting)
            rec_b_per = masked_mse_batchwise(yb_hat, yb_tgt, m_tgt)
            rec_a_per = masked_mse_batchwise(ya_hat, ya_tgt, m_tgt)
            rec_per = rec_b_per + rec_a_per

            # Classification loss (per-sample)
            if labels is None:
                cls_per = torch.zeros(xb.size(0), device=device)
            else:
                cls_per = ce(logits, labels)

            # Optional matching-cost weighting
            if use_matching_cost and matching_costs is not None:
                w_i = compute_sample_weights(matching_costs)
            else:
                w_i = torch.ones(xb.size(0), device=device)

            loss_rec = (rec_per * w_i).mean()
            loss_cls = (cls_per * w_i).mean()

            # HI regularizations - separated smoothness for before/after
            loss_smooth = slope_loss(h_a, mask, "tv") + slope_loss(h_b, mask, "tv")  # Use TV instead of second-order
            loss_mono   = slope_loss(h_a, mask, "mono_dec") + slope_loss(h_b, mask, "mono_dec")

            # Separated smoothness enhancement: lower weight for h_b, higher for h_a
            loss_smooth_enhanced = (smoothness_enhancement_loss_separated(h_b, h_a, mask, order=2, weight_b=0.2, weight_a=1.0) +
                                   smoothness_enhancement_loss_separated(h_b, h_a, mask, order=3, weight_b=0.1, weight_a=0.5))

            loss_liquid_weight = enhanced_liquid_weight_regularization(
                weights_b, weights_a, mask,
                lambda_tv=lambda_tv, lambda_ent=lambda_ent,
                lambda_bal=lambda_bal, lambda_div=lambda_div
            )

            # Enhanced maintenance effect loss with stronger after-maintenance smoothing
            loss_maintenance_effect = maintenance_effect_loss(
                h_b, h_a, mask,
                lambda_diff=1.0,           # Encourage difference between before/after
                lambda_smooth_after=2.5,   # INCREASED: Encourage after-maintenance HI to be smoother
                lambda_level_constraint=2.0 # Encourage min(h_a) > max(h_b)
            )

            # NEW: Plateau-specific losses for after-maintenance sequences
            loss_plateau_tv = plateau_tv_loss(h_a, mask)
            loss_plateau_curv = plateau_curvature_loss(h_a, mask)
            loss_steady_state = steady_state_loss(h_a, steady_a, mask)
            loss_flat_prior = torch.tensor(0.0, device=weights_a.device)  # disable
            loss_dynamic_plateau = dynamic_plateau_loss(h_b, h_a, steady_a, mask)

            # Combined plateau loss - removed flat_prior
            loss_plateau = (loss_plateau_tv + 0.5 * loss_plateau_curv +
                           loss_steady_state + loss_dynamic_plateau)

            # Total loss - adjusted weights for reduced plateau emphasis
            w_rec=1.0; w_smooth=0.003; w_mono=0.05; w_cls=1.0; w_smooth_enh=0.02; w_liquid=0.30; w_maintenance=1.5; w_plateau=0.8
            loss = (w_rec*loss_rec + w_smooth*loss_smooth + w_mono*loss_mono +
                    w_cls*loss_cls + w_smooth_enh*loss_smooth_enhanced + w_liquid*loss_liquid_weight +
                    w_maintenance*loss_maintenance_effect + w_plateau*loss_plateau)

            loss_rec, loss_cls, loss_smooth, loss_mono, loss_smooth_enhanced, loss_liquid_weight, loss_maintenance_effect, loss_plateau, loss = sanitize_tensors(
                loss_rec, loss_cls, loss_smooth, loss_mono, loss_smooth_enhanced, loss_liquid_weight, loss_maintenance_effect, loss_plateau, loss
            )

            if torch.isnan(loss) or torch.isinf(loss):
                batch_nan_count += 1
                if batch_nan_count == 1:
                    print(f"    [Warning] Epoch {ep}: Found NaN/Inf loss, skipping abnormal batch")
                continue

            opt.zero_grad()
            loss.backward()

            # Adaptive gradient clipping
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            grad_norms.append(grad_norm.item())

            opt.step()

            logs["mse_b"] += rec_b_per.mean().item()
            logs["mse_a"] += rec_a_per.mean().item()
            logs["smooth"]+= loss_smooth.item()
            logs["mono"]  += loss_mono.item()
            logs["cls"]   += loss_cls.item()
            logs["smooth_enh"]+= loss_smooth_enhanced.item()
            logs["liquid_weight"]+= loss_liquid_weight.item()
            logs["maintenance_effect"]+= loss_maintenance_effect.item()
            logs["plateau"]+= loss_plateau.item()
            logs["all"]   += loss.item()
            n_bt += 1

        if batch_nan_count > 0:
            nan_batches += batch_nan_count

        for k in logs: logs[k] /= max(n_bt,1)

        # Compute entropy and temperature statistics
        with torch.no_grad():
            def _entropy(w):
                return (-(w * (w+1e-8).log()).sum(-1)).mean().item()
            mean_ent_b = _entropy(weights_b)
            mean_ent_a = _entropy(weights_a)
            mean_tau_b = temp_b.mean().item()
            mean_tau_a = temp_a.mean().item()

        print("    Validating...")
        vl = eval_epoch(model, val_loader, device, K, compute_acc=True)
        # Improved early-stopping metric with configurable weight
        vl_total = (vl['mse_b'] + vl['mse_a']) + es_alpha * (1.0 - vl['acc'])

        # Log gradient statistics
        avg_grad_norm = np.mean(grad_norms) if grad_norms else 0.0
        max_grad_norm = np.max(grad_norms) if grad_norms else 0.0

        print(f"[Epoch {ep:03d}] "
              f"Train: L={logs['all']:.4f} rec_b={logs['mse_b']:.4f} rec_a={logs['mse_a']:.4f} "
              f"smooth={logs['smooth']:.4f} mono={logs['mono']:.4f} liquid_w={logs['liquid_weight']:.4f} "
              f"maint_eff={logs['maintenance_effect']:.4f} plateau={logs['plateau']:.4f} cls={logs['cls']:.4f} | "
              f"Val: mse_b={vl['mse_b']:.4f} mse_a={vl['mse_a']:.4f} acc={vl['acc']:.3f} ΔHI_impr={vl['delta_mean']:.3f} "
              f"(ES metric={vl_total:.4f}) | Grad: avg={avg_grad_norm:.3f} max={max_grad_norm:.3f}")
        print(f"Entropy pre/post: {mean_ent_b:.3f}/{mean_ent_a:.3f} | Temp pre/post: {mean_tau_b:.2f}/{mean_tau_a:.2f}")

        if vl_total < best_v:
            best_v = vl_total
            best_epoch = ep
            best_state = {k: v.clone() if hasattr(v, 'clone') else v for k, v in model.state_dict().items()}
            no_improve = 0
            print(f"    ✓ Saved new best model (ES metric: {best_v:.4f})")
        else:
            no_improve += 1
            print(f"    No improvement for {no_improve}/{patience} epochs")
            if no_improve >= patience:
                print(f"\n[Early Stopping] Patience reached at epoch {ep}.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"\n[Best Checkpoint] ES metric: {best_v:.4f} (Epoch {best_epoch})")
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        torch.save(model.state_dict(), save_path)
        print(f"[Model Saved] {save_path}")

        # Save metadata alongside model
        meta_path = save_path.replace('.pth', '_meta.json')
        meta_data = {
            "K": K,
            "id_to_name": id_to_name,
            "sensor_dim": C,
            "best_es": best_v,
            "best_epoch": best_epoch,
            "class_weights": class_weights.cpu().numpy().tolist() if class_weights is not None else None
        }
        import json
        with open(meta_path, 'w') as f:
            json.dump(meta_data, f, indent=2)
        print(f"[Metadata Saved] {meta_path}")

    if nan_batches > 0:
        print(f"[Warning] Total skipped {nan_batches} batches containing NaN/Inf during training.")

    meta = {"K": K, "id_to_name": id_to_name, "sensor_dim": C, "best_es": best_v, "best_epoch": best_epoch}
    return save_path, meta



ckpt_path, meta = train_only(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=1000,
    lr=1e-3,
    device=None,     # auto
    patience=20,
    use_matching_cost=True,
    save_path=str(resolve_data_file("best_model_NGFAID_250_100_5.pth")),
    class_weights=torch.as_tensor(class_weights, dtype=torch.float32)   # Pass your computed class weights here
)


## Class-Specific Operator Weight Heatmaps


In [ ]:
import os, json, math, random
import numpy as np
import torch
import matplotlib.pyplot as plt

def plot_class_operator_weight_heatmaps(
    checkpoint_path: str,
    test_loader,
    target_class,                     # int id (e.g., 3) or class name (e.g., "Class_3")
    device: torch.device = None,
    max_samples: int = 9,
    seed: int = 0,
    max_curve_keep: int = 2000,       # ensure we keep enough samples with weights
    fallback_to_pred_if_no_labels: bool = True,
):
    """
    Load the best model checkpoint and plot operator-weight heatmaps (pre/post) for samples
    belonging to a specified class.

    Args:
        checkpoint_path: Path to the saved .pth model (the function will also try to read the
                         matching '<checkpoint>_meta.json' if it exists).
        test_loader:     A DataLoader that yields batches as defined in your snippet.
        target_class:    Either a class id (int) or a class name (str). If labels are missing
                         or ignored, can fallback to predicted label when enabled.
        device:          torch.device to perform inference on. Defaults to CUDA if available.
        max_samples:     Maximum number of sample heatmaps to draw.
        seed:            RNG seed for shuffling/selection.
        max_curve_keep:  How many curves to retain from collect_test_predictions (must be high
                         enough to cover your filtered class subset).
        fallback_to_pred_if_no_labels:
                         If True, and ground-truth labels are missing/ignored, filter by
                         predicted labels instead.
    """

    # ---------- 0) Device ----------
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ---------- 1) Read meta or infer K/C/id_to_name ----------
    meta_path = checkpoint_path.replace(".pth", "_meta.json")
    has_meta = os.path.isfile(meta_path)
    if has_meta:
        with open(meta_path, "r") as f:
            meta = json.load(f)
        K = int(meta["K"])
        id_to_name = {int(k): v for k, v in meta["id_to_name"].items()}
        C = int(meta["sensor_dim"])
    else:
        # Fallback: infer from loader
        K = scan_num_classes_from_loader(test_loader)
        id_to_name = build_default_id_to_name(K)
        first_batch = next(iter(test_loader))
        C = first_batch["x_before"].shape[-1]

    # Map class name -> id if needed
    if isinstance(target_class, str):
        # Exact match first, else try case-insensitive
        name_to_id = {v: k for k, v in id_to_name.items()}
        if target_class in name_to_id:
            target_id = name_to_id[target_class]
        else:
            ci = {v.lower(): k for k, v in id_to_name.items()}
            if target_class.lower() in ci:
                target_id = ci[target_class.lower()]
            else:
                raise ValueError(f"Class name '{target_class}' not found in id_to_name: {id_to_name}")
    else:
        target_id = int(target_class)

    # ---------- 2) Rebuild model and load weights ----------
    model = DiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=K).to(device)
    state = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state, strict=True)
    model.eval()

    # ---------- 3) Collect predictions/curves (keep many for filtering) ----------
    y_true, y_pred, _, _, curves = collect_test_predictions(
        model=model,
        loader=test_loader,
        device=device,
        max_curve_keep=max_curve_keep,
        K=K,
        id_to_name=id_to_name
    )

    # ---------- 4) Filter curves by target class ----------
    def valid_label(v):
        return (v is not None) and (v >= 0)

    filtered = []
    for ex in curves:
        w_b = ex.get("weights_before", None)
        w_a = ex.get("weights_after", None)
        if w_b is None or w_a is None:
            continue  # need weights to plot

        t = ex.get("true", -1)
        p = ex.get("pred", -1)

        if valid_label(t) and t == target_id:
            filtered.append(ex)
        elif (not valid_label(t)) and fallback_to_pred_if_no_labels and p == target_id:
            filtered.append(ex)

    if len(filtered) == 0:
        print(f"[Info] No samples found for class '{target_class}' (id={target_id}).")
        return

    # ---------- 5) Plot settings ----------
    op_names = ["MonotonicLinear", "MonotonicFlat", "ConcaveLog", "SaturationSigmoid",
                "HingeReLU", "Polynomial", "DampedSin", "PiecewiseLinear"]

    random.Random(seed).shuffle(filtered)
    n_show = min(max_samples, len(filtered))
    cols = 3
    rows = math.ceil(n_show / cols)

    fig_w = 5.4 * cols
    fig_h = 3.8 * rows
    fig = plt.figure(figsize=(fig_w, fig_h))

    for idx in range(n_show):
        ex = filtered[idx]
        w_b = np.asarray(ex["weights_before"])  # (T_b, K)
        w_a = np.asarray(ex["weights_after"])   # (T_a, K)

        if w_b.ndim != 2 or w_a.ndim != 2:
            continue

        T_b, K_b = w_b.shape
        T_a, K_a = w_a.shape
        K_eff = min(K_b, K_a, len(op_names))
        w_b = w_b[:, :K_eff]
        w_a = w_a[:, :K_eff]

        # Concatenate along time to align pre | post; add separator via a line (not NaN)
        combined = np.concatenate([w_b, w_a], axis=0)  # (T_b+T_a, K_eff)

        ax = plt.subplot(rows, cols, idx + 1)
        # Heatmap expects (rows=ops, cols=time). Transpose to (K, T)
        im = ax.imshow(combined.T, aspect="auto", interpolation="nearest")
        # Vertical separator between pre and post
        ax.axvline(T_b - 0.5, color="k", linestyle=":", linewidth=1.0)

        # Axes labels and ticks
        ax.set_yticks(range(K_eff))
        ax.set_yticklabels([op_names[i] for i in range(K_eff)], fontsize=8)
        ax.set_xlabel("Time (pre  |  post)")
        ax.set_title(
            f"ID={ex.get('sample_id','?')} | True={id_to_name.get(ex.get('true',-1),'?')} | "
            f"Pred={id_to_name.get(ex.get('pred',-1),'?')} | p={ex['prob'][ex['pred']]:.2f}"
            if ex.get("prob") is not None else
            f"ID={ex.get('sample_id','?')} | True={id_to_name.get(ex.get('true',-1),'?')} | "
            f"Pred={id_to_name.get(ex.get('pred',-1),'?')}"
        , fontsize=9)

        # Colorbar (small)
        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.set_ylabel("Weight", rotation=270, labelpad=10)

    plt.suptitle(
        f"Operator Weight Heatmaps — Class: {id_to_name.get(target_id, f'Class_{target_id}')}",
        fontsize=12, y=0.995
    )
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

# -----------------------
# Example usage (English)
# -----------------------
plot_class_operator_weight_heatmaps(
    checkpoint_path=str(resolve_data_file("best_model_NGFAID_250_100_4.pth")),
    test_loader=test_loader,     # provide your test DataLoader
    target_class=3,              # or "Class_3"
    device=None,                 # auto-select
    max_samples=6,
    seed=0
)


## Part 3F: Finalized Training Variant


In [ ]:


# ============================================================
# Monotonic-Decreasing HI + aligned plotting (before -> after)
# Adapted to external train/val/test loaders that yield:
#   batch = {
#       "x_before": (B, L, C), "x_after": (B, L, C),
#       "y": LongTensor[B] or None,
#       "meta": list[dict] (each may contain 'pair_id', 'matching_cost', ...)
#   }
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import os
import random
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix

# ---------------------------------------------------------------------
# Utilities for the NEW loaders
# ---------------------------------------------------------------------
def adapt_batch(batch, device):
    """
    Adapt a batch from your loaders to the tensors expected by the model.
    Ensures fixed-length mask/lengths and extracts optional meta.
    """
    xb = batch["x_before"].to(device)  # (B, L, C)
    xa = batch["x_after"].to(device)   # (B, L, C)


    B, L, _ = xb.shape
    mask = torch.ones(B, L, dtype=xb.dtype, device=device)
    lengths = torch.full((B,), L, dtype=torch.long, device=device)

    labels = batch.get("y", None)
    if labels is not None and isinstance(labels, torch.Tensor):
        # Filter out ignore_index labels
        valid_mask = labels != -100
        if valid_mask.any():
            labels = labels.to(device)
        else:
            labels = None
    elif labels is not None:
        labels = None

    meta = batch.get("meta", None)
    sample_ids = []
    matching_costs = torch.zeros(B, dtype=torch.float32, device=device)
    if isinstance(meta, (list, tuple)) and len(meta) == B:
        for i, m in enumerate(meta):
            if isinstance(m, dict):
                sample_ids.append(str(m.get("pair_id", f"sample_{i}")))
                if "matching_cost" in m and m["matching_cost"] is not None:
                    matching_costs[i] = float(m["matching_cost"])
            else:
                sample_ids.append(f"sample_{i}")
    else:
        sample_ids = [f"sample_{i}" for i in range(B)]

    return xb, xa, labels, mask, lengths, sample_ids, matching_costs


def scan_num_classes_from_loader(train_loader):
    """
    Determine K (number of classes) from the training loader.
    """
    max_class = -1
    for batch in train_loader:
        y = batch.get("y", None)
        if y is None:
            continue
        if isinstance(y, torch.Tensor):
            y_np = y.detach().cpu().numpy()
            valid_labels = y_np[y_np >= 0]  # filter out negative labels
            if len(valid_labels) > 0:
                max_class = max(max_class, int(valid_labels.max()))

    if max_class < 0:
        raise ValueError("No valid labels found in the training loader to determine K.")

    K = max_class + 1
    return K


def build_default_id_to_name(K):
    return {i: f"Class_{i}" for i in range(K)}


# ---------------------------------------------------------------------
# Core model components
# ---------------------------------------------------------------------
class BoltzmannKAN(nn.Module):
    def __init__(self, in_ch, out_ch=4):
        super().__init__()
        self.E  = nn.Parameter(torch.zeros(out_ch, in_ch))
        self.kT = nn.Parameter(torch.ones(out_ch, in_ch))
    def forward(self, x):
        B,T,C = x.shape
        kT = torch.clamp(F.softplus(self.kT), 0.01, 10.0).unsqueeze(0).unsqueeze(2)
        E  = torch.clamp(self.E, -10.0, 10.0).unsqueeze(0).unsqueeze(2)
        x_ = torch.clamp(x.unsqueeze(1), -10.0, 10.0)
        exp = torch.clamp((E - x_) / kT, -50, 50)
        w   = torch.sigmoid(exp)
        h   = (w * x_).sum(dim=3).permute(0, 2, 1)
        return torch.clamp(F.softplus(h), 0.0, 100.0)

class BaseOp(nn.Module):
    def __init__(self):
        super().__init__()
        self.param_modulator = None
    def set_param_modulator(self, modulator): self.param_modulator = modulator
    def get_modulated_params(self, context):
        if self.param_modulator is None: return {}
        return self.param_modulator(context)

class MonotonicLinearOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 2.0  # Reduced from 5.0 to 2.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_bias = self.bias
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        bias  = torch.clamp(base_bias + mod.get('bias_offset', 0.0), -5.0, 5.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * (xm + bias)), 0.0, 100.0)

class MonotonicFlatOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 1e-3, 1.0
    def _cum(self, x):
        B, T = x.shape
        diff = F.relu(x[:, 1:] - x[:, :-1])
        zero_init = torch.zeros(B, 1, device=x.device, dtype=x.dtype)
        return torch.cat([zero_init, torch.cumsum(diff, dim=1)], dim=1)
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_bias = self.bias
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        bias  = torch.clamp(base_bias + mod.get('bias_offset', 0.0), -2.0, 2.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        cum_result = self._cum(xm)
        return torch.clamp(F.softplus(scale * (cum_result + bias)), 0.0, 100.0)

class ConcaveLogOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.eps = 1e-3
        self.smin, self.smax = 0.01, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        eps  = torch.clamp(self.eps + mod.get('eps_offset', 0.0), 1e-6, 1e-1)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * torch.log(torch.abs(xm) + eps)), 0.0, 100.0)

class SaturationSigmoidOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.raw_slope = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
        self.lmin, self.lmax = 0.1, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_slope = torch.clamp(F.softplus(self.raw_slope), self.lmin, self.lmax)
        base_bias  = self.bias
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        slope = torch.clamp(base_slope + mod.get('slope_offset', 0.0), self.lmin, self.lmax)
        bias  = torch.clamp(base_bias  + mod.get('bias_offset', 0.0), -3.0, 3.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * torch.sigmoid(slope * (xm - bias))), 0.0, 100.0)

class HingeReLUOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.threshold = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_threshold = self.threshold
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        threshold = torch.clamp(base_threshold + mod.get('threshold_offset', 0.0), -3.0, 3.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * F.relu(xm - threshold)), 0.0, 100.0)

class PolynomialOp(BaseOp):
    def __init__(self, deg=2):  # Reduced default degree for stability
        super().__init__()
        self.raw_coeff = nn.Parameter(torch.zeros(deg))
        self.deg = deg
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_coeff = torch.clamp(F.softplus(self.raw_coeff), 0.01, 2.0)  # Reduced from 5.0 to 2.0
        xm = torch.clamp(torch.tanh(h.mean(-1)), -3.0, 3.0)  # Added tanh for stability
        coeff_offset = mod.get('coeff_offset', torch.zeros_like(base_coeff))
        if coeff_offset.dim() > 1:
            coeff_offset = coeff_offset.mean(0)
        coeff = torch.clamp(base_coeff + coeff_offset, 0.01, 2.0)  # Also reduced to 2.0
        y = torch.zeros_like(xm)
        for i in range(self.deg):
            y = y + coeff[i] * torch.clamp(xm ** (i + 1), -100.0, 100.0)
        y = torch.tanh(y) * 5.0  # Added tanh scaling
        return torch.clamp(F.softplus(y), 0.0, 100.0)

class DampedSinOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.raw_freq = nn.Parameter(torch.tensor(0.0))
        self.raw_lambda = nn.Parameter(torch.tensor(0.0))
        self.phase = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
        self.fmin, self.fmax = 0.1, 5.0
        self.lmin, self.lmax = 0.01, 3.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_freq  = torch.clamp(F.softplus(self.raw_freq), self.fmin, self.fmax)
        base_lam   = torch.clamp(F.softplus(self.raw_lambda), self.lmin, self.lmax)
        base_phase = torch.clamp(self.phase, -10.0, 10.0)
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        freq  = torch.clamp(base_freq  + mod.get('freq_offset', 0.0), self.fmin, self.fmax)
        lam   = torch.clamp(base_lam   + mod.get('lambda_offset', 0.0), self.lmin, self.lmax)
        phase = torch.clamp(base_phase + mod.get('phase_offset', 0.0), -10.0, 10.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        damp = 1 / (1 + lam * torch.abs(xm))
        return torch.clamp(F.softplus(scale * damp * (torch.sin(freq * xm + phase) + 1)), 0.0, 100.0)

class PiecewiseLinearOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_k1 = nn.Parameter(torch.tensor(0.0))
        self.raw_k2 = nn.Parameter(torch.tensor(0.0))
        self.threshold = nn.Parameter(torch.tensor(0.0))
        self.kmin, self.kmax = 0.01, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_k1 = torch.clamp(F.softplus(self.raw_k1), self.kmin, self.kmax)
        base_k2 = torch.clamp(F.softplus(self.raw_k2), self.kmin, self.kmax)
        base_thr = torch.clamp(self.threshold, -5.0, 5.0)
        k1 = torch.clamp(base_k1 + mod.get('k1_offset', 0.0), self.kmin, self.kmax)
        k2 = torch.clamp(base_k2 + mod.get('k2_offset', 0.0), self.kmin, self.kmax)
        thr= torch.clamp(base_thr + mod.get('threshold_offset', 0.0), -5.0, 5.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        left = k1 * xm
        right = k1 * thr + k2 * (xm - thr)
        out = torch.where(xm <= thr, left, right)
        return torch.clamp(F.softplus(out), 0.0, 100.0)

class ParameterModulator(nn.Module):
    def __init__(self, context_dim, param_configs):
        super().__init__()
        self.param_configs = param_configs
        self.param_predictors = nn.ModuleDict()
        for name, info in param_configs.items():
            dim = info.get('dim', 1)
            self.param_predictors[name] = nn.Sequential(
                nn.Linear(context_dim, 64), nn.ReLU(), nn.Dropout(0.1),
                nn.Linear(64, 32), nn.ReLU(),
                nn.Linear(32, dim), nn.Tanh()
            )
    def forward(self, context):
        global_context = context.mean(dim=1)  # (B, context_dim)
        modulated = {}
        for name, predictor in self.param_predictors.items():
            info = self.param_configs[name]
            raw_off = predictor(global_context)
            scale = info.get('scale', 0.1)
            modulated[name] = raw_off * scale
        return modulated

class LiquidWeightGenerator(nn.Module):
    def __init__(self, h_dim, x_dim, n_ops, hidden_dim=64, tau_min=0.5, tau_max=3.0):  # Changed from 1.0-5.0 to 0.5-3.0
        super().__init__()
        self.n_ops = n_ops
        self.tau_min, self.tau_max = tau_min, tau_max
        self.h_feature_net = nn.Sequential(nn.Linear(h_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(0.1))
        self.x_feature_net = nn.Sequential(nn.Linear(x_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(0.1))
        self.temporal_encoder = nn.Sequential(nn.Linear(3, hidden_dim // 4), nn.ReLU())
        combined_dim = hidden_dim + hidden_dim // 4
        self.feature_fusion = nn.Sequential(
            nn.Linear(combined_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.op_branches = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 4), nn.ReLU(), nn.Linear(hidden_dim // 4, 1))
            for _ in range(n_ops)
        ])
        self.temp_predictor = nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 4), nn.ReLU(), nn.Linear(hidden_dim // 4, 1))
        self.global_bias_scale = 0.0  # Disabled random bias injection
        for branch in self.op_branches:
            nn.init.xavier_uniform_(branch[0].weight)
            nn.init.xavier_uniform_(branch[2].weight)

    def forward(self, h_multi, x, is_after=False, training_noise=0.0):
        B, T, h_dim = h_multi.shape
        _, _, x_dim = x.shape
        h_features = self.h_feature_net(h_multi)
        x_features = self.x_feature_net(x)
        t_norm = torch.arange(T, device=x.device, dtype=x.dtype).unsqueeze(0).expand(B, -1) / max(T-1, 1)
        dt = torch.zeros_like(t_norm)
        dt[:, 1:] = t_norm[:, 1:] - t_norm[:, :-1]
        phase = torch.sin(2 * 3.14159 * t_norm)
        temporal_input = torch.stack([t_norm, dt, phase], dim=-1)
        temporal_features = self.temporal_encoder(temporal_input)
        combined = torch.cat([h_features, x_features, temporal_features], dim=-1)
        fused = self.feature_fusion(combined)
        raw_logits = []
        for branch in self.op_branches:
            raw_logits.append(branch(fused).squeeze(-1))
        raw_weights = torch.stack(raw_logits, dim=-1)                     # (B, T, K)

        # Add training noise for exploration
        if self.training and training_noise > 0:
            noise = torch.randn_like(raw_weights) * training_noise
            raw_weights = raw_weights + noise

        # NaN protection and clipping
        raw_weights = torch.nan_to_num(raw_weights, nan=0.0, posinf=1e6, neginf=-1e6)
        raw_weights = raw_weights - raw_weights.mean(dim=-1, keepdim=True)
        raw_weights = raw_weights.clamp(-20.0, 20.0)  # Added clipping

        raw_temp = self.temp_predictor(fused).squeeze(-1)                 # (B, T)
        raw_temp = torch.nan_to_num(raw_temp, nan=0.0, posinf=1e6, neginf=-1e6)

        temperature = torch.clamp(F.softplus(raw_temp) + self.tau_min, self.tau_min, self.tau_max)

        weights = F.softmax(raw_weights / temperature.unsqueeze(-1), dim=-1)
        return weights, temperature

class CustomKAN(nn.Module):
    def __init__(self, ops, h_dim, x_dim):
        super().__init__()
        self.ops = nn.ModuleList(ops)
        self.n_ops = len(ops)
        self.weight_generator = LiquidWeightGenerator(h_dim=h_dim, x_dim=x_dim, n_ops=self.n_ops, hidden_dim=128)
        self.op_feature_extractors = nn.ModuleList([
            nn.Sequential(nn.Linear(h_dim, h_dim), nn.ReLU(), nn.Linear(h_dim, h_dim))
            for _ in range(self.n_ops)
        ])
        self.param_modulators = nn.ModuleList()
        context_dim = h_dim + x_dim
        for op in self.ops:
            param_configs = self._get_param_configs_for_op(op)
            if param_configs:
                modulator = ParameterModulator(context_dim, param_configs)
                self.param_modulators.append(modulator)
                op.set_param_modulator(modulator)
            else:
                self.param_modulators.append(None)
        self.raw_gain = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))

    def _get_param_configs_for_op(self, op):
        # Reduced modulation scales to prevent single-operator dominance
        if isinstance(op, MonotonicLinearOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.07}, 'bias_offset': {'dim': 1, 'scale': 0.1}}
        if isinstance(op, MonotonicFlatOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.03}, 'bias_offset': {'dim': 1, 'scale': 0.07}}
        if isinstance(op, ConcaveLogOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.07}, 'eps_offset': {'dim': 1, 'scale': 0.003}}
        if isinstance(op, SaturationSigmoidOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.07}, 'slope_offset': {'dim': 1, 'scale': 0.1}, 'bias_offset': {'dim': 1, 'scale': 0.1}}
        if isinstance(op, HingeReLUOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.07}, 'threshold_offset': {'dim': 1, 'scale': 0.1}}
        if isinstance(op, PolynomialOp):
            return {'coeff_offset': {'dim': op.deg, 'scale': 0.07}}
        if isinstance(op, DampedSinOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.07}, 'freq_offset': {'dim': 1, 'scale': 0.1},
                    'lambda_offset': {'dim': 1, 'scale': 0.07}, 'phase_offset': {'dim': 1, 'scale': 0.17}}
        if isinstance(op, PiecewiseLinearOp):
            return {'k1_offset': {'dim': 1, 'scale': 0.07}, 'k2_offset': {'dim': 1, 'scale': 0.07},
                    'threshold_offset': {'dim': 1, 'scale': 0.1}}
        return {}

    def forward(self, h_multi, x, is_after=False, training_noise=0.0, freeze_modulator=False):
        op_features = [ext(h_multi) for ext in self.op_feature_extractors]
        context = torch.cat([h_multi, x], dim=-1)

        # Optionally freeze parameter modulation
        if freeze_modulator:
            with torch.no_grad():
                outs = [op(op_features[i], context) for i, op in enumerate(self.ops)]
        else:
            outs = [op(op_features[i], context) for i, op in enumerate(self.ops)]

        Tm = min(o.size(1) for o in outs)
        outs = [o[:, :Tm] for o in outs]
        h_multi_aligned = h_multi[:, :Tm, :]
        x_aligned = x[:, :Tm, :]
        op_stack = torch.stack(outs, dim=-1)  # (B, Tm, K)
        weights, temperature = self.weight_generator(h_multi_aligned, x_aligned, is_after=is_after, training_noise=training_noise)
        damage = torch.sum(op_stack * weights, dim=-1)  # (B, Tm)
        damage = torch.clamp(damage, 0.0, 100.0)
        gain = torch.clamp(F.softplus(self.raw_gain), 0.1, 2.0)
        bias_val = torch.clamp(self.bias, -2.0, 2.0)
        damage = torch.clamp(gain * damage + bias_val, 0.0, 100.0)
        return damage, weights, temperature, op_stack

class TrendEncoder(nn.Module):
    """
    Vectorized monotonic decreasing HI projection with learnable nonlinear decay.
    Enhanced with plateau-specific regularization for after-maintenance sequences.
    """
    def __init__(self, in_ch, trend_ch=4):
        super().__init__()
        self.boltz = BoltzmannKAN(in_ch, out_ch=trend_ch)
        ops = [MonotonicLinearOp(), MonotonicFlatOp(), ConcaveLogOp(),
               SaturationSigmoidOp(), HingeReLUOp(), PolynomialOp(),
               DampedSinOp(), PiecewiseLinearOp()]
        self.customkan = CustomKAN(ops, h_dim=trend_ch, x_dim=in_ch)

        self.proj_gain = nn.Parameter(torch.tensor(1.0))
        self.proj_bias = nn.Parameter(torch.tensor(0.0))

        self.health_aware_transform = nn.Sequential(
            nn.Linear(in_ch, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )

        # Learnable nonlinear decay rate predictor
        self.rate_head = nn.Sequential(
            nn.Linear(in_ch + 1, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )
        self.rate_scale = nn.Parameter(torch.tensor(0.1))  # Global decay intensity

        # Reduced linear enforcer influence
        self.linear_enforcer = nn.Sequential(
            nn.Linear(in_ch, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )

        # Steady-state level predictor for plateau targeting
        self.steady_state_head = nn.Sequential(
            nn.Linear(in_ch, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Sigmoid()
        )

    def forward(self, x, is_after=False, training_noise=0.0, freeze_modulator=False, mixture_weight=0.6):  # x:(B,T,C)
        B, T, C = x.shape
        h_multi = self.boltz(x)                               # (B,T,trend_ch)
        damage, weights, temperature, op_outputs = self.customkan(h_multi, x, is_after=is_after,
                                                                  training_noise=training_noise,
                                                                  freeze_modulator=freeze_modulator)

        x_reshaped = x.view(-1, C)
        health_direct = self.health_aware_transform(x_reshaped).view(B, T)  # (B,T)

        t = torch.arange(T, device=x.device, dtype=x.dtype).unsqueeze(0).expand(B, -1)
        t_normalized = t / (T - 1 + 1e-6)

        # Learnable nonlinear decay with plateau-aware rate control
        time_feat = t_normalized.unsqueeze(-1)                 # (B,T,1)
        rate_in = torch.cat([x, time_feat], dim=-1)           # (B,T,C+1)
        raw_rate = self.rate_head(rate_in).squeeze(-1)         # (B,T)
        rate = F.softplus(raw_rate) * torch.clamp(self.rate_scale, 1e-3, 2.0)

        # For after-maintenance, encourage rate to approach zero in later stages
        if is_after:
            tau = int(0.3 * T)  # 30% cutoff for plateau region
            if T > tau:
                late_stage_penalty = torch.zeros_like(rate)
                late_indices = torch.arange(T, device=rate.device) >= tau
                late_stage_penalty[:, late_indices] = rate[:, late_indices] * 2.0  # Penalty for high rates in plateau region
                rate = rate - late_stage_penalty
                rate = torch.clamp(rate, 1e-6, None)  # Ensure positive rates

        cum_rate = torch.cumsum(rate, dim=1)                   # (B,T)
        nonlinear_decay = torch.exp(-cum_rate)                 # Monotonic decreasing and flexible

        g = torch.clamp(F.softplus(self.proj_gain), 0.1, 5.0)
        b = torch.clamp(self.proj_bias, -5.0, 5.0)
        damage_normalized = torch.sigmoid(g*(damage + b))

        # Increased mixture weight for operator outputs
        combined_health = health_direct * (1 - mixture_weight * damage_normalized)

        # Reduced linear channel influence with time-dependent weighting
        lw = self.linear_enforcer(x_reshaped).view(B, T)
        linear_weight = 0.1 * lw * (1.0 - t_normalized)   # Further reduced linear influence

        hi = linear_weight * nonlinear_decay + (1 - linear_weight) * combined_health

        # ---- Vectorized strictly non-increasing enforcement with adaptive epsilon ----
        if is_after:
            # Smaller epsilon step for after-maintenance to allow for plateaus
            eps = 5e-4
        else:
            eps = 1e-3
        idx = torch.arange(T, device=hi.device, dtype=hi.dtype).unsqueeze(0)  # (1, T)
        hi_mon, _ = torch.cummin(hi + eps * idx, dim=1)
        hi = hi_mon - eps * idx
        # -------------------------------------------------------------------------

        # Predict steady-state level for plateau targeting
        x_global = x.mean(dim=1)  # (B, C) - global sensor statistics
        steady_level = self.steady_state_head(x_global)  # (B, 1)

        return hi, h_multi, weights, temperature, steady_level, op_outputs

class ReconHead(nn.Module):
    def __init__(self, C, hidden=128):
        super().__init__()
        self.gru = nn.GRU(input_size=C+1, hidden_size=hidden, batch_first=True)
        self.out = nn.Linear(hidden, C)
    def forward(self, x_in, h_in):
        B,T,C = x_in.shape
        h_in_clamped = torch.clamp(h_in, 0.0, 10.0)
        feat = torch.cat([x_in, h_in_clamped.unsqueeze(-1)], dim=-1)
        H,_ = self.gru(feat)
        y = self.out(H)
        return y

class MaintClassifier(nn.Module):
    def __init__(self, sensor_dim, hidden=64, n_classes=14):
        super().__init__()
        self.hi_mlp = nn.Sequential(
            nn.Linear(6, hidden), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden//2)
        )
        self.sensor_mlp = nn.Sequential(
            nn.Linear(sensor_dim * 2, hidden), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden//2)
        )
        self.final_classifier = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, n_classes)
        )

    def forward(self, h_b, h_a, x_b, x_a, mask):
        m = mask
        T = m.size(1)
        valid = m.sum(1, keepdim=True).clamp_min(1.0)

        db = h_a - h_b
        mean_d = (db*m).sum(1,keepdim=True)/valid
        var_d  = (((db-mean_d)*m)**2).sum(1,keepdim=True)/valid
        std_d  = (var_d + 1e-8).sqrt()
        d0     = (db[:, :1])
        dmax   = (db.masked_fill(m==0, -1e9)).max(dim=1, keepdim=True).values
        pos_ratio = ((db>0).float()*m).sum(1,keepdim=True)/valid

        t = torch.arange(T, device=h_b.device, dtype=h_b.dtype)
        t = (t - t.mean())/(t.std()+1e-6)
        def slope(x):
            num = (x*t).sum(1) - x.sum(1)*t.sum()/T
            den = (t**2).sum() - (t.sum()**2)/T + 1e-8
            return (num/den).unsqueeze(1)
        slope_diff = slope(h_a) - slope(h_b)

        hi_feat = torch.cat([mean_d, d0, dmax, std_d, slope_diff, pos_ratio], dim=1)
        hi_feat = torch.clamp(hi_feat, -10.0, 10.0)
        hi_features = self.hi_mlp(hi_feat)

        x_b_mean = (x_b * m.unsqueeze(-1)).sum(1) / valid
        x_a_mean = (x_a * m.unsqueeze(-1)).sum(1) / valid
        sensor_change = torch.cat([x_b_mean, x_a_mean], dim=1)
        sensor_features = self.sensor_mlp(sensor_change)

        combined = torch.cat([hi_features, sensor_features], dim=1)
        logits = self.final_classifier(combined)
        return logits

def sanitize_tensors(*tensors):
    result = []
    for t in tensors:
        if t is not None:
            t_clean = torch.nan_to_num(t, nan=0.0, posinf=1e6, neginf=-1e6)
            result.append(t_clean)
        else:
            result.append(t)
    return result[0] if len(result) == 1 else result

class DiffAwareReconstructor(nn.Module):
    """
    - Two encoders share parameters: encode x_before / x_after → h_b / h_a (monotonic decreasing HI)
    - Two reconstruction heads: one-step recursive reconstruction
    - Classification head: ΔHI + sensor change features
    """
    def __init__(self, in_ch, trend_ch=4, hidden=128, n_classes=14):
        super().__init__()
        self.encoder = TrendEncoder(in_ch, trend_ch)
        self.recon_b = ReconHead(in_ch, hidden)
        self.recon_a = ReconHead(in_ch, hidden)
        self.clf     = MaintClassifier(sensor_dim=in_ch, hidden=64, n_classes=n_classes)
        self.consistency_weight = nn.Parameter(torch.tensor(1.0))

    def forward(self, x_b, x_a, mask, training_noise=0.0, freeze_modulator=False, mixture_weight=0.6):
        h_b, h_multi_b, weights_b, temp_b, steady_b, op_outputs_b = self.encoder(x_b, is_after=False,
                                                                                  training_noise=training_noise,
                                                                                  freeze_modulator=freeze_modulator,
                                                                                  mixture_weight=mixture_weight)
        h_a, h_multi_a, weights_a, temp_a, steady_a, op_outputs_a = self.encoder(x_a, is_after=True,
                                                                                  training_noise=training_noise,
                                                                                  freeze_modulator=freeze_modulator,
                                                                                  mixture_weight=mixture_weight)

        h_b, h_a = sanitize_tensors(h_b, h_a)
        weights_b, weights_a = sanitize_tensors(weights_b, weights_a)
        steady_b, steady_a = sanitize_tensors(steady_b, steady_a)

        xb_in, hb_in = x_b[:, :-2], h_b[:, :-2]
        xa_in, ha_in = x_a[:, :-2], h_a[:, :-2]
        yb_hat = self.recon_b(xb_in, hb_in)   # (B, L-2, C) ≈ x_b[:,1:L-1]
        ya_hat = self.recon_a(xa_in, ha_in)   # (B, L-2, C) ≈ x_a[:,1:L-1]

        yb_hat, ya_hat = sanitize_tensors(yb_hat, ya_hat)
        logits = self.clf(h_b, h_a, x_b, x_a, mask)
        logits = sanitize_tensors(logits)
        return yb_hat, ya_hat, h_b, h_a, logits, weights_b, weights_a, temp_b, temp_a, steady_b, steady_a, op_outputs_b, op_outputs_a

# ---------------------------------------------------------------------
# Losses / Regularizers
# ---------------------------------------------------------------------
def masked_mse(a, b, mask):
    diff = (a - b)**2
    mse  = (diff.mean(-1) * mask).sum() / (mask.sum() + 1e-6)
    return mse

def masked_mse_batchwise(a, b, mask):
    diff = (a - b)**2  # (B, T, C)
    mse_t = diff.mean(-1) * mask  # (B, T)
    denom = mask.sum(1).clamp_min(1.0)  # (B,)
    return mse_t.sum(1) / denom

def slope_loss(h, mask, kind="smooth"):
    if kind=="smooth":
        d2 = h[:,2:] - 2*h[:,1:-1] + h[:,:-2]
        m  = mask[:,2:]
        return (d2.abs() * m).sum() / (m.sum()+1e-6)
    elif kind=="mono_dec":
        dh = h[:,1:] - h[:,:-1]
        m  = mask[:,1:]
        return (F.relu(dh) * m).sum() / (m.sum()+1e-6)
    elif kind=="tv":
        # Total variation (first-order differences) - allows more curvature
        dh = h[:,1:] - h[:,:-1]
        m  = mask[:,1:]
        return (dh.abs() * m).sum() / (m.sum()+1e-6)
    else:
        return torch.tensor(0., device=h.device)

def plateau_tv_loss(h_a, mask, tau_ratio=0.3):
    """
    Time-weighted total variation loss for the plateau region (later 70% of sequence).
    Encourages h_a to become flat in the later stages.
    """
    B, T = h_a.shape
    tau = int(tau_ratio * T)

    if T <= tau + 1:
        return torch.tensor(0.0, device=h_a.device)

    # Focus on the plateau region (later 70%)
    h_plateau = h_a[:, tau:]  # (B, T-tau)
    mask_plateau = mask[:, tau:]  # (B, T-tau)

    if h_plateau.size(1) <= 1:
        return torch.tensor(0.0, device=h_a.device)

    # First-order differences in plateau region
    dh = h_plateau[:, 1:] - h_plateau[:, :-1]  # (B, T-tau-1)
    mask_diff = mask_plateau[:, 1:]  # (B, T-tau-1)

    # Time-increasing weights (later time steps get higher penalty)
    T_plateau = h_plateau.size(1) - 1
    if T_plateau > 0:
        time_weights = torch.linspace(1.0, 3.0, T_plateau, device=h_a.device).unsqueeze(0)  # (1, T-tau-1)
        weighted_tv = (dh.abs() * time_weights * mask_diff).sum() / (mask_diff.sum() + 1e-6)
    else:
        weighted_tv = torch.tensor(0.0, device=h_a.device)

    return weighted_tv

def plateau_curvature_loss(h_a, mask, tau_ratio=0.3):
    """
    Time-weighted second-order difference loss for the plateau region.
    Reduces curvature/oscillations in the later stages.
    """
    B, T = h_a.shape
    tau = int(tau_ratio * T)

    if T <= tau + 2:
        return torch.tensor(0.0, device=h_a.device)

    # Focus on the plateau region
    h_plateau = h_a[:, tau:]  # (B, T-tau)
    mask_plateau = mask[:, tau:]  # (B, T-tau)

    if h_plateau.size(1) <= 2:
        return torch.tensor(0.0, device=h_a.device)

    # Second-order differences in plateau region
    d2h = h_plateau[:, 2:] - 2 * h_plateau[:, 1:-1] + h_plateau[:, :-2]  # (B, T-tau-2)
    mask_d2 = mask_plateau[:, 2:]  # (B, T-tau-2)

    # Time-increasing weights
    T_plateau = h_plateau.size(1) - 2
    if T_plateau > 0:
        time_weights = torch.linspace(1.0, 2.0, T_plateau, device=h_a.device).unsqueeze(0)  # (1, T-tau-2)
        weighted_curv = (d2h.abs() * time_weights * mask_d2).sum() / (mask_d2.sum() + 1e-6)
    else:
        weighted_curv = torch.tensor(0.0, device=h_a.device)

    return weighted_curv

def steady_state_loss(h_a, steady_level, mask, tau_ratio=0.3):
    """
    Encourage h_a to approach the predicted steady-state level in the plateau region.
    """
    B, T = h_a.shape
    tau = int(tau_ratio * T)

    if T <= tau:
        return torch.tensor(0.0, device=h_a.device)

    # Focus on the plateau region
    h_plateau = h_a[:, tau:]  # (B, T-tau)
    mask_plateau = mask[:, tau:]  # (B, T-tau)

    # Expand steady_level to match plateau dimensions
    steady_expanded = steady_level.expand(-1, h_plateau.size(1))  # (B, T-tau)

    # L2 loss to steady state
    steady_loss = ((h_plateau - steady_expanded) ** 2 * mask_plateau).sum() / (mask_plateau.sum() + 1e-6)

    return steady_loss

def flat_operator_prior_loss(weights_a, mask, tau_ratio=0.3, flat_op_indices=[1, 3]):
    """
    Encourage flat/saturation operators to have higher weights in the plateau region.
    flat_op_indices: indices of MonotonicFlatOp and SaturationSigmoidOp
    """
    B, T, K = weights_a.shape
    tau = int(tau_ratio * T)

    if T <= tau:
        return torch.tensor(0.0, device=weights_a.device)

    # Focus on plateau region
    weights_plateau = weights_a[:, tau:, :]  # (B, T-tau, K)
    mask_plateau = mask[:, tau:]  # (B, T-tau)

    # Target distribution: higher probability for flat operators
    target_dist = torch.ones(K, device=weights_a.device) / K
    for idx in flat_op_indices:
        if idx < K:
            target_dist[idx] *= 2.0  # Boost flat operators
    target_dist = target_dist / target_dist.sum()  # Renormalize

    # KL divergence from current weights to target distribution
    eps = 1e-8
    weights_plateau_norm = weights_plateau + eps
    target_expanded = target_dist.unsqueeze(0).unsqueeze(0).expand_as(weights_plateau_norm)

    kl_div = (weights_plateau_norm * torch.log(weights_plateau_norm / target_expanded)).sum(-1)  # (B, T-tau)
    kl_loss = (kl_div * mask_plateau).sum() / (mask_plateau.sum() + 1e-6)

    return kl_loss

def enhanced_liquid_weight_regularization(weights_b, weights_a, mask, epoch, total_epochs,
                                          lambda_tv=0.0, lambda_ent=0.05,
                                          lambda_bal=0.3, lambda_div=0.05, lambda_var=0.02):
    """
    Enhanced liquid weight regularization with phased training approach.
    """
    total_loss = torch.tensor(0.0, device=weights_b.device)
    K = weights_b.size(-1)

    # Phase determination
    progress = epoch / total_epochs
    if progress <= 0.3:  # Phase A: Exploration (0-30%)
        phase = "exploration"
    elif progress <= 0.7:  # Phase B: Formation (30-70%)
        phase = "formation"
    else:  # Phase C: Convergence (70-100%)
        phase = "convergence"

    def tv_loss(weights, mask):
        if weights.size(1) <= 1:
            return torch.tensor(0.0, device=weights.device)
        diff = weights[:, 1:, :] - weights[:, :-1, :]
        mask_diff = mask[:, 1:]
        tv = (diff.abs().sum(-1) * mask_diff).sum() / (mask_diff.sum() + 1e-6)
        return tv

    # Adaptive TV loss based on phase
    if phase == "exploration":
        tv_weight = 0.0  # No temporal smoothing in exploration
    elif phase == "formation":
        tv_weight = lambda_tv * 0.5  # Light smoothing
    else:
        tv_weight = lambda_tv  # Full smoothing in convergence

    if tv_weight > 0:
        tv_loss_b = tv_loss(weights_b, mask)
        tv_loss_a = tv_loss(weights_a, mask)
        total_loss += tv_weight * (tv_loss_b + tv_loss_a)

    def target_entropy_loss(weights, mask, target_entropy):
        eps = 1e-8
        entropy = -(weights * torch.log(weights + eps)).sum(-1)  # (B, T)
        entropy_diff = (entropy - target_entropy) ** 2
        return (entropy_diff * mask).sum() / (mask.sum() + 1e-6)

    # Phased entropy targets
    if phase == "exploration":
        # Encourage uniform distribution (high entropy)
        target_entropy = np.log(K)  # Maximum entropy
        uniform_dist = torch.ones_like(weights_b) / K
        kl_to_uniform_b = F.kl_div(torch.log(weights_b + 1e-8), uniform_dist, reduction='none').sum(-1)
        kl_to_uniform_a = F.kl_div(torch.log(weights_a + 1e-8), uniform_dist, reduction='none').sum(-1)
        kl_loss = (kl_to_uniform_b * mask).sum() / (mask.sum() + 1e-6) + (kl_to_uniform_a * mask).sum() / (mask.sum() + 1e-6)
        total_loss += lambda_ent * kl_loss
    elif phase == "formation":
        target_entropy = 0.6 * np.log(K)  # Medium entropy
        ent_loss_b = target_entropy_loss(weights_b, mask, target_entropy)
        ent_loss_a = target_entropy_loss(weights_a, mask, target_entropy)
        total_loss += lambda_ent * (ent_loss_b + ent_loss_a)
    else:
        target_entropy = 0.35 * np.log(K)  # Lower entropy
        ent_loss_b = target_entropy_loss(weights_b, mask, target_entropy)
        ent_loss_a = target_entropy_loss(weights_a, mask, target_entropy)
        total_loss += lambda_ent * (ent_loss_b + ent_loss_a)

    def moe_balance_loss(weights, mask):
        """MoE-style load balancing loss"""
        B, T, K = weights.shape
        if T <= 1:
            return torch.tensor(0.0, device=weights.device)

        # Importance: average weight per operator
        valid_weights = weights * mask.unsqueeze(-1)  # (B, T, K)
        importance = valid_weights.sum([0, 1]) / (mask.sum() + 1e-6)  # (K,)

        # Load: fraction of tokens where each operator is top-1
        top1_indices = weights.argmax(dim=-1)  # (B, T)
        load = torch.zeros(K, device=weights.device)
        for k in range(K):
            load[k] = ((top1_indices == k).float() * mask).sum() / (mask.sum() + 1e-6)

        # Balance loss: encourage uniform importance and load
        importance_var = K * importance.var()
        load_var = K * load.var()
        return importance_var + load_var

    bal_loss_b = moe_balance_loss(weights_b, mask)
    bal_loss_a = moe_balance_loss(weights_a, mask)
    total_loss += lambda_bal * (bal_loss_b + bal_loss_a)

    def temporal_variance_loss(weights, mask, min_variance=0.01):
        """Encourage temporal variance in weights"""
        B, T, K = weights.shape
        if T <= 1:
            return torch.tensor(0.0, device=weights.device)

        # Compute temporal variance for each operator across time
        valid_weights = weights * mask.unsqueeze(-1)  # (B, T, K)
        # Normalize by valid timesteps per sample
        valid_counts = mask.sum(1, keepdim=True).clamp_min(1.0)  # (B, 1)
        mean_weights = valid_weights.sum(1, keepdim=True) / valid_counts.unsqueeze(-1)  # (B, 1, K)

        # Variance computation
        var_weights = ((valid_weights - mean_weights) ** 2 * mask.unsqueeze(-1)).sum(1) / valid_counts.unsqueeze(-1)  # (B, K)
        total_var = var_weights.sum(-1)  # (B,)

        # Penalty for insufficient variance
        var_penalty = F.relu(min_variance - total_var)
        return var_penalty.mean()

    # Encourage temporal variance (weights should change over time)
    var_loss_b = temporal_variance_loss(weights_b, mask, min_variance=0.01 + 0.04 * progress)
    var_loss_a = temporal_variance_loss(weights_a, mask, min_variance=0.01 + 0.04 * progress)
    total_loss += lambda_var * (var_loss_b + var_loss_a)

    def orthogonality_loss(weights, mask):
        """Encourage operator diversity through weight decorrelation"""
        B, T, K = weights.shape
        if T <= 1:
            return torch.tensor(0.0, device=weights.device)

        valid_weights = weights * mask.unsqueeze(-1)
        weights_flat = valid_weights.view(-1, K)  # (B*T, K)

        # Center the weights
        weights_centered = weights_flat - weights_flat.mean(0, keepdim=True)

        # Compute correlation matrix
        cov = torch.mm(weights_centered.T, weights_centered) / (weights_flat.size(0) - 1 + 1e-6)

        # Penalize off-diagonal correlations
        mask_offdiag = torch.ones_like(cov) - torch.eye(K, device=cov.device)
        corr_penalty = (cov.abs() * mask_offdiag).sum()
        return corr_penalty

    div_loss_b = orthogonality_loss(weights_b, mask)
    div_loss_a = orthogonality_loss(weights_a, mask)
    total_loss += lambda_div * (div_loss_b + div_loss_a)

    return total_loss

def operator_orthogonality_loss(op_outputs_b, op_outputs_a, weights_b, weights_a, mask, lambda_ortho=0.1):
    """
    Encourage different operators to produce orthogonal/diverse outputs.
    """
    total_loss = torch.tensor(0.0, device=op_outputs_b.device)

    def compute_ortho_loss(op_outputs, weights, mask):
        B, T, K = op_outputs.shape
        if K <= 1:
            return torch.tensor(0.0, device=op_outputs.device)

        # Normalize outputs and apply mask
        op_outputs_masked = op_outputs * mask.unsqueeze(-1)  # (B, T, K)

        # Compute pairwise cosine similarities
        ortho_loss = torch.tensor(0.0, device=op_outputs.device)
        count = 0

        for i in range(K):
            for j in range(i + 1, K):
                # Extract operator outputs
                out_i = op_outputs_masked[:, :, i]  # (B, T)
                out_j = op_outputs_masked[:, :, j]  # (B, T)

                # Compute cosine similarity
                dot_product = (out_i * out_j * mask).sum()
                norm_i = torch.sqrt((out_i ** 2 * mask).sum() + 1e-8)
                norm_j = torch.sqrt((out_j ** 2 * mask).sum() + 1e-8)

                cos_sim = dot_product / (norm_i * norm_j + 1e-8)
                ortho_loss += cos_sim ** 2  # Penalize high correlation
                count += 1

        return ortho_loss / max(count, 1)

    ortho_loss_b = compute_ortho_loss(op_outputs_b, weights_b, mask)
    ortho_loss_a = compute_ortho_loss(op_outputs_a, weights_a, mask)
    total_loss += lambda_ortho * (ortho_loss_b + ortho_loss_a)

    return total_loss

def smoothness_enhancement_loss_separated(h_b, h_a, mask, order=2, weight_b=0.3, weight_a=1.0):
    """
    Separated smoothness loss: lower weight for h_b, higher weight for h_a.
    """
    def compute_smoothness(h, mask, order):
        if order == 2:
            if h.size(1) <= 2:
                return torch.tensor(0.0, device=h.device)
            d2 = h[:,2:] - 2*h[:,1:-1] + h[:,:-2]
            m = mask[:,2:]
            return (d2.abs() * m).sum() / (m.sum() + 1e-6)
        elif order == 3:
            if h.size(1) <= 3:
                return torch.tensor(0.0, device=h.device)
            d3 = h[:,3:] - 3*h[:,2:-1] + 3*h[:,1:-2] - h[:,:-3]
            m = mask[:,3:]
            return (d3.abs() * m).sum() / (m.sum() + 1e-6)
        else:
            return torch.tensor(0., device=h.device)

    smooth_b = compute_smoothness(h_b, mask, order)
    smooth_a = compute_smoothness(h_a, mask, order)

    return weight_b * smooth_b + weight_a * smooth_a

def compute_sample_weights(matching_costs, alpha=0.5):
    return 1.0 / (1.0 + alpha * matching_costs)

def maintenance_effect_loss(h_b, h_a, mask, lambda_diff=1.0, lambda_smooth_after=2.5, lambda_level_constraint=3.0):
    """
    Enhanced maintenance effect loss with stronger after-maintenance smoothing.
    """
    total_loss = torch.tensor(0.0, device=h_b.device)

    # 1. Encourage difference between before and after HI (improved stability)
    hi_diff = torch.abs(h_a - h_b)  # (B, T)
    hi_diff_clamped = torch.clamp(hi_diff, min=1e-3)  # Prevent extreme gradients
    diff_loss = F.softplus(-10.0 * hi_diff_clamped)  # Smooth penalty for small differences
    diff_loss = (diff_loss * mask).sum() / (mask.sum() + 1e-6)
    total_loss += lambda_diff * diff_loss

    # 2. Encourage after-maintenance HI to be smoother (INCREASED WEIGHT)
    if h_a.size(1) > 1:
        h_a_diff = torch.abs(h_a[:, 1:] - h_a[:, :-1])  # (B, T-1)
        mask_diff = mask[:, 1:]
        smooth_after_loss = (h_a_diff * mask_diff).sum() / (mask_diff.sum() + 1e-6)
        total_loss += lambda_smooth_after * smooth_after_loss

    # 3. Encourage min(h_a) > max(h_b) for proper level separation
    h_b_masked = h_b.masked_fill(mask == 0, -1e9)  # Fill invalid positions with very low values
    h_a_masked = h_a.masked_fill(mask == 0, 1e9)   # Fill invalid positions with very high values

    h_b_max = h_b_masked.max(dim=1)[0]  # (B,) - max of each sequence
    h_a_min = h_a_masked.min(dim=1)[0]  # (B,) - min of each sequence

    # Encourage h_a_min > h_b_max with a margin
    margin = 0.1
    level_constraint = F.relu(h_b_max - h_a_min + margin)  # (B,)
    level_constraint_loss = level_constraint.mean()
    total_loss += lambda_level_constraint * level_constraint_loss

    # Pointwise gap constraint
    margin_pt = 0.05
    pointwise_gap = (h_a - h_b) + (-margin_pt)
    pointwise_loss = F.relu(-pointwise_gap)      # Penalty when h_a < h_b + margin_pt
    pointwise_loss = (pointwise_loss * mask).sum() / (mask.sum() + 1e-6)
    total_loss += 1.0 * pointwise_loss

    return total_loss

def dynamic_plateau_loss(h_b, h_a, steady_a, mask, k=10.0, m=0.05):
    """
    Dynamic plateau loss that activates when maintenance effect is significant.
    """
    # Calculate maintenance effect magnitude
    delta_mean = ((h_a - h_b) * mask).sum(1, keepdim=True) / (mask.sum(1, keepdim=True) + 1e-6)  # (B, 1)

    # Activation function: stronger plateau enforcement when delta is large
    gamma = torch.sigmoid(k * (delta_mean - m))  # (B, 1)

    # Plateau losses
    tv_loss = plateau_tv_loss(h_a, mask)
    curv_loss = plateau_curvature_loss(h_a, mask)
    steady_loss = steady_state_loss(h_a, steady_a, mask)

    # Weight by activation
    weighted_plateau_loss = gamma.mean() * (tv_loss + 0.5 * curv_loss + steady_loss)

    return weighted_plateau_loss

# ---------------------------------------------------------------------
# Evaluation / Prediction
# ---------------------------------------------------------------------
@torch.no_grad()
def eval_epoch(model, loader, device, K, compute_acc=True):
    model.eval()
    total = {"mse_b":0.0, "mse_a":0.0, "acc":0.0, "delta_mean":0.0}
    n_batch=0
    n_acc = 0; n_tot=0
    for batch in loader:
        xb, xa, labels, mask, lengths, sample_ids, matching_costs = adapt_batch(batch, device)
        m_tgt  = (torch.arange(xb.size(1)-2, device=device)[None,:] < (lengths-2)[:,None]).float()
        yb_hat, ya_hat, h_b, h_a, logits, _, _, _, _, _, _, _, _ = model(xb, xa, mask)
        yb_hat, ya_hat, h_b, h_a, logits = sanitize_tensors(yb_hat, ya_hat, h_b, h_a, logits)

        # Ensure logits dimension matches expected K
        if logits.shape[-1] != K:
            print(f"Warning: Logits dim {logits.shape[-1]} != expected K={K}, adjusting...")
            if logits.shape[-1] < K:
                # Pad with zeros
                padding = torch.zeros(logits.shape[0], K - logits.shape[-1], device=logits.device)
                logits = torch.cat([logits, padding], dim=1)
            else:
                # Truncate
                logits = logits[:, :K]

        yb_tgt = xb[:,1:-1,:]
        ya_tgt = xa[:,1:-1,:]
        mse_b = masked_mse(yb_hat, yb_tgt, m_tgt)
        mse_a = masked_mse(ya_hat, ya_tgt, m_tgt)
        if compute_acc and labels is not None:
            # Filter out ignore_index labels
            valid_labels = labels != -100
            if valid_labels.any():
                pred = logits[valid_labels].argmax(dim=1)
                labels_valid = labels[valid_labels]
                n_acc += (pred == labels_valid).sum().item()
                n_tot += labels_valid.numel()
        valid = mask.sum(1, keepdim=True).clamp_min(1.0)
        delta_mean = ((h_a - h_b)*mask).sum(1,keepdim=True)/valid
        delta_mean = torch.nan_to_num(delta_mean, nan=0.0)
        total["delta_mean"] += delta_mean.mean().item()
        total["mse_b"] += mse_b.item()
        total["mse_a"] += mse_a.item()
        n_batch += 1
    for k in ["mse_b","mse_a","delta_mean"]:
        total[k] /= max(n_batch,1)
    total["acc"] = (n_acc / max(n_tot,1)) if n_tot>0 else 0.0
    return total

@torch.no_grad()
def collect_test_predictions(model, loader, device, max_curve_keep=24, K=14, id_to_name=None):
    model.eval()
    y_true, y_pred = [], []
    all_delta_mean, all_sample_ids = [], []
    keep_curves = []
    for batch in loader:
        xb, xa, labels, mask, lengths, sample_ids, matching_costs = adapt_batch(batch, device)
        yb_hat, ya_hat, h_b, h_a, logits, weights_b, weights_a, temp_b, temp_a, steady_b, steady_a, op_outputs_b, op_outputs_a = model(xb, xa, mask)
        yb_hat, ya_hat, h_b, h_a, logits = sanitize_tensors(yb_hat, ya_hat, h_b, h_a, logits)

        # Ensure logits dimension matches expected K
        if logits.shape[-1] != K:
            if logits.shape[-1] < K:
                padding = torch.zeros(logits.shape[0], K - logits.shape[-1], device=logits.device)
                logits = torch.cat([logits, padding], dim=1)
            else:
                logits = logits[:, :K]

        prob = F.softmax(logits, dim=1)     # (B,K)
        pred = prob.argmax(1)               # (B,)

        valid = mask.sum(1, keepdim=True).clamp_min(1.0)
        delta_mean = ((h_a - h_b) * mask).sum(1, keepdim=True) / valid  # (B,1)
        delta_mean = torch.nan_to_num(delta_mean, nan=0.0)

        if labels is not None:
            # Filter out ignore_index labels
            valid_labels = labels != -100
            if valid_labels.any():
                y_true.append(labels[valid_labels].detach().cpu().numpy())
                y_pred.append(pred[valid_labels].detach().cpu().numpy())
        all_delta_mean.append(delta_mean.squeeze(1).cpu().numpy())
        all_sample_ids.extend(sample_ids)

        for i in range(xb.size(0)):
            if len(keep_curves) >= max_curve_keep: break
            L_i = int(lengths[i].item())
            L_recon = min(L_i - 2, yb_hat.size(1))
            keep_curves.append({
                "sample_id": sample_ids[i],
                "true": int(labels[i].cpu().item()) if labels is not None and labels[i] != -100 else -1,
                "pred": int(pred[i].cpu().item()),
                "prob": prob[i].cpu().numpy(),
                "h_before": h_b[i, :L_i].cpu().numpy(),
                "h_after":  h_a[i, :L_i].cpu().numpy(),
                "weights_before": weights_b[i, :L_i].cpu().numpy() if weights_b is not None else None,
                "weights_after": weights_a[i, :L_i].cpu().numpy() if weights_a is not None else None,
                "temp_before": temp_b[i, :L_i].cpu().numpy() if temp_b is not None else None,
                "temp_after": temp_a[i, :L_i].cpu().numpy() if temp_a is not None else None,
                "steady_before": steady_b[i].cpu().item() if steady_b is not None else None,
                "steady_after": steady_a[i].cpu().item() if steady_a is not None else None,
                "op_outputs_before": op_outputs_b[i, :L_i].cpu().numpy() if op_outputs_b is not None else [],
                "op_outputs_after":  op_outputs_a[i, :L_i].cpu().numpy() if op_outputs_a is not None else [],
                "yb_hat":   yb_hat[i, :L_recon].cpu().numpy(),
                "ya_hat":   ya_hat[i, :L_recon].cpu().numpy(),
                "x_before": xb[i, :L_i].cpu().numpy(),
                "x_after":  xa[i, :L_i].cpu().numpy(),
            })
    y_true = np.concatenate(y_true, axis=0) if len(y_true)>0 else np.array([])
    y_pred = np.concatenate(y_pred, axis=0) if len(y_pred)>0 else np.array([])
    all_delta_mean = np.concatenate(all_delta_mean, axis=0) if len(all_delta_mean)>0 else np.array([])
    return y_true, y_pred, all_delta_mean, all_sample_ids, keep_curves

# ---------------------------------------------------------------------
# Plotting
# ---------------------------------------------------------------------
def remove_constant_segments(hi_values, threshold=1e-6):
    if len(hi_values) <= 1:
        return hi_values, np.arange(len(hi_values))
    diffs = np.abs(np.diff(hi_values))
    valid_mask = np.ones(len(hi_values), dtype=bool)
    valid_mask[1:] = diffs > threshold
    if np.sum(valid_mask) < 2:
        valid_mask[0] = True
        valid_mask[-1] = True
    valid_indices = np.where(valid_mask)[0]
    return hi_values[valid_indices], valid_indices

def plot_hi_examples_aligned(curves, id_to_name, n_show=6, seed=0):
    if len(curves)==0:
        print("(No visualization samples)")
        return
    random.Random(seed).shuffle(curves)
    n_show = min(n_show, len(curves))
    cols = 3
    rows = int(np.ceil(n_show/cols))
    plt.figure(figsize=(cols*4.6, rows*3.2))
    for k in range(n_show):
        ex = curves[k]
        hb = ex["h_before"]; ha = ex["h_after"]
        L  = min(len(hb), len(ha))
        hb, ha = hb[:L], ha[:L]
        hb_clean, hb_indices = remove_constant_segments(hb)
        ha_clean, ha_indices = remove_constant_segments(ha)
        t_b = hb_indices
        t_a = ha_indices + L

        sample_id = ex["sample_id"]
        pred = ex["pred"]; true = ex["true"]; prob = ex["prob"]
        true_name = id_to_name.get(true, f"Class_{true}")
        pred_name = id_to_name.get(pred, f"Class_{pred}")

        plt.subplot(rows, cols, k+1)
        if len(hb_clean) > 1:
            plt.plot(t_b, hb_clean, label="Learned HI (Pre)", linewidth=1.8, marker='o', markersize=3)
        if len(ha_clean) > 1:
            plt.plot(t_a, ha_clean, label="Learned HI (Post)",  linewidth=1.8, linestyle='--', marker='s', markersize=3)
        plt.axvline(L-1, color='k', linestyle=':', linewidth=1.0, alpha=0.7)
        d_mean = float(np.mean(ha - hb))
        d_max  = float(np.max(ha - hb))

        # Add steady state info if available
        steady_info = ""
        if ex.get("steady_after") is not None:
            steady_info = f" | SS={ex['steady_after']:.3f}"

        plt.title(f"ID={sample_id}\nTrue={true_name} | Pred={pred_name} "
                  f"| p={prob[pred]:.2f}\nΔHI_mean={d_mean:.3f}, ΔHI_max={d_max:.3f}{steady_info}", fontsize=9)
        plt.xlabel("Cycle (Pre | Post)")
        plt.ylabel("Health Index (higher=better)")
        plt.grid(ls='--', alpha=.35)
        if k==0: plt.legend()
    plt.tight_layout(); plt.show()




def topk_by_delta(df_delta, id_to_name, k=3):
    if len(df_delta)==0:
        return
    unique_classes = sorted(df_delta["true"].unique())
    for cls in unique_classes[:5]:
        sub = df_delta[df_delta["true"]==cls].sort_values("delta_hi_mean", ascending=False).head(k)
        class_name = id_to_name.get(cls, f"Class_{cls}")
        print(f"\n[True={class_name}] Top {k} samples with highest ΔHI_mean:")
        print(sub[["uid","delta_hi_mean","pred"]].reset_index(drop=True))

# ---------------------------------------------------------------------
# Training (separate) and Testing (separate)
# ---------------------------------------------------------------------
def build_model_and_setup(train_loader, device, class_weights=None):
    """
    Build model and setup training components.

    Args:
        train_loader: Training data loader
        device: Device to use
        class_weights: Pre-computed class weights tensor (optional)
    """
    # Determine K from training loader
    K = scan_num_classes_from_loader(train_loader)
    id_to_name = build_default_id_to_name(K)

    first_batch = next(iter(train_loader))
    C = first_batch["x_before"].shape[-1]

    print(f"\n[Setup] Sensor dim C={C}, Num classes K={K}")

    # Use provided class weights or create uniform weights
    if class_weights is not None:
        if isinstance(class_weights, np.ndarray):
            class_weights = torch.from_numpy(class_weights).float().to(device)
        elif isinstance(class_weights, torch.Tensor):
            class_weights = class_weights.float().to(device)

        # Ensure class_weights has the right size
        if len(class_weights) < K:
            # Pad with ones if needed
            padded_weights = torch.ones(K, dtype=torch.float32, device=device)
            padded_weights[:len(class_weights)] = class_weights
            class_weights = padded_weights
        elif len(class_weights) > K:
            # Truncate if needed
            class_weights = class_weights[:K]

        print(f"[Class Weights] Using provided weights: {class_weights.detach().cpu().numpy()}")
    else:
        class_weights = torch.ones(K, dtype=torch.float32, device=device)
        print(f"[Class Weights] Using uniform weights: {class_weights.detach().cpu().numpy()}")

    model = DiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=K).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    ce  = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100, reduction='none')

    return model, opt, ce, K, id_to_name, C


def train_only(train_loader, val_loader, epochs=300, lr=1e-3, device=None, patience=20,
               use_matching_cost=True, save_path="best_model.pth", class_weights=None,
               es_alpha=1.0):
    """
    Train and save the best model with phased liquid weight training.

    Args:
        class_weights: Pre-computed class weights (numpy array or torch tensor)
        es_alpha: Weight for accuracy in early stopping metric
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Build model and training setup
    model, opt, ce, K, id_to_name, C = build_model_and_setup(train_loader, device, class_weights)
    # Override LR if provided
    for g in opt.param_groups:
        g['lr'] = lr

    best_v = float("inf")
    best_state = None
    best_epoch = 0
    no_improve = 0
    nan_batches = 0

    print(f"\nStarting phased liquid weight training for {epochs} epochs...")
    print(f"Device: {device}")
    print(f"Early stopping alpha: {es_alpha}")
    print(f"Phase A (0-30%): Exploration with uniform priors")
    print(f"Phase B (30-70%): Formation with target entropy")
    print(f"Phase C (70-100%): Convergence with refined entropy")

    for ep in range(1, epochs+1):
        progress = ep / epochs

        # Determine training phase
        if progress <= 0.3:
            phase = "exploration"
            freeze_modulator = True
            mixture_weight = 0.8
            training_noise = 0.05
        elif progress <= 0.7:
            phase = "formation"
            freeze_modulator = False
            mixture_weight = 0.7
            training_noise = 0.02
        else:
            phase = "convergence"
            freeze_modulator = False
            mixture_weight = 0.6
            training_noise = 0.0

        print(f"\n[Epoch {ep}/{epochs}] Phase: {phase.upper()}, Training...")
        model.train()
        logs = {"mse_b":0.0, "mse_a":0.0, "smooth":0.0, "mono":0.0,
                "cls":0.0, "smooth_enh":0.0, "liquid_weight":0.0, "maintenance_effect":0.0,
                "plateau":0.0, "ortho":0.0, "all":0.0}
        n_bt = 0
        batch_nan_count = 0

        # Adaptive regularization weights
        lambda_tv = 0.01 if phase == "convergence" else 0.0
        lambda_ent = 0.10
        lambda_bal = 0.3
        lambda_div = 0.05
        lambda_var = 0.02
        lambda_ortho = 0.1

        # Gradient norm tracking
        grad_norms = []

        # Monitoring metrics
        total_entropy_b = 0.0
        total_entropy_a = 0.0
        total_temp_b = 0.0
        total_temp_a = 0.0
        utilization_counts = torch.zeros(8, device=device)  # 8 operators
        switch_rates = []

        for batch_idx, batch in enumerate(train_loader):
            xb, xa, labels, mask, lengths, sample_ids, matching_costs = adapt_batch(batch, device)
            m_tgt  = (torch.arange(xb.size(1)-2, device=device)[None,:] < (lengths-2)[:,None]).float()

            yb_hat, ya_hat, h_b, h_a, logits, weights_b, weights_a, temp_b, temp_a, steady_b, steady_a, op_outputs_b, op_outputs_a = model(
                xb, xa, mask, training_noise=training_noise, freeze_modulator=freeze_modulator, mixture_weight=mixture_weight)

            yb_hat, ya_hat, h_b, h_a, logits = sanitize_tensors(yb_hat, ya_hat, h_b, h_a, logits)
            weights_b, weights_a = sanitize_tensors(weights_b, weights_a)
            steady_b, steady_a = sanitize_tensors(steady_b, steady_a)

            # Reconstruction targets
            yb_tgt = xb[:,1:-1,:]
            ya_tgt = xa[:,1:-1,:]

            # Per-sample reconstruction losses (for optional cost-weighting)
            rec_b_per = masked_mse_batchwise(yb_hat, yb_tgt, m_tgt)
            rec_a_per = masked_mse_batchwise(ya_hat, ya_tgt, m_tgt)
            rec_per = rec_b_per + rec_a_per

            # Classification loss (per-sample)
            if labels is None:
                cls_per = torch.zeros(xb.size(0), device=device)
            else:
                cls_per = ce(logits, labels)

            # Optional matching-cost weighting
            if use_matching_cost and matching_costs is not None:
                w_i = compute_sample_weights(matching_costs)
            else:
                w_i = torch.ones(xb.size(0), device=device)

            loss_rec = (rec_per * w_i).mean()
            loss_cls = (cls_per * w_i).mean()

            # HI regularizations - separated smoothness for before/after
            loss_smooth = slope_loss(h_a, mask, "tv") + slope_loss(h_b, mask, "tv")  # Use TV instead of second-order
            loss_mono   = slope_loss(h_a, mask, "mono_dec") + slope_loss(h_b, mask, "mono_dec")

            # Separated smoothness enhancement: lower weight for h_b, higher for h_a
            loss_smooth_enhanced = (smoothness_enhancement_loss_separated(h_b, h_a, mask, order=2, weight_b=0.2, weight_a=1.0) +
                                   smoothness_enhancement_loss_separated(h_b, h_a, mask, order=3, weight_b=0.1, weight_a=0.5))

            loss_liquid_weight = enhanced_liquid_weight_regularization(
                weights_b, weights_a, mask, ep, epochs,
                lambda_tv=lambda_tv, lambda_ent=lambda_ent,
                lambda_bal=lambda_bal, lambda_div=lambda_div
            )

            # Enhanced maintenance effect loss with stronger after-maintenance smoothing
            loss_maintenance_effect = maintenance_effect_loss(
                h_b, h_a, mask,
                lambda_diff=1.0,           # Encourage difference between before/after
                lambda_smooth_after=2.5,   # INCREASED: Encourage after-maintenance HI to be smoother
                lambda_level_constraint=2.0 # Encourage min(h_a) > max(h_b)
            )

            # NEW: Plateau-specific losses for after-maintenance sequences
            loss_plateau_tv = plateau_tv_loss(h_a, mask)
            loss_plateau_curv = plateau_curvature_loss(h_a, mask)
            loss_steady_state = steady_state_loss(h_a, steady_a, mask)
            loss_flat_prior = torch.tensor(0.0, device=weights_a.device)  # disable
            loss_dynamic_plateau = dynamic_plateau_loss(h_b, h_a, steady_a, mask)

            # Combined plateau loss - removed flat_prior
            loss_plateau = (loss_plateau_tv + 0.5 * loss_plateau_curv +
                           loss_steady_state + loss_dynamic_plateau)

            # Total loss - adjusted weights for reduced plateau emphasis
            w_rec=1.0; w_smooth=0.003; w_mono=0.05; w_cls=1.0; w_smooth_enh=0.02; w_liquid=0.30; w_maintenance=1.5; w_plateau=0.8
            loss = (w_rec*loss_rec + w_smooth*loss_smooth + w_mono*loss_mono +
                    w_cls*loss_cls + w_smooth_enh*loss_smooth_enhanced + w_liquid*loss_liquid_weight +
                    w_maintenance*loss_maintenance_effect + w_plateau*loss_plateau)

            loss_rec, loss_cls, loss_smooth, loss_mono, loss_smooth_enhanced, loss_liquid_weight, loss_maintenance_effect, loss_plateau, loss = sanitize_tensors(
                loss_rec, loss_cls, loss_smooth, loss_mono, loss_smooth_enhanced, loss_liquid_weight, loss_maintenance_effect, loss_plateau, loss
            )

            if torch.isnan(loss) or torch.isinf(loss):
                batch_nan_count += 1
                if batch_nan_count == 1:
                    print(f"    [Warning] Epoch {ep}: Found NaN/Inf loss, skipping abnormal batch")
                continue

            opt.zero_grad()
            loss.backward()

            # Adaptive gradient clipping
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            grad_norms.append(grad_norm.item())

            opt.step()

            logs["mse_b"] += rec_b_per.mean().item()
            logs["mse_a"] += rec_a_per.mean().item()
            logs["smooth"]+= loss_smooth.item()
            logs["mono"]  += loss_mono.item()
            logs["cls"]   += loss_cls.item()
            logs["smooth_enh"]+= loss_smooth_enhanced.item()
            logs["liquid_weight"]+= loss_liquid_weight.item()
            logs["maintenance_effect"]+= loss_maintenance_effect.item()
            logs["plateau"]+= loss_plateau.item()
            logs["all"]   += loss.item()
            n_bt += 1

        if batch_nan_count > 0:
            nan_batches += batch_nan_count

        for k in logs: logs[k] /= max(n_bt,1)

        # Compute entropy and temperature statistics
        with torch.no_grad():
            def _entropy(w):
                return (-(w * (w+1e-8).log()).sum(-1)).mean().item()
            mean_ent_b = _entropy(weights_b)
            mean_ent_a = _entropy(weights_a)
            mean_tau_b = temp_b.mean().item()
            mean_tau_a = temp_a.mean().item()

        print("    Validating...")
        vl = eval_epoch(model, val_loader, device, K, compute_acc=True)
        # Improved early-stopping metric with configurable weight
        vl_total = (vl['mse_b'] + vl['mse_a']) + es_alpha * (1.0 - vl['acc'])

        # Log gradient statistics
        avg_grad_norm = np.mean(grad_norms) if grad_norms else 0.0
        max_grad_norm = np.max(grad_norms) if grad_norms else 0.0

        print(f"[Epoch {ep:03d}] "
              f"Train: L={logs['all']:.4f} rec_b={logs['mse_b']:.4f} rec_a={logs['mse_a']:.4f} "
              f"smooth={logs['smooth']:.4f} mono={logs['mono']:.4f} liquid_w={logs['liquid_weight']:.4f} "
              f"maint_eff={logs['maintenance_effect']:.4f} plateau={logs['plateau']:.4f} cls={logs['cls']:.4f} | "
              f"Val: mse_b={vl['mse_b']:.4f} mse_a={vl['mse_a']:.4f} acc={vl['acc']:.3f} ΔHI_impr={vl['delta_mean']:.3f} "
              f"(ES metric={vl_total:.4f}) | Grad: avg={avg_grad_norm:.3f} max={max_grad_norm:.3f}")
        print(f"Entropy pre/post: {mean_ent_b:.3f}/{mean_ent_a:.3f} | Temp pre/post: {mean_tau_b:.2f}/{mean_tau_a:.2f}")

        if vl_total < best_v:
            best_v = vl_total
            best_epoch = ep
            best_state = {k: v.clone() if hasattr(v, 'clone') else v for k, v in model.state_dict().items()}
            no_improve = 0
            print(f"    ✓ Saved new best model (ES metric: {best_v:.4f})")
        else:
            no_improve += 1
            print(f"    No improvement for {no_improve}/{patience} epochs")
            if no_improve >= patience:
                print(f"\n[Early Stopping] Patience reached at epoch {ep}.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"\n[Best Checkpoint] ES metric: {best_v:.4f} (Epoch {best_epoch})")
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        torch.save(model.state_dict(), save_path)
        print(f"[Model Saved] {save_path}")

        # Save metadata alongside model
        meta_path = save_path.replace('.pth', '_meta.json')
        meta_data = {
            "K": K,
            "id_to_name": id_to_name,
            "sensor_dim": C,
            "best_es": best_v,
            "best_epoch": best_epoch,
            "class_weights": class_weights.cpu().numpy().tolist() if class_weights is not None else None
        }
        import json
        with open(meta_path, 'w') as f:
            json.dump(meta_data, f, indent=2)
        print(f"[Metadata Saved] {meta_path}")

    if nan_batches > 0:
        print(f"[Warning] Total skipped {nan_batches} batches containing NaN/Inf during training.")

    meta = {"K": K, "id_to_name": id_to_name, "sensor_dim": C, "best_es": best_v, "best_epoch": best_epoch}
    return save_path, meta


ckpt_path, meta = train_only(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=1000,
    lr=1e-3,
    device=None,     # auto
    patience=20,
    use_matching_cost=True,
    save_path=str(resolve_data_file("best_model_NGFAID_250_100_6.pth")),
    class_weights=torch.as_tensor(class_weights, dtype=torch.float32)   # Pass your computed class weights here
)


In [ ]:

# -------- Infer sensor dimension C and number of classes K from the existing loaders --------
# Prefer train_loader when inferring K; fall back to test_loader if needed.
def _infer_C_and_K(train_loader=None, test_loader=None):
    # Inspect one batch to obtain C
    probe_loader = train_loader if (train_loader is not None) else test_loader
    assert probe_loader is not None, "An available train_loader or test_loader is required to infer the dimensions."
    first_batch = next(iter(probe_loader))
    C = first_batch["x_before"].shape[-1]

    K = None
    if train_loader is not None:
        try:
            K = scan_num_classes_from_loader(train_loader)
        except Exception:
            K = None
    if K is None and test_loader is not None:
        try:
            K = scan_num_classes_from_loader(test_loader)
        except Exception:
            K = None
    assert K is not None and K > 0, "Failed to infer K (number of classes) from the data. Please check the loader labels y."
    return C, K

# train_loader / test_loader should already be constructed outside this snippet
# If train_loader is unavailable, replace it with None below and infer using only test_loader.
C, K = _infer_C_and_K(train_loader=train_loader, test_loader=test_loader)
id_to_name = build_default_id_to_name(K)

print(f"[Info] Inferred sensor dim C={C}, num classes K={K}")

# -------- Backfill / rebuild metadata (optional) --------
# If you want to store the training-time class_weights in the metadata as well, provide them here (tensor / numpy / list are all acceptable).
# Demonstration: write class_weights_obj if it is available; otherwise keep None.
class_weights_obj = None
# For example, if you still have the class_weights array, you can use:
# import numpy as np

def plot_enhanced_liquid_weights(curves, n_show=3, seed=0):
    if len(curves) == 0:
        print("(No visualization samples)")
        return
    curves_with_weights = [c for c in curves if c.get("weights_before") is not None]
    if len(curves_with_weights) == 0:
        print("(No weight information available)")
        return
    random.Random(seed).shuffle(curves_with_weights)
    n_show = min(n_show, len(curves_with_weights))
    op_names = ["MonotonicLinear", "MonotonicFlat", "ConcaveLog", "SaturationSigmoid",
                "HingeReLU", "Polynomial", "DampedSin", "PiecewiseLinear"]
    for i in range(n_show):
        ex = curves_with_weights[i]
        weights_b = ex["weights_before"]
        weights_a = ex["weights_after"]
        temp_b = ex.get("temp_before")
        temp_a = ex.get("temp_after")
        sample_id = ex["sample_id"]
        true_label = ex["true"]
        pred_label = ex["pred"]

        # Crop weight data starting from the 10th point
        start_idx = 10
        if weights_b.shape[0] > start_idx:
            weights_b = weights_b[start_idx:]
        if weights_a.shape[0] > start_idx:
            weights_a = weights_a[start_idx:]
        if temp_b is not None and len(temp_b) > start_idx:
            temp_b = temp_b[start_idx:]
        if temp_a is not None and len(temp_a) > start_idx:
            temp_a = temp_a[start_idx:]

        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle(f"Dynamic Parameter Modulation - ID={sample_id}, True=Class_{true_label}, Pred=Class_{pred_label}", fontsize=14)
        ax = axes[0, 0]
        T_b, K = weights_b.shape
        for k in range(K):
            ax.plot(range(T_b), weights_b[:, k], label=op_names[k] if k < len(op_names) else f"Op_{k}",
                   marker='o', markersize=2, linewidth=1.5)
        ax.set_title("Pre-maintenance Weights"); ax.set_xlabel("Time Step"); ax.set_ylabel("Weight")
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left'); ax.grid(True, alpha=0.3)
        ax = axes[0, 1]
        T_a, _ = weights_a.shape
        for k in range(K):
            ax.plot(range(T_a), weights_a[:, k], label=op_names[k] if k < len(op_names) else f"Op_{k}",
                   marker='s', markersize=2, linewidth=1.5, linestyle='--')
        ax.set_title("Post-maintenance Weights"); ax.set_xlabel("Time Step"); ax.set_ylabel("Weight")
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left'); ax.grid(True, alpha=0.3)
        ax = axes[0, 2]
        entropy_b = -np.sum(weights_b * np.log(weights_b + 1e-8), axis=1)
        entropy_a = -np.sum(weights_a * np.log(weights_a + 1e-8), axis=1)
        ax.plot(range(T_b), entropy_b, label="Pre", marker='o', linewidth=2)
        ax.plot(range(T_a), entropy_a, label="Post", marker='s', linewidth=2, linestyle='--')
        ax.axhline(np.log(K), color='r', linestyle=':', label=f'Max Entropy ({np.log(K):.2f})')
        ax.set_title("Weight Entropy Over Time"); ax.set_xlabel("Time Step"); ax.set_ylabel("Entropy")
        ax.legend(); ax.grid(True, alpha=0.3)
        ax = axes[1, 0]
        if temp_b is not None and temp_a is not None:
            ax.plot(range(T_b), temp_b, label="Pre", marker='o', linewidth=2)
            ax.plot(range(T_a), temp_a, label="Post", marker='s', linewidth=2, linestyle='--')
            ax.set_title("Temperature Evolution"); ax.set_xlabel("Time Step"); ax.set_ylabel("Temperature"); ax.legend()
        else:
            ax.text(0.5, 0.5, "Temperature data\nnot available", ha='center', va='center', transform=ax.transAxes)
            ax.set_title("Temperature Evolution")
        ax.grid(True, alpha=0.3)
        ax = axes[1, 1]
        ax.text(0.5, 0.5, "Operator output variance\n(not collected)", ha='center', va='center', transform=ax.transAxes)
        ax.set_title("Operator Output Variance"); ax.grid(True, alpha=0.3)
        ax = axes[1, 2]
        combined_weights = np.concatenate([weights_b, weights_a], axis=0)
        # Apply smoothing for correlation stability
        from scipy.ndimage import uniform_filter1d
        smoothed_weights = uniform_filter1d(combined_weights, size=3, axis=0)
        corr_matrix = np.corrcoef(smoothed_weights.T)
        im = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
        ax.set_title("Operator Weight Correlation")
        ax.set_xticks(range(K)); ax.set_yticks(range(K))
        short_names = [op_names[k][:8] if k < len(op_names) else f"Op_{k}" for k in range(K)]
        ax.set_xticklabels(short_names, rotation=45); ax.set_yticklabels(short_names)
        cbar = plt.colorbar(im, ax=ax); cbar.set_label('Correlation')
        plt.tight_layout(); plt.show()
def plot_liquid_weights(curves, n_show=6, seed=0):
    plot_enhanced_liquid_weights(curves, n_show, seed)

def plot_sensor_examples_aligned(curves, sensor_idx_list=None, n_cols=4):
    if len(curves)==0:
        print("(No visualization samples)")
        return
    ex = curves[0]
    xb = ex["x_before"]; xa = ex["x_after"]; ya = ex["ya_hat"]
    L, C = xb.shape
    Lhat = ya.shape[0]
    if sensor_idx_list is None:
        step = max(1, C//8); sensor_idx_list = list(range(0, C, step))[:8]
    n = len(sensor_idx_list)
    n_rows = int(np.ceil(n/n_cols))
    plt.figure(figsize=(n_cols*4.3, n_rows*3.1))
    for i, s in enumerate(sensor_idx_list):
        plt.subplot(n_rows, n_cols, i+1)
        t_b = np.arange(L); t_a = np.arange(L, 2*L)
        plt.plot(t_b, xb[:,s], label="Original (pre)", linewidth=1.2)
        plt.plot(t_a, xa[:,s], label="Original (post)",  linewidth=1.0, linestyle="--", alpha=0.7)
        t_pred = np.arange(L+1, L+1+Lhat)
        plt.plot(t_pred, ya[:,s], label="Post prediction", linewidth=1.6)
        plt.axvline(L-1, color='k', linestyle=':', linewidth='1.0')
        plt.title(f"Sensor_{s:02d}")
        if i==0: plt.legend()
        plt.grid(ls="--", alpha=.35)
    plt.tight_layout(); plt.show()
def _serialize_class_weights(obj):
    if obj is None:
        return None
    try:
        import numpy as np
        if isinstance(obj, torch.Tensor):
            return obj.detach().cpu().numpy().tolist()
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, (list, tuple)):
            return list(obj)
        else:
            return None
    except Exception:
        return None

def test_only(test_loader, ckpt_path, K=None, id_to_name=None, device=None,
              visualize=True, max_curve_keep=24):
    """
    Load a saved checkpoint and evaluate on the official test set.
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Peek one batch to get sensor dimension C (and fallback K if needed)
    first_batch = next(iter(test_loader))
    C = first_batch["x_before"].shape[-1]

    if K is None:
        # If K not provided, try to infer from labels present in test loader
        # (May undercount if some classes are absent. Prefer passing K from training.)
        print("[Info] K not provided; attempting to infer from test loader labels (may be lower than full K).")
        K = scan_num_classes_from_loader(test_loader)

    if id_to_name is None:
        id_to_name = build_default_id_to_name(K)

    # Build model and load weights
    model = DiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=K).to(device)
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state, strict=False)
    model.eval()

    print("\n" + "="*60)
    print("Final Test Set Evaluation")
    print("="*60)

    te = eval_epoch(model, test_loader, device, K, compute_acc=True)
    print(f"[Test] mse_b={te['mse_b']:.4f} mse_a={te['mse_a']:.4f} acc={te['acc']:.3f} ΔHI_impr={te['delta_mean']:.3f}")

    # Predictions & visualization data
    y_true, y_pred, all_delta_mean, all_uids, keep_curves = collect_test_predictions(
        model, test_loader, device, max_curve_keep=max_curve_keep, K=K, id_to_name=id_to_name
    )

    # Reports (only if labels present)
    if y_true.size > 0:
        print("\n[Classification Report]")
        labels_full = list(range(K))
        target_names = [id_to_name.get(i, f"Class_{i}") for i in labels_full]
        # --- FIX: force labels + names to avoid mismatch when some classes are absent in predictions ---
        print(classification_report(
            y_true, y_pred,
            labels=labels_full,
            target_names=target_names,
            zero_division=0
        ))

        print("\n[Confusion Matrix]")
        print(confusion_matrix(y_true, y_pred, labels=labels_full))

        df_delta = pd.DataFrame({
            "uid": all_uids[:len(y_true)],
            "true": y_true,
            "pred": y_pred,
            "delta_hi_mean": all_delta_mean[:len(y_true)]
        })
        print("\n[Learned ΔHI Statistics] By true class")
        print(df_delta.groupby("true")["delta_hi_mean"].describe())
        topk_by_delta(df_delta, id_to_name, k=3)

        if visualize:
            print("\n[Visualization] Learned health index curves with liquid weight adaptation...")
            plot_hi_examples_aligned(keep_curves, id_to_name, n_show=16, seed=0)

            print("\n[Visualization] Liquid operator weights evolution...")
            plot_liquid_weights(keep_curves, n_show=16, seed=0)

            print("\n[Visualization] Sensor reconstruction predictions...")
            plot_sensor_examples_aligned(keep_curves, sensor_idx_list=None, n_cols=4)
    else:
        print("\n[Info] No labels in test loader; skipping classification report and confusion matrix.")

    return {"test_metrics": te, "K": K, "id_to_name": id_to_name, "curves": keep_curves}
meta = {
    "K": int(K),
    "id_to_name": {int(k): str(v) for k, v in id_to_name.items()},
    "sensor_dim": int(C),
    "best_es": None,             # Leave blank or fill manually if the early-stopping metric from training is unknown
    "best_epoch": None,          # Same as above
    "class_weights": _serialize_class_weights(class_weights_obj)
}



results = test_only(
    test_loader=test_loader,
    ckpt_path=str(resolve_data_file("best_model_NGFAID_250_100_6.pth")),
    K=K,
    id_to_name=id_to_name,
    device=None,
    visualize=True,          # Set to True to generate plots; use False in headless environments
    max_curve_keep=24
)


## Aggregate Operator Weight Visualization


In [ ]:
import os, json, math, random
import numpy as np
import torch
import matplotlib.pyplot as plt

def plot_class_operator_weight_heatmaps(
    checkpoint_path: str,
    test_loader,
    target_class,                     # int id (e.g., 3) or class name (e.g., "Class_3")
    device: torch.device = None,
    max_samples: int = 9,
    seed: int = 0,
    max_curve_keep: int = 2000,       # ensure we keep enough samples with weights
    fallback_to_pred_if_no_labels: bool = True,
):
    """
    Load the best model checkpoint and plot operator-weight heatmaps (pre/post) for samples
    belonging to a specified class.

    Args:
        checkpoint_path: Path to the saved .pth model (the function will also try to read the
                         matching '<checkpoint>_meta.json' if it exists).
        test_loader:     A DataLoader that yields batches as defined in your snippet.
        target_class:    Either a class id (int) or a class name (str). If labels are missing
                         or ignored, can fallback to predicted label when enabled.
        device:          torch.device to perform inference on. Defaults to CUDA if available.
        max_samples:     Maximum number of sample heatmaps to draw.
        seed:            RNG seed for shuffling/selection.
        max_curve_keep:  How many curves to retain from collect_test_predictions (must be high
                         enough to cover your filtered class subset).
        fallback_to_pred_if_no_labels:
                         If True, and ground-truth labels are missing/ignored, filter by
                         predicted labels instead.
    """

    # ---------- 0) Device ----------
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ---------- 1) Read meta or infer K/C/id_to_name ----------
    meta_path = checkpoint_path.replace(".pth", "_meta.json")
    has_meta = os.path.isfile(meta_path)
    if has_meta:
        with open(meta_path, "r") as f:
            meta = json.load(f)
        K = int(meta["K"])
        id_to_name = {int(k): v for k, v in meta["id_to_name"].items()}
        C = int(meta["sensor_dim"])
    else:
        # Fallback: infer from loader
        K = scan_num_classes_from_loader(test_loader)
        id_to_name = build_default_id_to_name(K)
        first_batch = next(iter(test_loader))
        C = first_batch["x_before"].shape[-1]

    # Map class name -> id if needed
    if isinstance(target_class, str):
        # Exact match first, else try case-insensitive
        name_to_id = {v: k for k, v in id_to_name.items()}
        if target_class in name_to_id:
            target_id = name_to_id[target_class]
        else:
            ci = {v.lower(): k for k, v in id_to_name.items()}
            if target_class.lower() in ci:
                target_id = ci[target_class.lower()]
            else:
                raise ValueError(f"Class name '{target_class}' not found in id_to_name: {id_to_name}")
    else:
        target_id = int(target_class)

    # ---------- 2) Rebuild model and load weights ----------
    model = DiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=K).to(device)
    state = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state, strict=True)
    model.eval()

    # ---------- 3) Collect predictions/curves (keep many for filtering) ----------
    y_true, y_pred, _, _, curves = collect_test_predictions(
        model=model,
        loader=test_loader,
        device=device,
        max_curve_keep=max_curve_keep,
        K=K,
        id_to_name=id_to_name
    )

    # ---------- 4) Filter curves by target class ----------
    def valid_label(v):
        return (v is not None) and (v >= 0)

    filtered = []
    for ex in curves:
        w_b = ex.get("weights_before", None)
        w_a = ex.get("weights_after", None)
        if w_b is None or w_a is None:
            continue  # need weights to plot

        t = ex.get("true", -1)
        p = ex.get("pred", -1)

        if valid_label(t) and t == target_id:
            filtered.append(ex)
        elif (not valid_label(t)) and fallback_to_pred_if_no_labels and p == target_id:
            filtered.append(ex)

    if len(filtered) == 0:
        print(f"[Info] No samples found for class '{target_class}' (id={target_id}).")
        return

    # ---------- 5) Plot settings ----------
    op_names = ["MonotonicLinear", "MonotonicFlat", "ConcaveLog", "SaturationSigmoid",
                "HingeReLU", "Polynomial", "DampedSin", "PiecewiseLinear"]

    random.Random(seed).shuffle(filtered)
    n_show = min(max_samples, len(filtered))
    cols = 3
    rows = math.ceil(n_show / cols)

    fig_w = 5.4 * cols
    fig_h = 3.8 * rows
    fig = plt.figure(figsize=(fig_w, fig_h))

    for idx in range(n_show):
        ex = filtered[idx]
        w_b = np.asarray(ex["weights_before"])  # (T_b, K)
        w_a = np.asarray(ex["weights_after"])   # (T_a, K)

        if w_b.ndim != 2 or w_a.ndim != 2:
            continue

        T_b, K_b = w_b.shape
        T_a, K_a = w_a.shape
        K_eff = min(K_b, K_a, len(op_names))
        w_b = w_b[:, :K_eff]
        w_a = w_a[:, :K_eff]

        # Concatenate along time to align pre | post; add separator via a line (not NaN)
        combined = np.concatenate([w_b, w_a], axis=0)  # (T_b+T_a, K_eff)

        ax = plt.subplot(rows, cols, idx + 1)
        # Heatmap expects (rows=ops, cols=time). Transpose to (K, T)
        im = ax.imshow(combined.T, aspect="auto", interpolation="nearest")
        # Vertical separator between pre and post
        ax.axvline(T_b - 0.5, color="k", linestyle=":", linewidth=1.0)

        # Axes labels and ticks
        ax.set_yticks(range(K_eff))
        ax.set_yticklabels([op_names[i] for i in range(K_eff)], fontsize=8)
        ax.set_xlabel("Time (pre  |  post)")
        ax.set_title(
            f"ID={ex.get('sample_id','?')} | True={id_to_name.get(ex.get('true',-1),'?')} | "
            f"Pred={id_to_name.get(ex.get('pred',-1),'?')} | p={ex['prob'][ex['pred']]:.2f}"
            if ex.get("prob") is not None else
            f"ID={ex.get('sample_id','?')} | True={id_to_name.get(ex.get('true',-1),'?')} | "
            f"Pred={id_to_name.get(ex.get('pred',-1),'?')}"
        , fontsize=9)

        # Colorbar (small)
        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.set_ylabel("Weight", rotation=270, labelpad=10)

    plt.suptitle(
        f"Operator Weight Heatmaps — Class: {id_to_name.get(target_id, f'Class_{target_id}')}",
        fontsize=12, y=0.995
    )
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

# -----------------------
# Example usage (English)
# -----------------------
plot_class_operator_weight_heatmaps(
    checkpoint_path=str(resolve_data_file("best_model_NGFAID_250_100_6.pth")),
    test_loader=test_loader,     # provide your test DataLoader
    target_class=3,              # or "Class_3"
    device=None,                 # auto-select
    max_samples=6,
    seed=0
)


In [ ]:
plot_class_operator_weight_heatmaps(
    checkpoint_path=str(resolve_data_file("best_model_NGFAID_250_100_6.pth")),
    test_loader=test_loader,     # provide your test DataLoader
    target_class=8,              # or "Class_3"
    device=None,                 # auto-select
    max_samples=6,
    seed=0
)


### Next Step: Aggregate Heatmap Comparison


In [ ]:
import os, json, math, random
import numpy as np
import torch
import matplotlib.pyplot as plt

def plot_aggregate_class_operator_heatmap(
    checkpoint_path: str,
    test_loader,
    target_class,                      # int id (e.g., 3) or class name (e.g., "Class_3")
    device: torch.device = None,
    pre_bins: int = 60,
    post_bins: int = 60,
    normalize: str = "mean",           # "mean" (default) or "sum" for overlay style
    max_curve_keep: int = 200000,      # keep enough curves to cover the whole class
    fallback_to_pred_if_no_labels: bool = True,
    seed: int = 0,
):
    """
    Load best checkpoint and plot a single aggregated heatmap of operator weights
    (pre and post maintenance) for ALL samples in a target class.

    Steps:
      1) Load model and meta (K, id_to_name, C).
      2) Collect predictions and keep per-sample operator weights.
      3) Filter curves by target class (true label if available; else predicted).
      4) Time-normalize (resample) each sample's pre and post weights to fixed bins.
      5) Aggregate across samples (mean or sum) and plot one combined heatmap.

    Args:
        checkpoint_path: Path to model .pth file; will also try '<pth>_meta.json'.
        test_loader:     DataLoader yielding batches defined in your snippet.
        target_class:    Class id (int) or class name (str).
        device:          Device for inference; auto if None.
        pre_bins:        Target time bins for the pre-maintenance part.
        post_bins:       Target time bins for the post-maintenance part.
        normalize:       "mean" for average aggregation (default), "sum" for pure overlay.
        max_curve_keep:  Max curves stored from `collect_test_predictions`.
        fallback_to_pred_if_no_labels:
                         Use predicted label to filter when ground-truth is absent/ignored.
        seed:            RNG seed for deterministic selection (not strictly needed here).
    """

    # ----------------- helpers -----------------
    def resample_time_major(arr_TK: np.ndarray, tgt_len: int) -> np.ndarray:
        """Resample a (T, K) array along time axis to target length using linear interpolation."""
        T, K = arr_TK.shape
        if T <= 1 or tgt_len == T:
            # trivial cases: repeat or return as-is
            if T == tgt_len:
                return arr_TK.copy()
            if T <= 1:
                return np.repeat(arr_TK, tgt_len, axis=0)
        # Build source and target positions in [0, 1]
        src = np.linspace(0.0, 1.0, num=T, endpoint=True)
        dst = np.linspace(0.0, 1.0, num=tgt_len, endpoint=True)
        out = np.empty((tgt_len, K), dtype=arr_TK.dtype)
        # vectorized per-operator interpolation
        for k in range(K):
            out[:, k] = np.interp(dst, src, arr_TK[:, k])
        return out

    # ----------------- device -----------------
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ----------------- meta / model -----------------
    meta_path = checkpoint_path.replace(".pth", "_meta.json")
    if os.path.isfile(meta_path):
        with open(meta_path, "r") as f:
            meta = json.load(f)
        K = int(meta["K"])
        id_to_name = {int(k): v for k, v in meta["id_to_name"].items()}
        C = int(meta["sensor_dim"])
    else:
        # Fallback: infer from test loader
        K = scan_num_classes_from_loader(test_loader)
        id_to_name = build_default_id_to_name(K)
        first_batch = next(iter(test_loader))
        C = first_batch["x_before"].shape[-1]

    # Map class name -> id if needed
    if isinstance(target_class, str):
        name_to_id = {v: k for k, v in id_to_name.items()}
        if target_class in name_to_id:
            target_id = name_to_id[target_class]
        else:
            ci = {v.lower(): k for k, v in id_to_name.items()}
            if target_class.lower() in ci:
                target_id = ci[target_class.lower()]
            else:
                raise ValueError(f"Class name '{target_class}' not found in id_to_name: {id_to_name}")
    else:
        target_id = int(target_class)

    # Rebuild model and load weights
    model = DiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=K).to(device)
    state = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state, strict=True)
    model.eval()

    # ----------------- collect curves -----------------
    _, _, _, _, curves = collect_test_predictions(
        model=model,
        loader=test_loader,
        device=device,
        max_curve_keep=max_curve_keep,
        K=K,
        id_to_name=id_to_name
    )

    # ----------------- filter by class -----------------
    def valid_label(v): return (v is not None) and (v >= 0)
    selected = []
    for ex in curves:
        wb = ex.get("weights_before", None)
        wa = ex.get("weights_after", None)
        if wb is None or wa is None:
            continue
        t = ex.get("true", -1)
        p = ex.get("pred", -1)
        if valid_label(t) and t == target_id:
            selected.append(ex)
        elif (not valid_label(t)) and fallback_to_pred_if_no_labels and p == target_id:
            selected.append(ex)

    if len(selected) == 0:
        print(f"[Info] No samples found for class '{target_class}' (id={target_id}).")
        return

    # ----------------- resample & aggregate -----------------
    # Operator count (should be 8 by your design). Take min across examples to be safe.
    K_ops = min(
        min(ex["weights_before"].shape[1] for ex in selected),
        min(ex["weights_after"].shape[1]  for ex in selected)
    )
    # Optional: align to known operator list length
    op_names = ["MonotonicLinear", "MonotonicFlat", "ConcaveLog", "SaturationSigmoid",
                "HingeReLU", "Polynomial", "DampedSin", "PiecewiseLinear"]
    K_eff = min(K_ops, len(op_names))

    pre_stack  = []
    post_stack = []

    random.Random(seed).shuffle(selected)  # no effect on aggregation, keeps determinism

    for ex in selected:
        wb = np.asarray(ex["weights_before"])[:, :K_eff]  # (T_b, K_eff)
        wa = np.asarray(ex["weights_after"])[:,  :K_eff]  # (T_a, K_eff)
        wb_rs = resample_time_major(wb, pre_bins)   # (pre_bins,  K_eff)
        wa_rs = resample_time_major(wa, post_bins)  # (post_bins, K_eff)
        pre_stack.append(wb_rs)
        post_stack.append(wa_rs)

    pre_stack  = np.stack(pre_stack,  axis=0)  # (N, pre_bins,  K_eff)
    post_stack = np.stack(post_stack, axis=0)  # (N, post_bins, K_eff)

    if normalize.lower() == "sum":
        agg_pre  = pre_stack.sum(axis=0)       # (pre_bins,  K_eff)
        agg_post = post_stack.sum(axis=0)      # (post_bins, K_eff)
        cbar_label = "Aggregated weight (sum)"
    else:
        agg_pre  = pre_stack.mean(axis=0)      # (pre_bins,  K_eff)
        agg_post = post_stack.mean(axis=0)     # (post_bins, K_eff)
        cbar_label = "Aggregated weight (mean)"

    combined = np.concatenate([agg_pre, agg_post], axis=0)  # (pre_bins+post_bins, K_eff)

    # ----------------- plot -----------------
    fig = plt.figure(figsize=(8.8, 4.8))
    ax = plt.gca()
    im = ax.imshow(combined.T, aspect="auto", interpolation="nearest")
    ax.axvline(pre_bins - 0.5, color="k", linestyle=":", linewidth=1.0)

    ax.set_yticks(range(K_eff))
    ax.set_yticklabels([op_names[i] for i in range(K_eff)], fontsize=8)
    ax.set_xlabel("Time (pre  |  post)")
    title_cls = id_to_name.get(target_id, f"Class_{target_id}")
    ax.set_title(f"Aggregated Operator Weights — Class: {title_cls} (N={len(selected)})", fontsize=11)

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.set_ylabel(cbar_label, rotation=270, labelpad=10)

    plt.tight_layout()
    plt.show()

    # Optionally return arrays for further analysis
    return {
        "class_id": target_id,
        "class_name": title_cls,
        "n_samples": len(selected),
        "agg_pre": agg_pre,            # shape: (pre_bins,  K_eff)
        "agg_post": agg_post,          # shape: (post_bins, K_eff)
        "op_names": [op_names[i] for i in range(K_eff)],
    }

# -----------------------
# Example (English)
# -----------------------
out = plot_aggregate_class_operator_heatmap(
    checkpoint_path=str(resolve_data_file("best_model_NGFAID_250_100_5.pth")),
    test_loader=test_loader,
    target_class="Class_5",     # or 3
    device=None,                # auto
    pre_bins=60,
    post_bins=60,
    normalize="mean"            # or "sum" for literal overlay
)


## Part 3G: Final Training and Evaluation Pass


In [ ]:


# ============================================================
# Monotonic-Decreasing HI + aligned plotting (before -> after)
# Adapted to external train/val/test loaders that yield:
#   batch = {
#       "x_before": (B, L, C), "x_after": (B, L, C),
#       "y": LongTensor[B] or None,
#       "meta": list[dict] (each may contain 'pair_id', 'matching_cost', ...)
#   }
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import os
import random
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix

# ---------------------------------------------------------------------
# Utilities for the NEW loaders
# ---------------------------------------------------------------------
def adapt_batch(batch, device):
    """
    Adapt a batch from your loaders to the tensors expected by the model.
    Ensures fixed-length mask/lengths and extracts optional meta.
    """
    xb = batch["x_before"].to(device)  # (B, L, C)
    xa = batch["x_after"].to(device)   # (B, L, C)


    B, L, _ = xb.shape
    mask = torch.ones(B, L, dtype=xb.dtype, device=device)
    lengths = torch.full((B,), L, dtype=torch.long, device=device)

    labels = batch.get("y", None)
    if labels is not None and isinstance(labels, torch.Tensor):
        # Filter out ignore_index labels
        valid_mask = labels != -100
        if valid_mask.any():
            labels = labels.to(device)
        else:
            labels = None
    elif labels is not None:
        labels = None

    meta = batch.get("meta", None)
    sample_ids = []
    matching_costs = torch.zeros(B, dtype=torch.float32, device=device)
    if isinstance(meta, (list, tuple)) and len(meta) == B:
        for i, m in enumerate(meta):
            if isinstance(m, dict):
                sample_ids.append(str(m.get("pair_id", f"sample_{i}")))
                if "matching_cost" in m and m["matching_cost"] is not None:
                    matching_costs[i] = float(m["matching_cost"])
            else:
                sample_ids.append(f"sample_{i}")
    else:
        sample_ids = [f"sample_{i}" for i in range(B)]

    return xb, xa, labels, mask, lengths, sample_ids, matching_costs


def scan_num_classes_from_loader(train_loader):
    """
    Determine K (number of classes) from the training loader.
    """
    max_class = -1
    for batch in train_loader:
        y = batch.get("y", None)
        if y is None:
            continue
        if isinstance(y, torch.Tensor):
            y_np = y.detach().cpu().numpy()
            valid_labels = y_np[y_np >= 0]  # filter out negative labels
            if len(valid_labels) > 0:
                max_class = max(max_class, int(valid_labels.max()))

    if max_class < 0:
        raise ValueError("No valid labels found in the training loader to determine K.")

    K = max_class + 1
    return K


def build_default_id_to_name(K):
    return {i: f"Class_{i}" for i in range(K)}


# ---------------------------------------------------------------------
# Core model components
# ---------------------------------------------------------------------
class BoltzmannKAN(nn.Module):
    def __init__(self, in_ch, out_ch=4):
        super().__init__()
        self.E  = nn.Parameter(torch.zeros(out_ch, in_ch))
        self.kT = nn.Parameter(torch.ones(out_ch, in_ch))
    def forward(self, x):
        B,T,C = x.shape
        kT = torch.clamp(F.softplus(self.kT), 0.01, 10.0).unsqueeze(0).unsqueeze(2)
        E  = torch.clamp(self.E, -10.0, 10.0).unsqueeze(0).unsqueeze(2)
        x_ = torch.clamp(x.unsqueeze(1), -10.0, 10.0)
        exp = torch.clamp((E - x_) / kT, -50, 50)
        w   = torch.sigmoid(exp)
        h   = (w * x_).sum(dim=3).permute(0, 2, 1)
        return torch.clamp(F.softplus(h), 0.0, 100.0)

class BaseOp(nn.Module):
    def __init__(self):
        super().__init__()
        self.param_modulator = None
    def set_param_modulator(self, modulator): self.param_modulator = modulator
    def get_modulated_params(self, context):
        if self.param_modulator is None: return {}
        return self.param_modulator(context)

class MonotonicLinearOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 2.0  # Reduced from 5.0 to 2.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_bias = self.bias
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        bias  = torch.clamp(base_bias + mod.get('bias_offset', 0.0), -5.0, 5.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * (xm + bias)), 0.0, 100.0)

class MonotonicFlatOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 1e-3, 1.0
    def _cum(self, x):
        B, T = x.shape
        diff = F.relu(x[:, 1:] - x[:, :-1])
        zero_init = torch.zeros(B, 1, device=x.device, dtype=x.dtype)
        return torch.cat([zero_init, torch.cumsum(diff, dim=1)], dim=1)
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_bias = self.bias
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        bias  = torch.clamp(base_bias + mod.get('bias_offset', 0.0), -2.0, 2.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        cum_result = self._cum(xm)
        return torch.clamp(F.softplus(scale * (cum_result + bias)), 0.0, 100.0)

class ConcaveLogOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.eps = 1e-3
        self.smin, self.smax = 0.01, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        eps  = torch.clamp(self.eps + mod.get('eps_offset', 0.0), 1e-6, 1e-1)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * torch.log(torch.abs(xm) + eps)), 0.0, 100.0)

class SaturationSigmoidOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.raw_slope = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
        self.lmin, self.lmax = 0.1, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_slope = torch.clamp(F.softplus(self.raw_slope), self.lmin, self.lmax)
        base_bias  = self.bias
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        slope = torch.clamp(base_slope + mod.get('slope_offset', 0.0), self.lmin, self.lmax)
        bias  = torch.clamp(base_bias  + mod.get('bias_offset', 0.0), -3.0, 3.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * torch.sigmoid(slope * (xm - bias))), 0.0, 100.0)

class HingeReLUOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.threshold = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_threshold = self.threshold
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        threshold = torch.clamp(base_threshold + mod.get('threshold_offset', 0.0), -3.0, 3.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * F.relu(xm - threshold)), 0.0, 100.0)

class PolynomialOp(BaseOp):
    def __init__(self, deg=2):  # Reduced default degree for stability
        super().__init__()
        self.raw_coeff = nn.Parameter(torch.zeros(deg))
        self.deg = deg
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_coeff = torch.clamp(F.softplus(self.raw_coeff), 0.01, 2.0)  # Reduced from 5.0 to 2.0
        xm = torch.clamp(torch.tanh(h.mean(-1)), -3.0, 3.0)  # Added tanh for stability
        coeff_offset = mod.get('coeff_offset', torch.zeros_like(base_coeff))
        if coeff_offset.dim() > 1:
            coeff_offset = coeff_offset.mean(0)
        coeff = torch.clamp(base_coeff + coeff_offset, 0.01, 2.0)  # Also reduced to 2.0
        y = torch.zeros_like(xm)
        for i in range(self.deg):
            y = y + coeff[i] * torch.clamp(xm ** (i + 1), -100.0, 100.0)
        y = torch.tanh(y) * 5.0  # Added tanh scaling
        return torch.clamp(F.softplus(y), 0.0, 100.0)

class DampedSinOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.raw_freq = nn.Parameter(torch.tensor(0.0))
        self.raw_lambda = nn.Parameter(torch.tensor(0.0))
        self.phase = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
        self.fmin, self.fmax = 0.1, 5.0
        self.lmin, self.lmax = 0.01, 3.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_freq  = torch.clamp(F.softplus(self.raw_freq), self.fmin, self.fmax)
        base_lam   = torch.clamp(F.softplus(self.raw_lambda), self.lmin, self.lmax)
        base_phase = torch.clamp(self.phase, -10.0, 10.0)
        scale = torch.clamp(base_scale + mod.get('scale_offset', 0.0), self.smin, self.smax)
        freq  = torch.clamp(base_freq  + mod.get('freq_offset', 0.0), self.fmin, self.fmax)
        lam   = torch.clamp(base_lam   + mod.get('lambda_offset', 0.0), self.lmin, self.lmax)
        phase = torch.clamp(base_phase + mod.get('phase_offset', 0.0), -10.0, 10.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        damp = 1 / (1 + lam * torch.abs(xm))
        return torch.clamp(F.softplus(scale * damp * (torch.sin(freq * xm + phase) + 1)), 0.0, 100.0)

class PiecewiseLinearOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_k1 = nn.Parameter(torch.tensor(0.0))
        self.raw_k2 = nn.Parameter(torch.tensor(0.0))
        self.threshold = nn.Parameter(torch.tensor(0.0))
        self.kmin, self.kmax = 0.01, 5.0
    def forward(self, h, context=None):
        mod = self.get_modulated_params(context) if context is not None else {}
        base_k1 = torch.clamp(F.softplus(self.raw_k1), self.kmin, self.kmax)
        base_k2 = torch.clamp(F.softplus(self.raw_k2), self.kmin, self.kmax)
        base_thr = torch.clamp(self.threshold, -5.0, 5.0)
        k1 = torch.clamp(base_k1 + mod.get('k1_offset', 0.0), self.kmin, self.kmax)
        k2 = torch.clamp(base_k2 + mod.get('k2_offset', 0.0), self.kmin, self.kmax)
        thr= torch.clamp(base_thr + mod.get('threshold_offset', 0.0), -5.0, 5.0)
        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        left = k1 * xm
        right = k1 * thr + k2 * (xm - thr)
        out = torch.where(xm <= thr, left, right)
        return torch.clamp(F.softplus(out), 0.0, 100.0)

class ParameterModulator(nn.Module):
    def __init__(self, context_dim, param_configs):
        super().__init__()
        self.param_configs = param_configs
        self.param_predictors = nn.ModuleDict()
        for name, info in param_configs.items():
            dim = info.get('dim', 1)
            self.param_predictors[name] = nn.Sequential(
                nn.Linear(context_dim, 64), nn.ReLU(), nn.Dropout(0.1),
                nn.Linear(64, 32), nn.ReLU(),
                nn.Linear(32, dim), nn.Tanh()
            )
    def forward(self, context):
        global_context = context.mean(dim=1)  # (B, context_dim)
        modulated = {}
        for name, predictor in self.param_predictors.items():
            info = self.param_configs[name]
            raw_off = predictor(global_context)
            scale = info.get('scale', 0.1)
            modulated[name] = raw_off * scale
        return modulated

class LiquidWeightGenerator(nn.Module):
    def __init__(self, h_dim, x_dim, n_ops, hidden_dim=64, tau_min=0.5, tau_max=3.0):
        super().__init__()
        self.n_ops = n_ops
        self.tau_min, self.tau_max = tau_min, tau_max
        self.h_feature_net = nn.Sequential(nn.Linear(h_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(0.1))
        self.x_feature_net = nn.Sequential(nn.Linear(x_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(0.1))
        self.temporal_encoder = nn.Sequential(nn.Linear(3, hidden_dim // 4), nn.ReLU())
        combined_dim = hidden_dim + hidden_dim // 4
        self.feature_fusion = nn.Sequential(
            nn.Linear(combined_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.op_branches = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 4), nn.ReLU(), nn.Linear(hidden_dim // 4, 1))
            for _ in range(n_ops)
        ])
        self.temp_predictor = nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 4), nn.ReLU(), nn.Linear(hidden_dim // 4, 1))
        self.global_bias_scale = 0.0  # Disabled random bias injection
        for branch in self.op_branches:
            nn.init.xavier_uniform_(branch[0].weight)
            nn.init.xavier_uniform_(branch[2].weight)

    def forward(self, h_multi, x, is_after=False, training_noise=0.0):
        B, T, h_dim = h_multi.shape
        _, _, x_dim = x.shape
        h_features = self.h_feature_net(h_multi)
        x_features = self.x_feature_net(x)
        t_norm = torch.arange(T, device=x.device, dtype=x.dtype).unsqueeze(0).expand(B, -1) / max(T-1, 1)
        dt = torch.zeros_like(t_norm)
        dt[:, 1:] = t_norm[:, 1:] - t_norm[:, :-1]
        phase = torch.sin(2 * 3.14159 * t_norm)
        temporal_input = torch.stack([t_norm, dt, phase], dim=-1)
        temporal_features = self.temporal_encoder(temporal_input)
        combined = torch.cat([h_features, x_features, temporal_features], dim=-1)
        fused = self.feature_fusion(combined)
        raw_logits = []
        for branch in self.op_branches:
            raw_logits.append(branch(fused).squeeze(-1))
        raw_weights = torch.stack(raw_logits, dim=-1)                     # (B, T, K)

        # Add training noise for exploration
        if self.training and training_noise > 0:
            noise = torch.randn_like(raw_weights) * training_noise
            raw_weights = raw_weights + noise

        # NaN protection and clipping
        raw_weights = torch.nan_to_num(raw_weights, nan=0.0, posinf=1e6, neginf=-1e6)
        raw_weights = raw_weights - raw_weights.mean(dim=-1, keepdim=True)
        raw_weights = raw_weights.clamp(-20.0, 20.0)  # Added clipping

        raw_temp = self.temp_predictor(fused).squeeze(-1)                 # (B, T)
        raw_temp = torch.nan_to_num(raw_temp, nan=0.0, posinf=1e6, neginf=-1e6)

        temperature = torch.clamp(F.softplus(raw_temp) + self.tau_min, self.tau_min, self.tau_max)

        # Asymmetric temperature scaling: before sharper (×0.6), after smoother (×1.4)
        if is_after:
            temperature = temperature * 1.4
        else:
            temperature = temperature * 0.6

        weights = F.softmax(raw_weights / temperature.unsqueeze(-1), dim=-1)
        return weights, temperature

class CustomKAN(nn.Module):
    def __init__(self, ops, h_dim, x_dim):
        super().__init__()
        self.ops = nn.ModuleList(ops)
        self.n_ops = len(ops)
        self.weight_generator = LiquidWeightGenerator(h_dim=h_dim, x_dim=x_dim, n_ops=self.n_ops, hidden_dim=128)
        self.op_feature_extractors = nn.ModuleList([
            nn.Sequential(nn.Linear(h_dim, h_dim), nn.ReLU(), nn.Linear(h_dim, h_dim))
            for _ in range(self.n_ops)
        ])
        self.param_modulators = nn.ModuleList()
        context_dim = h_dim + x_dim
        for op in self.ops:
            param_configs = self._get_param_configs_for_op(op)
            if param_configs:
                modulator = ParameterModulator(context_dim, param_configs)
                self.param_modulators.append(modulator)
                op.set_param_modulator(modulator)
            else:
                self.param_modulators.append(None)
        self.raw_gain = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))

    def _get_param_configs_for_op(self, op):
        # Reduced modulation scales to prevent single-operator dominance
        if isinstance(op, MonotonicLinearOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.07}, 'bias_offset': {'dim': 1, 'scale': 0.1}}
        if isinstance(op, MonotonicFlatOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.03}, 'bias_offset': {'dim': 1, 'scale': 0.07}}
        if isinstance(op, ConcaveLogOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.07}, 'eps_offset': {'dim': 1, 'scale': 0.003}}
        if isinstance(op, SaturationSigmoidOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.07}, 'slope_offset': {'dim': 1, 'scale': 0.1}, 'bias_offset': {'dim': 1, 'scale': 0.1}}
        if isinstance(op, HingeReLUOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.07}, 'threshold_offset': {'dim': 1, 'scale': 0.1}}
        if isinstance(op, PolynomialOp):
            return {'coeff_offset': {'dim': op.deg, 'scale': 0.07}}
        if isinstance(op, DampedSinOp):
            return {'scale_offset': {'dim': 1, 'scale': 0.07}, 'freq_offset': {'dim': 1, 'scale': 0.1},
                    'lambda_offset': {'dim': 1, 'scale': 0.07}, 'phase_offset': {'dim': 1, 'scale': 0.17}}
        if isinstance(op, PiecewiseLinearOp):
            return {'k1_offset': {'dim': 1, 'scale': 0.07}, 'k2_offset': {'dim': 1, 'scale': 0.07},
                    'threshold_offset': {'dim': 1, 'scale': 0.1}}
        return {}

    def forward(self, h_multi, x, is_after=False, training_noise=0.0, freeze_modulator=False):
        op_features = [ext(h_multi) for ext in self.op_feature_extractors]
        context = torch.cat([h_multi, x], dim=-1)

        # Optionally freeze parameter modulation
        if freeze_modulator:
            with torch.no_grad():
                outs = [op(op_features[i], context) for i, op in enumerate(self.ops)]
        else:
            outs = [op(op_features[i], context) for i, op in enumerate(self.ops)]

        Tm = min(o.size(1) for o in outs)
        outs = [o[:, :Tm] for o in outs]
        h_multi_aligned = h_multi[:, :Tm, :]
        x_aligned = x[:, :Tm, :]
        op_stack = torch.stack(outs, dim=-1)  # (B, Tm, K)
        weights, temperature = self.weight_generator(h_multi_aligned, x_aligned, is_after=is_after, training_noise=training_noise)
        damage = torch.sum(op_stack * weights, dim=-1)  # (B, Tm)
        damage = torch.clamp(damage, 0.0, 100.0)
        gain = torch.clamp(F.softplus(self.raw_gain), 0.1, 2.0)
        bias_val = torch.clamp(self.bias, -2.0, 2.0)
        damage = torch.clamp(gain * damage + bias_val, 0.0, 100.0)
        return damage, weights, temperature, op_stack

class TrendEncoder(nn.Module):
    """
    Vectorized monotonic decreasing HI projection with learnable nonlinear decay.
    Enhanced with asymmetric epsilon and light jitter for natural step-like monotonicity.
    """
    def __init__(self, in_ch, trend_ch=4):
        super().__init__()
        self.boltz = BoltzmannKAN(in_ch, out_ch=trend_ch)
        ops = [MonotonicLinearOp(), MonotonicFlatOp(), ConcaveLogOp(),
               SaturationSigmoidOp(), HingeReLUOp(), PolynomialOp(),
               DampedSinOp(), PiecewiseLinearOp()]
        self.customkan = CustomKAN(ops, h_dim=trend_ch, x_dim=in_ch)

        self.proj_gain = nn.Parameter(torch.tensor(1.0))
        self.proj_bias = nn.Parameter(torch.tensor(0.0))

        self.health_aware_transform = nn.Sequential(
            nn.Linear(in_ch, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )

        # Learnable nonlinear decay rate predictor
        self.rate_head = nn.Sequential(
            nn.Linear(in_ch + 1, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )
        self.rate_scale = nn.Parameter(torch.tensor(0.1))  # Global decay intensity

        # Reduced linear enforcer influence
        self.linear_enforcer = nn.Sequential(
            nn.Linear(in_ch, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )

        # Steady-state level predictor for plateau targeting
        self.steady_state_head = nn.Sequential(
            nn.Linear(in_ch, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Sigmoid()
        )

    def forward(self, x, is_after=False, training_noise=0.0, freeze_modulator=False, mixture_weight=0.6):  # x:(B,T,C)
        B, T, C = x.shape
        h_multi = self.boltz(x)                               # (B,T,trend_ch)
        damage, weights, temperature, op_outputs = self.customkan(h_multi, x, is_after=is_after,
                                                                  training_noise=training_noise,
                                                                  freeze_modulator=freeze_modulator)

        x_reshaped = x.view(-1, C)
        health_direct = self.health_aware_transform(x_reshaped).view(B, T)  # (B,T)

        t = torch.arange(T, device=x.device, dtype=x.dtype).unsqueeze(0).expand(B, -1)
        t_normalized = t / (T - 1 + 1e-6)

        # Learnable nonlinear decay with plateau-aware rate control
        time_feat = t_normalized.unsqueeze(-1)                 # (B,T,1)
        rate_in = torch.cat([x, time_feat], dim=-1)           # (B,T,C+1)
        raw_rate = self.rate_head(rate_in).squeeze(-1)         # (B,T)
        rate = F.softplus(raw_rate) * torch.clamp(self.rate_scale, 1e-3, 2.0)

        cum_rate = torch.cumsum(rate, dim=1)                   # (B,T)
        nonlinear_decay = torch.exp(-cum_rate)                 # Monotonic decreasing and flexible

        g = torch.clamp(F.softplus(self.proj_gain), 0.1, 5.0)
        b = torch.clamp(self.proj_bias, -5.0, 5.0)
        damage_normalized = torch.sigmoid(g*(damage + b))

        # Asymmetric mixture weight: before relies more on damage (0.9), after less (0.45)
        combined_health = health_direct * (1 - mixture_weight * damage_normalized)

        # Asymmetric linear channel influence: before=looser, after=slightly tighter
        lw = self.linear_enforcer(x_reshaped).view(B, T)
        linear_weight = (0.03 if not is_after else 0.06) * lw * (1.0 - t_normalized)

        hi = linear_weight * nonlinear_decay + (1 - linear_weight) * combined_health

        # ---- Strictly non-increasing with asymmetric epsilon & light jitter ----
        if is_after:
            eps = 1e-3       # After maintenance: larger epsilon for smoother monotonicity
        else:
            eps = 5e-5       # Before maintenance: very small epsilon for fine steps
            if self.training:
                hi = hi + 1e-3 * torch.randn_like(hi)  # Light jitter during training to create steps

        idx = torch.arange(T, device=hi.device, dtype=hi.dtype).unsqueeze(0)  # (1, T)
        hi_mon, _ = torch.cummin(hi + eps * idx, dim=1)
        hi = hi_mon - eps * idx
        # -------------------------------------------------------------------------

        # Predict steady-state level for plateau targeting
        x_global = x.mean(dim=1)  # (B, C) - global sensor statistics
        steady_level = self.steady_state_head(x_global)  # (B, 1)

        return hi, h_multi, weights, temperature, steady_level, op_outputs

class ReconHead(nn.Module):
    def __init__(self, C, hidden=128):
        super().__init__()
        self.gru = nn.GRU(input_size=C+1, hidden_size=hidden, batch_first=True)
        self.out = nn.Linear(hidden, C)
    def forward(self, x_in, h_in):
        B,T,C = x_in.shape
        h_in_clamped = torch.clamp(h_in, 0.0, 10.0)
        feat = torch.cat([x_in, h_in_clamped.unsqueeze(-1)], dim=-1)
        H,_ = self.gru(feat)
        y = self.out(H)
        return y

class MaintClassifier(nn.Module):
    def __init__(self, sensor_dim, hidden=64, n_classes=14):
        super().__init__()
        self.hi_mlp = nn.Sequential(
            nn.Linear(6, hidden), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden//2)
        )
        self.sensor_mlp = nn.Sequential(
            nn.Linear(sensor_dim * 2, hidden), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden//2)
        )
        self.final_classifier = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, n_classes)
        )

    def forward(self, h_b, h_a, x_b, x_a, mask):
        m = mask
        T = m.size(1)
        valid = m.sum(1, keepdim=True).clamp_min(1.0)

        db = h_a - h_b
        mean_d = (db*m).sum(1,keepdim=True)/valid
        var_d  = (((db-mean_d)*m)**2).sum(1,keepdim=True)/valid
        std_d  = (var_d + 1e-8).sqrt()
        d0     = (db[:, :1])
        dmax   = (db.masked_fill(m==0, -1e9)).max(dim=1, keepdim=True).values
        pos_ratio = ((db>0).float()*m).sum(1,keepdim=True)/valid

        t = torch.arange(T, device=h_b.device, dtype=h_b.dtype)
        t = (t - t.mean())/(t.std()+1e-6)
        def slope(x):
            num = (x*t).sum(1) - x.sum(1)*t.sum()/T
            den = (t**2).sum() - (t.sum()**2)/T + 1e-8
            return (num/den).unsqueeze(1)
        slope_diff = slope(h_a) - slope(h_b)

        hi_feat = torch.cat([mean_d, d0, dmax, std_d, slope_diff, pos_ratio], dim=1)
        hi_feat = torch.clamp(hi_feat, -10.0, 10.0)
        hi_features = self.hi_mlp(hi_feat)

        x_b_mean = (x_b * m.unsqueeze(-1)).sum(1) / valid
        x_a_mean = (x_a * m.unsqueeze(-1)).sum(1) / valid
        sensor_change = torch.cat([x_b_mean, x_a_mean], dim=1)
        sensor_features = self.sensor_mlp(sensor_change)

        combined = torch.cat([hi_features, sensor_features], dim=1)
        logits = self.final_classifier(combined)
        return logits

def sanitize_tensors(*tensors):
    result = []
    for t in tensors:
        if t is not None:
            t_clean = torch.nan_to_num(t, nan=0.0, posinf=1e6, neginf=-1e6)
            result.append(t_clean)
        else:
            result.append(t)
    return result[0] if len(result) == 1 else result

class DiffAwareReconstructor(nn.Module):
    """
    - Two encoders share parameters: encode x_before / x_after → h_b / h_a (monotonic decreasing HI)
    - Two reconstruction heads: one-step recursive reconstruction
    - Classification head: ΔHI + sensor change features
    """
    def __init__(self, in_ch, trend_ch=4, hidden=128, n_classes=14):
        super().__init__()
        self.encoder = TrendEncoder(in_ch, trend_ch)
        self.recon_b = ReconHead(in_ch, hidden)
        self.recon_a = ReconHead(in_ch, hidden)
        self.clf     = MaintClassifier(sensor_dim=in_ch, hidden=64, n_classes=n_classes)
        self.consistency_weight = nn.Parameter(torch.tensor(1.0))

    def forward(self, x_b, x_a, mask, training_noise=0.0, freeze_modulator=False, mixture_weight_b=0.9, mixture_weight_a=0.45):
        h_b, h_multi_b, weights_b, temp_b, steady_b, op_outputs_b = self.encoder(x_b, is_after=False,
                                                                                  training_noise=training_noise,
                                                                                  freeze_modulator=freeze_modulator,
                                                                                  mixture_weight=mixture_weight_b)
        h_a, h_multi_a, weights_a, temp_a, steady_a, op_outputs_a = self.encoder(x_a, is_after=True,
                                                                                  training_noise=training_noise,
                                                                                  freeze_modulator=freeze_modulator,
                                                                                  mixture_weight=mixture_weight_a)

        h_b, h_a = sanitize_tensors(h_b, h_a)
        weights_b, weights_a = sanitize_tensors(weights_b, weights_a)
        steady_b, steady_a = sanitize_tensors(steady_b, steady_a)

        xb_in, hb_in = x_b[:, :-2], h_b[:, :-2]
        xa_in, ha_in = x_a[:, :-2], h_a[:, :-2]
        yb_hat = self.recon_b(xb_in, hb_in)   # (B, L-2, C) ≈ x_b[:,1:L-1]
        ya_hat = self.recon_a(xa_in, ha_in)   # (B, L-2, C) ≈ x_a[:,1:L-1]

        yb_hat, ya_hat = sanitize_tensors(yb_hat, ya_hat)
        logits = self.clf(h_b, h_a, x_b, x_a, mask)
        logits = sanitize_tensors(logits)
        return yb_hat, ya_hat, h_b, h_a, logits, weights_b, weights_a, temp_b, temp_a, steady_b, steady_a, op_outputs_b, op_outputs_a

# ---------------------------------------------------------------------
# Losses / Regularizers
# ---------------------------------------------------------------------
def masked_mse(a, b, mask):
    diff = (a - b)**2
    mse  = (diff.mean(-1) * mask).sum() / (mask.sum() + 1e-6)
    return mse

def masked_mse_batchwise(a, b, mask):
    diff = (a - b)**2  # (B, T, C)
    mse_t = diff.mean(-1) * mask  # (B, T)
    denom = mask.sum(1).clamp_min(1.0)  # (B,)
    return mse_t.sum(1) / denom

def slope_loss(h, mask, kind="smooth"):
    if kind=="smooth":
        d2 = h[:,2:] - 2*h[:,1:-1] + h[:,:-2]
        m  = mask[:,2:]
        return (d2.abs() * m).sum() / (m.sum()+1e-6)
    elif kind=="mono_dec":
        dh = h[:,1:] - h[:,:-1]
        m  = mask[:,1:]
        return (F.relu(dh) * m).sum() / (m.sum()+1e-6)
    elif kind=="tv":
        # Total variation (first-order differences) - allows more curvature
        dh = h[:,1:] - h[:,:-1]
        m  = mask[:,1:]
        return (dh.abs() * m).sum() / (m.sum()+1e-6)
    else:
        return torch.tensor(0., device=h.device)

def plateau_tv_loss(h_a, mask, tau_ratio=0.3):
    """
    Time-weighted total variation loss for the plateau region (later 70% of sequence).
    Encourages h_a to become flat in the later stages.
    """
    B, T = h_a.shape
    tau = int(tau_ratio * T)

    if T <= tau + 1:
        return torch.tensor(0.0, device=h_a.device)

    # Focus on the plateau region (later 70%)
    h_plateau = h_a[:, tau:]  # (B, T-tau)
    mask_plateau = mask[:, tau:]  # (B, T-tau)

    if h_plateau.size(1) <= 1:
        return torch.tensor(0.0, device=h_a.device)

    # First-order differences in plateau region
    dh = h_plateau[:, 1:] - h_plateau[:, :-1]  # (B, T-tau-1)
    mask_diff = mask_plateau[:, 1:]  # (B, T-tau-1)

    # Time-increasing weights (later time steps get higher penalty)
    T_plateau = h_plateau.size(1) - 1
    if T_plateau > 0:
        time_weights = torch.linspace(1.0, 3.0, T_plateau, device=h_a.device).unsqueeze(0)  # (1, T-tau-1)
        weighted_tv = (dh.abs() * time_weights * mask_diff).sum() / (mask_diff.sum() + 1e-6)
    else:
        weighted_tv = torch.tensor(0.0, device=h_a.device)

    return weighted_tv

def plateau_curvature_loss(h_a, mask, tau_ratio=0.3):
    """
    Time-weighted second-order difference loss for the plateau region.
    Reduces curvature/oscillations in the later stages.
    """
    B, T = h_a.shape
    tau = int(tau_ratio * T)

    if T <= tau + 2:
        return torch.tensor(0.0, device=h_a.device)

    # Focus on the plateau region
    h_plateau = h_a[:, tau:]  # (B, T-tau)
    mask_plateau = mask[:, tau:]  # (B, T-tau)

    if h_plateau.size(1) <= 2:
        return torch.tensor(0.0, device=h_a.device)

    # Second-order differences in plateau region
    d2h = h_plateau[:, 2:] - 2 * h_plateau[:, 1:-1] + h_plateau[:, :-2]  # (B, T-tau-2)
    mask_d2 = mask_plateau[:, 2:]  # (B, T-tau-2)

    # Time-increasing weights
    T_plateau = h_plateau.size(1) - 2
    if T_plateau > 0:
        time_weights = torch.linspace(1.0, 2.0, T_plateau, device=h_a.device).unsqueeze(0)  # (1, T-tau-2)
        weighted_curv = (d2h.abs() * time_weights * mask_d2).sum() / (mask_d2.sum() + 1e-6)
    else:
        weighted_curv = torch.tensor(0.0, device=h_a.device)

    return weighted_curv

def steady_state_loss(h_a, steady_level, mask, tau_ratio=0.3):
    """
    Encourage h_a to approach the predicted steady-state level in the plateau region.
    """
    B, T = h_a.shape
    tau = int(tau_ratio * T)

    if T <= tau:
        return torch.tensor(0.0, device=h_a.device)

    # Focus on the plateau region
    h_plateau = h_a[:, tau:]  # (B, T-tau)
    mask_plateau = mask[:, tau:]  # (B, T-tau)

    # Expand steady_level to match plateau dimensions
    steady_expanded = steady_level.expand(-1, h_plateau.size(1))  # (B, T-tau)

    # L2 loss to steady state
    steady_loss = ((h_plateau - steady_expanded) ** 2 * mask_plateau).sum() / (mask_plateau.sum() + 1e-6)

    return steady_loss

def flat_operator_prior_loss(weights_a, mask, tau_ratio=0.3, flat_op_indices=[1, 3]):
    """
    Encourage flat/saturation operators to have higher weights in the plateau region.
    flat_op_indices: indices of MonotonicFlatOp and SaturationSigmoidOp
    """
    B, T, K = weights_a.shape
    tau = int(tau_ratio * T)

    if T <= tau:
        return torch.tensor(0.0, device=weights_a.device)

    # Focus on plateau region
    weights_plateau = weights_a[:, tau:, :]  # (B, T-tau, K)
    mask_plateau = mask[:, tau:]  # (B, T-tau)

    # Target distribution: higher probability for flat operators
    target_dist = torch.ones(K, device=weights_a.device) / K
    for idx in flat_op_indices:
        if idx < K:
            target_dist[idx] *= 2.0  # Boost flat operators
    target_dist = target_dist / target_dist.sum()  # Renormalize

    # KL divergence from current weights to target distribution
    eps = 1e-8
    weights_plateau_norm = weights_plateau + eps
    target_expanded = target_dist.unsqueeze(0).unsqueeze(0).expand_as(weights_plateau_norm)

    kl_div = (weights_plateau_norm * torch.log(weights_plateau_norm / target_expanded)).sum(-1)  # (B, T-tau)
    kl_loss = (kl_div * mask_plateau).sum() / (mask_plateau.sum() + 1e-6)

    return kl_loss

def enhanced_liquid_weight_regularization(weights_b, weights_a, mask, epoch, total_epochs,
                                          lambda_tv=0.0, lambda_ent=0.05,
                                          lambda_bal=0.3, lambda_div=0.05, lambda_var=0.08):
    """
    Enhanced liquid weight regularization with phased training approach.
    Adjusted: lambda_tv=0.0 (no temporal smoothing), lambda_ent=0.05 (reduced), lambda_var=0.08 (increased)
    """
    total_loss = torch.tensor(0.0, device=weights_b.device)
    K = weights_b.size(-1)

    # Phase determination
    progress = epoch / total_epochs
    if progress <= 0.3:  # Phase A: Exploration (0-30%)
        phase = "exploration"
    elif progress <= 0.7:  # Phase B: Formation (30-70%)
        phase = "formation"
    else:  # Phase C: Convergence (70-100%)
        phase = "convergence"

    def tv_loss(weights, mask):
        if weights.size(1) <= 1:
            return torch.tensor(0.0, device=weights.device)
        diff = weights[:, 1:, :] - weights[:, :-1, :]
        mask_diff = mask[:, 1:]
        tv = (diff.abs().sum(-1) * mask_diff).sum() / (mask_diff.sum() + 1e-6)
        return tv

    # No temporal smoothing on weights (lambda_tv=0.0)
    if lambda_tv > 0:
        tv_loss_b = tv_loss(weights_b, mask)
        tv_loss_a = tv_loss(weights_a, mask)
        total_loss += lambda_tv * (tv_loss_b + tv_loss_a)

    def target_entropy_loss(weights, mask, target_entropy):
        eps = 1e-8
        entropy = -(weights * torch.log(weights + eps)).sum(-1)  # (B, T)
        entropy_diff = (entropy - target_entropy) ** 2
        return (entropy_diff * mask).sum() / (mask.sum() + 1e-6)

    # Phased entropy targets
    if phase == "exploration":
        # Encourage uniform distribution (high entropy)
        target_entropy = np.log(K)  # Maximum entropy
        uniform_dist = torch.ones_like(weights_b) / K
        kl_to_uniform_b = F.kl_div(torch.log(weights_b + 1e-8), uniform_dist, reduction='none').sum(-1)
        kl_to_uniform_a = F.kl_div(torch.log(weights_a + 1e-8), uniform_dist, reduction='none').sum(-1)
        kl_loss = (kl_to_uniform_b * mask).sum() / (mask.sum() + 1e-6) + (kl_to_uniform_a * mask).sum() / (mask.sum() + 1e-6)
        total_loss += lambda_ent * kl_loss
    elif phase == "formation":
        target_entropy = 0.6 * np.log(K)  # Medium entropy
        ent_loss_b = target_entropy_loss(weights_b, mask, target_entropy)
        ent_loss_a = target_entropy_loss(weights_a, mask, target_entropy)
        total_loss += lambda_ent * (ent_loss_b + ent_loss_a)
    else:
        target_entropy = 0.35 * np.log(K)  # Lower entropy
        ent_loss_b = target_entropy_loss(weights_b, mask, target_entropy)
        ent_loss_a = target_entropy_loss(weights_a, mask, target_entropy)
        total_loss += lambda_ent * (ent_loss_b + ent_loss_a)

    def moe_balance_loss(weights, mask):
        """MoE-style load balancing loss"""
        B, T, K = weights.shape
        if T <= 1:
            return torch.tensor(0.0, device=weights.device)

        # Importance: average weight per operator
        valid_weights = weights * mask.unsqueeze(-1)  # (B, T, K)
        importance = valid_weights.sum([0, 1]) / (mask.sum() + 1e-6)  # (K,)

        # Load: fraction of tokens where each operator is top-1
        top1_indices = weights.argmax(dim=-1)  # (B, T)
        load = torch.zeros(K, device=weights.device)
        for k in range(K):
            load[k] = ((top1_indices == k).float() * mask).sum() / (mask.sum() + 1e-6)

        # Balance loss: encourage uniform importance and load
        importance_var = K * importance.var()
        load_var = K * load.var()
        return importance_var + load_var

    bal_loss_b = moe_balance_loss(weights_b, mask)
    bal_loss_a = moe_balance_loss(weights_a, mask)
    total_loss += lambda_bal * (bal_loss_b + bal_loss_a)

    def temporal_variance_loss(weights, mask, min_variance=0.01):
        """Encourage temporal variance in weights"""
        B, T, K = weights.shape
        if T <= 1:
            return torch.tensor(0.0, device=weights.device)

        # Compute temporal variance for each operator across time
        valid_weights = weights * mask.unsqueeze(-1)  # (B, T, K)
        # Normalize by valid timesteps per sample
        valid_counts = mask.sum(1, keepdim=True).clamp_min(1.0)  # (B, 1)
        mean_weights = valid_weights.sum(1, keepdim=True) / valid_counts.unsqueeze(-1)  # (B, 1, K)

        # Variance computation
        var_weights = ((valid_weights - mean_weights) ** 2 * mask.unsqueeze(-1)).sum(1) / valid_counts.unsqueeze(-1)  # (B, K)
        total_var = var_weights.sum(-1)  # (B,)

        # Penalty for insufficient variance
        var_penalty = F.relu(min_variance - total_var)
        return var_penalty.mean()

    # Encourage temporal variance (weights should change over time) - increased lambda_var to 0.08
    var_loss_b = temporal_variance_loss(weights_b, mask, min_variance=0.01 + 0.04 * progress)
    var_loss_a = temporal_variance_loss(weights_a, mask, min_variance=0.01 + 0.04 * progress)
    total_loss += lambda_var * (var_loss_b + var_loss_a)

    def orthogonality_loss(weights, mask):
        """Encourage operator diversity through weight decorrelation"""
        B, T, K = weights.shape
        if T <= 1:
            return torch.tensor(0.0, device=weights.device)

        valid_weights = weights * mask.unsqueeze(-1)
        weights_flat = valid_weights.view(-1, K)  # (B*T, K)

        # Center the weights
        weights_centered = weights_flat - weights_flat.mean(0, keepdim=True)

        # Compute correlation matrix
        cov = torch.mm(weights_centered.T, weights_centered) / (weights_flat.size(0) - 1 + 1e-6)

        # Penalize off-diagonal correlations
        mask_offdiag = torch.ones_like(cov) - torch.eye(K, device=cov.device)
        corr_penalty = (cov.abs() * mask_offdiag).sum()
        return corr_penalty

    div_loss_b = orthogonality_loss(weights_b, mask)
    div_loss_a = orthogonality_loss(weights_a, mask)
    total_loss += lambda_div * (div_loss_b + div_loss_a)

    return total_loss

def operator_orthogonality_loss(op_outputs_b, op_outputs_a, weights_b, weights_a, mask, lambda_ortho=0.1):
    """
    Encourage different operators to produce orthogonal/diverse outputs.
    """
    total_loss = torch.tensor(0.0, device=op_outputs_b.device)

    def compute_ortho_loss(op_outputs, weights, mask):
        B, T, K = op_outputs.shape
        if K <= 1:
            return torch.tensor(0.0, device=op_outputs.device)

        # Normalize outputs and apply mask
        op_outputs_masked = op_outputs * mask.unsqueeze(-1)  # (B, T, K)

        # Compute pairwise cosine similarities
        ortho_loss = torch.tensor(0.0, device=op_outputs.device)
        count = 0

        for i in range(K):
            for j in range(i + 1, K):
                # Extract operator outputs
                out_i = op_outputs_masked[:, :, i]  # (B, T)
                out_j = op_outputs_masked[:, :, j]  # (B, T)

                # Compute cosine similarity
                dot_product = (out_i * out_j * mask).sum()
                norm_i = torch.sqrt((out_i ** 2 * mask).sum() + 1e-8)
                norm_j = torch.sqrt((out_j ** 2 * mask).sum() + 1e-8)

                cos_sim = dot_product / (norm_i * norm_j + 1e-8)
                ortho_loss += cos_sim ** 2  # Penalize high correlation
                count += 1

        return ortho_loss / max(count, 1)

    ortho_loss_b = compute_ortho_loss(op_outputs_b, weights_b, mask)
    ortho_loss_a = compute_ortho_loss(op_outputs_a, weights_a, mask)
    total_loss += lambda_ortho * (ortho_loss_b + ortho_loss_a)

    return total_loss

def smoothness_enhancement_loss_separated(h_b, h_a, mask, order=2, weight_b=0.0, weight_a=0.001):
    """
    Separated smoothness loss: zero weight for h_b, very light TV for h_a.
    """
    def compute_smoothness(h, mask, order):
        if order == 2:
            if h.size(1) <= 2:
                return torch.tensor(0.0, device=h.device)
            d2 = h[:,2:] - 2*h[:,1:-1] + h[:,:-2]
            m = mask[:,2:]
            return (d2.abs() * m).sum() / (m.sum() + 1e-6)
        elif order == 3:
            if h.size(1) <= 3:
                return torch.tensor(0.0, device=h.device)
            d3 = h[:,3:] - 3*h[:,2:-1] + 3*h[:,1:-2] - h[:,:-3]
            m = mask[:,3:]
            return (d3.abs() * m).sum() / (m.sum() + 1e-6)
        else:
            return torch.tensor(0., device=h.device)

    # Use TV (first-order) for after, zero for before
    tv_b = slope_loss(h_b, mask, "tv") if weight_b > 0 else torch.tensor(0.0, device=h_b.device)
    tv_a = slope_loss(h_a, mask, "tv")

    return weight_b * tv_b + weight_a * tv_a

def compute_sample_weights(matching_costs, alpha=0.5):
    return 1.0 / (1.0 + alpha * matching_costs)

def maintenance_effect_loss(h_b, h_a, mask, lambda_diff=1.0, lambda_smooth_after=0.2, lambda_level_constraint=2.0):
    """
    Maintenance effect loss with very light after-maintenance smoothing.
    """
    total_loss = torch.tensor(0.0, device=h_b.device)

    # 1. Encourage difference between before and after HI (improved stability)
    hi_diff = torch.abs(h_a - h_b)  # (B, T)
    hi_diff_clamped = torch.clamp(hi_diff, min=1e-3)  # Prevent extreme gradients
    diff_loss = F.softplus(-10.0 * hi_diff_clamped)  # Smooth penalty for small differences
    diff_loss = (diff_loss * mask).sum() / (mask.sum() + 1e-6)
    total_loss += lambda_diff * diff_loss

    # 2. Very light smoothing for after-maintenance HI (reduced from 0.8 to 0.2)
    if h_a.size(1) > 1:
        h_a_diff = torch.abs(h_a[:, 1:] - h_a[:, :-1])  # (B, T-1)
        mask_diff = mask[:, 1:]
        smooth_after_loss = (h_a_diff * mask_diff).sum() / (mask_diff.sum() + 1e-6)
        total_loss += lambda_smooth_after * smooth_after_loss

    # 3. Encourage min(h_a) > max(h_b) for proper level separation
    h_b_masked = h_b.masked_fill(mask == 0, -1e9)  # Fill invalid positions with very low values
    h_a_masked = h_a.masked_fill(mask == 0, 1e9)   # Fill invalid positions with very high values

    h_b_max = h_b_masked.max(dim=1)[0]  # (B,) - max of each sequence
    h_a_min = h_a_masked.min(dim=1)[0]  # (B,) - min of each sequence

    # Encourage h_a_min > h_b_max with a margin
    margin = 0.2
    level_constraint = F.relu(h_b_max - h_a_min + margin)  # (B,)
    level_constraint_loss = level_constraint.mean()
    total_loss += lambda_level_constraint * level_constraint_loss

    # Pointwise gap constraint
    margin_pt = 0.1
    pointwise_gap = (h_a - h_b) + (-margin_pt)
    pointwise_loss = F.relu(-pointwise_gap)      # Penalty when h_a < h_b + margin_pt
    pointwise_loss = (pointwise_loss * mask).sum() / (mask.sum() + 1e-6)
    total_loss += 2.5 * pointwise_loss

    return total_loss

def dynamic_plateau_loss(h_b, h_a, steady_a, mask, k=10.0, m=0.05):
    """
    Dynamic plateau loss that activates when maintenance effect is significant.
    """
    # Calculate maintenance effect magnitude
    delta_mean = ((h_a - h_b) * mask).sum(1, keepdim=True) / (mask.sum(1, keepdim=True) + 1e-6)  # (B, 1)

    # Activation function: stronger plateau enforcement when delta is large
    gamma = torch.sigmoid(k * (delta_mean - m))  # (B, 1)

    # Plateau losses
    tv_loss = plateau_tv_loss(h_a, mask)
    curv_loss = plateau_curvature_loss(h_a, mask)
    steady_loss = steady_state_loss(h_a, steady_a, mask)

    # Weight by activation
    weighted_plateau_loss = gamma.mean() * (tv_loss + 0.5 * curv_loss + steady_loss)

    return weighted_plateau_loss

# ---------------------------------------------------------------------
# Evaluation / Prediction
# ---------------------------------------------------------------------
@torch.no_grad()
def eval_epoch(model, loader, device, K, compute_acc=True):
    model.eval()
    total = {"mse_b":0.0, "mse_a":0.0, "acc":0.0, "delta_mean":0.0}
    n_batch=0
    n_acc = 0; n_tot=0
    for batch in loader:
        xb, xa, labels, mask, lengths, sample_ids, matching_costs = adapt_batch(batch, device)
        m_tgt  = (torch.arange(xb.size(1)-2, device=device)[None,:] < (lengths-2)[:,None]).float()
        yb_hat, ya_hat, h_b, h_a, logits, _, _, _, _, _, _, _, _ = model(xb, xa, mask)
        yb_hat, ya_hat, h_b, h_a, logits = sanitize_tensors(yb_hat, ya_hat, h_b, h_a, logits)

        # Ensure logits dimension matches expected K
        if logits.shape[-1] != K:
            print(f"Warning: Logits dim {logits.shape[-1]} != expected K={K}, adjusting...")
            if logits.shape[-1] < K:
                # Pad with zeros
                padding = torch.zeros(logits.shape[0], K - logits.shape[-1], device=logits.device)
                logits = torch.cat([logits, padding], dim=1)
            else:
                # Truncate
                logits = logits[:, :K]

        yb_tgt = xb[:,1:-1,:]
        ya_tgt = xa[:,1:-1,:]
        mse_b = masked_mse(yb_hat, yb_tgt, m_tgt)
        mse_a = masked_mse(ya_hat, ya_tgt, m_tgt)
        if compute_acc and labels is not None:
            # Filter out ignore_index labels
            valid_labels = labels != -100
            if valid_labels.any():
                pred = logits[valid_labels].argmax(dim=1)
                labels_valid = labels[valid_labels]
                n_acc += (pred == labels_valid).sum().item()
                n_tot += labels_valid.numel()
        valid = mask.sum(1, keepdim=True).clamp_min(1.0)
        delta_mean = ((h_a - h_b)*mask).sum(1,keepdim=True)/valid
        delta_mean = torch.nan_to_num(delta_mean, nan=0.0)
        total["delta_mean"] += delta_mean.mean().item()
        total["mse_b"] += mse_b.item()
        total["mse_a"] += mse_a.item()
        n_batch += 1
    for k in ["mse_b","mse_a","delta_mean"]:
        total[k] /= max(n_batch,1)
    total["acc"] = (n_acc / max(n_tot,1)) if n_tot>0 else 0.0
    return total






def topk_by_delta(df_delta, id_to_name, k=3):
    if len(df_delta)==0:
        return
    unique_classes = sorted(df_delta["true"].unique())
    for cls in unique_classes[:5]:
        sub = df_delta[df_delta["true"]==cls].sort_values("delta_hi_mean", ascending=False).head(k)
        class_name = id_to_name.get(cls, f"Class_{cls}")
        print(f"\n[True={class_name}] Top {k} samples with highest ΔHI_mean:")
        print(sub[["uid","delta_hi_mean","pred"]].reset_index(drop=True))

# ---------------------------------------------------------------------
# Training (separate) and Testing (separate)
# ---------------------------------------------------------------------
def build_model_and_setup(train_loader, device, class_weights=None):
    """
    Build model and setup training components.

    Args:
        train_loader: Training data loader
        device: Device to use
        class_weights: Pre-computed class weights tensor (optional)
    """
    # Determine K from training loader
    K = scan_num_classes_from_loader(train_loader)
    id_to_name = build_default_id_to_name(K)

    first_batch = next(iter(train_loader))
    C = first_batch["x_before"].shape[-1]

    print(f"\n[Setup] Sensor dim C={C}, Num classes K={K}")

    # Use provided class weights or create uniform weights
    if class_weights is not None:
        if isinstance(class_weights, np.ndarray):
            class_weights = torch.from_numpy(class_weights).float().to(device)
        elif isinstance(class_weights, torch.Tensor):
            class_weights = class_weights.float().to(device)

        # Ensure class_weights has the right size
        if len(class_weights) < K:
            # Pad with ones if needed
            padded_weights = torch.ones(K, dtype=torch.float32, device=device)
            padded_weights[:len(class_weights)] = class_weights
            class_weights = padded_weights
        elif len(class_weights) > K:
            # Truncate if needed
            class_weights = class_weights[:K]

        print(f"[Class Weights] Using provided weights: {class_weights.detach().cpu().numpy()}")
    else:
        class_weights = torch.ones(K, dtype=torch.float32, device=device)
        print(f"[Class Weights] Using uniform weights: {class_weights.detach().cpu().numpy()}")

    model = DiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=K).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    ce  = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100, reduction='none')

    return model, opt, ce, K, id_to_name, C


def train_only(train_loader, val_loader, epochs=300, lr=1e-3, device=None, patience=20,
               use_matching_cost=True, save_path="best_model.pth", class_weights=None,
               es_alpha=1.0):
    """
    Train and save the best model with phased liquid weight training.

    Args:
        class_weights: Pre-computed class weights (numpy array or torch tensor)
        es_alpha: Weight for accuracy in early stopping metric
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Build model and training setup
    model, opt, ce, K, id_to_name, C = build_model_and_setup(train_loader, device, class_weights)
    # Override LR if provided
    for g in opt.param_groups:
        g['lr'] = lr

    best_v = float("inf")
    best_state = None
    best_epoch = 0
    no_improve = 0
    nan_batches = 0

    print(f"\nStarting phased liquid weight training for {epochs} epochs...")
    print(f"Device: {device}")
    print(f"Early stopping alpha: {es_alpha}")
    print(f"Phase A (0-30%): Exploration with uniform priors")
    print(f"Phase B (30-70%): Formation with target entropy")
    print(f"Phase C (70-100%): Convergence with refined entropy")

    for ep in range(1, epochs+1):
        progress = ep / epochs

        # Determine training phase
        if progress <= 0.3:
            phase = "exploration"
            freeze_modulator = True
            mixture_weight_b = 0.9
            mixture_weight_a = 0.45
            training_noise = 0.03
        elif progress <= 0.7:
            phase = "formation"
            freeze_modulator = False
            mixture_weight_b = 0.9
            mixture_weight_a = 0.45
            training_noise = 0.03
        else:
            phase = "convergence"
            freeze_modulator = False
            mixture_weight_b = 0.9
            mixture_weight_a = 0.45
            training_noise = 0.03

        print(f"\n[Epoch {ep}/{epochs}] Phase: {phase.upper()}, Training...")
        model.train()
        logs = {"mse_b":0.0, "mse_a":0.0, "smooth":0.0, "mono":0.0,
                "cls":0.0, "liquid_weight":0.0, "maintenance_effect":0.0,
                "plateau":0.0, "all":0.0}
        n_bt = 0
        batch_nan_count = 0

        # Adaptive regularization weights
        lambda_tv = 0.0  # No temporal smoothing on weights
        lambda_ent = 0.05  # Reduced from 0.10
        lambda_bal = 0.3
        lambda_div = 0.05
        lambda_var = 0.08  # Increased from 0.06

        # Gradient norm tracking
        grad_norms = []

        # Monitoring metrics
        total_entropy_b = 0.0
        total_entropy_a = 0.0
        total_temp_b = 0.0
        total_temp_a = 0.0
        utilization_counts = torch.zeros(8, device=device)  # 8 operators
        switch_rates = []

        for batch_idx, batch in enumerate(train_loader):
            xb, xa, labels, mask, lengths, sample_ids, matching_costs = adapt_batch(batch, device)
            m_tgt  = (torch.arange(xb.size(1)-2, device=device)[None,:] < (lengths-2)[:,None]).float()

            yb_hat, ya_hat, h_b, h_a, logits, weights_b, weights_a, temp_b, temp_a, steady_b, steady_a, op_outputs_b, op_outputs_a = model(
                xb, xa, mask, training_noise=training_noise, freeze_modulator=freeze_modulator,
                mixture_weight_b=mixture_weight_b, mixture_weight_a=mixture_weight_a)

            yb_hat, ya_hat, h_b, h_a, logits = sanitize_tensors(yb_hat, ya_hat, h_b, h_a, logits)
            weights_b, weights_a = sanitize_tensors(weights_b, weights_a)
            steady_b, steady_a = sanitize_tensors(steady_b, steady_a)

            # Reconstruction targets
            yb_tgt = xb[:,1:-1,:]
            ya_tgt = xa[:,1:-1,:]

            # Per-sample reconstruction losses (for optional cost-weighting)
            rec_b_per = masked_mse_batchwise(yb_hat, yb_tgt, m_tgt)
            rec_a_per = masked_mse_batchwise(ya_hat, ya_tgt, m_tgt)
            rec_per = rec_b_per + rec_a_per

            # Classification loss (per-sample)
            if labels is None:
                cls_per = torch.zeros(xb.size(0), device=device)
            else:
                cls_per = ce(logits, labels)

            # Optional matching-cost weighting
            if use_matching_cost and matching_costs is not None:
                w_i = compute_sample_weights(matching_costs)
            else:
                w_i = torch.ones(xb.size(0), device=device)

            loss_rec = (rec_per * w_i).mean()
            loss_cls = (cls_per * w_i).mean()

            # HI regularizations - only very light TV for after, zero for before
            loss_smooth = smoothness_enhancement_loss_separated(h_b, h_a, mask, order=2, weight_b=0.0, weight_a=0.001)
            loss_mono = torch.tensor(0.0, device=h_b.device)  # Monotonicity enforced by cummin

            loss_liquid_weight = enhanced_liquid_weight_regularization(
                weights_b, weights_a, mask, ep, epochs,
                lambda_tv=lambda_tv, lambda_ent=lambda_ent,
                lambda_bal=lambda_bal, lambda_div=lambda_div, lambda_var=lambda_var
            )

            # Maintenance effect loss with very light after-maintenance smoothing
            loss_maintenance_effect = maintenance_effect_loss(
                h_b, h_a, mask,
                lambda_diff=1.0,
                lambda_smooth_after=0.2,  # Reduced from 0.8
                lambda_level_constraint=2.0
            )

            # Reduced plateau losses
            loss_plateau_tv = plateau_tv_loss(h_a, mask)
            loss_plateau_curv = plateau_curvature_loss(h_a, mask)
            loss_steady_state = steady_state_loss(h_a, steady_a, mask)
            loss_dynamic_plateau = dynamic_plateau_loss(h_b, h_a, steady_a, mask)

            # Combined plateau loss with very reduced weight
            loss_plateau = (loss_plateau_tv + 0.5 * loss_plateau_curv +
                           loss_steady_state + loss_dynamic_plateau)

            # Total loss - very reduced plateau weight
            w_rec=1.0; w_smooth=1.0; w_mono=0.0; w_cls=1.0; w_liquid=0.30; w_maintenance=2.5; w_plateau=0.05
            loss = (w_rec*loss_rec + w_smooth*loss_smooth + w_mono*loss_mono +
                    w_cls*loss_cls + w_liquid*loss_liquid_weight +
                    w_maintenance*loss_maintenance_effect + w_plateau*loss_plateau)

            loss_rec, loss_cls, loss_smooth, loss_mono, loss_liquid_weight, loss_maintenance_effect, loss_plateau, loss = sanitize_tensors(
                loss_rec, loss_cls, loss_smooth, loss_mono, loss_liquid_weight, loss_maintenance_effect, loss_plateau, loss
            )

            if torch.isnan(loss) or torch.isinf(loss):
                batch_nan_count += 1
                if batch_nan_count == 1:
                    print(f"    [Warning] Epoch {ep}: Found NaN/Inf loss, skipping abnormal batch")
                continue

            opt.zero_grad()
            loss.backward()

            # Adaptive gradient clipping
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            grad_norms.append(grad_norm.item())

            opt.step()

            logs["mse_b"] += rec_b_per.mean().item()
            logs["mse_a"] += rec_a_per.mean().item()
            logs["smooth"]+= loss_smooth.item()
            logs["mono"]  += loss_mono.item()
            logs["cls"]   += loss_cls.item()
            logs["liquid_weight"]+= loss_liquid_weight.item()
            logs["maintenance_effect"]+= loss_maintenance_effect.item()
            logs["plateau"]+= loss_plateau.item()
            logs["all"]   += loss.item()
            n_bt += 1

        if batch_nan_count > 0:
            nan_batches += batch_nan_count

        for k in logs: logs[k] /= max(n_bt,1)

        # Compute entropy and temperature statistics
        with torch.no_grad():
            def _entropy(w):
                return (-(w * (w+1e-8).log()).sum(-1)).mean().item()
            mean_ent_b = _entropy(weights_b)
            mean_ent_a = _entropy(weights_a)
            mean_tau_b = temp_b.mean().item()
            mean_tau_a = temp_a.mean().item()

        print("    Validating...")
        vl = eval_epoch(model, val_loader, device, K, compute_acc=True)
        # Improved early-stopping metric with configurable weight
        vl_total = (vl['mse_b'] + vl['mse_a']) + es_alpha * (1.0 - vl['acc'])

        # Log gradient statistics
        avg_grad_norm = np.mean(grad_norms) if grad_norms else 0.0
        max_grad_norm = np.max(grad_norms) if grad_norms else 0.0

        print(f"[Epoch {ep:03d}] "
              f"Train: L={logs['all']:.4f} rec_b={logs['mse_b']:.4f} rec_a={logs['mse_a']:.4f} "
              f"smooth={logs['smooth']:.4f} mono={logs['mono']:.4f} liquid_w={logs['liquid_weight']:.4f} "
              f"maint_eff={logs['maintenance_effect']:.4f} plateau={logs['plateau']:.4f} cls={logs['cls']:.4f} | "
              f"Val: mse_b={vl['mse_b']:.4f} mse_a={vl['mse_a']:.4f} acc={vl['acc']:.3f} ΔHI_impr={vl['delta_mean']:.3f} "
              f"(ES metric={vl_total:.4f}) | Grad: avg={avg_grad_norm:.3f} max={max_grad_norm:.3f}")
        print(f"Entropy pre/post: {mean_ent_b:.3f}/{mean_ent_a:.3f} | Temp pre/post: {mean_tau_b:.2f}/{mean_tau_a:.2f}")

        if vl_total < best_v:
            best_v = vl_total
            best_epoch = ep
            best_state = {k: v.clone() if hasattr(v, 'clone') else v for k, v in model.state_dict().items()}
            no_improve = 0
            print(f"    ✓ Saved new best model (ES metric: {best_v:.4f})")
        else:
            no_improve += 1
            print(f"    No improvement for {no_improve}/{patience} epochs")
            if no_improve >= patience:
                print(f"\n[Early Stopping] Patience reached at epoch {ep}.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"\n[Best Checkpoint] ES metric: {best_v:.4f} (Epoch {best_epoch})")
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        torch.save(model.state_dict(), save_path)
        print(f"[Model Saved] {save_path}")

        # Save metadata alongside model
        meta_path = save_path.replace('.pth', '_meta.json')
        meta_data = {
            "K": K,
            "id_to_name": id_to_name,
            "sensor_dim": C,
            "best_es": best_v,
            "best_epoch": best_epoch,
            "class_weights": class_weights.cpu().numpy().tolist() if class_weights is not None else None
        }
        import json
        with open(meta_path, 'w') as f:
            json.dump(meta_data, f, indent=2)
        print(f"[Metadata Saved] {meta_path}")

    if nan_batches > 0:
        print(f"[Warning] Total skipped {nan_batches} batches containing NaN/Inf during training.")

    meta = {"K": K, "id_to_name": id_to_name, "sensor_dim": C, "best_es": best_v, "best_epoch": best_epoch}
    return save_path, meta


ckpt_path, meta = train_only(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=1000,
    lr=1e-3,
    device=None,     # auto
    patience=20,
    use_matching_cost=True,
    save_path=str(resolve_data_file("best_model_NGFAID_250_100_6.pth")),
    class_weights=torch.as_tensor(class_weights, dtype=torch.float32)   # Pass your computed class weights here
)


## Results: Damage-Reconstruction Evaluation


In [ ]:
# -*- coding: utf-8 -*-
import torch
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import os
import pickle
from typing import Dict, Tuple, Optional
import matplotlib.pyplot as plt
import pandas as pd
import random

# ============================================
# Read "real class labels" and replace default Class_i labels
# ============================================

SAVE_DIR = str(DATA_DIR)

def _invert_name_mapping(name_to_id: Dict) -> Dict[int, str]:
    """Invert {name -> id} to {id -> name}, ensuring all names are str with capitalized first letter."""
    id_to_name = {}
    for name, idx in name_to_id.items():
        try:
            idx_int = int(idx)
        except Exception:
            continue
        # Capitalize first letter of class name
        name_str = str(name)
        if name_str:
            name_str = name_str[0].upper() + name_str[1:]
        id_to_name[idx_int] = name_str
    return id_to_name

def read_real_class_id_to_name(
    save_dir: str = SAVE_DIR,
    preferred_filename: Optional[str] = None
) -> Tuple[Optional[Dict[int, str]], Optional[int], Optional[str]]:
    """
    Read real class names from saved label_mappings_*.pkl:
      - Prioritize using 'label_to_id' (usually string class -> id)
      - Otherwise fallback to 'class_to_id' / 'target_class_to_id' / 'hclass_to_id'
    Returns:
      (id_to_name, K, used_file_path)
      Returns (None, None, None) if reading fails
    """
    if preferred_filename:
        candidates = [os.path.join(save_dir, preferred_filename)]
    else:
        # Automatically search for pkl files starting with label_mappings_sample_split_window_
        candidates = []
        if os.path.isdir(save_dir):
            for fn in os.listdir(save_dir):
                if fn.startswith("label_mappings_sample_split_window_") and fn.endswith(".pkl"):
                    candidates.append(os.path.join(save_dir, fn))
        # Select the newest
        candidates.sort(key=lambda p: os.path.getmtime(p), reverse=True)

    for path in candidates:
        try:
            with open(path, "rb") as f:
                lm = pickle.load(f)
        except Exception:
            continue

        # Prioritize using string labels
        if isinstance(lm, dict) and "label_to_id" in lm and lm["label_to_id"]:
            id_to_name = _invert_name_mapping(lm["label_to_id"])
            if id_to_name:
                K = max(id_to_name.keys()) + 1
                return id_to_name, K, path

        # Otherwise try numeric class labels (convert original numeric values to string for display)
        for key in ["class_to_id", "target_class_to_id", "hclass_to_id"]:
            if isinstance(lm, dict) and key in lm and lm[key]:
                id_to_name = _invert_name_mapping(lm[key])
                if id_to_name:
                    # Note: these mappings are {original class value -> encoded_id}, already inverted to {id -> original class value(str)}
                    K = max(id_to_name.keys()) + 1
                    return id_to_name, K, path

    return None, None, None

def _infer_C(train_loader=None, test_loader=None):
    """Infer sensor dimension C from loader."""
    probe_loader = train_loader if train_loader is not None else test_loader
    assert probe_loader is not None, "Need available train_loader or test_loader to infer C."
    first_batch = next(iter(probe_loader))
    C = first_batch["x_before"].shape[-1]
    return int(C)

def build_default_id_to_name(K: int):
    return {i: f"Class_{i}" for i in range(K)}

# ========== Use real class names to replace default Class_i ==========
# 1) Still use loader to infer C (sensor dimension)
# 2) K prioritizes using real class count from reading; fallback to inferring from loader if reading fails
C = _infer_C(train_loader=train_loader, test_loader=test_loader)

# First try to read real class labels from label_mappings
real_id_to_name, real_K, used_path = read_real_class_id_to_name(SAVE_DIR)

if real_id_to_name is not None and real_K is not None:
    id_to_name = real_id_to_name
    K = int(real_K)
    print(f"[Info] Using real class names (total {K} classes), source: {used_path}")
else:
    # Fallback: infer K from loader, then assign default names
    try:
        K = scan_num_classes_from_loader(train_loader) if train_loader is not None else scan_num_classes_from_loader(test_loader)
    except Exception:
        raise RuntimeError("Cannot infer K from label_mappings or loader. Please check data/mapping files.")
    id_to_name = build_default_id_to_name(K)
    print(f"[Warn] label_mappings*.pkl not found, fallback to default class names. K={K}")

print(f"[Info] Inferred sensor dim C={C}, num classes K={K}")

class_weights_obj = None

# ============================================
# GAFID Aviation Maintenance Dataset - Sensor Data Description
# ============================================
SENSOR_NAMES = {
    0: "volt1 (Main bus voltage)",
    1: "volt2 (Essential bus voltage)",
    2: "amp1 (Main battery ammeter)",
    3: "amp2 (Standby battery ammeter)",
    4: "FQtyL (Fuel quantity left)",
    5: "FQtyR (Fuel quantity right)",
    6: "E1 FFlow (Engine fuel flow)",
    7: "E1 OilT (Engine oil temp)",
    8: "E1 OilP (Engine oil pressure)",
    9: "E1 RPM (Engine RPM)",
    10: "E1 CHT1 (Cylinder 1 head temp)",
    11: "E1 CHT2 (Cylinder 2 head temp)",
    12: "E1 CHT3 (Cylinder 3 head temp)",
    13: "E1 CHT4 (Cylinder 4 head temp)",
    14: "E1 EGT1 (Cylinder 1 exhaust temp)",
    15: "E1 EGT2 (Cylinder 2 exhaust temp)",
    16: "E1 EGT3 (Cylinder 3 exhaust temp)",
    17: "E1 EGT4 (Cylinder 4 exhaust temp)",
    18: "OAT (Outside air temp)",
    19: "IAS (Indicated air speed)",
    20: "VSpd (Vertical speed)",
    21: "NormAc (Normal acceleration)",
    22: "AltMSL (Altitude MSL)",
    23: "Throttle (Engine power)",
    24: "Fuel flow control",
    25: "Flight stability",
    26: "Aerodynamic drag",
    27: "Total weight"
}

@torch.no_grad()
def collect_test_predictions(
    model, loader, device, max_curve_keep=24, K=14, id_to_name=None,
    use_damage_for_curves=True  # Optional new switch; use damage curves for plotting by default
):
    model.eval()
    y_true, y_pred = [], []
    all_delta_mean, all_sample_ids = [], []
    keep_curves = []

    # === Read CustomKAN gain / bias to reconstruct damage ===
    customkan = model.encoder.customkan
    gain_ck = torch.clamp(F.softplus(customkan.raw_gain), 0.1, 2.0)
    bias_ck = torch.clamp(customkan.bias, -2.0, 2.0)

    for batch in loader:
        xb, xa, labels, mask, lengths, sample_ids, matching_costs = adapt_batch(batch, device)

        # Raw forward pass
        yb_hat, ya_hat, h_b, h_a, logits, weights_b, weights_a, temp_b, temp_a, steady_b, steady_a, op_outputs_b, op_outputs_a = model(xb, xa, mask)

        # Clean NaN/Inf values
        yb_hat, ya_hat, h_b, h_a, logits = sanitize_tensors(yb_hat, ya_hat, h_b, h_a, logits)
        op_outputs_b = torch.nan_to_num(op_outputs_b, nan=0.0, posinf=1e6, neginf=-1e6)
        op_outputs_a = torch.nan_to_num(op_outputs_a, nan=0.0, posinf=1e6, neginf=-1e6)

        # Ensure logits dimension matches expected K
        if logits.shape[-1] != K:
            if logits.shape[-1] < K:
                padding = torch.zeros(logits.shape[0], K - logits.shape[-1], device=logits.device)
                logits = torch.cat([logits, padding], dim=1)
            else:
                logits = logits[:, :K]

        prob = F.softmax(logits, dim=1)     # (B,K)
        pred = prob.argmax(1)               # (B,)

        valid = mask.sum(1, keepdim=True).clamp_min(1.0)
        delta_mean = ((h_a - h_b) * mask).sum(1, keepdim=True) / valid  # (B,1)
        delta_mean = torch.nan_to_num(delta_mean, nan=0.0)

        if labels is not None:
            # Filter out ignore_index labels
            valid_labels = labels != -100
            if valid_labels.any():
                y_true.append(labels[valid_labels].detach().cpu().numpy())
                y_pred.append(pred[valid_labels].detach().cpu().numpy())
        all_delta_mean.append(delta_mean.squeeze(1).cpu().numpy())
        all_sample_ids.extend(sample_ids)

        # === Key step: reconstruct CustomKAN damage using op_outputs and weights ===
        # shape: op_outputs_* = (B, Tm, K), weights_* = (B, Tm, K)
        raw_dmg_b = (op_outputs_b * weights_b).sum(dim=-1)              # (B, Tm_b)
        raw_dmg_a = (op_outputs_a * weights_a).sum(dim=-1)              # (B, Tm_a)
        damage_b  = torch.clamp(gain_ck * raw_dmg_b + bias_ck, 0.0, 100.0)
        damage_a  = torch.clamp(gain_ck * raw_dmg_a + bias_ck, 0.0, 100.0)

        for i in range(xb.size(0)):
            if len(keep_curves) >= max_curve_keep: break
            L_i = int(lengths[i].item())
            L_recon = min(L_i - 2, yb_hat.size(1))

            # Align damage lengths by taking the minimum length across all three arrays so pre/post plots remain safe
            Tmb = damage_b.size(1)
            Tma = damage_a.size(1)
            L_eff = max(1, min(L_i, Tmb, Tma))

            # Choose which curve to plot: damage or HI
            if use_damage_for_curves:
                curve_pre  = damage_b[i, :L_eff].detach().cpu().numpy()
                curve_post = damage_a[i, :L_eff].detach().cpu().numpy()
            else:
                curve_pre  = h_b[i, :L_eff].detach().cpu().numpy()
                curve_post = h_a[i, :L_eff].detach().cpu().numpy()

            keep_curves.append({
                "sample_id": sample_ids[i],
                "true": int(labels[i].cpu().item()) if labels is not None and labels[i] != -100 else -1,
                "pred": int(pred[i].cpu().item()),
                "prob": prob[i].detach().cpu().numpy(),

                # === Fill h_before/h_after with damage so the existing plot* helpers can render damage directly ===
                "h_before": curve_pre,
                "h_after":  curve_post,

                # Keep the original HI as well in case it is needed for analysis
                "hi_before": h_b[i, :L_eff].detach().cpu().numpy(),
                "hi_after":  h_a[i, :L_eff].detach().cpu().numpy(),

                "weights_before": weights_b[i, :L_eff].detach().cpu().numpy() if weights_b is not None else None,
                "weights_after":  weights_a[i, :L_eff].detach().cpu().numpy() if weights_a is not None else None,
                "temp_before": temp_b[i, :L_eff].detach().cpu().numpy() if temp_b is not None else None,
                "temp_after":  temp_a[i, :L_eff].detach().cpu().numpy() if temp_a is not None else None,
                "steady_before": float(steady_b[i].detach().cpu().item()) if steady_b is not None else None,
                "steady_after":  float(steady_a[i].detach().cpu().item()) if steady_a is not None else None,
                "op_outputs_before": op_outputs_b[i, :L_eff].detach().cpu().numpy() if op_outputs_b is not None else [],
                "op_outputs_after":  op_outputs_a[i, :L_eff].detach().cpu().numpy() if op_outputs_a is not None else [],

                "yb_hat":   yb_hat[i, :L_recon].detach().cpu().numpy(),
                "ya_hat":   ya_hat[i, :L_recon].detach().cpu().numpy(),
                "x_before": xb[i, :L_eff].detach().cpu().numpy(),
                "x_after":  xa[i, :L_eff].detach().cpu().numpy(),
            })
    y_true = np.concatenate(y_true, axis=0) if len(y_true)>0 else np.array([])
    y_pred = np.concatenate(y_pred, axis=0) if len(y_pred)>0 else np.array([])
    all_delta_mean = np.concatenate(all_delta_mean, axis=0) if len(all_delta_mean)>0 else np.array([])
    return y_true, y_pred, all_delta_mean, all_sample_ids, keep_curves
# ---------------------------------------------------------------------
# Plotting
# ---------------------------------------------------------------------
def remove_constant_segments(hi_values, threshold=1e-6):
    if len(hi_values) <= 1:
        return hi_values, np.arange(len(hi_values))
    diffs = np.abs(np.diff(hi_values))
    valid_mask = np.ones(len(hi_values), dtype=bool)
    valid_mask[1:] = diffs > threshold
    if np.sum(valid_mask) < 2:
        valid_mask[0] = True
        valid_mask[-1] = True
    valid_indices = np.where(valid_mask)[0]
    return hi_values[valid_indices], valid_indices

def plot_hi_examples_aligned(curves, id_to_name, n_show=6, seed=0):
    """
    Plot 2 samples per class, skipping first and last 10 points of HI curves.
    Title format: "Flight sample XX | Class_name | p=... | ΔHI_mean=... | ΔHI_max=..."
    """
    if len(curves) == 0:
        print("(No visualization samples)")
        return

    # Group curves by true class
    class_to_curves = {}
    for ex in curves:
        true_label = ex["true"]
        if true_label == -1:
            continue  # skip samples without valid labels
        if true_label not in class_to_curves:
            class_to_curves[true_label] = []
        class_to_curves[true_label].append(ex)

    # Select 2 samples per class
    selected_curves = []
    for cls_id in sorted(class_to_curves.keys()):
        cls_curves = class_to_curves[cls_id]
        random.Random(seed).shuffle(cls_curves)
        selected_curves.extend(cls_curves[:2])

    n_show = len(selected_curves)
    if n_show == 0:
        print("(No valid samples to visualize)")
        return

    cols = 2  # 2 samples per row
    rows = int(np.ceil(n_show / cols))
    plt.figure(figsize=(cols * 6, rows * 4))

    for k, ex in enumerate(selected_curves):
        hb = ex["h_before"]
        ha = ex["h_after"]
        L = min(len(hb), len(ha))
        hb, ha = hb[:L], ha[:L]

        # Skip first and last 10 points
        skip = 10
        if L > 2 * skip:
            hb = hb[skip:-skip]
            ha = ha[skip:-skip]
        else:
            # If too short, just use all points
            pass

        # Reverse damage trend (plot damage in reverse order)
        hb = hb[::-1]
        ha = ha[::-1]

        hb_clean, hb_indices = remove_constant_segments(hb)
        ha_clean, ha_indices = remove_constant_segments(ha)
        t_b = hb_indices
        t_a = ha_indices  # The post-phase x-axis starts at 0 and is not accumulated

        sample_id = ex["sample_id"]
        pred = ex["pred"]
        true = ex["true"]
        prob = ex["prob"]
        true_name = id_to_name.get(true, f"Class_{true}")
        pred_name = id_to_name.get(pred, f"Class_{pred}")

        plt.subplot(rows, cols, k + 1)
        if len(hb_clean) > 1:
            plt.plot(t_b, hb_clean, label="Learned HI (Pre)", linewidth=1.8, marker='o', markersize=3)
        if len(ha_clean) > 1:
            plt.plot(t_a, ha_clean, label="Learned HI (Post)", linewidth=1.8, linestyle='--', marker='s', markersize=3)
        d_mean = float(np.mean(ha - hb))
        d_max = float(np.max(ha - hb))

        # Add steady state info if available
        steady_info = ""
        if ex.get("steady_after") is not None:
            steady_info = f" | SS={ex['steady_after']:.3f}"

        plt.title(f"Flight sample {sample_id} | {true_name}\n"
                  f"p={prob[pred]:.2f} | ΔHI_mean={d_mean:.3f}, ΔHI_max={d_max:.3f}{steady_info}", fontsize=9)
        plt.xlabel("Cycle")
        plt.ylabel("Health Index (higher=better)")
        plt.grid(ls='--', alpha=.35)
        if k % cols == 0:
            plt.legend()
    plt.tight_layout()
    plt.show()

def plot_enhanced_liquid_weights(curves, id_to_name, n_show=3, seed=0):
    """
    Plot 2 samples per class for liquid weights visualization, skipping first and last 10 points.
    Title format: "Flight sample XX | True=Class_name | Pred=Class_name"
    """
    if len(curves) == 0:
        print("(No visualization samples)")
        return
    curves_with_weights = [c for c in curves if c.get("weights_before") is not None]
    if len(curves_with_weights) == 0:
        print("(No weight information available)")
        return

    # Group curves by true class
    class_to_curves = {}
    for ex in curves_with_weights:
        true_label = ex["true"]
        if true_label == -1:
            continue
        if true_label not in class_to_curves:
            class_to_curves[true_label] = []
        class_to_curves[true_label].append(ex)

    # Select 2 samples per class
    selected_curves = []
    for cls_id in sorted(class_to_curves.keys()):
        cls_curves = class_to_curves[cls_id]
        random.Random(seed).shuffle(cls_curves)
        selected_curves.extend(cls_curves[:2])

    n_show = len(selected_curves)
    if n_show == 0:
        print("(No valid samples to visualize)")
        return

    op_names = ["MonotonicLinear", "MonotonicFlat", "ConcaveLog", "SaturationSigmoid",
                "HingeReLU", "Polynomial", "DampedSin", "PiecewiseLinear"]

    for i in range(n_show):
        ex = selected_curves[i]
        weights_b = ex["weights_before"]
        weights_a = ex["weights_after"]
        temp_b = ex.get("temp_before")
        temp_a = ex.get("temp_after")
        sample_id = ex["sample_id"]
        true_label = ex["true"]
        pred_label = ex["pred"]
        true_name = id_to_name.get(true_label, f"Class_{true_label}")
        pred_name = id_to_name.get(pred_label, f"Class_{pred_label}")

        # Skip first and last 10 points
        skip = 10
        if weights_b.shape[0] > 2 * skip:
            weights_b = weights_b[skip:-skip]
        if weights_a.shape[0] > 2 * skip:
            weights_a = weights_a[skip:-skip]
        if temp_b is not None and len(temp_b) > 2 * skip:
            temp_b = temp_b[skip:-skip]
        if temp_a is not None and len(temp_a) > 2 * skip:
            temp_a = temp_a[skip:-skip]

        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle(f"Dynamic Parameter Modulation - Flight sample {sample_id} | True={true_name} | Pred={pred_name}", fontsize=14)
        ax = axes[0, 0]
        T_b, K = weights_b.shape
        for k in range(K):
            ax.plot(range(T_b), weights_b[:, k], label=op_names[k] if k < len(op_names) else f"Op_{k}",
                   marker='o', markersize=2, linewidth=1.5)
        ax.set_title("Pre-maintenance Weights"); ax.set_xlabel("Time Step"); ax.set_ylabel("Weight")
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left'); ax.grid(True, alpha=0.3)
        ax = axes[0, 1]
        T_a, _ = weights_a.shape
        for k in range(K):
            ax.plot(range(T_a), weights_a[:, k], label=op_names[k] if k < len(op_names) else f"Op_{k}",
                   marker='s', markersize=2, linewidth=1.5, linestyle='--')
        ax.set_title("Post-maintenance Weights"); ax.set_xlabel("Time Step"); ax.set_ylabel("Weight")
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left'); ax.grid(True, alpha=0.3)
        ax = axes[0, 2]
        entropy_b = -np.sum(weights_b * np.log(weights_b + 1e-8), axis=1)
        entropy_a = -np.sum(weights_a * np.log(weights_a + 1e-8), axis=1)
        ax.plot(range(T_b), entropy_b, label="Pre", marker='o', linewidth=2)
        ax.plot(range(T_a), entropy_a, label="Post", marker='s', linewidth=2, linestyle='--')
        ax.axhline(np.log(K), color='r', linestyle=':', label=f'Max Entropy ({np.log(K):.2f})')
        ax.set_title("Weight Entropy Over Time"); ax.set_xlabel("Time Step"); ax.set_ylabel("Entropy")
        ax.legend(); ax.grid(True, alpha=0.3)
        ax = axes[1, 0]
        if temp_b is not None and temp_a is not None:
            ax.plot(range(T_b), temp_b, label="Pre", marker='o', linewidth=2)
            ax.plot(range(T_a), temp_a, label="Post", marker='s', linewidth=2, linestyle='--')
            ax.set_title("Temperature Evolution"); ax.set_xlabel("Time Step"); ax.set_ylabel("Temperature"); ax.legend()
        else:
            ax.text(0.5, 0.5, "Temperature data\nnot available", ha='center', va='center', transform=ax.transAxes)
            ax.set_title("Temperature Evolution")
        ax.grid(True, alpha=0.3)
        ax = axes[1, 1]
        ax.text(0.5, 0.5, "Operator output variance\n(not collected)", ha='center', va='center', transform=ax.transAxes)
        ax.set_title("Operator Output Variance"); ax.grid(True, alpha=0.3)
        ax = axes[1, 2]
        combined_weights = np.concatenate([weights_b, weights_a], axis=0)
        # Apply smoothing for correlation stability
        from scipy.ndimage import uniform_filter1d
        smoothed_weights = uniform_filter1d(combined_weights, size=3, axis=0)
        corr_matrix = np.corrcoef(smoothed_weights.T)
        im = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
        ax.set_title("Operator Weight Correlation")
        ax.set_xticks(range(K)); ax.set_yticks(range(K))
        short_names = [op_names[k][:8] if k < len(op_names) else f"Op_{k}" for k in range(K)]
        ax.set_xticklabels(short_names, rotation=45); ax.set_yticklabels(short_names)
        cbar = plt.colorbar(im, ax=ax); cbar.set_label('Correlation')
        plt.tight_layout(); plt.show()
def plot_liquid_weights(curves, id_to_name, n_show=6, seed=0):
    plot_enhanced_liquid_weights(curves, id_to_name, n_show, seed)

def plot_sensor_examples_aligned(curves, id_to_name, sensor_idx_list=None, n_cols=4):
    """
    Plot 2 samples per class for sensor reconstruction, showing selected sensors.
    """
    if len(curves)==0:
        print("(No visualization samples)")
        return

    # Group curves by true class
    class_to_curves = {}
    for ex in curves:
        true_label = ex["true"]
        if true_label == -1:
            continue
        if true_label not in class_to_curves:
            class_to_curves[true_label] = []
        class_to_curves[true_label].append(ex)

    # Select 2 samples per class
    selected_curves = []
    for cls_id in sorted(class_to_curves.keys()):
        cls_curves = class_to_curves[cls_id]
        random.shuffle(cls_curves)
        selected_curves.extend(cls_curves[:2])

    if len(selected_curves) == 0:
        print("(No valid samples to visualize)")
        return

    ex = selected_curves[0]
    xb = ex["x_before"]; xa = ex["x_after"]; ya = ex["ya_hat"]
    L, C = xb.shape
    Lhat = ya.shape[0]
    if sensor_idx_list is None:
        step = max(1, C//8); sensor_idx_list = list(range(0, C, step))[:8]

    # Plot each selected sample
    for sample_idx, ex in enumerate(selected_curves):
        xb = ex["x_before"]; xa = ex["x_after"]; ya = ex["ya_hat"]
        sample_id = ex["sample_id"]
        true_label = ex["true"]
        pred_label = ex["pred"]
        true_name = id_to_name.get(true_label, f"Class_{true_label}")
        pred_name = id_to_name.get(pred_label, f"Class_{pred_label}")

        L, C = xb.shape
        Lhat = ya.shape[0]

        n = len(sensor_idx_list)
        n_rows = int(np.ceil(n/n_cols))
        plt.figure(figsize=(n_cols*4.3, n_rows*3.1))
        plt.suptitle(f"Sensor Reconstruction - Flight sample {sample_id} | True={true_name} | Pred={pred_name}", fontsize=14)

        for i, s in enumerate(sensor_idx_list):
            plt.subplot(n_rows, n_cols, i+1)
            t_b = np.arange(L); t_a = np.arange(L)  # The post-phase x-axis starts at 0
            plt.plot(t_b, xb[:,s], label="Original (pre)", linewidth=1.2)
            plt.plot(t_a, xa[:,s], label="Original (post)",  linewidth=1.0, linestyle="--", alpha=0.7)
            t_pred = np.arange(1, 1+Lhat)  # Predictions start at 1
            plt.plot(t_pred, ya[:,s], label="Post prediction", linewidth=1.6)
            sensor_name = SENSOR_NAMES.get(s, f"Sensor_{s:02d}")
            plt.title(sensor_name, fontsize=9)
            if i==0: plt.legend()
            plt.grid(ls="--", alpha=.35)
        plt.tight_layout(); plt.show()

def _serialize_class_weights(obj):
    if obj is None:
        return None
    try:
        import numpy as np
        if isinstance(obj, torch.Tensor):
            return obj.detach().cpu().numpy().tolist()
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, (list, tuple)):
            return list(obj)
        else:
            return None
    except Exception:
        return None

def test_only(test_loader, ckpt_path, K=None, id_to_name=None, device=None,
              visualize=True, max_curve_keep=24):
    """
    Load a saved checkpoint and evaluate on the official test set.
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Peek one batch to get sensor dimension C (and fallback K if needed)
    first_batch = next(iter(test_loader))
    C = first_batch["x_before"].shape[-1]

    if K is None:
        # If K not provided, try to infer from labels present in test loader
        # (May undercount if some classes are absent. Prefer passing K from training.)
        print("[Info] K not provided; attempting to infer from test loader labels (may be lower than full K).")
        K = scan_num_classes_from_loader(test_loader)

    if id_to_name is None:
        id_to_name = build_default_id_to_name(K)

    # Build model and load weights
    model = DiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=K).to(device)
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state, strict=False)
    model.eval()

    print("\n" + "="*60)
    print("Final Test Set Evaluation")
    print("="*60)

    te = eval_epoch(model, test_loader, device, K, compute_acc=True)
    print(f"[Test] mse_b={te['mse_b']:.4f} mse_a={te['mse_a']:.4f} acc={te['acc']:.3f} ΔHI_impr={te['delta_mean']:.3f}")

    # Predictions & visualization data
    y_true, y_pred, all_delta_mean, all_uids, keep_curves = collect_test_predictions(
        model, test_loader, device, max_curve_keep=max_curve_keep, K=K, id_to_name=id_to_name
    )

    # Reports (only if labels present)
    if y_true.size > 0:
        print("\n[Classification Report]")
        labels_full = list(range(K))
        target_names = [id_to_name.get(i, f"Class_{i}") for i in labels_full]
        # --- FIX: force labels + names to avoid mismatch when some classes are absent in predictions ---
        print(classification_report(
            y_true, y_pred,
            labels=labels_full,
            target_names=target_names,
            zero_division=0
        ))

        print("\n[Confusion Matrix]")
        print(confusion_matrix(y_true, y_pred, labels=labels_full))

        df_delta = pd.DataFrame({
            "uid": all_uids[:len(y_true)],
            "true": y_true,
            "pred": y_pred,
            "delta_hi_mean": all_delta_mean[:len(y_true)]
        })
        print("\n[Learned ΔHI Statistics] By true class")
        print(df_delta.groupby("true")["delta_hi_mean"].describe())
        topk_by_delta(df_delta, id_to_name, k=3)

        if visualize:
            print("\n[Visualization] Learned health index curves with liquid weight adaptation...")
            plot_hi_examples_aligned(keep_curves[1:], id_to_name, n_show=16, seed=0)

            print("\n[Visualization] Liquid operator weights evolution...")
            plot_liquid_weights(keep_curves[1:], id_to_name, n_show=16, seed=0)

            print("\n[Visualization] Sensor reconstruction predictions...")
            plot_sensor_examples_aligned(keep_curves[1:], id_to_name, sensor_idx_list=None, n_cols=4)
    else:
        print("\n[Info] No labels in test loader; skipping classification report and confusion matrix.")

    return {"test_metrics": te, "K": K, "id_to_name": id_to_name, "curves": keep_curves}
meta = {
    "K": int(K),
    "id_to_name": {int(k): str(v) for k, v in id_to_name.items()},
    "sensor_dim": int(C),
    "best_es": None,             # Leave blank or fill manually if the early-stopping metric from training is unknown
    "best_epoch": None,          # Same as above
    "class_weights": _serialize_class_weights(class_weights_obj)
}



results = test_only(
    test_loader=test_loader,
    ckpt_path=str(resolve_data_file("best_model_NGFAID_250_100_7.pth")),
    K=K,
    id_to_name=id_to_name,
    device=None,
    visualize=True,          # Set to True to generate plots; use False in headless environments
    max_curve_keep=240
)

# Extract y_true, y_pred for plotting
y_true = results.get("y_true", np.array([]))
y_pred = results.get("y_pred", np.array([]))

def plot_test_results(
    y_true,
    y_pred,
    id_to_name=None,
    normalize_cm=False,
    cmap="Blues",
    title_left="Confusion Matrix",
    title_right="Classification Report",
    save_path=None,
    figsize=(18, 8),
    rotate_xticks=45
):
    """
    Plot confusion matrix + classification report side-by-side from TEST RESULTS.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # --------- determine class set & names ----------
    classes = np.unique(np.concatenate([y_true, y_pred]))
    classes = np.sort(classes)

    if id_to_name is None:
        class_names = [f"Class_{int(i)}" for i in classes]
    else:
        # id_to_name can be dict or list
        if isinstance(id_to_name, dict):
            class_names = [str(id_to_name.get(int(i), f"Class_{int(i)}")) for i in classes]
        else:
            # assume list-like indexable by class id
            class_names = [str(id_to_name[int(i)]) if int(i) < len(id_to_name) else f"Class_{int(i)}"
                           for i in classes]

    # --------- confusion matrix ----------
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    cm_display = cm.astype(float)
    if normalize_cm:
        row_sums = cm_display.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        cm_display = cm_display / row_sums

    # --------- classification report (as table) ----------
    # Use the *ordered* class names corresponding to `classes`
    report_dict = classification_report(
        y_true,
        y_pred,
        labels=classes,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    # Build a pandas DataFrame for neat table
    rows = class_names + ["accuracy", "macro avg", "weighted avg"]
    cols = ["precision", "recall", "f1-score", "support"]

    data = []
    for name in class_names:
        r = report_dict.get(name, {})
        data.append([r.get("precision", 0.0), r.get("recall", 0.0), r.get("f1-score", 0.0), int(r.get("support", 0))])

    # accuracy row
    acc = report_dict.get("accuracy", 0.0)
    data.append([acc, acc, acc, int(len(y_true))])

    # macro avg
    r = report_dict.get("macro avg", {})
    data.append([r.get("precision", 0.0), r.get("recall", 0.0), r.get("f1-score", 0.0), int(r.get("support", 0))])

    # weighted avg
    r = report_dict.get("weighted avg", {})
    data.append([r.get("precision", 0.0), r.get("recall", 0.0), r.get("f1-score", 0.0), int(r.get("support", 0))])

    rep_df = pd.DataFrame(data, index=rows, columns=cols)

    # --------- plotting ----------
    fig = plt.figure(figsize=figsize)

    # Left subplot: confusion matrix heatmap
    ax1 = plt.subplot(1, 2, 1)
    im = ax1.imshow(cm_display, interpolation="nearest", cmap=cmap, aspect="auto")
    ax1.set_title(title_left, fontsize=14)
    ax1.set_xlabel("Predicted label")
    ax1.set_ylabel("True label")
    ax1.set_xticks(np.arange(len(classes)))
    ax1.set_yticks(np.arange(len(classes)))
    ax1.set_xticklabels(class_names, rotation=rotate_xticks, ha="right")
    ax1.set_yticklabels(class_names)

    # Annotate cells (counts or normalized)
    fmt = ".2f" if normalize_cm else "d"
    thresh = cm_display.max() / 2.0 if cm_display.size > 0 else 0.0
    for i in range(cm_display.shape[0]):
        for j in range(cm_display.shape[1]):
            val = cm_display[i, j]
            # show both normalized and counts if desired (here we show one)
            text_color = "white" if val > thresh else "black"
            # Fix: convert to int when using 'd' format
            if normalize_cm:
                ax1.text(j, i, format(val, fmt), ha="center", va="center", color=text_color, fontsize=9)
            else:
                ax1.text(j, i, format(int(val), fmt), ha="center", va="center", color=text_color, fontsize=9)

    cbar = fig.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)
    cbar.ax.set_ylabel("Proportion" if normalize_cm else "Count", rotation=270, labelpad=12)

    # Right subplot: classification report table
    ax2 = plt.subplot(1, 2, 2)
    ax2.axis("off")
    ax2.set_title(title_right, fontsize=14, pad=10)

    # Convert numeric columns to strings with formatting
    rep_to_show = rep_df.copy()
    rep_to_show["precision"] = rep_to_show["precision"].map(lambda x: f"{x:.2f}")
    rep_to_show["recall"]    = rep_to_show["recall"].map(lambda x: f"{x:.2f}")
    rep_to_show["f1-score"]  = rep_to_show["f1-score"].map(lambda x: f"{x:.2f}")
    rep_to_show["support"]   = rep_to_show["support"].map(lambda x: f"{int(x)}")

    table = ax2.table(
        cellText=rep_to_show.values,
        rowLabels=rep_to_show.index,
        colLabels=rep_to_show.columns,
        loc="center",
        cellLoc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.0, 1.2)  # widen rows a bit

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        print(f"[Saved] {save_path}")

    plt.show()

# Plot test results
if y_true.size > 0 and y_pred.size > 0:
    plot_test_results(
        y_true=y_true,
        y_pred=y_pred,
        id_to_name=id_to_name,
        normalize_cm=False,
        save_path="test_eval.png"
    )


## Results: Severity-Aware Damage Visualization


In [ ]:
# -*- coding: utf-8 -*-
import torch
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import os
import pickle
from typing import Dict, Tuple, Optional
import matplotlib.pyplot as plt
import pandas as pd
import random

# ============================================
# Read "real class labels" and replace default Class_i labels
# ============================================

SAVE_DIR = str(DATA_DIR)

def _invert_name_mapping(name_to_id: Dict) -> Dict[int, str]:
    """Invert {name -> id} to {id -> name}, ensuring all names are str with capitalized first letter."""
    id_to_name = {}
    for name, idx in name_to_id.items():
        try:
            idx_int = int(idx)
        except Exception:
            continue
        # Capitalize first letter of class name
        name_str = str(name)
        if name_str:
            name_str = name_str[0].upper() + name_str[1:]
        id_to_name[idx_int] = name_str
    return id_to_name

def read_real_class_id_to_name(
    save_dir: str = SAVE_DIR,
    preferred_filename: Optional[str] = None
) -> Tuple[Optional[Dict[int, str]], Optional[int], Optional[str]]:
    """
    Read real class names from saved label_mappings_*.pkl:
      - Prioritize using 'label_to_id' (usually string class -> id)
      - Otherwise fallback to 'class_to_id' / 'target_class_to_id' / 'hclass_to_id'
    Returns:
      (id_to_name, K, used_file_path)
      Returns (None, None, None) if reading fails
    """
    if preferred_filename:
        candidates = [os.path.join(save_dir, preferred_filename)]
    else:
        # Automatically search for pkl files starting with label_mappings_sample_split_window_
        candidates = []
        if os.path.isdir(save_dir):
            for fn in os.listdir(save_dir):
                if fn.startswith("label_mappings_sample_split_window_") and fn.endswith(".pkl"):
                    candidates.append(os.path.join(save_dir, fn))
        # Select the newest
        candidates.sort(key=lambda p: os.path.getmtime(p), reverse=True)

    for path in candidates:
        try:
            with open(path, "rb") as f:
                lm = pickle.load(f)
        except Exception:
            continue

        # Prioritize using string labels
        if isinstance(lm, dict) and "label_to_id" in lm and lm["label_to_id"]:
            id_to_name = _invert_name_mapping(lm["label_to_id"])
            if id_to_name:
                K = max(id_to_name.keys()) + 1
                return id_to_name, K, path

        # Otherwise try numeric class labels (convert original numeric values to string for display)
        for key in ["class_to_id", "target_class_to_id", "hclass_to_id"]:
            if isinstance(lm, dict) and key in lm and lm[key]:
                id_to_name = _invert_name_mapping(lm[key])
                if id_to_name:
                    # Note: these mappings are {original class value -> encoded_id}, already inverted to {id -> original class value(str)}
                    K = max(id_to_name.keys()) + 1
                    return id_to_name, K, path

    return None, None, None

def _infer_C(train_loader=None, test_loader=None):
    """Infer sensor dimension C from loader."""
    probe_loader = train_loader if train_loader is not None else test_loader
    assert probe_loader is not None, "Need available train_loader or test_loader to infer C."
    try:
        first_batch = next(iter(probe_loader))
        C = first_batch["x_before"].shape[-1]
        return int(C)
    except Exception as e:
        print(f"[Error] Failed to infer C from loader: {e}")
        raise

def build_default_id_to_name(K: int):
    return {i: f"Class_{i}" for i in range(K)}

# ========== Use real class names to replace default Class_i ==========
# 1) Still use loader to infer C (sensor dimension)
# 2) K prioritizes using real class count from reading; fallback to inferring from loader if reading fails

# NOTE: train_loader and test_loader must be defined before this code block
# If they are not defined, you will get a NameError
try:
    C = _infer_C(train_loader=train_loader, test_loader=test_loader)
except (NameError, RuntimeError) as e:
    print(f"[Error] Could not infer C: {e}")
    print("[Info] Setting C to default value 28")
    C = 28  # Default sensor dimension for this dataset

# First try to read real class labels from label_mappings
real_id_to_name, real_K, used_path = read_real_class_id_to_name(SAVE_DIR)

if real_id_to_name is not None and real_K is not None:
    id_to_name = real_id_to_name
    K = int(real_K)
    print(f"[Info] Using real class names (total {K} classes), source: {used_path}")
else:
    # Fallback: infer K from loader, then assign default names
    try:
        if 'train_loader' in globals():
            K = scan_num_classes_from_loader(train_loader)
        elif 'test_loader' in globals():
            K = scan_num_classes_from_loader(test_loader)
        else:
            raise RuntimeError("Neither train_loader nor test_loader is defined")
    except Exception as e:
        print(f"[Error] Cannot infer K from loader: {e}")
        print("[Info] Setting K to default value 14")
        K = 14  # Default number of classes
    id_to_name = build_default_id_to_name(K)
    print(f"[Warn] label_mappings*.pkl not found, fallback to default class names. K={K}")

print(f"[Info] Inferred sensor dim C={C}, num classes K={K}")

class_weights_obj = None

# ============================================
# GAFID Aviation Maintenance Dataset - Sensor Data Description
# ============================================
SENSOR_NAMES = {
    0: "volt1 (Main bus voltage)",
    1: "volt2 (Essential bus voltage)",
    2: "amp1 (Main battery ammeter)",
    3: "amp2 (Standby battery ammeter)",
    4: "FQtyL (Fuel quantity left)",
    5: "FQtyR (Fuel quantity right)",
    6: "E1 FFlow (Engine fuel flow)",
    7: "E1 OilT (Engine oil temp)",
    8: "E1 OilP (Engine oil pressure)",
    9: "E1 RPM (Engine RPM)",
    10: "E1 CHT1 (Cylinder 1 head temp)",
    11: "E1 CHT2 (Cylinder 2 head temp)",
    12: "E1 CHT3 (Cylinder 3 head temp)",
    13: "E1 CHT4 (Cylinder 4 head temp)",
    14: "E1 EGT1 (Cylinder 1 exhaust temp)",
    15: "E1 EGT2 (Cylinder 2 exhaust temp)",
    16: "E1 EGT3 (Cylinder 3 exhaust temp)",
    17: "E1 EGT4 (Cylinder 4 exhaust temp)",
    18: "OAT (Outside air temp)",
    19: "IAS (Indicated air speed)",
    20: "VSpd (Vertical speed)",
    21: "NormAc (Normal acceleration)",
    22: "AltMSL (Altitude MSL)",
    23: "Throttle (Engine power)",
    24: "Fuel flow control",
    25: "Flight stability",
    26: "Aerodynamic drag",
    27: "Total weight"
}

# ============================================
# Cessna 172 Fault Severity Ranking (for intelligent sample selection)
# ============================================
FAULT_SEVERITY_MAP = {
    # Critical issues (Severity 1) - Immediate flight safety risk
    "Engine/propeller overspeed or damage": 1,
    "Propeller damage": 1,
    "Engine overspeed": 1,

    # Severe issues (Severity 2) - Unsafe to fly, grounding required
    "Cylinder compression issue": 2,
    "Compression issue": 2,
    "Cylinder failure": 2,

    # Moderate to severe (Severity 3) - Repair before flight
    "Intake gasket leak/damage": 3,
    "Intake leak": 3,
    "Gasket damage": 3,

    # Moderate (Severity 4) - Inspect/repair, long-term damage risk
    "Baffle crack/damage/loose/miss": 4,
    "Baffle damage": 4,
    "Baffle loose": 4,

    # Default for unlisted faults
    "General": 5,
    "Poor": 5,
    "Perfect": 6
}

def get_fault_severity(fault_name: str) -> int:
    """
    Get severity level for a fault name.
    Lower number = more severe (1=critical, 6=perfect condition)
    """
    fault_lower = fault_name.lower()
    for key, severity in FAULT_SEVERITY_MAP.items():
        if key.lower() in fault_lower:
            return severity
    return 5  # default moderate severity

def select_samples_by_severity(curves, id_to_name, samples_per_class=2, prioritize_severe=True):
    """
    Intelligently select samples for visualization based on fault severity.
    For severe faults: prioritize samples with larger ΔHI (better maintenance effect)
    For minor faults: show representative samples

    Args:
        curves: list of sample curves
        id_to_name: mapping of class ID to fault name
        samples_per_class: number of samples to show per class
        prioritize_severe: if True, use severity-aware selection

    Returns:
        List of selected curves
    """
    if not prioritize_severe:
        # Fallback to random selection
        class_to_curves = {}
        for ex in curves:
            true_label = ex["true"]
            if true_label == -1:
                continue
            if true_label not in class_to_curves:
                class_to_curves[true_label] = []
            class_to_curves[true_label].append(ex)

        selected = []
        for cls_id in sorted(class_to_curves.keys()):
            cls_curves = class_to_curves[cls_id]
            random.shuffle(cls_curves)
            selected.extend(cls_curves[:samples_per_class])
        return selected

    # Severity-aware selection
    class_to_curves = {}
    for ex in curves:
        true_label = ex["true"]
        if true_label == -1:
            continue
        fault_name = id_to_name.get(true_label, f"Class_{true_label}")
        severity = get_fault_severity(fault_name)

        # Calculate ΔHI for this sample
        h_b = ex.get("h_before", np.array([]))
        h_a = ex.get("h_after", np.array([]))
        if len(h_b) > 0 and len(h_a) > 0:
            delta_hi = float(np.mean(h_a - h_b))
        else:
            delta_hi = 0.0

        if true_label not in class_to_curves:
            class_to_curves[true_label] = []

        class_to_curves[true_label].append({
            "curve": ex,
            "severity": severity,
            "delta_hi": delta_hi,
            "fault_name": fault_name
        })

    # Select samples
    selected = []
    for cls_id in sorted(class_to_curves.keys()):
        cls_items = class_to_curves[cls_id]
        severity = cls_items[0]["severity"]

        # For critical/severe faults (1-3): prioritize high ΔHI (good maintenance effect)
        if severity <= 3:
            cls_items.sort(key=lambda x: x["delta_hi"], reverse=True)
        # For moderate faults (4): show balanced samples
        elif severity == 4:
            cls_items.sort(key=lambda x: abs(x["delta_hi"]))
        # For minor/perfect (5-6): show representative samples
        else:
            random.shuffle(cls_items)

        selected.extend([item["curve"] for item in cls_items[:samples_per_class]])

    return selected

@torch.no_grad()
def collect_test_predictions(
    model, loader, device, max_curve_keep=24, K=14, id_to_name=None,
    use_damage_for_curves=True  # Optional new switch; use damage curves for plotting by default
):
    model.eval()
    y_true, y_pred = [], []
    all_delta_mean, all_sample_ids = [], []
    keep_curves = []

    # === Read CustomKAN gain / bias to reconstruct damage ===
    customkan = model.encoder.customkan
    gain_ck = torch.clamp(F.softplus(customkan.raw_gain), 0.1, 2.0)
    bias_ck = torch.clamp(customkan.bias, -2.0, 2.0)

    for batch in loader:
        xb, xa, labels, mask, lengths, sample_ids, matching_costs = adapt_batch(batch, device)

        # Raw forward pass
        yb_hat, ya_hat, h_b, h_a, logits, weights_b, weights_a, temp_b, temp_a, steady_b, steady_a, op_outputs_b, op_outputs_a = model(xb, xa, mask)

        # Clean NaN/Inf values
        yb_hat, ya_hat, h_b, h_a, logits = sanitize_tensors(yb_hat, ya_hat, h_b, h_a, logits)
        op_outputs_b = torch.nan_to_num(op_outputs_b, nan=0.0, posinf=1e6, neginf=-1e6)
        op_outputs_a = torch.nan_to_num(op_outputs_a, nan=0.0, posinf=1e6, neginf=-1e6)

        # Ensure logits dimension matches expected K
        if logits.shape[-1] != K:
            if logits.shape[-1] < K:
                padding = torch.zeros(logits.shape[0], K - logits.shape[-1], device=logits.device)
                logits = torch.cat([logits, padding], dim=1)
            else:
                logits = logits[:, :K]

        prob = F.softmax(logits, dim=1)     # (B,K)
        pred = prob.argmax(1)               # (B,)

        valid = mask.sum(1, keepdim=True).clamp_min(1.0)
        delta_mean = ((h_a - h_b) * mask).sum(1, keepdim=True) / valid  # (B,1)
        delta_mean = torch.nan_to_num(delta_mean, nan=0.0)

        if labels is not None:
            # Filter out ignore_index labels
            valid_labels = labels != -100
            if valid_labels.any():
                y_true.append(labels[valid_labels].detach().cpu().numpy())
                y_pred.append(pred[valid_labels].detach().cpu().numpy())
        all_delta_mean.append(delta_mean.squeeze(1).cpu().numpy())
        all_sample_ids.extend(sample_ids)

        # === Key step: reconstruct CustomKAN damage using op_outputs and weights ===
        # shape: op_outputs_* = (B, Tm, K), weights_* = (B, Tm, K)
        raw_dmg_b = (op_outputs_b * weights_b).sum(dim=-1)              # (B, Tm_b)
        raw_dmg_a = (op_outputs_a * weights_a).sum(dim=-1)              # (B, Tm_a)
        damage_b  = torch.clamp(gain_ck * raw_dmg_b + bias_ck, 0.0, 100.0)
        damage_a  = torch.clamp(gain_ck * raw_dmg_a + bias_ck, 0.0, 100.0)

        for i in range(xb.size(0)):
            if len(keep_curves) >= max_curve_keep: break
            L_i = int(lengths[i].item())
            L_recon = min(L_i - 2, yb_hat.size(1))

            # Align damage lengths by taking the minimum length across all three arrays so pre/post plots remain safe
            Tmb = damage_b.size(1)
            Tma = damage_a.size(1)
            L_eff = max(1, min(L_i, Tmb, Tma))

            # Choose which curve to plot: damage or HI
            if use_damage_for_curves:
                curve_pre  = damage_b[i, :L_eff].detach().cpu().numpy()
                curve_post = damage_a[i, :L_eff].detach().cpu().numpy()
            else:
                curve_pre  = h_b[i, :L_eff].detach().cpu().numpy()
                curve_post = h_a[i, :L_eff].detach().cpu().numpy()

            keep_curves.append({
                "sample_id": sample_ids[i],
                "true": int(labels[i].cpu().item()) if labels is not None and labels[i] != -100 else -1,
                "pred": int(pred[i].cpu().item()),
                "prob": prob[i].detach().cpu().numpy(),

                # === Fill h_before/h_after with damage so the existing plot* helpers can render damage directly ===
                "h_before": curve_pre,
                "h_after":  curve_post,

                # Keep the original HI as well in case it is needed for analysis
                "hi_before": h_b[i, :L_eff].detach().cpu().numpy(),
                "hi_after":  h_a[i, :L_eff].detach().cpu().numpy(),

                "weights_before": weights_b[i, :L_eff].detach().cpu().numpy() if weights_b is not None else None,
                "weights_after":  weights_a[i, :L_eff].detach().cpu().numpy() if weights_a is not None else None,
                "temp_before": temp_b[i, :L_eff].detach().cpu().numpy() if temp_b is not None else None,
                "temp_after":  temp_a[i, :L_eff].detach().cpu().numpy() if temp_a is not None else None,
                "steady_before": float(steady_b[i].detach().cpu().item()) if steady_b is not None else None,
                "steady_after":  float(steady_a[i].detach().cpu().item()) if steady_a is not None else None,
                "op_outputs_before": op_outputs_b[i, :L_eff].detach().cpu().numpy() if op_outputs_b is not None else [],
                "op_outputs_after":  op_outputs_a[i, :L_eff].detach().cpu().numpy() if op_outputs_a is not None else [],

                "yb_hat":   yb_hat[i, :L_recon].detach().cpu().numpy(),
                "ya_hat":   ya_hat[i, :L_recon].detach().cpu().numpy(),
                "x_before": xb[i, :L_eff].detach().cpu().numpy(),
                "x_after":  xa[i, :L_eff].detach().cpu().numpy(),
            })
    y_true = np.concatenate(y_true, axis=0) if len(y_true)>0 else np.array([])
    y_pred = np.concatenate(y_pred, axis=0) if len(y_pred)>0 else np.array([])
    all_delta_mean = np.concatenate(all_delta_mean, axis=0) if len(all_delta_mean)>0 else np.array([])
    return y_true, y_pred, all_delta_mean, all_sample_ids, keep_curves
# ---------------------------------------------------------------------
# Plotting
# ---------------------------------------------------------------------
def remove_constant_segments(hi_values, threshold=1e-6):
    if len(hi_values) <= 1:
        return hi_values, np.arange(len(hi_values))
    diffs = np.abs(np.diff(hi_values))
    valid_mask = np.ones(len(hi_values), dtype=bool)
    valid_mask[1:] = diffs > threshold
    if np.sum(valid_mask) < 2:
        valid_mask[0] = True
        valid_mask[-1] = True
    valid_indices = np.where(valid_mask)[0]
    return hi_values[valid_indices], valid_indices

def plot_hi_examples_aligned(curves, id_to_name, n_show=6, seed=0):
    """
    Plot samples intelligently selected based on fault severity.
    For critical/severe faults: show samples with best maintenance effect (high ΔHI)
    Title format: "Flight sample XX | [Severity] Class_name | p=... | ΔHI_mean=... | ΔHI_max=..."
    """
    if len(curves) == 0:
        print("(No visualization samples)")
        return

    # Intelligently select samples based on severity
    selected_curves = select_samples_by_severity(
        curves, id_to_name,
        samples_per_class=2,
        prioritize_severe=True
    )

    n_show = len(selected_curves)
    if n_show == 0:
        print("(No valid samples to visualize)")
        return

    cols = 2  # 2 samples per row
    rows = int(np.ceil(n_show / cols))
    plt.figure(figsize=(cols * 6, rows * 4))

    for k, ex in enumerate(selected_curves):
        hb = ex["h_before"]
        ha = ex["h_after"]
        L = min(len(hb), len(ha))
        hb, ha = hb[:L], ha[:L]

        # Skip first and last 10 points
        skip = 10
        if L > 2 * skip:
            hb = hb[skip:-skip]
            ha = ha[skip:-skip]
        else:
            # If too short, just use all points
            pass

        # Reverse damage trend (plot damage in reverse order)
        hb = hb[::-1]
        ha = ha[::-1]

        hb_clean, hb_indices = remove_constant_segments(hb)
        ha_clean, ha_indices = remove_constant_segments(ha)
        t_b = hb_indices
        t_a = ha_indices  # The post-phase x-axis starts at 0 and is not accumulated

        sample_id = ex["sample_id"]
        pred = ex["pred"]
        true = ex["true"]
        prob = ex["prob"]
        true_name = id_to_name.get(true, f"Class_{true}")
        pred_name = id_to_name.get(pred, f"Class_{pred}")

        # Get severity indicator
        severity = get_fault_severity(true_name)
        severity_icons = {1: "🚨", 2: "⚠️", 3: "⚡", 4: "🔧", 5: "ℹ️", 6: "✅"}
        severity_icon = severity_icons.get(severity, "")

        plt.subplot(rows, cols, k + 1)
        if len(hb_clean) > 1:
            plt.plot(t_b, hb_clean, label="Learned HI (Pre)", linewidth=1.8, marker='o', markersize=3)
        if len(ha_clean) > 1:
            plt.plot(t_a, ha_clean, label="Learned HI (Post)", linewidth=1.8, linestyle='--', marker='s', markersize=3)
        d_mean = float(np.mean(np.abs(ha - hb)))
        d_max = float(np.max(ha - hb))

        # Add steady state info if available
        steady_info = ""
        # if ex.get("steady_after") is not None:
        #     steady_info = f" | SS={ex['steady_after']:.3f}"

        plt.title(f" Flight sample {sample_id} | {true_name}\n"
                  f"ΔHI_mean={d_mean:.3f}", fontsize=9)#, ΔHI_max={d_max:.3f} {severity_icon}{steady_info}p={prob[pred]:.2f} |
        plt.xlabel("Cycle")
        plt.ylabel("Health Index (higher=better)")
        plt.grid(ls='--', alpha=.35)
        if k % cols == 0:
            plt.legend()
    plt.tight_layout()
    plt.show()

def plot_enhanced_liquid_weights(curves, id_to_name, n_show=3, seed=0):
    """
    Plot liquid weights for intelligently selected samples based on fault severity.
    """
    if len(curves) == 0:
        print("(No visualization samples)")
        return
    curves_with_weights = [c for c in curves if c.get("weights_before") is not None]
    if len(curves_with_weights) == 0:
        print("(No weight information available)")
        return

    # Intelligently select samples based on severity
    selected_curves = select_samples_by_severity(
        curves_with_weights, id_to_name,
        samples_per_class=2,
        prioritize_severe=True
    )

    n_show = len(selected_curves)
    if n_show == 0:
        print("(No valid samples to visualize)")
        return

    op_names = ["MonotonicLinear", "MonotonicFlat", "ConcaveLog", "SaturationSigmoid",
                "HingeReLU", "Polynomial", "DampedSin", "PiecewiseLinear"]

    for i in range(n_show):
        ex = selected_curves[i]
        weights_b = ex["weights_before"]
        weights_a = ex["weights_after"]
        temp_b = ex.get("temp_before")
        temp_a = ex.get("temp_after")
        sample_id = ex["sample_id"]
        true_label = ex["true"]
        pred_label = ex["pred"]
        true_name = id_to_name.get(true_label, f"Class_{true_label}")
        pred_name = id_to_name.get(pred_label, f"Class_{pred_label}")

        # Get severity indicator
        severity = get_fault_severity(true_name)
        severity_icons = {1: "🚨", 2: "⚠️", 3: "⚡", 4: "🔧", 5: "ℹ️", 6: "✅"}
        severity_icon = severity_icons.get(severity, "")

        # Skip first and last 10 points
        skip = 10
        if weights_b.shape[0] > 2 * skip:
            weights_b = weights_b[skip:-skip]
        if weights_a.shape[0] > 2 * skip:
            weights_a = weights_a[skip:-skip]
        if temp_b is not None and len(temp_b) > 2 * skip:
            temp_b = temp_b[skip:-skip]
        if temp_a is not None and len(temp_a) > 2 * skip:
            temp_a = temp_a[skip:-skip]

        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle(f"{severity_icon} Dynamic Parameter Modulation - Flight sample {sample_id} | True={true_name} | Pred={pred_name}", fontsize=14)
        ax = axes[0, 0]
        T_b, K = weights_b.shape
        for k in range(K):
            ax.plot(range(T_b), weights_b[:, k], label=op_names[k] if k < len(op_names) else f"Op_{k}",
                   marker='o', markersize=2, linewidth=1.5)
        ax.set_title("Pre-maintenance Weights"); ax.set_xlabel("Time Step"); ax.set_ylabel("Weight")
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left'); ax.grid(True, alpha=0.3)
        ax = axes[0, 1]
        T_a, _ = weights_a.shape
        for k in range(K):
            ax.plot(range(T_a), weights_a[:, k], label=op_names[k] if k < len(op_names) else f"Op_{k}",
                   marker='s', markersize=2, linewidth=1.5, linestyle='--')
        ax.set_title("Post-maintenance Weights"); ax.set_xlabel("Time Step"); ax.set_ylabel("Weight")
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left'); ax.grid(True, alpha=0.3)
        ax = axes[0, 2]
        entropy_b = -np.sum(weights_b * np.log(weights_b + 1e-8), axis=1)
        entropy_a = -np.sum(weights_a * np.log(weights_a + 1e-8), axis=1)
        ax.plot(range(T_b), entropy_b, label="Pre", marker='o', linewidth=2)
        ax.plot(range(T_a), entropy_a, label="Post", marker='s', linewidth=2, linestyle='--')
        ax.axhline(np.log(K), color='r', linestyle=':', label=f'Max Entropy ({np.log(K):.2f})')
        ax.set_title("Weight Entropy Over Time"); ax.set_xlabel("Time Step"); ax.set_ylabel("Entropy")
        ax.legend(); ax.grid(True, alpha=0.3)
        ax = axes[1, 0]
        if temp_b is not None and temp_a is not None:
            ax.plot(range(T_b), temp_b, label="Pre", marker='o', linewidth=2)
            ax.plot(range(T_a), temp_a, label="Post", marker='s', linewidth=2, linestyle='--')
            ax.set_title("Temperature Evolution"); ax.set_xlabel("Time Step"); ax.set_ylabel("Temperature"); ax.legend()
        else:
            ax.text(0.5, 0.5, "Temperature data\nnot available", ha='center', va='center', transform=ax.transAxes)
            ax.set_title("Temperature Evolution")
        ax.grid(True, alpha=0.3)
        ax = axes[1, 1]
        ax.text(0.5, 0.5, "Operator output variance\n(not collected)", ha='center', va='center', transform=ax.transAxes)
        ax.set_title("Operator Output Variance"); ax.grid(True, alpha=0.3)
        ax = axes[1, 2]
        combined_weights = np.concatenate([weights_b, weights_a], axis=0)
        # Apply smoothing for correlation stability
        from scipy.ndimage import uniform_filter1d
        smoothed_weights = uniform_filter1d(combined_weights, size=3, axis=0)
        corr_matrix = np.corrcoef(smoothed_weights.T)
        im = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
        ax.set_title("Operator Weight Correlation")
        ax.set_xticks(range(K)); ax.set_yticks(range(K))
        short_names = [op_names[k][:8] if k < len(op_names) else f"Op_{k}" for k in range(K)]
        ax.set_xticklabels(short_names, rotation=45); ax.set_yticklabels(short_names)
        cbar = plt.colorbar(im, ax=ax); cbar.set_label('Correlation')
        plt.tight_layout(); plt.show()
def plot_liquid_weights(curves, id_to_name, n_show=6, seed=0):
    plot_enhanced_liquid_weights(curves, id_to_name, n_show, seed)

def plot_sensor_examples_aligned(curves, id_to_name, sensor_idx_list=None, n_cols=4):
    """
    Plot sensor reconstruction for intelligently selected samples based on fault severity.
    """
    if len(curves)==0:
        print("(No visualization samples)")
        return

    # Intelligently select samples based on severity
    selected_curves = select_samples_by_severity(
        curves, id_to_name,
        samples_per_class=2,
        prioritize_severe=True
    )

    if len(selected_curves) == 0:
        print("(No valid samples to visualize)")
        return

    ex = selected_curves[0]
    xb = ex["x_before"]; xa = ex["x_after"]; ya = ex["ya_hat"]
    L, C = xb.shape
    Lhat = ya.shape[0]
    if sensor_idx_list is None:
        step = max(1, C//8); sensor_idx_list = list(range(0, C, step))[:8]

    # Plot each selected sample
    for sample_idx, ex in enumerate(selected_curves):
        xb = ex["x_before"]; xa = ex["x_after"]; ya = ex["ya_hat"]
        sample_id = ex["sample_id"]
        true_label = ex["true"]
        pred_label = ex["pred"]
        true_name = id_to_name.get(true_label, f"Class_{true_label}")
        pred_name = id_to_name.get(pred_label, f"Class_{pred_label}")

        # Get severity indicator
        severity = get_fault_severity(true_name)
        severity_icons = {1: "🚨", 2: "⚠️", 3: "⚡", 4: "🔧", 5: "ℹ️", 6: "✅"}
        severity_icon = severity_icons.get(severity, "")

        L, C = xb.shape
        Lhat = ya.shape[0]

        n = len(sensor_idx_list)
        n_rows = int(np.ceil(n/n_cols))
        plt.figure(figsize=(n_cols*4.3, n_rows*3.1))
        plt.suptitle(f"{severity_icon} Sensor Reconstruction - Flight sample {sample_id} | True={true_name} | Pred={pred_name}", fontsize=14)

        for i, s in enumerate(sensor_idx_list):
            plt.subplot(n_rows, n_cols, i+1)
            t_b = np.arange(L); t_a = np.arange(L)  # The post-phase x-axis starts at 0
            plt.plot(t_b, xb[:,s], label="Original (pre)", linewidth=1.2)
            plt.plot(t_a, xa[:,s], label="Original (post)",  linewidth=1.0, linestyle="--", alpha=0.7)
            t_pred = np.arange(1, 1+Lhat)  # Predictions start at 1
            plt.plot(t_pred, ya[:,s], label="Post prediction", linewidth=1.6)
            sensor_name = SENSOR_NAMES.get(s, f"Sensor_{s:02d}")
            plt.title(sensor_name, fontsize=9)
            if i==0: plt.legend()
            plt.grid(ls="--", alpha=.35)
        plt.tight_layout(); plt.show()

def _serialize_class_weights(obj):
    if obj is None:
        return None
    try:
        import numpy as np
        if isinstance(obj, torch.Tensor):
            return obj.detach().cpu().numpy().tolist()
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, (list, tuple)):
            return list(obj)
        else:
            return None
    except Exception:
        return None

def test_only(test_loader, ckpt_path, K=None, id_to_name=None, device=None,
              visualize=True, max_curve_keep=24):
    """
    Load a saved checkpoint and evaluate on the official test set.
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Peek one batch to get sensor dimension C (and fallback K if needed)
    try:
        first_batch = next(iter(test_loader))
        C = first_batch["x_before"].shape[-1]
    except Exception as e:
        print(f"[Error] Failed to read first batch from test_loader: {e}")
        print("[Info] Using default C=28")
        C = 28

    if K is None:
        # If K not provided, try to infer from labels present in test loader
        # (May undercount if some classes are absent. Prefer passing K from training.)
        print("[Info] K not provided; attempting to infer from test loader labels (may be lower than full K).")
        try:
            K = scan_num_classes_from_loader(test_loader)
        except Exception as e:
            print(f"[Error] Failed to infer K: {e}")
            print("[Info] Using default K=14")
            K = 14

    if id_to_name is None:
        id_to_name = build_default_id_to_name(K)

    # Build model and load weights
    model = DiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=K).to(device)
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state, strict=False)
    model.eval()

    print("\n" + "="*60)
    print("Final Test Set Evaluation")
    print("="*60)

    te = eval_epoch(model, test_loader, device, K, compute_acc=True)
    print(f"[Test] mse_b={te['mse_b']:.4f} mse_a={te['mse_a']:.4f} acc={te['acc']:.3f} ΔHI_impr={te['delta_mean']:.3f}")

    # Predictions & visualization data
    y_true, y_pred, all_delta_mean, all_uids, keep_curves = collect_test_predictions(
        model, test_loader, device, max_curve_keep=max_curve_keep, K=K, id_to_name=id_to_name
    )

    # Reports (only if labels present)
    if y_true.size > 0:
        print("\n[Classification Report]")
        labels_full = list(range(K))
        target_names = [id_to_name.get(i, f"Class_{i}") for i in labels_full]
        # --- FIX: force labels + names to avoid mismatch when some classes are absent in predictions ---
        print(classification_report(
            y_true, y_pred,
            labels=labels_full,
            target_names=target_names,
            zero_division=0
        ))

        print("\n[Confusion Matrix]")
        print(confusion_matrix(y_true, y_pred, labels=labels_full))

        df_delta = pd.DataFrame({
            "uid": all_uids[:len(y_true)],
            "true": y_true,
            "pred": y_pred,
            "delta_hi_mean": all_delta_mean[:len(y_true)]
        })
        print("\n[Learned ΔHI Statistics] By true class")
        print(df_delta.groupby("true")["delta_hi_mean"].describe())
        topk_by_delta(df_delta, id_to_name, k=3)

        if visualize:
            print("\n[Visualization] Learned health index curves with intelligent severity-based sample selection...")
            print("🚨=Critical | ⚠️=Severe | ⚡=Moderate-Severe | 🔧=Moderate | ℹ️=Minor | ✅=Perfect")
            plot_hi_examples_aligned(keep_curves[1:], id_to_name, n_show=16, seed=0)

            print("\n[Visualization] Liquid operator weights evolution (severity-aware)...")
            plot_liquid_weights(keep_curves[1:], id_to_name, n_show=16, seed=0)

            print("\n[Visualization] Sensor reconstruction predictions (severity-aware)...")
            plot_sensor_examples_aligned(keep_curves[1:], id_to_name, sensor_idx_list=None, n_cols=4)
    else:
        print("\n[Info] No labels in test loader; skipping classification report and confusion matrix.")

    return {"test_metrics": te, "K": K, "id_to_name": id_to_name, "curves": keep_curves, "y_true": y_true, "y_pred": y_pred}
meta = {
    "K": int(K),
    "id_to_name": {int(k): str(v) for k, v in id_to_name.items()},
    "sensor_dim": int(C),
    "best_es": None,             # Leave blank or fill manually if the early-stopping metric from training is unknown
    "best_epoch": None,          # Same as above
    "class_weights": _serialize_class_weights(class_weights_obj)
}

# Only run test_only if loaders are defined and files exist
if 'test_loader' in globals() and test_loader is not None:
    try:
        results = test_only(
            test_loader=test_loader,
            ckpt_path=str(resolve_data_file("best_model_NGFAID_250_100_7.pth")),
            K=K,
            id_to_name=id_to_name,
            device=None,
            visualize=True,          # Set to True to generate plots; use False in headless environments
            max_curve_keep=240
        )

        # Save results to pickle for later use
        RESULTS_PKL = os.path.join(SAVE_DIR, "results_from_test_only.pkl")
        with open(RESULTS_PKL, 'wb') as f:
            pickle.dump({
                'curves': results['curves'],
                'id_to_name': results['id_to_name'],
                'K': results['K'],
                'y_true': results.get('y_true', np.array([])),
                'y_pred': results.get('y_pred', np.array([])),
                'meta': meta  # Include metadata in saved results
            }, f)
        print(f"[Info] Saved test results to {RESULTS_PKL}")

        # Extract y_true, y_pred for plotting
        y_true = results.get("y_true", np.array([]))
        y_pred = results.get("y_pred", np.array([]))

        def plot_test_results(
            y_true,
            y_pred,
            id_to_name=None,
            normalize_cm=False,
            cmap="Blues",
            title_left="Confusion Matrix",
            title_right="Classification Report",
            save_path=None,
            figsize=(18, 8),
            rotate_xticks=45
        ):
            """
            Plot confusion matrix + classification report side-by-side from TEST RESULTS.
            """
            y_true = np.asarray(y_true)
            y_pred = np.asarray(y_pred)

            # --------- determine class set & names ----------
            classes = np.unique(np.concatenate([y_true, y_pred]))
            classes = np.sort(classes)

            if id_to_name is None:
                class_names = [f"Class_{int(i)}" for i in classes]
            else:
                # id_to_name can be dict or list
                if isinstance(id_to_name, dict):
                    class_names = [str(id_to_name.get(int(i), f"Class_{int(i)}")) for i in classes]
                else:
                    # assume list-like indexable by class id
                    class_names = [str(id_to_name[int(i)]) if int(i) < len(id_to_name) else f"Class_{int(i)}"
                                   for i in classes]

            # --------- confusion matrix ----------
            cm = confusion_matrix(y_true, y_pred, labels=classes)
            cm_display = cm.astype(float)
            if normalize_cm:
                row_sums = cm_display.sum(axis=1, keepdims=True)
                row_sums[row_sums == 0] = 1.0
                cm_display = cm_display / row_sums

            # --------- classification report (as table) ----------
            # Use the *ordered* class names corresponding to `classes`
            report_dict = classification_report(
                y_true,
                y_pred,
                labels=classes,
                target_names=class_names,
                output_dict=True,
                zero_division=0
            )

            # Build a pandas DataFrame for neat table
            rows = class_names + ["accuracy", "macro avg", "weighted avg"]
            cols = ["precision", "recall", "f1-score", "support"]

            data = []
            for name in class_names:
                r = report_dict.get(name, {})
                data.append([r.get("precision", 0.0), r.get("recall", 0.0), r.get("f1-score", 0.0), int(r.get("support", 0))])

            # accuracy row
            acc = report_dict.get("accuracy", 0.0)
            data.append([acc, acc, acc, int(len(y_true))])

            # macro avg
            r = report_dict.get("macro avg", {})
            data.append([r.get("precision", 0.0), r.get("recall", 0.0), r.get("f1-score", 0.0), int(r.get("support", 0))])

            # weighted avg
            r = report_dict.get("weighted avg", {})
            data.append([r.get("precision", 0.0), r.get("recall", 0.0), r.get("f1-score", 0.0), int(r.get("support", 0))])

            rep_df = pd.DataFrame(data, index=rows, columns=cols)

            # --------- plotting ----------
            fig = plt.figure(figsize=figsize)

            # Left subplot: confusion matrix heatmap
            ax1 = plt.subplot(1, 2, 1)
            im = ax1.imshow(cm_display, interpolation="nearest", cmap=cmap, aspect="auto")
            ax1.set_title(title_left, fontsize=14)
            ax1.set_xlabel("Predicted label")
            ax1.set_ylabel("True label")
            ax1.set_xticks(np.arange(len(classes)))
            ax1.set_yticks(np.arange(len(classes)))
            ax1.set_xticklabels(class_names, rotation=rotate_xticks, ha="right")
            ax1.set_yticklabels(class_names)

            # Annotate cells (counts or normalized)
            fmt = ".2f" if normalize_cm else "d"
            thresh = cm_display.max() / 2.0 if cm_display.size > 0 else 0.0
            for i in range(cm_display.shape[0]):
                for j in range(cm_display.shape[1]):
                    val = cm_display[i, j]
                    # show both normalized and counts if desired (here we show one)
                    text_color = "white" if val > thresh else "black"
                    # Fix: convert to int when using 'd' format
                    if normalize_cm:
                        ax1.text(j, i, format(val, fmt), ha="center", va="center", color=text_color, fontsize=9)
                    else:
                        ax1.text(j, i, format(int(val), fmt), ha="center", va="center", color=text_color, fontsize=9)

            cbar = fig.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)
            cbar.ax.set_ylabel("Proportion" if normalize_cm else "Count", rotation=270, labelpad=12)

            # Right subplot: classification report table
            ax2 = plt.subplot(1, 2, 2)
            ax2.axis("off")
            ax2.set_title(title_right, fontsize=14, pad=10)

            # Convert numeric columns to strings with formatting
            rep_to_show = rep_df.copy()
            rep_to_show["precision"] = rep_to_show["precision"].map(lambda x: f"{x:.2f}")
            rep_to_show["recall"]    = rep_to_show["recall"].map(lambda x: f"{x:.2f}")
            rep_to_show["f1-score"]  = rep_to_show["f1-score"].map(lambda x: f"{x:.2f}")
            rep_to_show["support"]   = rep_to_show["support"].map(lambda x: f"{int(x)}")

            table = ax2.table(
                cellText=rep_to_show.values,
                rowLabels=rep_to_show.index,
                colLabels=rep_to_show.columns,
                loc="center",
                cellLoc="center",
            )
            table.auto_set_font_size(False)
            table.set_fontsize(9)
            table.scale(1.0, 1.2)  # widen rows a bit

            plt.tight_layout()

            if save_path:
                plt.savefig(save_path, dpi=200, bbox_inches="tight")
                print(f"[Saved] {save_path}")

            plt.show()

        # Plot test results
        if y_true.size > 0 and y_pred.size > 0:
            plot_test_results(
                y_true=y_true,
                y_pred=y_pred,
                id_to_name=id_to_name,
                normalize_cm=False,
                save_path="test_eval.png"
            )
    except Exception as e:
        print(f"[Error] Failed to run test_only: {e}")
        import traceback
        traceback.print_exc()
else:
    print("[Warning] test_loader not defined or None, skipping test_only execution")


## Test: Standalone Evaluation Pipeline


In [ ]:
# -*- coding: utf-8 -*-
import torch
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import os
import pickle
from typing import Dict, Tuple, Optional

# ============================================
# Read "real class labels" and replace default Class_i labels - Complete runnable snippet
# Description:
# 1) Prioritize reading real labels from previously saved label_mappings_*.pkl (use 'label_to_id' if available)
# 2) If file not found, fallback to default Class_i
# 3) Generate id_to_name: {encoded_id -> real class name(str)}
# 4) Use real class names to override id_to_name / K in test and visualization code
# ============================================

# -------------------------
# Modify as needed: directory containing label_mappings
# Keep consistent with save_dir when saving data
# -------------------------
SAVE_DIR = str(DATA_DIR)

def _invert_name_mapping(name_to_id: Dict) -> Dict[int, str]:
    """Invert {name -> id} to {id -> name}, ensuring all names are str with capitalized first letter."""
    id_to_name = {}
    for name, idx in name_to_id.items():
        try:
            idx_int = int(idx)
        except Exception:
            continue
        # Capitalize first letter of class name
        name_str = str(name)
        if name_str:
            name_str = name_str[0].upper() + name_str[1:]
        id_to_name[idx_int] = name_str
    return id_to_name

def read_real_class_id_to_name(
    save_dir: str = SAVE_DIR,
    preferred_filename: Optional[str] = None
) -> Tuple[Optional[Dict[int, str]], Optional[int], Optional[str]]:
    """
    Read real class names from saved label_mappings_*.pkl:
      - Prioritize using 'label_to_id' (usually string class -> id)
      - Otherwise fallback to 'class_to_id' / 'target_class_to_id' / 'hclass_to_id'
    Returns:
      (id_to_name, K, used_file_path)
      Returns (None, None, None) if reading fails
    """
    if preferred_filename:
        candidates = [os.path.join(save_dir, preferred_filename)]
    else:
        # Automatically search for pkl files starting with label_mappings_sample_split_window_
        candidates = []
        if os.path.isdir(save_dir):
            for fn in os.listdir(save_dir):
                if fn.startswith("label_mappings_sample_split_window_") and fn.endswith(".pkl"):
                    candidates.append(os.path.join(save_dir, fn))
        # Select the newest
        candidates.sort(key=lambda p: os.path.getmtime(p), reverse=True)

    for path in candidates:
        try:
            with open(path, "rb") as f:
                lm = pickle.load(f)
        except Exception:
            continue

        # Prioritize using string labels
        if isinstance(lm, dict) and "label_to_id" in lm and lm["label_to_id"]:
            id_to_name = _invert_name_mapping(lm["label_to_id"])
            if id_to_name:
                K = max(id_to_name.keys()) + 1
                return id_to_name, K, path

        # Otherwise try numeric class labels (convert original numeric values to string for display)
        for key in ["class_to_id", "target_class_to_id", "hclass_to_id"]:
            if isinstance(lm, dict) and key in lm and lm[key]:
                id_to_name = _invert_name_mapping(lm[key])
                if id_to_name:
                    # Note: these mappings are {original class value -> encoded_id}, already inverted to {id -> original class value(str)}
                    K = max(id_to_name.keys()) + 1
                    return id_to_name, K, path

    return None, None, None

# ---------- 1) Infer C, K and id_to_name ----------
def _infer_C_and_K(train_loader=None, test_loader=None):
    probe_loader = train_loader if (train_loader is not None) else test_loader
    assert probe_loader is not None, "Need available train_loader or test_loader to infer dimensions."
    first_batch = next(iter(probe_loader))
    C = int(first_batch["x_before"].shape[-1])

    K = None
    if train_loader is not None:
        try:
            K = scan_num_classes_from_loader(train_loader)
        except Exception:
            K = None
    if K is None and test_loader is not None:
        try:
            K = scan_num_classes_from_loader(test_loader)
        except Exception:
            K = None
    assert K is not None and K > 0, "Cannot infer K (number of classes) from data. Please check loader y labels."
    return C, int(K)

def _infer_C(train_loader=None, test_loader=None):
    """Infer sensor dimension C from loader."""
    probe_loader = train_loader if train_loader is not None else test_loader
    assert probe_loader is not None, "Need available train_loader or test_loader to infer C."
    first_batch = next(iter(probe_loader))
    C = first_batch["x_before"].shape[-1]
    return int(C)

def build_default_id_to_name(K: int):
    return {i: f"Class_{i}" for i in range(K)}

# ========== Use real class names to replace default Class_i ==========
# 1) Still use loader to infer C (sensor dimension)
# 2) K prioritizes using real class count from reading; fallback to inferring from loader if reading fails
C = _infer_C(train_loader=train_loader, test_loader=test_loader)

# First try to read real class labels from label_mappings
real_id_to_name, real_K, used_path = read_real_class_id_to_name(SAVE_DIR)

if real_id_to_name is not None and real_K is not None:
    id_to_name = real_id_to_name
    K = int(real_K)
    print(f"[Info] Using real class names (total {K} classes), source: {used_path}")
else:
    # Fallback: infer K from loader, then assign default names
    try:
        K = scan_num_classes_from_loader(train_loader) if train_loader is not None else scan_num_classes_from_loader(test_loader)
    except Exception:
        raise RuntimeError("Cannot infer K from label_mappings or loader. Please check data/mapping files.")
    id_to_name = build_default_id_to_name(K)
    print(f"[Warn] label_mappings*.pkl not found, fallback to default class names. K={K}")

print(f"[Info] Inferred sensor dim C={C}, num classes K={K}")
# If class weights needed, write as needed
class_weights_obj = None

# ---------- 2) Load best model and run test, explicitly return y_true / y_pred ----------
def load_best_and_test(
    ckpt_path: str,
    train_loader=None,
    test_loader=None,
    K: int = None,
    id_to_name: dict = None,
    max_curve_keep: int = 0,   # Set 0 if no curve visualization needed (faster)
    device: torch.device = None
):
    """
    Returns:
      y_true: np.ndarray [N]
      y_pred: np.ndarray [N]
      id_to_name: dict[int,str]
      C: int, K: int
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Infer C, K
    C_infer, K_infer = _infer_C_and_K(train_loader=train_loader, test_loader=test_loader)
    if K is None:
        K = K_infer
    if id_to_name is None:
        id_to_name = build_default_id_to_name(K)

    # Build model and load weights
    model = DiffAwareReconstructor(in_ch=C_infer, trend_ch=4, hidden=128, n_classes=K).to(device)
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state, strict=False)
    model.eval()

    # Use official collection function to get predictions
    y_true, y_pred, all_delta_mean, all_uids, curves = collect_test_predictions(
        model=model,
        loader=test_loader,
        device=device,
        max_curve_keep=max_curve_keep,
        K=K,
        id_to_name=id_to_name
    )

    # Convert to numpy (in case they are lists)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Print a brief report (optional)
    print("============================================================")
    print("Final Test (quick summary)")
    print("============================================================")
    print(classification_report(
        y_true, y_pred,
        labels=list(range(K)),
        target_names=[id_to_name.get(i, f"Class_{i}") for i in range(K)],
        zero_division=0
    ))
    print("[Confusion Matrix]")
    print(confusion_matrix(y_true, y_pred, labels=list(range(K))))

    return y_true, y_pred, id_to_name, C_infer, K

# ---------- 3) One-click execution: load -> test -> plot ----------
# Make sure:
#  - You have train_loader / test_loader variables
#  - Change path to your best model path
ckpt_path = str(resolve_data_file("best_model_NGFAID_250_100_7.pth"))

y_true, y_pred, id_to_name, C, K = load_best_and_test(
    ckpt_path=ckpt_path,
    train_loader=train_loader,
    test_loader=test_loader,
    K=K,                    # Use K from real class labels
    id_to_name=id_to_name,  # Use id_to_name from real class labels
    max_curve_keep=0,       # Set 0 when no curve visualization needed, faster
    device=None             # Auto select GPU/CPU
)
# -*- coding: utf-8 -*-
"""
Plot confusion matrix + classification report side-by-side
from TEST RESULTS (y_true, y_pred), not from pasted text.

Requirements:
  - numpy
  - matplotlib
  - sklearn
  - pandas (for neat table formatting; can be removed if not desired)
"""

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd


def plot_test_results(
    y_true,
    y_pred,
    id_to_name=None,
    normalize_cm=False,
    cmap="Blues",
    title_left="Confusion Matrix",
    title_right="Classification Report",
    save_path=None,
    figsize=(18, 8),
    rotate_xticks=45
):
    """
    Parameters
    ----------
    y_true : array-like (n_samples,)
        Ground-truth encoded class ids (e.g., from your test set).
    y_pred : array-like (n_samples,)
        Predicted encoded class ids.
    id_to_name : dict[int,str] or list[str], optional
        Mapping from encoded id -> readable class name. If None, uses "Class_i".
    normalize_cm : bool
        If True, show row-wise normalized confusion matrix.
    cmap : str
        Matplotlib colormap name for confusion matrix.
    save_path : str or None
        If provided, save the figure to this path (e.g., "test_report.png").
    figsize : tuple
        Figure size.
    rotate_xticks : int or float
        Rotation angle for x tick labels (right subplot table uses no rotation).
    """

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # --------- determine class set & names ----------
    classes = np.unique(np.concatenate([y_true, y_pred]))
    classes = np.sort(classes)

    if id_to_name is None:
        class_names = [f"Class_{int(i)}" for i in classes]
    else:
        # id_to_name can be dict or list
        if isinstance(id_to_name, dict):
            class_names = [str(id_to_name.get(int(i), f"Class_{int(i)}")) for i in classes]
        else:
            # assume list-like indexable by class id
            class_names = [str(id_to_name[int(i)]) if int(i) < len(id_to_name) else f"Class_{int(i)}"
                           for i in classes]

    # --------- confusion matrix ----------
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    cm_display = cm.astype(float)
    if normalize_cm:
        row_sums = cm_display.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        cm_display = cm_display / row_sums

    # --------- classification report (as table) ----------
    # Use the *ordered* class names corresponding to `classes`
    report_dict = classification_report(
        y_true,
        y_pred,
        labels=classes,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    # Build a pandas DataFrame for neat table
    rows = class_names + ["accuracy", "macro avg", "weighted avg"]
    cols = ["precision", "recall", "f1-score", "support"]

    data = []
    for name in class_names:
        r = report_dict.get(name, {})
        data.append([r.get("precision", 0.0), r.get("recall", 0.0), r.get("f1-score", 0.0), int(r.get("support", 0))])

    # accuracy row
    acc = report_dict.get("accuracy", 0.0)
    data.append([acc, acc, acc, int(len(y_true))])

    # macro avg
    r = report_dict.get("macro avg", {})
    data.append([r.get("precision", 0.0), r.get("recall", 0.0), r.get("f1-score", 0.0), int(r.get("support", 0))])

    # weighted avg
    r = report_dict.get("weighted avg", {})
    data.append([r.get("precision", 0.0), r.get("recall", 0.0), r.get("f1-score", 0.0), int(r.get("support", 0))])

    rep_df = pd.DataFrame(data, index=rows, columns=cols)

    # --------- plotting ----------
    fig = plt.figure(figsize=figsize)

    # Left subplot: confusion matrix heatmap
    ax1 = plt.subplot(1, 2, 1)
    im = ax1.imshow(cm_display, interpolation="nearest", cmap=cmap, aspect="auto")
    ax1.set_title(title_left, fontsize=14)
    ax1.set_xlabel("Predicted label")
    ax1.set_ylabel("True label")
    ax1.set_xticks(np.arange(len(classes)))
    ax1.set_yticks(np.arange(len(classes)))
    ax1.set_xticklabels(class_names, rotation=rotate_xticks, ha="right")
    ax1.set_yticklabels(class_names)

    # Annotate cells (counts or normalized)
    fmt = ".2f" if normalize_cm else "d"
    thresh = cm_display.max() / 2.0 if cm_display.size > 0 else 0.0
    for i in range(cm_display.shape[0]):
        for j in range(cm_display.shape[1]):
            val = cm_display[i, j]
            # show both normalized and counts if desired (here we show one)
            text_color = "white" if val > thresh else "black"
            # Fix: convert to int when using 'd' format
            if normalize_cm:
                ax1.text(j, i, format(val, fmt), ha="center", va="center", color=text_color, fontsize=9)
            else:
                ax1.text(j, i, format(int(val), fmt), ha="center", va="center", color=text_color, fontsize=9)

    cbar = fig.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)
    cbar.ax.set_ylabel("Proportion" if normalize_cm else "Count", rotation=270, labelpad=12)

    # Right subplot: classification report table
    ax2 = plt.subplot(1, 2, 2)
    ax2.axis("off")
    ax2.set_title(title_right, fontsize=14, pad=10)

    # Convert numeric columns to strings with formatting
    rep_to_show = rep_df.copy()
    rep_to_show["precision"] = rep_to_show["precision"].map(lambda x: f"{x:.2f}")
    rep_to_show["recall"]    = rep_to_show["recall"].map(lambda x: f"{x:.2f}")
    rep_to_show["f1-score"]  = rep_to_show["f1-score"].map(lambda x: f"{x:.2f}")
    rep_to_show["support"]   = rep_to_show["support"].map(lambda x: f"{int(x)}")

    table = ax2.table(
        cellText=rep_to_show.values,
        rowLabels=rep_to_show.index,
        colLabels=rep_to_show.columns,
        loc="center",
        cellLoc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.0, 1.2)  # widen rows a bit

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        print(f"[Saved] {save_path}")

    plt.show()


# ============================
# Example usage (pseudo-code):
# ============================
# Assume you already ran your test step and got arrays:
#   y_true, y_pred = collect_from_test(...)
# and you have 'id_to_name' from label_mappings (id -> readable name).
#
plot_test_results(
    y_true=y_true,
    y_pred=y_pred,
    id_to_name=id_to_name,    # or None to use "Class_i"
    normalize_cm=False,       # set True for row-normalized CM
    save_path="test_eval.png" # optional
)


## Discussion: All-Class Operator Weight Heatmaps


In [ ]:

# =============== (Optional) Plot Operator Weight Heatmaps for All Classes: Keep Real Class Names ================
import math, random
import numpy as np
import torch
import matplotlib.pyplot as plt

def plot_all_classes_operator_weight_heatmaps(
    checkpoint_path: str,
    test_loader,
    id_to_name: Optional[Dict[int, str]] = None,
    device: torch.device = None,
    max_samples_per_class: int = 6,
    seed: int = 0,
    max_curve_keep: int = 2000,
    fallback_to_pred_if_no_labels: bool = True,
):
    """
    Load the best model and plot operator weight heatmaps (pre | post) for samples of all classes.
    If id_to_name is provided, it will be used as the real class mapping (recommended).
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Try to read K/C/id_to_name from meta; if id_to_name is provided externally, use it
    meta_path = checkpoint_path.replace(".pth", "_meta.json")
    K_meta = None
    C_meta = None
    id_to_name_meta = None
    if os.path.isfile(meta_path):
        with open(meta_path, "r") as f:
            meta = json.load(f)
        K_meta = int(meta.get("K", 0) or 0)
        C_meta = int(meta.get("sensor_dim", 0) or 0)
        itm = meta.get("id_to_name", None)
        if isinstance(itm, dict):
            id_to_name_meta = {int(k): str(v) for k, v in itm.items()}

    # Final id_to_name / K to use
    if id_to_name is None:
        # If not provided, use meta's; if meta doesn't have it, use real mapping reading function; fallback to default
        id_to_name, K_from_labels, _ = read_real_class_id_to_name(SAVE_DIR)
        if id_to_name is None:
            id_to_name = id_to_name_meta if id_to_name_meta is not None else build_default_id_to_name(K_meta or scan_num_classes_from_loader(test_loader))
    K_final = len(id_to_name)

    # Rebuild model and load parameters
    # Create a temporary loader with num_workers=0 to avoid DataLoader worker issues
    temp_loader = torch.utils.data.DataLoader(
        test_loader.dataset,
        batch_size=1,
        shuffle=False,
        num_workers=0
    )
    first_batch = next(iter(temp_loader))
    C_loader = int(first_batch["x_before"].shape[-1])
    C_use = C_meta or C_loader
    model = DiffAwareReconstructor(in_ch=C_use, trend_ch=4, hidden=128, n_classes=K_final).to(device)
    state = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state, strict=True)
    model.eval()

    # Collect curves - use a safe loader with num_workers=0
    safe_loader = torch.utils.data.DataLoader(
        test_loader.dataset,
        batch_size=test_loader.batch_size if hasattr(test_loader, 'batch_size') else 32,
        shuffle=False,
        num_workers=0
    )

    from sklearn.metrics import classification_report, confusion_matrix  # In case not imported
    y_true, y_pred, _, _, curves = collect_test_predictions(
        model=model,
        loader=safe_loader,
        device=device,
        max_curve_keep=max_curve_keep,
        K=K_final,
        id_to_name=id_to_name
    )

    # Filter samples by class
    def valid_label(v):
        return (v is not None) and (v >= 0)

    # Collect samples for each class
    class_samples = {i: [] for i in range(K_final)}

    for ex in curves:
        w_b = ex.get("weights_before", None)
        w_a = ex.get("weights_after", None)
        if w_b is None or w_a is None:
            continue
        t = ex.get("true", -1)
        p = ex.get("pred", -1)

        if valid_label(t):
            class_samples[t].append(ex)
        elif (not valid_label(t)) and fallback_to_pred_if_no_labels and valid_label(p):
            class_samples[p].append(ex)

    # Plotting parameters
    op_names = ["MonotonicLinear", "MonotonicFlat", "ConcaveLog", "SaturationSigmoid",
                "HingeReLU", "Polynomial", "DampedSin", "PiecewiseLinear"]

    # Plot for each class
    for target_id in range(K_final):
        filtered = class_samples[target_id]

        if len(filtered) == 0:
            print(f"[Info] Class '{id_to_name.get(target_id, str(target_id))}' has no visualization samples.")
            continue

        random.Random(seed).shuffle(filtered)
        n_show = min(max_samples_per_class, len(filtered))
        cols = 3
        rows = math.ceil(n_show / cols)

        fig_w = 5.4 * cols
        fig_h = 3.8 * rows
        fig = plt.figure(figsize=(fig_w, fig_h))

        for idx in range(n_show):
            ex = filtered[idx]
            w_b = np.asarray(ex["weights_before"])  # (T_b, K)
            w_a = np.asarray(ex["weights_after"])   # (T_a, K)

            if w_b.ndim != 2 or w_a.ndim != 2:
                continue

            T_b, K_b = w_b.shape
            T_a, K_a = w_a.shape
            K_eff = min(K_b, K_a, len(op_names))
            w_b = w_b[:, :K_eff]
            w_a = w_a[:, :K_eff]
            combined = np.concatenate([w_b, w_a], axis=0)  # (T_b+T_a, K_eff)

            ax = plt.subplot(rows, cols, idx + 1)
            im = ax.imshow(combined.T, aspect="auto", interpolation="nearest")

            # Draw a clear dividing line between pre and post
            ax.axvline(T_b - 0.5, color='red', linestyle='-', linewidth=2.5)

            ax.set_yticks(range(K_eff))
            ax.set_yticklabels([op_names[i] for i in range(K_eff)], fontsize=8)

            # Custom x-axis: pre shows 0 to T_b-1, post shows 0 to T_a-1
            tick_positions = []
            tick_labels = []

            # Pre section ticks
            pre_step = max(1, T_b // 5)
            for i in range(0, T_b, pre_step):
                tick_positions.append(i)
                tick_labels.append(str(i))

            # Post section ticks (display from 0, but position after T_b)
            post_step = max(1, T_a // 5)
            for i in range(0, T_a, post_step):
                tick_positions.append(T_b + i)
                tick_labels.append(str(i))

            ax.set_xticks(tick_positions)
            ax.set_xticklabels(tick_labels, fontsize=8)
            ax.set_xlabel("Time (pre  |  post)", fontsize=9)

            cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            cbar.ax.set_ylabel("Weight", rotation=270, labelpad=10)

        plt.suptitle(
            f"Operator Weight Heatmaps - Class: {id_to_name.get(target_id, f'Class_{target_id}')}",
            fontsize=12, y=0.995
        )
        plt.tight_layout(rect=[0, 0, 1, 0.97])
        plt.show()


# ============ Call visualization with real class names (example) ============
# Plot all classes
plot_all_classes_operator_weight_heatmaps(
    checkpoint_path=str(resolve_data_file("best_model_NGFAID_250_100_7.pth")),
    test_loader=test_loader,
    id_to_name=id_to_name,          # Pass in real class name mapping
    device=None,
    max_samples_per_class=6,
    seed=0
)


## Discussion: Degradation-Mode Importance Analysis


In [ ]:
# ============================================================
# BoltzmannKAN Importance Analysis (by degradation mode & channel) + Rule Consistency Check
# Paste this code at the bottom of your existing script or in an appropriate location to run
# Dependencies: DiffAwareReconstructor, adapt_batch, SENSOR_NAMES, id_to_name, K, test_loader
# ============================================================
import torch
import numpy as np
import pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt
import re

# ----------------------------
# 1) Load the trained best model weights (as used in your test_only checkpoint)
# ----------------------------
def load_model_for_analysis(ckpt_path, C, K, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = DiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=K).to(device)
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state, strict=False)
    model.eval()
    # Freeze parameters
    for p in model.parameters():
        p.requires_grad = False
    return model, device

# ----------------------------
# 2) Extract BoltzmannKAN's E and kT (effective temperature after softplus->clamp)
# ----------------------------
def get_boltz_params(model):
    boltz = model.encoder.boltz
    # Original parameters
    E_raw  = boltz.E.data.clone()   # (trend_ch, C)
    kT_raw = boltz.kT.data.clone()  # (trend_ch, C)
    # Effective range during forward: kT=softplus(kT_raw) clamped to [0.01, 10.0]
    kT_eff = torch.clamp(torch.nn.functional.softplus(kT_raw), 0.01, 10.0)
    # E is also clamped to [-10, 10] during forward, but threshold meaning remains
    E_eff = torch.clamp(E_raw, -10.0, 10.0)
    return E_eff.cpu().numpy(), kT_eff.cpu().numpy()  # Shape: (trend_ch, C)

# ----------------------------
# 3) Data-driven: Sensor importance based on derivatives (by class, separately for before and after)
# ----------------------------
def compute_data_driven_importance_separate(model, loader, device, K):
    """
    Returns two DataFrames: average sensitivity importance for each class-sensor pair.
    One for 'before' data, one for 'after' data - to reflect different sensor importance.
    Importance scores are normalized across all channels (sum to 1.0 for each class).
    """
    boltz = model.encoder.boltz
    trend_ch = boltz.E.shape[0]

    # Aggregate: each class -> sensor -> [cumulative importance, sample count] for before/after separately
    agg_imp_before = defaultdict(lambda: {"sum": np.zeros(boltz.E.shape[1], dtype=np.float64),
                                          "cnt": 0})
    agg_imp_after = defaultdict(lambda: {"sum": np.zeros(boltz.E.shape[1], dtype=np.float64),
                                         "cnt": 0})

    sigmoid = torch.sigmoid

    # Create a safe loader with num_workers=0 to avoid worker process issues
    safe_loader = torch.utils.data.DataLoader(
        loader.dataset,
        batch_size=loader.batch_size if hasattr(loader, 'batch_size') else 32,
        shuffle=False,
        num_workers=0,
        persistent_workers=False
    )

    for batch in safe_loader:
        xb, xa, labels, mask, lengths, sample_ids, _ = adapt_batch(batch, device)

        # Need labels for per-class statistics; skip if no labels
        if labels is None:
            continue
        valid_labels = (labels != -100)
        if not valid_labels.any():
            continue

        # Take valid samples
        xb_valid = xb[valid_labels]
        xa_valid = xa[valid_labels]
        y = labels[valid_labels]
        m = mask[valid_labels]

        if xb_valid.numel() == 0:
            continue

        # Process before and after separately
        with torch.no_grad():
            E = torch.clamp(boltz.E, -10.0, 10.0)
            kT = torch.clamp(torch.nn.functional.softplus(boltz.kT), 0.01, 10.0)

            # Process 'before' data
            x = xb_valid
            B, T, C = x.shape
            x_ = torch.clamp(x, -10.0, 10.0).unsqueeze(1)
            E_ = E.unsqueeze(0).unsqueeze(2)
            kT_= kT.unsqueeze(0).unsqueeze(2)
            z  = torch.clamp((E_ - x_) / kT_, -50, 50)
            w  = sigmoid(z)

            sigp = w * (1 - w)
            ds_dx = sigp * (-1.0/kT_) * x_ + w
            ds_dx_ch_mean = ds_dx.mean(dim=1)

            m_exp = m.unsqueeze(-1)
            denom = m_exp.sum(dim=(0,1)).clamp_min(1.0)
            imp_bt = (ds_dx_ch_mean.abs() * m_exp).sum(dim=(0,1)) / denom

            # Accumulate by class for 'before'
            for cls_id in y.detach().cpu().numpy().tolist():
                agg_imp_before[int(cls_id)]["sum"] += imp_bt.detach().cpu().numpy()
                agg_imp_before[int(cls_id)]["cnt"] += 1

            # Process 'after' data
            x = xa_valid
            B, T, C = x.shape
            x_ = torch.clamp(x, -10.0, 10.0).unsqueeze(1)
            E_ = E.unsqueeze(0).unsqueeze(2)
            kT_= kT.unsqueeze(0).unsqueeze(2)
            z  = torch.clamp((E_ - x_) / kT_, -50, 50)
            w  = sigmoid(z)

            sigp = w * (1 - w)
            ds_dx = sigp * (-1.0/kT_) * x_ + w
            ds_dx_ch_mean = ds_dx.mean(dim=1)

            m_exp = m.unsqueeze(-1)
            denom = m_exp.sum(dim=(0,1)).clamp_min(1.0)
            imp_bt = (ds_dx_ch_mean.abs() * m_exp).sum(dim=(0,1)) / denom

            # Accumulate by class for 'after'
            for cls_id in y.detach().cpu().numpy().tolist():
                agg_imp_after[int(cls_id)]["sum"] += imp_bt.detach().cpu().numpy()
                agg_imp_after[int(cls_id)]["cnt"] += 1

    # Compile into DataFrames with normalization
    records_before = []
    for cls_id, v in agg_imp_before.items():
        if v["cnt"] == 0:
            continue
        avg_imp = v["sum"] / max(v["cnt"], 1)
        # Normalize: make importance sum to 1.0 across all channels for this class
        total_imp = avg_imp.sum()
        if total_imp > 0:
            avg_imp_normalized = avg_imp / total_imp
        else:
            avg_imp_normalized = avg_imp
        for c_idx, score in enumerate(avg_imp_normalized):
            records_before.append({
                "class_id": cls_id,
                "sensor_id": c_idx,
                "sensor_name": SENSOR_NAMES.get(c_idx, f"Sensor_{c_idx}"),
                "importance_before": float(score)
            })

    records_after = []
    for cls_id, v in agg_imp_after.items():
        if v["cnt"] == 0:
            continue
        avg_imp = v["sum"] / max(v["cnt"], 1)
        # Normalize: make importance sum to 1.0 across all channels for this class
        total_imp = avg_imp.sum()
        if total_imp > 0:
            avg_imp_normalized = avg_imp / total_imp
        else:
            avg_imp_normalized = avg_imp
        for c_idx, score in enumerate(avg_imp_normalized):
            records_after.append({
                "class_id": cls_id,
                "sensor_id": c_idx,
                "sensor_name": SENSOR_NAMES.get(c_idx, f"Sensor_{c_idx}"),
                "importance_after": float(score)
            })

    df_before = pd.DataFrame(records_before)
    df_after = pd.DataFrame(records_after)
    return df_before, df_after

# ----------------------------
# 4) Static parameter importance (based on 1/kT and E-coverage)
# ----------------------------
def compute_static_importance(E_eff, kT_eff, loader, device):
    """
    Returns DataFrame: static importance for each sensor (mean of 1/kT and E's match to data distribution).
    Importance scores are normalized across all channels (sum to 1.0).
    """
    trend_ch, C = E_eff.shape
    # Estimate value range/distribution for each sensor (overall, not per-class)
    xs = []

    # Create a safe loader with num_workers=0
    safe_loader = torch.utils.data.DataLoader(
        loader.dataset,
        batch_size=loader.batch_size if hasattr(loader, 'batch_size') else 32,
        shuffle=False,
        num_workers=0,
        persistent_workers=False
    )

    with torch.no_grad():
        for batch in safe_loader:
            xb, xa, labels, mask, lengths, sample_ids, _ = adapt_batch(batch, device)
            xs.append(torch.cat([xb, xa], dim=1).detach().cpu())  # Concatenate before/after
    if len(xs) == 0:
        return None
    X = torch.cat(xs, dim=0).numpy()   # (N, Ttot, C)

    # Estimate coverage interval using quantiles
    q_low  = np.nanpercentile(X, 5,  axis=(0,1))
    q_high = np.nanpercentile(X, 95, axis=(0,1))

    # Higher 1/kT means "sharper" response
    inv_kT_mean = (1.0 / kT_eff).mean(axis=0)  # (C,)

    # Does E fall within the main data distribution; calculate "coverage score"
    cover = []
    for c in range(C):
        E_c = np.clip(E_eff[:, c], q_low[c]-5.0, q_high[c]+5.0)  # Approach sample interval
        # Count how many trend_ch E values fall within [5%, 95%] quantile interval
        hit = np.logical_and(E_c >= q_low[c], E_c <= q_high[c]).mean()
        cover.append(hit)  # [0,1]
    cover = np.asarray(cover)

    # Calculate static importance
    static_imp = inv_kT_mean * (0.5 + 0.5*cover)

    # Normalize: make importance sum to 1.0 across all channels
    total_static_imp = static_imp.sum()
    if total_static_imp > 0:
        static_imp_normalized = static_imp / total_static_imp
    else:
        static_imp_normalized = static_imp

    df = pd.DataFrame({
        "sensor_id": np.arange(C, dtype=int),
        "sensor_name": [SENSOR_NAMES.get(i, f"Sensor_{i}") for i in range(C)],
        "inv_kT_mean": inv_kT_mean,
        "E_coverage_ratio": cover,
        "static_importance": static_imp_normalized
    }).sort_values("static_importance", ascending=False)
    return df

# ----------------------------
# 5) Select top 6 sensors per class (separately for before and after)
# ----------------------------
def select_top_sensors_per_class_separate(df_imp_before, df_imp_after, id_to_name, topk=6):
    """
    For each class, select top 6 sensors based on before and after importance separately.
    Returns DataFrame with class_id, class_name, and top sensor recommendations for before/after.
    """
    rows = []
    all_classes = set(df_imp_before["class_id"].unique()) | set(df_imp_after["class_id"].unique())

    for cls in sorted(all_classes):
        nm = id_to_name.get(int(cls), f"Class_{cls}")

        # Before sensors
        sub_before = df_imp_before[df_imp_before["class_id"]==cls].sort_values("importance_before", ascending=False).head(topk)
        top_ids_before = sub_before["sensor_id"].tolist()
        top_names_before = sub_before["sensor_name"].tolist()
        top_scores_before = sub_before["importance_before"].tolist()

        # After sensors
        sub_after = df_imp_after[df_imp_after["class_id"]==cls].sort_values("importance_after", ascending=False).head(topk)
        top_ids_after = sub_after["sensor_id"].tolist()
        top_names_after = sub_after["sensor_name"].tolist()
        top_scores_after = sub_after["importance_after"].tolist()

        rows.append({
            "class_id": int(cls),
            "class_name": nm,
            "top_sensor_ids_before": top_ids_before,
            "top_sensor_names_before": top_names_before,
            "importance_scores_before": top_scores_before,
            "top_sensor_ids_after": top_ids_after,
            "top_sensor_names_after": top_names_after,
            "importance_scores_after": top_scores_after
        })
    return pd.DataFrame(rows)

# ----------------------------
# 6) Create comprehensive sensor significance table for all classes (before and after separately)
# ----------------------------
def create_sensor_significance_table_separate(df_imp_before, df_imp_after, id_to_name, topk=6):
    """
    Create comprehensive tables showing sensor significance for all classes.
    Returns two formatted DataFrames: one for before, one for after.
    """
    all_classes = set(df_imp_before["class_id"].unique()) | set(df_imp_after["class_id"].unique())

    # Before table
    before_data = []
    for cls in sorted(all_classes):
        cls_name = id_to_name.get(int(cls), f"Class_{cls}")
        sub = df_imp_before[df_imp_before["class_id"]==cls].sort_values("importance_before", ascending=False).head(topk)

        row_data = {"Class": cls_name}
        for rank, (idx, row) in enumerate(sub.iterrows(), 1):
            sensor_col = f"Sensor_{rank}"
            score_col = f"Score_{rank}"
            row_data[sensor_col] = row["sensor_name"]
            row_data[score_col] = f"{row['importance_before']:.6f}"
        before_data.append(row_data)

    # After table
    after_data = []
    for cls in sorted(all_classes):
        cls_name = id_to_name.get(int(cls), f"Class_{cls}")
        sub = df_imp_after[df_imp_after["class_id"]==cls].sort_values("importance_after", ascending=False).head(topk)

        row_data = {"Class": cls_name}
        for rank, (idx, row) in enumerate(sub.iterrows(), 1):
            sensor_col = f"Sensor_{rank}"
            score_col = f"Score_{rank}"
            row_data[sensor_col] = row["sensor_name"]
            row_data[score_col] = f"{row['importance_after']:.6f}"
        after_data.append(row_data)

    return pd.DataFrame(before_data), pd.DataFrame(after_data)

# ============================================================
# ▶ Main workflow: Run analysis
# ============================================================
# Get C, K, device, and your best checkpoint path (same as used in testing)
# Create a temporary loader with num_workers=0 to get the first batch safely
temp_loader = torch.utils.data.DataLoader(
    test_loader.dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    persistent_workers=False
)
_first_batch = next(iter(temp_loader))
C_an = _first_batch["x_before"].shape[-1]
ckpt_for_analysis = str(resolve_data_file("best_model_NGFAID_250_100_7.pth"))

model_an, device_an = load_model_for_analysis(ckpt_for_analysis, C=C_an, K=K, device=None)
E_eff, kT_eff = get_boltz_params(model_an)

# --- Static importance ---
df_static = compute_static_importance(E_eff, kT_eff, test_loader, device_an)
print("\n[Static importance by Boltzmann parameters (normalized)]")
print(df_static.head(20))
print(f"Sum of static importance: {df_static['static_importance'].sum():.6f}")

# --- Data-driven importance (separately for before and after, by class) ---
df_imp_before, df_imp_after = compute_data_driven_importance_separate(model_an, test_loader, device_an, K)

print("\n[Per-class sensor importance (data-driven, BEFORE, normalized) — head]")
print(df_imp_before.head())

print("\n[Per-class sensor importance (data-driven, AFTER, normalized) — head]")
print(df_imp_after.head())

# Verify normalization for each class
print("\n[Verification: Sum of importance per class]")
for cls_id in sorted(df_imp_before["class_id"].unique()):
    sum_before = df_imp_before[df_imp_before["class_id"]==cls_id]["importance_before"].sum()
    sum_after = df_imp_after[df_imp_after["class_id"]==cls_id]["importance_after"].sum()
    print(f"Class {cls_id}: Before sum = {sum_before:.6f}, After sum = {sum_after:.6f}")

# --- Select top 6 sensors per class (separately for before and after) ---
df_top_sensors = select_top_sensors_per_class_separate(df_imp_before, df_imp_after, id_to_name, topk=12)

print("\n" + "="*80)
print("Top 6 Sensors for Each Class (Separately for BEFORE and AFTER, Normalized)")
print("="*80)
for idx, row in df_top_sensors.iterrows():
    print(f"\nClass: {row['class_name']}")
    print("-" * 60)
    print("  BEFORE fault:")
    for i, (sensor_name, score) in enumerate(zip(row['top_sensor_names_before'], row['importance_scores_before']), 1):
        print(f"    {i}. {sensor_name:30s} - importance: {score:.6f} ({score*100:.2f}%)")
    print("  AFTER fault:")
    for i, (sensor_name, score) in enumerate(zip(row['top_sensor_names_after'], row['importance_scores_after']), 1):
        print(f"    {i}. {sensor_name:30s} - importance: {score:.6f} ({score*100:.2f}%)")

# --- Create comprehensive sensor significance tables ---
df_sig_before, df_sig_after = create_sensor_significance_table_separate(df_imp_before, df_imp_after, id_to_name, topk=12)

print("\n" + "="*80)
print("Sensor Significance Table for All Classes - BEFORE Fault (Normalized)")
print("="*80)
print(df_sig_before.to_string(index=False))

print("\n" + "="*80)
print("Sensor Significance Table for All Classes - AFTER Fault (Normalized)")
print("="*80)
print(df_sig_after.to_string(index=False))

# --- Optional: Export results to CSV
df_top_sensors.to_csv("top_sensors_per_class_separate.csv", index=False)
df_sig_before.to_csv("sensor_significance_before.csv", index=False)
df_sig_after.to_csv("sensor_significance_after.csv", index=False)
print("\n[Info] Results exported to CSV files:")
print("  - top_sensors_per_class_separate.csv")
print("  - sensor_significance_before.csv")
print("  - sensor_significance_after.csv")


## Discussion: Flight-Level Data Pipeline Review


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, RobustScaler
from scipy.optimize import linear_sum_assignment
import warnings
warnings.filterwarnings('ignore')
import pickle

# ------------------------------
# Set random seed for reproducibility
# ------------------------------
def set_random_seeds(seed=42):
	"""Set all random seeds for reproducibility"""
	random.seed(seed)
	np.random.seed(seed)
	torch.manual_seed(seed)
	if torch.cuda.is_available():
		torch.cuda.manual_seed(seed)
		torch.cuda.manual_seed_all(seed)
	torch.backends.cudnn.deterministic = True
	torch.backends.cudnn.benchmark = False
	os.environ['PYTHONHASHSEED'] = str(seed)

set_random_seeds(42)

# ------------------------------
# Load processed data
# ------------------------------
save_dir = str(DATA_DIR)
window_size = 250
step_size = 100

# Load the processed datasets
with open(os.path.join(save_dir, f"train_pairs_sample_split_window_{window_size}_step_{step_size}.pkl"), 'rb') as f:
	pairs_train = pickle.load(f)

with open(os.path.join(save_dir, f"val_pairs_sample_split_window_{window_size}_step_{step_size}.pkl"), 'rb') as f:
	pairs_val = pickle.load(f)

with open(os.path.join(save_dir, f"test_pairs_sample_split_window_{window_size}_step_{step_size}.pkl"), 'rb') as f:
	pairs_test = pickle.load(f)

with open(os.path.join(save_dir, f"label_mappings_sample_split_window_{window_size}_step_{step_size}.pkl"), 'rb') as f:
	label_mappings = pickle.load(f)

print("Data loaded successfully!")
print(f"Training samples: {len(pairs_train)}")
print(f"Validation samples: {len(pairs_val)}")
print(f"Test samples: {len(pairs_test)}")

# ------------------------------
# Function to create concatenated sequences with anomaly insertion
# ------------------------------
def create_sequences_with_anomalies(dataset, label_mappings, n_after_samples=45, n_sequences_per_class=4, anomaly_window_size=20):
	"""
	Extract top 3 classes by sample count, then for each class:
	1. Concatenate n_after_samples 'after maintenance' samples
	2. In the last anomaly_window_size samples, randomly insert 2 consecutive 'before maintenance' samples
	3. Create n_sequences_per_class such sequences with replacement sampling

	Args:
		dataset: Dictionary of samples
		label_mappings: Label mapping dictionary
		n_after_samples: Number of after-maintenance samples to concatenate (default: 45)
		n_sequences_per_class: Number of sequences to create per class (default: 4)
		anomaly_window_size: Window size for anomaly insertion at the end (default: 20)

	Returns:
		sequences: Dictionary of created sequences with metadata
	"""
	print("\n" + "="*80)
	print("CREATING CONCATENATED SEQUENCES WITH ANOMALY INSERTION")
	print("="*80)

	# Determine which label field to use (priority: class_encoded > label_encoded)
	label_field = None
	if any('class_encoded' in sample for sample in dataset.values()):
		label_field = 'class_encoded'
	elif any('label_encoded' in sample for sample in dataset.values()):
		label_field = 'label_encoded'

	if label_field is None:
		raise ValueError("No encoded label field found in dataset!")

	print(f"Using label field: {label_field}")

	# Count samples per class
	class_counts = {}
	for sample in dataset.values():
		if label_field in sample:
			class_id = sample[label_field]
			class_counts[class_id] = class_counts.get(class_id, 0) + 1

	# Get top 3 classes by sample count
	sorted_classes = sorted(class_counts.items(), key=lambda x: x[1], reverse=True)
	top_3_classes = [cls_id for cls_id, count in sorted_classes[:3]]

	print(f"\nTop 3 classes by sample count:")
	for i, cls_id in enumerate(top_3_classes, 1):
		print(f"  {i}. Class {cls_id}: {class_counts[cls_id]} samples")

	# Organize samples by class (separate before and after)
	class_samples = {cls_id: {'before': [], 'after': []} for cls_id in top_3_classes}

	for sample_id, sample in dataset.items():
		if label_field not in sample:
			continue

		class_id = sample[label_field]
		if class_id not in top_3_classes:
			continue

		# Store both before and after samples with their IDs
		class_samples[class_id]['before'].append({
			'sample_id': sample_id,
			'data': sample['x_before'],
			'sample': sample
		})
		class_samples[class_id]['after'].append({
			'sample_id': sample_id,
			'data': sample['x_after'],
			'sample': sample
		})

	print(f"\nSamples organized by class:")
	for cls_id in top_3_classes:
		print(f"  Class {cls_id}:")
		print(f"    Before-maintenance samples: {len(class_samples[cls_id]['before'])}")
		print(f"    After-maintenance samples: {len(class_samples[cls_id]['after'])}")

	# Create sequences for each class
	sequences = {}
	sequence_id = 0

	for cls_id in top_3_classes:
		before_samples = class_samples[cls_id]['before']
		after_samples = class_samples[cls_id]['after']

		if len(after_samples) < n_after_samples:
			print(f"\nWarning: Class {cls_id} has only {len(after_samples)} after-maintenance samples, need {n_after_samples}")
			print(f"  Will use sampling with replacement")

		if len(before_samples) < 2:
			print(f"\nWarning: Class {cls_id} has only {len(before_samples)} before-maintenance samples, need at least 2")
			print(f"  Skipping this class")
			continue

		print(f"\nCreating {n_sequences_per_class} sequences for class {cls_id}...")

		for seq_idx in range(n_sequences_per_class):
			# Sample after-maintenance samples with replacement
			sampled_after = random.choices(after_samples, k=n_after_samples)

			# Concatenate the after-maintenance samples
			concatenated_samples = [s['data'] for s in sampled_after]

			# Determine insertion position in the last anomaly_window_size samples
			# Position is within [n_after_samples - anomaly_window_size, n_after_samples - 2]
			# to ensure we have room for 2 consecutive samples
			insert_start = n_after_samples - anomaly_window_size
			insert_end = n_after_samples - 2

			if insert_start < 0:
				insert_start = 0

			insertion_pos = random.randint(insert_start, insert_end)

			# Sample 2 consecutive before-maintenance samples
			# We need to ensure we can get 2 consecutive samples from the same pair
			# For simplicity, we'll just sample 2 before samples (not necessarily from same pair)
			sampled_before = random.sample(before_samples, k=2)

			# Insert the 2 before-maintenance samples at the insertion position
			concatenated_samples[insertion_pos] = sampled_before[0]['data']
			concatenated_samples[insertion_pos + 1] = sampled_before[1]['data']

			# Stack all samples into a single sequence
			# Shape: (n_after_samples, window_size, n_features)
			sequence_data = np.stack(concatenated_samples, axis=0)

			# Create sequence metadata
			sequence_info = {
				'class_id': cls_id,
				'sequence_id': sequence_id,
				'sequence_index': seq_idx,
				'n_samples': n_after_samples,
				'data': sequence_data,  # Shape: (n_after_samples, window_size, n_features)
				'anomaly_positions': [insertion_pos, insertion_pos + 1],
				'anomaly_window_start': insert_start,
				'sample_ids_after': [s['sample_id'] for s in sampled_after],
				'sample_ids_before_inserted': [s['sample_id'] for s in sampled_before],
				'label_field': label_field
			}

			sequences[sequence_id] = sequence_info
			sequence_id += 1

			if (seq_idx + 1) % 2 == 0 or seq_idx == n_sequences_per_class - 1:
				print(f"  Created sequence {seq_idx + 1}/{n_sequences_per_class}: "
				      f"anomaly inserted at positions {insertion_pos}-{insertion_pos+1}")

	print(f"\n" + "="*80)
	print(f"Sequence creation completed!")
	print(f"  Total sequences created: {len(sequences)}")
	print(f"  Sequences per class: {n_sequences_per_class}")
	print(f"  Samples per sequence: {n_after_samples}")
	print(f"  Anomaly window size: {anomaly_window_size}")
	print("="*80)

	return sequences

# ------------------------------
# Create sequences from training data
# ------------------------------
sequences = create_sequences_with_anomalies(
	dataset=pairs_train,
	label_mappings=label_mappings,
	n_after_samples=45,
	n_sequences_per_class=4,
	anomaly_window_size=20
)

# ------------------------------
# Display sequence information
# ------------------------------
print("\n" + "="*80)
print("SEQUENCE ANALYSIS")
print("="*80)

# Analyze sequences by class
class_sequence_counts = {}
for seq_id, seq_info in sequences.items():
	cls_id = seq_info['class_id']
	class_sequence_counts[cls_id] = class_sequence_counts.get(cls_id, 0) + 1

print("\nSequences per class:")
for cls_id, count in sorted(class_sequence_counts.items()):
	print(f"  Class {cls_id}: {count} sequences")

# Display sample sequence details
if len(sequences) > 0:
	sample_seq_id = list(sequences.keys())[0]
	sample_seq = sequences[sample_seq_id]

	print(f"\nSample sequence information (sequence ID: {sample_seq_id}):")
	print(f"  Class ID: {sample_seq['class_id']}")
	print(f"  Sequence index: {sample_seq['sequence_index']}")
	print(f"  Data shape: {sample_seq['data'].shape}")
	print(f"  Number of samples: {sample_seq['n_samples']}")
	print(f"  Anomaly positions: {sample_seq['anomaly_positions']}")
	print(f"  Anomaly window starts at: {sample_seq['anomaly_window_start']}")
	print(f"  Label field used: {sample_seq['label_field']}")

# ------------------------------
# Save sequences
# ------------------------------
output_filename = f"sequences_with_anomalies_window_{window_size}_step_{step_size}.pkl"
save_data(sequences, output_filename, save_dir)

print(f"\nSequences saved successfully to: {os.path.join(save_dir, output_filename)}")
print("\nAll processing completed!")


### Prototype: Long-Horizon Reliability Analysis


In [ ]:
# -*- coding: utf-8 -*-
"""
Long-horizon synthetic trajectory generation and reliability analysis
ΔSM + graded conjunction rule + refractory/NMS + EMA smoothing + RS z-score logistic calibration (monotonic)
"""

import os
import json
import pickle
import random
from typing import Dict, Any, List, Tuple, Optional
from collections import Counter

import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.linear_model import LogisticRegression

# ==============================
#            CONFIG
# ==============================
SAVE_DIR = str(DATA_DIR)
WINDOW_SIZE = 250
STEP_SIZE = 100

# Multi-seed evaluation
RANDOM_SEEDS = [7, 11, 19, 23, 29]
TOP_K = 3

# Target total trajectory length (~3 months ≈ 90 days → 45 segments of 2 days each)
TARGET_DAYS = 90
DAYS_PER_SEGMENT = 2.0
TARGET_SEGMENTS = int(TARGET_DAYS / DAYS_PER_SEGMENT)

# Markov chain parameters
P_ACTION = 0.30   # pre -> post
P_RELAPSE = 0.10  # post -> pre
P_RUN_LENGTH = 0.20  # geometric run length

# Window sizing for metrics (converted to sample counts for each seed)
STRIDE_FRACTION = 0.25      # stride = 1/4 day
METRIC_WINDOW_DAYS = 2.0    # window for PME/SM

# Detection parameters(tightened to reduce false positives)
USE_CONJ_RULE = True
DETECTION_TOLERANCE_DAYS = 1.0
STABILIZATION_WINDOW_DAYS = 2.0
REFRACTORY_DAYS = 1.5         # Extended refractory period: 1.5 days
NMS_NEIGHBOR = 0.5            # Wider NMS window: 0.5 day

# Threshold percentiles (more conservative: raise the thresholds)
PME_P_LOW = 60        # p40 -> p60
DSM_P_LOW = 60        # p40 -> p60
RS_P_MID = 75         # p60 -> p75
RS_P_HIGH = 85        # p70 -> p85
MIN_POS_FRAC = 0.05   # Fall back when the fraction of positive values is too low

# Plotting
DO_PLOTS = True
SAVE_FIGS = True
FIGS_DIR = os.path.join(SAVE_DIR, "reliability_figs")
os.makedirs(FIGS_DIR, exist_ok=True)

# ==============================
#     HELPERS: IO & LABELS
# ==============================
def _load_pickle(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)

def _fname(stem: str) -> str:
    return os.path.join(SAVE_DIR, f"{stem}_window_{WINDOW_SIZE}_step_{STEP_SIZE}.pkl")

def load_sample_split_artifacts() -> Dict[str, Any]:
    return {
        "train": _load_pickle(_fname("train_pairs_sample_split")),
        "val": _load_pickle(_fname("val_pairs_sample_split")),
        "test": _load_pickle(_fname("test_pairs_sample_split")),
        "pair_mapping": _load_pickle(_fname("pair_mapping_sample_split")),
        "label_mappings": _load_pickle(_fname("label_mappings_sample_split")),
        "scaler": _load_pickle(_fname("scaler_sample_split")),
    }

def _determine_label_field(d: Dict[str, dict]) -> Tuple[str, bool]:
    if len(d) == 0:
        return "class", False
    if any("class" in s for s in d.values()):
        return "class", False
    encoded_keys = ("class_encoded", "label_encoded", "target_class_encoded", "hclass_encoded")
    coverage = {k: sum(1 for s in d.values() if k in s and s[k] is not None and s[k] >= 0) for k in encoded_keys}
    if max(coverage.values()) == 0:
        return "class", False
    best = max(coverage.items(), key=lambda kv: kv[1])[0]
    return best, True

def count_by_field(d: Dict[str, dict], field: str) -> Dict[Any, int]:
    ctr = Counter()
    for s in d.values():
        if field in s:
            ctr[s[field]] += 1
    return dict(sorted(ctr.items(), key=lambda kv: str(kv[0])))

def resolve_name_for_id(id_: int, label_mappings: dict) -> str:
    for key, mapping in label_mappings.items():
        if not isinstance(mapping, dict):
            continue
        rev = {int(v): str(k) for k, v in mapping.items() if isinstance(v, (int, np.integer))}
        if id_ in rev:
            return rev[id_]
    return f"Class_{int(id_)}"

# ==============================
#          SMOOTHING
# ==============================
def ema(series: np.ndarray, alpha: float = 0.2) -> np.ndarray:
    """Exponential moving average."""
    y = np.copy(series).astype(np.float64)
    if len(y) == 0:
        return y
    s = y[0]
    out = np.zeros_like(y)
    out[0] = s
    for i in range(1, len(y)):
        s = alpha * y[i] + (1 - alpha) * s
        out[i] = s
    return out.astype(series.dtype)

# ==============================
#  SEGMENT LIBRARY
# ==============================
def build_segment_library(
    train_data: Dict[str, dict],
    top_classes: List[int],
    label_field: str
) -> Dict[int, Dict[str, List[np.ndarray]]]:
    lib = {c: {'pre': [], 'post': []} for c in top_classes}
    for _, sample in train_data.items():
        cid = sample.get(label_field)
        if cid not in top_classes:
            continue
        x_pre = sample.get('x_before')
        x_post = sample.get('x_after')
        if x_pre is not None and x_post is not None:
            lib[cid]['pre'].append(np.asarray(x_pre, dtype=np.float32))
            lib[cid]['post'].append(np.asarray(x_post, dtype=np.float32))
    return lib

# ==============================
#  SYNTHETIC TRAJECTORY
# ==============================
def sample_run_length(rng: random.Random, p: float) -> int:
    return int(np.random.geometric(p))

def markov_regime_sequence(run_length: int, p_action: float, p_relapse: float, rng: random.Random) -> List[str]:
    regimes, current = [], 'pre'
    for _ in range(run_length):
        regimes.append(current)
        if current == 'pre':
            if rng.random() < p_action:
                current = 'post'
        else:
            if rng.random() < p_relapse:
                current = 'pre'
    return regimes

def generate_trajectory(
    segment_library: Dict[int, Dict[str, List[np.ndarray]]],
    top_classes: List[int],
    class_frequencies: Dict[int, int],
    target_segments: int,
    p_run: float,
    p_action: float,
    p_relapse: float,
    seed: int
) -> Dict[str, Any]:
    rng = random.Random(seed)
    np.random.seed(seed)
    total = sum(class_frequencies.values())
    class_probs = {c: class_frequencies[c] / total for c in top_classes}

    segs, regs, clss, boundaries = [], [], [], []
    current_pos = 0
    while len(segs) < target_segments:
        sel_c = rng.choices(top_classes, weights=[class_probs[c] for c in top_classes])[0]
        L = sample_run_length(rng, p_run)
        L = min(L, target_segments - len(segs))
        if L <= 0:
            break
        sequence = markov_regime_sequence(L, p_action, p_relapse, rng)
        for r in sequence:
            avail = segment_library[sel_c][r]
            if len(avail) == 0:
                continue
            seg = rng.choice(avail)
            segs.append(seg)
            regs.append(r)
            clss.append(sel_c)
            if len(regs) > 1 and regs[-1] != regs[-2]:
                boundaries.append(current_pos)
            current_pos += len(seg)

    if segs:
        traj = np.concatenate(segs, axis=0)
    else:
        traj = np.zeros((0, 1), dtype=np.float32)

    return {
        'trajectory': traj,
        'regimes': regs,
        'classes': clss,
        'boundaries': boundaries,
        'segment_starts': np.cumsum([0] + [len(s) for s in segs[:-1]]).tolist()
    }

# ==============================
#  HI EXTRACTION
# ==============================
def extract_hi_from_trajectory(
    trajectory: np.ndarray,
    model: torch.nn.Module,
    device: torch.device,
    window_size: int = 250
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    T, C = trajectory.shape
    hi = np.zeros(T, dtype=np.float32)
    W = None
    model.eval()
    with torch.no_grad():
        for t in range(0, T, max(1, window_size // 4)):
            end_t = min(t + window_size, T)
            win = trajectory[t:end_t]
            if len(win) < window_size // 2:
                continue
            if len(win) < window_size:
                pad = window_size - len(win)
                win = np.vstack([win, np.zeros((pad, C), dtype=np.float32)])
            x = torch.from_numpy(win).unsqueeze(0).to(device)
            try:
                hi_b, _, w_b, _, _, _ = model.encoder(x)
                vals = hi_b.cpu().numpy().flatten()
                vlen = min(len(vals), end_t - t)
                hi[t:t+vlen] = vals[:vlen]
                if w_b is not None:
                    wv = w_b.cpu().numpy()
                    if W is None:
                        K = wv.shape[-1]
                        W = np.zeros((T, K), dtype=np.float32)
                    W[t:t+vlen] = wv[0, :vlen]
            except Exception as e:
                print(f"[WARN] HI extraction failed at t={t}: {e}")
                continue
    return hi, W

# ==============================
#  METRICS (PME, SM, ΔSM, RS)
# ==============================
def _ols_slope(x: np.ndarray) -> float:
    n = len(x)
    if n < 2:
        return 0.0
    t = np.arange(n)
    return np.polyfit(t, x, 1)[0]

def compute_pme(hi: np.ndarray, t: int, win_pre: int, win_post: int) -> float:
    if t < win_pre or t + win_post > len(hi):
        return 0.0
    pre = hi[t - win_pre:t]
    post = hi[t:t + win_post]
    if len(pre) < 2 or len(post) < 2:
        return 0.0
    mean_diff = np.mean(post) - np.mean(pre)
    slope_diff = _ols_slope(pre) - _ols_slope(post)
    return float(mean_diff + slope_diff)

def _sm_on_window(weights_slice: np.ndarray, op_groups: Dict[str, List[int]]) -> float:
    """SM = linear - (concave + oscillatory) on one window (row-wise normalized before averaging)."""
    if len(weights_slice) == 0:
        return 0.0
    w_norm = weights_slice / (weights_slice.sum(axis=1, keepdims=True) + 1e-8)
    m_linear = np.mean(w_norm[:, op_groups.get('linear', [])]) if op_groups.get('linear') else 0.0
    m_concave = np.mean(w_norm[:, op_groups.get('concave', [])]) if op_groups.get('concave') else 0.0
    m_osc = np.mean(w_norm[:, op_groups.get('oscillatory', [])]) if op_groups.get('oscillatory') else 0.0
    return float(m_linear - (m_concave + m_osc))

def compute_sm_pair(weights: Optional[np.ndarray], t: int, win_pre: int, win_post: int,
                    op_groups: Dict[str, List[int]]) -> Tuple[Optional[float], Optional[float]]:
    """Return (SM_post, ΔSM = SM_post - SM_pre)."""
    if weights is None or t + win_post > len(weights) or t - win_pre < 0:
        return None, None
    w_pre = weights[t - win_pre:t]
    w_post = weights[t:t + win_post]
    sm_pre = _sm_on_window(w_pre, op_groups)
    sm_post = _sm_on_window(w_post, op_groups)
    return sm_post, (sm_post - sm_pre)

def compute_rs(pme: float, dsm: Optional[float], a0: float, a1: float, a2: float) -> float:
    """Use PME and ΔSM as RS inputs, consistent with the improvement proposal."""
    z = a0 + a1 * pme + (a2 * (0.0 if dsm is None else dsm))
    z = np.clip(z, -50, 50)
    return float(1.0 / (1.0 + np.exp(-z)))

def compute_metric_timeline(
    hi: np.ndarray,
    weights: Optional[np.ndarray],
    win_size: int,
    stride: int,
    op_groups: Dict[str, List[int]],
    a0: float, a1: float, a2: float,
    ema_alpha: float = 0.2
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Returns: PME_tl, SM_post_tl, ΔSM_tl, RS_tl
    """
    T = len(hi)
    win_pre = win_size // 2
    win_post = win_size // 2
    pme_tl = np.zeros(T, dtype=np.float32)
    sm_post_tl = np.zeros(T, dtype=np.float32)
    dsm_tl = np.zeros(T, dtype=np.float32)
    rs_tl = np.zeros(T, dtype=np.float32)

    for t in range(win_pre, T - win_post, stride):
        pme = compute_pme(hi, t, win_pre, win_post)
        sm_post, dsm = compute_sm_pair(weights, t, win_pre, win_post, op_groups)
        rs = compute_rs(pme, dsm, a0, a1, a2)

        pme_tl[t] = pme
        sm_post_tl[t] = 0.0 if sm_post is None else sm_post
        dsm_tl[t] = 0.0 if dsm is None else dsm
        rs_tl[t] = rs

    # EMA smoothing
    pme_tl = ema(pme_tl, ema_alpha)
    sm_post_tl = ema(sm_post_tl, ema_alpha)
    dsm_tl = ema(dsm_tl, ema_alpha)
    rs_tl = ema(rs_tl, ema_alpha)

    # Edge padding
    if win_pre < T:
        for arr in (pme_tl, sm_post_tl, dsm_tl, rs_tl):
            arr[:win_pre] = arr[win_pre]
    if T - win_post > 0:
        for arr in (pme_tl, sm_post_tl, dsm_tl, rs_tl):
            arr[T - win_post:] = arr[T - win_post - 1]

    return pme_tl, sm_post_tl, dsm_tl, rs_tl

# ==============================
#  DETECTION (graded conjunction + NMS + refractory)
# ==============================
def _local_maxima(x: np.ndarray) -> List[int]:
    idx = []
    for i in range(1, len(x) - 1):
        if x[i] > x[i - 1] and x[i] > x[i + 1]:
            idx.append(i)
    return idx

def _nms_1d(idx: List[int], scores: np.ndarray, min_dist: int) -> List[int]:
    if not idx:
        return []
    order = sorted(idx, key=lambda i: scores[i], reverse=True)
    keep = []
    taken = np.zeros_like(scores, dtype=bool)
    for i in order:
        if taken[i]:
            continue
        keep.append(i)
        l = max(0, i - min_dist)
        r = min(len(scores), i + min_dist + 1)
        taken[l:r] = True
    return sorted(keep)

def detect_boundaries_conj_graded(
    pme_tl: np.ndarray,
    dsm_tl: np.ndarray,
    rs_tl: np.ndarray,
    tau_pme_low: float,
    tau_dsm_low: float,
    tau_rs_mid: float,
    tau_rs_high: float,
    refractory: int,
    nms_neighbor: int
) -> List[int]:
    """
    Graded conjunction(stricter AND logic):
      1) If the RS peak > τ_RS_high:require both (PME>τ_PME_low AND ΔSM>τ_ΔSM_low)
      2) If the RS peak > τ_RS_mid:require both (PME>τ_PME_low AND ΔSM>τ_ΔSM_low)
    Then apply NMS and enforce a minimum interval.
    """
    peaks = _local_maxima(rs_tl)
    cand = []
    for i in peaks:
        # Strong condition:high RS value and both PME and ΔSM pass
        strong = rs_tl[i] > tau_rs_high and (pme_tl[i] > tau_pme_low and dsm_tl[i] > tau_dsm_low)
        # Medium condition:medium RS value and both PME and ΔSM pass
        medium = (rs_tl[i] > tau_rs_mid) and (pme_tl[i] > tau_pme_low and dsm_tl[i] > tau_dsm_low)
        if strong or medium:
            cand.append(i)

    cand = _nms_1d(cand, rs_tl, nms_neighbor)

    detected = []
    last = -10**9
    for i in cand:
        if i - last >= refractory:
            detected.append(i)
            last = i
    return detected

def evaluate_detection(true_boundaries: List[int], detected_boundaries: List[int], tolerance: int) -> Dict[str, Any]:
    if len(true_boundaries) == 0:
        return {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'mabe': 0.0}
    true_matched = set()
    det_matched = set()
    errs = []
    for det in detected_boundaries:
        for j, tru in enumerate(true_boundaries):
            if abs(det - tru) <= tolerance and j not in true_matched:
                true_matched.add(j)
                det_matched.add(det)
                errs.append(abs(det - tru))
                break
    tp = len(true_matched)
    fp = len(detected_boundaries) - len(det_matched)
    fn = len(true_boundaries) - len(true_matched)
    precision = tp / max(1, (tp + fp))
    recall = tp / max(1, (tp + fn))
    f1 = 2 * precision * recall / max(1e-8, (precision + recall))
    mabe = float(np.mean(errs)) if errs else 0.0
    return {'precision': precision, 'recall': recall, 'f1': f1, 'mabe': mabe, 'tp': tp, 'fp': fp, 'fn': fn}

# ==============================
#  STABILIZATION & RISK
# ==============================
def compute_stabilization_time(sm_post_tl: np.ndarray, rs_tl: np.ndarray, detected: List[int],
                               tau_sm_post: float, tau_rs: float, delta_window: int) -> List[int]:
    tts = []
    for b in detected:
        for t in range(b, len(sm_post_tl) - delta_window):
            if np.all(sm_post_tl[t:t + delta_window] > tau_sm_post) and np.all(rs_tl[t:t + delta_window] > tau_rs):
                tts.append(t - b)
                break
    return tts

def evaluate_risk_calibration(rs_tl: np.ndarray, true_regimes: List[str], segment_starts: List[int]) -> Dict[str, float]:
    """
    Risk is defined as whether the next segment returns to pre(1=returns to pre).Use 1 - RS as the risk score to keep the direction consistent.
    """
    labels, scores = [], []
    for i in range(len(segment_starts) - 1):
        s, e = segment_starts[i], segment_starts[i + 1]
        rs_avg = float(np.mean(rs_tl[s:e]))
        scores.append(1.0 - rs_avg)  # Key point: keep the direction aligned
        nxt = true_regimes[i + 1] if i + 1 < len(true_regimes) else 'post'
        labels.append(1 if nxt == 'pre' else 0)
    if len(set(labels)) < 2:
        return {'auc': 0.5, 'brier': 0.25}
    auc = roc_auc_score(labels, scores)
    brier = brier_score_loss(labels, scores)
    return {'auc': float(auc), 'brier': float(brier)}

# ==============================
#  RS CALIBRATION (z-score + monotonic)
# ==============================
def calibrate_rs_with_logit(
    pme_tl: np.ndarray,
    dsm_tl: np.ndarray,
    true_regimes: List[str],
    segment_starts: List[int]
) -> Tuple[float, float, float]:
    """
    Segment-level features(PME, ΔSM) → z-score → logistic fit:
      z = a0 + a1*PME_z + a2*ΔSM_z ; RS = sigmoid(z)
    Monotonic prior: clamp a1 and a2 to 0 when they are negative so that better recovery does not imply lower RS.
    """
    X_raw, y = [], []
    for i in range(len(segment_starts) - 1):
        s, e = segment_starts[i], segment_starts[i + 1]
        pme_avg = float(np.mean(pme_tl[s:e]))
        dsm_avg = float(np.mean(dsm_tl[s:e]))
        X_raw.append([pme_avg, dsm_avg])
        nxt = true_regimes[i + 1] if i + 1 < len(true_regimes) else 'post'
        y.append(1 if nxt == 'pre' else 0)

    if len(set(y)) < 2:
        return (-3.0, 15.0, 10.0)

    X_raw = np.array(X_raw)
    # z-score
    mu = X_raw.mean(axis=0, keepdims=True)
    std = X_raw.std(axis=0, keepdims=True) + 1e-8
    X = (X_raw - mu) / std

    try:
        clf = LogisticRegression(max_iter=400, solver="lbfgs")
        clf.fit(X, np.array(y))
        a0 = float(clf.intercept_[0])
        a1 = float(clf.coef_[0][0])
        a2 = float(clf.coef_[0][1])
        # Monotonic prior: clamp negative coefficients to 0
        a1 = max(0.0, a1)
        a2 = max(0.0, a2)
        # To reuse the z-score transformation during inference, map (a1, a2) back to the original scale:
        #  σ(a0 + a1*PME_z + a2*ΔSM_z) = σ(a0 + a1*(PME-μ1)/σ1 + a2*(ΔSM-μ2)/σ2)
        #  => Equivalent form: a0' = a0 - a1*μ1/σ1 - a2*μ2/σ2 ; a1' = a1/σ1 ; a2' = a2/σ2
        mu1, mu2 = float(mu[0, 0]), float(mu[0, 1])
        std1, std2 = float(std[0, 0]), float(std[0, 1])
        a1_prime = a1 / std1
        a2_prime = a2 / std2
        a0_prime = a0 - a1 * (mu1 / std1) - a2 * (mu2 / std2)
        return (float(a0_prime), float(a1_prime), float(a2_prime))
    except Exception as e:
        print(f"[WARN] Logistic calibration failed: {e}")
        return (-3.0, 15.0, 10.0)

# ==============================
#  MAIN
# ==============================
if __name__ == "__main__":
    print("[INFO] Starting long-horizon synthetic trajectory analysis...")

    # 1) Load artifacts
    artifacts = load_sample_split_artifacts()
    train_data = artifacts["train"]
    label_mappings = artifacts.get("label_mappings", {})

    # 2) Top-K classes
    label_field, is_encoded = _determine_label_field(train_data)
    freq = count_by_field(train_data, label_field)
    top_items = sorted(freq.items(), key=lambda kv: kv[1], reverse=True)[:TOP_K]
    top_classes = [k for k, _ in top_items]
    class_frequencies = {k: v for k, v in top_items}
    print("[INFO] Top {} classes:".format(TOP_K))
    for k, v in top_items:
        name = resolve_name_for_id(int(k), label_mappings) if is_encoded else k
        print(f"  - {name} ({k}): {v} samples")

    # 3) Segment library
    print("[INFO] Building segment library...")
    segment_library = build_segment_library(train_data, top_classes, label_field)
    for c in top_classes:
        print(f"  Class {c}: {len(segment_library[c]['pre'])} pre, {len(segment_library[c]['post'])} post")

    # 4) Load model (ensure DiffAwareReconstructor is available)
    print("[INFO] Loading model...")
    ckpt_path = os.path.join(SAVE_DIR, "best_model_NGFAID_250_100_6.pth")
    meta_path = ckpt_path.replace('.pth', '_meta.json')
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if os.path.exists(meta_path):
        with open(meta_path, 'r') as f:
            meta_data = json.load(f)
        K = meta_data.get('K', len(top_classes))
        C = meta_data.get('sensor_dim', segment_library[top_classes[0]]['pre'][0].shape[1])
        print(f"[INFO] Loaded metadata: K={K}, C={C}")
    else:
        first_segment = segment_library[top_classes[0]]['pre'][0]
        C = first_segment.shape[1]
        K = len(top_classes)
        print(f"[INFO] Inferred from data: K={K}, C={C}")

    # >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
    model = DiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=K).to(device)
    state_dict = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state_dict, strict=False)
    model.eval()
    print("[INFO] Model loaded successfully (with partial matching)")
    # <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

    # 5) Operator groups
    op_groups = {
        'linear': [0, 1, 7],
        'concave': [2, 3, 5],
        'oscillatory': [4, 6]
    }

    results = []

    # -------- Calibrate RS using the first seed --------
    seed0 = RANDOM_SEEDS[0]
    print(f"\n[INFO] Calibration seed {seed0} ...")
    traj0 = generate_trajectory(segment_library, top_classes, class_frequencies,
                                TARGET_SEGMENTS, P_RUN_LENGTH, P_ACTION, P_RELAPSE, seed0)
    arr0 = traj0['trajectory']
    regimes0 = traj0['regimes']
    seg_starts0 = traj0['segment_starts']

    hi0, W0 = extract_hi_from_trajectory(arr0, model, device, window_size=WINDOW_SIZE)
    spd0 = len(arr0) / (len(traj0['regimes']) * DAYS_PER_SEGMENT)
    win0 = int(METRIC_WINDOW_DAYS * spd0)
    stride0 = max(1, int(STRIDE_FRACTION * spd0))
    pme0, sm_post0, dsm0, _ = compute_metric_timeline(hi0, W0, win0, stride0, op_groups,
                                                      a0=-3.0, a1=15.0, a2=10.0, ema_alpha=0.2)
    a0, a1, a2 = calibrate_rs_with_logit(pme0, dsm0, regimes0, seg_starts0)
    print(f"[INFO] Calibrated RS head: a0={a0:.3f}, a1={a1:.3f}, a2={a2:.3f}")

    seeds_to_run = RANDOM_SEEDS

    for seed in seeds_to_run:
        print(f"\n[INFO] Processing seed {seed}...")
        traj = generate_trajectory(segment_library, top_classes, class_frequencies,
                                   TARGET_SEGMENTS, P_RUN_LENGTH, P_ACTION, P_RELAPSE, seed)
        arr = traj['trajectory']
        true_bounds = traj['boundaries']
        regimes = traj['regimes']
        seg_starts = traj['segment_starts']
        print(f"  Generated trajectory: {len(arr)} samples, {len(true_bounds)} boundaries")

        # HI and metrics
        hi, W = extract_hi_from_trajectory(arr, model, device, window_size=WINDOW_SIZE)
        spd = len(arr) / (len(traj['regimes']) * DAYS_PER_SEGMENT)
        win = int(METRIC_WINDOW_DAYS * spd)
        stride = max(1, int(STRIDE_FRACTION * spd))
        pme_tl, sm_post_tl, dsm_tl, rs_tl = compute_metric_timeline(
            hi, W, win, stride, op_groups, a0=a0, a1=a1, a2=a2, ema_alpha=0.2
        )

        # Robust percentiles
        def robust_percentile(x: np.ndarray, p: int, default: float) -> float:
            pos = x[np.isfinite(x) & (x > 0)]
            if len(pos) < max(5, int(MIN_POS_FRAC * len(x))):
                return default
            return float(np.percentile(pos, p))

        tau_pme_low = robust_percentile(pme_tl, PME_P_LOW, 0.0)
        tau_dsm_low = robust_percentile(dsm_tl, DSM_P_LOW, 0.0)
        tau_rs_mid = robust_percentile(rs_tl, RS_P_MID, np.median(rs_tl))
        tau_rs_high = robust_percentile(rs_tl, RS_P_HIGH, tau_rs_mid)

        # Detection (graded conjunction + suppression + NMS)
        tol = int(DETECTION_TOLERANCE_DAYS * spd)
        refractory = int(REFRACTORY_DAYS * spd)
        nms_nb = int(NMS_NEIGHBOR * spd)

        if USE_CONJ_RULE:
            detected = detect_boundaries_conj_graded(
                pme_tl, dsm_tl, rs_tl,
                tau_pme_low=tau_pme_low, tau_dsm_low=tau_dsm_low,
                tau_rs_mid=tau_rs_mid, tau_rs_high=tau_rs_high,
                refractory=refractory, nms_neighbor=nms_nb
            )
        else:
            peaks = _local_maxima(rs_tl)
            detected = [i for i in peaks if rs_tl[i] >= tau_rs_mid]

        # Evaluation
        det_metrics = evaluate_detection(true_boundaries=true_bounds,
                                         detected_boundaries=detected,
                                         tolerance=tol)
        print(f"  Detection - P: {det_metrics['precision']:.3f}, "
              f"R: {det_metrics['recall']:.3f}, "
              f"F1: {det_metrics['f1']:.3f}, "
              f"MABE: {det_metrics['mabe']:.1f}")

        # Stabilization (using SM_post + RS)
        delta_samples = int(STABILIZATION_WINDOW_DAYS * spd)
        tts_vals = compute_stabilization_time(sm_post_tl, rs_tl, detected, tau_sm_post=tau_pme_low*0.0+robust_percentile(sm_post_tl, 50, 0.0),
                                              tau_rs=tau_rs_mid, delta_window=delta_samples)
        if tts_vals:
            print(f"  Stabilization - Median TTS: {np.median(tts_vals):.1f} "
                  f"[{np.percentile(tts_vals,25):.1f}-{np.percentile(tts_vals,75):.1f}]")

        # Risk calibration evaluation (using 1 - RS)
        risk_metrics = evaluate_risk_calibration(rs_tl, regimes, seg_starts)
        print(f"  Risk calibration - AUC: {risk_metrics['auc']:.3f}, Brier: {risk_metrics['brier']:.3f}")

        # Save
        results.append({
            'seed': seed,
            'detection': det_metrics,
            'tts': tts_vals,
            'risk': risk_metrics,
            'hi_timeline': hi,
            'pme_timeline': pme_tl,
            'sm_post_timeline': sm_post_tl,
            'dsm_timeline': dsm_tl,
            'rs_timeline': rs_tl,
            'true_boundaries': true_bounds,
            'detected_boundaries': detected,
            'thresholds': {
                'tau_pme_low': tau_pme_low,
                'tau_dsm_low': tau_dsm_low,
                'tau_rs_mid': tau_rs_mid,
                'tau_rs_high': tau_rs_high
            },
            'samples_per_day': spd
        })

        # Plotting (first seed)
        if seed == seeds_to_run[0] and DO_PLOTS:
            fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

            # HI
            axes[0].plot(hi, alpha=0.85, lw=0.8, label='HI')
            for b in true_bounds:
                axes[0].axvline(b, color='red', alpha=0.35, ls=':', label='True' if b == true_bounds[0] else None)
            for d in detected:
                axes[0].axvline(d, color='green', alpha=0.75, ls='--', label='Detected' if d == detected[0] else None)
            axes[0].set_ylabel('Health Index'); axes[0].legend(loc='upper right'); axes[0].grid(alpha=0.3)

            # PME / ΔSM(with thresholds)
            axes[1].bar(np.arange(len(pme_tl)), pme_tl, width=1.0, alpha=0.8, label='PME')
            axes[1].axhline(tau_pme_low, color='tab:blue', ls='--', alpha=0.7, label=f'τ_PME (p{PME_P_LOW})')
            axes[1].bar(np.arange(len(dsm_tl)), dsm_tl, width=1.0, alpha=0.5, color='tab:purple', label='ΔSM')
            axes[1].axhline(tau_dsm_low, color='tab:purple', ls='--', alpha=0.7, label=f'τ_ΔSM (p{DSM_P_LOW})')
            axes[1].legend(loc='upper right'); axes[1].grid(alpha=0.3)

            # RS(with thresholds)
            axes[2].plot(rs_tl, lw=0.9, color='orange', label='RS')
            axes[2].axhline(tau_rs_mid, color='green', ls='--', alpha=0.6, label=f'τ_RS (p{RS_P_MID})')
            axes[2].axhline(tau_rs_high, color='green', ls='-.', alpha=0.6, label=f'τ_RS (p{RS_P_HIGH})')
            for b in true_bounds:
                axes[2].axvline(b, color='red', alpha=0.35, ls=':')
            for d in detected:
                axes[2].axvline(d, color='green', alpha=0.75, ls='--')
            axes[2].set_xlabel('Time Index'); axes[2].set_ylabel('Reliability Score'); axes[2].legend(loc='upper right'); axes[2].grid(alpha=0.3)

            plt.suptitle(f'Long-horizon Trajectory Analysis (Seed {seed})', fontsize=13, fontweight='bold')
            plt.tight_layout()
            if SAVE_FIGS:
                fig_path = os.path.join(FIGS_DIR, f'trajectory_seed_{seed}.png')
                plt.savefig(fig_path, dpi=160, bbox_inches='tight')
                print(f"  [Saved] {fig_path}")
            plt.show()

    # Summary
    print("\n" + "="*80)
    print("SUMMARY STATISTICS (mean ± std)")
    print("="*80)
    all_precision = [r['detection']['precision'] for r in results]
    all_recall = [r['detection']['recall'] for r in results]
    all_f1 = [r['detection']['f1'] for r in results]
    all_mabe = [r['detection']['mabe'] for r in results]
    print("Boundary Detection:")
    print(f"  Precision: {np.mean(all_precision):.3f} ± {np.std(all_precision):.3f}")
    print(f"  Recall:    {np.mean(all_recall):.3f} ± {np.std(all_recall):.3f}")
    print(f"  F1-Score:  {np.mean(all_f1):.3f} ± {np.std(all_f1):.3f}")
    print(f"  MABE:      {np.mean(all_mabe):.1f} ± {np.std(all_mabe):.1f} samples")

    all_tts = [t for r in results for t in r['tts']]
    if all_tts:
        print("\nStabilization:")
        print(f"  Median TTS: {np.median(all_tts):.1f} samples")
        print(f"  IQR: [{np.percentile(all_tts,25):.1f}, {np.percentile(all_tts,75):.1f}]")

    all_auc = [r['risk']['auc'] for r in results]
    all_brier = [r['risk']['brier'] for r in results]
    print("\nRisk Calibration (labels: next is pre; scores=1-RS):")
    print(f"  AUC:   {np.mean(all_auc):.3f} ± {np.std(all_auc):.3f}")
    print(f"  Brier: {np.mean(all_brier):.3f} ± {np.std(all_brier):.3f}")

    # Save
    output_path = os.path.join(SAVE_DIR, 'long_horizon_results.pkl')
    with open(output_path, 'wb') as f:
        pickle.dump({
            'config': {
                'seeds': RANDOM_SEEDS,
                'top_k': TOP_K,
                'target_days': TARGET_DAYS,
                'p_action': P_ACTION,
                'p_relapse': P_RELAPSE,
                'p_run_length': P_RUN_LENGTH,
                'use_conj_rule': USE_CONJ_RULE,
                'refractory_days': REFRACTORY_DAYS,
                'nms_neighbor_days': NMS_NEIGHBOR,
                'ema_alpha': 0.2,
                'percentiles': {
                    'PME_P_LOW': PME_P_LOW, 'DSM_P_LOW': DSM_P_LOW,
                    'RS_P_MID': RS_P_MID, 'RS_P_HIGH': RS_P_HIGH
                }
            },
            'results': results
        }, f)
    print(f"\n[INFO] Results saved to {output_path}")
    print("="*80 + "\n")


In [ ]:
# -*- coding: utf-8 -*-
"""
Long-horizon synthetic trajectory generation and reliability analysis
ΔSM + graded conjunction rule + refractory/NMS + EMA smoothing + RS z-score logistic calibration (monotonic)
"""

import os
import json
import pickle
import random
from typing import Dict, Any, List, Tuple, Optional
from collections import Counter

import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.linear_model import LogisticRegression

# ==============================
#            CONFIG
# ==============================
SAVE_DIR = str(DATA_DIR)
WINDOW_SIZE = 250
STEP_SIZE = 100

# Multi-seed evaluation
RANDOM_SEEDS = [7, 11, 19, 23, 29]
TOP_K = 3

# Target total trajectory length (~3 months ≈ 90 days → 45 segments of 2 days each)
TARGET_DAYS = 90
DAYS_PER_SEGMENT = 2.0
TARGET_SEGMENTS = int(TARGET_DAYS / DAYS_PER_SEGMENT)

# Markov chain parameters
P_ACTION = 0.30   # pre -> post
P_RELAPSE = 0.10  # post -> pre
P_RUN_LENGTH = 0.20  # geometric run length

# Window sizing for metrics (converted to sample counts for each seed)
STRIDE_FRACTION = 0.25      # stride = 1/4 day
METRIC_WINDOW_DAYS = 2.0    # window for PME/SM

# Detection parameters(tightened to reduce false positives)
USE_CONJ_RULE = True
DETECTION_TOLERANCE_DAYS = 1.0
STABILIZATION_WINDOW_DAYS = 2.0
REFRACTORY_DAYS = 1.5         # Extended refractory period: 1.5 days
NMS_NEIGHBOR = 0.5            # Wider NMS window: 0.5 day

# Threshold percentiles (more conservative: raise the thresholds)
PME_P_LOW = 60        # p40 -> p60
DSM_P_LOW = 60        # p40 -> p60
RS_P_MID = 75         # p60 -> p75
RS_P_HIGH = 85        # p70 -> p85
MIN_POS_FRAC = 0.05   # Fall back when the fraction of positive values is too low

# Plotting
DO_PLOTS = True
SAVE_FIGS = True
FIGS_DIR = os.path.join(SAVE_DIR, "reliability_figs")
os.makedirs(FIGS_DIR, exist_ok=True)

# ==============================
#     HELPERS: IO & LABELS
# ==============================
def _load_pickle(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)

def _fname(stem: str) -> str:
    return os.path.join(SAVE_DIR, f"{stem}_window_{WINDOW_SIZE}_step_{STEP_SIZE}.pkl")

def load_sample_split_artifacts() -> Dict[str, Any]:
    return {
        "train": _load_pickle(_fname("train_pairs_sample_split")),
        "val": _load_pickle(_fname("val_pairs_sample_split")),
        "test": _load_pickle(_fname("test_pairs_sample_split")),
        "pair_mapping": _load_pickle(_fname("pair_mapping_sample_split")),
        "label_mappings": _load_pickle(_fname("label_mappings_sample_split")),
        "scaler": _load_pickle(_fname("scaler_sample_split")),
    }

def _determine_label_field(d: Dict[str, dict]) -> Tuple[str, bool]:
    if len(d) == 0:
        return "class", False
    if any("class" in s for s in d.values()):
        return "class", False
    encoded_keys = ("class_encoded", "label_encoded", "target_class_encoded", "hclass_encoded")
    coverage = {k: sum(1 for s in d.values() if k in s and s[k] is not None and s[k] >= 0) for k in encoded_keys}
    if max(coverage.values()) == 0:
        return "class", False
    best = max(coverage.items(), key=lambda kv: kv[1])[0]
    return best, True

def count_by_field(d: Dict[str, dict], field: str) -> Dict[Any, int]:
    ctr = Counter()
    for s in d.values():
        if field in s:
            ctr[s[field]] += 1
    return dict(sorted(ctr.items(), key=lambda kv: str(kv[0])))

def resolve_name_for_id(id_: int, label_mappings: dict) -> str:
    for key, mapping in label_mappings.items():
        if not isinstance(mapping, dict):
            continue
        rev = {int(v): str(k) for k, v in mapping.items() if isinstance(v, (int, np.integer))}
        if id_ in rev:
            return rev[id_]
    return f"Class_{int(id_)}"

# ==============================
#          SMOOTHING
# ==============================
def ema(series: np.ndarray, alpha: float = 0.2) -> np.ndarray:
    """Exponential moving average."""
    y = np.copy(series).astype(np.float64)
    if len(y) == 0:
        return y
    s = y[0]
    out = np.zeros_like(y)
    out[0] = s
    for i in range(1, len(y)):
        s = alpha * y[i] + (1 - alpha) * s
        out[i] = s
    return out.astype(series.dtype)

# ==============================
#  SEGMENT LIBRARY
# ==============================
def build_segment_library(
    train_data: Dict[str, dict],
    top_classes: List[int],
    label_field: str
) -> Dict[int, Dict[str, List[np.ndarray]]]:
    lib = {c: {'pre': [], 'post': []} for c in top_classes}
    for _, sample in train_data.items():
        cid = sample.get(label_field)
        if cid not in top_classes:
            continue
        x_pre = sample.get('x_before')
        x_post = sample.get('x_after')
        if x_pre is not None and x_post is not None:
            lib[cid]['pre'].append(np.asarray(x_pre, dtype=np.float32))
            lib[cid]['post'].append(np.asarray(x_post, dtype=np.float32))
    return lib

# ==============================
#  SYNTHETIC TRAJECTORY
# ==============================
def sample_run_length(rng: random.Random, p: float) -> int:
    return int(np.random.geometric(p))

def markov_regime_sequence(run_length: int, p_action: float, p_relapse: float, rng: random.Random) -> List[str]:
    regimes, current = [], 'pre'
    for _ in range(run_length):
        regimes.append(current)
        if current == 'pre':
            if rng.random() < p_action:
                current = 'post'
        else:
            if rng.random() < p_relapse:
                current = 'pre'
    return regimes

def generate_trajectory(
    segment_library: Dict[int, Dict[str, List[np.ndarray]]],
    top_classes: List[int],
    class_frequencies: Dict[int, int],
    target_segments: int,
    p_run: float,
    p_action: float,
    p_relapse: float,
    seed: int
) -> Dict[str, Any]:
    rng = random.Random(seed)
    np.random.seed(seed)
    total = sum(class_frequencies.values())
    class_probs = {c: class_frequencies[c] / total for c in top_classes}

    segs, regs, clss, boundaries = [], [], [], []
    current_pos = 0
    while len(segs) < target_segments:
        sel_c = rng.choices(top_classes, weights=[class_probs[c] for c in top_classes])[0]
        L = sample_run_length(rng, p_run)
        L = min(L, target_segments - len(segs))
        if L <= 0:
            break
        sequence = markov_regime_sequence(L, p_action, p_relapse, rng)
        for r in sequence:
            avail = segment_library[sel_c][r]
            if len(avail) == 0:
                continue
            seg = rng.choice(avail)
            segs.append(seg)
            regs.append(r)
            clss.append(sel_c)
            if len(regs) > 1 and regs[-1] != regs[-2]:
                boundaries.append(current_pos)
            current_pos += len(seg)

    if segs:
        traj = np.concatenate(segs, axis=0)
    else:
        traj = np.zeros((0, 1), dtype=np.float32)

    return {
        'trajectory': traj,
        'regimes': regs,
        'classes': clss,
        'boundaries': boundaries,
        'segment_starts': np.cumsum([0] + [len(s) for s in segs[:-1]]).tolist()
    }

# ==============================
#  HI EXTRACTION
# ==============================
def extract_hi_from_trajectory(
    trajectory: np.ndarray,
    model: torch.nn.Module,
    device: torch.device,
    window_size: int = 250
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    T, C = trajectory.shape
    hi = np.zeros(T, dtype=np.float32)
    W = None
    model.eval()
    with torch.no_grad():
        for t in range(0, T, max(1, window_size // 4)):
            end_t = min(t + window_size, T)
            win = trajectory[t:end_t]
            if len(win) < window_size // 2:
                continue
            if len(win) < window_size:
                pad = window_size - len(win)
                win = np.vstack([win, np.zeros((pad, C), dtype=np.float32)])
            x = torch.from_numpy(win).unsqueeze(0).to(device)
            try:
                hi_b, _, w_b, _, _, _ = model.encoder(x)
                vals = hi_b.cpu().numpy().flatten()
                vlen = min(len(vals), end_t - t)
                hi[t:t+vlen] = vals[:vlen]
                if w_b is not None:
                    wv = w_b.cpu().numpy()
                    if W is None:
                        K = wv.shape[-1]
                        W = np.zeros((T, K), dtype=np.float32)
                    W[t:t+vlen] = wv[0, :vlen]
            except Exception as e:
                print(f"[WARN] HI extraction failed at t={t}: {e}")
                continue
    return hi, W

# ==============================
#  METRICS (PME, SM, ΔSM, RS)
# ==============================
def _ols_slope(x: np.ndarray) -> float:
    n = len(x)
    if n < 2:
        return 0.0
    t = np.arange(n)
    return np.polyfit(t, x, 1)[0]

def compute_pme(hi: np.ndarray, t: int, win_pre: int, win_post: int) -> float:
    if t < win_pre or t + win_post > len(hi):
        return 0.0
    pre = hi[t - win_pre:t]
    post = hi[t:t + win_post]
    if len(pre) < 2 or len(post) < 2:
        return 0.0
    mean_diff = np.mean(post) - np.mean(pre)
    slope_diff = _ols_slope(pre) - _ols_slope(post)
    return float(mean_diff + slope_diff)

def _sm_on_window(weights_slice: np.ndarray, op_groups: Dict[str, List[int]]) -> float:
    """SM = linear - (concave + oscillatory) on one window (row-wise normalized before averaging)."""
    if len(weights_slice) == 0:
        return 0.0
    w_norm = weights_slice / (weights_slice.sum(axis=1, keepdims=True) + 1e-8)
    m_linear = np.mean(w_norm[:, op_groups.get('linear', [])]) if op_groups.get('linear') else 0.0
    m_concave = np.mean(w_norm[:, op_groups.get('concave', [])]) if op_groups.get('concave') else 0.0
    m_osc = np.mean(w_norm[:, op_groups.get('oscillatory', [])]) if op_groups.get('oscillatory') else 0.0
    return float(m_linear - (m_concave + m_osc))

def compute_sm_pair(weights: Optional[np.ndarray], t: int, win_pre: int, win_post: int,
                    op_groups: Dict[str, List[int]]) -> Tuple[Optional[float], Optional[float]]:
    """Return (SM_post, ΔSM = SM_post - SM_pre)."""
    if weights is None or t + win_post > len(weights) or t - win_pre < 0:
        return None, None
    w_pre = weights[t - win_pre:t]
    w_post = weights[t:t + win_post]
    sm_pre = _sm_on_window(w_pre, op_groups)
    sm_post = _sm_on_window(w_post, op_groups)
    return sm_post, (sm_post - sm_pre)

def compute_rs(pme: float, dsm: Optional[float], a0: float, a1: float, a2: float) -> float:
    """Use PME and ΔSM as RS inputs, consistent with the improvement proposal."""
    z = a0 + a1 * pme + (a2 * (0.0 if dsm is None else dsm))
    z = np.clip(z, -50, 50)
    return float(1.0 / (1.0 + np.exp(-z)))

def compute_metric_timeline(
    hi: np.ndarray,
    weights: Optional[np.ndarray],
    win_size: int,
    stride: int,
    op_groups: Dict[str, List[int]],
    a0: float, a1: float, a2: float,
    ema_alpha: float = 0.2
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Returns: PME_tl, SM_post_tl, ΔSM_tl, RS_tl
    """
    T = len(hi)
    win_pre = win_size // 2
    win_post = win_size // 2
    pme_tl = np.zeros(T, dtype=np.float32)
    sm_post_tl = np.zeros(T, dtype=np.float32)
    dsm_tl = np.zeros(T, dtype=np.float32)
    rs_tl = np.zeros(T, dtype=np.float32)

    for t in range(win_pre, T - win_post, stride):
        pme = compute_pme(hi, t, win_pre, win_post)
        sm_post, dsm = compute_sm_pair(weights, t, win_pre, win_post, op_groups)
        rs = compute_rs(pme, dsm, a0, a1, a2)

        pme_tl[t] = pme
        sm_post_tl[t] = 0.0 if sm_post is None else sm_post
        dsm_tl[t] = 0.0 if dsm is None else dsm
        rs_tl[t] = rs

    # EMA smoothing
    pme_tl = ema(pme_tl, ema_alpha)
    sm_post_tl = ema(sm_post_tl, ema_alpha)
    dsm_tl = ema(dsm_tl, ema_alpha)
    rs_tl = ema(rs_tl, ema_alpha)

    # Edge padding
    if win_pre < T:
        for arr in (pme_tl, sm_post_tl, dsm_tl, rs_tl):
            arr[:win_pre] = arr[win_pre]
    if T - win_post > 0:
        for arr in (pme_tl, sm_post_tl, dsm_tl, rs_tl):
            arr[T - win_post:] = arr[T - win_post - 1]

    return pme_tl, sm_post_tl, dsm_tl, rs_tl

# ==============================
#  DETECTION (graded conjunction + NMS + refractory)
# ==============================
def _local_maxima(x: np.ndarray) -> List[int]:
    idx = []
    for i in range(1, len(x) - 1):
        if x[i] > x[i - 1] and x[i] > x[i + 1]:
            idx.append(i)
    return idx

def _nms_1d(idx: List[int], scores: np.ndarray, min_dist: int) -> List[int]:
    if not idx:
        return []
    order = sorted(idx, key=lambda i: scores[i], reverse=True)
    keep = []
    taken = np.zeros_like(scores, dtype=bool)
    for i in order:
        if taken[i]:
            continue
        keep.append(i)
        l = max(0, i - min_dist)
        r = min(len(scores), i + min_dist + 1)
        taken[l:r] = True
    return sorted(keep)

def detect_boundaries_conj_graded(
    pme_tl: np.ndarray,
    dsm_tl: np.ndarray,
    rs_tl: np.ndarray,
    tau_pme_low: float,
    tau_dsm_low: float,
    tau_rs_mid: float,
    tau_rs_high: float,
    refractory: int,
    nms_neighbor: int
) -> List[int]:
    """
    Graded conjunction(stricter AND logic):
      1) If the RS peak > τ_RS_high:require both (PME>τ_PME_low AND ΔSM>τ_ΔSM_low)
      2) If the RS peak > τ_RS_mid:require both (PME>τ_PME_low AND ΔSM>τ_ΔSM_low)
    Then apply NMS and enforce a minimum interval.
    """
    peaks = _local_maxima(rs_tl)
    cand = []
    for i in peaks:
        # Strong condition:high RS value and both PME and ΔSM pass
        strong = rs_tl[i] > tau_rs_high and (pme_tl[i] > tau_pme_low and dsm_tl[i] > tau_dsm_low)
        # Medium condition:medium RS value and both PME and ΔSM pass
        medium = (rs_tl[i] > tau_rs_mid) and (pme_tl[i] > tau_pme_low and dsm_tl[i] > tau_dsm_low)
        if strong or medium:
            cand.append(i)

    cand = _nms_1d(cand, rs_tl, nms_neighbor)

    detected = []
    last = -10**9
    for i in cand:
        if i - last >= refractory:
            detected.append(i)
            last = i
    return detected

def evaluate_detection(true_boundaries: List[int], detected_boundaries: List[int], tolerance: int) -> Dict[str, Any]:
    if len(true_boundaries) == 0:
        return {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'mabe': 0.0}
    true_matched = set()
    det_matched = set()
    errs = []
    for det in detected_boundaries:
        for j, tru in enumerate(true_boundaries):
            if abs(det - tru) <= tolerance and j not in true_matched:
                true_matched.add(j)
                det_matched.add(det)
                errs.append(abs(det - tru))
                break
    tp = len(true_matched)
    fp = len(detected_boundaries) - len(det_matched)
    fn = len(true_boundaries) - len(true_matched)
    precision = tp / max(1, (tp + fp))
    recall = tp / max(1, (tp + fn))
    f1 = 2 * precision * recall / max(1e-8, (precision + recall))
    mabe = float(np.mean(errs)) if errs else 0.0
    return {'precision': precision, 'recall': recall, 'f1': f1, 'mabe': mabe, 'tp': tp, 'fp': fp, 'fn': fn}

# ==============================
#  STABILIZATION & RISK
# ==============================
def compute_stabilization_time(sm_post_tl: np.ndarray, rs_tl: np.ndarray, detected: List[int],
                               tau_sm_post: float, tau_rs: float, delta_window: int) -> List[int]:
    tts = []
    for b in detected:
        for t in range(b, len(sm_post_tl) - delta_window):
            if np.all(sm_post_tl[t:t + delta_window] > tau_sm_post) and np.all(rs_tl[t:t + delta_window] > tau_rs):
                tts.append(t - b)
                break
    return tts

def evaluate_risk_calibration(rs_tl: np.ndarray, true_regimes: List[str], segment_starts: List[int]) -> Dict[str, float]:
    """
    Risk is defined as whether the next segment returns to pre(1=returns to pre).Use 1 - RS as the risk score to keep the direction consistent.
    """
    labels, scores = [], []
    for i in range(len(segment_starts) - 1):
        s, e = segment_starts[i], segment_starts[i + 1]
        rs_avg = float(np.mean(rs_tl[s:e]))
        scores.append(1.0 - rs_avg)  # Key point: keep the direction aligned
        nxt = true_regimes[i + 1] if i + 1 < len(true_regimes) else 'post'
        labels.append(1 if nxt == 'pre' else 0)
    if len(set(labels)) < 2:
        return {'auc': 0.5, 'brier': 0.25}
    auc = roc_auc_score(labels, scores)
    brier = brier_score_loss(labels, scores)
    return {'auc': float(auc), 'brier': float(brier)}

# ==============================
#  RS CALIBRATION (z-score + monotonic)
# ==============================
def calibrate_rs_with_logit(
    pme_tl: np.ndarray,
    dsm_tl: np.ndarray,
    true_regimes: List[str],
    segment_starts: List[int]
) -> Tuple[float, float, float]:
    """
    Segment-level features(PME, ΔSM) → z-score → logistic fit:
      z = a0 + a1*PME_z + a2*ΔSM_z ; RS = sigmoid(z)
    Monotonic prior: clamp a1 and a2 to 0 when they are negative so that better recovery does not imply lower RS.
    """
    X_raw, y = [], []
    for i in range(len(segment_starts) - 1):
        s, e = segment_starts[i], segment_starts[i + 1]
        pme_avg = float(np.mean(pme_tl[s:e]))
        dsm_avg = float(np.mean(dsm_tl[s:e]))
        X_raw.append([pme_avg, dsm_avg])
        nxt = true_regimes[i + 1] if i + 1 < len(true_regimes) else 'post'
        y.append(1 if nxt == 'pre' else 0)

    if len(set(y)) < 2:
        return (-3.0, 15.0, 10.0)

    X_raw = np.array(X_raw)
    # z-score
    mu = X_raw.mean(axis=0, keepdims=True)
    std = X_raw.std(axis=0, keepdims=True) + 1e-8
    X = (X_raw - mu) / std

    try:
        clf = LogisticRegression(max_iter=400, solver="lbfgs")
        clf.fit(X, np.array(y))
        a0 = float(clf.intercept_[0])
        a1 = float(clf.coef_[0][0])
        a2 = float(clf.coef_[0][1])
        # Monotonic prior: clamp negative coefficients to 0
        a1 = max(0.0, a1)
        a2 = max(0.0, a2)
        # To reuse the z-score transformation during inference, map (a1, a2) back to the original scale:
        #  σ(a0 + a1*PME_z + a2*ΔSM_z) = σ(a0 + a1*(PME-μ1)/σ1 + a2*(ΔSM-μ2)/σ2)
        #  => Equivalent form: a0' = a0 - a1*μ1/σ1 - a2*μ2/σ2 ; a1' = a1/σ1 ; a2' = a2/σ2
        mu1, mu2 = float(mu[0, 0]), float(mu[0, 1])
        std1, std2 = float(std[0, 0]), float(std[0, 1])
        a1_prime = a1 / std1
        a2_prime = a2 / std2
        a0_prime = a0 - a1 * (mu1 / std1) - a2 * (mu2 / std2)
        return (float(a0_prime), float(a1_prime), float(a2_prime))
    except Exception as e:
        print(f"[WARN] Logistic calibration failed: {e}")
        return (-3.0, 15.0, 10.0)

# ==============================
#  MAIN
# ==============================
if __name__ == "__main__":
    print("[INFO] Starting long-horizon synthetic trajectory analysis...")

    # 1) Load artifacts
    artifacts = load_sample_split_artifacts()
    train_data = artifacts["train"]
    label_mappings = artifacts.get("label_mappings", {})

    # 2) Top-K classes
    label_field, is_encoded = _determine_label_field(train_data)
    freq = count_by_field(train_data, label_field)
    top_items = sorted(freq.items(), key=lambda kv: kv[1], reverse=True)[:TOP_K]
    top_classes = [k for k, _ in top_items]
    class_frequencies = {k: v for k, v in top_items}
    print("[INFO] Top {} classes:".format(TOP_K))
    for k, v in top_items:
        name = resolve_name_for_id(int(k), label_mappings) if is_encoded else k
        print(f"  - {name} ({k}): {v} samples")

    # 3) Segment library
    print("[INFO] Building segment library...")
    segment_library = build_segment_library(train_data, top_classes, label_field)
    for c in top_classes:
        print(f"  Class {c}: {len(segment_library[c]['pre'])} pre, {len(segment_library[c]['post'])} post")

    # 4) Load model (ensure DiffAwareReconstructor is available)
    print("[INFO] Loading model...")
    ckpt_path = os.path.join(SAVE_DIR, "best_model_NGFAID_250_100_6.pth")
    meta_path = ckpt_path.replace('.pth', '_meta.json')
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if os.path.exists(meta_path):
        with open(meta_path, 'r') as f:
            meta_data = json.load(f)
        K = meta_data.get('K', len(top_classes))
        C = meta_data.get('sensor_dim', segment_library[top_classes[0]]['pre'][0].shape[1])
        print(f"[INFO] Loaded metadata: K={K}, C={C}")
    else:
        first_segment = segment_library[top_classes[0]]['pre'][0]
        C = first_segment.shape[1]
        K = len(top_classes)
        print(f"[INFO] Inferred from data: K={K}, C={C}")

    # >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
    model = DiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=K).to(device)
    state_dict = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state_dict, strict=False)
    model.eval()
    print("[INFO] Model loaded successfully (with partial matching)")
    # <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

    # 5) Operator groups
    op_groups = {
        'linear': [0, 1, 7],
        'concave': [2, 3, 5],
        'oscillatory': [4, 6]
    }

    results = []

    # -------- Calibrate RS using the first seed --------
    seed0 = RANDOM_SEEDS[0]
    print(f"\n[INFO] Calibration seed {seed0} ...")
    traj0 = generate_trajectory(segment_library, top_classes, class_frequencies,
                                TARGET_SEGMENTS, P_RUN_LENGTH, P_ACTION, P_RELAPSE, seed0)
    arr0 = traj0['trajectory']
    regimes0 = traj0['regimes']
    seg_starts0 = traj0['segment_starts']

    hi0, W0 = extract_hi_from_trajectory(arr0, model, device, window_size=WINDOW_SIZE)
    spd0 = len(arr0) / (len(traj0['regimes']) * DAYS_PER_SEGMENT)
    win0 = int(METRIC_WINDOW_DAYS * spd0)
    stride0 = max(1, int(STRIDE_FRACTION * spd0))
    pme0, sm_post0, dsm0, _ = compute_metric_timeline(hi0, W0, win0, stride0, op_groups,
                                                      a0=-3.0, a1=15.0, a2=10.0, ema_alpha=0.2)
    a0, a1, a2 = calibrate_rs_with_logit(pme0, dsm0, regimes0, seg_starts0)
    print(f"[INFO] Calibrated RS head: a0={a0:.3f}, a1={a1:.3f}, a2={a2:.3f}")

    seeds_to_run = RANDOM_SEEDS

    for seed in seeds_to_run:
        print(f"\n[INFO] Processing seed {seed}...")
        traj = generate_trajectory(segment_library, top_classes, class_frequencies,
                                   TARGET_SEGMENTS, P_RUN_LENGTH, P_ACTION, P_RELAPSE, seed)
        arr = traj['trajectory']
        true_bounds = traj['boundaries']
        regimes = traj['regimes']
        seg_starts = traj['segment_starts']
        print(f"  Generated trajectory: {len(arr)} samples, {len(true_bounds)} boundaries")

        # HI and metrics
        hi, W = extract_hi_from_trajectory(arr, model, device, window_size=WINDOW_SIZE)
        spd = len(arr) / (len(traj['regimes']) * DAYS_PER_SEGMENT)
        win = int(METRIC_WINDOW_DAYS * spd)
        stride = max(1, int(STRIDE_FRACTION * spd))
        pme_tl, sm_post_tl, dsm_tl, rs_tl = compute_metric_timeline(
            hi, W, win, stride, op_groups, a0=a0, a1=a1, a2=a2, ema_alpha=0.2
        )

        # Robust percentiles
        def robust_percentile(x: np.ndarray, p: int, default: float) -> float:
            pos = x[np.isfinite(x) & (x > 0)]
            if len(pos) < max(5, int(MIN_POS_FRAC * len(x))):
                return default
            return float(np.percentile(pos, p))

        tau_pme_low = robust_percentile(pme_tl, PME_P_LOW, 0.0)
        tau_dsm_low = robust_percentile(dsm_tl, DSM_P_LOW, 0.0)
        tau_rs_mid = robust_percentile(rs_tl, RS_P_MID, np.median(rs_tl))
        tau_rs_high = robust_percentile(rs_tl, RS_P_HIGH, tau_rs_mid)

        # Detection (graded conjunction + suppression + NMS)
        tol = int(DETECTION_TOLERANCE_DAYS * spd)
        refractory = int(REFRACTORY_DAYS * spd)
        nms_nb = int(NMS_NEIGHBOR * spd)

        if USE_CONJ_RULE:
            detected = detect_boundaries_conj_graded(
                pme_tl, dsm_tl, rs_tl,
                tau_pme_low=tau_pme_low, tau_dsm_low=tau_dsm_low,
                tau_rs_mid=tau_rs_mid, tau_rs_high=tau_rs_high,
                refractory=refractory, nms_neighbor=nms_nb
            )
        else:
            peaks = _local_maxima(rs_tl)
            detected = [i for i in peaks if rs_tl[i] >= tau_rs_mid]

        # Evaluation
        det_metrics = evaluate_detection(true_boundaries=true_bounds,
                                         detected_boundaries=detected,
                                         tolerance=tol)
        print(f"  Detection - P: {det_metrics['precision']:.3f}, "
              f"R: {det_metrics['recall']:.3f}, "
              f"F1: {det_metrics['f1']:.3f}, "
              f"MABE: {det_metrics['mabe']:.1f}")

        # Stabilization (using SM_post + RS)
        delta_samples = int(STABILIZATION_WINDOW_DAYS * spd)
        tts_vals = compute_stabilization_time(sm_post_tl, rs_tl, detected, tau_sm_post=tau_pme_low*0.0+robust_percentile(sm_post_tl, 50, 0.0),
                                              tau_rs=tau_rs_mid, delta_window=delta_samples)
        if tts_vals:
            print(f"  Stabilization - Median TTS: {np.median(tts_vals):.1f} "
                  f"[{np.percentile(tts_vals,25):.1f}-{np.percentile(tts_vals,75):.1f}]")

        # Risk calibration evaluation (using 1 - RS)
        risk_metrics = evaluate_risk_calibration(rs_tl, regimes, seg_starts)
        print(f"  Risk calibration - AUC: {risk_metrics['auc']:.3f}, Brier: {risk_metrics['brier']:.3f}")

        # Save
        results.append({
            'seed': seed,
            'detection': det_metrics,
            'tts': tts_vals,
            'risk': risk_metrics,
            'hi_timeline': hi,
            'pme_timeline': pme_tl,
            'sm_post_timeline': sm_post_tl,
            'dsm_timeline': dsm_tl,
            'rs_timeline': rs_tl,
            'true_boundaries': true_bounds,
            'detected_boundaries': detected,
            'thresholds': {
                'tau_pme_low': tau_pme_low,
                'tau_dsm_low': tau_dsm_low,
                'tau_rs_mid': tau_rs_mid,
                'tau_rs_high': tau_rs_high
            },
            'samples_per_day': spd
        })

        # Plotting (first seed)
        if seed == seeds_to_run[0] and DO_PLOTS:
            fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

            # HI
            axes[0].plot(hi, alpha=0.85, lw=0.8, label='HI')
            # Plot true boundaries
            for idx, b in enumerate(true_bounds):
                axes[0].axvline(b, color='red', alpha=0.35, ls=':', label='True' if idx == 0 else '')
            # Plot detected boundaries
            for idx, d in enumerate(detected):
                axes[0].axvline(d, color='green', alpha=0.75, ls='--', label='Detected' if idx == 0 else '')
            axes[0].set_ylabel('Health Index')
            axes[0].legend(loc='upper right')
            axes[0].grid(alpha=0.3)

            # PME / ΔSM(with thresholds)
            axes[1].bar(np.arange(len(pme_tl)), pme_tl, width=1.0, alpha=0.8, label='PME')
            axes[1].axhline(tau_pme_low, color='tab:blue', ls='--', alpha=0.7, label=f'τ_PME (p{PME_P_LOW})')
            axes[1].bar(np.arange(len(dsm_tl)), dsm_tl, width=1.0, alpha=0.5, color='tab:purple', label='ΔSM')
            axes[1].axhline(tau_dsm_low, color='tab:purple', ls='--', alpha=0.7, label=f'τ_ΔSM (p{DSM_P_LOW})')
            axes[1].legend(loc='upper right')
            axes[1].grid(alpha=0.3)

            # RS(with thresholds)
            axes[2].plot(rs_tl, lw=0.9, color='orange', label='RS')
            axes[2].axhline(tau_rs_mid, color='green', ls='--', alpha=0.6, label=f'τ_RS (p{RS_P_MID})')
            axes[2].axhline(tau_rs_high, color='green', ls='-.', alpha=0.6, label=f'τ_RS (p{RS_P_HIGH})')
            # Plot true boundaries
            for idx, b in enumerate(true_bounds):
                axes[2].axvline(b, color='red', alpha=0.35, ls=':')
            # Plot detected boundaries
            for idx, d in enumerate(detected):
                axes[2].axvline(d, color='green', alpha=0.75, ls='--')
            axes[2].set_xlabel('Time Index')
            axes[2].set_ylabel('Reliability Score')
            axes[2].legend(loc='upper right')
            axes[2].grid(alpha=0.3)

            plt.suptitle(f'Long-horizon Trajectory Analysis (Seed {seed})', fontsize=13, fontweight='bold')
            plt.tight_layout()
            if SAVE_FIGS:
                fig_path = os.path.join(FIGS_DIR, f'trajectory_seed_{seed}.png')
                plt.savefig(fig_path, dpi=160, bbox_inches='tight')
                print(f"  [Saved] {fig_path}")
            plt.show()

    # Summary
    print("\n" + "="*80)
    print("SUMMARY STATISTICS (mean ± std)")
    print("="*80)
    all_precision = [r['detection']['precision'] for r in results]
    all_recall = [r['detection']['recall'] for r in results]
    all_f1 = [r['detection']['f1'] for r in results]
    all_mabe = [r['detection']['mabe'] for r in results]
    print("Boundary Detection:")
    print(f"  Precision: {np.mean(all_precision):.3f} ± {np.std(all_precision):.3f}")
    print(f"  Recall:    {np.mean(all_recall):.3f} ± {np.std(all_recall):.3f}")
    print(f"  F1-Score:  {np.mean(all_f1):.3f} ± {np.std(all_f1):.3f}")
    print(f"  MABE:      {np.mean(all_mabe):.1f} ± {np.std(all_mabe):.1f} samples")

    all_tts = [t for r in results for t in r['tts']]
    if all_tts:
        print("\nStabilization:")
        print(f"  Median TTS: {np.median(all_tts):.1f} samples")
        print(f"  IQR: [{np.percentile(all_tts,25):.1f}, {np.percentile(all_tts,75):.1f}]")

    all_auc = [r['risk']['auc'] for r in results]
    all_brier = [r['risk']['brier'] for r in results]
    print("\nRisk Calibration (labels: next is pre; scores=1-RS):")
    print(f"  AUC:   {np.mean(all_auc):.3f} ± {np.std(all_auc):.3f}")
    print(f"  Brier: {np.mean(all_brier):.3f} ± {np.std(all_brier):.3f}")

    # Save
    output_path = os.path.join(SAVE_DIR, 'long_horizon_results.pkl')
    with open(output_path, 'wb') as f:
        pickle.dump({
            'config': {
                'seeds': RANDOM_SEEDS,
                'top_k': TOP_K,
                'target_days': TARGET_DAYS,
                'p_action': P_ACTION,
                'p_relapse': P_RELAPSE,
                'p_run_length': P_RUN_LENGTH,
                'use_conj_rule': USE_CONJ_RULE,
                'refractory_days': REFRACTORY_DAYS,
                'nms_neighbor_days': NMS_NEIGHBOR,
                'ema_alpha': 0.2,
                'percentiles': {
                    'PME_P_LOW': PME_P_LOW, 'DSM_P_LOW': DSM_P_LOW,
                    'RS_P_MID': RS_P_MID, 'RS_P_HIGH': RS_P_HIGH
                }
            },
            'results': results
        }, f)
    print(f"\n[INFO] Results saved to {output_path}")
    print("="*80 + "\n")


### Enhanced Multi-Class Trajectory Analysis


In [ ]:
# -*- coding: utf-8 -*-
"""
Long-horizon synthetic trajectory generation and reliability analysis
ΔSM + graded conjunction rule + refractory/NMS + EMA smoothing + RS z-score logistic calibration (monotonic)
Enhanced with:
  - Robust RS calibration (pooled across seeds, IQR-based standardization, shrinkage)
  - Multi-scale consensus fusion (K-of-N agreement instead of union)
  - Peak prominence & width filtering
  - Adaptive thresholding with full distribution
  - Stronger evidence gates (AND logic for high-confidence)
  - HMM smoothing for realistic run lengths
  - Per-class reliability visualization with fault type annotations
  - Segment-level "next=pre" risk calibration with reliability diagram
  - Non-overlapping interval labels with automatic layout
  - Comprehensive dashboard (9-panel) with distribution/lead-lag/run-length analysis
  - Improved plotting: per-class views + multi-class overview with color-coded segments
  - Multi-class trajectory composition: different classes for different segments
"""

import os
import json
import pickle
import random
from typing import Dict, Any, List, Tuple, Optional
from collections import Counter

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.colors as mcolors
from sklearn.metrics import roc_auc_score, brier_score_loss, roc_curve, precision_recall_curve
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d

# ==============================
#            CONFIG
# ==============================
SAVE_DIR = str(DATA_DIR)
WINDOW_SIZE = 250
STEP_SIZE = 100

# Multi-seed evaluation
RANDOM_SEEDS = [7]
TOP_K = 3

# Target total trajectory length (48 days → 24 segments of 2 days each)
TARGET_DAYS = 48
DAYS_PER_SEGMENT = 2.0
TARGET_SEGMENTS = int(TARGET_DAYS / DAYS_PER_SEGMENT)  # 24 segments

# Markov chain parameters
P_ACTION = 0.30   # pre -> post
P_RELAPSE = 0.10  # post -> pre
P_RUN_LENGTH = 0.20  # geometric run length

# Window sizing for metrics
STRIDE_FRACTION = 0.125      # denser stride (WS/8) for smoother HI
METRIC_WINDOW_DAYS = 10.0

# Detection parameters (enhanced with consensus and prominence)
USE_CONJ_RULE = True
DETECTION_TOLERANCE_DAYS = 1.0
STABILIZATION_WINDOW_DAYS = 5.0
REFRACTORY_DAYS = 7.0         # increased for fewer duplicates
NMS_NEIGHBOR = 1.2            # increased to merge near-duplicates

# Thresholds (will be optimized on calibration set)
PME_P_LOW = 70        # more conservative
DSM_P_LOW = 70
RS_P_MID = 75
RS_P_HIGH = 85
MIN_POS_FRAC = 0.10   # require more positive mass

# Multi-scale detection with consensus
USE_MULTISCALE = True
SCALE_FACTORS = [0.5, 1.0, 2.0]
CONSENSUS_K = 2  # require agreement on K-of-N scales

# Peak filtering
MIN_PEAK_PROMINENCE = 0.05  # relative to MAD
MIN_PEAK_WIDTH = 2  # samples

# HMM smoothing
USE_HMM_SMOOTHING = True

# EMA parameters
EMA_ALPHA_METRICS = 0.3  # stronger smoothing for metrics
EMA_ALPHA_HI = 0.2       # lighter for HI

# Calibration
NUM_CALIBRATION_TRAJECTORIES = 10  # pool across multiple trajectories
LOGISTIC_C = 0.1  # stronger L2 regularization

# Segment-level risk calibration
USE_SEGMENT_RISK_CALIBRATION = True
SEGMENT_RISK_FEATURES = ['pme_avg', 'dsm_avg', 'rs_avg', 'rs_slope', 'rs_var', 'hi_slope']

# Plotting
DO_PLOTS = True
SAVE_FIGS = True
FIGS_DIR = os.path.join(SAVE_DIR, "reliability_figs")
os.makedirs(FIGS_DIR, exist_ok=True)
os.makedirs(os.path.join(FIGS_DIR, "dashboards"), exist_ok=True)
os.makedirs(os.path.join(FIGS_DIR, "classes"), exist_ok=True)
os.makedirs(os.path.join(FIGS_DIR, "overview"), exist_ok=True)

# ==============================
#     HELPERS: IO & LABELS
# ==============================
def _load_pickle(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)

def _fname(stem: str) -> str:
    return os.path.join(SAVE_DIR, f"{stem}_window_{WINDOW_SIZE}_step_{STEP_SIZE}.pkl")

def load_sample_split_artifacts() -> Dict[str, Any]:
    return {
        "train": _load_pickle(_fname("train_pairs_sample_split")),
        "val": _load_pickle(_fname("val_pairs_sample_split")),
        "test": _load_pickle(_fname("test_pairs_sample_split")),
        "pair_mapping": _load_pickle(_fname("pair_mapping_sample_split")),
        "label_mappings": _load_pickle(_fname("label_mappings_sample_split")),
        "scaler": _load_pickle(_fname("scaler_sample_split")),
    }

def _determine_label_field(d: Dict[str, dict]) -> Tuple[str, bool]:
    if len(d) == 0:
        return "class", False
    if any("class" in s for s in d.values()):
        return "class", False
    encoded_keys = ("class_encoded", "label_encoded", "target_class_encoded", "hclass_encoded")
    coverage = {k: sum(1 for s in d.values() if k in s and s[k] is not None and s[k] >= 0) for k in encoded_keys}
    if max(coverage.values()) == 0:
        return "class", False
    best = max(coverage.items(), key=lambda kv: kv[1])[0]
    return best, True

def count_by_field(d: Dict[str, dict], field: str) -> Dict[Any, int]:
    ctr = Counter()
    for s in d.values():
        if field in s:
            ctr[s[field]] += 1
    return dict(sorted(ctr.items(), key=lambda kv: str(kv[0])))

def resolve_name_for_id(id_: int, label_mappings: dict) -> str:
    for key, mapping in label_mappings.items():
        if not isinstance(mapping, dict):
            continue
        rev = {int(v): str(k) for k, v in mapping.items() if isinstance(v, (int, np.integer))}
        if id_ in rev:
            return rev[id_]
    return f"Class_{int(id_)}"

# ==============================
#          SMOOTHING
# ==============================
def ema(series: np.ndarray, alpha: float = 0.2) -> np.ndarray:
    """Exponential moving average."""
    y = np.copy(series).astype(np.float64)
    if len(y) == 0:
        return y
    s = y[0]
    out = np.zeros_like(y)
    out[0] = s
    for i in range(1, len(y)):
        s = alpha * y[i] + (1 - alpha) * s
        out[i] = s
    return out.astype(series.dtype)

# ==============================
#  SEGMENT LIBRARY
# ==============================
def build_segment_library(
    train_data: Dict[str, dict],
    top_classes: List[int],
    label_field: str
) -> Dict[int, Dict[str, List[np.ndarray]]]:
    lib = {c: {'pre': [], 'post': []} for c in top_classes}
    for _, sample in train_data.items():
        cid = sample.get(label_field)
        if cid not in top_classes:
            continue
        x_pre = sample.get('x_before')
        x_post = sample.get('x_after')
        if x_pre is not None and x_post is not None:
            lib[cid]['pre'].append(np.asarray(x_pre, dtype=np.float32))
            lib[cid]['post'].append(np.asarray(x_post, dtype=np.float32))
    return lib

# ==============================
#  SYNTHETIC TRAJECTORY (multi-class composition)
# ==============================
def generate_trajectory_rule_based(
    segment_library: Dict[int, Dict[str, List[np.ndarray]]],
    top_classes: List[int],
    class_frequencies: Dict[int, int],
    target_segments: int,
    seed: int
) -> Dict[str, Any]:
    """
    Rule-based construction(multi-class version):
    1. Assign 24 segments to different classes(ensure every class appears at least once)
    2. Generate all segments as post first
    3. randomly choose an insertion position between segments 10 and 20
    4. Insert 3 consecutive pre segments at that position (they may belong to different classes)
    """
    rng = random.Random(seed)
    np.random.seed(seed)

    total = sum(class_frequencies.values())
    class_probs = {c: class_frequencies[c] / total for c in top_classes}

    # 1. Assign classes to the target_segments segments while ensuring diversity
    # First ensure every class appears at least once
    assigned_classes = list(top_classes)

    # Randomly assign the remaining segments according to class probabilities
    remaining = target_segments - len(top_classes)
    for _ in range(remaining):
        sel_c = rng.choices(top_classes, weights=[class_probs[c] for c in top_classes])[0]
        assigned_classes.append(sel_c)

    # Shuffle the order so the class distribution looks more natural
    rng.shuffle(assigned_classes)

    # 2. Generate target_segments post segments according to the assigned classes
    segs = []
    regs = []
    clss = []

    for cls_id in assigned_classes:
        avail = segment_library[cls_id]['post']
        if len(avail) == 0:
            # If this class has no post samples, skip it
            continue
        seg = rng.choice(avail)
        segs.append(seg)
        regs.append('post')
        clss.append(cls_id)

    # 3. Randomly choose an insertion index between segments 10 and 20 (indices 10-19)
    if len(segs) < 13:  # At least 13 segments are needed to insert 3 segments between 10 and 20
        insert_idx = max(3, len(segs) - 3)
    else:
        insert_idx = rng.randint(10, min(20, len(segs) - 3))

    # 4. Insert 3 consecutive pre segments (possibly from different classes)
    pre_segs = []
    pre_regs = []
    pre_clss = []

    for _ in range(3):
        sel_c = rng.choices(top_classes, weights=[class_probs[c] for c in top_classes])[0]
        avail = segment_library[sel_c]['pre']
        if len(avail) == 0:
            # If no pre sample exists for this class, try a different class
            for alt_c in top_classes:
                if alt_c != sel_c and len(segment_library[alt_c]['pre']) > 0:
                    sel_c = alt_c
                    avail = segment_library[alt_c]['pre']
                    break
        if len(avail) == 0:
            continue
        seg = rng.choice(avail)
        pre_segs.append(seg)
        pre_regs.append('pre')
        pre_clss.append(sel_c)

    # Insert at the chosen position
    segs = segs[:insert_idx] + pre_segs + segs[insert_idx:]
    regs = regs[:insert_idx] + pre_regs + regs[insert_idx:]
    clss = clss[:insert_idx] + pre_clss + clss[insert_idx:]

    # Record class distribution(for verification)
    class_counts = Counter(clss)
    print(f"  [Trajectory composition] Classes used: {dict(class_counts)}")

    # Compute boundaries(positions where the regime changes)
    boundaries = []
    current_pos = 0
    for i in range(len(regs)):
        if i > 0 and regs[i] != regs[i-1]:
            boundaries.append(current_pos)
        current_pos += len(segs[i])

    # Concatenate the trajectory
    if segs:
        traj = np.concatenate(segs, axis=0)
    else:
        traj = np.zeros((0, 1), dtype=np.float32)

    return {
        'trajectory': traj,
        'regimes': regs,
        'classes': clss,
        'boundaries': boundaries,
        'segment_starts': np.cumsum([0] + [len(s) for s in segs[:-1]]).tolist()
    }

# ==============================
#  HI EXTRACTION (denser stride)
# ==============================
def extract_hi_from_trajectory(
    trajectory: np.ndarray,
    model: torch.nn.Module,
    device: torch.device,
    window_size: int = 250
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    T, C = trajectory.shape
    hi = np.zeros(T, dtype=np.float32)
    W = None
    model.eval()
    stride = max(1, window_size // 8)  # denser stride
    with torch.no_grad():
        for t in range(0, T, stride):
            end_t = min(t + window_size, T)
            win = trajectory[t:end_t]
            if len(win) < window_size // 2:
                continue
            if len(win) < window_size:
                pad = window_size - len(win)
                win = np.vstack([win, np.zeros((pad, C), dtype=np.float32)])
            x = torch.from_numpy(win).unsqueeze(0).to(device)
            try:
                hi_b, _, w_b, _, _, _ = model.encoder(x)
                vals = hi_b.cpu().numpy().flatten()
                vlen = min(len(vals), end_t - t)
                hi[t:t+vlen] = vals[:vlen]
                if w_b is not None:
                    wv = w_b.cpu().numpy()
                    if W is None:
                        K = wv.shape[-1]
                        W = np.zeros((T, K), dtype=np.float32)
                    W[t:t+vlen] = wv[0, :vlen]
            except Exception as e:
                print(f"[WARN] HI extraction failed at t={t}: {e}")
                continue
    return hi, W

# ==============================
#  METRICS (PME, SM, ΔSM, RS)
# ==============================
def _ols_slope(x: np.ndarray) -> float:
    n = len(x)
    if n < 2:
        return 0.0
    t = np.arange(n)
    return np.polyfit(t, x, 1)[0]

def compute_pme(hi: np.ndarray, t: int, win_pre: int, win_post: int) -> float:
    if t < win_pre or t + win_post > len(hi):
        return 0.0
    pre = hi[t - win_pre:t]
    post = hi[t:t + win_post]
    if len(pre) < 2 or len(post) < 2:
        return 0.0
    mean_diff = np.mean(post) - np.mean(pre)
    slope_diff = _ols_slope(pre) - _ols_slope(post)
    return float(mean_diff + slope_diff)

def _sm_on_window(weights_slice: np.ndarray, op_groups: Dict[str, List[int]]) -> float:
    """SM = linear - (concave + oscillatory) on one window (row-wise normalized before averaging)."""
    if len(weights_slice) == 0:
        return 0.0
    w_norm = weights_slice / (weights_slice.sum(axis=1, keepdims=True) + 1e-8)
    m_linear = np.mean(w_norm[:, op_groups.get('linear', [])]) if op_groups.get('linear') else 0.0
    m_concave = np.mean(w_norm[:, op_groups.get('concave', [])]) if op_groups.get('concave') else 0.0
    m_osc = np.mean(w_norm[:, op_groups.get('oscillatory', [])]) if op_groups.get('oscillatory') else 0.0
    return float(m_linear - (m_concave + m_osc))

def compute_sm_pair(weights: Optional[np.ndarray], t: int, win_pre: int, win_post: int,
                    op_groups: Dict[str, List[int]]) -> Tuple[Optional[float], Optional[float]]:
    """Return (SM_post, ΔSM = SM_post - SM_pre)."""
    if weights is None or t + win_post > len(weights) or t - win_pre < 0:
        return None, None
    w_pre = weights[t - win_pre:t]
    w_post = weights[t:t + win_post]
    sm_pre = _sm_on_window(w_pre, op_groups)
    sm_post = _sm_on_window(w_post, op_groups)
    return sm_post, (sm_post - sm_pre)

def compute_rs(pme: float, dsm: Optional[float], a0: float, a1: float, a2: float) -> float:
    """Use PME and ΔSM as RS inputs"""
    z = a0 + a1 * pme + (a2 * (0.0 if dsm is None else dsm))
    z = np.clip(z, -50, 50)
    return float(1.0 / (1.0 + np.exp(-z)))

def compute_metric_timeline(
    hi: np.ndarray,
    weights: Optional[np.ndarray],
    win_size: int,
    stride: int,
    op_groups: Dict[str, List[int]],
    a0: float, a1: float, a2: float,
    ema_alpha: float = 0.3
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Returns: PME_tl, SM_post_tl, ΔSM_tl, RS_tl
    """
    T = len(hi)
    win_pre = win_size // 2
    win_post = win_size // 2
    pme_tl = np.zeros(T, dtype=np.float32)
    sm_post_tl = np.zeros(T, dtype=np.float32)
    dsm_tl = np.zeros(T, dtype=np.float32)
    rs_tl = np.zeros(T, dtype=np.float32)

    for t in range(win_pre, T - win_post, stride):
        pme = compute_pme(hi, t, win_pre, win_post)
        sm_post, dsm = compute_sm_pair(weights, t, win_pre, win_post, op_groups)
        rs = compute_rs(pme, dsm, a0, a1, a2)

        pme_tl[t] = pme
        sm_post_tl[t] = 0.0 if sm_post is None else sm_post
        dsm_tl[t] = 0.0 if dsm is None else dsm
        rs_tl[t] = rs

    # EMA smoothing
    pme_tl = ema(pme_tl, ema_alpha)
    sm_post_tl = ema(sm_post_tl, ema_alpha)
    dsm_tl = ema(dsm_tl, ema_alpha)
    rs_tl = ema(rs_tl, ema_alpha)

    # Edge padding
    if win_pre < T:
        for arr in (pme_tl, sm_post_tl, dsm_tl, rs_tl):
            arr[:win_pre] = arr[win_pre]
    if T - win_post > 0:
        for arr in (pme_tl, sm_post_tl, dsm_tl, rs_tl):
            arr[T - win_post:] = arr[T - win_post - 1]

    return pme_tl, sm_post_tl, dsm_tl, rs_tl

# ==============================
#  SEGMENT-LEVEL FEATURES
# ==============================
def make_segment_features(
    hi: np.ndarray,
    pme_tl: np.ndarray,
    dsm_tl: np.ndarray,
    rs_tl: np.ndarray,
    segment_starts: List[int],
    feature_names: List[str]
) -> np.ndarray:
    """Extract segment-level features for risk calibration."""
    features = []
    for i in range(len(segment_starts) - 1):
        s, e = segment_starts[i], segment_starts[i + 1]
        feat_dict = {
            'pme_avg': float(np.mean(pme_tl[s:e])),
            'dsm_avg': float(np.mean(dsm_tl[s:e])),
            'rs_avg': float(np.mean(rs_tl[s:e])),
            'rs_slope': _ols_slope(rs_tl[s:e]),
            'rs_var': float(np.var(rs_tl[s:e])),
            'hi_slope': _ols_slope(hi[s:e])
        }
        features.append([feat_dict[fn] for fn in feature_names])
    return np.array(features)

# ==============================
#  DETECTION (consensus + prominence + HMM)
# ==============================
def _find_peaks_with_prominence(x: np.ndarray, min_prominence: float, min_width: int) -> List[int]:
    """Find peaks with minimum prominence and width."""
    if len(x) < 3:
        return []
    mad = np.median(np.abs(x - np.median(x)))
    prom_threshold = max(min_prominence, mad * min_prominence)
    peaks, properties = find_peaks(x, prominence=prom_threshold, width=min_width)
    return peaks.tolist()

def _nms_1d(idx: List[int], scores: np.ndarray, min_dist: int) -> List[int]:
    if not idx:
        return []
    order = sorted(idx, key=lambda i: scores[i], reverse=True)
    keep = []
    taken = np.zeros_like(scores, dtype=bool)
    for i in order:
        if taken[i]:
            continue
        keep.append(i)
        l = max(0, i - min_dist)
        r = min(len(scores), i + min_dist + 1)
        taken[l:r] = True
    return sorted(keep)

def detect_boundaries_consensus(
    pme_tl: np.ndarray,
    dsm_tl: np.ndarray,
    rs_tl: np.ndarray,
    tau_pme_low: float,
    tau_dsm_low: float,
    tau_rs_mid: float,
    tau_rs_high: float,
    refractory: int,
    nms_neighbor: int,
    min_prominence: float,
    min_width: int
) -> List[int]:
    """
    Stronger evidence gates:
      - High confidence: RS > τ_high AND PME > τ_pme AND ΔSM > τ_dsm
      - Medium confidence: RS > τ_mid AND (PME > τ_pme OR ΔSM > τ_dsm) AND peak shape test
    """
    peaks = _find_peaks_with_prominence(rs_tl, min_prominence, min_width)
    cand = []

    for i in peaks:
        # High confidence: strict AND logic
        high_conf = (rs_tl[i] > tau_rs_high and
                    pme_tl[i] > tau_pme_low and
                    dsm_tl[i] > tau_dsm_low)

        # Medium confidence: OR logic with peak shape test
        if not high_conf:
            has_metric = pme_tl[i] > tau_pme_low or dsm_tl[i] > tau_dsm_low
            # Peak shape: RS slope positive before, negative after
            slope_before = rs_tl[i] - rs_tl[max(0, i-3)] if i >= 3 else 0
            slope_after = rs_tl[min(len(rs_tl)-1, i+3)] - rs_tl[i] if i < len(rs_tl)-3 else 0
            peak_shape = slope_before > 0 and slope_after < 0
            mid_conf = rs_tl[i] > tau_rs_mid and has_metric and peak_shape
        else:
            mid_conf = False

        if high_conf or mid_conf:
            cand.append(i)

    cand = _nms_1d(cand, rs_tl, nms_neighbor)

    # Refractory period
    detected = []
    last = -10**9
    for i in cand:
        if i - last >= refractory:
            detected.append(i)
            last = i
    return detected

def detect_boundaries_multiscale_consensus(
    hi: np.ndarray,
    weights: Optional[np.ndarray],
    op_groups: Dict[str, List[int]],
    base_win: int,
    stride: int,
    scale_factors: List[float],
    consensus_k: int,
    a0: float, a1: float, a2: float,
    tau_pme_low: float,
    tau_dsm_low: float,
    tau_rs_mid: float,
    tau_rs_high: float,
    refractory: int,
    nms_neighbor: int,
    min_prominence: float,
    min_width: int,
    ema_alpha: float = 0.3
) -> List[int]:
    """Multi-scale detection with K-of-N consensus."""
    scale_detections = []
    scale_scores = []

    for scale in scale_factors:
        win = int(base_win * scale)
        if win < 10:
            continue

        pme_tl, sm_post_tl, dsm_tl, rs_tl = compute_metric_timeline(
            hi, weights, win, stride, op_groups, a0, a1, a2, ema_alpha
        )

        detected = detect_boundaries_consensus(
            pme_tl, dsm_tl, rs_tl,
            tau_pme_low=tau_pme_low, tau_dsm_low=tau_dsm_low,
            tau_rs_mid=tau_rs_mid, tau_rs_high=tau_rs_high,
            refractory=refractory, nms_neighbor=nms_neighbor,
            min_prominence=min_prominence, min_width=min_width
        )

        scale_detections.append(detected)
        scale_scores.append(rs_tl)

    if not scale_detections:
        return []

    # Consensus voting: require K-of-N agreement
    all_candidates = set()
    for det in scale_detections:
        all_candidates.update(det)

    consensus_detections = []
    for cand in sorted(all_candidates):
        votes = 0
        for det in scale_detections:
            # Check if any detection in this scale is within tolerance
            if any(abs(d - cand) <= nms_neighbor for d in det):
                votes += 1

        if votes >= consensus_k:
            consensus_detections.append(cand)

    # Merge nearby consensus detections
    if not consensus_detections:
        return []

    merged = []
    current_cluster = [consensus_detections[0]]

    for i in range(1, len(consensus_detections)):
        if consensus_detections[i] - current_cluster[-1] <= nms_neighbor:
            current_cluster.append(consensus_detections[i])
        else:
            merged.append(int(np.median(current_cluster)))
            current_cluster = [consensus_detections[i]]

    if current_cluster:
        merged.append(int(np.median(current_cluster)))

    return merged

def hmm_smooth_detections(
    detected: List[int],
    rs_tl: np.ndarray,
    p_action: float,
    p_relapse: float,
    trajectory_length: int
) -> List[int]:
    """Simple HMM smoothing to enforce realistic run lengths."""
    if len(detected) < 2:
        return detected

    # Build state sequence: 0=post, 1=pre
    states = np.zeros(trajectory_length, dtype=int)
    for i in range(len(detected)):
        if i % 2 == 0:  # even transitions are post->pre
            start = detected[i]
            end = detected[i+1] if i+1 < len(detected) else trajectory_length
            states[start:end] = 1

    # Viterbi-like smoothing: penalize short runs
    min_run_length = int(-np.log(p_action) / np.log(1 - p_action))  # expected run length
    smoothed = []
    current_state = 0
    run_start = 0

    for i in range(trajectory_length):
        if states[i] != current_state:
            run_length = i - run_start
            if run_length >= min_run_length or current_state == 0:  # keep long runs or post states
                if current_state == 1 and run_start > 0:
                    smoothed.append(run_start)
            current_state = states[i]
            run_start = i

    return smoothed

def evaluate_detection(true_boundaries: List[int], detected_boundaries: List[int], tolerance: int) -> Dict[str, Any]:
    if len(true_boundaries) == 0:
        return {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'mabe': 0.0, 'tp': 0, 'fp': 0, 'fn': 0, 'lags': []}
    true_matched = set()
    det_matched = set()
    errs = []
    lags = []
    for det in detected_boundaries:
        for j, tru in enumerate(true_boundaries):
            if abs(det - tru) <= tolerance and j not in true_matched:
                true_matched.add(j)
                det_matched.add(det)
                errs.append(abs(det - tru))
                lags.append(det - tru)
                break
    tp = len(true_matched)
    fp = len(detected_boundaries) - len(det_matched)
    fn = len(true_boundaries) - len(true_matched)
    precision = tp / max(1, (tp + fp))
    recall = tp / max(1, (tp + fn))
    f1 = 2 * precision * recall / max(1e-8, (precision + recall))
    mabe = float(np.mean(errs)) if errs else 0.0
    return {'precision': precision, 'recall': recall, 'f1': f1, 'mabe': mabe, 'tp': tp, 'fp': fp, 'fn': fn, 'lags': lags}

# ==============================
#  STABILIZATION & RISK
# ==============================
def compute_stabilization_time(sm_post_tl: np.ndarray, rs_tl: np.ndarray, detected: List[int],
                               tau_sm_post: float, tau_rs: float, delta_window: int) -> List[int]:
    tts = []
    for b in detected:
        for t in range(b, len(sm_post_tl) - delta_window):
            if np.all(sm_post_tl[t:t + delta_window] > tau_sm_post) and np.all(rs_tl[t:t + delta_window] > tau_rs):
                tts.append(t - b)
                break
    return tts

def evaluate_risk_calibration(rs_tl: np.ndarray, true_regimes: List[str], segment_starts: List[int]) -> Dict[str, float]:
    """
    Risk is defined as whether the next segment returns to pre(1=returns to pre).Use 1 - RS as the risk score to keep the direction consistent.
    """
    labels, scores = [], []
    for i in range(len(segment_starts) - 1):
        s, e = segment_starts[i], segment_starts[i + 1]
        rs_avg = float(np.mean(rs_tl[s:e]))
        scores.append(1.0 - rs_avg)
        nxt = true_regimes[i + 1] if i + 1 < len(true_regimes) else 'post'
        labels.append(1 if nxt == 'pre' else 0)
    if len(set(labels)) < 2:
        return {'auc': 0.5, 'brier': 0.25}
    auc = roc_auc_score(labels, scores)
    brier = brier_score_loss(labels, scores)
    return {'auc': float(auc), 'brier': float(brier)}

# ==============================
#  RS CALIBRATION (robust, pooled)
# ==============================
def calibrate_rs_robust_pooled(
    calibration_data: List[Tuple[np.ndarray, np.ndarray, List[str], List[int]]]
) -> Tuple[float, float, float]:
    """
    Robust RS calibration pooled across multiple trajectories:
      - IQR-based standardization
      - Stronger L2 regularization
      - Winsorization to prevent outliers
    """
    X_raw, y = [], []

    for pme_tl, dsm_tl, regimes, seg_starts in calibration_data:
        for i in range(len(seg_starts) - 1):
            s, e = seg_starts[i], seg_starts[i + 1]
            pme_avg = float(np.mean(pme_tl[s:e]))
            dsm_avg = float(np.mean(dsm_tl[s:e]))
            X_raw.append([pme_avg, dsm_avg])
            nxt = regimes[i + 1] if i + 1 < len(regimes) else 'post'
            y.append(1 if nxt == 'post' else 0)  # 1=reliable (post), 0=risky (pre)

    if len(set(y)) < 2 or len(X_raw) < 10:
        return (-1.0, 5.0, 3.0)

    X_raw = np.array(X_raw)

    # Winsorize to 1st-99th percentile
    for col in range(X_raw.shape[1]):
        p1, p99 = np.percentile(X_raw[:, col], [1, 99])
        X_raw[:, col] = np.clip(X_raw[:, col], p1, p99)

    # IQR-based standardization
    q25 = np.percentile(X_raw, 25, axis=0, keepdims=True)
    q75 = np.percentile(X_raw, 75, axis=0, keepdims=True)
    iqr = q75 - q25 + 1e-8
    median = np.median(X_raw, axis=0, keepdims=True)
    X = (X_raw - median) / iqr

    try:
        clf = LogisticRegression(
            max_iter=1000,
            solver="lbfgs",
            C=LOGISTIC_C,  # stronger regularization
            class_weight='balanced'
        )
        clf.fit(X, np.array(y))
        a0 = float(clf.intercept_[0])
        a1 = float(clf.coef_[0][0])
        a2 = float(clf.coef_[0][1])

        # Map back to original scale
        med1, med2 = float(median[0, 0]), float(median[0, 1])
        iqr1, iqr2 = float(iqr[0, 0]), float(iqr[0, 1])
        a1_prime = a1 / iqr1
        a2_prime = a2 / iqr2
        a0_prime = a0 - a1 * (med1 / iqr1) - a2 * (med2 / iqr2)

        return (float(a0_prime), float(a1_prime), float(a2_prime))
    except Exception as e:
        print(f"[WARN] Robust calibration failed: {e}")
        return (-1.0, 5.0, 3.0)

# ==============================
#  SEGMENT-LEVEL RISK CALIBRATION
# ==============================
def calibrate_segment_risk(
    calibration_data: List[Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, List[str], List[int]]],
    feature_names: List[str]
) -> Optional[LogisticRegression]:
    """
    Calibrate segment-level risk predictor: P(next_regime='pre' | segment_features).
    Returns trained LogisticRegression model.
    """
    X_all, y_all = [], []

    for hi, pme_tl, dsm_tl, rs_tl, regimes, seg_starts in calibration_data:
        X_seg = make_segment_features(hi, pme_tl, dsm_tl, rs_tl, seg_starts, feature_names)
        for i in range(len(seg_starts) - 1):
            nxt = regimes[i + 1] if i + 1 < len(regimes) else 'post'
            y_all.append(1 if nxt == 'pre' else 0)
        X_all.append(X_seg)

    if not X_all:
        return None

    X = np.vstack(X_all)
    y = np.array(y_all)

    if len(set(y)) < 2 or len(X) < 10:
        return None

    # Winsorize
    for col in range(X.shape[1]):
        p1, p99 = np.percentile(X[:, col], [1, 99])
        X[:, col] = np.clip(X[:, col], p1, p99)

    # Standardize
    X_mean = np.mean(X, axis=0, keepdims=True)
    X_std = np.std(X, axis=0, keepdims=True) + 1e-8
    X = (X - X_mean) / X_std

    try:
        clf = LogisticRegression(
            max_iter=1000,
            solver='lbfgs',
            C=LOGISTIC_C,
            class_weight='balanced'
        )
        clf.fit(X, y)
        # Store normalization params
        clf.X_mean_ = X_mean
        clf.X_std_ = X_std
        return clf
    except Exception as e:
        print(f"[WARN] Segment risk calibration failed: {e}")
        return None

# ==============================
#  NON-OVERLAPPING INTERVAL LABELS
# ==============================
def layout_interval_labels(segments: List[Dict[str, Any]], min_gap: int = 10) -> Tuple[List[int], int]:
    """
    segments: List[dict] with keys {start, end, text}
    Return the y-level for each segment(0,1,2,...)for vertical staggering, avoid overlap.
    Greedy sweep-line layout with multiple occupancy layers.
    """
    layers = []  # Rightmost occupied boundary for each layer
    levels = []
    for seg in sorted(segments, key=lambda s: (s['start'], s['end'])):
        placed = False
        for li, rightmost in enumerate(layers):
            if seg['start'] > rightmost + min_gap:
                layers[li] = seg['end']
                levels.append(li)
                placed = True
                break
        if not placed:
            layers.append(seg['end'])
            levels.append(len(layers) - 1)
    return levels, len(layers)

# ==============================
#  PLOTTING FUNCTIONS
# ==============================
def plot_trajectory_with_annotations(
    hi: np.ndarray,
    pme_tl: np.ndarray,
    dsm_tl: np.ndarray,
    true_boundaries: List[int],
    detected_boundaries: List[int],
    regimes: List[str],
    classes: List[int],
    segment_starts: List[int],
    label_mappings: dict,
    thresholds: Dict[str, float],
    seed: int,
    save_path: Optional[str] = None
):
    """
    Overview plot: multi-class stitched trajectory with class-specific background colors and a light red overlay for pre/post regimes
    """
    # Assign a distinct base color to each class
    base_colors = ['lightblue', 'lightgreen', 'lightyellow', 'lightcoral', 'lightcyan', 'lavender']
    unique_classes = sorted(set(classes))
    class_color_map = {cls: base_colors[i % len(base_colors)] for i, cls in enumerate(unique_classes)}

    fig, axes = plt.subplots(3, 1, figsize=(16, 9), sharex=True)

    # (1) HI with class-colored background + pre/post overlay
    ax = axes[0]
    for i in range(len(segment_starts)):
        start = segment_starts[i]
        end = segment_starts[i+1] if i+1 < len(segment_starts) else len(hi)
        cls = classes[i]
        regime = regimes[i]

        # Base class color
        base_color = class_color_map.get(cls, 'whitesmoke')
        ax.axvspan(start, end, color=base_color, alpha=0.4)

        # Add a light red overlay for pre segments
        if regime == 'pre':
            ax.axvspan(start, end, color='red', alpha=0.15)

    ax.plot(hi, lw=1.2, color='darkblue', alpha=0.8, label='Health Index')

    # Annotate boundaries
    for b in true_boundaries:
        ax.axvline(b, color='red', ls=':', alpha=0.6, lw=1.5, label='True' if b == true_boundaries[0] else '')
    for d in detected_boundaries:
        ax.axvline(d, color='green', ls='--', alpha=0.8, lw=1.5, label='Detected' if d == detected_boundaries[0] else '')

    # Top labels:ClassX(post) / ClassX(pre)
    segments_for_layout = []
    for i in range(len(segment_starts)):
        start = segment_starts[i]
        end = segment_starts[i+1] if i+1 < len(segment_starts) else len(hi)
        if end - start >= 150:  # label only sufficiently long segments
            cls = classes[i]
            regime = regimes[i]
            class_name = resolve_name_for_id(int(cls), label_mappings)
            segments_for_layout.append({
                'start': start,
                'end': end,
                'text': f'{class_name}({regime})',
                'color': class_color_map.get(cls, 'gray')
            })

    # Non-overlapping label layout
    if segments_for_layout:
        levels, n_layers = layout_interval_labels(segments_for_layout, min_gap=50)
        ylim = ax.get_ylim()
        dy = (ylim[1] - ylim[0]) * 0.06
        for seg, lv in zip(segments_for_layout, levels):
            mid = 0.5 * (seg['start'] + seg['end'])
            ax.text(mid, ylim[1] + dy * (lv + 0.5), seg['text'],
                   ha='center', va='bottom', fontsize=6, rotation=0,
                   bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.85, edgecolor=seg['color']),
                   clip_on=False)

    ax.set_ylabel('Health Index', fontsize=10)
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(alpha=0.3)

    # (2) PME & ΔSM
    ax = axes[1]
    ax.bar(np.arange(len(pme_tl)), pme_tl, width=1.0, alpha=0.6, label='PME', color='tab:blue')
    ax.bar(np.arange(len(dsm_tl)), dsm_tl, width=1.0, alpha=0.4, label='ΔSM', color='tab:purple')
    ax.axhline(thresholds['tau_pme_low'], color='tab:blue', ls='--', alpha=0.7, label='τ_PME')
    ax.axhline(thresholds['tau_dsm_low'], color='tab:purple', ls='--', alpha=0.7, label='τ_ΔSM')
    ax.set_ylabel('Metric Values', fontsize=10)
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(alpha=0.3)

    # (3) Class segments timeline
    ax = axes[2]
    for i in range(len(segment_starts)):
        start = segment_starts[i]
        end = segment_starts[i+1] if i+1 < len(segment_starts) else len(hi)
        cls = classes[i]
        regime = regimes[i]
        color = class_color_map.get(cls, 'gray')
        alpha = 0.5 if regime == 'post' else 0.3
        ax.axvspan(start, end, color=color, alpha=alpha)

    ax.set_ylabel('Class Segments', fontsize=10)
    ax.set_xlabel('Time Index', fontsize=10)
    ax.set_ylim([0, 1])
    ax.set_yticks([])
    ax.grid(alpha=0.3)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  [Saved] {save_path}")

    plt.show()

# ==============================
#  MAIN
# ==============================
if __name__ == "__main__":
    print("[INFO] Starting long-horizon synthetic trajectory analysis...")
    print("[INFO] Enhanced with robust calibration, consensus fusion, and HMM smoothing")
    print("[INFO] Multi-class trajectory composition enabled")

    # 1) Load artifacts
    artifacts = load_sample_split_artifacts()
    train_data = artifacts["train"]
    label_mappings = artifacts.get("label_mappings", {})

    # 2) Top-K classes
    label_field, is_encoded = _determine_label_field(train_data)
    freq = count_by_field(train_data, label_field)
    top_items = sorted(freq.items(), key=lambda kv: kv[1], reverse=True)[:TOP_K]
    top_classes = [k for k, _ in top_items]
    class_frequencies = {k: v for k, v in top_items}
    print("[INFO] Top {} classes:".format(TOP_K))
    for k, v in top_items:
        name = resolve_name_for_id(int(k), label_mappings) if is_encoded else k
        print(f"  - {name} ({k}): {v} samples")

    # 3) Segment library
    print("[INFO] Building segment library...")
    segment_library = build_segment_library(train_data, top_classes, label_field)
    for c in top_classes:
        print(f"  Class {c}: {len(segment_library[c]['pre'])} pre, {len(segment_library[c]['post'])} post")

    # 4) Load model
    print("[INFO] Loading model...")
    ckpt_path = os.path.join(SAVE_DIR, "best_model_NGFAID_250_100_6.pth")
    meta_path = ckpt_path.replace('.pth', '_meta.json')
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if os.path.exists(meta_path):
        with open(meta_path, 'r') as f:
            meta_data = json.load(f)
        K = meta_data.get('K', len(top_classes))
        C = meta_data.get('sensor_dim', segment_library[top_classes[0]]['pre'][0].shape[1])
        print(f"[INFO] Loaded metadata: K={K}, C={C}")
    else:
        first_segment = segment_library[top_classes[0]]['pre'][0]
        C = first_segment.shape[1]
        K = len(top_classes)
        print(f"[INFO] Inferred from data: K={K}, C={C}")

    model = DiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=K).to(device)
    state_dict = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state_dict, strict=False)
    model.eval()
    print("[INFO] Model loaded successfully")

    # 5) Operator groups
    op_groups = {
        'linear': [0, 1, 7],
        'concave': [2, 3, 5],
        'oscillatory': [4, 6]
    }

    # -------- Pooled RS calibration --------
    print(f"\n[INFO] Generating {NUM_CALIBRATION_TRAJECTORIES} calibration trajectories...")
    calibration_data = []
    for cal_seed in range(1000, 1000 + NUM_CALIBRATION_TRAJECTORIES):
        traj = generate_trajectory_rule_based(segment_library, top_classes, class_frequencies,
                                              TARGET_SEGMENTS, cal_seed)
        arr = traj['trajectory']
        regimes = traj['regimes']
        seg_starts = traj['segment_starts']

        hi, W = extract_hi_from_trajectory(arr, model, device, window_size=WINDOW_SIZE)
        spd = len(arr) / (len(traj['regimes']) * DAYS_PER_SEGMENT)
        win = int(METRIC_WINDOW_DAYS * spd)
        stride = max(1, int(STRIDE_FRACTION * spd))
        pme, sm_post, dsm, _ = compute_metric_timeline(
            hi, W, win, stride, op_groups,
            a0=-1.0, a1=5.0, a2=3.0, ema_alpha=EMA_ALPHA_METRICS
        )
        calibration_data.append((pme, dsm, regimes, seg_starts))

    a0, a1, a2 = calibrate_rs_robust_pooled(calibration_data)
    print(f"[INFO] Calibrated RS (pooled, robust): a0={a0:.3f}, a1={a1:.3f}, a2={a2:.3f}")

    results = []

    # -------- Process each trajectory --------
    for seed in RANDOM_SEEDS:
        print(f"\n[INFO] Processing trajectory for seed {seed}...")

        traj = generate_trajectory_rule_based(segment_library, top_classes, class_frequencies,
                                              TARGET_SEGMENTS, seed)
        arr = traj['trajectory']
        true_bounds = traj['boundaries']
        regimes = traj['regimes']
        classes = traj['classes']
        seg_starts = traj['segment_starts']
        print(f"  Generated trajectory: {len(arr)} samples, {len(true_bounds)} boundaries")

        hi, W = extract_hi_from_trajectory(arr, model, device, window_size=WINDOW_SIZE)
        spd = len(arr) / (len(traj['regimes']) * DAYS_PER_SEGMENT)
        win = int(METRIC_WINDOW_DAYS * spd)
        stride = max(1, int(STRIDE_FRACTION * spd))
        pme_tl, sm_post_tl, dsm_tl, rs_tl = compute_metric_timeline(
            hi, W, win, stride, op_groups, a0=a0, a1=a1, a2=a2, ema_alpha=EMA_ALPHA_METRICS
        )

        # Adaptive thresholds (full distribution with winsorization)
        def robust_percentile_full(x: np.ndarray, p: int) -> float:
            valid = x[np.isfinite(x)]
            if len(valid) < 10:
                return 0.0
            # Winsorize
            p1, p99 = np.percentile(valid, [1, 99])
            clipped = np.clip(valid, p1, p99)
            return float(np.percentile(clipped, p))

        tau_pme_low = robust_percentile_full(pme_tl, PME_P_LOW)
        tau_dsm_low = robust_percentile_full(dsm_tl, DSM_P_LOW)
        tau_rs_mid = robust_percentile_full(rs_tl, RS_P_MID)
        tau_rs_high = robust_percentile_full(rs_tl, RS_P_HIGH)

        # Detection
        tol = int(DETECTION_TOLERANCE_DAYS * spd)
        refractory = int(REFRACTORY_DAYS * spd)
        nms_nb = int(NMS_NEIGHBOR * spd)
        min_prom = MIN_PEAK_PROMINENCE
        min_width = MIN_PEAK_WIDTH

        if USE_MULTISCALE:
            detected = detect_boundaries_multiscale_consensus(
                hi, W, op_groups, win, stride, SCALE_FACTORS, CONSENSUS_K,
                a0, a1, a2,
                tau_pme_low=tau_pme_low, tau_dsm_low=tau_dsm_low,
                tau_rs_mid=tau_rs_mid, tau_rs_high=tau_rs_high,
                refractory=refractory, nms_neighbor=nms_nb,
                min_prominence=min_prom, min_width=min_width
            )
        else:
            detected = detect_boundaries_consensus(
                pme_tl, dsm_tl, rs_tl,
                tau_pme_low=tau_pme_low, tau_dsm_low=tau_dsm_low,
                tau_rs_mid=tau_rs_mid, tau_rs_high=tau_rs_high,
                refractory=refractory, nms_neighbor=nms_nb,
                min_prominence=min_prom, min_width=min_width
            )

        # HMM smoothing
        if USE_HMM_SMOOTHING and detected:
            detected = hmm_smooth_detections(detected, rs_tl, P_ACTION, P_RELAPSE, len(arr))

        # Evaluation
        det_metrics = evaluate_detection(true_boundaries=true_bounds,
                                         detected_boundaries=detected,
                                         tolerance=tol)
        print(f"  Detection - P: {det_metrics['precision']:.3f}, "
              f"R: {det_metrics['recall']:.3f}, "
              f"F1: {det_metrics['f1']:.3f}, "
              f"MABE: {det_metrics['mabe']:.1f}")

        # Stabilization
        delta_samples = int(STABILIZATION_WINDOW_DAYS * spd)
        tts_vals = compute_stabilization_time(sm_post_tl, rs_tl, detected,
                                              tau_sm_post=robust_percentile_full(sm_post_tl, 50),
                                              tau_rs=tau_rs_mid, delta_window=delta_samples)
        if tts_vals:
            print(f"  Stabilization - Median TTS: {np.median(tts_vals):.1f} "
                  f"[{np.percentile(tts_vals,25):.1f}-{np.percentile(tts_vals,75):.1f}]")

        # Risk calibration
        risk_metrics = evaluate_risk_calibration(rs_tl, regimes, seg_starts)
        print(f"  Risk calibration - AUC: {risk_metrics['auc']:.3f}, Brier: {risk_metrics['brier']:.3f}")

        results.append({
            'seed': seed,
            'detection': det_metrics,
            'tts': tts_vals,
            'risk': risk_metrics,
            'hi_timeline': hi,
            'pme_timeline': pme_tl,
            'sm_post_timeline': sm_post_tl,
            'dsm_timeline': dsm_tl,
            'rs_timeline': rs_tl,
            'true_boundaries': true_bounds,
            'detected_boundaries': detected,
            'regimes': regimes,
            'classes': classes,
            'segment_starts': seg_starts,
            'thresholds': {
                'tau_pme_low': tau_pme_low,
                'tau_dsm_low': tau_dsm_low,
                'tau_rs_mid': tau_rs_mid,
                'tau_rs_high': tau_rs_high
            },
            'samples_per_day': spd
        })

        # Plotting with annotations (overview plot)
        if DO_PLOTS:
            save_path = os.path.join(FIGS_DIR, 'overview', f'trajectory_seed_{seed}_overview.png') if SAVE_FIGS else None
            plot_trajectory_with_annotations(
                hi, pme_tl, dsm_tl,
                true_bounds, detected,
                regimes, classes, seg_starts,
                label_mappings,
                results[-1]['thresholds'],
                seed, save_path
            )

    # Summary
    print("\n" + "="*80)
    print("SUMMARY STATISTICS (Enhanced with Multi-Class Composition)")
    print("="*80)
    all_precision = [r['detection']['precision'] for r in results]
    all_recall = [r['detection']['recall'] for r in results]
    all_f1 = [r['detection']['f1'] for r in results]
    all_mabe = [r['detection']['mabe'] for r in results]
    print("Boundary Detection (Consensus + HMM):")
    print(f"  Precision: {np.mean(all_precision):.3f} ± {np.std(all_precision):.3f}")
    print(f"  Recall:    {np.mean(all_recall):.3f} ± {np.std(all_recall):.3f}")
    print(f"  F1-Score:  {np.mean(all_f1):.3f} ± {np.std(all_f1):.3f}")
    print(f"  MABE:      {np.mean(all_mabe):.1f} ± {np.std(all_mabe):.1f} samples")

    all_tts = [t for r in results for t in r['tts']]
    if all_tts:
        print("\nStabilization:")
        print(f"  Median TTS: {np.median(all_tts):.1f} samples")
        print(f"  IQR: [{np.percentile(all_tts,25):.1f}, {np.percentile(all_tts,75):.1f}]")

    all_auc = [r['risk']['auc'] for r in results]
    all_brier = [r['risk']['brier'] for r in results]
    print("\nRisk Calibration (Robust, Pooled):")
    print(f"  AUC:   {np.mean(all_auc):.3f} ± {np.std(all_auc):.3f}")
    print(f"  Brier: {np.mean(all_brier):.3f} ± {np.std(all_brier):.3f}")

    # Save
    output_path = os.path.join(SAVE_DIR, 'long_horizon_results_enhanced.pkl')
    with open(output_path, 'wb') as f:
        pickle.dump({
            'config': {
                'seeds': RANDOM_SEEDS,
                'calibration_trajectories': NUM_CALIBRATION_TRAJECTORIES,
                'consensus_k': CONSENSUS_K,
                'use_hmm': USE_HMM_SMOOTHING,
                'logistic_c': LOGISTIC_C,
                'rs_coefficients': {'a0': a0, 'a1': a1, 'a2': a2}
            },
            'results': results
        }, f)
    print(f"\n[INFO] Results saved to {output_path}")
    print("="*80 + "\n")


In [ ]:
# -*- coding: utf-8 -*-
"""
Long-horizon synthetic trajectory generation and reliability analysis
ΔSM + graded conjunction rule + refractory/NMS + EMA smoothing + RS z-score logistic calibration (monotonic)
Enhanced with:
  - Robust RS calibration (pooled across seeds, IQR-based standardization, shrinkage)
  - Multi-scale consensus fusion (K-of-N agreement instead of union)
  - Peak prominence & width filtering
  - Adaptive thresholding with full distribution
  - Stronger evidence gates (AND logic for high-confidence)
  - HMM smoothing for realistic run lengths
  - Per-class reliability visualization with fault type annotations
  - Segment-level "next=pre" risk calibration with reliability diagram
  - Non-overlapping interval labels with automatic layout
  - Comprehensive dashboard (9-panel) with distribution/lead-lag/run-length analysis
  - Improved plotting: per-class views + multi-class overview with color-coded segments
  - Multi-class trajectory composition: different classes for different segments
  - Single pre sample from different class insertion
"""

import os
import json
import pickle
import random
from typing import Dict, Any, List, Tuple, Optional
from collections import Counter

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.colors as mcolors
from sklearn.metrics import roc_auc_score, brier_score_loss, roc_curve, precision_recall_curve
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d

# ==============================
#            CONFIG
# ==============================
SAVE_DIR = str(DATA_DIR)
WINDOW_SIZE = 250
STEP_SIZE = 100

# Multi-seed evaluation
RANDOM_SEEDS = [7]
TOP_K = 3

# Target total trajectory length (48 days → 24 segments of 2 days each)
TARGET_DAYS = 36
DAYS_PER_SEGMENT = 2.0
TARGET_SEGMENTS = int(TARGET_DAYS / DAYS_PER_SEGMENT)  # 24 segments

# Markov chain parameters
P_ACTION = 0.20   # pre -> post
P_RELAPSE = 0.10  # post -> pre
P_RUN_LENGTH = 0.20  # geometric run length

# Window sizing for metrics
STRIDE_FRACTION = 0.125      # denser stride (WS/8) for smoother HI
METRIC_WINDOW_DAYS = 10.0

# Detection parameters (enhanced with consensus and prominence)
USE_CONJ_RULE = True
DETECTION_TOLERANCE_DAYS = 1.0
STABILIZATION_WINDOW_DAYS = 5.0
REFRACTORY_DAYS = 7.0         # increased for fewer duplicates
NMS_NEIGHBOR = 1.2            # increased to merge near-duplicates

# Thresholds (will be optimized on calibration set)
PME_P_LOW = 70        # more conservative
DSM_P_LOW = 70
RS_P_MID = 75
RS_P_HIGH = 85
MIN_POS_FRAC = 0.10   # require more positive mass

# Multi-scale detection with consensus
USE_MULTISCALE = True
SCALE_FACTORS = [0.5, 1.0, 2.0]
CONSENSUS_K = 2  # require agreement on K-of-N scales

# Peak filtering
MIN_PEAK_PROMINENCE = 0.05  # relative to MAD
MIN_PEAK_WIDTH = 2  # samples

# HMM smoothing
USE_HMM_SMOOTHING = True

# EMA parameters
EMA_ALPHA_METRICS = 0.3  # stronger smoothing for metrics
EMA_ALPHA_HI = 0.2       # lighter for HI

# Calibration
NUM_CALIBRATION_TRAJECTORIES = 10  # pool across multiple trajectories
LOGISTIC_C = 0.1  # stronger L2 regularization

# Segment-level risk calibration
USE_SEGMENT_RISK_CALIBRATION = True
SEGMENT_RISK_FEATURES = ['pme_avg', 'dsm_avg', 'rs_avg', 'rs_slope', 'rs_var', 'hi_slope']

# Plotting
DO_PLOTS = True
SAVE_FIGS = True
FIGS_DIR = os.path.join(SAVE_DIR, "reliability_figs")
os.makedirs(FIGS_DIR, exist_ok=True)
os.makedirs(os.path.join(FIGS_DIR, "dashboards"), exist_ok=True)
os.makedirs(os.path.join(FIGS_DIR, "classes"), exist_ok=True)
os.makedirs(os.path.join(FIGS_DIR, "overview"), exist_ok=True)

# ==============================
#     HELPERS: IO & LABELS
# ==============================
def _load_pickle(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)

def _fname(stem: str) -> str:
    return os.path.join(SAVE_DIR, f"{stem}_window_{WINDOW_SIZE}_step_{STEP_SIZE}.pkl")

def load_sample_split_artifacts() -> Dict[str, Any]:
    return {
        "train": _load_pickle(_fname("train_pairs_sample_split")),
        "val": _load_pickle(_fname("val_pairs_sample_split")),
        "test": _load_pickle(_fname("test_pairs_sample_split")),
        "pair_mapping": _load_pickle(_fname("pair_mapping_sample_split")),
        "label_mappings": _load_pickle(_fname("label_mappings_sample_split")),
        "scaler": _load_pickle(_fname("scaler_sample_split")),
    }

def _determine_label_field(d: Dict[str, dict]) -> Tuple[str, bool]:
    if len(d) == 0:
        return "class", False
    if any("class" in s for s in d.values()):
        return "class", False
    encoded_keys = ("class_encoded", "label_encoded", "target_class_encoded", "hclass_encoded")
    coverage = {k: sum(1 for s in d.values() if k in s and s[k] is not None and s[k] >= 0) for k in encoded_keys}
    if max(coverage.values()) == 0:
        return "class", False
    best = max(coverage.items(), key=lambda kv: kv[1])[0]
    return best, True

def count_by_field(d: Dict[str, dict], field: str) -> Dict[Any, int]:
    ctr = Counter()
    for s in d.values():
        if field in s:
            ctr[s[field]] += 1
    return dict(sorted(ctr.items(), key=lambda kv: str(kv[0])))

def resolve_name_for_id(id_: int, label_mappings: dict) -> str:
    for key, mapping in label_mappings.items():
        if not isinstance(mapping, dict):
            continue
        rev = {int(v): str(k) for k, v in mapping.items() if isinstance(v, (int, np.integer))}
        if id_ in rev:
            return rev[id_]
    return f"Class_{int(id_)}"

# ==============================
#          SMOOTHING
# ==============================
def ema(series: np.ndarray, alpha: float = 0.2) -> np.ndarray:
    """Exponential moving average."""
    y = np.copy(series).astype(np.float64)
    if len(y) == 0:
        return y
    s = y[0]
    out = np.zeros_like(y)
    out[0] = s
    for i in range(1, len(y)):
        s = alpha * y[i] + (1 - alpha) * s
        out[i] = s
    return out.astype(series.dtype)

# ==============================
#  SEGMENT LIBRARY
# ==============================
def build_segment_library(
    train_data: Dict[str, dict],
    top_classes: List[int],
    label_field: str
) -> Dict[int, Dict[str, List[np.ndarray]]]:
    lib = {c: {'pre': [], 'post': []} for c in top_classes}
    for _, sample in train_data.items():
        cid = sample.get(label_field)
        if cid not in top_classes:
            continue
        x_pre = sample.get('x_before')
        x_post = sample.get('x_after')
        if x_pre is not None and x_post is not None:
            lib[cid]['pre'].append(np.asarray(x_pre, dtype=np.float32))
            lib[cid]['post'].append(np.asarray(x_post, dtype=np.float32))
    return lib

# ==============================
#  SYNTHETIC TRAJECTORY (multi-class composition)
# ==============================
def generate_trajectory_rule_based(
    segment_library: Dict[int, Dict[str, List[np.ndarray]]],
    top_classes: List[int],
    class_frequencies: Dict[int, int],
    target_segments: int,
    seed: int
) -> Dict[str, Any]:
    """
    Rule-based construction(multi-class version):
    1. Assign 24 segments to different classes(ensure every class appears at least once)
    2. Generate all segments as post first
    3. randomly choose an insertion position between segments 10 and 20
    4. Insert 1 pre segment at that position (from a different class)
    """
    rng = random.Random(seed)
    np.random.seed(seed)

    total = sum(class_frequencies.values())
    class_probs = {c: class_frequencies[c] / total for c in top_classes}

    # 1. Assign classes to the target_segments segments while ensuring diversity
    # First ensure every class appears at least once
    assigned_classes = list(top_classes)

    # Randomly assign the remaining segments according to class probabilities
    remaining = target_segments - len(top_classes)
    for _ in range(remaining):
        sel_c = rng.choices(top_classes, weights=[class_probs[c] for c in top_classes])[0]
        assigned_classes.append(sel_c)

    # Shuffle the order so the class distribution looks more natural
    rng.shuffle(assigned_classes)

    # 2. Generate target_segments post segments according to the assigned classes
    segs = []
    regs = []
    clss = []

    for cls_id in assigned_classes:
        avail = segment_library[cls_id]['post']
        if len(avail) == 0:
            # If this class has no post samples, skip it
            continue
        seg = rng.choice(avail)
        segs.append(seg)
        regs.append('post')
        clss.append(cls_id)

    # 3. Randomly choose an insertion index between segments 10 and 20 (indices 10-19)
    if len(segs) < 11:  # At least 11 segments are needed for insertion between 10 and 20
        insert_idx = max(3, len(segs) - 1)
    else:
        insert_idx = rng.randint(10, min(20, len(segs) - 1))

    # 4. Insert 1 pre segment (from a different class)
    # Choose a class that differs from the segments before and after the insertion point
    prev_class = clss[insert_idx - 1] if insert_idx > 0 else None
    next_class = clss[insert_idx] if insert_idx < len(clss) else None

    # Identify candidate classes that differ from the neighbors
    available_classes = [c for c in top_classes if c != prev_class and c != next_class]
    if not available_classes:
        available_classes = top_classes

    sel_c = rng.choice(available_classes)
    avail = segment_library[sel_c]['pre']

    # If the selected class has no pre sample, try other classes
    if len(avail) == 0:
        for alt_c in top_classes:
            if len(segment_library[alt_c]['pre']) > 0:
                sel_c = alt_c
                avail = segment_library[alt_c]['pre']
                break

    if len(avail) > 0:
        seg = rng.choice(avail)
        # Insert at the chosen position
        segs.insert(insert_idx, seg)
        regs.insert(insert_idx, 'pre')
        clss.insert(insert_idx, sel_c)

    # Record class distribution(for verification)
    class_counts = Counter(clss)
    print(f"  [Trajectory composition] Classes used: {dict(class_counts)}")

    # Compute boundaries(positions where the regime changes)
    boundaries = []
    current_pos = 0
    for i in range(len(regs)):
        if i > 0 and regs[i] != regs[i-1]:
            boundaries.append(current_pos)
        current_pos += len(segs[i])

    # Concatenate the trajectory
    if segs:
        traj = np.concatenate(segs, axis=0)
    else:
        traj = np.zeros((0, 1), dtype=np.float32)

    return {
        'trajectory': traj,
        'regimes': regs,
        'classes': clss,
        'boundaries': boundaries,
        'segment_starts': np.cumsum([0] + [len(s) for s in segs[:-1]]).tolist()
    }

# ==============================
#  HI EXTRACTION (denser stride)
# ==============================
def extract_hi_from_trajectory(
    trajectory: np.ndarray,
    model: torch.nn.Module,
    device: torch.device,
    window_size: int = 250
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    T, C = trajectory.shape
    hi = np.zeros(T, dtype=np.float32)
    W = None
    model.eval()
    stride = max(1, window_size // 8)  # denser stride
    with torch.no_grad():
        for t in range(0, T, stride):
            end_t = min(t + window_size, T)
            win = trajectory[t:end_t]
            if len(win) < window_size // 2:
                continue
            if len(win) < window_size:
                pad = window_size - len(win)
                win = np.vstack([win, np.zeros((pad, C), dtype=np.float32)])
            x = torch.from_numpy(win).unsqueeze(0).to(device)
            try:
                hi_b, _, w_b, _, _, _ = model.encoder(x)
                vals = hi_b.cpu().numpy().flatten()
                vlen = min(len(vals), end_t - t)
                hi[t:t+vlen] = vals[:vlen]
                if w_b is not None:
                    wv = w_b.cpu().numpy()
                    if W is None:
                        K = wv.shape[-1]
                        W = np.zeros((T, K), dtype=np.float32)
                    W[t:t+vlen] = wv[0, :vlen]
            except Exception as e:
                print(f"[WARN] HI extraction failed at t={t}: {e}")
                continue
    return hi, W

# ==============================
#  METRICS (PME, SM, ΔSM, RS)
# ==============================
def _ols_slope(x: np.ndarray) -> float:
    n = len(x)
    if n < 2:
        return 0.0
    t = np.arange(n)
    return np.polyfit(t, x, 1)[0]

def compute_pme(hi: np.ndarray, t: int, win_pre: int, win_post: int) -> float:
    if t < win_pre or t + win_post > len(hi):
        return 0.0
    pre = hi[t - win_pre:t]
    post = hi[t:t + win_post]
    if len(pre) < 2 or len(post) < 2:
        return 0.0
    mean_diff = np.mean(post) - np.mean(pre)
    slope_diff = _ols_slope(pre) - _ols_slope(post)
    return float(mean_diff + slope_diff)

def _sm_on_window(weights_slice: np.ndarray, op_groups: Dict[str, List[int]]) -> float:
    """SM = linear - (concave + oscillatory) on one window (row-wise normalized before averaging)."""
    if len(weights_slice) == 0:
        return 0.0
    w_norm = weights_slice / (weights_slice.sum(axis=1, keepdims=True) + 1e-8)
    m_linear = np.mean(w_norm[:, op_groups.get('linear', [])]) if op_groups.get('linear') else 0.0
    m_concave = np.mean(w_norm[:, op_groups.get('concave', [])]) if op_groups.get('concave') else 0.0
    m_osc = np.mean(w_norm[:, op_groups.get('oscillatory', [])]) if op_groups.get('oscillatory') else 0.0
    return float(m_linear - (m_concave + m_osc))

def compute_sm_pair(weights: Optional[np.ndarray], t: int, win_pre: int, win_post: int,
                    op_groups: Dict[str, List[int]]) -> Tuple[Optional[float], Optional[float]]:
    """Return (SM_post, ΔSM = SM_post - SM_pre)."""
    if weights is None or t + win_post > len(weights) or t - win_pre < 0:
        return None, None
    w_pre = weights[t - win_pre:t]
    w_post = weights[t:t + win_post]
    sm_pre = _sm_on_window(w_pre, op_groups)
    sm_post = _sm_on_window(w_post, op_groups)
    return sm_post, (sm_post - sm_pre)

def compute_rs(pme: float, dsm: Optional[float], a0: float, a1: float, a2: float) -> float:
    """Use PME and ΔSM as RS inputs"""
    z = a0 + a1 * pme + (a2 * (0.0 if dsm is None else dsm))
    z = np.clip(z, -50, 50)
    return float(1.0 / (1.0 + np.exp(-z)))

def compute_metric_timeline(
    hi: np.ndarray,
    weights: Optional[np.ndarray],
    win_size: int,
    stride: int,
    op_groups: Dict[str, List[int]],
    a0: float, a1: float, a2: float,
    ema_alpha: float = 0.3
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Returns: PME_tl, SM_post_tl, ΔSM_tl, RS_tl
    """
    T = len(hi)
    win_pre = win_size // 2
    win_post = win_size // 2
    pme_tl = np.zeros(T, dtype=np.float32)
    sm_post_tl = np.zeros(T, dtype=np.float32)
    dsm_tl = np.zeros(T, dtype=np.float32)
    rs_tl = np.zeros(T, dtype=np.float32)

    for t in range(win_pre, T - win_post, stride):
        pme = compute_pme(hi, t, win_pre, win_post)
        sm_post, dsm = compute_sm_pair(weights, t, win_pre, win_post, op_groups)
        rs = compute_rs(pme, dsm, a0, a1, a2)

        pme_tl[t] = pme
        sm_post_tl[t] = 0.0 if sm_post is None else sm_post
        dsm_tl[t] = 0.0 if dsm is None else dsm
        rs_tl[t] = rs

    # EMA smoothing
    pme_tl = ema(pme_tl, ema_alpha)
    sm_post_tl = ema(sm_post_tl, ema_alpha)
    dsm_tl = ema(dsm_tl, ema_alpha)
    rs_tl = ema(rs_tl, ema_alpha)

    # Edge padding
    if win_pre < T:
        for arr in (pme_tl, sm_post_tl, dsm_tl, rs_tl):
            arr[:win_pre] = arr[win_pre]
    if T - win_post > 0:
        for arr in (pme_tl, sm_post_tl, dsm_tl, rs_tl):
            arr[T - win_post:] = arr[T - win_post - 1]

    return pme_tl, sm_post_tl, dsm_tl, rs_tl

# ==============================
#  SEGMENT-LEVEL FEATURES
# ==============================
def make_segment_features(
    hi: np.ndarray,
    pme_tl: np.ndarray,
    dsm_tl: np.ndarray,
    rs_tl: np.ndarray,
    segment_starts: List[int],
    feature_names: List[str]
) -> np.ndarray:
    """Extract segment-level features for risk calibration."""
    features = []
    for i in range(len(segment_starts) - 1):
        s, e = segment_starts[i], segment_starts[i + 1]
        feat_dict = {
            'pme_avg': float(np.mean(pme_tl[s:e])),
            'dsm_avg': float(np.mean(dsm_tl[s:e])),
            'rs_avg': float(np.mean(rs_tl[s:e])),
            'rs_slope': _ols_slope(rs_tl[s:e]),
            'rs_var': float(np.var(rs_tl[s:e])),
            'hi_slope': _ols_slope(hi[s:e])
        }
        features.append([feat_dict[fn] for fn in feature_names])
    return np.array(features)

# ==============================
#  DETECTION (consensus + prominence + HMM)
# ==============================
def _find_peaks_with_prominence(x: np.ndarray, min_prominence: float, min_width: int) -> List[int]:
    """Find peaks with minimum prominence and width."""
    if len(x) < 3:
        return []
    mad = np.median(np.abs(x - np.median(x)))
    prom_threshold = max(min_prominence, mad * min_prominence)
    peaks, properties = find_peaks(x, prominence=prom_threshold, width=min_width)
    return peaks.tolist()

def _nms_1d(idx: List[int], scores: np.ndarray, min_dist: int) -> List[int]:
    if not idx:
        return []
    order = sorted(idx, key=lambda i: scores[i], reverse=True)
    keep = []
    taken = np.zeros_like(scores, dtype=bool)
    for i in order:
        if taken[i]:
            continue
        keep.append(i)
        l = max(0, i - min_dist)
        r = min(len(scores), i + min_dist + 1)
        taken[l:r] = True
    return sorted(keep)

def detect_boundaries_consensus(
    pme_tl: np.ndarray,
    dsm_tl: np.ndarray,
    rs_tl: np.ndarray,
    tau_pme_low: float,
    tau_dsm_low: float,
    tau_rs_mid: float,
    tau_rs_high: float,
    refractory: int,
    nms_neighbor: int,
    min_prominence: float,
    min_width: int
) -> List[int]:
    """
    Stronger evidence gates:
      - High confidence: RS > τ_high AND PME > τ_pme AND ΔSM > τ_dsm
      - Medium confidence: RS > τ_mid AND (PME > τ_pme OR ΔSM > τ_dsm) AND peak shape test
    """
    peaks = _find_peaks_with_prominence(rs_tl, min_prominence, min_width)
    cand = []

    for i in peaks:
        # High confidence: strict AND logic
        high_conf = (rs_tl[i] > tau_rs_high and
                    pme_tl[i] > tau_pme_low and
                    dsm_tl[i] > tau_dsm_low)

        # Medium confidence: OR logic with peak shape test
        if not high_conf:
            has_metric = pme_tl[i] > tau_pme_low or dsm_tl[i] > tau_dsm_low
            # Peak shape: RS slope positive before, negative after
            slope_before = rs_tl[i] - rs_tl[max(0, i-3)] if i >= 3 else 0
            slope_after = rs_tl[min(len(rs_tl)-1, i+3)] - rs_tl[i] if i < len(rs_tl)-3 else 0
            peak_shape = slope_before > 0 and slope_after < 0
            mid_conf = rs_tl[i] > tau_rs_mid and has_metric and peak_shape
        else:
            mid_conf = False

        if high_conf or mid_conf:
            cand.append(i)

    cand = _nms_1d(cand, rs_tl, nms_neighbor)

    # Refractory period
    detected = []
    last = -10**9
    for i in cand:
        if i - last >= refractory:
            detected.append(i)
            last = i
    return detected

def detect_boundaries_multiscale_consensus(
    hi: np.ndarray,
    weights: Optional[np.ndarray],
    op_groups: Dict[str, List[int]],
    base_win: int,
    stride: int,
    scale_factors: List[float],
    consensus_k: int,
    a0: float, a1: float, a2: float,
    tau_pme_low: float,
    tau_dsm_low: float,
    tau_rs_mid: float,
    tau_rs_high: float,
    refractory: int,
    nms_neighbor: int,
    min_prominence: float,
    min_width: int,
    ema_alpha: float = 0.3
) -> List[int]:
    """Multi-scale detection with K-of-N consensus."""
    scale_detections = []
    scale_scores = []

    for scale in scale_factors:
        win = int(base_win * scale)
        if win < 10:
            continue

        pme_tl, sm_post_tl, dsm_tl, rs_tl = compute_metric_timeline(
            hi, weights, win, stride, op_groups, a0, a1, a2, ema_alpha
        )

        detected = detect_boundaries_consensus(
            pme_tl, dsm_tl, rs_tl,
            tau_pme_low=tau_pme_low, tau_dsm_low=tau_dsm_low,
            tau_rs_mid=tau_rs_mid, tau_rs_high=tau_rs_high,
            refractory=refractory, nms_neighbor=nms_neighbor,
            min_prominence=min_prominence, min_width=min_width
        )

        scale_detections.append(detected)
        scale_scores.append(rs_tl)

    if not scale_detections:
        return []

    # Consensus voting: require K-of-N agreement
    all_candidates = set()
    for det in scale_detections:
        all_candidates.update(det)

    consensus_detections = []
    for cand in sorted(all_candidates):
        votes = 0
        for det in scale_detections:
            # Check if any detection in this scale is within tolerance
            if any(abs(d - cand) <= nms_neighbor for d in det):
                votes += 1

        if votes >= consensus_k:
            consensus_detections.append(cand)

    # Merge nearby consensus detections
    if not consensus_detections:
        return []

    merged = []
    current_cluster = [consensus_detections[0]]

    for i in range(1, len(consensus_detections)):
        if consensus_detections[i] - current_cluster[-1] <= nms_neighbor:
            current_cluster.append(consensus_detections[i])
        else:
            merged.append(int(np.median(current_cluster)))
            current_cluster = [consensus_detections[i]]

    if current_cluster:
        merged.append(int(np.median(current_cluster)))

    return merged

def hmm_smooth_detections(
    detected: List[int],
    rs_tl: np.ndarray,
    p_action: float,
    p_relapse: float,
    trajectory_length: int
) -> List[int]:
    """Simple HMM smoothing to enforce realistic run lengths."""
    if len(detected) < 2:
        return detected

    # Build state sequence: 0=post, 1=pre
    states = np.zeros(trajectory_length, dtype=int)
    for i in range(len(detected)):
        if i % 2 == 0:  # even transitions are post->pre
            start = detected[i]
            end = detected[i+1] if i+1 < len(detected) else trajectory_length
            states[start:end] = 1

    # Viterbi-like smoothing: penalize short runs
    min_run_length = int(-np.log(p_action) / np.log(1 - p_action))  # expected run length
    smoothed = []
    current_state = 0
    run_start = 0

    for i in range(trajectory_length):
        if states[i] != current_state:
            run_length = i - run_start
            if run_length >= min_run_length or current_state == 0:  # keep long runs or post states
                if current_state == 1 and run_start > 0:
                    smoothed.append(run_start)
            current_state = states[i]
            run_start = i

    return smoothed

def evaluate_detection(true_boundaries: List[int], detected_boundaries: List[int], tolerance: int) -> Dict[str, Any]:
    if len(true_boundaries) == 0:
        return {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'mabe': 0.0, 'tp': 0, 'fp': 0, 'fn': 0, 'lags': []}
    true_matched = set()
    det_matched = set()
    errs = []
    lags = []
    for det in detected_boundaries:
        for j, tru in enumerate(true_boundaries):
            if abs(det - tru) <= tolerance and j not in true_matched:
                true_matched.add(j)
                det_matched.add(det)
                errs.append(abs(det - tru))
                lags.append(det - tru)
                break
    tp = len(true_matched)
    fp = len(detected_boundaries) - len(det_matched)
    fn = len(true_boundaries) - len(true_matched)
    precision = tp / max(1, (tp + fp))
    recall = tp / max(1, (tp + fn))
    f1 = 2 * precision * recall / max(1e-8, (precision + recall))
    mabe = float(np.mean(errs)) if errs else 0.0
    return {'precision': precision, 'recall': recall, 'f1': f1, 'mabe': mabe, 'tp': tp, 'fp': fp, 'fn': fn, 'lags': lags}

# ==============================
#  STABILIZATION & RISK
# ==============================
def compute_stabilization_time(sm_post_tl: np.ndarray, rs_tl: np.ndarray, detected: List[int],
                               tau_sm_post: float, tau_rs: float, delta_window: int) -> List[int]:
    tts = []
    for b in detected:
        for t in range(b, len(sm_post_tl) - delta_window):
            if np.all(sm_post_tl[t:t + delta_window] > tau_sm_post) and np.all(rs_tl[t:t + delta_window] > tau_rs):
                tts.append(t - b)
                break
    return tts

def evaluate_risk_calibration(rs_tl: np.ndarray, true_regimes: List[str], segment_starts: List[int]) -> Dict[str, float]:
    """
    Risk is defined as whether the next segment returns to pre(1=returns to pre).Use 1 - RS as the risk score to keep the direction consistent.
    """
    labels, scores = [], []
    for i in range(len(segment_starts) - 1):
        s, e = segment_starts[i], segment_starts[i + 1]
        rs_avg = float(np.mean(rs_tl[s:e]))
        scores.append(1.0 - rs_avg)
        nxt = true_regimes[i + 1] if i + 1 < len(true_regimes) else 'post'
        labels.append(1 if nxt == 'pre' else 0)
    if len(set(labels)) < 2:
        return {'auc': 0.5, 'brier': 0.25}
    auc = roc_auc_score(labels, scores)
    brier = brier_score_loss(labels, scores)
    return {'auc': float(auc), 'brier': float(brier)}

# ==============================
#  RS CALIBRATION (robust, pooled)
# ==============================
def calibrate_rs_robust_pooled(
    calibration_data: List[Tuple[np.ndarray, np.ndarray, List[str], List[int]]]
) -> Tuple[float, float, float]:
    """
    Robust RS calibration pooled across multiple trajectories:
      - IQR-based standardization
      - Stronger L2 regularization
      - Winsorization to prevent outliers
    """
    X_raw, y = [], []

    for pme_tl, dsm_tl, regimes, seg_starts in calibration_data:
        for i in range(len(seg_starts) - 1):
            s, e = seg_starts[i], seg_starts[i + 1]
            pme_avg = float(np.mean(pme_tl[s:e]))
            dsm_avg = float(np.mean(dsm_tl[s:e]))
            X_raw.append([pme_avg, dsm_avg])
            nxt = regimes[i + 1] if i + 1 < len(regimes) else 'post'
            y.append(1 if nxt == 'post' else 0)  # 1=reliable (post), 0=risky (pre)

    if len(set(y)) < 2 or len(X_raw) < 10:
        return (-1.0, 5.0, 3.0)

    X_raw = np.array(X_raw)

    # Winsorize to 1st-99th percentile
    for col in range(X_raw.shape[1]):
        p1, p99 = np.percentile(X_raw[:, col], [1, 99])
        X_raw[:, col] = np.clip(X_raw[:, col], p1, p99)

    # IQR-based standardization
    q25 = np.percentile(X_raw, 25, axis=0, keepdims=True)
    q75 = np.percentile(X_raw, 75, axis=0, keepdims=True)
    iqr = q75 - q25 + 1e-8
    median = np.median(X_raw, axis=0, keepdims=True)
    X = (X_raw - median) / iqr

    try:
        clf = LogisticRegression(
            max_iter=1000,
            solver="lbfgs",
            C=LOGISTIC_C,  # stronger regularization
            class_weight='balanced'
        )
        clf.fit(X, np.array(y))
        a0 = float(clf.intercept_[0])
        a1 = float(clf.coef_[0][0])
        a2 = float(clf.coef_[0][1])

        # Map back to original scale
        med1, med2 = float(median[0, 0]), float(median[0, 1])
        iqr1, iqr2 = float(iqr[0, 0]), float(iqr[0, 1])
        a1_prime = a1 / iqr1
        a2_prime = a2 / iqr2
        a0_prime = a0 - a1 * (med1 / iqr1) - a2 * (med2 / iqr2)

        return (float(a0_prime), float(a1_prime), float(a2_prime))
    except Exception as e:
        print(f"[WARN] Robust calibration failed: {e}")
        return (-1.0, 5.0, 3.0)

# ==============================
#  SEGMENT-LEVEL RISK CALIBRATION
# ==============================
def calibrate_segment_risk(
    calibration_data: List[Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, List[str], List[int]]],
    feature_names: List[str]
) -> Optional[LogisticRegression]:
    """
    Calibrate segment-level risk predictor: P(next_regime='pre' | segment_features).
    Returns trained LogisticRegression model.
    """
    X_all, y_all = [], []

    for hi, pme_tl, dsm_tl, rs_tl, regimes, seg_starts in calibration_data:
        X_seg = make_segment_features(hi, pme_tl, dsm_tl, rs_tl, seg_starts, feature_names)
        for i in range(len(seg_starts) - 1):
            nxt = regimes[i + 1] if i + 1 < len(regimes) else 'post'
            y_all.append(1 if nxt == 'pre' else 0)
        X_all.append(X_seg)

    if not X_all:
        return None

    X = np.vstack(X_all)
    y = np.array(y_all)

    if len(set(y)) < 2 or len(X) < 10:
        return None

    # Winsorize
    for col in range(X.shape[1]):
        p1, p99 = np.percentile(X[:, col], [1, 99])
        X[:, col] = np.clip(X[:, col], p1, p99)

    # Standardize
    X_mean = np.mean(X, axis=0, keepdims=True)
    X_std = np.std(X, axis=0, keepdims=True) + 1e-8
    X = (X - X_mean) / X_std

    try:
        clf = LogisticRegression(
            max_iter=1000,
            solver='lbfgs',
            C=LOGISTIC_C,
            class_weight='balanced'
        )
        clf.fit(X, y)
        # Store normalization params
        clf.X_mean_ = X_mean
        clf.X_std_ = X_std
        return clf
    except Exception as e:
        print(f"[WARN] Segment risk calibration failed: {e}")
        return None

# ==============================
#  NON-OVERLAPPING INTERVAL LABELS
# ==============================
def layout_interval_labels(segments: List[Dict[str, Any]], min_gap: int = 10) -> Tuple[List[int], int]:
    """
    segments: List[dict] with keys {start, end, text}
    Return the y-level for each segment(0,1,2,...)for vertical staggering, avoid overlap.
    Greedy sweep-line layout with multiple occupancy layers.
    """
    layers = []  # Rightmost occupied boundary for each layer
    levels = []
    for seg in sorted(segments, key=lambda s: (s['start'], s['end'])):
        placed = False
        for li, rightmost in enumerate(layers):
            if seg['start'] > rightmost + min_gap:
                layers[li] = seg['end']
                levels.append(li)
                placed = True
                break
        if not placed:
            layers.append(seg['end'])
            levels.append(len(layers) - 1)
    return levels, len(layers)

# ==============================
#  PLOTTING FUNCTIONS
# ==============================
def plot_trajectory_with_annotations(
    hi: np.ndarray,
    pme_tl: np.ndarray,
    dsm_tl: np.ndarray,
    true_boundaries: List[int],
    detected_boundaries: List[int],
    regimes: List[str],
    classes: List[int],
    segment_starts: List[int],
    label_mappings: dict,
    thresholds: Dict[str, float],
    seed: int,
    save_path: Optional[str] = None
):
    """
    Overview plot: multi-class stitched trajectory with class-specific background colors and a light red overlay for pre/post regimes
    """
    # Assign a distinct base color to each class
    base_colors = ['lightblue', 'lightgreen', 'lightyellow', 'lightcoral', 'lightcyan', 'lavender']
    unique_classes = sorted(set(classes))
    class_color_map = {cls: base_colors[i % len(base_colors)] for i, cls in enumerate(unique_classes)}

    fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

    # (1) HI with class-colored background + pre/post overlay
    ax = axes[0]
    for i in range(len(segment_starts)):
        start = segment_starts[i]
        end = segment_starts[i+1] if i+1 < len(segment_starts) else len(hi)
        cls = classes[i]
        regime = regimes[i]

        # Base class color
        base_color = class_color_map.get(cls, 'whitesmoke')
        ax.axvspan(start, end, color=base_color, alpha=0.4)

        # Add a light red overlay for pre segments
        if regime == 'pre':
            ax.axvspan(start, end, color='red', alpha=0.15)

    ax.plot(hi, lw=1.2, color='darkblue', alpha=0.8, label='Health Index')

    # Annotate boundaries
    for b in true_boundaries:
        ax.axvline(b, color='red', ls=':', alpha=0.6, lw=1.5, label='True' if b == true_boundaries[0] else '')
    for d in detected_boundaries:
        shifted_d=d+200
        ax.axvline(d+200, color='green', ls='--', alpha=0.8, lw=1.5, label='Detected' if d == detected_boundaries[0] else '')
        # Find the true boundary closest to this detected boundary
        closest_b = min(true_boundaries, key=lambda x: abs(x - shifted_d))
    # Compute the distance
        diff = shifted_d - closest_b
        # Annotate the midpoint between them
        mid_x = (closest_b + shifted_d) / 2
        ax.text(mid_x, ax.get_ylim()[1] * 0.95, f'{diff:.1f}', color='black',
                fontsize=9, ha='center', va='top', rotation=0,
                bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='gray', alpha=0.7))
    # Top labels:ClassX(post) / ClassX(pre)
    segments_for_layout = []
    for i in range(len(segment_starts)):
        start = segment_starts[i]
        end = segment_starts[i+1] if i+1 < len(segment_starts) else len(hi)
        if end - start >= 150:  # label only sufficiently long segments
            cls = classes[i]
            regime = regimes[i]
            class_name = resolve_name_for_id(int(cls), label_mappings)
            segments_for_layout.append({
                'start': start,
                'end': end,
                'text': f'{class_name}({regime})',
                'color': class_color_map.get(cls, 'gray')
            })

    # Non-overlapping label layout
    if segments_for_layout:
        levels, n_layers = layout_interval_labels(segments_for_layout, min_gap=50)
        ylim = ax.get_ylim()
        dy = (ylim[1] - ylim[0]) * 0.06
        for seg, lv in zip(segments_for_layout, levels):
            mid = 0.5 * (seg['start'] + seg['end'])
            ax.text(mid, ylim[1] + dy * (lv + 0.5), seg['text'],
                   ha='center', va='bottom', fontsize=6, rotation=0,
                   bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.85, edgecolor=seg['color']),
                   clip_on=False)

    ax.set_ylabel('Health Index', fontsize=10)
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(alpha=0.3)

    # (2) PME & ΔSM
    ax = axes[1]
    ax.bar(np.arange(len(pme_tl)), pme_tl, width=1.0, alpha=0.6, label='PME', color='tab:blue')
    ax.bar(np.arange(len(dsm_tl)), dsm_tl, width=1.0, alpha=0.4, label='ΔSM', color='tab:purple')
    ax.axhline(thresholds['tau_pme_low'], color='tab:blue', ls='--', alpha=0.7, label='τ_PME')
    ax.axhline(thresholds['tau_dsm_low'], color='tab:purple', ls='--', alpha=0.7, label='τ_ΔSM')
    ax.set_ylabel('Metric Values', fontsize=10)
    ax.set_xlabel('Time Index', fontsize=10)
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(alpha=0.3)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  [Saved] {save_path}")

    plt.show()

# ==============================
#  MAIN
# ==============================
if __name__ == "__main__":
    print("[INFO] Starting long-horizon synthetic trajectory analysis...")
    print("[INFO] Enhanced with robust calibration, consensus fusion, and HMM smoothing")
    print("[INFO] Multi-class trajectory composition enabled (single pre sample insertion)")

    # 1) Load artifacts
    artifacts = load_sample_split_artifacts()
    train_data = artifacts["train"]
    label_mappings = artifacts.get("label_mappings", {})

    # 2) Top-K classes
    label_field, is_encoded = _determine_label_field(train_data)
    freq = count_by_field(train_data, label_field)
    top_items = sorted(freq.items(), key=lambda kv: kv[1], reverse=True)[:TOP_K]
    top_classes = [k for k, _ in top_items]
    class_frequencies = {k: v for k, v in top_items}
    print("[INFO] Top {} classes:".format(TOP_K))
    for k, v in top_items:
        name = resolve_name_for_id(int(k), label_mappings) if is_encoded else k
        print(f"  - {name} ({k}): {v} samples")

    # 3) Segment library
    print("[INFO] Building segment library...")
    segment_library = build_segment_library(train_data, top_classes, label_field)
    for c in top_classes:
        print(f"  Class {c}: {len(segment_library[c]['pre'])} pre, {len(segment_library[c]['post'])} post")

    # 4) Load model
    print("[INFO] Loading model...")
    ckpt_path = os.path.join(SAVE_DIR, "best_model_NGFAID_250_100_6.pth")
    meta_path = ckpt_path.replace('.pth', '_meta.json')
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if os.path.exists(meta_path):
        with open(meta_path, 'r') as f:
            meta_data = json.load(f)
        K = meta_data.get('K', len(top_classes))
        C = meta_data.get('sensor_dim', segment_library[top_classes[0]]['pre'][0].shape[1])
        print(f"[INFO] Loaded metadata: K={K}, C={C}")
    else:
        first_segment = segment_library[top_classes[0]]['pre'][0]
        C = first_segment.shape[1]
        K = len(top_classes)
        print(f"[INFO] Inferred from data: K={K}, C={C}")

    model = DiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=K).to(device)
    state_dict = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state_dict, strict=False)
    model.eval()
    print("[INFO] Model loaded successfully")

    # 5) Operator groups
    op_groups = {
        'linear': [0, 1, 7],
        'concave': [2, 3, 5],
        'oscillatory': [4, 6]
    }

    # -------- Pooled RS calibration --------
    print(f"\n[INFO] Generating {NUM_CALIBRATION_TRAJECTORIES} calibration trajectories...")
    calibration_data = []
    for cal_seed in range(1000, 1000 + NUM_CALIBRATION_TRAJECTORIES):
        traj = generate_trajectory_rule_based(segment_library, top_classes, class_frequencies,
                                              TARGET_SEGMENTS, cal_seed)
        arr = traj['trajectory']
        regimes = traj['regimes']
        seg_starts = traj['segment_starts']

        hi, W = extract_hi_from_trajectory(arr, model, device, window_size=WINDOW_SIZE)
        spd = len(arr) / (len(traj['regimes']) * DAYS_PER_SEGMENT)
        win = int(METRIC_WINDOW_DAYS * spd)
        stride = max(1, int(STRIDE_FRACTION * spd))
        pme, sm_post, dsm, _ = compute_metric_timeline(
            hi, W, win, stride, op_groups,
            a0=-1.0, a1=5.0, a2=3.0, ema_alpha=EMA_ALPHA_METRICS
        )
        calibration_data.append((pme, dsm, regimes, seg_starts))

    a0, a1, a2 = calibrate_rs_robust_pooled(calibration_data)
    print(f"[INFO] Calibrated RS (pooled, robust): a0={a0:.3f}, a1={a1:.3f}, a2={a2:.3f}")

    results = []

    # -------- Process each trajectory --------
    for seed in RANDOM_SEEDS:
        print(f"\n[INFO] Processing trajectory for seed {seed}...")

        traj = generate_trajectory_rule_based(segment_library, top_classes, class_frequencies,
                                              TARGET_SEGMENTS, seed)
        arr = traj['trajectory']
        true_bounds = traj['boundaries']
        regimes = traj['regimes']
        classes = traj['classes']
        seg_starts = traj['segment_starts']
        print(f"  Generated trajectory: {len(arr)} samples, {len(true_bounds)} boundaries")

        hi, W = extract_hi_from_trajectory(arr, model, device, window_size=WINDOW_SIZE)
        spd = len(arr) / (len(traj['regimes']) * DAYS_PER_SEGMENT)
        win = int(METRIC_WINDOW_DAYS * spd)
        stride = max(1, int(STRIDE_FRACTION * spd))
        pme_tl, sm_post_tl, dsm_tl, rs_tl = compute_metric_timeline(
            hi, W, win, stride, op_groups, a0=a0, a1=a1, a2=a2, ema_alpha=EMA_ALPHA_METRICS
        )

        # Adaptive thresholds (full distribution with winsorization)
        def robust_percentile_full(x: np.ndarray, p: int) -> float:
            valid = x[np.isfinite(x)]
            if len(valid) < 10:
                return 0.0
            # Winsorize
            p1, p99 = np.percentile(valid, [1, 99])
            clipped = np.clip(valid, p1, p99)
            return float(np.percentile(clipped, p))

        tau_pme_low = robust_percentile_full(pme_tl, PME_P_LOW)
        tau_dsm_low = robust_percentile_full(dsm_tl, DSM_P_LOW)
        tau_rs_mid = robust_percentile_full(rs_tl, RS_P_MID)
        tau_rs_high = robust_percentile_full(rs_tl, RS_P_HIGH)

        # Detection
        tol = int(DETECTION_TOLERANCE_DAYS * spd)
        refractory = int(REFRACTORY_DAYS * spd)
        nms_nb = int(NMS_NEIGHBOR * spd)
        min_prom = MIN_PEAK_PROMINENCE
        min_width = MIN_PEAK_WIDTH

        if USE_MULTISCALE:
            detected = detect_boundaries_multiscale_consensus(
                hi, W, op_groups, win, stride, SCALE_FACTORS, CONSENSUS_K,
                a0, a1, a2,
                tau_pme_low=tau_pme_low, tau_dsm_low=tau_dsm_low,
                tau_rs_mid=tau_rs_mid, tau_rs_high=tau_rs_high,
                refractory=refractory, nms_neighbor=nms_nb,
                min_prominence=min_prom, min_width=min_width
            )
        else:
            detected = detect_boundaries_consensus(
                pme_tl, dsm_tl, rs_tl,
                tau_pme_low=tau_pme_low, tau_dsm_low=tau_dsm_low,
                tau_rs_mid=tau_rs_mid, tau_rs_high=tau_rs_high,
                refractory=refractory, nms_neighbor=nms_nb,
                min_prominence=min_prom, min_width=min_width
            )

        # HMM smoothing
        if USE_HMM_SMOOTHING and detected:
            detected = hmm_smooth_detections(detected, rs_tl, P_ACTION, P_RELAPSE, len(arr))

        # Evaluation
        det_metrics = evaluate_detection(true_boundaries=true_bounds,
                                         detected_boundaries=detected,
                                         tolerance=tol)
        print(f"  Detection - P: {det_metrics['precision']:.3f}, "
              f"R: {det_metrics['recall']:.3f}, "
              f"F1: {det_metrics['f1']:.3f}, "
              f"MABE: {det_metrics['mabe']:.1f}")

        # Stabilization
        delta_samples = int(STABILIZATION_WINDOW_DAYS * spd)
        tts_vals = compute_stabilization_time(sm_post_tl, rs_tl, detected,
                                              tau_sm_post=robust_percentile_full(sm_post_tl, 50),
                                              tau_rs=tau_rs_mid, delta_window=delta_samples)
        if tts_vals:
            print(f"  Stabilization - Median TTS: {np.median(tts_vals):.1f} "
                  f"[{np.percentile(tts_vals,25):.1f}-{np.percentile(tts_vals,75):.1f}]")

        # Risk calibration
        risk_metrics = evaluate_risk_calibration(rs_tl, regimes, seg_starts)
        print(f"  Risk calibration - AUC: {risk_metrics['auc']:.3f}, Brier: {risk_metrics['brier']:.3f}")

        results.append({
            'seed': seed,
            'detection': det_metrics,
            'tts': tts_vals,
            'risk': risk_metrics,
            'hi_timeline': hi,
            'pme_timeline': pme_tl,
            'sm_post_timeline': sm_post_tl,
            'dsm_timeline': dsm_tl,
            'rs_timeline': rs_tl,
            'true_boundaries': true_bounds,
            'detected_boundaries': detected,
            'regimes': regimes,
            'classes': classes,
            'segment_starts': seg_starts,
            'thresholds': {
                'tau_pme_low': tau_pme_low,
                'tau_dsm_low': tau_dsm_low,
                'tau_rs_mid': tau_rs_mid,
                'tau_rs_high': tau_rs_high
            },
            'samples_per_day': spd
        })

        # Plotting with annotations (overview plot)
        if DO_PLOTS:
            save_path = os.path.join(FIGS_DIR, 'overview', f'trajectory_seed_{seed}_overview.png') if SAVE_FIGS else None
            plot_trajectory_with_annotations(
                hi, pme_tl, dsm_tl,
                true_bounds, detected,
                regimes, classes, seg_starts,
                label_mappings,
                results[-1]['thresholds'],
                seed, save_path
            )

    # Summary
    print("\n" + "="*80)
    print("SUMMARY STATISTICS (Enhanced with Multi-Class Composition)")
    print("="*80)
    all_precision = [r['detection']['precision'] for r in results]
    all_recall = [r['detection']['recall'] for r in results]
    all_f1 = [r['detection']['f1'] for r in results]
    all_mabe = [r['detection']['mabe'] for r in results]
    print("Boundary Detection (Consensus + HMM):")
    print(f"  Precision: {np.mean(all_precision):.3f} ± {np.std(all_precision):.3f}")
    print(f"  Recall:    {np.mean(all_recall):.3f} ± {np.std(all_recall):.3f}")
    print(f"  F1-Score:  {np.mean(all_f1):.3f} ± {np.std(all_f1):.3f}")
    print(f"  MABE:      {np.mean(all_mabe):.1f} ± {np.std(all_mabe):.1f} samples")

    all_tts = [t for r in results for t in r['tts']]
    if all_tts:
        print("\nStabilization:")
        print(f"  Median TTS: {np.median(all_tts):.1f} samples")
        print(f"  IQR: [{np.percentile(all_tts,25):.1f}, {np.percentile(all_tts,75):.1f}]")

    all_auc = [r['risk']['auc'] for r in results]
    all_brier = [r['risk']['brier'] for r in results]
    print("\nRisk Calibration (Robust, Pooled):")
    print(f"  AUC:   {np.mean(all_auc):.3f} ± {np.std(all_auc):.3f}")
    print(f"  Brier: {np.mean(all_brier):.3f} ± {np.std(all_brier):.3f}")

    # Save
    output_path = os.path.join(SAVE_DIR, 'long_horizon_results_enhanced.pkl')
    with open(output_path, 'wb') as f:
        pickle.dump({
            'config': {
                'seeds': RANDOM_SEEDS,
                'calibration_trajectories': NUM_CALIBRATION_TRAJECTORIES,
                'consensus_k': CONSENSUS_K,
                'use_hmm': USE_HMM_SMOOTHING,
                'logistic_c': LOGISTIC_C,
                'rs_coefficients': {'a0': a0, 'a1': a1, 'a2': a2}
            },
            'results': results
        }, f)
    print(f"\n[INFO] Results saved to {output_path}")
    print("="*80 + "\n")


## Operator Weight Visualization Utilities


In [ ]:
import os, json, math, random
import numpy as np
import torch
import matplotlib.pyplot as plt

def plot_class_operator_weight_heatmaps(
    checkpoint_path: str,
    test_loader,
    target_class,                     # int id (e.g., 3) or class name (e.g., "Class_3")
    device: torch.device = None,
    max_samples: int = 9,
    seed: int = 0,
    max_curve_keep: int = 2000,       # ensure we keep enough samples with weights
    fallback_to_pred_if_no_labels: bool = True,
):
    """
    Load the best model checkpoint and plot operator-weight heatmaps (pre/post) for samples
    belonging to a specified class.

    Args:
        checkpoint_path: Path to the saved .pth model (the function will also try to read the
                         matching '<checkpoint>_meta.json' if it exists).
        test_loader:     A DataLoader that yields batches as defined in your snippet.
        target_class:    Either a class id (int) or a class name (str). If labels are missing
                         or ignored, can fallback to predicted label when enabled.
        device:          torch.device to perform inference on. Defaults to CUDA if available.
        max_samples:     Maximum number of sample heatmaps to draw.
        seed:            RNG seed for shuffling/selection.
        max_curve_keep:  How many curves to retain from collect_test_predictions (must be high
                         enough to cover your filtered class subset).
        fallback_to_pred_if_no_labels:
                         If True, and ground-truth labels are missing/ignored, filter by
                         predicted labels instead.
    """

    # ---------- 0) Device ----------
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ---------- 1) Read meta or infer K/C/id_to_name ----------
    meta_path = checkpoint_path.replace(".pth", "_meta.json")
    has_meta = os.path.isfile(meta_path)
    if has_meta:
        with open(meta_path, "r") as f:
            meta = json.load(f)
        K = int(meta["K"])
        id_to_name = {int(k): v for k, v in meta["id_to_name"].items()}
        C = int(meta["sensor_dim"])
    else:
        # Fallback: infer from loader
        K = scan_num_classes_from_loader(test_loader)
        id_to_name = build_default_id_to_name(K)
        first_batch = next(iter(test_loader))
        C = first_batch["x_before"].shape[-1]

    # Map class name -> id if needed
    if isinstance(target_class, str):
        # Exact match first, else try case-insensitive
        name_to_id = {v: k for k, v in id_to_name.items()}
        if target_class in name_to_id:
            target_id = name_to_id[target_class]
        else:
            ci = {v.lower(): k for k, v in id_to_name.items()}
            if target_class.lower() in ci:
                target_id = ci[target_class.lower()]
            else:
                raise ValueError(f"Class name '{target_class}' not found in id_to_name: {id_to_name}")
    else:
        target_id = int(target_class)

    # ---------- 2) Rebuild model and load weights ----------
    model = DiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=K).to(device)
    state = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state, strict=True)
    model.eval()

    # ---------- 3) Collect predictions/curves (keep many for filtering) ----------
    y_true, y_pred, _, _, curves = collect_test_predictions(
        model=model,
        loader=test_loader,
        device=device,
        max_curve_keep=max_curve_keep,
        K=K,
        id_to_name=id_to_name
    )

    # ---------- 4) Filter curves by target class ----------
    def valid_label(v):
        return (v is not None) and (v >= 0)

    filtered = []
    for ex in curves:
        w_b = ex.get("weights_before", None)
        w_a = ex.get("weights_after", None)
        if w_b is None or w_a is None:
            continue  # need weights to plot

        t = ex.get("true", -1)
        p = ex.get("pred", -1)

        if valid_label(t) and t == target_id:
            filtered.append(ex)
        elif (not valid_label(t)) and fallback_to_pred_if_no_labels and p == target_id:
            filtered.append(ex)

    if len(filtered) == 0:
        print(f"[Info] No samples found for class '{target_class}' (id={target_id}).")
        return

    # ---------- 5) Plot settings ----------
    op_names = ["MonotonicLinear", "MonotonicFlat", "ConcaveLog", "SaturationSigmoid",
                "HingeReLU", "Polynomial", "DampedSin", "PiecewiseLinear"]

    random.Random(seed).shuffle(filtered)
    n_show = min(max_samples, len(filtered))
    cols = 3
    rows = math.ceil(n_show / cols)

    fig_w = 5.4 * cols
    fig_h = 3.8 * rows
    fig = plt.figure(figsize=(fig_w, fig_h))

    for idx in range(n_show):
        ex = filtered[idx]
        w_b = np.asarray(ex["weights_before"])  # (T_b, K)
        w_a = np.asarray(ex["weights_after"])   # (T_a, K)

        if w_b.ndim != 2 or w_a.ndim != 2:
            continue

        T_b, K_b = w_b.shape
        T_a, K_a = w_a.shape
        K_eff = min(K_b, K_a, len(op_names))
        w_b = w_b[:, :K_eff]
        w_a = w_a[:, :K_eff]

        # Concatenate along time to align pre | post; add separator via a line (not NaN)
        combined = np.concatenate([w_b, w_a], axis=0)  # (T_b+T_a, K_eff)

        ax = plt.subplot(rows, cols, idx + 1)
        # Heatmap expects (rows=ops, cols=time). Transpose to (K, T)
        im = ax.imshow(combined.T, aspect="auto", interpolation="nearest")
        # Vertical separator between pre and post
        ax.axvline(T_b - 0.5, color="k", linestyle=":", linewidth=1.0)

        # Axes labels and ticks
        ax.set_yticks(range(K_eff))
        ax.set_yticklabels([op_names[i] for i in range(K_eff)], fontsize=8)
        ax.set_xlabel("Time (pre  |  post)")
        ax.set_title(
            f"ID={ex.get('sample_id','?')} | True={id_to_name.get(ex.get('true',-1),'?')} | "
            f"Pred={id_to_name.get(ex.get('pred',-1),'?')} | p={ex['prob'][ex['pred']]:.2f}"
            if ex.get("prob") is not None else
            f"ID={ex.get('sample_id','?')} | True={id_to_name.get(ex.get('true',-1),'?')} | "
            f"Pred={id_to_name.get(ex.get('pred',-1),'?')}"
        , fontsize=9)

        # Colorbar (small)
        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.set_ylabel("Weight", rotation=270, labelpad=10)

    plt.suptitle(
        f"Operator Weight Heatmaps — Class: {id_to_name.get(target_id, f'Class_{target_id}')}",
        fontsize=12, y=0.995
    )
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

# -----------------------
# Example usage (English)
# -----------------------
plot_class_operator_weight_heatmaps(
    checkpoint_path=str(resolve_data_file("best_model_NGFAID_250_100_4.pth")),
    test_loader=test_loader,     # provide your test DataLoader
    target_class=3,              # or "Class_3"
    device=None,                 # auto-select
    max_samples=6,
    seed=0
)


In [ ]:
# import os, json, math, random
# import numpy as np
# import torch
# import matplotlib.pyplot as plt

# def plot_aggregate_class_operator_heatmap(
#     checkpoint_path: str,
#     test_loader,
#     target_class,                      # int id (e.g., 3) or class name (e.g., "Class_3")
#     device: torch.device = None,
#     pre_bins: int = 60,
#     post_bins: int = 60,
#     normalize: str = "mean",           # "mean" (default) or "sum" for overlay style
#     max_curve_keep: int = 200000,      # keep enough curves to cover the whole class
#     fallback_to_pred_if_no_labels: bool = True,
#     seed: int = 0,
# ):
#     """
#     Load best checkpoint and plot a single aggregated heatmap of operator weights
#     (pre and post maintenance) for ALL samples in a target class.

#     Steps:
#       1) Load model and meta (K, id_to_name, C).
#       2) Collect predictions and keep per-sample operator weights.
#       3) Filter curves by target class (true label if available; else predicted).
#       4) Time-normalize (resample) each sample's pre and post weights to fixed bins.
#       5) Aggregate across samples (mean or sum) and plot one combined heatmap.

#     Args:
#         checkpoint_path: Path to model .pth file; will also try '<pth>_meta.json'.
#         test_loader:     DataLoader yielding batches defined in your snippet.
#         target_class:    Class id (int) or class name (str).
#         device:          Device for inference; auto if None.
#         pre_bins:        Target time bins for the pre-maintenance part.
#         post_bins:       Target time bins for the post-maintenance part.
#         normalize:       "mean" for average aggregation (default), "sum" for pure overlay.
#         max_curve_keep:  Max curves stored from `collect_test_predictions`.
#         fallback_to_pred_if_no_labels:
#                          Use predicted label to filter when ground-truth is absent/ignored.
#         seed:            RNG seed for deterministic selection (not strictly needed here).
#     """

#     # ----------------- helpers -----------------
#     def resample_time_major(arr_TK: np.ndarray, tgt_len: int) -> np.ndarray:
#         """Resample a (T, K) array along time axis to target length using linear interpolation."""
#         T, K = arr_TK.shape
#         if T <= 1 or tgt_len == T:
#             # trivial cases: repeat or return as-is
#             if T == tgt_len:
#                 return arr_TK.copy()
#             if T <= 1:
#                 return np.repeat(arr_TK, tgt_len, axis=0)
#         # Build source and target positions in [0, 1]
#         src = np.linspace(0.0, 1.0, num=T, endpoint=True)
#         dst = np.linspace(0.0, 1.0, num=tgt_len, endpoint=True)
#         out = np.empty((tgt_len, K), dtype=arr_TK.dtype)
#         # vectorized per-operator interpolation
#         for k in range(K):
#             out[:, k] = np.interp(dst, src, arr_TK[:, k])
#         return out

#     # ----------------- device -----------------
#     if device is None:
#         device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#     # ----------------- meta / model -----------------
#     meta_path = checkpoint_path.replace(".pth", "_meta.json")
#     if os.path.isfile(meta_path):
#         with open(meta_path, "r") as f:
#             meta = json.load(f)
#         K = int(meta["K"])
#         id_to_name = {int(k): v for k, v in meta["id_to_name"].items()}
#         C = int(meta["sensor_dim"])
#     else:
#         # Fallback: infer from test loader
#         K = scan_num_classes_from_loader(test_loader)
#         id_to_name = build_default_id_to_name(K)
#         first_batch = next(iter(test_loader))
#         C = first_batch["x_before"].shape[-1]

#     # Map class name -> id if needed
#     if isinstance(target_class, str):
#         name_to_id = {v: k for k, v in id_to_name.items()}
#         if target_class in name_to_id:
#             target_id = name_to_id[target_class]
#         else:
#             ci = {v.lower(): k for k, v in id_to_name.items()}
#             if target_class.lower() in ci:
#                 target_id = ci[target_class.lower()]
#             else:
#                 raise ValueError(f"Class name '{target_class}' not found in id_to_name: {id_to_name}")
#     else:
#         target_id = int(target_class)

#     # Rebuild model and load weights
#     model = DiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=K).to(device)
#     state = torch.load(checkpoint_path, map_location=device)
#     model.load_state_dict(state, strict=True)
#     model.eval()

#     # ----------------- collect curves -----------------
#     _, _, _, _, curves = collect_test_predictions(
#         model=model,
#         loader=test_loader,
#         device=device,
#         max_curve_keep=max_curve_keep,
#         K=K,
#         id_to_name=id_to_name
#     )

#     # ----------------- filter by class -----------------
#     def valid_label(v): return (v is not None) and (v >= 0)
#     selected = []
#     for ex in curves:
#         wb = ex.get("weights_before", None)
#         wa = ex.get("weights_after", None)
#         if wb is None or wa is None:
#             continue
#         t = ex.get("true", -1)
#         p = ex.get("pred", -1)
#         if valid_label(t) and t == target_id:
#             selected.append(ex)
#         elif (not valid_label(t)) and fallback_to_pred_if_no_labels and p == target_id:
#             selected.append(ex)

#     if len(selected) == 0:
#         print(f"[Info] No samples found for class '{target_class}' (id={target_id}).")
#         return

#     # ----------------- resample & aggregate -----------------
#     # Operator count (should be 8 by your design). Take min across examples to be safe.
#     K_ops = min(
#         min(ex["weights_before"].shape[1] for ex in selected),
#         min(ex["weights_after"].shape[1]  for ex in selected)
#     )
#     # Optional: align to known operator list length
#     op_names = ["MonotonicLinear", "MonotonicFlat", "ConcaveLog", "SaturationSigmoid",
#                 "HingeReLU", "Polynomial", "DampedSin", "PiecewiseLinear"]
#     K_eff = min(K_ops, len(op_names))

#     pre_stack  = []
#     post_stack = []

#     random.Random(seed).shuffle(selected)  # no effect on aggregation, keeps determinism

#     for ex in selected:
#         wb = np.asarray(ex["weights_before"])[:, :K_eff]  # (T_b, K_eff)
#         wa = np.asarray(ex["weights_after"])[:,  :K_eff]  # (T_a, K_eff)
#         wb_rs = resample_time_major(wb, pre_bins)   # (pre_bins,  K_eff)
#         wa_rs = resample_time_major(wa, post_bins)  # (post_bins, K_eff)
#         pre_stack.append(wb_rs)
#         post_stack.append(wa_rs)

#     pre_stack  = np.stack(pre_stack,  axis=0)  # (N, pre_bins,  K_eff)
#     post_stack = np.stack(post_stack, axis=0)  # (N, post_bins, K_eff)

#     if normalize.lower() == "sum":
#         agg_pre  = pre_stack.sum(axis=0)       # (pre_bins,  K_eff)
#         agg_post = post_stack.sum(axis=0)      # (post_bins, K_eff)
#         cbar_label = "Aggregated weight (sum)"
#     else:
#         agg_pre  = pre_stack.mean(axis=0)      # (pre_bins,  K_eff)
#         agg_post = post_stack.mean(axis=0)     # (post_bins, K_eff)
#         cbar_label = "Aggregated weight (mean)"

#     combined = np.concatenate([agg_pre, agg_post], axis=0)  # (pre_bins+post_bins, K_eff)

#     # ----------------- plot -----------------
#     fig = plt.figure(figsize=(8.8, 4.8))
#     ax = plt.gca()
#     im = ax.imshow(combined.T, aspect="auto", interpolation="nearest")
#     ax.axvline(pre_bins - 0.5, color="k", linestyle=":", linewidth=1.0)

#     ax.set_yticks(range(K_eff))
#     ax.set_yticklabels([op_names[i] for i in range(K_eff)], fontsize=8)
#     ax.set_xlabel("Time (pre  |  post)")
#     title_cls = id_to_name.get(target_id, f"Class_{target_id}")
#     ax.set_title(f"Aggregated Operator Weights — Class: {title_cls} (N={len(selected)})", fontsize=11)

#     cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
#     cbar.ax.set_ylabel(cbar_label, rotation=270, labelpad=10)

#     plt.tight_layout()
#     plt.show()

#     # Optionally return arrays for further analysis
#     return {
#         "class_id": target_id,
#         "class_name": title_cls,
#         "n_samples": len(selected),
#         "agg_pre": agg_pre,            # shape: (pre_bins,  K_eff)
#         "agg_post": agg_post,          # shape: (post_bins, K_eff)
#         "op_names": [op_names[i] for i in range(K_eff)],
#     }

# # -----------------------
# # Example (English)
# # -----------------------
# out = plot_aggregate_class_operator_heatmap(
#     checkpoint_path=str(resolve_data_file("best_model_NGFAID_250_100_4.pth")),
#     test_loader=test_loader,
#     target_class="Class_3",     # or 3
#     device=None,                # auto
#     pre_bins=60,
#     post_bins=60,
#     normalize="mean"            # or "sum" for literal overlay
# )


In [ ]:
out = plot_aggregate_class_operator_heatmap(
    checkpoint_path=str(resolve_data_file("best_model_NGFAID_250_100_4.pth")),
    test_loader=test_loader,
    target_class="Class_10",     # or 3
    device=None,                # auto
    pre_bins=60,
    post_bins=60,
    normalize="mean"            # or "sum" for literal overlay
)


## GAFID Sensor Reference
GAFID Aviation Maintenance Dataset - Sensor Description
| Column Name  | Description |
|-------------|-------------|
| **flight_id** | Unique identifier for each flight |
| **volt1** | Main electrical system bus voltage (alternators and main battery) |
| **volt2** | Essential bus (standby battery) bus voltage |
| **amp1** | Ammeter on the main battery (+ charging, - discharging) |
| **amp2** | Ammeter on the standby battery (+ charging, - discharging) |
| **FQtyL** | Fuel quantity in the left tank |
| **FQtyR** | Fuel quantity in the right tank |
| **E1 FFlow** | Engine fuel flow rate |
| **E1 OilT** | Engine oil temperature |
| **E1 OilP** | Engine oil pressure |
| **E1 RPM** | Engine revolutions per minute |
| **E1 CHT1** | Cylinder head temperature of the 1st cylinder |
| **E1 CHT2** | Cylinder head temperature of the 2nd cylinder |
| **E1 CHT3** | Cylinder head temperature of the 3rd cylinder |
| **E1 CHT4** | Cylinder head temperature of the 4th cylinder |
| **E1 EGT1** | Exhaust gas temperature of the 1st cylinder |
| **E1 EGT2** | Exhaust gas temperature of the 2nd cylinder |
| **E1 EGT3** | Exhaust gas temperature of the 3rd cylinder |
| **E1 EGT4** | Exhaust gas temperature of the 4th cylinder |
| **OAT** | Outside air temperature |
| **IAS** | Indicated air speed |
| **VSpd** | Vertical speed |
| **NormAc** | Normal acceleration |
| **AltMSL** | Altitude above mean sea level |
| **Column 18** | Unknown variable (Possibly related to throttle or engine power) |
| **Column 19** | Unknown variable (Possibly related to fuel flow or control input) |
| **Column 20** | Unknown variable (Possibly related to flight stability or attitude) |
| **Column 21** | Unknown variable (Possibly aerodynamic drag or velocity changes) |
| **Column 22** | Unknown variable (Possibly aircraft total weight or system state) |




## Section 1: Import Libraries and Read Data Files


In [ ]:

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
import warnings
warnings.filterwarnings('ignore')
import pickle

# Set random seeds for reproducibility
def set_random_seeds(seed=42):
    """Set all random seeds"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    # Enable deterministic computation for reproducibility
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

# Set random seed
set_random_seeds(42)

# Read CSV and Pickle data files
flight_header = pd.read_csv(str(resolve_data_file("flight_header.csv")))
flight_data_dict = pd.read_pickle(str(resolve_data_file("flight_data.pkl")))

print(flight_header.head())
print("flight_header shape:", flight_header.shape)
print("flight_data_dict shape:", flight_data_dict[2].shape)
print("flight_data_dict length:", len(flight_data_dict))

def preprocess_flight_data(flight_data_dict, flight_header):
    """
    Data preprocessing: handle missing values, infinite values, etc.

    Args:
    - flight_data_dict: Raw flight data dictionary
    - flight_header: Label information dataframe

    Returns:
    - processed_data_dict: Preprocessed data dictionary
    - processing_stats: Processing statistics
    """
    processed_data_dict = {}
    processing_stats = {
        'total_flights': len(flight_data_dict),
        'nan_ratios': [],
        'inf_ratios': [],
        'processed_flights': 0
    }

    print("Starting data preprocessing...")

    for master_idx, seq_data in flight_data_dict.items():
        # Check if exists in header
        if master_idx not in flight_header['Master Index'].values:
            continue

        # Convert to float32
        seq_data = seq_data.astype(np.float32)

        # Count missing values and infinite values
        nan_mask = np.isnan(seq_data)
        inf_mask = np.isinf(seq_data)

        nan_ratio = np.sum(nan_mask) / seq_data.size
        inf_ratio = np.sum(inf_mask) / seq_data.size

        processing_stats['nan_ratios'].append(nan_ratio)
        processing_stats['inf_ratios'].append(inf_ratio)

        # Handle infinite values: replace with NaN for unified processing
        seq_data[inf_mask] = np.nan

        # Handle missing values: time-series friendly interpolation
        if np.any(np.isnan(seq_data)):
            # Process each feature column separately
            for col_idx in range(seq_data.shape[1]):
                col_data = seq_data[:, col_idx]
                if np.any(np.isnan(col_data)):
                    # Forward fill
                    col_data = pd.Series(col_data).fillna(method='ffill')
                    # Backward fill (handle NaN at the beginning)
                    col_data = col_data.fillna(method='bfill')
                    # If still has NaN, fill with column mean
                    if col_data.isna().any():
                        col_mean = col_data.mean()
                        col_data = col_data.fillna(col_mean)
                    seq_data[:, col_idx] = col_data.values

        processed_data_dict[master_idx] = seq_data
        processing_stats['processed_flights'] += 1

    # Calculate processing statistics
    processing_stats['avg_nan_ratio'] = np.mean(processing_stats['nan_ratios'])
    processing_stats['avg_inf_ratio'] = np.mean(processing_stats['inf_ratios'])

    print(f"Data preprocessing completed!")
    print(f"Processed flights: {processing_stats['processed_flights']}")
    print(f"Average NaN ratio: {processing_stats['avg_nan_ratio']:.4f}")
    print(f"Average Inf ratio: {processing_stats['avg_inf_ratio']:.4f}")

    return processed_data_dict, processing_stats

def split_master_indices(flight_header, test_size=0.3, val_size=0.3, random_state=42):
    """
    Split master indices by flight-level grouping, then stratify by class to split train, validation and test sets
    Ensure all windows from the same flight only appear in one dataset to avoid data leakage

    Args:
    - flight_header: Label information dataframe
    - test_size: Test set ratio
    - val_size: Validation set ratio relative to train+validation set
    - random_state: Random seed

    Returns:
    - train_master_indices, val_master_indices, test_master_indices: Master index lists for each set
    """
    # Set random seed
    np.random.seed(random_state)
    random.seed(random_state)

    # Group master_index by class
    class_groups = flight_header.groupby('class')['Master Index'].apply(list).to_dict()

    train_indices = []
    val_indices = []
    test_indices = []

    print("Performing flight-level stratified dataset splitting...")
    print("Flight distribution by class:")

    for class_label, master_indices in class_groups.items():
        unique_indices = list(set(master_indices))  # Remove duplicates
        n_flights = len(unique_indices)

        print(f"Class {class_label}: {n_flights} flights")

        if n_flights < 10:
            print(f"  Too few flights, assigning all to training set")
            train_indices.extend(unique_indices)
            continue

        # First split test set
        train_val_indices, test_class_indices = train_test_split(
            unique_indices,
            test_size=test_size,
            random_state=random_state,
            shuffle=True
        )

        # Split train and validation from remaining data
        train_class_indices, val_class_indices = train_test_split(
            train_val_indices,
            test_size=val_size,
            random_state=random_state,
            shuffle=True
        )

        train_indices.extend(train_class_indices)
        val_indices.extend(val_class_indices)
        test_indices.extend(test_class_indices)

        print(f"  Training: {len(train_class_indices)} flights ({len(train_class_indices)/n_flights*100:.1f}%)")
        print(f"  Validation: {len(val_class_indices)} flights ({len(val_class_indices)/n_flights*100:.1f}%)")
        print(f"  Test: {len(test_class_indices)} flights ({len(test_class_indices)/n_flights*100:.1f}%)")

    print(f"\nOverall split results:")
    print(f"Training flights: {len(train_indices)}")
    print(f"Validation flights: {len(val_indices)}")
    print(f"Test flights: {len(test_indices)}")

    return train_indices, val_indices, test_indices

def sliding_window_with_masks(flight_data_dict, flight_header, master_indices, window_size=500, step_size=250, min_valid_ratio=0.8):
    """
    Generate sliding window samples for specified master_indices with explicit padding masks

    Args:
    - flight_data_dict: Preprocessed flight data dictionary
    - flight_header: Label information dataframe
    - master_indices: List of master_index to process
    - window_size: Sliding window size
    - step_size: Sliding window step size
    - min_valid_ratio: Minimum valid length ratio

    Returns:
    - samples: Sample dictionary containing data, masks and label information
    """
    samples = {}
    sample_id = 0

    # Create mapping from master_index to header information
    header_dict = {}
    for idx, row in flight_header.iterrows():
        master_idx = row['Master Index']
        header_dict[master_idx] = {
            'before_after': row['before_after'],
            'class': row['class'],
            'target_class': row['target_class'],
            'label': row['label'],
            'hclass': row['hclass'],
            'date_diff': row['date_diff']
        }

    print(f"Generating sliding window samples for {len(master_indices)} flights...")

    for master_idx in master_indices:
        if master_idx not in flight_data_dict or master_idx not in header_dict:
            continue

        seq_data = flight_data_dict[master_idx]
        label_info = header_dict[master_idx]
        seq_length = len(seq_data)

        if seq_length < window_size * min_valid_ratio:
            # Sequence too short, create a padded sample
            padded_data = np.zeros((window_size, seq_data.shape[1]), dtype=np.float32)
            mask = np.zeros(window_size, dtype=bool)

            actual_len = min(seq_length, window_size)
            padded_data[:actual_len, :] = seq_data[:actual_len, :]
            mask[:actual_len] = True

            samples[sample_id] = {
                "x": padded_data,
                "mask": mask,
                "valid_length": actual_len,
                "master_index": master_idx,
                "before_after": label_info['before_after'],
                "class": label_info['class'],
                "target_class": label_info['target_class'],
                "label": label_info['label'],
                "hclass": label_info['hclass'],
                "date_diff": label_info['date_diff'],
                "window_start": 0,
                "window_end": actual_len,
                "is_padded": True
            }
            sample_id += 1
        else:
            # Use sliding window to create multiple samples
            start = 0
            while start + window_size <= seq_length:
                window_data = seq_data[start:start + window_size, :].astype(np.float32)
                mask = np.ones(window_size, dtype=bool)  # Complete window, all valid

                samples[sample_id] = {
                    "x": window_data,
                    "mask": mask,
                    "valid_length": window_size,
                    "master_index": master_idx,
                    "before_after": label_info['before_after'],
                    "class": label_info['class'],
                    "target_class": label_info['target_class'],
                    "label": label_info['label'],
                    "hclass": label_info['hclass'],
                    "date_diff": label_info['date_diff'],
                    "window_start": start,
                    "window_end": start + window_size,
                    "is_padded": False
                }
                sample_id += 1
                start += step_size

            # Handle last incomplete window
            if start < seq_length:
                remaining_len = seq_length - start
                if remaining_len >= window_size * min_valid_ratio:
                    last_window = np.zeros((window_size, seq_data.shape[1]), dtype=np.float32)
                    mask = np.zeros(window_size, dtype=bool)

                    last_window[:remaining_len, :] = seq_data[start:, :]
                    mask[:remaining_len] = True

                    samples[sample_id] = {
                        "x": last_window,
                        "mask": mask,
                        "valid_length": remaining_len,
                        "master_index": master_idx,
                        "before_after": label_info['before_after'],
                        "class": label_info['class'],
                        "target_class": label_info['target_class'],
                        "label": label_info['label'],
                        "hclass": label_info['hclass'],
                        "date_diff": label_info['date_diff'],
                        "window_start": start,
                        "window_end": seq_length,
                        "is_padded": True
                    }
                    sample_id += 1

    return samples

def compute_scaling_params(train_samples):
    """
    Compute normalization parameters using only training set

    Args:
    - train_samples: Training set sample dictionary

    Returns:
    - scaler: Trained normalizer
    - scaling_stats: Normalization statistics
    """
    print("Computing training set normalization parameters...")

    # Collect all valid data points from training samples
    all_valid_data = []

    for sample in train_samples.values():
        x = sample['x']
        mask = sample['mask']
        # Only use valid data points
        valid_data = x[mask]
        if len(valid_data) > 0:
            all_valid_data.append(valid_data)

    # Combine all valid data
    if all_valid_data:
        combined_data = np.vstack(all_valid_data)
    else:
        raise ValueError("No valid data points in training set")

    # Use robust scaler (more robust to outliers)
    scaler = RobustScaler()
    scaler.fit(combined_data)

    scaling_stats = {
        'n_features': combined_data.shape[1],
        'n_samples': combined_data.shape[0],
        'center': scaler.center_,
        'scale': scaler.scale_
    }

    print(f"Normalization parameters computed!")
    print(f"Number of features: {scaling_stats['n_features']}")
    print(f"Total training sample points: {scaling_stats['n_samples']}")

    return scaler, scaling_stats

def apply_scaling(samples, scaler):
    """
    Apply normalization to sample data

    Args:
    - samples: Sample dictionary
    - scaler: Trained normalizer

    Returns:
    - scaled_samples: Normalized sample dictionary
    """
    scaled_samples = {}

    for sample_id, sample in samples.items():
        scaled_sample = sample.copy()
        x = sample['x'].copy()
        mask = sample['mask']

        # Save original shape
        original_shape = x.shape

        # Reshape to 2D for normalization
        x_reshaped = x.reshape(-1, x.shape[-1])
        x_scaled = scaler.transform(x_reshaped)

        # Restore original shape
        x_scaled = x_scaled.reshape(original_shape)

        # Keep padded parts as 0
        x_scaled[~mask] = 0.0

        scaled_sample['x'] = x_scaled.astype(np.float32)
        scaled_samples[sample_id] = scaled_sample

    return scaled_samples

def create_label_mappings(samples):
    """
    Create label mappings, converting string labels to numerical labels

    Args:
    - samples: Sample dictionary

    Returns:
    - label_mappings: Various label mapping dictionaries
    - encoded_samples: Samples with encoded labels
    """
    # Collect all unique labels
    unique_labels = set()
    unique_classes = set()
    unique_target_classes = set()
    unique_hclasses = set()
    unique_before_after = set()

    for sample_data in samples.values():
        unique_labels.add(sample_data['label'])
        unique_classes.add(sample_data['class'])
        unique_target_classes.add(sample_data['target_class'])
        unique_hclasses.add(sample_data['hclass'])
        unique_before_after.add(sample_data['before_after'])

    # Create label to number mappings
    label_mappings = {
        'label_to_id': {label: idx for idx, label in enumerate(sorted(unique_labels))},
        'class_to_id': {cls: idx for idx, cls in enumerate(sorted(unique_classes))},
        'target_class_to_id': {cls: idx for idx, cls in enumerate(sorted(unique_target_classes))},
        'hclass_to_id': {cls: idx for idx, cls in enumerate(sorted(unique_hclasses))},
        'before_after_to_id': {ba: idx for idx, ba in enumerate(sorted(unique_before_after))}
    }

    # Create reverse mappings
    label_mappings['id_to_label'] = {v: k for k, v in label_mappings['label_to_id'].items()}
    label_mappings['id_to_class'] = {v: k for k, v in label_mappings['class_to_id'].items()}
    label_mappings['id_to_target_class'] = {v: k for k, v in label_mappings['target_class_to_id'].items()}
    label_mappings['id_to_hclass'] = {v: k for k, v in label_mappings['hclass_to_id'].items()}
    label_mappings['id_to_before_after'] = {v: k for k, v in label_mappings['before_after_to_id'].items()}

    # Add encoded labels for each sample
    encoded_samples = {}
    for sample_id, sample_data in samples.items():
        encoded_sample = sample_data.copy()
        encoded_sample['label_encoded'] = label_mappings['label_to_id'][sample_data['label']]
        encoded_sample['class_encoded'] = label_mappings['class_to_id'][sample_data['class']]
        encoded_sample['target_class_encoded'] = label_mappings['target_class_to_id'][sample_data['target_class']]
        encoded_sample['hclass_encoded'] = label_mappings['hclass_to_id'][sample_data['hclass']]
        encoded_sample['before_after_encoded'] = label_mappings['before_after_to_id'][sample_data['before_after']]
        encoded_samples[sample_id] = encoded_sample

    return label_mappings, encoded_samples

def analyze_class_distribution(samples, dataset_name):
    """
    Analyze class distribution and detect class imbalance

    Args:
    - samples: Sample dictionary
    - dataset_name: Dataset name

    Returns:
    - class_weights: Class weight dictionary
    """
    class_count = {}
    total_samples = len(samples)

    for sample in samples.values():
        cls = sample['class_encoded']
        class_count[cls] = class_count.get(cls, 0) + 1

    print(f"\n{dataset_name} class distribution analysis:")
    print(f"Total samples: {total_samples}")

    class_weights = {}
    max_count = max(class_count.values())

    for cls, count in sorted(class_count.items()):
        ratio = count / total_samples
        weight = max_count / count  # Inverse proportion weight
        class_weights[cls] = weight
        print(f"  Class {cls}: {count} samples ({ratio:.2%}), weight: {weight:.3f}")

    # Check for severe imbalance
    min_ratio = min(class_count.values()) / max(class_count.values())
    if min_ratio < 0.1:
        print(f"⚠️  Severe class imbalance detected (min/max ratio: {min_ratio:.3f})")
        print("    Recommend using class weights or resampling techniques")

    return class_weights

def filter_nan_and_empty_samples(samples):
    """
    Filter out samples containing NaN or empty values

    Args:
    - samples: Sample dictionary

    Returns:
    - filtered_samples: Cleaned sample dictionary
    - removal_stats: Statistics about removed samples
    """
    filtered_samples = {}
    removal_stats = {
        'original_count': len(samples),
        'nan_count': 0,
        'empty_count': 0,
        'invalid_count': 0,
        'final_count': 0
    }

    print("Filtering NaN and empty values from samples...")

    for sample_id, sample in samples.items():
        x = sample['x']
        mask = sample['mask']

        # Check for NaN values in data
        has_nan = np.any(np.isnan(x))

        # Check for infinite values
        has_inf = np.any(np.isinf(x))

        # Check if sample is effectively empty (no valid data)
        valid_data = x[mask]
        is_empty = len(valid_data) == 0

        # Check if sample has invalid values in labels
        has_invalid_labels = any(pd.isna(val) for key, val in sample.items()
                                if key.endswith('_encoded') and isinstance(val, (int, float)))

        if has_nan or has_inf:
            removal_stats['nan_count'] += 1
        elif is_empty:
            removal_stats['empty_count'] += 1
        elif has_invalid_labels:
            removal_stats['invalid_count'] += 1
        else:
            # Sample is valid, keep it
            filtered_samples[sample_id] = sample

    removal_stats['final_count'] = len(filtered_samples)
    removal_stats['removed_count'] = removal_stats['original_count'] - removal_stats['final_count']

    print(f"Filtering completed:")
    print(f"  Original samples: {removal_stats['original_count']}")
    print(f"  Removed due to NaN/Inf: {removal_stats['nan_count']}")
    print(f"  Removed due to empty data: {removal_stats['empty_count']}")
    print(f"  Removed due to invalid labels: {removal_stats['invalid_count']}")
    print(f"  Final valid samples: {removal_stats['final_count']}")

    return filtered_samples, removal_stats

def save_processed_data(data_dict, filename, save_dir):
    """
    Save processed data to specified directory

    Args:
    - data_dict: Data dictionary to save
    - filename: File name
    - save_dir: Save directory
    """
    os.makedirs(save_dir, exist_ok=True)
    filepath = os.path.join(save_dir, filename)

    with open(filepath, 'wb') as f:
        pickle.dump(data_dict, f)

    print(f"Data saved to: {filepath}")
    print(f"File size: {os.path.getsize(filepath) / (1024*1024):.2f} MB")

# Set sliding window parameters
window_size = 50  # Sliding window size
step_size = 25    # Sliding window step size

print("="*60)
print("Starting improved data processing pipeline")
print("="*60)

# Step 1: Data preprocessing
processed_data_dict, processing_stats = preprocess_flight_data(flight_data_dict, flight_header)

# Step 2: Flight-level stratified dataset splitting (avoid data leakage)
train_master_indices, val_master_indices, test_master_indices = split_master_indices(
    flight_header, test_size=0.1, val_size=0.2, random_state=42
)

# Step 3: Generate sliding window samples for each dataset separately
print("\nGenerating training set sliding window samples...")
train_samples = sliding_window_with_masks(
    processed_data_dict, flight_header, train_master_indices,
    window_size=window_size, step_size=step_size
)

print("Generating validation set sliding window samples...")
val_samples = sliding_window_with_masks(
    processed_data_dict, flight_header, val_master_indices,
    window_size=window_size, step_size=step_size
)

print("Generating test set sliding window samples...")
test_samples = sliding_window_with_masks(
    processed_data_dict, flight_header, test_master_indices,
    window_size=window_size, step_size=step_size
)

print(f"\nSliding window sample generation completed!")
print(f"Training samples: {len(train_samples)}")
print(f"Validation samples: {len(val_samples)}")
print(f"Test samples: {len(test_samples)}")

# Step 4: Compute normalization parameters using only training set
scaler, scaling_stats = compute_scaling_params(train_samples)

# Step 5: Apply normalization to all datasets
print("\nApplying normalization...")
train_samples_scaled = apply_scaling(train_samples, scaler)
val_samples_scaled = apply_scaling(val_samples, scaler)
test_samples_scaled = apply_scaling(test_samples, scaler)

# Step 6: Create label mappings (based only on training set)
print("Creating label mappings...")
label_mappings, train_samples_encoded = create_label_mappings(train_samples_scaled)

# Apply same label mappings to validation and test sets
val_samples_encoded = {}
for sample_id, sample in val_samples_scaled.items():
    encoded_sample = sample.copy()
    encoded_sample['label_encoded'] = label_mappings['label_to_id'][sample['label']]
    encoded_sample['class_encoded'] = label_mappings['class_to_id'][sample['class']]
    encoded_sample['target_class_encoded'] = label_mappings['target_class_to_id'][sample['target_class']]
    encoded_sample['hclass_encoded'] = label_mappings['hclass_to_id'][sample['hclass']]
    encoded_sample['before_after_encoded'] = label_mappings['before_after_to_id'][sample['before_after']]
    val_samples_encoded[sample_id] = encoded_sample

test_samples_encoded = {}
for sample_id, sample in test_samples_scaled.items():
    encoded_sample = sample.copy()
    encoded_sample['label_encoded'] = label_mappings['label_to_id'][sample['label']]
    encoded_sample['class_encoded'] = label_mappings['class_to_id'][sample['class']]
    encoded_sample['target_class_encoded'] = label_mappings['target_class_to_id'][sample['target_class']]
    encoded_sample['hclass_encoded'] = label_mappings['hclass_to_id'][sample['hclass']]
    encoded_sample['before_after_encoded'] = label_mappings['before_after_to_id'][sample['before_after']]
    test_samples_encoded[sample_id] = encoded_sample

# Step 7: Filter NaN and empty values
print("\n" + "="*50)
print("FILTERING NaN AND EMPTY VALUES")
print("="*50)

train_samples_filtered, train_removal_stats = filter_nan_and_empty_samples(train_samples_encoded)
val_samples_filtered, val_removal_stats = filter_nan_and_empty_samples(val_samples_encoded)
test_samples_filtered, test_removal_stats = filter_nan_and_empty_samples(test_samples_encoded)

# Step 8: Analyze class distribution and compute class weights
train_class_weights = analyze_class_distribution(train_samples_filtered, "Training Set")
val_class_weights = analyze_class_distribution(val_samples_filtered, "Validation Set")
test_class_weights = analyze_class_distribution(test_samples_filtered, "Test Set")

# Update final sample dictionaries
pairs_train = train_samples_filtered
pairs_val = val_samples_filtered
pairs_test = test_samples_filtered

# Step 9: Save processed data to specified directory
save_dir = str(DATA_DIR)

print("\n" + "="*50)
print("SAVING PROCESSED DATA")
print("="*50)

# Save datasets
save_processed_data(pairs_train, "train_samples_processed.pkl", save_dir)
save_processed_data(pairs_val, "val_samples_processed.pkl", save_dir)
save_processed_data(pairs_test, "test_samples_processed.pkl", save_dir)

# Save additional files
save_processed_data(label_mappings, "label_mappings.pkl", save_dir)
save_processed_data(train_class_weights, "train_class_weights.pkl", save_dir)
save_processed_data(scaler, "scaler.pkl", save_dir)
save_processed_data(scaling_stats, "scaling_stats.pkl", save_dir)

# Save processing statistics
processing_summary = {
    'window_size': window_size,
    'step_size': step_size,
    'train_removal_stats': train_removal_stats,
    'val_removal_stats': val_removal_stats,
    'test_removal_stats': test_removal_stats,
    'processing_stats': processing_stats,
    'scaling_stats': scaling_stats
}
save_processed_data(processing_summary, "processing_summary.pkl", save_dir)

print("\n" + "="*60)
print("Data processing pipeline completion summary")
print("="*60)

print(f"\n📊 Final dataset statistics:")
print(f"  Training set: {len(pairs_train)} samples ({len(train_master_indices)} flights)")
print(f"  Validation set: {len(pairs_val)} samples ({len(val_master_indices)} flights)")
print(f"  Test set: {len(pairs_test)} samples ({len(test_master_indices)} flights)")

print(f"\n🧹 Data cleaning results:")
print(f"  Training set: {train_removal_stats['removed_count']} samples removed")
print(f"  Validation set: {val_removal_stats['removed_count']} samples removed")
print(f"  Test set: {test_removal_stats['removed_count']} samples removed")

print(f"\n🏷️ Label mapping information:")
print(f"  Label categories: {len(label_mappings['label_to_id'])}")
print(f"  Main classes: {len(label_mappings['class_to_id'])}")
print(f"  Target classes: {len(label_mappings['target_class_to_id'])}")
print(f"  Hierarchical classes: {len(label_mappings['hclass_to_id'])}")
print(f"  Before/after maintenance: {label_mappings['before_after_to_id']}")

print(f"\n📏 Data preprocessing information:")
print(f"  Window size: {window_size}")
print(f"  Step size: {step_size}")
print(f"  Feature dimensions: {scaling_stats['n_features']}")
print(f"  Normalization method: RobustScaler (robust to outliers)")
print(f"  Random seed: 42")

print(f"\n🔍 Data quality statistics:")
print(f"  Average NaN ratio: {processing_stats['avg_nan_ratio']:.4f}")
print(f"  Average Inf ratio: {processing_stats['avg_inf_ratio']:.4f}")

print(f"\n💾 Saved files:")
print(f"  📁 Save directory: {save_dir}")
print(f"  📄 train_samples_processed.pkl")
print(f"  📄 val_samples_processed.pkl")
print(f"  📄 test_samples_processed.pkl")
print(f"  📄 label_mappings.pkl")
print(f"  📄 train_class_weights.pkl")
print(f"  📄 scaler.pkl")
print(f"  📄 scaling_stats.pkl")
print(f"  📄 processing_summary.pkl")

print(f"\n💡 Key improvements:")
print("  ✅ Flight-level grouping to avoid data leakage")
print("  ✅ Normalization parameters computed only from training set")
print("  ✅ Explicit padding masks and valid lengths")
print("  ✅ Time-series friendly missing value handling")
print("  ✅ Class imbalance detection and weight calculation")
print("  ✅ Complete reproducibility settings")
print("  ✅ NaN and empty value filtering")
print("  ✅ Data saved to Google Drive")

# Display sample information
print(f"\n📝 Sample information:")
if len(pairs_train) > 0:
    sample_key = list(pairs_train.keys())[0]
    sample = pairs_train[sample_key]
    print(f"  Data shape: {sample['x'].shape}")
    print(f"  Mask shape: {sample['mask'].shape}")
    print(f"  Valid length: {sample['valid_length']}")
    print(f"  Contains padding: {sample['is_padded']}")
    print(f"  Encoded labels: class={sample['class_encoded']}, before_after={sample['before_after_encoded']}")

print(f"\n🎯 Variables ready for model training:")
print("  pairs_train, pairs_val, pairs_test - clean preprocessed sample dictionaries")
print("  label_mappings - label mapping dictionary")
print("  train_class_weights - training set class weights")
print("  scaler - normalizer (for new data prediction)")
print("  scaling_stats - normalization statistics")

print(f"\n✨ All processed data has been saved and is ready for model training!")


## Section 2: Raw Data Visualization


In [ ]:
import matplotlib.pyplot as plt

# Set Chinese font support
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# Select a sample for plotting
def plot_before_after_maintenance_data(flight_data_dict, flight_header):
    """
    Plot raw sensor channel data before and after maintenance for all channels

    Parameters:
    - flight_data_dict: Flight data dictionary
    - flight_header: Flight header information
    """
    # Find samples before and after maintenance
    before_sample = None
    after_sample = None

    for idx, row in flight_header.iterrows():
        master_idx = row['Master Index']
        if master_idx in flight_data_dict:
            if row['before_after'] == 1 and before_sample is None:  # Before maintenance
                before_sample = {
                    'master_idx': master_idx,
                    'data': flight_data_dict[master_idx],
                    'info': row
                }
            elif row['before_after'] == 0 and after_sample is None:  # After maintenance
                after_sample = {
                    'master_idx': master_idx,
                    'data': flight_data_dict[master_idx],
                    'info': row
                }

        # Exit after finding both samples
        if before_sample is not None and after_sample is not None:
            break

    if before_sample is None or after_sample is None:
        print("Unable to find suitable before and after maintenance sample data")
        return

    # Get number of sensor channels
    num_channels = before_sample['data'].shape[1]

    # Create subplots for all channels
    fig, axes = plt.subplots(num_channels, 1, figsize=(15, 3*num_channels))
    if num_channels == 1:
        axes = [axes]

    for sensor_channel in range(num_channels):
        # Get before maintenance data
        before_data = before_sample['data'][:, sensor_channel]
        before_time = range(len(before_data))

        # Get after maintenance data
        after_data = after_sample['data'][:, sensor_channel]
        # After maintenance time axis starts after before maintenance data ends
        after_time = range(len(before_data), len(before_data) + len(after_data))

        # Plot before maintenance data
        axes[sensor_channel].plot(before_time, before_data, 'b-', linewidth=1.5, alpha=0.8)

        # Plot after maintenance data
        axes[sensor_channel].plot(after_time, after_data, 'r-', linewidth=1.5, alpha=0.8)

        # Add dividing line
        axes[sensor_channel].axvline(x=len(before_data), color='green', linestyle='--', linewidth=2)

        # Set subplot properties
        axes[sensor_channel].set_title(f'Sensor Channel {sensor_channel} Raw Data Comparison Before and After Maintenance',
                                      fontsize=12, fontweight='bold')
        axes[sensor_channel].set_xlabel('Time Step', fontsize=10)
        axes[sensor_channel].set_ylabel('Sensor Value', fontsize=10)

    # Add overall title
    fig.suptitle(f'All Sensor Channels Data Comparison Before and After Maintenance\n'
                f'Before Maintenance Class: {before_sample["info"]["class"]}, After Maintenance Class: {after_sample["info"]["class"]}',
                fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.show()

    # Print detailed information
    print(f"Before Maintenance Sample (Master Index: {before_sample['master_idx']}):")
    print(f"  Data Shape: {before_sample['data'].shape}")
    print(f"  Label: {before_sample['info']['label']}")
    print(f"  Class: {before_sample['info']['class']}")
    print(f"  Date Difference: {before_sample['info']['date_diff']}")

    print(f"\nAfter Maintenance Sample (Master Index: {after_sample['master_idx']}):")
    print(f"  Data Shape: {after_sample['data'].shape}")
    print(f"  Label: {after_sample['info']['label']}")
    print(f"  Class: {after_sample['info']['class']}")
    print(f"  Date Difference: {after_sample['info']['date_diff']}")

# Plot before/after maintenance data comparison for all sensor channels
print("Plotting raw data comparison for all sensor channels before and after maintenance...")
plot_before_after_maintenance_data(flight_data_dict, flight_header)


## Section 3: Model Architecture


In [ ]:

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
import warnings
warnings.filterwarnings('ignore')
import pickle

# Set random seeds for reproducibility
def set_random_seeds(seed=42):
    """Set all random seeds"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    # Enable deterministic computation for reproducibility
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

# Set random seed
set_random_seeds(42)

# Read CSV and Pickle data files
flight_header = pd.read_csv(str(resolve_data_file("flight_header.csv")))
flight_data_dict = pd.read_pickle(str(resolve_data_file("flight_data.pkl")))

print(flight_header.head())
print("flight_header shape:", flight_header.shape)
print("flight_data_dict shape:", flight_data_dict[2].shape)
print("flight_data_dict length:", len(flight_data_dict))

def preprocess_flight_data(flight_data_dict, flight_header):
    """
    Data preprocessing: handle missing values, infinities, and related issues

    Args:
    - flight_data_dict: raw flight data dictionary
    - flight_header: dataframe containing label information

    Returns:
    - processed_data_dict: preprocessed data dictionary
    - processing_stats: preprocessing statistics
    """
    processed_data_dict = {}
    processing_stats = {
        'total_flights': len(flight_data_dict),
        'nan_ratios': [],
        'inf_ratios': [],
        'processed_flights': 0
    }

    print("Starting data preprocessing...")

    for master_idx, seq_data in flight_data_dict.items():
        # Check if exists in header
        if master_idx not in flight_header['Master Index'].values:
            continue

        # Convert to float32
        seq_data = seq_data.astype(np.float32)

        # Count missing values and infinite values
        nan_mask = np.isnan(seq_data)
        inf_mask = np.isinf(seq_data)

        nan_ratio = np.sum(nan_mask) / seq_data.size
        inf_ratio = np.sum(inf_mask) / seq_data.size

        processing_stats['nan_ratios'].append(nan_ratio)
        processing_stats['inf_ratios'].append(inf_ratio)

        # Handle infinite values: replace with NaN for unified processing
        seq_data[inf_mask] = np.nan

        # Handle missing values: time-series friendly interpolation
        if np.any(np.isnan(seq_data)):
            # Process each feature column separately
            for col_idx in range(seq_data.shape[1]):
                col_data = seq_data[:, col_idx]
                if np.any(np.isnan(col_data)):
                    # Forward fill
                    col_data = pd.Series(col_data).fillna(method='ffill')
                    # Backward fill (handle NaN at the beginning)
                    col_data = col_data.fillna(method='bfill')
                    # If still has NaN, fill with column mean
                    if col_data.isna().any():
                        col_mean = col_data.mean()
                        col_data = col_data.fillna(col_mean)
                    seq_data[:, col_idx] = col_data.values

        processed_data_dict[master_idx] = seq_data
        processing_stats['processed_flights'] += 1

    # Calculate processing statistics
    processing_stats['avg_nan_ratio'] = np.mean(processing_stats['nan_ratios'])
    processing_stats['avg_inf_ratio'] = np.mean(processing_stats['inf_ratios'])

    print(f"Data preprocessing completed!")
    print(f"Processed flights: {processing_stats['processed_flights']}")
    print(f"Average NaN ratio: {processing_stats['avg_nan_ratio']:.4f}")
    print(f"Average Inf ratio: {processing_stats['avg_inf_ratio']:.4f}")

    return processed_data_dict, processing_stats

def split_master_indices(flight_header, random_state=42):
    """
    Split the dataset using different ratios based on the number of samples per class
    For classes with many samples (>= 100 flights per class): randomly sample 10% for training, 10% for testing, and 10% for validation without replacement
    For classes with fewer samples (< 100 flights per class): use a 60% / 20% / 20% split for train / test / validation
    Ensure that all windows from the same flight appear in only one split to avoid data leakage

    Args:
    - flight_header: dataframe containing label information
    - random_state: Random seed

    Returns:
    - train_master_indices, val_master_indices, test_master_indices: master-index lists for each split
    """
    # Set the random seed
    np.random.seed(random_state)
    random.seed(random_state)

    # Group master indices by class
    class_groups = flight_header.groupby('class')['Master Index'].apply(list).to_dict()

    train_indices = []
    val_indices = []
    test_indices = []

    print("Running stratified dataset splitting based on per-class sample counts...")
    print("Flight distribution by class:")

    for class_label, master_indices in class_groups.items():
        unique_indices = list(set(master_indices))  # Remove duplicates
        n_flights = len(unique_indices)

        print(f"Class {class_label}: {n_flights} flights")

        if n_flights < 10:
            print(f"  Too few flights; assigning all of them to the training set")
            train_indices.extend(unique_indices)
            continue

        if n_flights >= 100:
            # For classes with many samples: sample 10% for training, 10% for testing, and 10% for validation without replacement
            # Leave the remaining 70% unused to balance the dataset
            train_size = int(n_flights * 0.1)
            test_size = int(n_flights * 0.1)
            val_size = int(n_flights * 0.1)

            # Randomly shuffle the indices
            shuffled_indices = unique_indices.copy()
            np.random.shuffle(shuffled_indices)

            # Assign samples
            train_class_indices = shuffled_indices[:train_size]
            test_class_indices = shuffled_indices[train_size:train_size + test_size]
            val_class_indices = shuffled_indices[train_size + test_size:train_size + test_size + val_size]

            print(f"  Sufficient samples available; using a 10% / 10% / 10% split (remaining 70% unused)")
            print(f"  Train: {len(train_class_indices)} flights (10%)")
            print(f"  Validation: {len(val_class_indices)} flights (10%)")
            print(f"  Test: {len(test_class_indices)} flights (10%)")
            print(f"  Unused: {n_flights - len(train_class_indices) - len(val_class_indices) - len(test_class_indices)} flights (70%)")
        else:
            # For classes with fewer samples: use a 60% / 20% / 20% split
            # First split out the test set (20%)
            train_val_indices, test_class_indices = train_test_split(
                unique_indices,
                test_size=0.2,
                random_state=random_state,
                shuffle=True
            )

            # Split the remaining data into train and validation (60% / 20% overall becomes 75% / 25% of the remainder)
            train_class_indices, val_class_indices = train_test_split(
                train_val_indices,
                test_size=0.25,  # 0.25 * 0.8 = 0.2 (20% of the full dataset)
                random_state=random_state,
                shuffle=True
            )

            print(f"  Fewer samples available; using a 60% / 20% / 20% split")
            print(f"  Train: {len(train_class_indices)} flights ({len(train_class_indices)/n_flights*100:.1f}%)")
            print(f"  Validation: {len(val_class_indices)} flights ({len(val_class_indices)/n_flights*100:.1f}%)")
            print(f"  Test: {len(test_class_indices)} flights ({len(test_class_indices)/n_flights*100:.1f}%)")

        train_indices.extend(train_class_indices)
        val_indices.extend(val_class_indices)
        test_indices.extend(test_class_indices)

    print(f"\nOverall split summary:")
    print(f"Training flights: {len(train_indices)}")
    print(f"Validation flights: {len(val_indices)}")
    print(f"Test flights: {len(test_indices)}")

    return train_indices, val_indices, test_indices

def sliding_window_with_masks(flight_data_dict, flight_header, master_indices, window_size=500, step_size=250, min_valid_ratio=0.8):
    """
    Generate sliding-window samples for the specified master_indices, with explicit padding masks

    Args:
    - flight_data_dict: preprocessed flight data dictionary
    - flight_header: dataframe containing label information
    - master_indices: list of master indices to process
    - window_size: sliding-window size
    - step_size: sliding-window step size
    - min_valid_ratio: minimum valid-length ratio

    Returns:
    - samples: sample dictionary containing data, masks, and label information
    """
    samples = {}
    sample_id = 0

    # Build a mapping from master_index to header information
    header_dict = {}
    for idx, row in flight_header.iterrows():
        master_idx = row['Master Index']
        header_dict[master_idx] = {
            'before_after': row['before_after'],
            'class': row['class'],
            'target_class': row['target_class'],
            'label': row['label'],
            'hclass': row['hclass'],
            'date_diff': row['date_diff']
        }

    print(f"Generating sliding-window samples for {len(master_indices)} flights...")

    for master_idx in master_indices:
        if master_idx not in flight_data_dict or master_idx not in header_dict:
            continue

        seq_data = flight_data_dict[master_idx]
        label_info = header_dict[master_idx]
        seq_length = len(seq_data)

        if seq_length < window_size * min_valid_ratio:
            # Sequence is too short; create one padded sample
            padded_data = np.zeros((window_size, seq_data.shape[1]), dtype=np.float32)
            mask = np.zeros(window_size, dtype=bool)

            actual_len = min(seq_length, window_size)
            padded_data[:actual_len, :] = seq_data[:actual_len, :]
            mask[:actual_len] = True

            samples[sample_id] = {
                "x": padded_data,
                "mask": mask,
                "valid_length": actual_len,
                "master_index": master_idx,
                "before_after": label_info['before_after'],
                "class": label_info['class'],
                "target_class": label_info['target_class'],
                "label": label_info['label'],
                "hclass": label_info['hclass'],
                "date_diff": label_info['date_diff'],
                "window_start": 0,
                "window_end": actual_len,
                "is_padded": True
            }
            sample_id += 1
        else:
            # Create multiple samples using a sliding window
            start = 0
            while start + window_size <= seq_length:
                window_data = seq_data[start:start + window_size, :].astype(np.float32)
                mask = np.ones(window_size, dtype=bool)  # Full window, all positions valid

                samples[sample_id] = {
                    "x": window_data,
                    "mask": mask,
                    "valid_length": window_size,
                    "master_index": master_idx,
                    "before_after": label_info['before_after'],
                    "class": label_info['class'],
                    "target_class": label_info['target_class'],
                    "label": label_info['label'],
                    "hclass": label_info['hclass'],
                    "date_diff": label_info['date_diff'],
                    "window_start": start,
                    "window_end": start + window_size,
                    "is_padded": False
                }
                sample_id += 1
                start += step_size

            # Handle the last incomplete window
            if start < seq_length:
                remaining_len = seq_length - start
                if remaining_len >= window_size * min_valid_ratio:
                    last_window = np.zeros((window_size, seq_data.shape[1]), dtype=np.float32)
                    mask = np.zeros(window_size, dtype=bool)

                    last_window[:remaining_len, :] = seq_data[start:, :]
                    mask[:remaining_len] = True

                    samples[sample_id] = {
                        "x": last_window,
                        "mask": mask,
                        "valid_length": remaining_len,
                        "master_index": master_idx,
                        "before_after": label_info['before_after'],
                        "class": label_info['class'],
                        "target_class": label_info['target_class'],
                        "label": label_info['label'],
                        "hclass": label_info['hclass'],
                        "date_diff": label_info['date_diff'],
                        "window_start": start,
                        "window_end": seq_length,
                        "is_padded": True
                    }
                    sample_id += 1

    return samples

def compute_scaling_params(train_samples):
    """
    Compute normalization parameters using only the training set

    Args:
    - train_samples: training sample dictionary

    Returns:
    - scaler: fitted scaler
    - scaling_stats: normalization statistics
    """
    print("Computing normalization parameters from the training set...")

    # Collect all valid data points from the training samples
    all_valid_data = []

    for sample in train_samples.values():
        x = sample['x']
        mask = sample['mask']
        # Use only valid data points
        valid_data = x[mask]
        if len(valid_data) > 0:
            all_valid_data.append(valid_data)

    # Concatenate all valid data
    if all_valid_data:
        combined_data = np.vstack(all_valid_data)
    else:
        raise ValueError("No valid data points were found in the training set")

    # Use RobustScaler for better resistance to outliers
    scaler = RobustScaler()
    scaler.fit(combined_data)

    scaling_stats = {
        'n_features': combined_data.shape[1],
        'n_samples': combined_data.shape[0],
        'center': scaler.center_,
        'scale': scaler.scale_
    }

    print(f"Normalization parameters computed successfully!")
    print(f"Number of features: {scaling_stats['n_features']}")
    print(f"Total number of training sample points: {scaling_stats['n_samples']}")

    return scaler, scaling_stats

def apply_scaling(samples, scaler):
    """
    Apply normalization to the sample data

    Args:
    - samples: sample dictionary
    - scaler: fitted scaler

    Returns:
    - scaled_samples: normalized sample dictionary
    """
    scaled_samples = {}

    for sample_id, sample in samples.items():
        scaled_sample = sample.copy()
        x = sample['x'].copy()
        mask = sample['mask']

        # Preserve the original shape
        original_shape = x.shape

        # Reshape to 2D for normalization
        x_reshaped = x.reshape(-1, x.shape[-1])
        x_scaled = scaler.transform(x_reshaped)

        # Restore the original shape
        x_scaled = x_scaled.reshape(original_shape)

        # Keep padded positions at 0
        x_scaled[~mask] = 0.0

        scaled_sample['x'] = x_scaled.astype(np.float32)
        scaled_samples[sample_id] = scaled_sample

    return scaled_samples

def create_label_mappings(samples):
    """
    Create label mappings that convert string labels to numeric labels

    Args:
    - samples: sample dictionary

    Returns:
    - label_mappings: dictionary of label-mapping tables
    - encoded_samples: samples with encoded labels
    """
    # Collect all unique labels
    unique_labels = set()
    unique_classes = set()
    unique_target_classes = set()
    unique_hclasses = set()
    unique_before_after = set()

    for sample_data in samples.values():
        unique_labels.add(sample_data['label'])
        unique_classes.add(sample_data['class'])
        unique_target_classes.add(sample_data['target_class'])
        unique_hclasses.add(sample_data['hclass'])
        unique_before_after.add(sample_data['before_after'])

    # Create mappings from labels to numeric IDs
    label_mappings = {
        'label_to_id': {label: idx for idx, label in enumerate(sorted(unique_labels))},
        'class_to_id': {cls: idx for idx, cls in enumerate(sorted(unique_classes))},
        'target_class_to_id': {cls: idx for idx, cls in enumerate(sorted(unique_target_classes))},
        'hclass_to_id': {cls: idx for idx, cls in enumerate(sorted(unique_hclasses))},
        'before_after_to_id': {ba: idx for idx, ba in enumerate(sorted(unique_before_after))}
    }

    # Create reverse mappings
    label_mappings['id_to_label'] = {v: k for k, v in label_mappings['label_to_id'].items()}
    label_mappings['id_to_class'] = {v: k for k, v in label_mappings['class_to_id'].items()}
    label_mappings['id_to_target_class'] = {v: k for k, v in label_mappings['target_class_to_id'].items()}
    label_mappings['id_to_hclass'] = {v: k for k, v in label_mappings['hclass_to_id'].items()}
    label_mappings['id_to_before_after'] = {v: k for k, v in label_mappings['before_after_to_id'].items()}

    # Add encoded labels to each sample
    encoded_samples = {}
    for sample_id, sample_data in samples.items():
        encoded_sample = sample_data.copy()
        encoded_sample['label_encoded'] = label_mappings['label_to_id'][sample_data['label']]
        encoded_sample['class_encoded'] = label_mappings['class_to_id'][sample_data['class']]
        encoded_sample['target_class_encoded'] = label_mappings['target_class_to_id'][sample_data['target_class']]
        encoded_sample['hclass_encoded'] = label_mappings['hclass_to_id'][sample_data['hclass']]
        encoded_sample['before_after_encoded'] = label_mappings['before_after_to_id'][sample_data['before_after']]
        encoded_samples[sample_id] = encoded_sample

    return label_mappings, encoded_samples

def analyze_class_distribution(samples, dataset_name):
    """
    Analyze class distribution and detect class imbalance

    Args:
    - samples: sample dictionary
    - dataset_name: dataset name

    Returns:
    - class_weights: class-weight dictionary
    """
    class_count = {}
    total_samples = len(samples)

    for sample in samples.values():
        cls = sample['class_encoded']
        class_count[cls] = class_count.get(cls, 0) + 1

    print(f"\n{dataset_name} class distribution analysis:")
    print(f"Total samples: {total_samples}")

    class_weights = {}
    max_count = max(class_count.values())

    for cls, count in sorted(class_count.items()):
        ratio = count / total_samples
        weight = max_count / count  # Inverse-frequency weight
        class_weights[cls] = weight
        print(f"  Class {cls}: {count} samples ({ratio:.2%}), weight: {weight:.3f}")

    # Check for severe imbalance
    min_ratio = min(class_count.values()) / max(class_count.values())
    if min_ratio < 0.1:
        print(f"⚠️  Severe class imbalance detected (min/max ratio: {min_ratio:.3f})")
        print("    Consider using class weights or resampling techniques")

    return class_weights

def filter_nan_and_empty_samples(samples):
    """
    Filter out samples containing NaN values or empty content

    Args:
    - samples: sample dictionary

    Returns:
    - filtered_samples: cleaned sample dictionary
    - removal_stats: sample-removal statistics
    """
    filtered_samples = {}
    removal_stats = {
        'original_count': len(samples),
        'nan_count': 0,
        'empty_count': 0,
        'invalid_count': 0,
        'final_count': 0
    }

    print("Filtering NaN and empty samples...")

    for sample_id, sample in samples.items():
        x = sample['x']
        mask = sample['mask']

        # Check for NaN values in the data
        has_nan = np.any(np.isnan(x))

        # Check for infinite values
        has_inf = np.any(np.isinf(x))

        # Check whether the sample is effectively empty(no valid data)
        valid_data = x[mask]
        is_empty = len(valid_data) == 0

        # Check whether the sample labels contain invalid values
        has_invalid_labels = any(pd.isna(val) for key, val in sample.items()
                                if key.endswith('_encoded') and isinstance(val, (int, float)))

        if has_nan or has_inf:
            removal_stats['nan_count'] += 1
        elif is_empty:
            removal_stats['empty_count'] += 1
        elif has_invalid_labels:
            removal_stats['invalid_count'] += 1
        else:
            # Sample is valid; keep it
            filtered_samples[sample_id] = sample

    removal_stats['final_count'] = len(filtered_samples)
    removal_stats['removed_count'] = removal_stats['original_count'] - removal_stats['final_count']

    print(f"Filtering completed:")
    print(f"  Original samples: {removal_stats['original_count']}")
    print(f"  Removed due to NaN/Inf: {removal_stats['nan_count']}")
    print(f"  Removed due to empty data: {removal_stats['empty_count']}")
    print(f"  Removed due to invalid labels: {removal_stats['invalid_count']}")
    print(f"  Final valid samples: {removal_stats['final_count']}")

    return filtered_samples, removal_stats

def save_processed_data(data_dict, filename, save_dir):
    """
    Save the processed data to the specified directory

    Args:
    - data_dict: data dictionary to save
    - filename: file name
    - save_dir: output directory
    """
    os.makedirs(save_dir, exist_ok=True)
    filepath = os.path.join(save_dir, filename)

    with open(filepath, 'wb') as f:
        pickle.dump(data_dict, f)

    print(f"Data saved to: {filepath}")
    print(f"File size: {os.path.getsize(filepath) / (1024*1024):.2f} MB")

# Set sliding-window parameters
window_size = 500  # sliding-window size
step_size = 250    # sliding-window step size

print("="*60)
print("Starting the improved data-processing pipeline")
print("="*60)

# Step 1: data preprocessing
processed_data_dict, processing_stats = preprocess_flight_data(flight_data_dict, flight_header)

# Step 2: split the dataset by per-class sample counts while avoiding data leakage
train_master_indices, val_master_indices, test_master_indices = split_master_indices(
    flight_header, random_state=42
)

# Step 3: generate sliding-window samples for each split
print("\nGenerating training sliding-window samples...")
train_samples = sliding_window_with_masks(
    processed_data_dict, flight_header, train_master_indices,
    window_size=window_size, step_size=step_size
)

print("Generating validation sliding-window samples...")
val_samples = sliding_window_with_masks(
    processed_data_dict, flight_header, val_master_indices,
    window_size=window_size, step_size=step_size
)

print("Generating test sliding-window samples...")
test_samples = sliding_window_with_masks(
    processed_data_dict, flight_header, test_master_indices,
    window_size=window_size, step_size=step_size
)

print(f"\nSliding-window sample generation completed！")
print(f"Training samples: {len(train_samples)}")
print(f"Validation samples: {len(val_samples)}")
print(f"Test samples: {len(test_samples)}")

# Step4:Compute normalization parameters using only the training set
scaler, scaling_stats = compute_scaling_params(train_samples)

# Step 5: apply normalization to all splits
print("\nApplying normalization...")
train_samples_scaled = apply_scaling(train_samples, scaler)
val_samples_scaled = apply_scaling(val_samples, scaler)
test_samples_scaled = apply_scaling(test_samples, scaler)

# Step 6: create label mappings using only the training set
print("Creating label mappings...")
label_mappings, train_samples_encoded = create_label_mappings(train_samples_scaled)

# Apply the same label mappings to the validation and test sets
val_samples_encoded = {}
for sample_id, sample in val_samples_scaled.items():
    encoded_sample = sample.copy()
    encoded_sample['label_encoded'] = label_mappings['label_to_id'][sample['label']]
    encoded_sample['class_encoded'] = label_mappings['class_to_id'][sample['class']]
    encoded_sample['target_class_encoded'] = label_mappings['target_class_to_id'][sample['target_class']]
    encoded_sample['hclass_encoded'] = label_mappings['hclass_to_id'][sample['hclass']]
    encoded_sample['before_after_encoded'] = label_mappings['before_after_to_id'][sample['before_after']]
    val_samples_encoded[sample_id] = encoded_sample

test_samples_encoded = {}
for sample_id, sample in test_samples_scaled.items():
    encoded_sample = sample.copy()
    encoded_sample['label_encoded'] = label_mappings['label_to_id'][sample['label']]
    encoded_sample['class_encoded'] = label_mappings['class_to_id'][sample['class']]
    encoded_sample['target_class_encoded'] = label_mappings['target_class_to_id'][sample['target_class']]
    encoded_sample['hclass_encoded'] = label_mappings['hclass_to_id'][sample['hclass']]
    encoded_sample['before_after_encoded'] = label_mappings['before_after_to_id'][sample['before_after']]
    test_samples_encoded[sample_id] = encoded_sample

# Step 7: filter NaN and empty values
print("\n" + "="*50)
print("Filtering NaN and empty values")
print("="*50)

train_samples_filtered, train_removal_stats = filter_nan_and_empty_samples(train_samples_encoded)
val_samples_filtered, val_removal_stats = filter_nan_and_empty_samples(val_samples_encoded)
test_samples_filtered, test_removal_stats = filter_nan_and_empty_samples(test_samples_encoded)

# Step8:analyze class distribution and compute class weights
train_class_weights = analyze_class_distribution(train_samples_filtered, "Training set")
val_class_weights = analyze_class_distribution(val_samples_filtered, "Validation set")
test_class_weights = analyze_class_distribution(test_samples_filtered, "Test set")

# Update the final sample dictionaries
pairs_train = train_samples_filtered
pairs_val = val_samples_filtered
pairs_test = test_samples_filtered

# Step9:Save the processed data to the specified directory
save_dir = str(DATA_DIR)

print("\n" + "="*50)
print("Saving processed data")
print("="*50)

# Save datasets
save_processed_data(pairs_train, "train_samples_processed_sample_len_150.pkl", save_dir)
save_processed_data(pairs_val, "val_samples_processed_sample_len_150.pkl", save_dir)
save_processed_data(pairs_test, "test_samples_processed_sample_len_150.pkl", save_dir)

# Save supporting files
save_processed_data(label_mappings, "label_mappings_sample_len_500.pkl", save_dir)
save_processed_data(train_class_weights, "train_class_weights_sample_len_500.pkl", save_dir)
save_processed_data(scaler, "scaler_sample_len_500.pkl", save_dir)
save_processed_data(scaling_stats, "scaling_stats_sample_len_500.pkl", save_dir)

# Savepreprocessing statistics
processing_summary = {
    'window_size': window_size,
    'step_size': step_size,
    'train_removal_stats': train_removal_stats,
    'val_removal_stats': val_removal_stats,
    'test_removal_stats': test_removal_stats,
    'processing_stats': processing_stats,
    'scaling_stats': scaling_stats
}
save_processed_data(processing_summary, "processing_summary_sample_len_150.pkl", save_dir)

print("\n" + "="*60)
print("Data-processing pipeline summary")
print("="*60)

print(f"\n📊 Final dataset statistics:")
print(f"  Training set: {len(pairs_train)} samples ({len(train_master_indices)} flights)")
print(f"  Validation set: {len(pairs_val)} samples ({len(val_master_indices)} flights)")
print(f"  Test set: {len(pairs_test)} samples ({len(test_master_indices)} flights)")

print(f"\n🧹 Data cleaning summary:")
print(f"  Training set: removed {train_removal_stats['removed_count']} samples")
print(f"  Validation set: removed {val_removal_stats['removed_count']} samples")
print(f"  Test set: removed {test_removal_stats['removed_count']} samples")

print(f"\n🏷️ Label mapping summary:")
print(f"  Label categories: {len(label_mappings['label_to_id'])}")
print(f"  Main classes: {len(label_mappings['class_to_id'])}")
print(f"  Target classes: {len(label_mappings['target_class_to_id'])}")
print(f"  Hierarchical classes: {len(label_mappings['hclass_to_id'])}")
print(f"  Before/after maintenance: {label_mappings['before_after_to_id']}")

print(f"\n📏 Preprocessing configuration:")
print(f"  Window size: {window_size}")
print(f"  Step size: {step_size}")
print(f"  Feature dimension: {scaling_stats['n_features']}")
print(f"  Normalization method: RobustScaler (robust to outliers)")
print(f"  Random seed: 42")

print(f"\n🔍 Data-quality statistics:")
print(f"  Average NaN ratio: {processing_stats['avg_nan_ratio']:.4f}")
print(f"  Average Inf ratio: {processing_stats['avg_inf_ratio']:.4f}")

print(f"\n💾 Saved files:")
print(f"  📁 output directory: {save_dir}")
print(f"  📄 train_samples_processed.pkl")
print(f"  📄 val_samples_processed.pkl")
print(f"  📄 test_samples_processed.pkl")
print(f"  📄 label_mappings.pkl")
print(f"  📄 train_class_weights.pkl")
print(f"  📄 scaler.pkl")
print(f"  📄 scaling_stats.pkl")
print(f"  📄 processing_summary.pkl")

print(f"\n💡 Key improvements:")
print("  ✅ Adaptive split strategy based on per-class sample counts")
print("  ✅ For data-rich classes: 10% / 10% / 10% split without replacement")
print("  ✅ For data-scarce classes: 60% / 20% / 20% split")
print("  ✅ Flight-level grouping to avoid data leakage")
print("  ✅ Normalization parameters computed from the training set only")
print("  ✅ Explicit padding masks and valid lengths")
print("  ✅ Time-series-friendly missing-value handling")
print("  ✅ Class-imbalance detection and weight calculation")
print("  ✅ Complete reproducibility settings")
print("  ✅ NaN and empty-value filtering")
print("  ✅ Data saved to Google Drive")

# Show sample information
print(f"\n📝 Sample information:")
if len(pairs_train) > 0:
    sample_key = list(pairs_train.keys())[0]
    sample = pairs_train[sample_key]
    print(f"  Data shape: {sample['x'].shape}")
    print(f"  Mask shape: {sample['mask'].shape}")
    print(f"  Valid length: {sample['valid_length']}")
    print(f"  Contains padding: {sample['is_padded']}")
    print(f"  Encoded labels: class={sample['class_encoded']}, before_after={sample['before_after_encoded']}")

print(f"\n🎯 Variables ready for model training:")
print("  pairs_train, pairs_val, pairs_test - cleaned and preprocessed sample dictionaries")
print("  label_mappings - label mapping dictionary")
print("  train_class_weights - class weights for the training set")
print("  scaler - fitted scaler (for inference on new data)")
print("  scaling_stats - normalization statistics")

print(f"\n✨ All processed data has been saved and is ready for model training!")


In [ ]:
# ============================================================
# Monotonic-Decreasing HI + aligned plotting (before -> after) + Hyper-Connections (DHC)
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import os
import random
import matplotlib.pyplot as plt
import pickle

# ============================================================
# Hyper-Connections Implementation
# ============================================================
class DynamicHyperConnection(nn.Module):
    """
    Dynamic Hyper-Connection (DHC) module based on 2409.19606v3
    Implements Alg.1 with dynamic prediction of B, A_m, A_r matrices
    """
    def __init__(self, d_model, n=2, use_tanh=True, scale_output=True):
        super().__init__()
        self.d_model = d_model
        self.n = n
        self.use_tanh = use_tanh
        self.scale_output = scale_output

        # Static matrices (equation 14)
        # B_static: depth connection (n x n)
        self.register_buffer('B_static', self._init_depth_matrix(n))

        # A_static: width connection [A_m, A_r] (n x 2n)
        self.register_buffer('A_static', self._init_width_matrix(n))

        # Dynamic prediction networks (equations 10-13)
        # Input normalization
        self.input_norm = nn.LayerNorm(d_model)

        # Dynamic weight predictors
        self.W_beta = nn.Linear(d_model, n * n, bias=False)
        self.W_m = nn.Linear(d_model, n * n, bias=False)
        self.W_r = nn.Linear(d_model, n * n, bias=False)

        # Small scale factors for stability
        self.scale_beta = nn.Parameter(torch.ones(1) * 0.1)
        self.scale_m = nn.Parameter(torch.ones(1) * 0.1)
        self.scale_r = nn.Parameter(torch.ones(1) * 0.1)

        # Initialize dynamic weights to 0 for Pre-Norm equivalent initialization
        nn.init.zeros_(self.W_beta.weight)
        nn.init.zeros_(self.W_m.weight)
        nn.init.zeros_(self.W_r.weight)

    def _init_depth_matrix(self, n):
        """Initialize depth connection matrix B (equation 14)"""
        B = torch.eye(n)  # Identity for stable initialization
        return B

    def _init_width_matrix(self, n):
        """Initialize width connection matrix A = [A_m, A_r] (equation 14)"""
        # A_m: previous layer connection (n x n)
        A_m = torch.eye(n)
        # A_r: residual connection (n x n)
        A_r = torch.eye(n)
        A = torch.cat([A_m, A_r], dim=1)  # (n x 2n)
        return A

    def forward(self, h_input, T_h0, context=None):
        """
        Forward pass implementing Algorithm 1

        Args:
            h_input: Input hidden vector (B, seq_len, d_model) or (B, d_model)
            T_h0: Transformed input from current layer (B, seq_len, d_model) or (B, d_model)
            context: Optional context for dynamic prediction (B, context_dim)

        Returns:
            output: (B, seq_len, d_model) or (B, d_model)
        """
        # Handle both 2D and 3D inputs
        if h_input.dim() == 3:
            B, seq_len, d_model = h_input.shape
            flatten_time = True
            h_input_flat = h_input.view(B * seq_len, d_model)
            T_h0_flat = T_h0.view(B * seq_len, d_model)
        else:
            B, d_model = h_input.shape
            flatten_time = False
            h_input_flat = h_input
            T_h0_flat = T_h0

        # Create hyper-hidden matrix H by repeating h_input n times (equation 1)
        H = h_input_flat.unsqueeze(1).repeat(1, self.n, 1)  # (B*seq_len, n, d_model)

        # Dynamic matrix prediction using input normalization
        if context is not None:
            # Use context for dynamic prediction
            norm_input = self.input_norm(context)
        else:
            # Use input itself for dynamic prediction
            norm_input = self.input_norm(h_input_flat)

        # Predict dynamic matrices (equations 10-13)
        beta_flat = self.W_beta(norm_input)  # (B*seq_len, n*n)
        m_flat = self.W_m(norm_input)       # (B*seq_len, n*n)
        r_flat = self.W_r(norm_input)       # (B*seq_len, n*n)

        if self.use_tanh:
            beta_flat = torch.tanh(beta_flat)
            m_flat = torch.tanh(m_flat)
            r_flat = torch.tanh(r_flat)

        # Apply small scale factors
        beta_flat = beta_flat * self.scale_beta
        m_flat = m_flat * self.scale_m
        r_flat = r_flat * self.scale_r

        # Reshape to matrices
        B_dynamic = beta_flat.view(-1, self.n, self.n)  # (B*seq_len, n, n)
        A_m_dynamic = m_flat.view(-1, self.n, self.n)   # (B*seq_len, n, n)
        A_r_dynamic = r_flat.view(-1, self.n, self.n)   # (B*seq_len, n, n)

        # Combine static and dynamic matrices (equations 10-13)
        B_total = self.B_static.unsqueeze(0) + B_dynamic  # (B*seq_len, n, n)
        A_m_total = self.A_static[:, :self.n].unsqueeze(0) + A_m_dynamic  # (B*seq_len, n, n)
        A_r_total = self.A_static[:, self.n:].unsqueeze(0) + A_r_dynamic  # (B*seq_len, n, n)

        # Apply connections (equations 2, 5-7)
        # Depth connection: H_hat = B^T @ T(h0) + A_r^T @ H
        T_h0_expanded = T_h0_flat.unsqueeze(1).repeat(1, self.n, 1)  # (B*seq_len, n, d_model)

        # Batch matrix multiplication
        depth_term = torch.bmm(B_total.transpose(1, 2), T_h0_expanded)  # (B*seq_len, n, d_model)
        residual_term = torch.bmm(A_r_total.transpose(1, 2), H)       # (B*seq_len, n, d_model)

        H_hat = depth_term + residual_term  # (B*seq_len, n, d_model)

        # Row-wise summation to get final output (Algorithm 1, line 11)
        output_flat = H_hat.sum(dim=1)  # (B*seq_len, d_model)

        # Apply variance scaling (divide by sqrt(n) for stable variance)
        if self.scale_output:
            output_flat = output_flat / np.sqrt(self.n)

        # Reshape back to original format
        if flatten_time:
            output = output_flat.view(B, seq_len, d_model)
        else:
            output = output_flat

        return output

class HyperConnectedBlock(nn.Module):
    """
    Wrapper that applies DHC to any functional block
    """
    def __init__(self, block, d_model, n=2, use_dhc=True):
        super().__init__()
        self.block = block
        self.use_dhc = use_dhc

        if use_dhc:
            self.dhc = DynamicHyperConnection(d_model, n=n)

    def forward(self, x, *args, **kwargs):
        # Apply the functional block
        if hasattr(self.block, '__call__') and not isinstance(self.block, nn.Module):
            # For functional blocks (like fusion functions)
            block_output = self.block(x, *args, **kwargs)
        else:
            # For nn.Module blocks
            block_output = self.block(x, *args, **kwargs)

        if self.use_dhc:
            # Apply DHC: input=x, transformed=block_output
            output = self.dhc(x, block_output)
        else:
            output = block_output

        return output

# ============================================================
# 1) Dataset: read samples from flight data dictionaries
# ============================================================
class FlightDataset(Dataset):
    """
    Each sample contains:
      x: (L,C) feature sequence
      mask: (L,) validity mask for padded sequences
      valid_length: actual sequence length
      label: encoded label for classification
      class_encoded: encoded main class
      before_after_encoded: 0 for before maintenance, 1 for after maintenance
    """
    def __init__(self, samples_dict, label_key='class_encoded'):
        self.samples = list(samples_dict.values())
        self.label_key = label_key

        # Extract feature dimension from first sample
        if len(self.samples) > 0:
            self.C = self.samples[0]['x'].shape[1]
        else:
            raise ValueError("Empty dataset")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        return {
            "x": torch.from_numpy(sample['x']).float(),  # (L,C)
            "mask": torch.from_numpy(sample['mask']).bool(),  # (L,)
            "valid_length": torch.tensor(sample['valid_length'], dtype=torch.long),
            "label": torch.tensor(sample[self.label_key], dtype=torch.long),
            "before_after": torch.tensor(sample['before_after_encoded'], dtype=torch.long),
            "master_index": sample['master_index'],
            "is_padded": sample['is_padded']
        }

def flight_collate_fn(batch):
    """
    Collate function for flight data - sequences are already padded to same length
    """
    batch_size = len(batch)
    if batch_size == 0:
        return {}

    # All sequences should have the same length after windowing
    x = torch.stack([b["x"] for b in batch], 0)  # (B,L,C)
    mask = torch.stack([b["mask"] for b in batch], 0)  # (B,L)
    valid_lengths = torch.stack([b["valid_length"] for b in batch], 0)  # (B,)
    labels = torch.stack([b["label"] for b in batch], 0)  # (B,)
    before_after = torch.stack([b["before_after"] for b in batch], 0)  # (B,)
    master_indices = [b["master_index"] for b in batch]

    return {
        "x": x,
        "mask": mask,
        "valid_lengths": valid_lengths,
        "labels": labels,
        "before_after": before_after,
        "master_indices": master_indices
    }

# ============================================================
# 2) Model: encode "damage" → project to monotonic decreasing HI → recursive reconstruction → classify
# ============================================================
class BoltzmannKAN(nn.Module):
    def __init__(self, in_ch, out_ch=4):
        super().__init__()
        self.E  = nn.Parameter(torch.zeros(out_ch, in_ch))
        self.kT = nn.Parameter(torch.ones(out_ch, in_ch))
    def forward(self, x):
        B,T,C = x.shape
        kT = torch.clamp(F.softplus(self.kT), 0.01, 10.0).unsqueeze(0).unsqueeze(2)  # (1,out_ch,1,in_ch)
        E  = torch.clamp(self.E, -10.0, 10.0).unsqueeze(0).unsqueeze(2)             # (1,out_ch,1,in_ch)
        x_ = torch.clamp(x.unsqueeze(1), -10.0, 10.0)                               # (B, out_ch, T, in_ch)
        exp = torch.clamp((E - x_) / kT, -50, 50)
        w   = torch.sigmoid(exp)
        h   = (w * x_).sum(dim=3).permute(0, 2, 1)           # (B,T,out_ch) >=0
        return torch.clamp(F.softplus(h), 0.0, 100.0)

class BaseOp(nn.Module):
    """
    Base operator class with support for dynamic parameter modulation
    """
    def __init__(self):
        super().__init__()
        self.param_modulator = None

    def set_param_modulator(self, modulator):
        """Set the parameter modulator"""
        self.param_modulator = modulator

    def get_modulated_params(self, context):
        """Get parameters after context-based modulation"""
        if self.param_modulator is None:
            return {}
        return self.param_modulator(context)

class MonotonicLinearOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0

    def forward(self, h, context=None):
        # Retrieve the modulated parameters
        modulated_params = self.get_modulated_params(context) if context is not None else {}

        # Base parameters
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_bias = self.bias

        # Parameter modulation
        scale_offset = modulated_params.get('scale_offset', 0.0)
        bias_offset = modulated_params.get('bias_offset', 0.0)

        # Modulated parameters
        scale = torch.clamp(base_scale + scale_offset, self.smin, self.smax)
        bias = torch.clamp(base_bias + bias_offset, -5.0, 5.0)

        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * (xm + bias)), 0.0, 100.0)

class MonotonicFlatOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 1e-3, 1.0

    def _cum(self, x):
        # Fix: ensure computations are carried out along the time dimension
        B, T = x.shape  # xshould have shape (B, T)
        diff = F.relu(x[:, 1:] - x[:, :-1])  # (B, T-1)
        zero_init = torch.zeros(B, 1, device=x.device, dtype=x.dtype)  # (B, 1)
        return torch.cat([zero_init, torch.cumsum(diff, dim=1)], dim=1)  # (B, T)

    def forward(self, h, context=None):
        modulated_params = self.get_modulated_params(context) if context is not None else {}

        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_bias = self.bias

        scale_offset = modulated_params.get('scale_offset', 0.0)
        bias_offset = modulated_params.get('bias_offset', 0.0)

        scale = torch.clamp(base_scale + scale_offset, self.smin, self.smax)
        bias = torch.clamp(base_bias + bias_offset, -2.0, 2.0)

        xm = torch.clamp(h.mean(-1), -10.0, 10.0)  # (B, T)
        cum_result = self._cum(xm)  # (B, T)
        return torch.clamp(F.softplus(scale * (cum_result + bias)), 0.0, 100.0)

class ConcaveLogOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.eps = 1e-3
        self.smin, self.smax = 0.01, 5.0

    def forward(self, h, context=None):
        modulated_params = self.get_modulated_params(context) if context is not None else {}

        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        scale_offset = modulated_params.get('scale_offset', 0.0)
        eps_offset = modulated_params.get('eps_offset', 0.0)

        scale = torch.clamp(base_scale + scale_offset, self.smin, self.smax)
        eps = torch.clamp(self.eps + eps_offset, 1e-6, 1e-1)

        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * torch.log(torch.abs(xm) + eps)), 0.0, 100.0)

class SaturationSigmoidOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.raw_slope = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
        self.lmin, self.lmax = 0.1, 5.0

    def forward(self, h, context=None):
        modulated_params = self.get_modulated_params(context) if context is not None else {}

        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_slope = torch.clamp(F.softplus(self.raw_slope), self.lmin, self.lmax)
        base_bias = self.bias

        scale_offset = modulated_params.get('scale_offset', 0.0)
        slope_offset = modulated_params.get('slope_offset', 0.0)
        bias_offset = modulated_params.get('bias_offset', 0.0)

        scale = torch.clamp(base_scale + scale_offset, self.smin, self.smax)
        slope = torch.clamp(base_slope + slope_offset, self.lmin, self.lmax)
        bias = torch.clamp(base_bias + bias_offset, -3.0, 3.0)

        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * torch.sigmoid(slope * (xm - bias))), 0.0, 100.0)

class HingeReLUOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.threshold = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0

    def forward(self, h, context=None):
        modulated_params = self.get_modulated_params(context) if context is not None else {}

        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_threshold = self.threshold

        scale_offset = modulated_params.get('scale_offset', 0.0)
        threshold_offset = modulated_params.get('threshold_offset', 0.0)

        scale = torch.clamp(base_scale + scale_offset, self.smin, self.smax)
        threshold = torch.clamp(base_threshold + threshold_offset, -3.0, 3.0)

        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * F.relu(xm - threshold)), 0.0, 100.0)

class PolynomialOp(BaseOp):
    def __init__(self, deg=3):
        super().__init__()
        self.raw_coeff = nn.Parameter(torch.zeros(deg))
        self.deg = deg

    def forward(self, h, context=None):
        modulated_params = self.get_modulated_params(context) if context is not None else {}

        base_coeff = torch.clamp(F.softplus(self.raw_coeff), 0.01, 5.0)

        # Fix: Use broadcasting compatible approach for batch dimension
        xm = torch.clamp(h.mean(-1), -5.0, 5.0)  # (B, T)
        B, T = xm.shape

        # Handle coefficient offsets properly for broadcasting
        coeff_offset = modulated_params.get('coeff_offset', torch.zeros_like(base_coeff))
        if coeff_offset.dim() > 1:
            # If coeff_offset is (B, deg), take mean over batch for simplicity
            coeff_offset = coeff_offset.mean(0)

        coeff = torch.clamp(base_coeff + coeff_offset, 0.01, 5.0)  # (deg,)

        y = torch.zeros_like(xm)  # (B, T)
        for i in range(self.deg):
            # Broadcast coefficient to match batch dimension
            power_term = torch.clamp(xm ** (i + 1), -100.0, 100.0)  # (B, T)
            y = y + coeff[i] * power_term  # (B, T) + scalar * (B, T)
        return torch.clamp(F.softplus(y), 0.0, 100.0)

class DampedSinOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.raw_freq = nn.Parameter(torch.tensor(0.0))
        self.raw_lambda = nn.Parameter(torch.tensor(0.0))
        self.phase = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
        self.fmin, self.fmax = 0.1, 5.0
        self.lmin, self.lmax = 0.01, 3.0

    def forward(self, h, context=None):
        modulated_params = self.get_modulated_params(context) if context is not None else {}

        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_freq = torch.clamp(F.softplus(self.raw_freq), self.fmin, self.fmax)
        base_lambda = torch.clamp(F.softplus(self.raw_lambda), self.lmin, self.lmax)
        base_phase = torch.clamp(self.phase, -10.0, 10.0)

        scale_offset = modulated_params.get('scale_offset', 0.0)
        freq_offset = modulated_params.get('freq_offset', 0.0)
        lambda_offset = modulated_params.get('lambda_offset', 0.0)
        phase_offset = modulated_params.get('phase_offset', 0.0)

        scale = torch.clamp(base_scale + scale_offset, self.smin, self.smax)
        freq = torch.clamp(base_freq + freq_offset, self.fmin, self.fmax)
        lam = torch.clamp(base_lambda + lambda_offset, self.lmin, self.lmax)
        phase = torch.clamp(base_phase + phase_offset, -10.0, 10.0)

        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        damp = 1 / (1 + lam * torch.abs(xm))
        return torch.clamp(F.softplus(scale * damp * (torch.sin(freq * xm + phase) + 1)), 0.0, 100.0)

class PiecewiseLinearOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_k1 = nn.Parameter(torch.tensor(0.0))
        self.raw_k2 = nn.Parameter(torch.tensor(0.0))
        self.threshold = nn.Parameter(torch.tensor(0.0))
        self.kmin, self.kmax = 0.01, 5.0

    def forward(self, h, context=None):
        modulated_params = self.get_modulated_params(context) if context is not None else {}

        base_k1 = torch.clamp(F.softplus(self.raw_k1), self.kmin, self.kmax)
        base_k2 = torch.clamp(F.softplus(self.raw_k2), self.kmin, self.kmax)
        base_threshold = torch.clamp(self.threshold, -5.0, 5.0)

        k1_offset = modulated_params.get('k1_offset', 0.0)
        k2_offset = modulated_params.get('k2_offset', 0.0)
        threshold_offset = modulated_params.get('threshold_offset', 0.0)

        k1 = torch.clamp(base_k1 + k1_offset, self.kmin, self.kmax)
        k2 = torch.clamp(base_k2 + k2_offset, self.kmin, self.kmax)
        threshold = torch.clamp(base_threshold + threshold_offset, -5.0, 5.0)

        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        left = k1 * xm
        right = k1 * threshold + k2 * (xm - threshold)
        out = torch.where(xm <= threshold, left, right)
        return torch.clamp(F.softplus(out), 0.0, 100.0)

class ParameterModulator(nn.Module):
    """
    Parameter modulator:dynamically adjust operator parameters based on input context
    """
    def __init__(self, context_dim, param_configs):
        super().__init__()
        self.param_configs = param_configs

        # Create a prediction network for each parameter
        self.param_predictors = nn.ModuleDict()
        for param_name, param_info in param_configs.items():
            param_dim = param_info.get('dim', 1)
            self.param_predictors[param_name] = nn.Sequential(
                nn.Linear(context_dim, 64),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Linear(32, param_dim),
                nn.Tanh()  # Output range [-1, 1]
            )

    def forward(self, context):
        """
        Args:
            context: (B, T, context_dim) context features
        Returns:
            dict: modulated parameter offsets
        """
        # Average across the time dimension to obtain a global context
        global_context = context.mean(dim=1)  # (B, context_dim)

        modulated_params = {}
        for param_name, predictor in self.param_predictors.items():
            param_info = self.param_configs[param_name]
            raw_offset = predictor(global_context)  # (B, param_dim)

            # Scale offsets according to the configuration
            scale = param_info.get('scale', 0.1)
            modulated_params[param_name] = raw_offset * scale

        return modulated_params

class LiquidWeightGenerator(nn.Module):
    """
    Improved Liquid Weight Generator with stronger operator diversity and better temporal dynamics
    """
    def __init__(self, h_dim, x_dim, n_ops, hidden_dim=64, tau_min=1.0, tau_max=3.0):
        super().__init__()
        self.n_ops = n_ops
        self.tau_min, self.tau_max = tau_min, tau_max

        # Enhanced feature encoder with more sophisticated temporal dynamics
        self.h_feature_net = nn.Sequential(
            nn.Linear(h_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.1)
        )

        self.x_feature_net = nn.Sequential(
            nn.Linear(x_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.1)
        )

        # Add temporal encoding
        self.temporal_encoder = nn.Sequential(
            nn.Linear(3, hidden_dim // 4),  # t, dt, phase
            nn.ReLU()
        )

        # Combined feature processing
        combined_dim = hidden_dim + hidden_dim // 4
        self.feature_fusion = nn.Sequential(
            nn.Linear(combined_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim)
        )

        # Weight prediction with explicit per-operator branches
        self.op_branches = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim // 4),
                nn.ReLU(),
                nn.Linear(hidden_dim // 4, 1)
            ) for _ in range(n_ops)
        ])

        # Temperature predictor
        self.temp_predictor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 4),
            nn.ReLU(),
            nn.Linear(hidden_dim // 4, 1)
        )

        # Remove global bias or make it minimal
        self.global_bias_scale = 0.01  # Very small influence

        # Initialize to encourage diversity
        for branch in self.op_branches:
            nn.init.xavier_uniform_(branch[0].weight)
            nn.init.xavier_uniform_(branch[2].weight)

    def forward(self, h_multi, x):
        """
        Args:
            h_multi: (B, T, h_dim) - Boltzmann KAN output
            x: (B, T, x_dim) - Raw sensor data
        Returns:
            weights: (B, T, K) - Per-timestep operator weights
            temperature: (B, T) - Temperature parameters
        """
        B, T, h_dim = h_multi.shape
        _, _, x_dim = x.shape

        # Process h and x separately to maintain diversity
        h_features = self.h_feature_net(h_multi)  # (B, T, hidden_dim//2)
        x_features = self.x_feature_net(x)        # (B, T, hidden_dim//2)

        # Add temporal encoding
        t_normalized = torch.arange(T, device=x.device, dtype=x.dtype).unsqueeze(0).expand(B, -1) / max(T-1, 1)
        dt = torch.zeros_like(t_normalized)
        dt[:, 1:] = t_normalized[:, 1:] - t_normalized[:, :-1]
        phase = torch.sin(2 * 3.14159 * t_normalized)  # Cyclical patterns

        temporal_input = torch.stack([t_normalized, dt, phase], dim=-1)  # (B, T, 3)
        temporal_features = self.temporal_encoder(temporal_input)  # (B, T, hidden_dim//4)

        # Fuse features
        combined = torch.cat([h_features, x_features, temporal_features], dim=-1)
        fused_features = self.feature_fusion(combined)  # (B, T, hidden_dim)

        # Generate per-operator logits using separate branches
        raw_logits = []
        for i, branch in enumerate(self.op_branches):
            logit = branch(fused_features).squeeze(-1)  # (B, T)
            raw_logits.append(logit)
        raw_weights = torch.stack(raw_logits, dim=-1)  # (B, T, K)

        # Zero-mean normalization to prevent systematic bias
        raw_weights = raw_weights - raw_weights.mean(dim=-1, keepdim=True)

        # Minimal global bias (optional)
        if hasattr(self, 'global_bias_scale') and self.global_bias_scale > 0:
            # Very small random bias to break symmetry
            device = raw_weights.device
            global_bias = torch.randn(self.n_ops, device=device) * self.global_bias_scale
            raw_weights = raw_weights + global_bias.unsqueeze(0).unsqueeze(0)

        # Predict temperature
        raw_temp = self.temp_predictor(fused_features).squeeze(-1)  # (B, T)
        temperature = torch.clamp(F.softplus(raw_temp) + self.tau_min, self.tau_min, self.tau_max)

        # Apply temperature-scaled softmax
        weights = F.softmax(raw_weights / temperature.unsqueeze(-1), dim=-1)  # (B, T, K)

        return weights, temperature

class CustomKAN(nn.Module):
    """
    Enhanced CustomKAN with improved operator diversity and multi-scale feature extraction
    Operators with support for dynamic parameter modulation
    """
    def __init__(self, ops, h_dim, x_dim):
        super().__init__()
        self.ops = nn.ModuleList(ops)
        self.n_ops = len(ops)

        # Enhanced weight generator
        self.weight_generator = LiquidWeightGenerator(
            h_dim=h_dim,
            x_dim=x_dim,
            n_ops=self.n_ops,
            hidden_dim=128
        )

        # Per-operator feature extraction to increase diversity
        self.op_feature_extractors = nn.ModuleList([
            nn.Sequential(
                nn.Linear(h_dim, h_dim),
                nn.ReLU(),
                nn.Linear(h_dim, h_dim)
            ) for _ in range(self.n_ops)
        ])

        # Create a parameter modulator for each operator
        self.param_modulators = nn.ModuleList()
        context_dim = h_dim + x_dim  # Context dimension

        for i, op in enumerate(self.ops):
            # Define parameter settings according to operator type
            param_configs = self._get_param_configs_for_op(op)
            if param_configs:
                modulator = ParameterModulator(context_dim, param_configs)
                self.param_modulators.append(modulator)
                # Attach the parameter modulator to the operator
                op.set_param_modulator(modulator)
            else:
                self.param_modulators.append(None)

        # Final transformation parameters
        self.raw_gain = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))

    def _get_param_configs_for_op(self, op):
        """Return the parameter configuration for a given operator type"""
        if isinstance(op, MonotonicLinearOp):
            return {
                'scale_offset': {'dim': 1, 'scale': 0.2},
                'bias_offset': {'dim': 1, 'scale': 0.3}
            }
        elif isinstance(op, MonotonicFlatOp):
            return {
                'scale_offset': {'dim': 1, 'scale': 0.1},
                'bias_offset': {'dim': 1, 'scale': 0.2}
            }
        elif isinstance(op, ConcaveLogOp):
            return {
                'scale_offset': {'dim': 1, 'scale': 0.2},
                'eps_offset': {'dim': 1, 'scale': 0.01}
            }
        elif isinstance(op, SaturationSigmoidOp):
            return {
                'scale_offset': {'dim': 1, 'scale': 0.2},
                'slope_offset': {'dim': 1, 'scale': 0.3},
                'bias_offset': {'dim': 1, 'scale': 0.3}
            }
        elif isinstance(op, HingeReLUOp):
            return {
                'scale_offset': {'dim': 1, 'scale': 0.2},
                'threshold_offset': {'dim': 1, 'scale': 0.3}
            }
        elif isinstance(op, PolynomialOp):
            return {
                'coeff_offset': {'dim': op.deg, 'scale': 0.2}
            }
        elif isinstance(op, DampedSinOp):
            return {
                'scale_offset': {'dim': 1, 'scale': 0.2},
                'freq_offset': {'dim': 1, 'scale': 0.3},
                'lambda_offset': {'dim': 1, 'scale': 0.2},
                'phase_offset': {'dim': 1, 'scale': 0.5}
            }
        elif isinstance(op, PiecewiseLinearOp):
            return {
                'k1_offset': {'dim': 1, 'scale': 0.2},
                'k2_offset': {'dim': 1, 'scale': 0.2},
                'threshold_offset': {'dim': 1, 'scale': 0.3}
            }
        return {}

    def forward(self, h_multi, x):  # h_multi:(B,T,h_dim), x:(B,T,x_dim)
        B, T, _ = h_multi.shape

        # Apply per-operator feature extraction for diversity
        op_features = []
        for i, extractor in enumerate(self.op_feature_extractors):
            feat = extractor(h_multi)  # (B, T, h_dim)
            op_features.append(feat)

        # Build context features for parameter modulation
        context = torch.cat([h_multi, x], dim=-1)  # (B, T, h_dim + x_dim)

        # Compute operator outputs with enhanced diversity and dynamic parameters
        outs = []
        for i, op in enumerate(self.ops):
            op_out = op(op_features[i], context)  # Pass context features for parameter modulation
            outs.append(op_out)

        # Align lengths
        Tm = min(o.size(1) for o in outs)
        outs = [o[:, :Tm] for o in outs]
        h_multi_aligned = h_multi[:, :Tm, :]
        x_aligned = x[:, :Tm, :]

        # Stack operator outputs
        op_stack = torch.stack(outs, dim=-1)  # (B, Tm, K)

        # Generate liquid weights
        weights, temperature = self.weight_generator(h_multi_aligned, x_aligned)  # (B, Tm, K)

        # Weighted combination (continuous weighting)
        damage = torch.sum(op_stack * weights, dim=-1)  # (B, Tm)
        damage = torch.clamp(damage, 0.0, 100.0)

        # Final transformation
        gain = torch.clamp(F.softplus(self.raw_gain), 0.1, 2.0)  # Reduced gain range
        bias_val = torch.clamp(self.bias, -2.0, 2.0)  # Reduced bias range
        damage = torch.clamp(gain * damage + bias_val, 0.0, 100.0)

        return damage, weights, temperature

class HealthFusionBlock(nn.Module):
    """
    Health fusion block that combines three health assessment paths using sequence-parallel duality
    """
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model

        # Projection layers for each path
        self.direct_proj = nn.Linear(d_model, d_model)
        self.damage_proj = nn.Linear(d_model, d_model)
        self.linear_proj = nn.Linear(d_model, d_model)

        # Fusion weights (learnable)
        self.fusion_weights = nn.Parameter(torch.ones(3) / 3.0)

    def forward(self, health_direct, damage_normalized, linear_decay):
        """
        Args:
            health_direct: (B, T) direct health assessment
            damage_normalized: (B, T) damage pattern
            linear_decay: (B, T) linear constraint
        Returns:
            fused_health: (B, T) fused health index
        """
        B, T = health_direct.shape

        # Project each path to d_model space
        direct_feat = self.direct_proj(health_direct.unsqueeze(-1).expand(-1, -1, self.d_model))
        damage_feat = self.damage_proj(damage_normalized.unsqueeze(-1).expand(-1, -1, self.d_model))
        linear_feat = self.linear_proj(linear_decay.unsqueeze(-1).expand(-1, -1, self.d_model))

        # Stack paths as parallel routes (B, T, 3, d_model)
        path_stack = torch.stack([direct_feat, damage_feat, linear_feat], dim=2)

        # Apply fusion weights with softmax normalization
        weights = F.softmax(self.fusion_weights, dim=0)  # (3,)
        weighted_stack = path_stack * weights.view(1, 1, 3, 1)  # Broadcast weights

        # Sum over paths and project back to (B, T)
        fused_feat = weighted_stack.sum(dim=2)  # (B, T, d_model)
        fused_health = fused_feat.mean(dim=-1)  # (B, T)

        return fused_health

class TrendEncoder(nn.Module):
    """
    Encoder outputs "damage", then projects to monotonic decreasing HI:
        HI is learned from sensor data without hard range constraints
    Without real HI, learns mapping from sensor data to infer health states
    Enhanced with improved Liquid Weight Generator, dynamic parameter modulation, and Hyper-Connections
    """
    def __init__(self, in_ch, trend_ch=4):
        super().__init__()
        self.in_ch = in_ch
        self.trend_ch = trend_ch

        # Core components
        self.boltz = BoltzmannKAN(in_ch, out_ch=trend_ch)
        ops = [MonotonicLinearOp(), MonotonicFlatOp(), ConcaveLogOp(),
               SaturationSigmoidOp(), HingeReLUOp(), PolynomialOp(),
               DampedSinOp(), PiecewiseLinearOp()]

        # Enhanced CustomKAN with dynamic parameter modulation
        self.customkan = CustomKAN(ops, h_dim=trend_ch, x_dim=in_ch)

        # Wrap KAN representation block with DHC
        self.kan_dhc = DynamicHyperConnection(d_model=trend_ch, n=2, use_tanh=True)

        # Projection parameters - learn flexible health states from sensor features
        self.proj_gain = nn.Parameter(torch.tensor(1.0))
        self.proj_bias = nn.Parameter(torch.tensor(0.0))

        # Add health-aware layer, directly infer from multi-dimensional sensor features
        self.health_aware_transform = nn.Sequential(
            nn.Linear(in_ch, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()  # Output 0-1 range health state
        )

        # Add linear constraint layer, enforce linear trend
        self.linear_enforcer = nn.Sequential(
            nn.Linear(in_ch, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()  # Output linear decay degree
        )

        # Health fusion block with DHC
        self.health_fusion = HealthFusionBlock(d_model=1)
        self.fusion_dhc = DynamicHyperConnection(d_model=1, n=2, use_tanh=True)

    def forward(self, x):  # x:(B,T,C)
        # Feature extraction based on Boltzmann KAN
        h_multi = self.boltz(x)              # (B,T,trend_ch) >=0

        # Enhanced liquid weight generation with dynamic parameter modulation
        damage, weights, temperature = self.customkan(h_multi, x)    # damage:(B,T), weights:(B,T,K), temp:(B,T)

        # Apply DHC to KAN representation
        # Use original h_multi as input, damage-enhanced features as transformed
        damage_enhanced = damage.unsqueeze(-1).expand(-1, -1, self.trend_ch)  # (B, T, trend_ch)
        h_multi_dhc = self.kan_dhc(h_multi, damage_enhanced)  # (B, T, trend_ch)

        # Directly learn health states from raw sensor data
        # Assess health state for sensor readings at each time step
        B, T, C = x.shape
        x_reshaped = x.view(-1, C)  # (B*T, C)
        health_direct = self.health_aware_transform(x_reshaped)  # (B*T, 1)
        health_direct = health_direct.view(B, T)  # (B, T)

        # Add linear constraint, force linear decay pattern
        linear_weight = self.linear_enforcer(x_reshaped).view(B, T)  # (B, T)

        # Generate ideal linear decay pattern
        t = torch.arange(T, device=x.device, dtype=x.dtype).unsqueeze(0).expand(B, -1)  # (B, T)
        t_normalized = t / (T - 1)  # Normalize to [0, 1]

        # Combine damage and direct health assessment
        g = torch.clamp(F.softplus(self.proj_gain), 0.1, 5.0)
        b = torch.clamp(self.proj_bias, -5.0, 5.0)

        # Convert damage to health state decay
        damage_normalized = torch.sigmoid(g*(damage + b))

        # Combine three health assessment methods: direct assessment, damage pattern, linear constraint
        combined_health = health_direct * (1 - 0.3 * damage_normalized)

        # Generate ideal linear decay curve (from high to low) - remove hard range constraints
        linear_decay = 1.0 - t_normalized * 0.5  # Simple linear decay from 1.0 to 0.5

        # Use DHC-enhanced health fusion instead of linear interpolation
        fused_health = self.health_fusion(health_direct, damage_normalized, linear_decay)  # (B, T)

        # Apply DHC to fusion output
        # Stack the three paths as input, fused result as transformed
        three_paths = torch.stack([health_direct, damage_normalized, linear_decay], dim=-1)  # (B, T, 3)
        three_paths_avg = three_paths.mean(dim=-1, keepdim=True)  # (B, T, 1)
        fused_health_expanded = fused_health.unsqueeze(-1)  # (B, T, 1)

        hi = self.fusion_dhc(three_paths_avg, fused_health_expanded).squeeze(-1)  # (B, T)

        # Force stronger monotonic decreasing constraint without hard range limits
        for t_idx in range(1, T):
            hi = torch.cat([
                hi[:, :t_idx],
                torch.min(hi[:, t_idx:t_idx+1], hi[:, t_idx-1:t_idx] - 0.001),  # Force at least 0.001 decrease
                hi[:, t_idx+1:]
            ], dim=1)

        return hi, h_multi_dhc, weights, temperature

class ReconHead(nn.Module):
    """
    Recursive reconstruction: given (x_t, h_t) → predict x_{t+1}
    """
    def __init__(self, C, hidden=128):
        super().__init__()
        self.gru = nn.GRU(input_size=C+1, hidden_size=hidden, batch_first=True)
        self.out = nn.Linear(hidden, C)
    def forward(self, x_in, h_in):
        # x_in:(B,T_in,C), h_in:(B,T_in)
        B,T,C = x_in.shape
        h_in_clamped = torch.clamp(h_in, 0.0, 10.0)  # Remove upper limit constraint
        feat = torch.cat([x_in, h_in_clamped.unsqueeze(-1)], dim=-1)  # (B,T,C+1)
        H,_ = self.gru(feat)                                  # (B,T,H)
        y = self.out(H)                                       # (B,T,C) aligned predict x_{t+1}
        return y

class FlightClassifier(nn.Module):
    """
    Enhanced Health Index difference-based fault classification with attention mechanism and DHC
    """
    def __init__(self, n_classes):
        super().__init__()

        # Attention mechanism for HI sequence
        self.attention = nn.MultiheadAttention(embed_dim=1, num_heads=1, batch_first=True)

        # Enhanced feature extraction
        self.feature_extractor = nn.Sequential(
            nn.Linear(6, 128),  # Extended feature vector
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        # DHC for classifier block
        self.classifier_dhc = DynamicHyperConnection(d_model=32, n=2, use_tanh=True)

        # Final classifier
        self.classifier = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(16, n_classes)
        )

    def forward(self, hi, before_after, mask):
        """
        Args:
            hi: (B, T) health index sequence
            before_after: (B,) 0=before maintenance, 1=after maintenance
            mask: (B, T) validity mask
        Returns:
            logits: (B, n_classes) classification logits
        """
        B, T = hi.shape

        # Calculate statistics from HI with attention
        valid_counts = mask.sum(dim=1, keepdim=True).clamp_min(1.0)  # (B, 1)

        # Apply attention to HI sequence
        hi_expanded = hi.unsqueeze(-1)  # (B, T, 1)
        attn_mask = ~mask  # Fix: Remove the .bool() call since mask is already boolean
        attended_hi, _ = self.attention(hi_expanded, hi_expanded, hi_expanded,
                                      key_padding_mask=attn_mask)  # (B, T, 1)
        attended_hi = attended_hi.squeeze(-1)  # (B, T)

        # Enhanced statistics calculation
        hi_mean = (hi * mask.float()).sum(dim=1, keepdim=True) / valid_counts  # (B, 1)
        hi_std = torch.sqrt(((hi - hi_mean) ** 2 * mask.float()).sum(dim=1, keepdim=True) / valid_counts)  # (B, 1)

        # Attention-weighted statistics
        attn_weights = F.softmax(attended_hi * mask.float() + (1 - mask.float()) * (-1e9), dim=1)  # (B, T)
        hi_attn_mean = (hi * attn_weights).sum(dim=1, keepdim=True)  # (B, 1)

        # Trend analysis - early vs late period difference
        half_t = T // 2
        early_mask = mask[:, :half_t]
        late_mask = mask[:, half_t:]
        early_hi = hi[:, :half_t]
        late_hi = hi[:, half_t:]

        early_mean = (early_hi * early_mask.float()).sum(dim=1, keepdim=True) / (early_mask.float().sum(dim=1, keepdim=True) + 1e-8)
        late_mean = (late_hi * late_mask.float()).sum(dim=1, keepdim=True) / (late_mask.float().sum(dim=1, keepdim=True) + 1e-8)
        hi_trend = early_mean - late_mean  # Positive if declining health

        # Feature vector: [before_after, mean_hi, std_hi, attn_mean_hi, trend, attn_weight_entropy]
        attn_entropy = -(attn_weights * torch.log(attn_weights + 1e-8)).sum(dim=1, keepdim=True)

        features = torch.cat([
            before_after.float().unsqueeze(1),  # (B, 1)
            hi_mean,  # (B, 1)
            hi_std,   # (B, 1)
            hi_attn_mean,  # (B, 1)
            hi_trend,  # (B, 1)
            attn_entropy  # (B, 1)
        ], dim=1)  # (B, 6)

        # Extract features
        feature_repr = self.feature_extractor(features)  # (B, 32)

        # Apply DHC to classifier block
        # Use features as input, feature_repr as transformed
        features_proj = torch.matmul(features, torch.randn(6, 32, device=features.device))  # Project to same dim
        feature_repr_dhc = self.classifier_dhc(features_proj, feature_repr)  # (B, 32)

        # Final classification
        logits = self.classifier(feature_repr_dhc)  # (B, n_classes)

        return logits

class FlightDiffAwareReconstructor(nn.Module):
    """
    Flight data model with monotonic HI learning, classification, and Hyper-Connections
    """
    def __init__(self, in_ch, trend_ch=4, hidden=128, n_classes=19):
        super().__init__()
        self.encoder = TrendEncoder(in_ch, trend_ch)
        self.recon = ReconHead(in_ch, hidden)
        self.classifier = FlightClassifier(n_classes)

    def forward(self, x, mask):
        """
        Args:
            x: (B, L, C) input sequences
            mask: (B, L) validity mask
        Returns:
            hi: (B, L) health index
            y_hat: (B, L-2, C) reconstruction prediction
            h_multi: (B, L, trend_ch) intermediate features
            weights: (B, L, K) operator weights
            temperature: (B, L) temperature parameters
        """
        B, L, C = x.shape

        # Generate health index and features
        hi, h_multi, weights, temperature = self.encoder(x)  # hi:(B,L)

        # Reconstruction on 0:L-2 → 1:L-1
        if L >= 3:
            x_in = x[:, :-2, :]  # (B, L-2, C)
            h_in = hi[:, :-2]    # (B, L-2)
            y_hat = self.recon(x_in, h_in)  # (B, L-2, C)
        else:
            y_hat = torch.zeros(B, 0, C, device=x.device)

        return hi, y_hat, h_multi, weights, temperature

# ============================================================
# 3) Loss functions adapted for flight data with DHC regularization
# ============================================================
def sanitize_tensors(*tensors):
    """Numerical stabilization"""
    sanitized = []
    for tensor in tensors:
        if tensor is None:
            sanitized.append(None)
            continue
        if isinstance(tensor, torch.Tensor):
            tensor = torch.nan_to_num(tensor, nan=0.0, posinf=1e6, neginf=-1e6)
            tensor = torch.clamp(tensor, -1e6, 1e6)
        sanitized.append(tensor)
    return sanitized if len(sanitized) > 1 else sanitized[0]

def masked_mse(pred, target, mask):
    """MSE loss with mask"""
    if pred.numel() == 0 or target.numel() == 0:
        return torch.tensor(0.0, device=pred.device)
    diff = (pred - target).pow(2)
    if mask.numel() > 0:
        masked_diff = diff * mask.unsqueeze(-1)
        loss = masked_diff.sum() / (mask.sum() * diff.size(-1) + 1e-8)
    else:
        loss = diff.mean()
    return torch.clamp(loss, 0.0, 1e6)

def slope_loss(h, mask, mode="smooth"):
    """Slope-based regularization"""
    if h.size(1) < 2:
        return torch.tensor(0.0, device=h.device)

    diff = h[:, 1:] - h[:, :-1]  # (B, T-1)
    mask_diff = mask[:, 1:].float() * mask[:, :-1].float()  # (B, T-1)

    if mode == "smooth":
        loss = (diff.abs() * mask_diff).sum() / (mask_diff.sum() + 1e-8)
    elif mode == "mono_dec":
        violation = F.relu(diff) * mask_diff  # Penalize increases
        loss = violation.sum() / (mask_diff.sum() + 1e-8)
    else:
        loss = torch.tensor(0.0, device=h.device)

    return torch.clamp(loss, 0.0, 1e6)

def enhanced_linear_slope_loss(h, mask, weight=10.0):
    """Enhanced linear slope constraint"""
    if h.size(1) < 3:
        return torch.tensor(0.0, device=h.device)

    B, T = h.shape

    # Create ideal linear decay
    t = torch.arange(T, device=h.device, dtype=h.dtype).unsqueeze(0).expand(B, -1)
    t_norm = t / (T - 1)

    # Fit linear trend to actual HI
    linear_loss = torch.tensor(0.0, device=h.device)

    for b in range(B):
        valid_indices = mask[b].nonzero(as_tuple=True)[0]
        if len(valid_indices) < 2:
            continue

        h_valid = h[b, valid_indices]
        t_valid = t_norm[b, valid_indices]

        if len(h_valid) < 2:
            continue

        # Linear regression
        x_mean = t_valid.mean()
        y_mean = h_valid.mean()

        numerator = ((t_valid - x_mean) * (h_valid - y_mean)).sum()
        denominator = ((t_valid - x_mean) ** 2).sum()

        if denominator < 1e-8:
            continue

        slope = numerator / denominator
        intercept = y_mean - slope * x_mean

        # Predicted linear values
        h_linear = slope * t_valid + intercept

        # Loss: deviation from linear trend
        deviation = (h_valid - h_linear).pow(2).mean()
        linear_loss = linear_loss + deviation

    return linear_loss / B * weight

def enhanced_liquid_weight_regularization(weights_list, mask_list, lambda_tv=0.02, lambda_ent=0.1, lambda_bal=0.05, lambda_div=0.1):
    """Enhanced liquid weight regularization"""
    total_loss = torch.tensor(0.0)
    count = 0

    for weights, mask in zip(weights_list, mask_list):
        if weights is None or mask is None:
            continue

        B, T, K = weights.shape

        # 1. Temporal variation penalty (smoothness)
        if T > 1:
            tv_loss = torch.sum(torch.abs(weights[:, 1:, :] - weights[:, :-1, :]) * mask[:, 1:].float().unsqueeze(-1))
            tv_loss = tv_loss / (mask[:, 1:].float().sum() * K + 1e-8)
            total_loss = total_loss + lambda_tv * tv_loss

        # 2. Entropy regularization (diversity)
        eps = 1e-8
        entropy = -torch.sum(weights * torch.log(weights + eps), dim=-1)  # (B, T)
        max_entropy = torch.log(torch.tensor(K, dtype=weights.dtype, device=weights.device))
        entropy_loss = torch.sum((max_entropy - entropy) * mask.float()) / (mask.float().sum() + 1e-8)
        total_loss = total_loss + lambda_ent * entropy_loss

        # 3. Balance regularization (prevent dominant operators)
        mean_weights = torch.sum(weights * mask.float().unsqueeze(-1), dim=1) / (mask.float().sum(dim=1, keepdim=True) + 1e-8)  # (B, K)
        uniform_target = 1.0 / K
        balance_loss = torch.mean((mean_weights - uniform_target).pow(2))
        total_loss = total_loss + lambda_bal * balance_loss

        # 4. Diversity regularization (encourage different operators)
        weight_corr = torch.corrcoef(mean_weights.t())
        if not torch.isnan(weight_corr).any():
            off_diag = weight_corr - torch.eye(K, device=weight_corr.device)
            diversity_loss = torch.mean(off_diag.pow(2))
            total_loss = total_loss + lambda_div * diversity_loss

        count += 1

    return total_loss / max(count, 1)

def dhc_row_utilization_loss(model, lambda_row=0.01):
    """
    Regularization for DHC row utilization to prevent single row dominance
    """
    total_loss = torch.tensor(0.0, device=next(model.parameters()).device)
    count = 0

    for name, module in model.named_modules():
        if isinstance(module, DynamicHyperConnection):
            # Get recent activations (if stored) or skip
            # For simplicity, we'll add a small penalty to encourage diversity
            n = module.n
            target_usage = 1.0 / n  # Uniform target

            # Regularize scale parameters to encourage diversity
            scale_params = [module.scale_beta, module.scale_m, module.scale_r]
            for scale_param in scale_params:
                # Encourage non-zero but balanced usage
                usage_penalty = torch.abs(scale_param - target_usage * torch.ones_like(scale_param))
                total_loss = total_loss + lambda_row * usage_penalty.mean()

            count += 1

    return total_loss / max(count, 1)

@torch.no_grad()
def eval_epoch_flight(model, loader, device):
    """Evaluation function for flight data"""
    model.eval()
    total = {"mse": 0.0, "acc": 0.0, "hi_mean": 0.0}
    n_batch = 0
    n_acc = 0
    n_tot = 0

    for batch in loader:
        x = batch["x"].to(device)
        mask = batch["mask"].to(device)
        labels = batch["labels"].to(device)
        before_after = batch["before_after"].to(device)
        valid_lengths = batch["valid_lengths"].to(device)

        B, L, C = x.shape

        # Generate predictions
        hi, y_hat, h_multi, weights, temperature = model(x, mask)

        # Numerical stabilization
        hi, y_hat = sanitize_tensors(hi, y_hat)

        # Reconstruction loss (if applicable)
        if y_hat.size(1) > 0 and L >= 3:
            target = x[:, 1:-1, :]  # (B, L-2, C)
            recon_mask = (torch.arange(L-2, device=device)[None, :] < (valid_lengths-2)[:, None]).float()
            mse = masked_mse(y_hat, target, recon_mask)

            if not torch.isnan(mse):
                total["mse"] += mse.item()

        # Classification using health index
        logits = model.classifier(hi, before_after, mask)
        pred = logits.argmax(dim=1)
        n_acc += (pred == labels).sum().item()
        n_tot += labels.numel()

        # HI statistics
        valid_counts = mask.float().sum(dim=1, keepdim=True).clamp_min(1.0)
        hi_mean = (hi * mask.float()).sum(dim=1, keepdim=True) / valid_counts
        total["hi_mean"] += hi_mean.mean().item()

        n_batch += 1

    for k in ["mse", "hi_mean"]:
        total[k] /= max(n_batch, 1)
    total["acc"] = n_acc / max(n_tot, 1)

    return total

# ============================================================
# 4) Data loading and visualization functions adapted for flight data
# ============================================================
def load_flight_data(data_dir):
    """Load processed flight data"""
    train_path = os.path.join(data_dir, "train_samples_processed_sample_len_150.pkl")
    val_path = os.path.join(data_dir, "val_samples_processed_sample_len_150.pkl")
    test_path = os.path.join(data_dir, "test_samples_processed_sample_len_150.pkl")
    label_mappings_path = os.path.join(data_dir, "label_mappings_sample_len_150.pkl")

    with open(train_path, 'rb') as f:
        train_samples = pickle.load(f)
    with open(val_path, 'rb') as f:
        val_samples = pickle.load(f)
    with open(test_path, 'rb') as f:
        test_samples = pickle.load(f)
    with open(label_mappings_path, 'rb') as f:
        label_mappings = pickle.load(f)

    return train_samples, val_samples, test_samples, label_mappings

def print_data_summary(train_samples, val_samples, test_samples, label_mappings):
    """Print dataset summary"""
    print(f"Dataset Summary:")
    print(f"  Train: {len(train_samples)} samples")
    print(f"  Val: {len(val_samples)} samples")
    print(f"  Test: {len(test_samples)} samples")
    print(f"  Classes: {len(label_mappings['class_to_id'])}")
    print(f"  Features: {list(train_samples.values())[0]['x'].shape[1]}")

@torch.no_grad()
def collect_flight_predictions(model, loader, device, max_curves=12):
    """Collect test predictions for visualization"""
    model.eval()
    y_true, y_pred = [], []
    curves = []

    for batch in loader:
        x = batch["x"].to(device)
        mask = batch["mask"].to(device)
        labels = batch["labels"].to(device)
        before_after = batch["before_after"].to(device)
        master_indices = batch["master_indices"]

        hi, y_hat, h_multi, weights, temperature = model(x, mask)
        hi, y_hat = sanitize_tensors(hi, y_hat)

        # Classification
        logits = model.classifier(hi, before_after, mask)
        prob = F.softmax(logits, dim=1)
        pred = prob.argmax(1)

        y_true.append(labels.cpu().numpy())
        y_pred.append(pred.cpu().numpy())

        # Collect sample curves
        for i in range(x.size(0)):
            if len(curves) >= max_curves:
                break

            valid_len = mask[i].sum().item()

            curves.append({
                "master_index": master_indices[i],
                "true": int(labels[i].cpu().item()),
                "pred": int(pred[i].cpu().item()),
                "prob": prob[i].cpu().numpy(),
                "hi": hi[i, :valid_len].cpu().numpy(),
                "x": x[i, :valid_len].cpu().numpy(),
                "before_after": int(before_after[i].cpu().item()),
                "weights": weights[i, :valid_len].cpu().numpy() if weights is not None else None,
                "temperature": temperature[i, :valid_len].cpu().numpy() if temperature is not None else None
            })

    y_true = np.concatenate(y_true) if y_true else np.array([])
    y_pred = np.concatenate(y_pred) if y_pred else np.array([])

    return y_true, y_pred, curves

def plot_flight_hi_examples(curves, label_mappings, n_show=6):
    """Plot health index examples for flight data"""
    if len(curves) == 0:
        print("(No visualization samples)")
        return

    random.shuffle(curves)
    n_show = min(n_show, len(curves))

    id_to_class = label_mappings.get('id_to_class', {})
    id_to_before_after = {0: 'Before', 1: 'After'}

    cols = 3
    rows = int(np.ceil(n_show / cols))
    plt.figure(figsize=(cols*4.5, rows*3.2))

    for k in range(n_show):
        ex = curves[k]
        hi = ex["hi"]

        master_idx = ex["master_index"]
        true_class = id_to_class.get(ex["true"], f"Class_{ex['true']}")
        pred_class = id_to_class.get(ex["pred"], f"Class_{ex['pred']}")
        prob = ex["prob"]
        before_after = id_to_before_after.get(ex["before_after"], "Unknown")

        plt.subplot(rows, cols, k+1)
        plt.plot(range(len(hi)), hi, linewidth=2, marker='o', markersize=2)

        title = (f"Flight {master_idx} ({before_after})\n"
                f"True: {true_class} | Pred: {pred_class}\n"
                f"Prob: {prob[ex['pred']]:.3f}")
        plt.title(title, fontsize=9)
        plt.xlabel("Time Step")
        plt.ylabel("Learned Health Index")
        plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def plot_flight_weights(curves, n_show=2):
    """Plot operator weights for flight data"""
    curves_with_weights = [c for c in curves if c.get("weights") is not None]
    if len(curves_with_weights) == 0:
        print("(No weight information available)")
        return

    random.shuffle(curves_with_weights)
    n_show = min(n_show, len(curves_with_weights))

    op_names = ["MonotonicLinear", "MonotonicFlat", "ConcaveLog", "SaturationSigmoid",
                "HingeReLU", "Polynomial", "DampedSin", "PiecewiseLinear"]

    for i in range(n_show):
        ex = curves_with_weights[i]
        weights = ex["weights"]  # (T, K)
        temperature = ex.get("temperature")

        master_idx = ex["master_index"]
        true_class = ex["true"]
        pred_class = ex["pred"]

        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        fig.suptitle(f"Flight {master_idx} - Operator Analysis (True: {true_class}, Pred: {pred_class})")

        # 1. Weights over time
        ax = axes[0]
        T, K = weights.shape
        for k in range(K):
            ax.plot(range(T), weights[:, k],
                   label=op_names[k] if k < len(op_names) else f"Op_{k}",
                   marker='o', markersize=1, linewidth=1.5)
        ax.set_title("Operator Weights")
        ax.set_xlabel("Time Step")
        ax.set_ylabel("Weight")
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.grid(True, alpha=0.3)

        # 2. Weight entropy
        ax = axes[1]
        entropy = -np.sum(weights * np.log(weights + 1e-8), axis=1)
        ax.plot(range(T), entropy, linewidth=2, color='red')
        ax.axhline(np.log(K), color='k', linestyle='--', alpha=0.7, label=f'Max Entropy ({np.log(K):.2f})')
        ax.set_title("Weight Entropy")
        ax.set_xlabel("Time Step")
        ax.set_ylabel("Entropy")
        ax.legend()
        ax.grid(True, alpha=0.3)

        # 3. Temperature (if available)
        ax = axes[2]
        if temperature is not None:
            ax.plot(range(T), temperature, linewidth=2, color='blue')
            ax.set_title("Temperature")
            ax.set_xlabel("Time Step")
            ax.set_ylabel("Temperature")
        else:
            ax.text(0.5, 0.5, "Temperature data\nnot available",
                   ha='center', va='center', transform=ax.transAxes)
            ax.set_title("Temperature")
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

# ============================================================
# 5) Training function adapted for flight data with DHC
# ============================================================
def train_flight_model(train_samples, val_samples, test_samples, label_mappings,
                      epochs=50, batch_size=32, lr=1e-3, device=None, patience=10):
    """Training function for flight data with DHC regularization"""
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"Training on device: {device}")

    # Create datasets
    train_dataset = FlightDataset(train_samples, label_key='class_encoded')
    val_dataset = FlightDataset(val_samples, label_key='class_encoded')
    test_dataset = FlightDataset(test_samples, label_key='class_encoded')

    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=flight_collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=flight_collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=flight_collate_fn)

    # Model parameters
    C = train_dataset.C  # Number of features
    n_classes = len(label_mappings['class_to_id'])

    print(f"Model config: features={C}, classes={n_classes}")
    print(f"Hyper-Connections: DHC enabled with n=2 for KAN, Fusion, and Classifier blocks")

    # Create model
    model = FlightDiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=n_classes).to(device)

    # Setup optimizer with different weight decay for DHC components
    dhc_params = []
    other_params = []

    for name, param in model.named_parameters():
        if 'dhc' in name and ('W_beta' in name or 'W_m' in name or 'W_r' in name):
            # Dynamic DHC parameters get weight decay
            dhc_params.append(param)
        elif 'dhc' in name and ('static' in name):
            # Static DHC parameters don't get weight decay
            other_params.append(param)
        else:
            other_params.append(param)

    opt = torch.optim.AdamW([
        {'params': dhc_params, 'weight_decay': 1e-4},  # DHC dynamic params
        {'params': other_params, 'weight_decay': 1e-5}  # Other params
    ], lr=lr)

    ce = nn.CrossEntropyLoss()

    best_val_loss = float('inf')
    best_model = None
    no_improve_count = 0
    model_save_path = str(resolve_data_file("flight_maintenance_dhc.pt"))

    print(f"Starting training for {epochs} epochs with early stopping (patience={patience})...")

    for epoch in range(1, epochs + 1):
        print(f"\n[Epoch {epoch}/{epochs}] Training...")
        model.train()

        train_loss = 0.0
        train_recon_loss = 0.0
        train_cls_loss = 0.0
        train_hi_loss = 0.0
        train_dhc_loss = 0.0
        n_batches = 0

        for batch_idx, batch in enumerate(train_loader):
            if batch_idx % max(1, len(train_loader)//5) == 0:
                progress = (batch_idx + 1) / len(train_loader) * 100
                print(f"    Progress: {progress:.1f}%")

            x = batch["x"].to(device)
            mask = batch["mask"].to(device)
            labels = batch["labels"].to(device)
            before_after = batch["before_after"].to(device)
            valid_lengths = batch["valid_lengths"].to(device)

            B, L, C = x.shape

            # Forward pass
            hi, y_hat, h_multi, weights, temperature = model(x, mask)
            hi, y_hat = sanitize_tensors(hi, y_hat)

            # Reconstruction loss
            recon_loss = torch.tensor(0.0, device=device)
            if y_hat.size(1) > 0 and L >= 3:
                target = x[:, 1:-1, :]
                recon_mask = (torch.arange(L-2, device=device)[None, :] < (valid_lengths-2)[:, None]).float()
                recon_loss = masked_mse(y_hat, target, recon_mask)

            # Classification loss
            logits = model.classifier(hi, before_after, mask)
            cls_loss = ce(logits, labels)

            # Health index regularization
            hi_smooth_loss = slope_loss(hi, mask, "smooth")
            hi_mono_loss = slope_loss(hi, mask, "mono_dec")
            hi_linear_loss = enhanced_linear_slope_loss(hi, mask, weight=5.0)

            # Liquid weight regularization
            weight_reg_loss = enhanced_liquid_weight_regularization([weights], [mask])

            # DHC row utilization regularization
            dhc_reg_loss = dhc_row_utilization_loss(model, lambda_row=0.01)

            # Total loss with DHC regularization
            total_loss = (0.5 * recon_loss +
                         5.0 * cls_loss +  # Increased classification weight
                         0.05 * hi_smooth_loss +  # Reduced regularization
                         0.1 * hi_mono_loss +
                         0.2 * hi_linear_loss +
                         0.1 * weight_reg_loss +  # Reduced regularization
                         0.05 * dhc_reg_loss)  # DHC regularization

            # Backward pass
            opt.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            # Accumulate losses
            train_loss += total_loss.item()
            train_recon_loss += recon_loss.item()
            train_cls_loss += cls_loss.item()
            train_hi_loss += (hi_smooth_loss + hi_mono_loss + hi_linear_loss).item()
            train_dhc_loss += dhc_reg_loss.item()
            n_batches += 1

        # Average losses
        train_loss /= n_batches
        train_recon_loss /= n_batches
        train_cls_loss /= n_batches
        train_hi_loss /= n_batches
        train_dhc_loss /= n_batches

        # Validation
        print("    Validating...")
        val_metrics = eval_epoch_flight(model, val_loader, device)

        print(f"[Epoch {epoch:03d}] Train: total={train_loss:.4f} recon={train_recon_loss:.4f} "
              f"cls={train_cls_loss:.4f} hi={train_hi_loss:.4f} dhc={train_dhc_loss:.4f} | "
              f"Val: mse={val_metrics['mse']:.4f} acc={val_metrics['acc']:.3f} hi_mean={val_metrics['hi_mean']:.3f}")

        # Early stopping check - prioritize accuracy
        val_loss = 2.0 * (1 - val_metrics['acc']) + 0.5 * val_metrics['mse']  # Weighted combination
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model = {k: v.clone() for k, v in model.state_dict().items()}

            # Save best model to specified path
            try:
                torch.save(model.state_dict(), model_save_path)
                print(f"    ✓ New best model saved to {model_save_path} (val_loss: {val_loss:.4f})")
            except Exception as e:
                print(f"    ✓ New best model (val_loss: {val_loss:.4f}) - Save failed: {e}")

            no_improve_count = 0
        else:
            no_improve_count += 1
            print(f"    No improvement ({no_improve_count}/{patience})")

            if no_improve_count >= patience:
                print(f"\n[Early Stopping] No improvement for {patience} epochs.")
                break

    # Load best model
    if best_model is not None:
        model.load_state_dict(best_model)
        print(f"\n[Best Model] Validation loss: {best_val_loss:.4f}")
        print(f"[Best Model] Saved at: {model_save_path}")

    # Test evaluation
    print("\n" + "="*60)
    print("Final Test Set Evaluation")
    print("="*60)
    test_metrics = eval_epoch_flight(model, test_loader, device)
    print(f"[Test] mse={test_metrics['mse']:.4f} acc={test_metrics['acc']:.3f} hi_mean={test_metrics['hi_mean']:.3f}")

    # Collect predictions for analysis
    y_true, y_pred, curves = collect_flight_predictions(model, test_loader, device)

    # Classification report
    if len(y_true) > 0:
        from sklearn.metrics import classification_report, confusion_matrix

        # Fix: Ensure class names are strings for sklearn
        class_names = []
        for i in range(len(label_mappings['class_to_id'])):
            class_name = label_mappings['id_to_class'].get(i, f"Class_{i}")
            class_names.append(str(class_name))  # Convert to string

        print("\n[Classification Report]")
        try:
            print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
        except Exception as e:
            print(f"Classification report error: {e}")
            print(classification_report(y_true, y_pred, zero_division=0))

        print("\n[Confusion Matrix]")
        print(confusion_matrix(y_true, y_pred))

        # Visualizations
        print("\n[Visualization] Health index curves...")
        plot_flight_hi_examples(curves, label_mappings, n_show=6)

        print("\n[Visualization] Operator weights...")
        plot_flight_weights(curves, n_show=2)

    return model, (y_true, y_pred, curves)

# Load flight data
data_dir = str(DATA_DIR)
train_samples, val_samples, test_samples, label_mappings = load_flight_data(data_dir)

print_data_summary(train_samples, val_samples, test_samples, label_mappings)

# Train model with DHC
print("\n" + "="*60)
print("Training with Dynamic Hyper-Connections (DHC)")
print("="*60)
print("DHC Features:")
print("- Dynamic matrix prediction with input normalization")
print("- n=2 expansion for KAN representation block")
print("- n=2 expansion for health fusion block")
print("- n=2 expansion for classifier block")
print("- Row utilization regularization")
print("- Separate weight decay for DHC components")
print("="*60)

model, results = train_flight_model(
    train_samples=train_samples,
    val_samples=val_samples,
    test_samples=test_samples,
    label_mappings=label_mappings,
    epochs=1000,
    batch_size=640,
    lr=1e-3,
    device=None,
    patience=15
)

print("\nTraining completed! Model features:")
print("- Dynamic Hyper-Connections (DHC) with n=2")
print("- Enhanced KAN representation with DHC")
print("- Health fusion using sequence-parallel duality")
print("- Attention-based classification with DHC")
print("- Row utilization regularization")
print("- Separate optimization for DHC components")


In [ ]:
# ============================================================
# Load best model and visualize health index curves before and after maintenance
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from collections import defaultdict
import random

def load_best_model_and_visualize(model_path, test_samples, label_mappings, device=None):
    """
    Load the best model and plot health index curve comparison before and after maintenance
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


    # Recreate model
    C = list(test_samples.values())[0]['x'].shape[1]  # Number of features
    n_classes = len(label_mappings['class_to_id'])

    model = FlightDiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=n_classes).to(device)

    # Load best model weights
    try:
        model.load_state_dict(torch.load(model_path, map_location=device))
        print(f"✓ Successfully loaded model: {model_path}")
    except Exception as e:
        print(f"✗ Failed to load model: {e}")
        return None

    model.eval()

    # Group samples by class and maintenance status
    class_samples = defaultdict(lambda: {"before": [], "after": []})

    for sample_id, sample in test_samples.items():
        class_id = sample['class_encoded']
        before_after = sample['before_after_encoded']

        if before_after == 0:  # Before maintenance
            class_samples[class_id]["before"].append((sample_id, sample))
        else:  # After maintenance
            class_samples[class_id]["after"].append((sample_id, sample))

    # Select a pair of before/after samples for each class
    selected_pairs = []

    for class_id in sorted(class_samples.keys()):
        before_samples = class_samples[class_id]["before"]
        after_samples = class_samples[class_id]["after"]

        if len(before_samples) > 0 and len(after_samples) > 0:
            # Randomly select one before and one after maintenance sample
            before_sample = random.choice(before_samples)
            after_sample = random.choice(after_samples)

            selected_pairs.append({
                "class_id": class_id,
                "class_name": label_mappings['id_to_class'].get(class_id, f"Class_{class_id}"),
                "before": before_sample,
                "after": after_sample
            })

    print(f"Found {len(selected_pairs)} class pairs with before/after maintenance samples")

    # Plot maintenance comparison of health index curves
    plot_maintenance_comparison(model, selected_pairs, device)

    return model, selected_pairs

def plot_maintenance_comparison(model, selected_pairs, device, max_classes=12):
    """
    Plot health index curve comparison before and after maintenance
    """
    if len(selected_pairs) == 0:
        print("No sample pairs available for plotting")
        return

    # Limit the number of classes to display
    pairs_to_plot = selected_pairs[:max_classes]
    n_pairs = len(pairs_to_plot)

    # Calculate subplot layout
    cols = 3
    rows = int(np.ceil(n_pairs / cols))

    plt.figure(figsize=(cols * 5, rows * 4))

    with torch.no_grad():
        for idx, pair in enumerate(pairs_to_plot):
            class_name = pair["class_name"]
            before_sample_id, before_sample = pair["before"]
            after_sample_id, after_sample = pair["after"]

            # Predict health index before maintenance
            x_before = torch.from_numpy(before_sample['x']).float().unsqueeze(0).to(device)
            mask_before = torch.from_numpy(before_sample['mask']).bool().unsqueeze(0).to(device)

            hi_before, _, _, _, _ = model(x_before, mask_before)
            hi_before = hi_before.squeeze(0).cpu().numpy()

            # Predict health index after maintenance
            x_after = torch.from_numpy(after_sample['x']).float().unsqueeze(0).to(device)
            mask_after = torch.from_numpy(after_sample['mask']).bool().unsqueeze(0).to(device)

            hi_after, _, _, _, _ = model(x_after, mask_after)
            hi_after = hi_after.squeeze(0).cpu().numpy()

            # Get valid lengths
            valid_len_before = before_sample['valid_length']
            valid_len_after = after_sample['valid_length']

            # Extract valid portions
            hi_before_valid = hi_before[:valid_len_before]
            hi_after_valid = hi_after[:valid_len_after]

            # Create continuous time axis
            time_before = np.arange(0, valid_len_before)
            time_after = np.arange(valid_len_before, valid_len_before + valid_len_after)

            # Plot subplot
            plt.subplot(rows, cols, idx + 1)

            # Plot health index curve before maintenance (blue)
            plt.plot(time_before, hi_before_valid, 'b-', linewidth=2, marker='o',
                    markersize=3, label='Before Maintenance', alpha=0.8)

            # Plot health index curve after maintenance (red)
            plt.plot(time_after, hi_after_valid, 'r-', linewidth=2, marker='s',
                    markersize=3, label='After Maintenance', alpha=0.8)

            # Add vertical divider at maintenance point
            plt.axvline(x=valid_len_before, color='green', linestyle='--',
                       linewidth=2, alpha=0.7, label='Maintenance Point')

            # Add maintenance effect arrow (if health index improves significantly after maintenance)
            if len(hi_before_valid) > 0 and len(hi_after_valid) > 0:
                last_before = hi_before_valid[-1]
                first_after = hi_after_valid[0]
                if first_after > last_before:
                    # Draw maintenance effect arrow
                    plt.annotate('', xy=(valid_len_before, first_after),
                               xytext=(valid_len_before, last_before),
                               arrowprops=dict(arrowstyle='->', color='green', lw=2))

            # Set title and labels
            plt.title(f'{class_name}\n'
                     f'Before Sample: {before_sample_id}\n'
                     f'After Sample: {after_sample_id}',
                     fontsize=10, pad=10)
            plt.xlabel('Time Step')
            plt.ylabel('Health Index (HI)')
            plt.grid(True, alpha=0.3)
            plt.legend(fontsize=8)

            # Set y-axis range to better display changes
            all_hi = np.concatenate([hi_before_valid, hi_after_valid])
            if len(all_hi) > 0:
                y_min, y_max = all_hi.min(), all_hi.max()
                y_range = y_max - y_min
                if y_range > 0:
                    plt.ylim(y_min - 0.1 * y_range, y_max + 0.1 * y_range)

    plt.suptitle('Aircraft Health Index Comparison Before and After Maintenance\n(Blue: Before, Red: After, Green Dashed: Maintenance Point)',
                fontsize=14, y=0.98)
    plt.tight_layout()
    plt.subplots_adjust(top=0.92)
    plt.show()

def analyze_maintenance_effect(model, selected_pairs, device):
    """
    Analyze statistical information of maintenance effects
    """
    print("\n" + "="*60)
    print("Maintenance Effect Analysis")
    print("="*60)

    maintenance_effects = []

    with torch.no_grad():
        for pair in selected_pairs:
            class_name = pair["class_name"]
            before_sample_id, before_sample = pair["before"]
            after_sample_id, after_sample = pair["after"]

            # Predict health indices
            x_before = torch.from_numpy(before_sample['x']).float().unsqueeze(0).to(device)
            mask_before = torch.from_numpy(before_sample['mask']).bool().unsqueeze(0).to(device)
            hi_before, _, _, _, _ = model(x_before, mask_before)
            hi_before = hi_before.squeeze(0).cpu().numpy()

            x_after = torch.from_numpy(after_sample['x']).float().unsqueeze(0).to(device)
            mask_after = torch.from_numpy(after_sample['mask']).bool().unsqueeze(0).to(device)
            hi_after, _, _, _, _ = model(x_after, mask_after)
            hi_after = hi_after.squeeze(0).cpu().numpy()

            # Calculate maintenance effect
            valid_len_before = before_sample['valid_length']
            valid_len_after = after_sample['valid_length']

            hi_before_valid = hi_before[:valid_len_before]
            hi_after_valid = hi_after[:valid_len_after]

            if len(hi_before_valid) > 0 and len(hi_after_valid) > 0:
                # Average and ending health index before maintenance
                avg_before = np.mean(hi_before_valid)
                end_before = hi_before_valid[-1]

                # Average and starting health index after maintenance
                avg_after = np.mean(hi_after_valid)
                start_after = hi_after_valid[0]

                # Calculate maintenance effects
                avg_improvement = avg_after - avg_before
                immediate_improvement = start_after - end_before

                maintenance_effects.append({
                    'class_name': class_name,
                    'avg_improvement': avg_improvement,
                    'immediate_improvement': immediate_improvement,
                    'before_avg': avg_before,
                    'after_avg': avg_after
                })

                print(f"{str(class_name):20s} | "
                      f"Before Avg: {avg_before:.3f} | "
                      f"After Avg: {avg_after:.3f} | "
                      f"Avg Improvement: {avg_improvement:+.3f} | "
                      f"Immediate Improvement: {immediate_improvement:+.3f}")

    # Overall statistics
    if maintenance_effects:
        avg_improvements = [effect['avg_improvement'] for effect in maintenance_effects]
        immediate_improvements = [effect['immediate_improvement'] for effect in maintenance_effects]

        print("\n" + "-"*60)
        print("Overall Maintenance Effect Statistics:")
        print(f"Average Health Index Improvement: {np.mean(avg_improvements):+.3f} ± {np.std(avg_improvements):.3f}")
        print(f"Immediate Health Index Improvement: {np.mean(immediate_improvements):+.3f} ± {np.std(immediate_improvements):.3f}")
        print(f"Effective Maintenance Rate: {np.sum(np.array(avg_improvements) > 0) / len(avg_improvements) * 100:.1f}%")

# Execute visualization
print("\n" + "="*60)
print("Load Best Model and Visualize Health Index Curves Before/After Maintenance")
print("="*60)

model_path = str(resolve_data_file("flight_maintenance_dhc.pt"))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model and visualize
model, selected_pairs = load_best_model_and_visualize(
    model_path=model_path,
    test_samples=test_samples,
    label_mappings=label_mappings,
    device=device
)

if model is not None and selected_pairs:
    # Analyze maintenance effects
    analyze_maintenance_effect(model, selected_pairs, device)

    print(f"\n✓ Successfully plotted health index curve comparisons for {len(selected_pairs)} classes before/after maintenance")
    print("Chart shows:")
    print("  - Blue line: Health index changes before maintenance")
    print("  - Red line: Health index changes after maintenance")
    print("  - Green dashed line: Maintenance time point")
    print("  - X-axis continuously shows time progression")
else:
    print("✗ Failed to load model or process data")


## Channel-Wise BoltzmannKAN Importance Analysis


In [ ]:
# ============================================================
# BoltzmannKAN Importance Analysis (by channel output proportion change before/after)
# Paste this code at the bottom of your existing script or in an appropriate location to run
# Dependencies: DiffAwareReconstructor, adapt_batch, SENSOR_NAMES, id_to_name, K, test_loader
# ============================================================
import torch
import numpy as np
import pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt
import re

# ----------------------------
# 1) Load the trained best model weights (as used in your test_only checkpoint)
# ----------------------------
def load_model_for_analysis(ckpt_path, C, K, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = DiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=K).to(device)
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state, strict=False)
    model.eval()
    # Freeze parameters
    for p in model.parameters():
        p.requires_grad = False
    return model, device

# ----------------------------
# 2) Compute Boltzmann layer channel output proportions before and after maintenance
# ----------------------------
def compute_boltzmann_channel_proportions(model, loader, device, K):
    """
    Returns DataFrame: importance for each class-sensor pair based on the change in
    Boltzmann layer output proportion between before and after maintenance.
    """
    boltz = model.encoder.boltz
    C = boltz.E.shape[1]

    # Aggregate: each class -> sensor -> {prop_before, prop_after, count}
    # Initialize with proper shape
    agg_before = {}
    agg_after = {}

    # Create a safe loader with num_workers=0 to avoid worker process issues
    safe_loader = torch.utils.data.DataLoader(
        loader.dataset,
        batch_size=loader.batch_size if hasattr(loader, 'batch_size') else 32,
        shuffle=False,
        num_workers=0,
        persistent_workers=False
    )

    with torch.no_grad():
        for batch in safe_loader:
            xb, xa, labels, mask, lengths, sample_ids, _ = adapt_batch(batch, device)

            # Need labels for per-class statistics; skip if no labels
            if labels is None:
                continue
            valid_labels = (labels != -100)
            if not valid_labels.any():
                continue

            # Take valid samples
            xb_valid = xb[valid_labels]
            xa_valid = xa[valid_labels]
            y = labels[valid_labels]

            if xb_valid.numel() == 0:
                continue

            # Get Boltzmann layer outputs for before and after
            # Shape: (B, T, C) -> we need to aggregate across time and batch
            boltz_out_before = boltz(xb_valid)  # (B, T_before, C)
            boltz_out_after = boltz(xa_valid)   # (B, T_after, C)

            # Compute mean absolute output per channel across time dimension
            # Then compute proportion of each channel relative to total output
            mean_out_before = boltz_out_before.abs().mean(dim=1)  # (B, C)
            mean_out_after = boltz_out_after.abs().mean(dim=1)    # (B, C)

            # Normalize to get proportions (sum to 1 across channels for each sample)
            prop_before = mean_out_before / (mean_out_before.sum(dim=1, keepdim=True) + 1e-8)  # (B, C)
            prop_after = mean_out_after / (mean_out_after.sum(dim=1, keepdim=True) + 1e-8)    # (B, C)

            # Accumulate by class
            y_np = y.detach().cpu().numpy()
            prop_before_np = prop_before.detach().cpu().numpy()
            prop_after_np = prop_after.detach().cpu().numpy()

            for i, cls_id in enumerate(y_np):
                cls_id = int(cls_id)

                # Initialize dictionaries for this class if not exists
                # Use the actual C dimension from the Boltzmann output
                if cls_id not in agg_before:
                    agg_before[cls_id] = {"output_sum": np.zeros(C, dtype=np.float64), "cnt": 0}
                if cls_id not in agg_after:
                    agg_after[cls_id] = {"output_sum": np.zeros(C, dtype=np.float64), "cnt": 0}

                # Ensure the shapes match before adding
                if len(prop_before_np[i]) == C and len(prop_after_np[i]) == C:
                    agg_before[cls_id]["output_sum"] += prop_before_np[i]
                    agg_before[cls_id]["cnt"] += 1
                    agg_after[cls_id]["output_sum"] += prop_after_np[i]
                    agg_after[cls_id]["cnt"] += 1
                else:
                    # Skip samples with mismatched dimensions
                    print(f"[Warning] Skipping sample with dimension mismatch: expected {C}, got before={len(prop_before_np[i])}, after={len(prop_after_np[i])}")
                    continue

    # Compute change for each class-sensor pair
    records = []
    all_classes = set(agg_before.keys()) | set(agg_after.keys())

    for cls_id in all_classes:
        # Average proportions
        cnt_before = max(agg_before.get(cls_id, {"cnt": 1})["cnt"], 1)
        cnt_after = max(agg_after.get(cls_id, {"cnt": 1})["cnt"], 1)
        prop_before_avg = agg_before.get(cls_id, {"output_sum": np.zeros(C)})["output_sum"] / cnt_before
        prop_after_avg = agg_after.get(cls_id, {"output_sum": np.zeros(C)})["output_sum"] / cnt_after

        # For each sensor, compute the change in proportion
        for c_idx in range(C):
            prop_change = abs(prop_after_avg[c_idx] - prop_before_avg[c_idx])

            records.append({
                "class_id": cls_id,
                "sensor_id": c_idx,
                "sensor_name": SENSOR_NAMES.get(c_idx, f"Sensor_{c_idx}"),
                "prop_before": float(prop_before_avg[c_idx]),
                "prop_after": float(prop_after_avg[c_idx]),
                "prop_change": float(prop_change),
                "importance": float(prop_change)
            })

    df = pd.DataFrame(records)
    return df

# ----------------------------
# 3) Static importance (based on average output proportion across all data)
# ----------------------------
def compute_static_importance(model, loader, device):
    """
    Returns DataFrame: static importance for each sensor based on average output proportion.
    """
    boltz = model.encoder.boltz
    C = boltz.E.shape[1]

    output_sum = None
    cnt = 0

    # Create a safe loader with num_workers=0
    safe_loader = torch.utils.data.DataLoader(
        loader.dataset,
        batch_size=loader.batch_size if hasattr(loader, 'batch_size') else 32,
        shuffle=False,
        num_workers=0,
        persistent_workers=False
    )

    with torch.no_grad():
        for batch in safe_loader:
            xb, xa, labels, mask, lengths, sample_ids, _ = adapt_batch(batch, device)

            # Get Boltzmann layer outputs for both before and after
            boltz_out_before = boltz(xb)  # (B, T_before, C)
            boltz_out_after = boltz(xa)   # (B, T_after, C)

            # Concatenate along time dimension
            boltz_out = torch.cat([boltz_out_before, boltz_out_after], dim=1)  # (B, T_total, C)

            # Mean across time and batch
            mean_out = boltz_out.abs().mean(dim=1)  # (B, C)
            prop = mean_out / (mean_out.sum(dim=1, keepdim=True) + 1e-8)

            prop_sum = prop.sum(dim=0).detach().cpu().numpy()

            # Initialize output_sum with correct shape on first batch
            if output_sum is None:
                output_sum = np.zeros_like(prop_sum, dtype=np.float64)

            output_sum += prop_sum
            cnt += prop.shape[0]

    avg_prop = output_sum / max(cnt, 1)

    df = pd.DataFrame({
        "sensor_id": np.arange(len(avg_prop), dtype=int),
        "sensor_name": [SENSOR_NAMES.get(i, f"Sensor_{i}") for i in range(len(avg_prop))],
        "avg_output_proportion": avg_prop,
        "static_importance": avg_prop
    }).sort_values("static_importance", ascending=False)
    return df

# ----------------------------
# 4) Select top 8 sensors per class based on proportion change
# ----------------------------
def select_top_sensors_per_class_by_change(df_imp, id_to_name, topk=8):
    """
    For each class, select top 8 sensors based on Boltzmann output proportion change.
    Returns DataFrame with class_id, class_name, and top sensor recommendations.
    """
    rows = []
    all_classes = df_imp["class_id"].unique()

    for cls in sorted(all_classes):
        nm = id_to_name.get(int(cls), f"Class_{cls}")

        # Top sensors by importance
        sub = df_imp[df_imp["class_id"]==cls].sort_values("importance", ascending=False).head(topk)
        top_ids = sub["sensor_id"].tolist()
        top_names = sub["sensor_name"].tolist()
        top_scores = sub["importance"].tolist()
        prop_befores = sub["prop_before"].tolist()
        prop_afters = sub["prop_after"].tolist()

        rows.append({
            "class_id": int(cls),
            "class_name": nm,
            "top_sensor_ids": top_ids,
            "top_sensor_names": top_names,
            "importance_scores": top_scores,
            "prop_before": prop_befores,
            "prop_after": prop_afters
        })
    return pd.DataFrame(rows)

# ----------------------------
# 5) Create comprehensive sensor significance table for all classes
# ----------------------------
def create_sensor_significance_table(df_imp, id_to_name, topk=8):
    """
    Create comprehensive table showing sensor significance for all classes.
    Returns formatted DataFrame.
    """
    all_classes = df_imp["class_id"].unique()

    table_data = []
    for cls in sorted(all_classes):
        cls_name = id_to_name.get(int(cls), f"Class_{cls}")
        sub = df_imp[df_imp["class_id"]==cls].sort_values("importance", ascending=False).head(topk)

        row_data = {"Class": cls_name}
        for rank, (idx, row) in enumerate(sub.iterrows(), 1):
            sensor_col = f"Sensor_{rank}"
            score_col = f"ΔProp_{rank}"
            before_col = f"Before_{rank}"
            after_col = f"After_{rank}"
            row_data[sensor_col] = row["sensor_name"]
            row_data[score_col] = f"{row['prop_change']:.6f}"
            row_data[before_col] = f"{row['prop_before']:.6f}"
            row_data[after_col] = f"{row['prop_after']:.6f}"
        table_data.append(row_data)

    return pd.DataFrame(table_data)

# ============================================================
# ▶ Main workflow: Run analysis
# ============================================================
# Get C, K, device, and your best checkpoint path (same as used in testing)
# Create a temporary loader with num_workers=0 to get the first batch safely
temp_loader = torch.utils.data.DataLoader(
    test_loader.dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    persistent_workers=False
)
_first_batch = next(iter(temp_loader))
C_an = _first_batch["x_before"].shape[-1]
ckpt_for_analysis = str(resolve_data_file("best_model_NGFAID_250_100_7.pth"))

model_an, device_an = load_model_for_analysis(ckpt_for_analysis, C=C_an, K=K, device=None)

# --- Static importance ---
df_static = compute_static_importance(model_an, test_loader, device_an)
print("\n[Static importance by Boltzmann output proportion]")
print(df_static.head(20))

# --- Channel output proportion change importance (before vs after maintenance) ---
df_imp = compute_boltzmann_channel_proportions(model_an, test_loader, device_an, K)

print("\n[Per-class sensor importance based on Boltzmann output proportion changes — head]")
print(df_imp.head(20))

# --- Select top 8 sensors per class based on proportion changes ---
df_top_sensors = select_top_sensors_per_class_by_change(df_imp, id_to_name, topk=8)

print("\n" + "="*80)
print("Top 8 Sensors for Each Class (Based on Boltzmann Output Proportion Changes)")
print("="*80)
for idx, row in df_top_sensors.iterrows():
    print(f"\nClass: {row['class_name']}")
    print("-" * 60)
    for i, (sensor_name, score, prop_b, prop_a) in enumerate(zip(
        row['top_sensor_names'],
        row['importance_scores'],
        row['prop_before'],
        row['prop_after']
    ), 1):
        print(f"  {i}. {sensor_name:30s} - ΔProp: {score:.6f} (Before: {prop_b:.6f}, After: {prop_a:.6f})")

# --- Create comprehensive sensor significance table ---
df_sig = create_sensor_significance_table(df_imp, id_to_name, topk=8)

print("\n" + "="*80)
print("Sensor Significance Table for All Classes (Boltzmann Proportion Changes)")
print("="*80)
print(df_sig.to_string(index=False))

# --- Optional: Export results to CSV
df_top_sensors.to_csv("top_sensors_per_class_by_boltzmann_proportion.csv", index=False)
df_sig.to_csv("sensor_significance_by_boltzmann_proportion.csv", index=False)
print("\n[Info] Results exported to CSV files:")
print("  - top_sensors_per_class_by_boltzmann_proportion.csv")
print("  - sensor_significance_by_boltzmann_proportion.csv")


## Monotonic HI Model Variant


In [ ]:
# ============================================================
# Monotonic-Decreasing HI + aligned plotting (before -> after)
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import os
import random
import matplotlib.pyplot as plt
import pickle

# ============================================================
# 1) Dataset: read samples from flight data dictionaries
# ============================================================
class FlightDataset(Dataset):
    """
    Each sample contains:
      x: (L,C) feature sequence
      mask: (L,) validity mask for padded sequences
      valid_length: actual sequence length
      label: encoded label for classification
      class_encoded: encoded main class
      before_after_encoded: 0 for before maintenance, 1 for after maintenance
    """
    def __init__(self, samples_dict, label_key='class'):
        self.samples = list(samples_dict.values())
        self.label_key = label_key

        # Extract feature dimension from first sample
        if len(self.samples) > 0:
            self.C = self.samples[0]['x'].shape[1]
        else:
            raise ValueError("Empty dataset")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        return {
            "x": torch.from_numpy(sample['x']).float(),  # (L,C)
            "mask": torch.from_numpy(sample['mask']).bool(),  # (L,)
            "valid_length": torch.tensor(sample['valid_length'], dtype=torch.long),
            "label": torch.tensor(sample[self.label_key], dtype=torch.long),
            "before_after": torch.tensor(sample['before_after'], dtype=torch.long),
            "master_index": sample['master_index'],
            "is_padded": sample['is_padded']
        }

def flight_collate_fn(batch):
    """
    Collate function for flight data - sequences are already padded to same length
    """
    batch_size = len(batch)
    if batch_size == 0:
        return {}

    # All sequences should have the same length after windowing
    x = torch.stack([b["x"] for b in batch], 0)  # (B,L,C)
    mask = torch.stack([b["mask"] for b in batch], 0)  # (B,L)
    valid_lengths = torch.stack([b["valid_length"] for b in batch], 0)  # (B,)
    labels = torch.stack([b["label"] for b in batch], 0)  # (B,)
    before_after = torch.stack([b["before_after"] for b in batch], 0)  # (B,)
    master_indices = [b["master_index"] for b in batch]

    return {
        "x": x,
        "mask": mask,
        "valid_lengths": valid_lengths,
        "labels": labels,
        "before_after": before_after,
        "master_indices": master_indices
    }

# ============================================================
# 2) Model: encode "damage" → project to monotonic decreasing HI → recursive reconstruction → classify
# ============================================================
class BoltzmannKAN(nn.Module):
    def __init__(self, in_ch, out_ch=4):
        super().__init__()
        self.E  = nn.Parameter(torch.zeros(out_ch, in_ch))
        self.kT = nn.Parameter(torch.ones(out_ch, in_ch))
    def forward(self, x):
        B,T,C = x.shape
        kT = torch.clamp(F.softplus(self.kT), 0.01, 10.0).unsqueeze(0).unsqueeze(2)  # (1,out_ch,1,in_ch)
        E  = torch.clamp(self.E, -10.0, 10.0).unsqueeze(0).unsqueeze(2)             # (1,out_ch,1,in_ch)
        x_ = torch.clamp(x.unsqueeze(1), -10.0, 10.0)                               # (B, out_ch, T, in_ch)
        exp = torch.clamp((E - x_) / kT, -50, 50)
        w   = torch.sigmoid(exp)
        h   = (w * x_).sum(dim=3).permute(0, 2, 1)           # (B,T,out_ch) >=0
        return torch.clamp(F.softplus(h), 0.0, 100.0)

class BaseOp(nn.Module):
    """
    Base operator class with support for dynamic parameter modulation
    """
    def __init__(self):
        super().__init__()
        self.param_modulator = None

    def set_param_modulator(self, modulator):
        """Set the parameter modulator"""
        self.param_modulator = modulator

    def get_modulated_params(self, context):
        """Get parameters after context-based modulation"""
        if self.param_modulator is None:
            return {}
        return self.param_modulator(context)

class MonotonicLinearOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0

    def forward(self, h, context=None):
        # Retrieve the modulated parameters
        modulated_params = self.get_modulated_params(context) if context is not None else {}

        # Base parameters
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_bias = self.bias

        # Parameter modulation
        scale_offset = modulated_params.get('scale_offset', 0.0)
        bias_offset = modulated_params.get('bias_offset', 0.0)

        # Modulated parameters
        scale = torch.clamp(base_scale + scale_offset, self.smin, self.smax)
        bias = torch.clamp(base_bias + bias_offset, -5.0, 5.0)

        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * (xm + bias)), 0.0, 100.0)

class MonotonicFlatOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 1e-3, 1.0

    def _cum(self, x):
        # Fix: ensure computations are carried out along the time dimension
        B, T = x.shape  # xshould have shape (B, T)
        diff = F.relu(x[:, 1:] - x[:, :-1])  # (B, T-1)
        zero_init = torch.zeros(B, 1, device=x.device, dtype=x.dtype)  # (B, 1)
        return torch.cat([zero_init, torch.cumsum(diff, dim=1)], dim=1)  # (B, T)

    def forward(self, h, context=None):
        modulated_params = self.get_modulated_params(context) if context is not None else {}

        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_bias = self.bias

        scale_offset = modulated_params.get('scale_offset', 0.0)
        bias_offset = modulated_params.get('bias_offset', 0.0)

        scale = torch.clamp(base_scale + scale_offset, self.smin, self.smax)
        bias = torch.clamp(base_bias + bias_offset, -2.0, 2.0)

        xm = torch.clamp(h.mean(-1), -10.0, 10.0)  # (B, T)
        cum_result = self._cum(xm)  # (B, T)
        return torch.clamp(F.softplus(scale * (cum_result + bias)), 0.0, 100.0)

class ConcaveLogOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.eps = 1e-3
        self.smin, self.smax = 0.01, 5.0

    def forward(self, h, context=None):
        modulated_params = self.get_modulated_params(context) if context is not None else {}

        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        scale_offset = modulated_params.get('scale_offset', 0.0)
        eps_offset = modulated_params.get('eps_offset', 0.0)

        scale = torch.clamp(base_scale + scale_offset, self.smin, self.smax)
        eps = torch.clamp(self.eps + eps_offset, 1e-6, 1e-1)

        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * torch.log(torch.abs(xm) + eps)), 0.0, 100.0)

class SaturationSigmoidOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.raw_slope = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
        self.lmin, self.lmax = 0.1, 5.0

    def forward(self, h, context=None):
        modulated_params = self.get_modulated_params(context) if context is not None else {}

        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_slope = torch.clamp(F.softplus(self.raw_slope), self.lmin, self.lmax)
        base_bias = self.bias

        scale_offset = modulated_params.get('scale_offset', 0.0)
        slope_offset = modulated_params.get('slope_offset', 0.0)
        bias_offset = modulated_params.get('bias_offset', 0.0)

        scale = torch.clamp(base_scale + scale_offset, self.smin, self.smax)
        slope = torch.clamp(base_slope + slope_offset, self.lmin, self.lmax)
        bias = torch.clamp(base_bias + bias_offset, -3.0, 3.0)

        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * torch.sigmoid(slope * (xm - bias))), 0.0, 100.0)

class HingeReLUOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.threshold = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0

    def forward(self, h, context=None):
        modulated_params = self.get_modulated_params(context) if context is not None else {}

        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_threshold = self.threshold

        scale_offset = modulated_params.get('scale_offset', 0.0)
        threshold_offset = modulated_params.get('threshold_offset', 0.0)

        scale = torch.clamp(base_scale + scale_offset, self.smin, self.smax)
        threshold = torch.clamp(base_threshold + threshold_offset, -3.0, 3.0)

        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * F.relu(xm - threshold)), 0.0, 100.0)

class PolynomialOp(BaseOp):
    def __init__(self, deg=3):
        super().__init__()
        self.raw_coeff = nn.Parameter(torch.zeros(deg))
        self.deg = deg

    def forward(self, h, context=None):
        modulated_params = self.get_modulated_params(context) if context is not None else {}

        base_coeff = torch.clamp(F.softplus(self.raw_coeff), 0.01, 5.0)

        # Fix: Use broadcasting compatible approach for batch dimension
        xm = torch.clamp(h.mean(-1), -5.0, 5.0)  # (B, T)
        B, T = xm.shape

        # Handle coefficient offsets properly for broadcasting
        coeff_offset = modulated_params.get('coeff_offset', torch.zeros_like(base_coeff))
        if coeff_offset.dim() > 1:
            # If coeff_offset is (B, deg), take mean over batch for simplicity
            coeff_offset = coeff_offset.mean(0)

        coeff = torch.clamp(base_coeff + coeff_offset, 0.01, 5.0)  # (deg,)

        y = torch.zeros_like(xm)  # (B, T)
        for i in range(self.deg):
            # Broadcast coefficient to match batch dimension
            power_term = torch.clamp(xm ** (i + 1), -100.0, 100.0)  # (B, T)
            y = y + coeff[i] * power_term  # (B, T) + scalar * (B, T)
        return torch.clamp(F.softplus(y), 0.0, 100.0)

class DampedSinOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.raw_freq = nn.Parameter(torch.tensor(0.0))
        self.raw_lambda = nn.Parameter(torch.tensor(0.0))
        self.phase = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
        self.fmin, self.fmax = 0.1, 5.0
        self.lmin, self.lmax = 0.01, 3.0

    def forward(self, h, context=None):
        modulated_params = self.get_modulated_params(context) if context is not None else {}

        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_freq = torch.clamp(F.softplus(self.raw_freq), self.fmin, self.fmax)
        base_lambda = torch.clamp(F.softplus(self.raw_lambda), self.lmin, self.lmax)
        base_phase = torch.clamp(self.phase, -10.0, 10.0)

        scale_offset = modulated_params.get('scale_offset', 0.0)
        freq_offset = modulated_params.get('freq_offset', 0.0)
        lambda_offset = modulated_params.get('lambda_offset', 0.0)
        phase_offset = modulated_params.get('phase_offset', 0.0)

        scale = torch.clamp(base_scale + scale_offset, self.smin, self.smax)
        freq = torch.clamp(base_freq + freq_offset, self.fmin, self.fmax)
        lam = torch.clamp(base_lambda + lambda_offset, self.lmin, self.lmax)
        phase = torch.clamp(base_phase + phase_offset, -10.0, 10.0)

        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        damp = 1 / (1 + lam * torch.abs(xm))
        return torch.clamp(F.softplus(scale * damp * (torch.sin(freq * xm + phase) + 1)), 0.0, 100.0)

class PiecewiseLinearOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_k1 = nn.Parameter(torch.tensor(0.0))
        self.raw_k2 = nn.Parameter(torch.tensor(0.0))
        self.threshold = nn.Parameter(torch.tensor(0.0))
        self.kmin, self.kmax = 0.01, 5.0

    def forward(self, h, context=None):
        modulated_params = self.get_modulated_params(context) if context is not None else {}

        base_k1 = torch.clamp(F.softplus(self.raw_k1), self.kmin, self.kmax)
        base_k2 = torch.clamp(F.softplus(self.raw_k2), self.kmin, self.kmax)
        base_threshold = torch.clamp(self.threshold, -5.0, 5.0)

        k1_offset = modulated_params.get('k1_offset', 0.0)
        k2_offset = modulated_params.get('k2_offset', 0.0)
        threshold_offset = modulated_params.get('threshold_offset', 0.0)

        k1 = torch.clamp(base_k1 + k1_offset, self.kmin, self.kmax)
        k2 = torch.clamp(base_k2 + k2_offset, self.kmin, self.kmax)
        threshold = torch.clamp(base_threshold + threshold_offset, -5.0, 5.0)

        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        left = k1 * xm
        right = k1 * threshold + k2 * (xm - threshold)
        out = torch.where(xm <= threshold, left, right)
        return torch.clamp(F.softplus(out), 0.0, 100.0)

class ParameterModulator(nn.Module):
    """
    Parameter modulator:dynamically adjust operator parameters based on input context
    """
    def __init__(self, context_dim, param_configs):
        super().__init__()
        self.param_configs = param_configs

        # Create a prediction network for each parameter
        self.param_predictors = nn.ModuleDict()
        for param_name, param_info in param_configs.items():
            param_dim = param_info.get('dim', 1)
            self.param_predictors[param_name] = nn.Sequential(
                nn.Linear(context_dim, 64),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Linear(32, param_dim),
                nn.Tanh()  # Output range [-1, 1]
            )

    def forward(self, context):
        """
        Args:
            context: (B, T, context_dim) context features
        Returns:
            dict: modulated parameter offsets
        """
        # Average across the time dimension to obtain a global context
        global_context = context.mean(dim=1)  # (B, context_dim)

        modulated_params = {}
        for param_name, predictor in self.param_predictors.items():
            param_info = self.param_configs[param_name]
            raw_offset = predictor(global_context)  # (B, param_dim)

            # Scale offsets according to the configuration
            scale = param_info.get('scale', 0.1)
            modulated_params[param_name] = raw_offset * scale

        return modulated_params

class LiquidWeightGenerator(nn.Module):
    """
    Improved Liquid Weight Generator based on Boltzmann KAN features
    """
    def __init__(self, h_dim, n_ops, hidden_dim=64, tau_min=1.0, tau_max=3.0):
        super().__init__()
        self.n_ops = n_ops
        self.tau_min, self.tau_max = tau_min, tau_max

        # Feature encoder for Boltzmann KAN output
        self.feature_encoder = nn.Sequential(
            nn.Linear(h_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim)
        )

        # Add temporal encoding
        self.temporal_encoder = nn.Sequential(
            nn.Linear(2, hidden_dim // 4),  # t, dt
            nn.ReLU()
        )

        # Combined feature processing
        combined_dim = hidden_dim + hidden_dim // 4
        self.feature_fusion = nn.Sequential(
            nn.Linear(combined_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim)
        )

        # Weight prediction with explicit per-operator branches
        self.op_branches = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim // 4),
                nn.ReLU(),
                nn.Linear(hidden_dim // 4, 1)
            ) for _ in range(n_ops)
        ])

        # Temperature predictor
        self.temp_predictor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 4),
            nn.ReLU(),
            nn.Linear(hidden_dim // 4, 1)
        )

        # Initialize to encourage diversity
        for branch in self.op_branches:
            nn.init.xavier_uniform_(branch[0].weight)
            nn.init.xavier_uniform_(branch[2].weight)

    def forward(self, h_multi):
        """
        Args:
            h_multi: (B, T, h_dim) - Boltzmann KAN output
        Returns:
            weights: (B, T, K) - Per-timestep operator weights
            temperature: (B, T) - Temperature parameters
        """
        B, T, h_dim = h_multi.shape

        # Process Boltzmann KAN features
        features = self.feature_encoder(h_multi)  # (B, T, hidden_dim)

        # Add temporal encoding
        t_normalized = torch.arange(T, device=h_multi.device, dtype=h_multi.dtype).unsqueeze(0).expand(B, -1) / max(T-1, 1)
        dt = torch.zeros_like(t_normalized)
        dt[:, 1:] = t_normalized[:, 1:] - t_normalized[:, :-1]

        temporal_input = torch.stack([t_normalized, dt], dim=-1)  # (B, T, 2)
        temporal_features = self.temporal_encoder(temporal_input)  # (B, T, hidden_dim//4)

        # Fuse features
        combined = torch.cat([features, temporal_features], dim=-1)
        fused_features = self.feature_fusion(combined)  # (B, T, hidden_dim)

        # Generate per-operator logits using separate branches
        raw_logits = []
        for i, branch in enumerate(self.op_branches):
            logit = branch(fused_features).squeeze(-1)  # (B, T)
            raw_logits.append(logit)
        raw_weights = torch.stack(raw_logits, dim=-1)  # (B, T, K)

        # Zero-mean normalization to prevent systematic bias
        raw_weights = raw_weights - raw_weights.mean(dim=-1, keepdim=True)

        # Predict temperature
        raw_temp = self.temp_predictor(fused_features).squeeze(-1)  # (B, T)
        temperature = torch.clamp(F.softplus(raw_temp) + self.tau_min, self.tau_min, self.tau_max)

        # Apply temperature-scaled softmax
        weights = F.softmax(raw_weights / temperature.unsqueeze(-1), dim=-1)  # (B, T, K)

        return weights, temperature

class CustomKAN(nn.Module):
    """
    Enhanced CustomKAN with improved operator diversity based on Boltzmann KAN features
    """
    def __init__(self, ops, h_dim):
        super().__init__()
        self.ops = nn.ModuleList(ops)
        self.n_ops = len(ops)

        # Enhanced weight generator based on Boltzmann KAN features
        self.weight_generator = LiquidWeightGenerator(
            h_dim=h_dim,
            n_ops=self.n_ops,
            hidden_dim=128
        )

        # Per-operator feature extraction to increase diversity
        self.op_feature_extractors = nn.ModuleList([
            nn.Sequential(
                nn.Linear(h_dim, h_dim),
                nn.ReLU(),
                nn.Linear(h_dim, h_dim)
            ) for _ in range(self.n_ops)
        ])

        # Create a parameter modulator for each operator
        self.param_modulators = nn.ModuleList()
        context_dim = h_dim  # Context dimension (based on BoltzmannKAN output)

        for i, op in enumerate(self.ops):
            # Define parameter settings according to operator type
            param_configs = self._get_param_configs_for_op(op)
            if param_configs:
                modulator = ParameterModulator(context_dim, param_configs)
                self.param_modulators.append(modulator)
                # Attach the parameter modulator to the operator
                op.set_param_modulator(modulator)
            else:
                self.param_modulators.append(None)

        # Final transformation parameters
        self.raw_gain = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))

    def _get_param_configs_for_op(self, op):
        """Return the parameter configuration for a given operator type"""
        if isinstance(op, MonotonicLinearOp):
            return {
                'scale_offset': {'dim': 1, 'scale': 0.2},
                'bias_offset': {'dim': 1, 'scale': 0.3}
            }
        elif isinstance(op, MonotonicFlatOp):
            return {
                'scale_offset': {'dim': 1, 'scale': 0.1},
                'bias_offset': {'dim': 1, 'scale': 0.2}
            }
        elif isinstance(op, ConcaveLogOp):
            return {
                'scale_offset': {'dim': 1, 'scale': 0.2},
                'eps_offset': {'dim': 1, 'scale': 0.01}
            }
        elif isinstance(op, SaturationSigmoidOp):
            return {
                'scale_offset': {'dim': 1, 'scale': 0.2},
                'slope_offset': {'dim': 1, 'scale': 0.3},
                'bias_offset': {'dim': 1, 'scale': 0.3}
            }
        elif isinstance(op, HingeReLUOp):
            return {
                'scale_offset': {'dim': 1, 'scale': 0.2},
                'threshold_offset': {'dim': 1, 'scale': 0.3}
            }
        elif isinstance(op, PolynomialOp):
            return {
                'coeff_offset': {'dim': op.deg, 'scale': 0.2}
            }
        elif isinstance(op, DampedSinOp):
            return {
                'scale_offset': {'dim': 1, 'scale': 0.2},
                'freq_offset': {'dim': 1, 'scale': 0.3},
                'lambda_offset': {'dim': 1, 'scale': 0.2},
                'phase_offset': {'dim': 1, 'scale': 0.5}
            }
        elif isinstance(op, PiecewiseLinearOp):
            return {
                'k1_offset': {'dim': 1, 'scale': 0.2},
                'k2_offset': {'dim': 1, 'scale': 0.2},
                'threshold_offset': {'dim': 1, 'scale': 0.3}
            }
        return {}

    def forward(self, h_multi):  # h_multi:(B,T,h_dim)
        B, T, _ = h_multi.shape

        # Apply per-operator feature extraction for diversity
        op_features = []
        for i, extractor in enumerate(self.op_feature_extractors):
            feat = extractor(h_multi)  # (B, T, h_dim)
            op_features.append(feat)

        # Compute operator outputs with enhanced diversity and dynamic parameters
        outs = []
        for i, op in enumerate(self.ops):
            op_out = op(op_features[i], h_multi)  # Use h_multi as context for parameter modulation
            outs.append(op_out)

        # Align lengths
        Tm = min(o.size(1) for o in outs)
        outs = [o[:, :Tm] for o in outs]
        h_multi_aligned = h_multi[:, :Tm, :]

        # Stack operator outputs
        op_stack = torch.stack(outs, dim=-1)  # (B, Tm, K)

        # Generate liquid weights based on Boltzmann KAN features
        weights, temperature = self.weight_generator(h_multi_aligned)  # (B, Tm, K)

        # Weighted combination (continuous weighting)
        damage = torch.sum(op_stack * weights, dim=-1)  # (B, Tm)
        damage = torch.clamp(damage, 0.0, 100.0)

        # Final transformation
        gain = torch.clamp(F.softplus(self.raw_gain), 0.1, 2.0)  # Reduced gain range
        bias_val = torch.clamp(self.bias, -2.0, 2.0)  # Reduced bias range
        damage = torch.clamp(gain * damage + bias_val, 0.0, 100.0)

        return damage, weights, temperature

class TrendEncoder(nn.Module):
    """
    Encoder outputs "damage", then projects to monotonic decreasing HI:
        HI is learned from sensor data without hard range constraints
    Without real HI, learns mapping from sensor data to infer health states
    Enhanced with improved Liquid Weight Generator based on Boltzmann KAN features
    """
    def __init__(self, in_ch, trend_ch=4):
        super().__init__()
        self.boltz = BoltzmannKAN(in_ch, out_ch=trend_ch)
        ops = [MonotonicLinearOp(), MonotonicFlatOp(), ConcaveLogOp(),
               SaturationSigmoidOp(), HingeReLUOp(), PolynomialOp(),
               DampedSinOp(), PiecewiseLinearOp()]

        # Enhanced CustomKAN with dynamic parameter modulation based on Boltzmann KAN features
        self.customkan = CustomKAN(ops, h_dim=trend_ch)

        # Projection parameters - learn flexible health states from sensor features
        self.proj_gain = nn.Parameter(torch.tensor(1.0))
        self.proj_bias = nn.Parameter(torch.tensor(0.0))

        # Add health-aware layer, directly infer from Boltzmann KAN features
        self.health_aware_transform = nn.Sequential(
            nn.Linear(trend_ch, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()  # Output 0-1 range health state
        )

        # Add linear constraint layer, enforce linear trend
        self.linear_enforcer = nn.Sequential(
            nn.Linear(trend_ch, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()  # Output linear decay degree
        )

    def forward(self, x):  # x:(B,T,C)
        # Feature extraction based on Boltzmann KAN
        h_multi = self.boltz(x)              # (B,T,trend_ch) >=0

        # Enhanced liquid weight generation based on Boltzmann KAN features
        damage, weights, temperature = self.customkan(h_multi)    # damage:(B,T), weights:(B,T,K), temp:(B,T)

        # Directly learn health states from Boltzmann KAN features
        # Assess health state based on Boltzmann KAN features at each time step
        B, T, C = h_multi.shape
        h_reshaped = h_multi.reshape(-1, C)  # Fix: Use reshape instead of view (B*T, C)
        health_direct = self.health_aware_transform(h_reshaped)  # (B*T, 1)
        health_direct = health_direct.view(B, T)  # (B, T)

        # Add linear constraint, force linear decay pattern
        linear_weight = self.linear_enforcer(h_reshaped).view(B, T)  # (B, T)

        # Generate ideal linear decay pattern
        t = torch.arange(T, device=x.device, dtype=x.dtype).unsqueeze(0).expand(B, -1)  # (B, T)
        t_normalized = t / (T - 1)  # Normalize to [0, 1]

        # Combine damage and direct health assessment
        g = torch.clamp(F.softplus(self.proj_gain), 0.1, 5.0)
        b = torch.clamp(self.proj_bias, -5.0, 5.0)

        # Convert damage to health state decay
        damage_normalized = torch.sigmoid(g*(damage + b))

        # Combine three health assessment methods: direct assessment, damage pattern, linear constraint
        combined_health = health_direct * (1 - 0.3 * damage_normalized)

        # Generate ideal linear decay curve (from high to low) - remove hard range constraints
        linear_decay = 1.0 - t_normalized * 0.5  # Simple linear decay from 1.0 to 0.5

        # Use linear_weight to mix linear decay and complex patterns
        # Higher linear_weight means more towards linear
        hi = linear_weight * linear_decay + (1 - linear_weight) * combined_health

        # Force stronger monotonic decreasing constraint without hard range limits
        for t_idx in range(1, T):
            hi = torch.cat([
                hi[:, :t_idx],
                torch.min(hi[:, t_idx:t_idx+1], hi[:, t_idx-1:t_idx] - 0.001),  # Force at least 0.001 decrease
                hi[:, t_idx+1:]
            ], dim=1)

        return hi, h_multi, weights, temperature

class ReconHead(nn.Module):
    """
    Recursive reconstruction: given (x_t, h_t) → predict x_{t+1}
    """
    def __init__(self, C, hidden=128):
        super().__init__()
        self.gru = nn.GRU(input_size=C+1, hidden_size=hidden, batch_first=True)
        self.out = nn.Linear(hidden, C)
    def forward(self, x_in, h_in):
        # x_in:(B,T_in,C), h_in:(B,T_in)
        B,T,C = x_in.shape
        h_in_clamped = torch.clamp(h_in, 0.0, 10.0)  # Remove upper limit constraint
        feat = torch.cat([x_in, h_in_clamped.unsqueeze(-1)], dim=-1)  # (B,T,C+1)
        H,_ = self.gru(feat)                                  # (B,T,H)
        y = self.out(H)                                       # (B,T,C) aligned predict x_{t+1}
        return y

class FlightClassifier(nn.Module):
    """
    Enhanced Health Index difference-based fault classification with attention mechanism
    """
    def __init__(self, n_classes):
        super().__init__()

        # Attention mechanism for HI sequence
        self.attention = nn.MultiheadAttention(embed_dim=1, num_heads=1, batch_first=True)

        # Enhanced feature extraction
        self.feature_extractor = nn.Sequential(
            nn.Linear(6, 128),  # Extended feature vector
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        # Final classifier
        self.classifier = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(16, n_classes)
        )

    def forward(self, hi, before_after, mask):
        """
        Args:
            hi: (B, T) health index sequence
            before_after: (B,) 0=before maintenance, 1=after maintenance
            mask: (B, T) validity mask
        Returns:
            logits: (B, n_classes) classification logits
        """
        B, T = hi.shape

        # Calculate statistics from HI with attention
        valid_counts = mask.sum(dim=1, keepdim=True).clamp_min(1.0)  # (B, 1)

        # Apply attention to HI sequence
        hi_expanded = hi.unsqueeze(-1)  # (B, T, 1)
        attn_mask = ~mask  # Fix: Remove the .bool() call since mask is already boolean
        attended_hi, _ = self.attention(hi_expanded, hi_expanded, hi_expanded,
                                      key_padding_mask=attn_mask)  # (B, T, 1)
        attended_hi = attended_hi.squeeze(-1)  # (B, T)

        # Enhanced statistics calculation
        hi_mean = (hi * mask.float()).sum(dim=1, keepdim=True) / valid_counts  # (B, 1)
        hi_std = torch.sqrt(((hi - hi_mean) ** 2 * mask.float()).sum(dim=1, keepdim=True) / valid_counts)  # (B, 1)

        # Attention-weighted statistics
        attn_weights = F.softmax(attended_hi * mask.float() + (1 - mask.float()) * (-1e9), dim=1)  # (B, T)
        hi_attn_mean = (hi * attn_weights).sum(dim=1, keepdim=True)  # (B, 1)

        # Trend analysis - early vs late period difference
        half_t = T // 2
        early_mask = mask[:, :half_t]
        late_mask = mask[:, half_t:]
        early_hi = hi[:, :half_t]
        late_hi = hi[:, half_t:]

        early_mean = (early_hi * early_mask.float()).sum(dim=1, keepdim=True) / (early_mask.float().sum(dim=1, keepdim=True) + 1e-8)
        late_mean = (late_hi * late_mask.float()).sum(dim=1, keepdim=True) / (late_mask.float().sum(dim=1, keepdim=True) + 1e-8)
        hi_trend = early_mean - late_mean  # Positive if declining health

        # Feature vector: [before_after, mean_hi, std_hi, attn_mean_hi, trend, attn_weight_entropy]
        attn_entropy = -(attn_weights * torch.log(attn_weights + 1e-8)).sum(dim=1, keepdim=True)

        features = torch.cat([
            before_after.float().unsqueeze(1),  # (B, 1)
            hi_mean,  # (B, 1)
            hi_std,   # (B, 1)
            hi_attn_mean,  # (B, 1)
            hi_trend,  # (B, 1)
            attn_entropy  # (B, 1)
        ], dim=1)  # (B, 6)

        # Extract features and classify
        feature_repr = self.feature_extractor(features)  # (B, 32)
        logits = self.classifier(feature_repr)  # (B, n_classes)

        return logits

class FlightDiffAwareReconstructor(nn.Module):
    """
    Flight data model with monotonic HI learning and classification
    """
    def __init__(self, in_ch, trend_ch=4, hidden=128, n_classes=19):
        super().__init__()
        self.encoder = TrendEncoder(in_ch, trend_ch)
        self.recon = ReconHead(in_ch, hidden)
        self.classifier = FlightClassifier(n_classes)

    def forward(self, x, mask):
        """
        Args:
            x: (B, L, C) input sequences
            mask: (B, L) validity mask
        Returns:
            hi: (B, L) health index
            y_hat: (B, L-2, C) reconstruction prediction
            h_multi: (B, L, trend_ch) intermediate features
            weights: (B, L, K) operator weights
            temperature: (B, L) temperature parameters
        """
        B, L, C = x.shape

        # Generate health index and features
        hi, h_multi, weights, temperature = self.encoder(x)  # hi:(B,L)

        # Reconstruction on 0:L-2 → 1:L-1
        if L >= 3:
            x_in = x[:, :-2, :]  # (B, L-2, C)
            h_in = hi[:, :-2]    # (B, L-2)
            y_hat = self.recon(x_in, h_in)  # (B, L-2, C)
        else:
            y_hat = torch.zeros(B, 0, C, device=x.device)

        return hi, y_hat, h_multi, weights, temperature

# ============================================================
# 3) Loss functions adapted for flight data
# ============================================================
def sanitize_tensors(*tensors):
    """Numerical stabilization"""
    sanitized = []
    for tensor in tensors:
        if tensor is None:
            sanitized.append(None)
            continue
        if isinstance(tensor, torch.Tensor):
            tensor = torch.nan_to_num(tensor, nan=0.0, posinf=1e6, neginf=-1e6)
            tensor = torch.clamp(tensor, -1e6, 1e6)
        sanitized.append(tensor)
    return sanitized if len(sanitized) > 1 else sanitized[0]

def masked_mse(pred, target, mask):
    """MSE loss with mask"""
    if pred.numel() == 0 or target.numel() == 0:
        return torch.tensor(0.0, device=pred.device)
    diff = (pred - target).pow(2)
    if mask.numel() > 0:
        masked_diff = diff * mask.unsqueeze(-1)
        loss = masked_diff.sum() / (mask.sum() * diff.size(-1) + 1e-8)
    else:
        loss = diff.mean()
    return torch.clamp(loss, 0.0, 1e6)

def slope_loss(h, mask, mode="smooth"):
    """Slope-based regularization"""
    if h.size(1) < 2:
        return torch.tensor(0.0, device=h.device)

    diff = h[:, 1:] - h[:, :-1]  # (B, T-1)
    mask_diff = mask[:, 1:].float() * mask[:, :-1].float()  # (B, T-1)

    if mode == "smooth":
        loss = (diff.abs() * mask_diff).sum() / (mask_diff.sum() + 1e-8)
    elif mode == "mono_dec":
        violation = F.relu(diff) * mask_diff  # Penalize increases
        loss = violation.sum() / (mask_diff.sum() + 1e-8)
    else:
        loss = torch.tensor(0.0, device=h.device)

    return torch.clamp(loss, 0.0, 1e6)

def enhanced_linear_slope_loss(h, mask, weight=10.0):
    """Enhanced linear slope constraint"""
    if h.size(1) < 3:
        return torch.tensor(0.0, device=h.device)

    B, T = h.shape

    # Create ideal linear decay
    t = torch.arange(T, device=h.device, dtype=h.dtype).unsqueeze(0).expand(B, -1)
    t_norm = t / (T - 1)

    # Fit linear trend to actual HI
    linear_loss = torch.tensor(0.0, device=h.device)

    for b in range(B):
        valid_indices = mask[b].nonzero(as_tuple=True)[0]
        if len(valid_indices) < 2:
            continue

        h_valid = h[b, valid_indices]
        t_valid = t_norm[b, valid_indices]

        if len(h_valid) < 2:
            continue

        # Linear regression
        x_mean = t_valid.mean()
        y_mean = h_valid.mean()

        numerator = ((t_valid - x_mean) * (h_valid - y_mean)).sum()
        denominator = ((t_valid - x_mean) ** 2).sum()

        if denominator < 1e-8:
            continue

        slope = numerator / denominator
        intercept = y_mean - slope * x_mean

        # Predicted linear values
        h_linear = slope * t_valid + intercept

        # Loss: deviation from linear trend
        deviation = (h_valid - h_linear).pow(2).mean()
        linear_loss = linear_loss + deviation

    return linear_loss / B * weight

def enhanced_liquid_weight_regularization(weights_list, mask_list, lambda_tv=0.02, lambda_ent=0.1, lambda_bal=0.05, lambda_div=0.1):
    """Enhanced liquid weight regularization"""
    total_loss = torch.tensor(0.0)
    count = 0

    for weights, mask in zip(weights_list, mask_list):
        if weights is None or mask is None:
            continue

        B, T, K = weights.shape

        # 1. Temporal variation penalty (smoothness)
        if T > 1:
            tv_loss = torch.sum(torch.abs(weights[:, 1:, :] - weights[:, :-1, :]) * mask[:, 1:].float().unsqueeze(-1))
            tv_loss = tv_loss / (mask[:, 1:].float().sum() * K + 1e-8)
            total_loss = total_loss + lambda_tv * tv_loss

        # 2. Entropy regularization (diversity)
        eps = 1e-8
        entropy = -torch.sum(weights * torch.log(weights + eps), dim=-1)  # (B, T)
        max_entropy = torch.log(torch.tensor(K, dtype=weights.dtype, device=weights.device))
        entropy_loss = torch.sum((max_entropy - entropy) * mask.float()) / (mask.float().sum() + 1e-8)
        total_loss = total_loss + lambda_ent * entropy_loss

        # 3. Balance regularization (prevent dominant operators)
        mean_weights = torch.sum(weights * mask.float().unsqueeze(-1), dim=1) / (mask.float().sum(dim=1, keepdim=True) + 1e-8)  # (B, K)
        uniform_target = 1.0 / K
        balance_loss = torch.mean((mean_weights - uniform_target).pow(2))
        total_loss = total_loss + lambda_bal * balance_loss

        # 4. Diversity regularization (encourage different operators)
        weight_corr = torch.corrcoef(mean_weights.t())
        if not torch.isnan(weight_corr).any():
            off_diag = weight_corr - torch.eye(K, device=weight_corr.device)
            diversity_loss = torch.mean(off_diag.pow(2))
            total_loss = total_loss + lambda_div * diversity_loss

        count += 1

    return total_loss / max(count, 1)

@torch.no_grad()
def eval_epoch_flight(model, loader, device, mode="full"):
    """Evaluation function for flight data with different modes"""
    model.eval()
    total = {"mse": 0.0, "acc": 0.0, "hi_mean": 0.0}
    n_batch = 0
    n_acc = 0
    n_tot = 0

    for batch in loader:
        x = batch["x"].to(device)
        mask = batch["mask"].to(device)
        labels = batch["labels"].to(device)
        before_after = batch["before_after"].to(device)
        valid_lengths = batch["valid_lengths"].to(device)

        B, L, C = x.shape

        # Generate predictions
        hi, y_hat, h_multi, weights, temperature = model(x, mask)

        # Numerical stabilization
        hi, y_hat = sanitize_tensors(hi, y_hat)

        # Reconstruction loss (if mode is "recon" or "full")
        if mode in ["recon", "full"] and y_hat.size(1) > 0 and L >= 3:
            target = x[:, 1:-1, :]  # (B, L-2, C)
            recon_mask = (torch.arange(L-2, device=device)[None, :] < (valid_lengths-2)[:, None]).float()
            mse = masked_mse(y_hat, target, recon_mask)

            if not torch.isnan(mse):
                total["mse"] += mse.item()

        # Classification (if mode is "cls" or "full")
        if mode in ["cls", "full"]:
            logits = model.classifier(hi, before_after, mask)
            pred = logits.argmax(dim=1)
            n_acc += (pred == labels).sum().item()
            n_tot += labels.numel()

        # HI statistics
        valid_counts = mask.float().sum(dim=1, keepdim=True).clamp_min(1.0)
        hi_mean = (hi * mask.float()).sum(dim=1, keepdim=True) / valid_counts
        total["hi_mean"] += hi_mean.mean().item()

        n_batch += 1

    for k in ["mse", "hi_mean"]:
        total[k] /= max(n_batch, 1)

    if mode in ["cls", "full"]:
        total["acc"] = n_acc / max(n_tot, 1)
    else:
        total["acc"] = 0.0

    return total

# ============================================================
# 4) Data loading and visualization functions adapted for flight data
# ============================================================
def load_flight_data(data_dir):
    """Load processed flight data with corrected key names"""
    train_path = os.path.join(data_dir, "train_samples_processed_sample_len_150.pkl")
    val_path = os.path.join(data_dir, "val_samples_processed_sample_len_150.pkl")
    test_path = os.path.join(data_dir, "test_samples_processed_sample_len_150.pkl")
    label_mappings_path = os.path.join(data_dir, "label_mappings_sample_len_150.pkl")

    with open(train_path, 'rb') as f:
        train_samples = pickle.load(f)
    with open(val_path, 'rb') as f:
        val_samples = pickle.load(f)
    with open(test_path, 'rb') as f:
        test_samples = pickle.load(f)
    with open(label_mappings_path, 'rb') as f:
        label_mappings = pickle.load(f)

    return train_samples, val_samples, test_samples, label_mappings

def print_data_summary(train_samples, val_samples, test_samples, label_mappings):
    """Print dataset summary"""
    print(f"Dataset Summary:")
    print(f"  Train: {len(train_samples)} samples")
    print(f"  Val: {len(val_samples)} samples")
    print(f"  Test: {len(test_samples)} samples")

    # Check the actual number of classes from label mappings
    num_classes = 0
    if 'class_to_id' in label_mappings:
        num_classes = len(label_mappings['class_to_id'])
    elif 'id_to_class' in label_mappings:
        num_classes = len(label_mappings['id_to_class'])

    print(f"  Classes: {num_classes}")
    print(f"  Features: {list(train_samples.values())[0]['x'].shape[1]}")

    # Print sample data structure
    sample = list(train_samples.values())[0]
    print(f"Sample data structure:")
    for key, value in sample.items():
        if hasattr(value, 'shape'):
            print(f"  {key}: {value.shape}")
        else:
            print(f"  {key}: {type(value)} = {value}")

@torch.no_grad()
def collect_flight_predictions(model, loader, device, max_curves=12):
    """Collect test predictions for visualization"""
    model.eval()
    y_true, y_pred = [], []
    curves = []

    for batch in loader:
        x = batch["x"].to(device)
        mask = batch["mask"].to(device)
        labels = batch["labels"].to(device)
        before_after = batch["before_after"].to(device)
        master_indices = batch["master_indices"]

        hi, y_hat, h_multi, weights, temperature = model(x, mask)
        hi, y_hat = sanitize_tensors(hi, y_hat)

        # Classification
        logits = model.classifier(hi, before_after, mask)
        prob = F.softmax(logits, dim=1)
        pred = prob.argmax(1)

        y_true.append(labels.cpu().numpy())
        y_pred.append(pred.cpu().numpy())

        # Collect sample curves
        for i in range(x.size(0)):
            if len(curves) >= max_curves:
                break

            valid_len = mask[i].sum().item()

            curves.append({
                "master_index": master_indices[i],
                "true": int(labels[i].cpu().item()),
                "pred": int(pred[i].cpu().item()),
                "prob": prob[i].cpu().numpy(),
                "hi": hi[i, :valid_len].cpu().numpy(),
                "x": x[i, :valid_len].cpu().numpy(),
                "before_after": int(before_after[i].cpu().item()),
                "weights": weights[i, :valid_len].cpu().numpy() if weights is not None else None,
                "temperature": temperature[i, :valid_len].cpu().numpy() if temperature is not None else None
            })

    y_true = np.concatenate(y_true) if y_true else np.array([])
    y_pred = np.concatenate(y_pred) if y_pred else np.array([])

    return y_true, y_pred, curves

def plot_flight_hi_examples(curves, label_mappings, n_show=6):
    """Plot health index examples for flight data"""
    if len(curves) == 0:
        print("(No visualization samples)")
        return

    random.shuffle(curves)
    n_show = min(n_show, len(curves))

    id_to_class = label_mappings.get('id_to_class', {})
    id_to_before_after = {0: 'Before', 1: 'After'}

    cols = 3
    rows = int(np.ceil(n_show / cols))
    plt.figure(figsize=(cols*4.5, rows*3.2))

    for k in range(n_show):
        ex = curves[k]
        hi = ex["hi"]

        master_idx = ex["master_index"]
        true_class = id_to_class.get(ex["true"], f"Class_{ex['true']}")
        pred_class = id_to_class.get(ex["pred"], f"Class_{ex['pred']}")
        prob = ex["prob"]
        before_after = id_to_before_after.get(ex["before_after"], "Unknown")

        plt.subplot(rows, cols, k+1)
        plt.plot(range(len(hi)), hi, linewidth=2, marker='o', markersize=2)

        title = (f"Flight {master_idx} ({before_after})\n"
                f"True: {true_class} | Pred: {pred_class}\n"
                f"Prob: {prob[ex['pred']]:.3f}")
        plt.title(title, fontsize=9)
        plt.xlabel("Time Step")
        plt.ylabel("Learned Health Index")
        plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def plot_flight_weights(curves, n_show=2):
    """Plot operator weights for flight data"""
    curves_with_weights = [c for c in curves if c.get("weights") is not None]
    if len(curves_with_weights) == 0:
        print("(No weight information available)")
        return

    random.shuffle(curves_with_weights)
    n_show = min(n_show, len(curves_with_weights))

    op_names = ["MonotonicLinear", "MonotonicFlat", "ConcaveLog", "SaturationSigmoid",
                "HingeReLU", "Polynomial", "DampedSin", "PiecewiseLinear"]

    for i in range(n_show):
        ex = curves_with_weights[i]
        weights = ex["weights"]  # (T, K)
        temperature = ex.get("temperature")

        master_idx = ex["master_index"]
        true_class = ex["true"]
        pred_class = ex["pred"]

        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        fig.suptitle(f"Flight {master_idx} - Operator Analysis (True: {true_class}, Pred: {pred_class})")

        # 1. Weights over time
        ax = axes[0]
        T, K = weights.shape
        for k in range(K):
            ax.plot(range(T), weights[:, k],
                   label=op_names[k] if k < len(op_names) else f"Op_{k}",
                   marker='o', markersize=1, linewidth=1.5)
        ax.set_title("Operator Weights")
        ax.set_xlabel("Time Step")
        ax.set_ylabel("Weight")
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.grid(True, alpha=0.3)

        # 2. Weight entropy
        ax = axes[1]
        entropy = -np.sum(weights * np.log(weights + 1e-8), axis=1)
        ax.plot(range(T), entropy, linewidth=2, color='red')
        ax.axhline(np.log(K), color='k', linestyle='--', alpha=0.7, label=f'Max Entropy ({np.log(K):.2f})')
        ax.set_title("Weight Entropy")
        ax.set_xlabel("Time Step")
        ax.set_ylabel("Entropy")
        ax.legend()
        ax.grid(True, alpha=0.3)

        # 3. Temperature (if available)
        ax = axes[2]
        if temperature is not None:
            ax.plot(range(T), temperature, linewidth=2, color='blue')
            ax.set_title("Temperature")
            ax.set_xlabel("Time Step")
            ax.set_ylabel("Temperature")
        else:
            ax.text(0.5, 0.5, "Temperature data\nnot available",
                   ha='center', va='center', transform=ax.transAxes)
            ax.set_title("Temperature")
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

# ============================================================
# 5) Three-stage training function adapted for flight data
# ============================================================
def train_flight_model_three_stage(train_samples, val_samples, test_samples, label_mappings,
                                  epochs_stage1=50, epochs_stage2=30, epochs_stage3=20,
                                  batch_size=32, lr=1e-3, device=None, patience=10):
    """Three-stage training function for flight data"""
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"Training on device: {device}")

    # Create datasets - use correct label key
    train_dataset = FlightDataset(train_samples, label_key='class')
    val_dataset = FlightDataset(val_samples, label_key='class')
    test_dataset = FlightDataset(test_samples, label_key='class')

    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=flight_collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=flight_collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=flight_collate_fn)

    # Model parameters
    C = train_dataset.C  # Number of features

    # Determine number of classes from label mappings
    if 'class_to_id' in label_mappings:
        n_classes = len(label_mappings['class_to_id'])
    elif 'id_to_class' in label_mappings:
        n_classes = len(label_mappings['id_to_class'])
    else:
        # If no mapping found, infer from data
        all_labels = set()
        for sample in train_samples.values():
            all_labels.add(sample['class'])
        n_classes = len(all_labels)

    print(f"Model config: features={C}, classes={n_classes}")

    # Create model
    model = FlightDiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=n_classes).to(device)

    model_save_path = str(resolve_data_file("flight_maintenance_3stage.pt"))

    print("\n" + "="*80)
    print("STAGE 1: RECONSTRUCTION TRAINING")
    print("="*80)

    # Stage 1: Train reconstruction (encoder + recon head)
    opt_stage1 = torch.optim.AdamW([
        {'params': model.encoder.parameters()},
        {'params': model.recon.parameters()}
    ], lr=lr, weight_decay=1e-5)

    best_recon_loss = float('inf')
    no_improve_count = 0

    for epoch in range(1, epochs_stage1 + 1):
        print(f"\n[Stage 1 - Epoch {epoch}/{epochs_stage1}] Training reconstruction...")
        model.train()

        train_loss = 0.0
        n_batches = 0

        for batch_idx, batch in enumerate(train_loader):
            if batch_idx % max(1, len(train_loader)//5) == 0:
                progress = (batch_idx + 1) / len(train_loader) * 100
                print(f"    Progress: {progress:.1f}%")

            x = batch["x"].to(device)
            mask = batch["mask"].to(device)
            valid_lengths = batch["valid_lengths"].to(device)

            B, L, C = x.shape

            # Forward pass
            hi, y_hat, h_multi, weights, temperature = model(x, mask)
            hi, y_hat = sanitize_tensors(hi, y_hat)

            # Reconstruction loss only
            recon_loss = torch.tensor(0.0, device=device)
            if y_hat.size(1) > 0 and L >= 3:
                target = x[:, 1:-1, :]
                recon_mask = (torch.arange(L-2, device=device)[None, :] < (valid_lengths-2)[:, None]).float()
                recon_loss = masked_mse(y_hat, target, recon_mask)

            # Health index regularization (light)
            hi_smooth_loss = slope_loss(hi, mask, "smooth")
            hi_mono_loss = slope_loss(hi, mask, "mono_dec")

            # Total loss
            total_loss = recon_loss + 0.01 * hi_smooth_loss + 0.02 * hi_mono_loss

            # Backward pass
            opt_stage1.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt_stage1.step()

            train_loss += total_loss.item()
            n_batches += 1

        train_loss /= n_batches

        # Validation (reconstruction only)
        val_metrics = eval_epoch_flight(model, val_loader, device, mode="recon")

        print(f"[Stage 1 - Epoch {epoch:03d}] Train: {train_loss:.4f} | Val: mse={val_metrics['mse']:.4f}")

        # Early stopping for reconstruction
        if val_metrics['mse'] < best_recon_loss:
            best_recon_loss = val_metrics['mse']
            no_improve_count = 0
            print(f"    ✓ New best reconstruction loss: {best_recon_loss:.4f}")
        else:
            no_improve_count += 1
            if no_improve_count >= patience:
                print(f"\n[Stage 1 Early Stopping] No improvement for {patience} epochs.")
                break

    print("\n" + "="*80)
    print("STAGE 2: CLASSIFICATION TRAINING (FROZEN ENCODER)")
    print("="*80)

    # Stage 2: Freeze encoder and recon, train classifier
    for param in model.encoder.parameters():
        param.requires_grad = False
    for param in model.recon.parameters():
        param.requires_grad = False

    opt_stage2 = torch.optim.AdamW(model.classifier.parameters(), lr=lr*0.5, weight_decay=1e-5)
    ce = nn.CrossEntropyLoss()

    best_cls_acc = 0.0
    no_improve_count = 0

    for epoch in range(1, epochs_stage2 + 1):
        print(f"\n[Stage 2 - Epoch {epoch}/{epochs_stage2}] Training classification...")
        model.train()

        train_loss = 0.0
        n_batches = 0

        for batch_idx, batch in enumerate(train_loader):
            if batch_idx % max(1, len(train_loader)//5) == 0:
                progress = (batch_idx + 1) / len(train_loader) * 100
                print(f"    Progress: {progress:.1f}%")

            x = batch["x"].to(device)
            mask = batch["mask"].to(device)
            labels = batch["labels"].to(device)
            before_after = batch["before_after"].to(device)

            # Forward pass (encoder frozen)
            with torch.no_grad():
                hi, _, _, _, _ = model(x, mask)
                hi = sanitize_tensors(hi)

            # Classification loss
            logits = model.classifier(hi, before_after, mask)
            cls_loss = ce(logits, labels)

            # Backward pass
            opt_stage2.zero_grad()
            cls_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.classifier.parameters(), 1.0)
            opt_stage2.step()

            train_loss += cls_loss.item()
            n_batches += 1

        train_loss /= n_batches

        # Validation (classification only)
        val_metrics = eval_epoch_flight(model, val_loader, device, mode="cls")

        print(f"[Stage 2 - Epoch {epoch:03d}] Train: {train_loss:.4f} | Val: acc={val_metrics['acc']:.3f}")

        # Early stopping for classification
        if val_metrics['acc'] > best_cls_acc:
            best_cls_acc = val_metrics['acc']
            no_improve_count = 0
            print(f"    ✓ New best classification accuracy: {best_cls_acc:.3f}")
        else:
            no_improve_count += 1
            if no_improve_count >= patience:
                print(f"\n[Stage 2 Early Stopping] No improvement for {patience} epochs.")
                break

    print("\n" + "="*80)
    print("STAGE 3: JOINT FINE-TUNING")
    print("="*80)

    # Stage 3: Unfreeze all parameters and fine-tune
    for param in model.parameters():
        param.requires_grad = True

    opt_stage3 = torch.optim.AdamW(model.parameters(), lr=lr*0.1, weight_decay=1e-5)

    best_combined_metric = float('inf')
    best_model = None
    no_improve_count = 0

    for epoch in range(1, epochs_stage3 + 1):
        print(f"\n[Stage 3 - Epoch {epoch}/{epochs_stage3}] Fine-tuning...")
        model.train()

        train_loss = 0.0
        train_recon_loss = 0.0
        train_cls_loss = 0.0
        n_batches = 0

        for batch_idx, batch in enumerate(train_loader):
            if batch_idx % max(1, len(train_loader)//5) == 0:
                progress = (batch_idx + 1) / len(train_loader) * 100
                print(f"    Progress: {progress:.1f}%")

            x = batch["x"].to(device)
            mask = batch["mask"].to(device)
            labels = batch["labels"].to(device)
            before_after = batch["before_after"].to(device)
            valid_lengths = batch["valid_lengths"].to(device)

            B, L, C = x.shape

            # Forward pass
            hi, y_hat, h_multi, weights, temperature = model(x, mask)
            hi, y_hat = sanitize_tensors(hi, y_hat)

            # Reconstruction loss
            recon_loss = torch.tensor(0.0, device=device)
            if y_hat.size(1) > 0 and L >= 3:
                target = x[:, 1:-1, :]
                recon_mask = (torch.arange(L-2, device=device)[None, :] < (valid_lengths-2)[:, None]).float()
                recon_loss = masked_mse(y_hat, target, recon_mask)

            # Classification loss
            logits = model.classifier(hi, before_after, mask)
            cls_loss = ce(logits, labels)

            # Light regularization
            hi_smooth_loss = slope_loss(hi, mask, "smooth")

            # Combined loss with balanced weights
            total_loss = 0.3 * recon_loss + 1.0 * cls_loss + 0.01 * hi_smooth_loss

            # Backward pass
            opt_stage3.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt_stage3.step()

            train_loss += total_loss.item()
            train_recon_loss += recon_loss.item()
            train_cls_loss += cls_loss.item()
            n_batches += 1

        train_loss /= n_batches
        train_recon_loss /= n_batches
        train_cls_loss /= n_batches

        # Full validation
        val_metrics = eval_epoch_flight(model, val_loader, device, mode="full")

        print(f"[Stage 3 - Epoch {epoch:03d}] Train: total={train_loss:.4f} recon={train_recon_loss:.4f} "
              f"cls={train_cls_loss:.4f} | Val: mse={val_metrics['mse']:.4f} acc={val_metrics['acc']:.3f}")

        # Combined metric for early stopping
        combined_metric = 2.0 * (1 - val_metrics['acc']) + 0.3 * val_metrics['mse']
        if combined_metric < best_combined_metric:
            best_combined_metric = combined_metric
            best_model = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve_count = 0

            # Save best model
            try:
                torch.save(model.state_dict(), model_save_path)
                print(f"    ✓ New best model saved (combined_metric: {combined_metric:.4f})")
            except Exception as e:
                print(f"    ✓ New best model (combined_metric: {combined_metric:.4f}) - Save failed: {e}")
        else:
            no_improve_count += 1
            if no_improve_count >= patience:
                print(f"\n[Stage 3 Early Stopping] No improvement for {patience} epochs.")
                break

    # Load best model
    if best_model is not None:
        model.load_state_dict(best_model)
        print(f"\n[Best Model] Combined metric: {best_combined_metric:.4f}")
        print(f"[Best Model] Saved at: {model_save_path}")

    # Final test evaluation
    print("\n" + "="*80)
    print("FINAL TEST SET EVALUATION")
    print("="*80)
    test_metrics = eval_epoch_flight(model, test_loader, device, mode="full")
    print(f"[Test] mse={test_metrics['mse']:.4f} acc={test_metrics['acc']:.3f} hi_mean={test_metrics['hi_mean']:.3f}")

    # Collect predictions for analysis
    y_true, y_pred, curves = collect_flight_predictions(model, test_loader, device)

    # Classification report
    if len(y_true) > 0:
        from sklearn.metrics import classification_report, confusion_matrix

        # Fix: Ensure class names are strings for sklearn
        class_names = []
        id_to_class = label_mappings.get('id_to_class', {})
        for i in range(n_classes):
            class_name = id_to_class.get(i, f"Class_{i}")
            class_names.append(str(class_name))  # Convert to string

        print("\n[Classification Report]")
        try:
            print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
        except Exception as e:
            print(f"Classification report error: {e}")
            print(classification_report(y_true, y_pred, zero_division=0))

        print("\n[Confusion Matrix]")
        print(confusion_matrix(y_true, y_pred))

        # Visualizations
        print("\n[Visualization] Health index curves...")
        plot_flight_hi_examples(curves, label_mappings, n_show=6)

        print("\n[Visualization] Operator weights...")
        plot_flight_weights(curves, n_show=2)

    return model, (y_true, y_pred, curves)

# Load flight data
data_dir = str(DATA_DIR)
train_samples, val_samples, test_samples, label_mappings = load_flight_data(data_dir)

print_data_summary(train_samples, val_samples, test_samples, label_mappings)

# Train model with three-stage approach
model, results = train_flight_model_three_stage(
    train_samples=train_samples,
    val_samples=val_samples,
    test_samples=test_samples,
    label_mappings=label_mappings,
    epochs_stage1=50,   # Reconstruction training
    epochs_stage2=30,   # Classification training (frozen encoder)
    epochs_stage3=20,   # Joint fine-tuning
    batch_size=640,
    lr=1e-3,
    device=None,
    patience=10
)

print("\nThree-stage training completed! Features:")
print("- Stage 1: Reconstruction training (encoder + recon head)")
print("- Stage 2: Classification training (frozen encoder)")
print("- Stage 3: Joint fine-tuning (all parameters)")
print("- Feature fusion based on Boltzmann KAN output")
print("- Enhanced Liquid Weight Generator from Boltzmann features")


In [ ]:
# ============================================================
# Load saved model and continue with Stage 2 training
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import os
import random
import matplotlib.pyplot as plt
import pickle

# ============================================================
# 1) Dataset: read samples from flight data dictionaries
# ============================================================
class FlightDataset(Dataset):
    """
    Each sample contains:
      x: (L,C) feature sequence
      mask: (L,) validity mask for padded sequences
      valid_length: actual sequence length
      label: encoded label for classification
      class_encoded: encoded main class
      before_after_encoded: 0 for before maintenance, 1 for after maintenance
    """
    def __init__(self, samples_dict, label_key='class'):
        self.samples = list(samples_dict.values())
        self.label_key = label_key

        # Extract feature dimension from first sample
        if len(self.samples) > 0:
            self.C = self.samples[0]['x'].shape[1]
        else:
            raise ValueError("Empty dataset")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        return {
            "x": torch.from_numpy(sample['x']).float(),  # (L,C)
            "mask": torch.from_numpy(sample['mask']).bool(),  # (L,)
            "valid_length": torch.tensor(sample['valid_length'], dtype=torch.long),
            "label": torch.tensor(sample[self.label_key], dtype=torch.long),
            "before_after": torch.tensor(sample['before_after'], dtype=torch.long),
            "master_index": sample['master_index'],
            "is_padded": sample['is_padded']
        }

def flight_collate_fn(batch):
    """
    Collate function for flight data - sequences are already padded to same length
    """
    batch_size = len(batch)
    if batch_size == 0:
        return {}

    # All sequences should have the same length after windowing
    x = torch.stack([b["x"] for b in batch], 0)  # (B,L,C)
    mask = torch.stack([b["mask"] for b in batch], 0)  # (B,L)
    valid_lengths = torch.stack([b["valid_length"] for b in batch], 0)  # (B,)
    labels = torch.stack([b["label"] for b in batch], 0)  # (B,)
    before_after = torch.stack([b["before_after"] for b in batch], 0)  # (B,)
    master_indices = [b["master_index"] for b in batch]

    return {
        "x": x,
        "mask": mask,
        "valid_lengths": valid_lengths,
        "labels": labels,
        "before_after": before_after,
        "master_indices": master_indices
    }

# ============================================================
# 2) Model: encode "damage" → project to monotonic decreasing HI → recursive reconstruction → classify
# ============================================================
class BoltzmannKAN(nn.Module):
    def __init__(self, in_ch, out_ch=4):
        super().__init__()
        self.E  = nn.Parameter(torch.zeros(out_ch, in_ch))
        self.kT = nn.Parameter(torch.ones(out_ch, in_ch))
    def forward(self, x):
        B,T,C = x.shape
        kT = torch.clamp(F.softplus(self.kT), 0.01, 10.0).unsqueeze(0).unsqueeze(2)  # (1,out_ch,1,in_ch)
        E  = torch.clamp(self.E, -10.0, 10.0).unsqueeze(0).unsqueeze(2)             # (1,out_ch,1,in_ch)
        x_ = torch.clamp(x.unsqueeze(1), -10.0, 10.0)                               # (B, out_ch, T, in_ch)
        exp = torch.clamp((E - x_) / kT, -50, 50)
        w   = torch.sigmoid(exp)
        h   = (w * x_).sum(dim=3).permute(0, 2, 1)           # (B,T,out_ch) >=0
        return torch.clamp(F.softplus(h), 0.0, 100.0)

class BaseOp(nn.Module):
    """
    Base operator class with support for dynamic parameter modulation
    """
    def __init__(self):
        super().__init__()
        self.param_modulator = None

    def set_param_modulator(self, modulator):
        """Set the parameter modulator"""
        self.param_modulator = modulator

    def get_modulated_params(self, context):
        """Get parameters after context-based modulation"""
        if self.param_modulator is None:
            return {}
        return self.param_modulator(context)

class MonotonicLinearOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0

    def forward(self, h, context=None):
        # Retrieve the modulated parameters
        modulated_params = self.get_modulated_params(context) if context is not None else {}

        # Base parameters
        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_bias = self.bias

        # Parameter modulation
        scale_offset = modulated_params.get('scale_offset', 0.0)
        bias_offset = modulated_params.get('bias_offset', 0.0)

        # Modulated parameters
        scale = torch.clamp(base_scale + scale_offset, self.smin, self.smax)
        bias = torch.clamp(base_bias + bias_offset, -5.0, 5.0)

        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * (xm + bias)), 0.0, 100.0)

class MonotonicFlatOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 1e-3, 1.0

    def _cum(self, x):
        # Fix: ensure computations are carried out along the time dimension
        B, T = x.shape  # xshould have shape (B, T)
        diff = F.relu(x[:, 1:] - x[:, :-1])  # (B, T-1)
        zero_init = torch.zeros(B, 1, device=x.device, dtype=x.dtype)  # (B, 1)
        return torch.cat([zero_init, torch.cumsum(diff, dim=1)], dim=1)  # (B, T)

    def forward(self, h, context=None):
        modulated_params = self.get_modulated_params(context) if context is not None else {}

        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_bias = self.bias

        scale_offset = modulated_params.get('scale_offset', 0.0)
        bias_offset = modulated_params.get('bias_offset', 0.0)

        scale = torch.clamp(base_scale + scale_offset, self.smin, self.smax)
        bias = torch.clamp(base_bias + bias_offset, -2.0, 2.0)

        xm = torch.clamp(h.mean(-1), -10.0, 10.0)  # (B, T)
        cum_result = self._cum(xm)  # (B, T)
        return torch.clamp(F.softplus(scale * (cum_result + bias)), 0.0, 100.0)

class ConcaveLogOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.eps = 1e-3
        self.smin, self.smax = 0.01, 5.0

    def forward(self, h, context=None):
        modulated_params = self.get_modulated_params(context) if context is not None else {}

        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        scale_offset = modulated_params.get('scale_offset', 0.0)
        eps_offset = modulated_params.get('eps_offset', 0.0)

        scale = torch.clamp(base_scale + scale_offset, self.smin, self.smax)
        eps = torch.clamp(self.eps + eps_offset, 1e-6, 1e-1)

        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * torch.log(torch.abs(xm) + eps)), 0.0, 100.0)

class SaturationSigmoidOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.raw_slope = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
        self.lmin, self.lmax = 0.1, 5.0

    def forward(self, h, context=None):
        modulated_params = self.get_modulated_params(context) if context is not None else {}

        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_slope = torch.clamp(F.softplus(self.raw_slope), self.lmin, self.lmax)
        base_bias = self.bias

        scale_offset = modulated_params.get('scale_offset', 0.0)
        slope_offset = modulated_params.get('slope_offset', 0.0)
        bias_offset = modulated_params.get('bias_offset', 0.0)

        scale = torch.clamp(base_scale + scale_offset, self.smin, self.smax)
        slope = torch.clamp(base_slope + slope_offset, self.lmin, self.lmax)
        bias = torch.clamp(base_bias + bias_offset, -3.0, 3.0)

        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * torch.sigmoid(slope * (xm - bias))), 0.0, 100.0)

class HingeReLUOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.threshold = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0

    def forward(self, h, context=None):
        modulated_params = self.get_modulated_params(context) if context is not None else {}

        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_threshold = self.threshold

        scale_offset = modulated_params.get('scale_offset', 0.0)
        threshold_offset = modulated_params.get('threshold_offset', 0.0)

        scale = torch.clamp(base_scale + scale_offset, self.smin, self.smax)
        threshold = torch.clamp(base_threshold + threshold_offset, -3.0, 3.0)

        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        return torch.clamp(F.softplus(scale * F.relu(xm - threshold)), 0.0, 100.0)

class PolynomialOp(BaseOp):
    def __init__(self, deg=3):
        super().__init__()
        self.raw_coeff = nn.Parameter(torch.zeros(deg))
        self.deg = deg

    def forward(self, h, context=None):
        modulated_params = self.get_modulated_params(context) if context is not None else {}

        base_coeff = torch.clamp(F.softplus(self.raw_coeff), 0.01, 5.0)

        # Fix: Use broadcasting compatible approach for batch dimension
        xm = torch.clamp(h.mean(-1), -5.0, 5.0)  # (B, T)
        B, T = xm.shape

        # Handle coefficient offsets properly for broadcasting
        coeff_offset = modulated_params.get('coeff_offset', torch.zeros_like(base_coeff))
        if coeff_offset.dim() > 1:
            # If coeff_offset is (B, deg), take mean over batch for simplicity
            coeff_offset = coeff_offset.mean(0)

        coeff = torch.clamp(base_coeff + coeff_offset, 0.01, 5.0)  # (deg,)

        y = torch.zeros_like(xm)  # (B, T)
        for i in range(self.deg):
            # Broadcast coefficient to match batch dimension
            power_term = torch.clamp(xm ** (i + 1), -100.0, 100.0)  # (B, T)
            y = y + coeff[i] * power_term  # (B, T) + scalar * (B, T)
        return torch.clamp(F.softplus(y), 0.0, 100.0)

class DampedSinOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_scale = nn.Parameter(torch.tensor(0.0))
        self.raw_freq = nn.Parameter(torch.tensor(0.0))
        self.raw_lambda = nn.Parameter(torch.tensor(0.0))
        self.phase = nn.Parameter(torch.tensor(0.0))
        self.smin, self.smax = 0.01, 5.0
        self.fmin, self.fmax = 0.1, 5.0
        self.lmin, self.lmax = 0.01, 3.0

    def forward(self, h, context=None):
        modulated_params = self.get_modulated_params(context) if context is not None else {}

        base_scale = torch.clamp(F.softplus(self.raw_scale), self.smin, self.smax)
        base_freq = torch.clamp(F.softplus(self.raw_freq), self.fmin, self.fmax)
        base_lambda = torch.clamp(F.softplus(self.raw_lambda), self.lmin, self.lmax)
        base_phase = torch.clamp(self.phase, -10.0, 10.0)

        scale_offset = modulated_params.get('scale_offset', 0.0)
        freq_offset = modulated_params.get('freq_offset', 0.0)
        lambda_offset = modulated_params.get('lambda_offset', 0.0)
        phase_offset = modulated_params.get('phase_offset', 0.0)

        scale = torch.clamp(base_scale + scale_offset, self.smin, self.smax)
        freq = torch.clamp(base_freq + freq_offset, self.fmin, self.fmax)
        lam = torch.clamp(base_lambda + lambda_offset, self.lmin, self.lmax)
        phase = torch.clamp(base_phase + phase_offset, -10.0, 10.0)

        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        damp = 1 / (1 + lam * torch.abs(xm))
        return torch.clamp(F.softplus(scale * damp * (torch.sin(freq * xm + phase) + 1)), 0.0, 100.0)

class PiecewiseLinearOp(BaseOp):
    def __init__(self):
        super().__init__()
        self.raw_k1 = nn.Parameter(torch.tensor(0.0))
        self.raw_k2 = nn.Parameter(torch.tensor(0.0))
        self.threshold = nn.Parameter(torch.tensor(0.0))
        self.kmin, self.kmax = 0.01, 5.0

    def forward(self, h, context=None):
        modulated_params = self.get_modulated_params(context) if context is not None else {}

        base_k1 = torch.clamp(F.softplus(self.raw_k1), self.kmin, self.kmax)
        base_k2 = torch.clamp(F.softplus(self.raw_k2), self.kmin, self.kmax)
        base_threshold = torch.clamp(self.threshold, -5.0, 5.0)

        k1_offset = modulated_params.get('k1_offset', 0.0)
        k2_offset = modulated_params.get('k2_offset', 0.0)
        threshold_offset = modulated_params.get('threshold_offset', 0.0)

        k1 = torch.clamp(base_k1 + k1_offset, self.kmin, self.kmax)
        k2 = torch.clamp(base_k2 + k2_offset, self.kmin, self.kmax)
        threshold = torch.clamp(base_threshold + threshold_offset, -5.0, 5.0)

        xm = torch.clamp(h.mean(-1), -10.0, 10.0)
        left = k1 * xm
        right = k1 * threshold + k2 * (xm - threshold)
        out = torch.where(xm <= threshold, left, right)
        return torch.clamp(F.softplus(out), 0.0, 100.0)

class ParameterModulator(nn.Module):
    """
    Parameter modulator:dynamically adjust operator parameters based on input context
    """
    def __init__(self, context_dim, param_configs):
        super().__init__()
        self.param_configs = param_configs

        # Create a prediction network for each parameter
        self.param_predictors = nn.ModuleDict()
        for param_name, param_info in param_configs.items():
            param_dim = param_info.get('dim', 1)
            self.param_predictors[param_name] = nn.Sequential(
                nn.Linear(context_dim, 64),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Linear(32, param_dim),
                nn.Tanh()  # Output range [-1, 1]
            )

    def forward(self, context):
        """
        Args:
            context: (B, T, context_dim) context features
        Returns:
            dict: modulated parameter offsets
        """
        # Average across the time dimension to obtain a global context
        global_context = context.mean(dim=1)  # (B, context_dim)

        modulated_params = {}
        for param_name, predictor in self.param_predictors.items():
            param_info = self.param_configs[param_name]
            raw_offset = predictor(global_context)  # (B, param_dim)

            # Scale offsets according to the configuration
            scale = param_info.get('scale', 0.1)
            modulated_params[param_name] = raw_offset * scale

        return modulated_params

class LiquidWeightGenerator(nn.Module):
    """
    Improved Liquid Weight Generator based on Boltzmann KAN features
    """
    def __init__(self, h_dim, n_ops, hidden_dim=64, tau_min=1.0, tau_max=3.0):
        super().__init__()
        self.n_ops = n_ops
        self.tau_min, self.tau_max = tau_min, tau_max

        # Feature encoder for Boltzmann KAN output
        self.feature_encoder = nn.Sequential(
            nn.Linear(h_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim)
        )

        # Add temporal encoding
        self.temporal_encoder = nn.Sequential(
            nn.Linear(2, hidden_dim // 4),  # t, dt
            nn.ReLU()
        )

        # Combined feature processing
        combined_dim = hidden_dim + hidden_dim // 4
        self.feature_fusion = nn.Sequential(
            nn.Linear(combined_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim)
        )

        # Weight prediction with explicit per-operator branches
        self.op_branches = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim // 4),
                nn.ReLU(),
                nn.Linear(hidden_dim // 4, 1)
            ) for _ in range(n_ops)
        ])

        # Temperature predictor
        self.temp_predictor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 4),
            nn.ReLU(),
            nn.Linear(hidden_dim // 4, 1)
        )

        # Initialize to encourage diversity
        for branch in self.op_branches:
            nn.init.xavier_uniform_(branch[0].weight)
            nn.init.xavier_uniform_(branch[2].weight)

    def forward(self, h_multi):
        """
        Args:
            h_multi: (B, T, h_dim) - Boltzmann KAN output
        Returns:
            weights: (B, T, K) - Per-timestep operator weights
            temperature: (B, T) - Temperature parameters
        """
        B, T, h_dim = h_multi.shape

        # Process Boltzmann KAN features
        features = self.feature_encoder(h_multi)  # (B, T, hidden_dim)

        # Add temporal encoding
        t_normalized = torch.arange(T, device=h_multi.device, dtype=h_multi.dtype).unsqueeze(0).expand(B, -1) / max(T-1, 1)
        dt = torch.zeros_like(t_normalized)
        dt[:, 1:] = t_normalized[:, 1:] - t_normalized[:, :-1]

        temporal_input = torch.stack([t_normalized, dt], dim=-1)  # (B, T, 2)
        temporal_features = self.temporal_encoder(temporal_input)  # (B, T, hidden_dim//4)

        # Fuse features
        combined = torch.cat([features, temporal_features], dim=-1)
        fused_features = self.feature_fusion(combined)  # (B, T, hidden_dim)

        # Generate per-operator logits using separate branches
        raw_logits = []
        for i, branch in enumerate(self.op_branches):
            logit = branch(fused_features).squeeze(-1)  # (B, T)
            raw_logits.append(logit)
        raw_weights = torch.stack(raw_logits, dim=-1)  # (B, T, K)

        # Zero-mean normalization to prevent systematic bias
        raw_weights = raw_weights - raw_weights.mean(dim=-1, keepdim=True)

        # Predict temperature
        raw_temp = self.temp_predictor(fused_features).squeeze(-1)  # (B, T)
        temperature = torch.clamp(F.softplus(raw_temp) + self.tau_min, self.tau_min, self.tau_max)

        # Apply temperature-scaled softmax
        weights = F.softmax(raw_weights / temperature.unsqueeze(-1), dim=-1)  # (B, T, K)

        return weights, temperature

class CustomKAN(nn.Module):
    """
    Enhanced CustomKAN with improved operator diversity based on Boltzmann KAN features
    """
    def __init__(self, ops, h_dim):
        super().__init__()
        self.ops = nn.ModuleList(ops)
        self.n_ops = len(ops)

        # Enhanced weight generator based on Boltzmann KAN features
        self.weight_generator = LiquidWeightGenerator(
            h_dim=h_dim,
            n_ops=self.n_ops,
            hidden_dim=128
        )

        # Per-operator feature extraction to increase diversity
        self.op_feature_extractors = nn.ModuleList([
            nn.Sequential(
                nn.Linear(h_dim, h_dim),
                nn.ReLU(),
                nn.Linear(h_dim, h_dim)
            ) for _ in range(self.n_ops)
        ])

        # Create a parameter modulator for each operator
        self.param_modulators = nn.ModuleList()
        context_dim = h_dim  # Context dimension (based on BoltzmannKAN output)

        for i, op in enumerate(self.ops):
            # Define parameter settings according to operator type
            param_configs = self._get_param_configs_for_op(op)
            if param_configs:
                modulator = ParameterModulator(context_dim, param_configs)
                self.param_modulators.append(modulator)
                # Attach the parameter modulator to the operator
                op.set_param_modulator(modulator)
            else:
                self.param_modulators.append(None)

        # Final transformation parameters
        self.raw_gain = nn.Parameter(torch.tensor(0.0))
        self.bias = nn.Parameter(torch.tensor(0.0))

    def _get_param_configs_for_op(self, op):
        """Return the parameter configuration for a given operator type"""
        if isinstance(op, MonotonicLinearOp):
            return {
                'scale_offset': {'dim': 1, 'scale': 0.2},
                'bias_offset': {'dim': 1, 'scale': 0.3}
            }
        elif isinstance(op, MonotonicFlatOp):
            return {
                'scale_offset': {'dim': 1, 'scale': 0.1},
                'bias_offset': {'dim': 1, 'scale': 0.2}
            }
        elif isinstance(op, ConcaveLogOp):
            return {
                'scale_offset': {'dim': 1, 'scale': 0.2},
                'eps_offset': {'dim': 1, 'scale': 0.01}
            }
        elif isinstance(op, SaturationSigmoidOp):
            return {
                'scale_offset': {'dim': 1, 'scale': 0.2},
                'slope_offset': {'dim': 1, 'scale': 0.3},
                'bias_offset': {'dim': 1, 'scale': 0.3}
            }
        elif isinstance(op, HingeReLUOp):
            return {
                'scale_offset': {'dim': 1, 'scale': 0.2},
                'threshold_offset': {'dim': 1, 'scale': 0.3}
            }
        elif isinstance(op, PolynomialOp):
            return {
                'coeff_offset': {'dim': op.deg, 'scale': 0.2}
            }
        elif isinstance(op, DampedSinOp):
            return {
                'scale_offset': {'dim': 1, 'scale': 0.2},
                'freq_offset': {'dim': 1, 'scale': 0.3},
                'lambda_offset': {'dim': 1, 'scale': 0.2},
                'phase_offset': {'dim': 1, 'scale': 0.5}
            }
        elif isinstance(op, PiecewiseLinearOp):
            return {
                'k1_offset': {'dim': 1, 'scale': 0.2},
                'k2_offset': {'dim': 1, 'scale': 0.2},
                'threshold_offset': {'dim': 1, 'scale': 0.3}
            }
        return {}

    def forward(self, h_multi):  # h_multi:(B,T,h_dim)
        B, T, _ = h_multi.shape

        # Apply per-operator feature extraction for diversity
        op_features = []
        for i, extractor in enumerate(self.op_feature_extractors):
            feat = extractor(h_multi)  # (B, T, h_dim)
            op_features.append(feat)

        # Compute operator outputs with enhanced diversity and dynamic parameters
        outs = []
        for i, op in enumerate(self.ops):
            op_out = op(op_features[i], h_multi)  # Use h_multi as context for parameter modulation
            outs.append(op_out)

        # Align lengths
        Tm = min(o.size(1) for o in outs)
        outs = [o[:, :Tm] for o in outs]
        h_multi_aligned = h_multi[:, :Tm, :]

        # Stack operator outputs
        op_stack = torch.stack(outs, dim=-1)  # (B, Tm, K)

        # Generate liquid weights based on Boltzmann KAN features
        weights, temperature = self.weight_generator(h_multi_aligned)  # (B, Tm, K)

        # Weighted combination (continuous weighting)
        damage = torch.sum(op_stack * weights, dim=-1)  # (B, Tm)
        damage = torch.clamp(damage, 0.0, 100.0)

        # Final transformation
        gain = torch.clamp(F.softplus(self.raw_gain), 0.1, 2.0)  # Reduced gain range
        bias_val = torch.clamp(self.bias, -2.0, 2.0)  # Reduced bias range
        damage = torch.clamp(gain * damage + bias_val, 0.0, 100.0)

        return damage, weights, temperature

class TrendEncoder(nn.Module):
    """
    Encoder outputs "damage", then projects to monotonic decreasing HI:
        HI is learned from sensor data without hard range constraints
    Without real HI, learns mapping from sensor data to infer health states
    Enhanced with improved Liquid Weight Generator based on Boltzmann KAN features
    """
    def __init__(self, in_ch, trend_ch=4):
        super().__init__()
        self.boltz = BoltzmannKAN(in_ch, out_ch=trend_ch)
        ops = [MonotonicLinearOp(), MonotonicFlatOp(), ConcaveLogOp(),
               SaturationSigmoidOp(), HingeReLUOp(), PolynomialOp(),
               DampedSinOp(), PiecewiseLinearOp()]

        # Enhanced CustomKAN with dynamic parameter modulation based on Boltzmann KAN features
        self.customkan = CustomKAN(ops, h_dim=trend_ch)

        # Projection parameters - learn flexible health states from sensor features
        self.proj_gain = nn.Parameter(torch.tensor(1.0))
        self.proj_bias = nn.Parameter(torch.tensor(0.0))

        # Add health-aware layer, directly infer from Boltzmann KAN features
        self.health_aware_transform = nn.Sequential(
            nn.Linear(trend_ch, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()  # Output 0-1 range health state
        )

        # Add linear constraint layer, enforce linear trend
        self.linear_enforcer = nn.Sequential(
            nn.Linear(trend_ch, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()  # Output linear decay degree
        )

    def forward(self, x):  # x:(B,T,C)
        # Feature extraction based on Boltzmann KAN
        h_multi = self.boltz(x)              # (B,T,trend_ch) >=0

        # Enhanced liquid weight generation based on Boltzmann KAN features
        damage, weights, temperature = self.customkan(h_multi)    # damage:(B,T), weights:(B,T,K), temp:(B,T)

        # Directly learn health states from Boltzmann KAN features
        # Assess health state based on Boltzmann KAN features at each time step
        B, T, C = h_multi.shape
        h_reshaped = h_multi.reshape(-1, C)  # Fix: Use reshape instead of view (B*T, C)
        health_direct = self.health_aware_transform(h_reshaped)  # (B*T, 1)
        health_direct = health_direct.view(B, T)  # (B, T)

        # Add linear constraint, force linear decay pattern
        linear_weight = self.linear_enforcer(h_reshaped).view(B, T)  # (B, T)

        # Generate ideal linear decay pattern
        t = torch.arange(T, device=x.device, dtype=x.dtype).unsqueeze(0).expand(B, -1)  # (B, T)
        t_normalized = t / (T - 1)  # Normalize to [0, 1]

        # Combine damage and direct health assessment
        g = torch.clamp(F.softplus(self.proj_gain), 0.1, 5.0)
        b = torch.clamp(self.proj_bias, -5.0, 5.0)

        # Convert damage to health state decay
        damage_normalized = torch.sigmoid(g*(damage + b))

        # Combine three health assessment methods: direct assessment, damage pattern, linear constraint
        combined_health = health_direct * (1 - 0.3 * damage_normalized)

        # Generate ideal linear decay curve (from high to low) - remove hard range constraints
        linear_decay = 1.0 - t_normalized * 0.5  # Simple linear decay from 1.0 to 0.5

        # Use linear_weight to mix linear decay and complex patterns
        # Higher linear_weight means more towards linear
        hi = linear_weight * linear_decay + (1 - linear_weight) * combined_health

        # Force stronger monotonic decreasing constraint without hard range limits
        for t_idx in range(1, T):
            hi = torch.cat([
                hi[:, :t_idx],
                torch.min(hi[:, t_idx:t_idx+1], hi[:, t_idx-1:t_idx] - 0.001),  # Force at least 0.001 decrease
                hi[:, t_idx+1:]
            ], dim=1)

        return hi, h_multi, weights, temperature

class ReconHead(nn.Module):
    """
    Recursive reconstruction: given (x_t, h_t) → predict x_{t+1}
    """
    def __init__(self, C, hidden=128):
        super().__init__()
        self.gru = nn.GRU(input_size=C+1, hidden_size=hidden, batch_first=True)
        self.out = nn.Linear(hidden, C)
    def forward(self, x_in, h_in):
        # x_in:(B,T_in,C), h_in:(B,T_in)
        B,T,C = x_in.shape
        h_in_clamped = torch.clamp(h_in, 0.0, 10.0)  # Remove upper limit constraint
        feat = torch.cat([x_in, h_in_clamped.unsqueeze(-1)], dim=-1)  # (B,T,C+1)
        H,_ = self.gru(feat)                                  # (B,T,H)
        y = self.out(H)                                       # (B,T,C) aligned predict x_{t+1}
        return y

class FlightClassifier(nn.Module):
    """
    Enhanced Health Index difference-based fault classification with attention mechanism
    """
    def __init__(self, n_classes):
        super().__init__()

        # Attention mechanism for HI sequence
        self.attention = nn.MultiheadAttention(embed_dim=1, num_heads=1, batch_first=True)

        # Enhanced feature extraction
        self.feature_extractor = nn.Sequential(
            nn.Linear(6, 128),  # Extended feature vector
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        # Final classifier
        self.classifier = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(16, n_classes)
        )

    def forward(self, hi, before_after, mask):
        """
        Args:
            hi: (B, T) health index sequence
            before_after: (B,) 0=before maintenance, 1=after maintenance
            mask: (B, T) validity mask
        Returns:
            logits: (B, n_classes) classification logits
        """
        B, T = hi.shape

        # Calculate statistics from HI with attention
        valid_counts = mask.sum(dim=1, keepdim=True).clamp_min(1.0)  # (B, 1)

        # Apply attention to HI sequence
        hi_expanded = hi.unsqueeze(-1)  # (B, T, 1)
        attn_mask = ~mask  # Fix: Remove the .bool() call since mask is already boolean
        attended_hi, _ = self.attention(hi_expanded, hi_expanded, hi_expanded,
                                      key_padding_mask=attn_mask)  # (B, T, 1)
        attended_hi = attended_hi.squeeze(-1)  # (B, T)

        # Enhanced statistics calculation
        hi_mean = (hi * mask.float()).sum(dim=1, keepdim=True) / valid_counts  # (B, 1)
        hi_std = torch.sqrt(((hi - hi_mean) ** 2 * mask.float()).sum(dim=1, keepdim=True) / valid_counts)  # (B, 1)

        # Attention-weighted statistics
        attn_weights = F.softmax(attended_hi * mask.float() + (1 - mask.float()) * (-1e9), dim=1)  # (B, T)
        hi_attn_mean = (hi * attn_weights).sum(dim=1, keepdim=True)  # (B, 1)

        # Trend analysis - early vs late period difference
        half_t = T // 2
        early_mask = mask[:, :half_t]
        late_mask = mask[:, half_t:]
        early_hi = hi[:, :half_t]
        late_hi = hi[:, half_t:]

        early_mean = (early_hi * early_mask.float()).sum(dim=1, keepdim=True) / (early_mask.float().sum(dim=1, keepdim=True) + 1e-8)
        late_mean = (late_hi * late_mask.float()).sum(dim=1, keepdim=True) / (late_mask.float().sum(dim=1, keepdim=True) + 1e-8)
        hi_trend = early_mean - late_mean  # Positive if declining health

        # Feature vector: [before_after, mean_hi, std_hi, attn_mean_hi, trend, attn_weight_entropy]
        attn_entropy = -(attn_weights * torch.log(attn_weights + 1e-8)).sum(dim=1, keepdim=True)

        features = torch.cat([
            before_after.float().unsqueeze(1),  # (B, 1)
            hi_mean,  # (B, 1)
            hi_std,   # (B, 1)
            hi_attn_mean,  # (B, 1)
            hi_trend,  # (B, 1)
            attn_entropy  # (B, 1)
        ], dim=1)  # (B, 6)

        # Extract features and classify
        feature_repr = self.feature_extractor(features)  # (B, 32)
        logits = self.classifier(feature_repr)  # (B, n_classes)

        return logits

class FlightDiffAwareReconstructor(nn.Module):
    """
    Flight data model with monotonic HI learning and classification
    """
    def __init__(self, in_ch, trend_ch=4, hidden=128, n_classes=19):
        super().__init__()
        self.encoder = TrendEncoder(in_ch, trend_ch)
        self.recon = ReconHead(in_ch, hidden)
        self.classifier = FlightClassifier(n_classes)

    def forward(self, x, mask):
        """
        Args:
            x: (B, L, C) input sequences
            mask: (B, L) validity mask
        Returns:
            hi: (B, L) health index
            y_hat: (B, L-2, C) reconstruction prediction
            h_multi: (B, L, trend_ch) intermediate features
            weights: (B, L, K) operator weights
            temperature: (B, L) temperature parameters
        """
        B, L, C = x.shape

        # Generate health index and features
        hi, h_multi, weights, temperature = self.encoder(x)  # hi:(B,L)

        # Reconstruction on 0:L-2 → 1:L-1
        if L >= 3:
            x_in = x[:, :-2, :]  # (B, L-2, C)
            h_in = hi[:, :-2]    # (B, L-2)
            y_hat = self.recon(x_in, h_in)  # (B, L-2, C)
        else:
            y_hat = torch.zeros(B, 0, C, device=x.device)

        return hi, y_hat, h_multi, weights, temperature

# ============================================================
# 3) Load data and model
# ============================================================
def load_flight_data(data_dir):
    """Load processed flight data with corrected key names"""
    train_path = os.path.join(data_dir, "train_samples_processed_sample_len_150.pkl")
    val_path = os.path.join(data_dir, "val_samples_processed_sample_len_150.pkl")
    test_path = os.path.join(data_dir, "test_samples_processed_sample_len_150.pkl")
    label_mappings_path = os.path.join(data_dir, "label_mappings_sample_len_150.pkl")

    with open(train_path, 'rb') as f:
        train_samples = pickle.load(f)
    with open(val_path, 'rb') as f:
        val_samples = pickle.load(f)
    with open(test_path, 'rb') as f:
        test_samples = pickle.load(f)
    with open(label_mappings_path, 'rb') as f:
        label_mappings = pickle.load(f)

    return train_samples, val_samples, test_samples, label_mappings

@torch.no_grad()
def eval_epoch_flight(model, loader, device, mode="full"):
    """Evaluation function for flight data with different modes"""
    model.eval()
    total = {"mse": 0.0, "acc": 0.0, "hi_mean": 0.0}
    n_batch = 0
    n_acc = 0
    n_tot = 0

    for batch in loader:
        x = batch["x"].to(device)
        mask = batch["mask"].to(device)
        labels = batch["labels"].to(device)
        before_after = batch["before_after"].to(device)
        valid_lengths = batch["valid_lengths"].to(device)

        B, L, C = x.shape

        # Generate predictions
        hi, y_hat, h_multi, weights, temperature = model(x, mask)

        # Classification
        if mode in ["cls", "full"]:
            # Clamp labels to ensure they are in valid range
            labels = torch.clamp(labels, 0, 18)  # Ensure labels stay within the range [0, 18]
            logits = model.classifier(hi, before_after, mask)
            pred = logits.argmax(dim=1)
            n_acc += (pred == labels).sum().item()
            n_tot += labels.numel()

        # HI statistics
        valid_counts = mask.float().sum(dim=1, keepdim=True).clamp_min(1.0)
        hi_mean = (hi * mask.float()).sum(dim=1, keepdim=True) / valid_counts
        total["hi_mean"] += hi_mean.mean().item()

        n_batch += 1

    for k in ["mse", "hi_mean"]:
        total[k] /= max(n_batch, 1)

    if mode in ["cls", "full"]:
        total["acc"] = n_acc / max(n_tot, 1)
    else:
        total["acc"] = 0.0

    return total

# ============================================================
# 4) Stage 2 Training Function
# ============================================================
def stage2_classification_training(train_samples, val_samples, test_samples, label_mappings,
                                 epochs=30, batch_size=640, lr=1e-3, device=None, patience=10,
                                 model_path=None):
    """Stage 2: Classification training with frozen encoder"""
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"Training on device: {device}")

    # Create datasets - use correct label key
    train_dataset = FlightDataset(train_samples, label_key='class')
    val_dataset = FlightDataset(val_samples, label_key='class')
    test_dataset = FlightDataset(test_samples, label_key='class')

    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=flight_collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=flight_collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=flight_collate_fn)

    # Model parameters
    C = train_dataset.C  # Number of features

    # Determine number of classes from label mappings
    if 'class_to_id' in label_mappings:
        n_classes = len(label_mappings['class_to_id'])
    elif 'id_to_class' in label_mappings:
        n_classes = len(label_mappings['id_to_class'])
    else:
        # If no mapping found, infer from data
        all_labels = set()
        for sample in train_samples.values():
            all_labels.add(sample['class'])
        n_classes = len(all_labels)

    print(f"Model config: features={C}, classes={n_classes}")

    # Create model
    model = FlightDiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=n_classes).to(device)

    # Load pre-trained model if path is provided
    if model_path and os.path.exists(model_path):
        print(f"Loading pre-trained model from: {model_path}")
        model.load_state_dict(torch.load(model_path, map_location=device))
        print("Model loaded successfully!")
    else:
        print("Warning: No pre-trained model found, training from scratch")

    # Print data summary
    print(f"Dataset Summary:")
    print(f"  Train: {len(train_samples)} samples")
    print(f"  Val: {len(val_samples)} samples")
    print(f"  Test: {len(test_samples)} samples")
    print(f"  Classes: {n_classes}")
    print(f"  Features: {C}")

    print("\n" + "="*80)
    print("STAGE 2: CLASSIFICATION TRAINING (FROZEN ENCODER)")
    print("="*80)

    # Freeze encoder and recon
    for param in model.encoder.parameters():
        param.requires_grad = False
    for param in model.recon.parameters():
        param.requires_grad = False

    # Only train classifier
    opt_stage2 = torch.optim.AdamW(model.classifier.parameters(), lr=lr*0.5, weight_decay=1e-5)
    ce = nn.CrossEntropyLoss()

    best_cls_acc = 0.0
    no_improve_count = 0

    for epoch in range(1, epochs + 1):
        print(f"\n[Stage 2 - Epoch {epoch}/{epochs}] Training classification...")
        model.train()

        train_loss = 0.0
        n_batches = 0

        for batch_idx, batch in enumerate(train_loader):
            if batch_idx % max(1, len(train_loader)//5) == 0:
                progress = (batch_idx + 1) / len(train_loader) * 100
                print(f"    Progress: {progress:.1f}%")

            x = batch["x"].to(device)
            mask = batch["mask"].to(device)
            labels = batch["labels"].to(device)
            before_after = batch["before_after"].to(device)

            # Clamp labels to ensure they are in valid range
            labels = torch.clamp(labels, 0, n_classes-1)

            # Forward pass (encoder frozen)
            with torch.no_grad():
                hi, _, _, _, _ = model(x, mask)

            # Classification loss
            logits = model.classifier(hi, before_after, mask)
            cls_loss = ce(logits, labels)

            # Backward pass
            opt_stage2.zero_grad()
            cls_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.classifier.parameters(), 1.0)
            opt_stage2.step()

            train_loss += cls_loss.item()
            n_batches += 1

        train_loss /= n_batches

        # Validation (classification only)
        val_metrics = eval_epoch_flight(model, val_loader, device, mode="cls")

        print(f"[Stage 2 - Epoch {epoch:03d}] Train: {train_loss:.4f} | Val: acc={val_metrics['acc']:.3f}")

        # Early stopping for classification
        if val_metrics['acc'] > best_cls_acc:
            best_cls_acc = val_metrics['acc']
            no_improve_count = 0
            print(f"    ✓ New best classification accuracy: {best_cls_acc:.3f}")

            # Save best model
            save_path = str(resolve_data_file("flight_maintenance_stage2.pt"))
            try:
                torch.save(model.state_dict(), save_path)
                print(f"    ✓ Best model saved at: {save_path}")
            except Exception as e:
                print(f"    ✗ Save failed: {e}")
        else:
            no_improve_count += 1
            if no_improve_count >= patience:
                print(f"\n[Stage 2 Early Stopping] No improvement for {patience} epochs.")
                break

    # Final test evaluation
    print("\n" + "="*80)
    print("FINAL TEST SET EVALUATION (STAGE 2)")
    print("="*80)
    test_metrics = eval_epoch_flight(model, test_loader, device, mode="cls")
    print(f"[Test] acc={test_metrics['acc']:.3f} hi_mean={test_metrics['hi_mean']:.3f}")

    return model

# ============================================================
# 5) Main execution
# ============================================================
# Load flight data
data_dir = str(DATA_DIR)
train_samples, val_samples, test_samples, label_mappings = load_flight_data(data_dir)

# Path to pre-trained model (from Stage 1)
model_path = str(resolve_data_file("flight_maintenance_3stage.pt"))

# Stage 2: Classification training
print("Starting Stage 2: Classification Training...")
model = stage2_classification_training(
    train_samples=train_samples,
    val_samples=val_samples,
    test_samples=test_samples,
    label_mappings=label_mappings,
    epochs=30,
    batch_size=640,
    lr=1e-3,
    device=None,
    patience=10,
    model_path=model_path
)

print("\nStage 2 training completed!")


In [ ]:
# ============================================================
# 6) Load best checkpoint (Stage3>Stage2) and plot prediction vs original
# ============================================================
import os
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# ---------- Configuration ----------
num_samples = 4     # Number of test samples to plot
channel_idx = 0     # Sensor channel index [0 .. C-1]
path_stage3 = str(resolve_data_file("flight_maintenance_3stage.pt"))
path_stage2 = str(resolve_data_file("flight_maintenance_stage2.pt"))
batch_size  = 640

# ---------- Construct test_loader (independent of training function's DataLoader) ----------
test_dataset = FlightDataset(test_samples, label_key='class')
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=flight_collate_fn)

# ---------- Reconstruct model architecture and load best weights ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
C = test_dataset.C

# Calculate number of classes
if 'class_to_id' in label_mappings:
    n_classes = len(label_mappings['class_to_id'])
elif 'id_to_class' in label_mappings:
    n_classes = len(label_mappings['id_to_class'])
else:
    n_classes = len(set(s['class'] for s in train_samples.values()))

model_plot = FlightDiffAwareReconstructor(in_ch=C, trend_ch=4, hidden=128, n_classes=n_classes).to(device)

ckpt_path = path_stage3 if os.path.exists(path_stage3) else (path_stage2 if os.path.exists(path_stage2) else None)
if ckpt_path is not None:
    print(f"[Load] Using checkpoint: {ckpt_path}")
    state = torch.load(ckpt_path, map_location=device)
    model_plot.load_state_dict(state, strict=False)
else:
    print("[Load] No checkpoint found, fallback to the in-memory 'model' if exists.")
    try:
        model_plot.load_state_dict(model.state_dict(), strict=False)
        print("[Load] Copied weights from the trained 'model' variable.")
    except Exception as e:
        raise RuntimeError("No available weights to load. Train the model or provide a checkpoint.") from e

model_plot.eval()

# ---------- Get a batch and forward pass ----------
batch = next(iter(test_loader))
x    = batch["x"].to(device)           # (B, L, C)
mask = batch["mask"].to(device)         # (B, L)
vlen = batch["valid_lengths"]           # (B,)
B, L, C = x.shape

with torch.no_grad():
    hi, y_hat, h_multi, weights, temperature = model_plot(x, mask)  # y_hat: (B, L-2, C) predicts x[t+1]

# ---------- Plot multiple samples ----------
num_samples = min(num_samples, x.size(0))  # Ensure we don't exceed batch size

fig, axes = plt.subplots(num_samples, 1, figsize=(12, 3*num_samples))
if num_samples == 1:
    axes = [axes]  # Make it iterable for single subplot

for idx in range(num_samples):
    i = idx  # Sample index

    # Valid length (prefer valid_lengths; fallback to mask.sum)
    valid_len = int(vlen[i].item()) if vlen is not None else int(mask[i].sum().item())

    # y_hat time length
    L_pred = y_hat.size(1)   # Usually L-2
    if L_pred <= 0:
        print(f"Sample {i}: Sequence too short (L < 3), skipping reconstruction comparison.")
        axes[idx].text(0.5, 0.5, f"Sample {i}: Sequence too short",
                      ha='center', va='center', transform=axes[idx].transAxes)
        continue

    # Can only compare up to valid end of target: target is x[:, 1:1+T_plot]
    # Valid target needs index < valid_len, and we need t+1 index to exist → max t is valid_len-2
    T_plot = max(0, min(L_pred, valid_len - 2))
    if T_plot <= 0:
        print(f"Sample {i}: Not enough valid length for reconstruction comparison (valid_len < 3), skipping.")
        axes[idx].text(0.5, 0.5, f"Sample {i}: Not enough valid length",
                      ha='center', va='center', transform=axes[idx].transAxes)
        continue

    # Extract original (target) and predicted sequences for the same channel, convert to CPU numpy
    orig_series = x[i, 1:1+T_plot, channel_idx].detach().cpu().numpy()    # Target: x[t+1]
    pred_series = y_hat[i, :T_plot,   channel_idx].detach().cpu().numpy()  # Prediction: y_hat[t] ≈ x[t+1]
    t = np.arange(1, 1+T_plot)

    # Plot (overlay on single subplot)
    axes[idx].plot(t, orig_series, label="Original sequence (target x[t+1])", color='blue')
    axes[idx].plot(t, pred_series, label="Predicted sequence (y_hat[t])", linestyle="--", color='red')
    axes[idx].set_xlabel("Time step")
    axes[idx].set_ylabel(f"Sensor channel {channel_idx}")
    axes[idx].set_title(f"Test sample {i} | Channel {channel_idx}: Original vs Predicted")
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

i = 600  # First sample
valid_len = int(vlen[i].item()) if vlen is not None else int(mask[i].sum().item())
L_pred = y_hat.size(1)
T_plot = max(0, min(L_pred, valid_len - 2))

if T_plot > 0:
    fig, axes = plt.subplots(C, 1, figsize=(12, 2*C))
    if C == 1:
        axes = [axes]

    for ch in range(C):
        orig_ch = x[i, 1:1+T_plot, ch].detach().cpu().numpy()
        pred_ch = y_hat[i, :T_plot, ch].detach().cpu().numpy()
        t = np.arange(1, 1+T_plot)

        axes[ch].plot(t, orig_ch, label="Original", color='blue')
        axes[ch].plot(t, pred_ch, label="Predicted", linestyle="--", color='red')
        axes[ch].set_title(f"Sample {i} | Channel {ch}")
        axes[ch].set_xlabel("Time step")
        axes[ch].set_ylabel(f"Channel {ch}")
        axes[ch].legend()
        axes[ch].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
